# 06_POINT_PROCESS_BASELINES

## Notebook purpose

This notebook evaluates whether the authoritative V0.1 BUY/SELL event stream exhibits temporal dependence that cannot be explained by simpler non-Hawkes point-process models.

The notebook will:

1. verify the Notebook 05 to Notebook 06 handoff;
2. preserve the authoritative `SAME_MS_SAME_SIDE_BURSTS` event definition;
3. preserve exact-time simultaneous-batch semantics;
4. characterize event rates, interarrival durations, dispersion, serial dependence, and BUY/SELL interaction;
5. fit and compare non-Hawkes point-process baselines;
6. determine whether deterministic or adaptive rate variation explains the observed clustering;
7. reconcile V0.1 results against V0.0 point-process reference artifacts;
8. decide whether Notebook 07 Hawkes estimation is authorized, restricted, conditional, or not authorized.

This notebook ends before Hawkes-process estimation.

---

## Authoritative upstream state

### Source run prefix

`BTCUSDT_spot_20260710T063746Z_c8b5bf12`

### V0.1 run ID

`v0_1_20260714T090616Z_e82325081a81`

### Primary event representation

`SAME_MS_SAME_SIDE_BURSTS`

### Primary event time

`event_time_ns`

### Primary event partition

`event_partition`

### Primary timestamp interface

`SIMULTANEOUS_EVENT_BATCH_REQUIRED`

### Authoritative event-stream producer

`04_EVENT_STREAM_CONSTRUCTION.ipynb`

### Authoritative market-state feature producer

`05_MARKET_STATE_FEATURES.ipynb`

### Expected upstream counts

- primary event rows: `13,887`
- primary scoring batches: `13,564`
- simultaneous exact-time batches: `249`
- mixed BUY/SELL exact-time batches: `41`

All expected values must be verified against authoritative manifests and handoff artifacts before analysis begins.

---

## Timing and causality contract

For every exact-time scoring batch:

1. all model intensities and forecasts must be computed using history strictly before `event_time_ns`;
2. all events in the batch must be scored against the same pre-batch history;
3. the full batch may update model state only after all batch observations have been scored.

The following are forbidden:

- zero-lag within-batch excitation;
- arbitrary timestamp jitter;
- deletion of tied events;
- artificial ordering of BUY and SELL events inside the same exact-time batch;
- treating collector sequence as elapsed physical time;
- using current-batch information in pre-batch forecasts;
- using future labels as model inputs;
- crossing partition boundaries when constructing durations, lags, conditional-response windows, or future expectations.

---

## Partition-use contract

### DEVELOPMENT

May be used for:

- baseline specification selection;
- rolling-window or half-life selection;
- diagnostic-grid selection;
- numerical tolerance selection;
- model-family selection;
- implementation debugging.

### CALIBRATION

May be used only after DEVELOPMENT choices are frozen.

May be used for:

- final parameter estimation;
- locked prequential baseline comparison;
- engineering confirmation of DEVELOPMENT findings.

CALIBRATION is not an untouched out-of-sample partition because it participated in earlier event-definition work.

### VALIDATION

Event content must not be loaded in this notebook.

VALIDATION remains reserved for later Hawkes-versus-baseline evaluation.

### ENGINEERING_HOLDOUT

Must remain completely unopened.

The final notebook audit must prove that neither VALIDATION nor ENGINEERING_HOLDOUT event observations entered any analytical dataframe, array, model, diagnostic, or plot.

---

## Permitted baseline families

This notebook may estimate and evaluate:

1. pooled homogeneous Poisson;
2. independent BUY/SELL homogeneous Poisson;
3. low-dimensional deterministic elapsed-time Poisson;
4. causal rolling-window event-rate models;
5. causal exponentially weighted event-rate models;
6. exponential, gamma, and Weibull renewal models;
7. an optional first-order exact-time batch-mark transition model.

A formal time-of-day, day-of-week, or cross-day seasonal model is not estimable from the current single-session collection and must be recorded as:

`NOT_APPLICABLE_IN_CURRENT_DATA`

---

## Prohibited models and claims

This notebook must not:

- estimate Hawkes kernels;
- estimate Hawkes branching ratios;
- estimate Hawkes spectral radii;
- optimize Hawkes decay parameters;
- fit state-dependent Hawkes models;
- use the Notebook 05 future-label table;
- train a predictive model on the full Notebook 05 feature set;
- construct a quoting policy;
- simulate fills or queue position;
- report strategy P&L;
- report Sharpe ratio or drawdown;
- claim Hawkes superiority;
- claim profitable predictability;
- claim cross-session or publication-level generalization.

---

## Primary comparison principle

The primary comparison criterion is locked chronological probabilistic performance on CALIBRATION after all model choices have been frozen using DEVELOPMENT.

The notebook must compare models using compatible observation exposure and scoring semantics.

Primary quantities include:

- Poisson count log score;
- point-process likelihood where applicable;
- score per second;
- score per event;
- improvement over homogeneous Poisson;
- block-bootstrap uncertainty for score differences;
- residual dispersion;
- residual autocorrelation;
- residual BUY/SELL cross-correlation;
- time-rescaling diagnostics.

In-sample fit alone is not sufficient to authorize Hawkes estimation.

---

## Required scientific decisions

The notebook must separately determine:

1. whether homogeneous Poisson is inadequate;
2. whether deterministic rate variation is sufficient;
3. whether a causal adaptive-rate baseline is sufficient;
4. whether BUY self-dependence remains;
5. whether SELL self-dependence remains;
6. whether BUY-to-SELL dependence remains;
7. whether SELL-to-BUY dependence remains;
8. whether Hawkes estimation is justified as the next engineering step.

Each scientific decision must be recorded as one of:

- `PRESENT`
- `ABSENT`
- `INCONCLUSIVE`
- `NOT_APPLICABLE`

---

## Permitted terminal statuses

The notebook must finish with exactly one of:

- `PASS_HAWKES_BIVARIATE_AUTHORIZED`
- `PASS_HAWKES_RESTRICTED_FIRST`
- `CONDITIONAL_PASS_HAWKES_DIAGNOSTIC_ONLY`
- `PASS_SIMPLE_BASELINE_SUFFICIENT_HAWKES_NOT_AUTHORIZED`
- `FAIL_BASELINE_PIPELINE_INVALID`

A successful authorization applies only to the current frozen engineering run.

It does not establish cross-session stability, live-trading suitability, economic value, or publication-level evidence.

---

## Output contract

The notebook must write authoritative V0.1 artifacts under the V0.1 project tree only.

Every persisted artifact must record:

- source run prefix;
- V0.1 run ID;
- source-set hash;
- run-config hash;
- producing notebook;
- schema version;
- partition coverage;
- row counts;
- model or diagnostic specification;
- random seed where applicable;
- SHA-256 checksum;
- acceptance status.

No V0.0 file may be modified.

V0.0 point-process outputs may be read only for reconciliation and provenance.

---

## Initial authorization state

- point-process baseline estimation: `AUTHORIZED`
- Hawkes estimation: `NOT_AUTHORIZED`
- Hawkes superiority claim: `NOT_AUTHORIZED`
- state-dependent Hawkes estimation: `NOT_AUTHORIZED`
- strategy construction: `NOT_AUTHORIZED`
- market-making result: `NOT_AUTHORIZED`

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import sys
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Final, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import scipy.optimize as optimize
import scipy.special as special
import scipy.stats as stats
import statsmodels.api as sm
from IPython.display import display


# ============================================================
# Runtime contract
# ============================================================

MINIMUM_PYTHON_VERSION: Final[tuple[int, int]] = (3, 10)

if sys.version_info < MINIMUM_PYTHON_VERSION:
    required = ".".join(map(str, MINIMUM_PYTHON_VERSION))
    observed = platform.python_version()
    raise RuntimeError(
        f"Python {required} or newer is required; observed Python {observed}."
    )

warnings.filterwarnings(
    "error",
    category=pd.errors.SettingWithCopyWarning,
)

pd.options.display.max_columns = 200
pd.options.display.width = 180
pd.options.display.float_format = "{:,.8g}".format


# ============================================================
# Frozen project identity
# ============================================================

NOTEBOOK_NAME: Final[str] = "06_POINT_PROCESS_BASELINES.ipynb"
NOTEBOOK_STAGE: Final[str] = "POINT_PROCESS_BASELINES"

SOURCE_RUN_PREFIX: Final[str] = (
    "BTCUSDT_spot_20260710T063746Z_c8b5bf12"
)
V01_RUN_ID: Final[str] = "v0_1_20260714T090616Z_e82325081a81"
COMBINED_OUTPUT_PREFIX: Final[str] = (
    f"{SOURCE_RUN_PREFIX}__{V01_RUN_ID}"
)

SOURCE_SET_SHA256: Final[str] = (
    "132c83531eec615d279408b5c06f402973114ba3058dfadd2fe58e2e67184c4b"
)
RUN_CONFIG_SHA256: Final[str] = (
    "14aea0efb3c7b6a193b8c4575a440babe8265d2935c6a770bee995f688d59617"
)
RUN_IDENTITY_SHA256: Final[str] = (
    "5eb89cf073c036d70e3767df4b35ace7c31822e52e9b9df19204c42e37fe4198"
)

PRIMARY_EVENT_REPRESENTATION: Final[str] = "SAME_MS_SAME_SIDE_BURSTS"
PRIMARY_EVENT_TIME_COLUMN: Final[str] = "event_time_ns"
PRIMARY_EVENT_PARTITION_COLUMN: Final[str] = "event_partition"
PRIMARY_TIMESTAMP_INTERFACE: Final[str] = (
    "SIMULTANEOUS_EVENT_BATCH_REQUIRED"
)

EXPECTED_PRIMARY_EVENT_ROWS: Final[int] = 13_887
EXPECTED_PRIMARY_SCORING_BATCHES: Final[int] = 13_564
EXPECTED_SIMULTANEOUS_BATCHES: Final[int] = 249
EXPECTED_MIXED_SIDE_BATCHES: Final[int] = 41


# ============================================================
# Project paths
# ============================================================

PROJECT_ROOT: Final[Path] = Path(r"D:\Clown Project")
V00_ROOT: Final[Path] = PROJECT_ROOT / "V0.0"
V01_ROOT: Final[Path] = PROJECT_ROOT / "V0.1"

V01_CONFIG_ROOT: Final[Path] = V01_ROOT / "config"
V01_DATA_ROOT: Final[Path] = V01_ROOT / "data"
V01_ARTIFACT_ROOT: Final[Path] = V01_ROOT / "artifacts"
V01_LOG_ROOT: Final[Path] = V01_ROOT / "logs"

NOTEBOOK06_ARTIFACT_ROOT: Final[Path] = (
    V01_ARTIFACT_ROOT / "point_process_baselines"
)
NOTEBOOK06_TABLE_ROOT: Final[Path] = (
    NOTEBOOK06_ARTIFACT_ROOT / "tables"
)
NOTEBOOK06_FIGURE_ROOT: Final[Path] = (
    NOTEBOOK06_ARTIFACT_ROOT / "figures"
)
NOTEBOOK06_MODEL_ROOT: Final[Path] = (
    NOTEBOOK06_ARTIFACT_ROOT / "models"
)
NOTEBOOK06_DIAGNOSTIC_ROOT: Final[Path] = (
    NOTEBOOK06_ARTIFACT_ROOT / "diagnostics"
)
NOTEBOOK06_HANDOFF_ROOT: Final[Path] = (
    V01_ARTIFACT_ROOT / "handoff"
)

# No directories are created in this cell.
# Path creation is deferred until upstream authority checks pass.


# ============================================================
# Deterministic random-state contract
# ============================================================

RANDOM_SEED: Final[int] = 20260720

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
RNG = np.random.default_rng(RANDOM_SEED)


# ============================================================
# Frozen diagnostic and model-selection grids
# ============================================================

NANOSECONDS_PER_MILLISECOND: Final[int] = 1_000_000
NANOSECONDS_PER_SECOND: Final[int] = 1_000_000_000

COUNT_BIN_WIDTHS_MS: Final[tuple[int, ...]] = (
    10,
    25,
    50,
    100,
    250,
    500,
    1_000,
    2_000,
    5_000,
    10_000,
    30_000,
)

PRIMARY_COUNT_BIN_WIDTH_MS: Final[int] = 100
SLOW_COUNT_BIN_WIDTH_MS: Final[int] = 1_000

PRIMARY_ACF_LAGS: Final[tuple[int, ...]] = (
    1,
    2,
    3,
    5,
    10,
    20,
    50,
)

SLOW_ACF_LAGS: Final[tuple[int, ...]] = (
    1,
    2,
    5,
    10,
    30,
)

CROSS_CORRELATION_BIN_WIDTHS_MS: Final[tuple[int, ...]] = (
    10,
    100,
    1_000,
)

CONDITIONAL_RESPONSE_BANDS_MS: Final[
    tuple[tuple[int, int], ...]
] = (
    (0, 10),
    (10, 25),
    (25, 50),
    (50, 100),
    (100, 250),
    (250, 500),
    (500, 1_000),
    (1_000, 2_000),
    (2_000, 5_000),
)

ROLLING_RATE_WINDOWS_MS: Final[tuple[int, ...]] = (
    250,
    500,
    1_000,
    2_000,
    5_000,
    10_000,
    30_000,
    60_000,
)

EWMA_HALF_LIVES_MS: Final[tuple[int, ...]] = (
    250,
    500,
    1_000,
    2_000,
    5_000,
    10_000,
    30_000,
    60_000,
)

DETERMINISTIC_RATE_SPECIFICATIONS: Final[tuple[str, ...]] = (
    "CONSTANT",
    "LINEAR_ELAPSED_TIME",
    "QUADRATIC_ELAPSED_TIME",
    "NATURAL_SPLINE_DF3",
)

RENEWAL_FAMILIES: Final[tuple[str, ...]] = (
    "EXPONENTIAL",
    "GAMMA",
    "WEIBULL",
)

CHRONOLOGICAL_BLOCK_SECONDS: Final[int] = 60
BOOTSTRAP_BLOCK_SECONDS: Final[int] = 10

N_MONTE_CARLO_SIMULATIONS: Final[int] = 2_000
N_BLOCK_BOOTSTRAP_REPLICATES: Final[int] = 2_000

INTENSITY_FLOOR_PER_SECOND: Final[float] = 1e-12
LOG_SCORE_EPSILON: Final[float] = np.finfo(np.float64).tiny
MINIMUM_RENEWAL_DURATIONS: Final[int] = 100

FLOAT_ABSOLUTE_TOLERANCE: Final[float] = 1e-10
FLOAT_RELATIVE_TOLERANCE: Final[float] = 1e-8


# ============================================================
# Immutable notebook configuration
# ============================================================

@dataclass(frozen=True, slots=True)
class Notebook06Config:
    notebook_name: str
    notebook_stage: str
    source_run_prefix: str
    v01_run_id: str
    random_seed: int
    primary_event_representation: str
    primary_event_time_column: str
    primary_event_partition_column: str
    primary_timestamp_interface: str
    primary_count_bin_width_ms: int
    chronological_block_seconds: int
    bootstrap_block_seconds: int
    monte_carlo_simulations: int
    block_bootstrap_replicates: int
    intensity_floor_per_second: float
    minimum_renewal_durations: int


NOTEBOOK_CONFIG = Notebook06Config(
    notebook_name=NOTEBOOK_NAME,
    notebook_stage=NOTEBOOK_STAGE,
    source_run_prefix=SOURCE_RUN_PREFIX,
    v01_run_id=V01_RUN_ID,
    random_seed=RANDOM_SEED,
    primary_event_representation=PRIMARY_EVENT_REPRESENTATION,
    primary_event_time_column=PRIMARY_EVENT_TIME_COLUMN,
    primary_event_partition_column=PRIMARY_EVENT_PARTITION_COLUMN,
    primary_timestamp_interface=PRIMARY_TIMESTAMP_INTERFACE,
    primary_count_bin_width_ms=PRIMARY_COUNT_BIN_WIDTH_MS,
    chronological_block_seconds=CHRONOLOGICAL_BLOCK_SECONDS,
    bootstrap_block_seconds=BOOTSTRAP_BLOCK_SECONDS,
    monte_carlo_simulations=N_MONTE_CARLO_SIMULATIONS,
    block_bootstrap_replicates=N_BLOCK_BOOTSTRAP_REPLICATES,
    intensity_floor_per_second=INTENSITY_FLOOR_PER_SECOND,
    minimum_renewal_durations=MINIMUM_RENEWAL_DURATIONS,
)


def canonical_json_sha256(payload: Mapping[str, Any]) -> str:
    """Return the SHA-256 digest of a canonical JSON-serializable mapping."""
    canonical_payload = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(canonical_payload).hexdigest()


NOTEBOOK_CONFIG_SHA256 = canonical_json_sha256(asdict(NOTEBOOK_CONFIG))


# ============================================================
# Runtime provenance
# ============================================================

runtime_provenance = pd.DataFrame(
    [
        {"component": "python", "version": platform.python_version()},
        {"component": "numpy", "version": np.__version__},
        {"component": "pandas", "version": pd.__version__},
        {"component": "scipy", "version": scipy.__version__},
        {"component": "statsmodels", "version": sm.__version__},
        {"component": "matplotlib", "version": plt.matplotlib.__version__},
        {"component": "platform", "version": platform.platform()},
    ]
)

setup_summary = pd.DataFrame(
    [
        {"field": "notebook", "value": NOTEBOOK_NAME},
        {"field": "source_run_prefix", "value": SOURCE_RUN_PREFIX},
        {"field": "v01_run_id", "value": V01_RUN_ID},
        {
            "field": "primary_event_representation",
            "value": PRIMARY_EVENT_REPRESENTATION,
        },
        {
            "field": "timestamp_interface",
            "value": PRIMARY_TIMESTAMP_INTERFACE,
        },
        {"field": "random_seed", "value": RANDOM_SEED},
        {
            "field": "notebook_config_sha256",
            "value": NOTEBOOK_CONFIG_SHA256,
        },
        {
            "field": "initial_hawkes_authorization",
            "value": "NOT_AUTHORIZED",
        },
    ]
)

display(setup_summary)
display(runtime_provenance)

print(
    "Notebook 06 runtime and immutable configuration initialized. "
    "No upstream data or protected partition content has been loaded."
)

,field,value
0,notebook,06_POINT_PROCESS_BASELINES.ipynb
1,source_run_prefix,BTCUSDT_spot_20260710T063746Z_c8b5bf12
2,v01_run_id,v0_1_20260714T090616Z_e82325081a81
3,primary_event_representation,SAME_MS_SAME_SIDE_BURSTS
4,timestamp_interface,SIMULTANEOUS_EVENT_BATCH_REQUIRED
5,random_seed,20260720
6,notebook_config_sha256,583b70096dd27c6ec4d7d807ef493e27b018f559f7eb1a...
7,initial_hawkes_authorization,NOT_AUTHORIZED


,component,version
0,python,3.11.9
1,numpy,2.2.0
2,pandas,2.3.2
3,scipy,1.16.1
4,statsmodels,0.14.5
5,matplotlib,3.10.6
6,platform,Windows-10-10.0.26200-SP0


Notebook 06 runtime and immutable configuration initialized. No upstream data or protected partition content has been loaded.


In [2]:
# ============================================================
# Upstream control-artifact discovery and verification
# ============================================================

from dataclasses import dataclass
from pathlib import Path
from typing import Literal


# ------------------------------------------------------------
# Frozen expected control-artifact hashes
# ------------------------------------------------------------

EXPECTED_NOTEBOOK_00_MANIFEST_FILE_SHA256: Final[str] = (
    "e12966301d92e61656715f2a6d397786820de836d8156dd70092250619cbb00e"
)

EXPECTED_NOTEBOOK_04_MANIFEST_PAYLOAD_SHA256: Final[str] = (
    "c8454d2059f1b5e50675aa156e1eabd7762642cbf624065b496bea1edb56ff53"
)
EXPECTED_NOTEBOOK_04_TO_05_HANDOFF_PAYLOAD_SHA256: Final[str] = (
    "f0544516d59fbabc24d1c8ff42063c96e7c6c4b72fcd5cedbd8ef0a2dfc38a28"
)
EXPECTED_NOTEBOOK_04_FINAL_ACCEPTANCE_PAYLOAD_SHA256: Final[str] = (
    "d056dab3330dff3be55ccc4d5a7aa3acf6ba7569712bd3588f3b088d2a1ef8cd"
)

EXPECTED_NOTEBOOK_05_MANIFEST_PAYLOAD_SHA256: Final[str] = (
    "b83bf3be102042259849e08afce17aba2bb1883b5cced04a57408f2524f0f168"
)
EXPECTED_NOTEBOOK_05_TO_06_HANDOFF_PAYLOAD_SHA256: Final[str] = (
    "89fb688420909c230e82bf8b43575d8a5f144ac372846a9a40fcfd1476966ce7"
)
EXPECTED_NOTEBOOK_05_FINAL_ACCEPTANCE_PAYLOAD_SHA256: Final[str] = (
    "199c7cc486f1f6b4878d4370c59737fb6e9a781efedf935b5de8ac9fd0efee42"
)


# ------------------------------------------------------------
# Exact control-artifact paths
# ------------------------------------------------------------

MANIFEST_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "manifests"
HANDOFF_ROOT: Final[Path] = V01_ARTIFACT_ROOT / "handoff"

NOTEBOOK_00_OUTPUT_MANIFEST_PATH: Final[Path] = (
    MANIFEST_ROOT
    / (
        f"{COMBINED_OUTPUT_PREFIX}"
        "__00_V01_RUN_CONTRACT"
        "__notebook_00_output_manifest.json"
    )
)

NOTEBOOK_04_OUTPUT_MANIFEST_PATH: Final[Path] = (
    MANIFEST_ROOT
    / (
        f"{COMBINED_OUTPUT_PREFIX}"
        "__04_EVENT_STREAM_CONSTRUCTION"
        "__notebook_04_output_manifest.json"
    )
)

NOTEBOOK_04_TO_05_HANDOFF_PATH: Final[Path] = (
    HANDOFF_ROOT
    / (
        f"{COMBINED_OUTPUT_PREFIX}"
        "__04_EVENT_STREAM_CONSTRUCTION"
        "__notebook_04_to_notebook_05_handoff.json"
    )
)

NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT_PATH: Final[Path] = (
    MANIFEST_ROOT
    / (
        f"{COMBINED_OUTPUT_PREFIX}"
        "__04_EVENT_STREAM_CONSTRUCTION"
        "__notebook_04_final_acceptance_report.json"
    )
)

NOTEBOOK_05_OUTPUT_MANIFEST_PATH: Final[Path] = (
    MANIFEST_ROOT
    / (
        f"{COMBINED_OUTPUT_PREFIX}"
        "__05_MARKET_STATE_FEATURES"
        "__notebook_05_output_manifest.json"
    )
)

NOTEBOOK_05_TO_06_HANDOFF_PATH: Final[Path] = (
    HANDOFF_ROOT
    / (
        f"{COMBINED_OUTPUT_PREFIX}"
        "__05_MARKET_STATE_FEATURES"
        "__to_06_POINT_PROCESS_BASELINES"
        "__handoff.json"
    )
)

NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT_PATH: Final[Path] = (
    MANIFEST_ROOT
    / (
        f"{COMBINED_OUTPUT_PREFIX}"
        "__05_MARKET_STATE_FEATURES"
        "__final_acceptance_report.json"
    )
)


# ------------------------------------------------------------
# Validation helpers
# ------------------------------------------------------------

def require(condition: bool, message: str) -> None:
    """Raise a runtime error when a required contract is not satisfied."""
    if not condition:
        raise RuntimeError(message)


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    """Return the SHA-256 digest of a file without loading it fully into memory."""
    path = Path(path)

    require(
        path.is_file(),
        f"Required file does not exist or is not a regular file: {path}",
    )

    hasher = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            hasher.update(block)

    return hasher.hexdigest()


def read_json_object(path: Path) -> dict[str, Any]:
    """Load a JSON document and require a top-level object."""
    path = Path(path)

    require(
        path.is_file(),
        f"Required JSON artifact does not exist: {path}",
    )

    with path.open("r", encoding="utf-8") as handle:
        document = json.load(handle)

    require(
        isinstance(document, dict),
        f"JSON artifact must contain a top-level object: {path}",
    )

    return document


def path_is_within(candidate: Path, root: Path) -> bool:
    """Return whether a path resolves within the declared root."""
    candidate_resolved = Path(candidate).resolve(strict=False)
    root_resolved = Path(root).resolve(strict=False)

    try:
        candidate_resolved.relative_to(root_resolved)
        return True
    except ValueError:
        return False


def verify_control_path(path: Path) -> None:
    """Require a control artifact to remain inside V0.1 and outside V0.0."""
    resolved = Path(path).resolve(strict=False)

    require(
        path_is_within(resolved, V01_ROOT),
        f"Control artifact is outside the V0.1 root: {resolved}",
    )
    require(
        not path_is_within(resolved, V00_ROOT),
        f"Control artifact improperly enters the immutable V0.0 tree: {resolved}",
    )


def verify_self_hashed_json(
    path: Path,
    *,
    hash_field: str,
    expected_payload_sha256: str,
) -> tuple[dict[str, Any], str, str]:
    """
    Verify a flat self-hashed JSON document.

    Returns
    -------
    document
        Parsed JSON object.
    observed_payload_sha256
        Canonical hash after removing the embedded hash field.
    observed_raw_sha256
        SHA-256 digest of the persisted file bytes.
    """
    document = read_json_object(path)

    embedded_payload_sha256 = document.get(hash_field)

    require(
        isinstance(embedded_payload_sha256, str),
        f"{path.name} is missing string field {hash_field!r}.",
    )

    payload_without_hash = {
        key: value
        for key, value in document.items()
        if key != hash_field
    }

    observed_payload_sha256 = canonical_json_sha256(
        payload_without_hash
    )
    observed_raw_sha256 = sha256_file(path)

    require(
        embedded_payload_sha256 == observed_payload_sha256,
        (
            f"Embedded self-hash verification failed for {path.name}: "
            f"embedded={embedded_payload_sha256}, "
            f"observed={observed_payload_sha256}"
        ),
    )
    require(
        observed_payload_sha256 == expected_payload_sha256,
        (
            f"Frozen payload hash mismatch for {path.name}: "
            f"expected={expected_payload_sha256}, "
            f"observed={observed_payload_sha256}"
        ),
    )

    return (
        document,
        observed_payload_sha256,
        observed_raw_sha256,
    )


def require_fields(
    document: Mapping[str, Any],
    expected_fields: Mapping[str, Any],
    *,
    artifact_label: str,
) -> None:
    """Require exact values for a declared subset of document fields."""
    mismatches: list[str] = []

    for field_name, expected_value in expected_fields.items():
        observed_value = document.get(field_name)

        if observed_value != expected_value:
            mismatches.append(
                f"{field_name}: expected={expected_value!r}, "
                f"observed={observed_value!r}"
            )

    require(
        not mismatches,
        (
            f"{artifact_label} identity or authority mismatch:\n- "
            + "\n- ".join(mismatches)
        ),
    )


@dataclass(frozen=True, slots=True)
class ControlArtifactAuditRow:
    artifact_label: str
    artifact_type: str
    resolved_path: str
    hash_mode: str
    expected_sha256: str
    observed_sha256: str
    raw_file_sha256: str
    inside_v01: bool
    outside_v00: bool
    identity_verified: bool
    authority_verified: bool
    verified: bool


control_audit_rows: list[ControlArtifactAuditRow] = []


# ------------------------------------------------------------
# Verify every declared path before parsing any document
# ------------------------------------------------------------

CONTROL_ARTIFACT_PATHS: Final[tuple[Path, ...]] = (
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH,
    NOTEBOOK_04_OUTPUT_MANIFEST_PATH,
    NOTEBOOK_04_TO_05_HANDOFF_PATH,
    NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT_PATH,
    NOTEBOOK_05_OUTPUT_MANIFEST_PATH,
    NOTEBOOK_05_TO_06_HANDOFF_PATH,
    NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT_PATH,
)

for control_path in CONTROL_ARTIFACT_PATHS:
    verify_control_path(control_path)

    require(
        control_path.is_file(),
        f"Required upstream control artifact is missing: {control_path}",
    )


# ------------------------------------------------------------
# Notebook 00 wrapped manifest verification
# ------------------------------------------------------------

NOTEBOOK_00_OUTPUT_MANIFEST_DOCUMENT = read_json_object(
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH
)
NOTEBOOK_00_OUTPUT_MANIFEST_RAW_SHA256 = sha256_file(
    NOTEBOOK_00_OUTPUT_MANIFEST_PATH
)

require(
    NOTEBOOK_00_OUTPUT_MANIFEST_RAW_SHA256
    == EXPECTED_NOTEBOOK_00_MANIFEST_FILE_SHA256,
    (
        "Notebook 00 output-manifest file hash mismatch: "
        f"expected={EXPECTED_NOTEBOOK_00_MANIFEST_FILE_SHA256}, "
        f"observed={NOTEBOOK_00_OUTPUT_MANIFEST_RAW_SHA256}"
    ),
)

NOTEBOOK_00_OUTPUT_MANIFEST_METADATA = (
    NOTEBOOK_00_OUTPUT_MANIFEST_DOCUMENT.get("artifact_metadata")
)
NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD = (
    NOTEBOOK_00_OUTPUT_MANIFEST_DOCUMENT.get("payload")
)

require(
    isinstance(NOTEBOOK_00_OUTPUT_MANIFEST_METADATA, dict),
    "Notebook 00 output manifest lacks artifact_metadata.",
)
require(
    isinstance(NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD, dict),
    "Notebook 00 output manifest lacks payload.",
)

NOTEBOOK_00_OBSERVED_PAYLOAD_SHA256 = canonical_json_sha256(
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD
)

require(
    NOTEBOOK_00_OUTPUT_MANIFEST_METADATA.get("payload_sha256")
    == NOTEBOOK_00_OBSERVED_PAYLOAD_SHA256,
    "Notebook 00 embedded payload hash verification failed.",
)

require_fields(
    NOTEBOOK_00_OUTPUT_MANIFEST_METADATA,
    {
        "artifact_type": "NOTEBOOK_00_OUTPUT_MANIFEST",
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "source_set_hash": SOURCE_SET_SHA256,
        "v0_1_run_id": V01_RUN_ID,
    },
    artifact_label="Notebook 00 output-manifest metadata",
)

require_fields(
    NOTEBOOK_00_OUTPUT_MANIFEST_PAYLOAD,
    {
        "manifest_type": "NOTEBOOK_00_OUTPUT_MANIFEST",
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "source_set_hash": SOURCE_SET_SHA256,
        "v0_1_run_id": V01_RUN_ID,
        "v0_1_run_config_hash": RUN_CONFIG_SHA256,
        "v0_1_run_identity_hash": RUN_IDENTITY_SHA256,
    },
    artifact_label="Notebook 00 output-manifest payload",
)

control_audit_rows.append(
    ControlArtifactAuditRow(
        artifact_label="notebook_00_output_manifest",
        artifact_type="NOTEBOOK_00_OUTPUT_MANIFEST",
        resolved_path=str(
            NOTEBOOK_00_OUTPUT_MANIFEST_PATH.resolve(strict=False)
        ),
        hash_mode="RAW_FILE_AND_WRAPPED_PAYLOAD",
        expected_sha256=EXPECTED_NOTEBOOK_00_MANIFEST_FILE_SHA256,
        observed_sha256=NOTEBOOK_00_OUTPUT_MANIFEST_RAW_SHA256,
        raw_file_sha256=NOTEBOOK_00_OUTPUT_MANIFEST_RAW_SHA256,
        inside_v01=True,
        outside_v00=True,
        identity_verified=True,
        authority_verified=True,
        verified=True,
    )
)


# ------------------------------------------------------------
# Notebook 04 flat control documents
# ------------------------------------------------------------

(
    NOTEBOOK_04_OUTPUT_MANIFEST,
    NOTEBOOK_04_OUTPUT_MANIFEST_PAYLOAD_SHA256,
    NOTEBOOK_04_OUTPUT_MANIFEST_RAW_SHA256,
) = verify_self_hashed_json(
    NOTEBOOK_04_OUTPUT_MANIFEST_PATH,
    hash_field="manifest_payload_sha256",
    expected_payload_sha256=(
        EXPECTED_NOTEBOOK_04_MANIFEST_PAYLOAD_SHA256
    ),
)

(
    NOTEBOOK_04_TO_05_HANDOFF,
    NOTEBOOK_04_TO_05_HANDOFF_PAYLOAD_SHA256,
    NOTEBOOK_04_TO_05_HANDOFF_RAW_SHA256,
) = verify_self_hashed_json(
    NOTEBOOK_04_TO_05_HANDOFF_PATH,
    hash_field="handoff_payload_sha256",
    expected_payload_sha256=(
        EXPECTED_NOTEBOOK_04_TO_05_HANDOFF_PAYLOAD_SHA256
    ),
)

(
    NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT,
    NOTEBOOK_04_FINAL_ACCEPTANCE_PAYLOAD_SHA256,
    NOTEBOOK_04_FINAL_ACCEPTANCE_RAW_SHA256,
) = verify_self_hashed_json(
    NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT_PATH,
    hash_field="final_acceptance_report_payload_sha256",
    expected_payload_sha256=(
        EXPECTED_NOTEBOOK_04_FINAL_ACCEPTANCE_PAYLOAD_SHA256
    ),
)

require_fields(
    NOTEBOOK_04_OUTPUT_MANIFEST,
    {
        "artifact_type": "NOTEBOOK_04_OUTPUT_MANIFEST",
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "v0_1_run_id": V01_RUN_ID,
        "primary_event_representation": (
            PRIMARY_EVENT_REPRESENTATION
        ),
        "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
        "acceptance_status": "PASS_WITH_EVENT_STREAM_WARNINGS",
    },
    artifact_label="Notebook 04 output manifest",
)

require_fields(
    NOTEBOOK_04_TO_05_HANDOFF,
    {
        "artifact_type": "NOTEBOOK_04_TO_NOTEBOOK_05_HANDOFF",
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "v0_1_run_id": V01_RUN_ID,
        "event_stream_authorized": True,
        "feature_construction_authorized": True,
        "hawkes_estimation_authorized": False,
        "market_making_authorized": False,
        "primary_event_representation": (
            PRIMARY_EVENT_REPRESENTATION
        ),
        "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
        "primary_event_rows": EXPECTED_PRIMARY_EVENT_ROWS,
        "primary_batch_rows": EXPECTED_PRIMARY_SCORING_BATCHES,
    },
    artifact_label="Notebook 04 to Notebook 05 handoff",
)

require_fields(
    NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT,
    {
        "artifact_type": "NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT",
        "source_run_prefix": SOURCE_RUN_PREFIX,
        "v0_1_run_id": V01_RUN_ID,
        "primary_event_representation": (
            PRIMARY_EVENT_REPRESENTATION
        ),
        "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
        "primary_event_rows": EXPECTED_PRIMARY_EVENT_ROWS,
        "primary_batch_rows": EXPECTED_PRIMARY_SCORING_BATCHES,
        "blocking_gate_failures": 0,
    },
    artifact_label="Notebook 04 final acceptance report",
)

notebook_04_warning_findings = (
    NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT.get("warning_findings")
)

require(
    isinstance(notebook_04_warning_findings, dict),
    "Notebook 04 final acceptance report lacks warning_findings.",
)
require(
    notebook_04_warning_findings.get("simultaneous_batch_count")
    == EXPECTED_SIMULTANEOUS_BATCHES,
    "Notebook 04 simultaneous-batch count differs from the frozen contract.",
)
require(
    notebook_04_warning_findings.get("mixed_side_batch_count")
    == EXPECTED_MIXED_SIDE_BATCHES,
    "Notebook 04 mixed-side batch count differs from the frozen contract.",
)

for (
    artifact_label,
    artifact_type,
    artifact_path,
    expected_hash,
    observed_hash,
    raw_hash,
) in (
    (
        "notebook_04_output_manifest",
        "NOTEBOOK_04_OUTPUT_MANIFEST",
        NOTEBOOK_04_OUTPUT_MANIFEST_PATH,
        EXPECTED_NOTEBOOK_04_MANIFEST_PAYLOAD_SHA256,
        NOTEBOOK_04_OUTPUT_MANIFEST_PAYLOAD_SHA256,
        NOTEBOOK_04_OUTPUT_MANIFEST_RAW_SHA256,
    ),
    (
        "notebook_04_to_notebook_05_handoff",
        "NOTEBOOK_04_TO_NOTEBOOK_05_HANDOFF",
        NOTEBOOK_04_TO_05_HANDOFF_PATH,
        EXPECTED_NOTEBOOK_04_TO_05_HANDOFF_PAYLOAD_SHA256,
        NOTEBOOK_04_TO_05_HANDOFF_PAYLOAD_SHA256,
        NOTEBOOK_04_TO_05_HANDOFF_RAW_SHA256,
    ),
    (
        "notebook_04_final_acceptance_report",
        "NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT",
        NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT_PATH,
        EXPECTED_NOTEBOOK_04_FINAL_ACCEPTANCE_PAYLOAD_SHA256,
        NOTEBOOK_04_FINAL_ACCEPTANCE_PAYLOAD_SHA256,
        NOTEBOOK_04_FINAL_ACCEPTANCE_RAW_SHA256,
    ),
):
    control_audit_rows.append(
        ControlArtifactAuditRow(
            artifact_label=artifact_label,
            artifact_type=artifact_type,
            resolved_path=str(
                artifact_path.resolve(strict=False)
            ),
            hash_mode="SELF_HASHED_PAYLOAD",
            expected_sha256=expected_hash,
            observed_sha256=observed_hash,
            raw_file_sha256=raw_hash,
            inside_v01=True,
            outside_v00=True,
            identity_verified=True,
            authority_verified=True,
            verified=True,
        )
    )


# ------------------------------------------------------------
# Notebook 05 flat control documents
# ------------------------------------------------------------

(
    NOTEBOOK_05_OUTPUT_MANIFEST,
    NOTEBOOK_05_OUTPUT_MANIFEST_PAYLOAD_SHA256,
    NOTEBOOK_05_OUTPUT_MANIFEST_RAW_SHA256,
) = verify_self_hashed_json(
    NOTEBOOK_05_OUTPUT_MANIFEST_PATH,
    hash_field="manifest_payload_sha256",
    expected_payload_sha256=(
        EXPECTED_NOTEBOOK_05_MANIFEST_PAYLOAD_SHA256
    ),
)

(
    NOTEBOOK_05_TO_06_HANDOFF,
    NOTEBOOK_05_TO_06_HANDOFF_PAYLOAD_SHA256,
    NOTEBOOK_05_TO_06_HANDOFF_RAW_SHA256,
) = verify_self_hashed_json(
    NOTEBOOK_05_TO_06_HANDOFF_PATH,
    hash_field="handoff_payload_sha256",
    expected_payload_sha256=(
        EXPECTED_NOTEBOOK_05_TO_06_HANDOFF_PAYLOAD_SHA256
    ),
)

(
    NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT,
    NOTEBOOK_05_FINAL_ACCEPTANCE_PAYLOAD_SHA256,
    NOTEBOOK_05_FINAL_ACCEPTANCE_RAW_SHA256,
) = verify_self_hashed_json(
    NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT_PATH,
    hash_field="final_acceptance_payload_sha256",
    expected_payload_sha256=(
        EXPECTED_NOTEBOOK_05_FINAL_ACCEPTANCE_PAYLOAD_SHA256
    ),
)

common_notebook_05_identity = {
    "source_run_prefix": SOURCE_RUN_PREFIX,
    "source_set_sha256": SOURCE_SET_SHA256,
    "v0_1_run_id": V01_RUN_ID,
    "run_config_sha256": RUN_CONFIG_SHA256,
    "run_identity_sha256": RUN_IDENTITY_SHA256,
    "combined_output_prefix": COMBINED_OUTPUT_PREFIX,
}

require_fields(
    NOTEBOOK_05_OUTPUT_MANIFEST,
    {
        **common_notebook_05_identity,
        "artifact_type": "NOTEBOOK_05_OUTPUT_MANIFEST",
        "manifest_type": "NOTEBOOK_OUTPUT_MANIFEST",
        "producing_notebook": "05_MARKET_STATE_FEATURES",
        "next_notebook": "06_POINT_PROCESS_BASELINES",
        "acceptance_status": "PASS_WITH_FEATURE_WARNINGS",
        "notebook_06_authorized": True,
        "observational_unit": "PRIMARY_SCORING_BATCH",
        "primary_event_representation": (
            PRIMARY_EVENT_REPRESENTATION
        ),
        "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
    },
    artifact_label="Notebook 05 output manifest",
)

require_fields(
    NOTEBOOK_05_TO_06_HANDOFF,
    {
        **common_notebook_05_identity,
        "artifact_type": "NOTEBOOK_05_TO_NOTEBOOK_06_HANDOFF",
        "producing_notebook": "05_MARKET_STATE_FEATURES",
        "next_notebook": "06_POINT_PROCESS_BASELINES",
        "acceptance_status": "PASS_WITH_FEATURE_WARNINGS",
        "point_process_baselines_authorized": True,
        "hawkes_estimation_authorized": False,
        "market_making_authorized": False,
        "quote_policy_authorized": False,
        "fill_simulation_authorized": False,
        "pnl_backtest_authorized": False,
        "observational_unit": "PRIMARY_SCORING_BATCH",
        "primary_event_representation": (
            PRIMARY_EVENT_REPRESENTATION
        ),
        "timestamp_interface": PRIMARY_TIMESTAMP_INTERFACE,
        "event_partition_field": (
            PRIMARY_EVENT_PARTITION_COLUMN
        ),
        "primary_time_field": PRIMARY_EVENT_TIME_COLUMN,
    },
    artifact_label="Notebook 05 to Notebook 06 handoff",
)

require_fields(
    NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT,
    {
        **common_notebook_05_identity,
        "artifact_type": "NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT",
        "producing_notebook": "05_MARKET_STATE_FEATURES",
        "next_notebook": "06_POINT_PROCESS_BASELINES",
        "acceptance_status": "PASS_WITH_FEATURE_WARNINGS",
        "blocking_failure_count": 0,
        "notebook_06_authorized": True,
    },
    artifact_label="Notebook 05 final acceptance report",
)


# ------------------------------------------------------------
# Notebook 05 causal and physical-separation contract
# ------------------------------------------------------------

notebook_05_causal_contract = NOTEBOOK_05_TO_06_HANDOFF.get(
    "causal_contract"
)

require(
    isinstance(notebook_05_causal_contract, dict),
    "Notebook 05 handoff lacks causal_contract.",
)

require_fields(
    notebook_05_causal_contract,
    {
        "features_are_tminus": True,
        "current_batch_raw_fields_attached": False,
        "tplus_fields_attached": False,
        "future_labels_attached_to_feature_table": False,
        "labels_physically_separate": True,
        "timestamp_jitter_applied": False,
        "events_removed": False,
        "causally_explained_feature_nan_policy": (
            "PRESERVE_WITHOUT_IMPUTATION"
        ),
    },
    artifact_label="Notebook 05 causal contract",
)

notebook_05_scope = NOTEBOOK_05_TO_06_HANDOFF.get(
    "notebook_06_scope"
)

require(
    isinstance(notebook_05_scope, dict),
    "Notebook 05 handoff lacks notebook_06_scope.",
)
require(
    notebook_05_scope.get("authorized") is True,
    "Notebook 06 is not authorized by the Notebook 05 scope.",
)

authorized_work = set(
    notebook_05_scope.get("authorized_work", [])
)
not_authorized_work = set(
    notebook_05_scope.get("not_authorized_work", [])
)

require(
    {
        "point_process_baselines",
        "event_arrival_baselines",
        "simple_control_models",
        "diagnostics_for_incremental_information_before_hawkes",
    }.issubset(authorized_work),
    "Notebook 05 handoff does not authorize all required Notebook 06 work.",
)

require(
    {
        "hawkes_estimation",
        "hawkes_superiority_claim",
        "market_making_strategy",
        "quote_policy",
        "fill_simulation",
        "pnl_backtest",
    }.issubset(not_authorized_work),
    "Notebook 05 handoff does not preserve all downstream prohibitions.",
)


# ------------------------------------------------------------
# Notebook 05 row and column contract
# ------------------------------------------------------------

notebook_05_row_counts = NOTEBOOK_05_OUTPUT_MANIFEST.get(
    "row_counts"
)
notebook_05_column_counts = NOTEBOOK_05_OUTPUT_MANIFEST.get(
    "column_counts"
)

require(
    isinstance(notebook_05_row_counts, dict),
    "Notebook 05 output manifest lacks row_counts.",
)
require(
    isinstance(notebook_05_column_counts, dict),
    "Notebook 05 output manifest lacks column_counts.",
)

require(
    notebook_05_row_counts.get(
        "primary_batch_market_state_features"
    )
    == EXPECTED_PRIMARY_SCORING_BATCHES,
    "Notebook 05 feature-row count differs from the scoring-batch contract.",
)
require(
    notebook_05_row_counts.get(
        "primary_batch_future_labels"
    )
    == EXPECTED_PRIMARY_SCORING_BATCHES,
    "Notebook 05 future-label row count differs from the scoring-batch contract.",
)
require(
    notebook_05_column_counts.get("feature_columns") == 571,
    "Notebook 05 feature count differs from the authority map.",
)
require(
    notebook_05_column_counts.get("label_columns") == 140,
    "Notebook 05 future-label count differs from the authority map.",
)


# ------------------------------------------------------------
# Cross-document lineage verification
# ------------------------------------------------------------

notebook_05_handoff_manifest_reference = (
    NOTEBOOK_05_TO_06_HANDOFF.get("output_manifest")
)

require(
    isinstance(notebook_05_handoff_manifest_reference, dict),
    "Notebook 05 handoff lacks output_manifest lineage.",
)

require_fields(
    notebook_05_handoff_manifest_reference,
    {
        "resolved_path": str(
            NOTEBOOK_05_OUTPUT_MANIFEST_PATH
        ),
        "manifest_payload_sha256": (
            NOTEBOOK_05_OUTPUT_MANIFEST_PAYLOAD_SHA256
        ),
        "raw_file_sha256": (
            NOTEBOOK_05_OUTPUT_MANIFEST_RAW_SHA256
        ),
    },
    artifact_label="Notebook 05 handoff manifest lineage",
)

notebook_05_final_manifest_reference = (
    NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT.get("manifest")
)
notebook_05_final_handoff_reference = (
    NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT.get("handoff")
)

require(
    isinstance(notebook_05_final_manifest_reference, dict),
    "Notebook 05 final acceptance report lacks manifest lineage.",
)
require(
    isinstance(notebook_05_final_handoff_reference, dict),
    "Notebook 05 final acceptance report lacks handoff lineage.",
)

require_fields(
    notebook_05_final_manifest_reference,
    {
        "resolved_path": str(
            NOTEBOOK_05_OUTPUT_MANIFEST_PATH
        ),
        "manifest_payload_sha256": (
            NOTEBOOK_05_OUTPUT_MANIFEST_PAYLOAD_SHA256
        ),
        "raw_file_sha256": (
            NOTEBOOK_05_OUTPUT_MANIFEST_RAW_SHA256
        ),
    },
    artifact_label="Notebook 05 final manifest lineage",
)

require_fields(
    notebook_05_final_handoff_reference,
    {
        "resolved_path": str(
            NOTEBOOK_05_TO_06_HANDOFF_PATH
        ),
        "handoff_payload_sha256": (
            NOTEBOOK_05_TO_06_HANDOFF_PAYLOAD_SHA256
        ),
        "raw_file_sha256": (
            NOTEBOOK_05_TO_06_HANDOFF_RAW_SHA256
        ),
    },
    artifact_label="Notebook 05 final handoff lineage",
)


for (
    artifact_label,
    artifact_type,
    artifact_path,
    expected_hash,
    observed_hash,
    raw_hash,
) in (
    (
        "notebook_05_output_manifest",
        "NOTEBOOK_05_OUTPUT_MANIFEST",
        NOTEBOOK_05_OUTPUT_MANIFEST_PATH,
        EXPECTED_NOTEBOOK_05_MANIFEST_PAYLOAD_SHA256,
        NOTEBOOK_05_OUTPUT_MANIFEST_PAYLOAD_SHA256,
        NOTEBOOK_05_OUTPUT_MANIFEST_RAW_SHA256,
    ),
    (
        "notebook_05_to_notebook_06_handoff",
        "NOTEBOOK_05_TO_NOTEBOOK_06_HANDOFF",
        NOTEBOOK_05_TO_06_HANDOFF_PATH,
        EXPECTED_NOTEBOOK_05_TO_06_HANDOFF_PAYLOAD_SHA256,
        NOTEBOOK_05_TO_06_HANDOFF_PAYLOAD_SHA256,
        NOTEBOOK_05_TO_06_HANDOFF_RAW_SHA256,
    ),
    (
        "notebook_05_final_acceptance_report",
        "NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT",
        NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT_PATH,
        EXPECTED_NOTEBOOK_05_FINAL_ACCEPTANCE_PAYLOAD_SHA256,
        NOTEBOOK_05_FINAL_ACCEPTANCE_PAYLOAD_SHA256,
        NOTEBOOK_05_FINAL_ACCEPTANCE_RAW_SHA256,
    ),
):
    control_audit_rows.append(
        ControlArtifactAuditRow(
            artifact_label=artifact_label,
            artifact_type=artifact_type,
            resolved_path=str(
                artifact_path.resolve(strict=False)
            ),
            hash_mode="SELF_HASHED_PAYLOAD",
            expected_sha256=expected_hash,
            observed_sha256=observed_hash,
            raw_file_sha256=raw_hash,
            inside_v01=True,
            outside_v00=True,
            identity_verified=True,
            authority_verified=True,
            verified=True,
        )
    )


# ------------------------------------------------------------
# Final control-artifact audit
# ------------------------------------------------------------

UPSTREAM_CONTROL_ARTIFACT_AUDIT = pd.DataFrame(
    [asdict(row) for row in control_audit_rows]
).sort_values(
    ["artifact_label"],
    kind="stable",
).reset_index(drop=True)

require(
    len(UPSTREAM_CONTROL_ARTIFACT_AUDIT) == 7,
    (
        "Unexpected number of verified upstream control artifacts: "
        f"{len(UPSTREAM_CONTROL_ARTIFACT_AUDIT)}"
    ),
)
require(
    bool(UPSTREAM_CONTROL_ARTIFACT_AUDIT["verified"].all()),
    "At least one upstream control artifact failed verification.",
)

PROTECTED_PARTITION_CONTENT_LOADED = {
    "DEVELOPMENT": False,
    "CALIBRATION": False,
    "VALIDATION": False,
    "ENGINEERING_HOLDOUT": False,
}

NOTEBOOK_06_BASELINE_WORK_AUTHORIZED = True
NOTEBOOK_06_HAWKES_ESTIMATION_AUTHORIZED = False

display(
    UPSTREAM_CONTROL_ARTIFACT_AUDIT[
        [
            "artifact_label",
            "artifact_type",
            "hash_mode",
            "identity_verified",
            "authority_verified",
            "verified",
        ]
    ]
)

print(
    "Upstream control chain verified: Notebook 00 → Notebook 04 "
    "→ Notebook 05 → Notebook 06."
)
print(
    "Point-process baseline work is authorized. "
    "Hawkes estimation remains unauthorized."
)
print(
    "No event tables, feature tables, future-label tables, "
    "or protected partition observations have been loaded."
)

,artifact_label,artifact_type,hash_mode,identity_verified,authority_verified,verified
0,notebook_00_output_manifest,NOTEBOOK_00_OUTPUT_MANIFEST,RAW_FILE_AND_WRAPPED_PAYLOAD,True,True,True
1,notebook_04_final_acceptance_report,NOTEBOOK_04_FINAL_ACCEPTANCE_REPORT,SELF_HASHED_PAYLOAD,True,True,True
2,notebook_04_output_manifest,NOTEBOOK_04_OUTPUT_MANIFEST,SELF_HASHED_PAYLOAD,True,True,True
3,notebook_04_to_notebook_05_handoff,NOTEBOOK_04_TO_NOTEBOOK_05_HANDOFF,SELF_HASHED_PAYLOAD,True,True,True
4,notebook_05_final_acceptance_report,NOTEBOOK_05_FINAL_ACCEPTANCE_REPORT,SELF_HASHED_PAYLOAD,True,True,True
5,notebook_05_output_manifest,NOTEBOOK_05_OUTPUT_MANIFEST,SELF_HASHED_PAYLOAD,True,True,True
6,notebook_05_to_notebook_06_handoff,NOTEBOOK_05_TO_NOTEBOOK_06_HANDOFF,SELF_HASHED_PAYLOAD,True,True,True


Upstream control chain verified: Notebook 00 → Notebook 04 → Notebook 05 → Notebook 06.
Point-process baseline work is authorized. Hawkes estimation remains unauthorized.
No event tables, feature tables, future-label tables, or protected partition observations have been loaded.


In [3]:
# ============================================================
# Authoritative analytical-input resolution and protected load
# ============================================================

import pyarrow as pa
import pyarrow.parquet as pq


# ------------------------------------------------------------
# Partition-access contract
# ------------------------------------------------------------

ANALYTICAL_PARTITIONS: Final[tuple[str, ...]] = (
    "DEVELOPMENT",
    "CALIBRATION",
)

PROTECTED_PARTITIONS: Final[tuple[str, ...]] = (
    "VALIDATION",
    "ENGINEERING_HOLDOUT",
)

ALL_PARTITIONS: Final[tuple[str, ...]] = (
    *ANALYTICAL_PARTITIONS,
    *PROTECTED_PARTITIONS,
)

FUTURE_LABEL_TABLE_LOADED = False


# ------------------------------------------------------------
# Artifact-record resolution
# ------------------------------------------------------------

def require_mapping(
    value: Any,
    *,
    label: str,
) -> dict[str, Any]:
    """Require a dictionary-like artifact record."""
    require(
        isinstance(value, Mapping),
        f"{label} must be a mapping; observed {type(value).__name__}.",
    )
    return dict(value)


def artifact_record_path(
    record: Mapping[str, Any],
    *,
    label: str,
) -> Path:
    """Resolve the persisted path from an upstream artifact record."""
    raw_path = (
        record.get("resolved_path")
        or record.get("path")
    )

    require(
        isinstance(raw_path, str) and raw_path.strip(),
        f"{label} does not contain a usable path.",
    )

    path = Path(raw_path)

    require(
        path.is_file(),
        f"{label} does not exist: {path}",
    )
    require(
        path_is_within(path, V01_ROOT),
        f"{label} is outside the V0.1 authority tree: {path}",
    )
    require(
        not path_is_within(path, V00_ROOT),
        f"{label} improperly points into the immutable V0.0 tree: {path}",
    )

    return path


def parquet_contract_path(
    parquet_path: Path,
    record: Mapping[str, Any],
) -> Path:
    """Resolve a parquet sidecar contract path."""
    registered_contract = record.get("contract_path")

    if isinstance(registered_contract, str) and registered_contract.strip():
        return Path(registered_contract)

    return parquet_path.with_name(
        f"{parquet_path.stem}__contract.json"
    )


@dataclass(frozen=True, slots=True)
class VerifiedParquetArtifact:
    label: str
    artifact_type: str
    parquet_path: Path
    contract_path: Path
    row_count: int
    column_count: int
    columns: tuple[str, ...]
    table_payload_sha256: str
    raw_file_sha256: str
    contract_payload_sha256: str


def verify_parquet_artifact(
    *,
    label: str,
    record: Mapping[str, Any],
    expected_artifact_type: str | None = None,
) -> VerifiedParquetArtifact:
    """
    Verify a parquet artifact without reading its data rows.

    Verification covers:
    - V0.1 path authority;
    - persisted raw-file hash;
    - self-hashed sidecar contract;
    - handoff/manifest payload-hash consistency;
    - schema and row-count metadata.
    """
    normalized_record = require_mapping(
        record,
        label=f"{label} artifact record",
    )

    parquet_path = artifact_record_path(
        normalized_record,
        label=label,
    )
    contract_path = parquet_contract_path(
        parquet_path,
        normalized_record,
    )

    require(
        contract_path.is_file(),
        f"{label} sidecar contract is missing: {contract_path}",
    )
    require(
        path_is_within(contract_path, V01_ROOT),
        f"{label} contract is outside V0.1: {contract_path}",
    )
    require(
        not path_is_within(contract_path, V00_ROOT),
        f"{label} contract points into V0.0: {contract_path}",
    )

    contract = read_json_object(contract_path)

    embedded_contract_hash = contract.get(
        "contract_payload_sha256"
    )
    require(
        isinstance(embedded_contract_hash, str),
        (
            f"{label} contract lacks "
            "'contract_payload_sha256'."
        ),
    )

    contract_without_hash = {
        key: value
        for key, value in contract.items()
        if key != "contract_payload_sha256"
    }
    observed_contract_hash = canonical_json_sha256(
        contract_without_hash
    )

    require(
        observed_contract_hash == embedded_contract_hash,
        (
            f"{label} contract self-hash mismatch: "
            f"embedded={embedded_contract_hash}; "
            f"observed={observed_contract_hash}"
        ),
    )

    registered_contract_hash = normalized_record.get(
        "contract_payload_sha256"
    )
    if registered_contract_hash is not None:
        require(
            registered_contract_hash == observed_contract_hash,
            (
                f"{label} record-to-contract hash mismatch: "
                f"record={registered_contract_hash}; "
                f"contract={observed_contract_hash}"
            ),
        )

    observed_raw_file_hash = sha256_file(parquet_path)

    contract_raw_file_hash = (
        contract.get("raw_file_sha256")
        or contract.get("file_sha256")
    )
    require(
        isinstance(contract_raw_file_hash, str),
        f"{label} contract lacks a parquet file hash.",
    )
    require(
        observed_raw_file_hash == contract_raw_file_hash,
        (
            f"{label} raw parquet hash mismatch: "
            f"contract={contract_raw_file_hash}; "
            f"observed={observed_raw_file_hash}"
        ),
    )

    registered_raw_file_hash = (
        normalized_record.get("raw_file_sha256")
        or normalized_record.get("file_sha256")
    )
    if registered_raw_file_hash is not None:
        require(
            registered_raw_file_hash == observed_raw_file_hash,
            (
                f"{label} record-to-file hash mismatch: "
                f"record={registered_raw_file_hash}; "
                f"observed={observed_raw_file_hash}"
            ),
        )

    contract_payload_hash = contract.get(
        "table_payload_sha256"
    )
    registered_payload_hash = normalized_record.get(
        "table_payload_sha256"
    )

    require(
        isinstance(contract_payload_hash, str),
        f"{label} contract lacks table_payload_sha256.",
    )

    if registered_payload_hash is not None:
        require(
            registered_payload_hash == contract_payload_hash,
            (
                f"{label} table-payload hash mismatch: "
                f"record={registered_payload_hash}; "
                f"contract={contract_payload_hash}"
            ),
        )

    parquet_metadata = pq.ParquetFile(parquet_path)
    parquet_schema_columns = tuple(
        parquet_metadata.schema_arrow.names
    )

    contract_columns = contract.get("columns")
    require(
        isinstance(contract_columns, list),
        f"{label} contract lacks its ordered column list.",
    )
    contract_columns_tuple = tuple(
        str(column)
        for column in contract_columns
    )

    require(
        parquet_schema_columns == contract_columns_tuple,
        (
            f"{label} parquet-schema mismatch.\n"
            f"Parquet: {parquet_schema_columns}\n"
            f"Contract: {contract_columns_tuple}"
        ),
    )

    metadata_row_count = int(
        parquet_metadata.metadata.num_rows
    )
    contract_row_count = int(contract["row_count"])
    contract_column_count = int(contract["column_count"])

    require(
        metadata_row_count == contract_row_count,
        (
            f"{label} metadata row-count mismatch: "
            f"parquet={metadata_row_count}; "
            f"contract={contract_row_count}"
        ),
    )
    require(
        len(parquet_schema_columns) == contract_column_count,
        (
            f"{label} metadata column-count mismatch: "
            f"parquet={len(parquet_schema_columns)}; "
            f"contract={contract_column_count}"
        ),
    )

    registered_row_count = normalized_record.get("row_count")
    registered_column_count = normalized_record.get(
        "column_count"
    )

    if registered_row_count is not None:
        require(
            int(registered_row_count) == contract_row_count,
            (
                f"{label} record row-count mismatch: "
                f"record={registered_row_count}; "
                f"contract={contract_row_count}"
            ),
        )

    if registered_column_count is not None:
        require(
            int(registered_column_count)
            == contract_column_count,
            (
                f"{label} record column-count mismatch: "
                f"record={registered_column_count}; "
                f"contract={contract_column_count}"
            ),
        )

    contract_artifact_type = str(
        contract.get("artifact_type", "")
    )
    record_artifact_type = str(
        normalized_record.get("artifact_type", "")
    )

    if expected_artifact_type is not None:
        valid_artifact_types = {
            expected_artifact_type,
            f"{expected_artifact_type}_CONTRACT",
        }

        require(
            (
                contract_artifact_type in valid_artifact_types
                or record_artifact_type
                in valid_artifact_types
            ),
            (
                f"{label} artifact-type mismatch: "
                f"expected one of {sorted(valid_artifact_types)}; "
                f"record={record_artifact_type!r}; "
                f"contract={contract_artifact_type!r}"
            ),
        )

    return VerifiedParquetArtifact(
        label=label,
        artifact_type=(
            record_artifact_type
            or contract_artifact_type
        ),
        parquet_path=parquet_path,
        contract_path=contract_path,
        row_count=contract_row_count,
        column_count=contract_column_count,
        columns=contract_columns_tuple,
        table_payload_sha256=contract_payload_hash,
        raw_file_sha256=observed_raw_file_hash,
        contract_payload_sha256=observed_contract_hash,
    )


# ------------------------------------------------------------
# Resolve authoritative Notebook 04 records
# ------------------------------------------------------------

NOTEBOOK_04_PRIMARY_EVENTS_RECORD = require_mapping(
    NOTEBOOK_04_TO_05_HANDOFF.get(
        "primary_estimation_events"
    ),
    label="Notebook 04 primary-estimation-events record",
)

NOTEBOOK_04_EXACT_TIME_BATCHES_RECORD = require_mapping(
    NOTEBOOK_04_TO_05_HANDOFF.get(
        "primary_exact_time_batches"
    ),
    label="Notebook 04 exact-time-batches record",
)

NOTEBOOK_04_SCORING_BATCHES_RECORD = require_mapping(
    NOTEBOOK_04_TO_05_HANDOFF.get(
        "primary_scoring_batches"
    ),
    label="Notebook 04 scoring-batches record",
)

NOTEBOOK_04_EVENT_TO_BATCH_RECORD = require_mapping(
    NOTEBOOK_04_TO_05_HANDOFF.get(
        "primary_event_to_batch_membership"
    ),
    label="Notebook 04 event-to-batch-membership record",
)

NOTEBOOK_04_OBSERVATION_WINDOWS_RECORD = require_mapping(
    NOTEBOOK_04_TO_05_HANDOFF.get(
        "observation_window_contract"
    ),
    label="Notebook 04 observation-window-contract record",
)

NOTEBOOK_05_FEATURE_TABLE_RECORD = require_mapping(
    NOTEBOOK_05_TO_06_HANDOFF.get("feature_table"),
    label="Notebook 05 feature-table record",
)


# ------------------------------------------------------------
# Verify parquet files and sidecar contracts before row access
# ------------------------------------------------------------

VERIFIED_PRIMARY_EVENTS = verify_parquet_artifact(
    label="primary_estimation_events",
    record=NOTEBOOK_04_PRIMARY_EVENTS_RECORD,
    expected_artifact_type=(
        "NOTEBOOK_04_PRIMARY_ESTIMATION_EVENTS"
    ),
)

VERIFIED_EXACT_TIME_BATCHES = verify_parquet_artifact(
    label="primary_exact_time_batches",
    record=NOTEBOOK_04_EXACT_TIME_BATCHES_RECORD,
    expected_artifact_type=(
        "NOTEBOOK_04_PRIMARY_EXACT_TIME_BATCHES"
    ),
)

VERIFIED_SCORING_BATCHES = verify_parquet_artifact(
    label="primary_scoring_batches",
    record=NOTEBOOK_04_SCORING_BATCHES_RECORD,
    expected_artifact_type=(
        "NOTEBOOK_04_PRIMARY_SCORING_BATCHES"
    ),
)

VERIFIED_EVENT_TO_BATCH_MEMBERSHIP = (
    verify_parquet_artifact(
        label="primary_event_to_batch_membership",
        record=NOTEBOOK_04_EVENT_TO_BATCH_RECORD,
        expected_artifact_type=(
            "NOTEBOOK_04_PRIMARY_EVENT_TO_BATCH_MEMBERSHIP"
        ),
    )
)

VERIFIED_OBSERVATION_WINDOWS = verify_parquet_artifact(
    label="observation_window_contract",
    record=NOTEBOOK_04_OBSERVATION_WINDOWS_RECORD,
    expected_artifact_type=(
        "NOTEBOOK_04_OBSERVATION_WINDOW_CONTRACT"
    ),
)

VERIFIED_FEATURE_TABLE = verify_parquet_artifact(
    label="primary_batch_market_state_features",
    record=NOTEBOOK_05_FEATURE_TABLE_RECORD,
    expected_artifact_type=(
        "PRIMARY_BATCH_MARKET_STATE_FEATURES"
    ),
)


# ------------------------------------------------------------
# Required schemas
# ------------------------------------------------------------

EVENT_REQUIRED_COLUMNS: Final[frozenset[str]] = frozenset(
    {
        "primary_event_id",
        "primary_event_number",
        "partition_event_index",
        "primary_event_batch_id",
        "primary_event_batch_number",
        "event_partition_order",
        "event_partition",
        "event_time_ns",
        "relative_event_time_ns",
        "relative_event_time_seconds",
        "event_side",
        "event_side_code",
        "event_print_count",
        "event_quantity",
        "event_notional",
    }
)

BATCH_REQUIRED_COLUMNS: Final[frozenset[str]] = frozenset(
    {
        "primary_event_batch_id",
        "primary_event_batch_number",
        "partition_batch_index",
        "event_partition_order",
        "event_partition",
        "event_time_ns",
        "relative_batch_time_ns",
        "relative_batch_time_seconds",
        "batch_event_count",
        "buy_event_count",
        "sell_event_count",
        "unique_side_count",
        "mixed_side_batch_flag",
        "simultaneous_batch_required_flag",
    }
)

SCORING_CONTRACT_COLUMNS: Final[frozenset[str]] = frozenset(
    {
        "score_with_history_strictly_before_batch_time",
        "zero_lag_within_batch_excitation_allowed",
        "apply_batch_excitation_after_all_members_scored",
        "collector_sequence_is_trace_order_only",
        "timestamp_jitter_allowed",
        "event_removal_allowed",
    }
)

MEMBERSHIP_REQUIRED_COLUMNS: Final[frozenset[str]] = frozenset(
    {
        "primary_event_id",
        "primary_event_batch_id",
        "event_partition",
        "event_time_ns",
    }
)

OBSERVATION_WINDOW_REQUIRED_COLUMNS: Final[
    frozenset[str]
] = frozenset(
    {
        "partition_order",
        "event_partition",
        "contract_sequence_start",
        "contract_sequence_end_exclusive",
        "contract_start_ns",
        "contract_end_exclusive_ns",
        "contract_duration_ns",
        "event_clock_origin_ns",
        "first_event_time_ns",
        "last_event_time_ns",
        "left_censoring_duration_ns",
        "observation_window_duration_ns",
        "primary_event_count",
        "primary_batch_count",
        "simultaneous_batch_count",
        "mixed_side_batch_count",
    }
)


def require_schema_columns(
    artifact: VerifiedParquetArtifact,
    required_columns: Iterable[str],
) -> None:
    """Require fields to exist before any parquet row scan begins."""
    missing_columns = sorted(
        set(required_columns)
        - set(artifact.columns)
    )

    require(
        not missing_columns,
        (
            f"{artifact.label} is missing required columns: "
            f"{missing_columns}"
        ),
    )


require_schema_columns(
    VERIFIED_PRIMARY_EVENTS,
    EVENT_REQUIRED_COLUMNS,
)
require_schema_columns(
    VERIFIED_EXACT_TIME_BATCHES,
    BATCH_REQUIRED_COLUMNS,
)
require_schema_columns(
    VERIFIED_SCORING_BATCHES,
    BATCH_REQUIRED_COLUMNS
    | SCORING_CONTRACT_COLUMNS,
)
require_schema_columns(
    VERIFIED_EVENT_TO_BATCH_MEMBERSHIP,
    MEMBERSHIP_REQUIRED_COLUMNS,
)
require_schema_columns(
    VERIFIED_OBSERVATION_WINDOWS,
    OBSERVATION_WINDOW_REQUIRED_COLUMNS,
)


# ------------------------------------------------------------
# Protected parquet loading
# ------------------------------------------------------------

def load_partition_subset(
    artifact: VerifiedParquetArtifact,
    *,
    partitions: Sequence[str],
    columns: Sequence[str] | None = None,
    partition_column: str = "event_partition",
) -> pd.DataFrame:
    """
    Load only explicitly authorized partitions through a parquet filter.

    Full-table read followed by an in-memory partition filter is forbidden.
    """
    authorized_partitions = tuple(
        str(partition)
        for partition in partitions
    )

    require(
        authorized_partitions,
        f"No partitions requested for {artifact.label}.",
    )
    require(
        set(authorized_partitions).issubset(
            set(ANALYTICAL_PARTITIONS)
        ),
        (
            f"Unauthorized partition requested for "
            f"{artifact.label}: {authorized_partitions}"
        ),
    )
    require(
        partition_column in artifact.columns,
        (
            f"{artifact.label} does not contain partition "
            f"column {partition_column!r}."
        ),
    )

    selected_columns: list[str] | None

    if columns is None:
        selected_columns = None
    else:
        selected_columns = list(
            dict.fromkeys(
                [
                    *columns,
                    partition_column,
                ]
            )
        )

        missing_columns = sorted(
            set(selected_columns)
            - set(artifact.columns)
        )
        require(
            not missing_columns,
            (
                f"{artifact.label} requested unavailable "
                f"columns: {missing_columns}"
            ),
        )

    frame = pd.read_parquet(
        artifact.parquet_path,
        engine="pyarrow",
        columns=selected_columns,
        filters=[
            (
                partition_column,
                "in",
                list(authorized_partitions),
            )
        ],
    )

    require(
        partition_column in frame.columns,
        (
            f"{artifact.label} filtered read lost "
            f"{partition_column!r}."
        ),
    )

    observed_partitions = set(
        frame[partition_column]
        .astype("string")
        .dropna()
        .tolist()
    )

    require(
        observed_partitions.issubset(
            set(authorized_partitions)
        ),
        (
            f"{artifact.label} loaded unauthorized partitions: "
            f"{sorted(observed_partitions - set(authorized_partitions))}"
        ),
    )
    require(
        not observed_partitions.intersection(
            PROTECTED_PARTITIONS
        ),
        (
            f"{artifact.label} loaded protected partition "
            f"content: "
            f"{sorted(observed_partitions.intersection(PROTECTED_PARTITIONS))}"
        ),
    )

    return frame.reset_index(drop=True)


def deterministic_sort(
    frame: pd.DataFrame,
    candidate_columns: Sequence[str],
) -> pd.DataFrame:
    """Sort by the available authority columns without mutating input."""
    sort_columns = [
        column
        for column in candidate_columns
        if column in frame.columns
    ]

    require(
        sort_columns,
        "No deterministic sort columns are available.",
    )

    return (
        frame.sort_values(
            sort_columns,
            kind="stable",
        )
        .reset_index(drop=True)
    )


# Observation-window rows contain partition metadata rather than
# event observations. Reading all four metadata rows is permitted.
OBSERVATION_WINDOW_CONTRACT = pd.read_parquet(
    VERIFIED_OBSERVATION_WINDOWS.parquet_path,
    engine="pyarrow",
).sort_values(
    "partition_order",
    kind="stable",
).reset_index(drop=True)

require(
    len(OBSERVATION_WINDOW_CONTRACT)
    == len(ALL_PARTITIONS),
    (
        "Observation-window contract must contain exactly "
        f"{len(ALL_PARTITIONS)} partition rows."
    ),
)
require(
    OBSERVATION_WINDOW_CONTRACT[
        "event_partition"
    ].is_unique,
    "Observation-window contract contains duplicate partitions.",
)
require(
    set(
        OBSERVATION_WINDOW_CONTRACT[
            "event_partition"
        ].astype("string")
    )
    == set(ALL_PARTITIONS),
    "Observation-window contract partition set is incorrect.",
)


# ------------------------------------------------------------
# Filtered analytical content loads
# ------------------------------------------------------------

PRIMARY_ESTIMATION_EVENTS_ANALYTICAL = load_partition_subset(
    VERIFIED_PRIMARY_EVENTS,
    partitions=ANALYTICAL_PARTITIONS,
)

PRIMARY_ESTIMATION_EVENTS_ANALYTICAL = deterministic_sort(
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL,
    (
        "event_partition_order",
        "partition_event_index",
        "primary_event_number",
        "primary_event_id",
    ),
)

PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL = (
    load_partition_subset(
        VERIFIED_EXACT_TIME_BATCHES,
        partitions=ANALYTICAL_PARTITIONS,
    )
)

PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL = deterministic_sort(
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL,
    (
        "event_partition_order",
        "partition_batch_index",
        "primary_event_batch_number",
        "primary_event_batch_id",
    ),
)

PRIMARY_SCORING_BATCHES_ANALYTICAL = (
    load_partition_subset(
        VERIFIED_SCORING_BATCHES,
        partitions=ANALYTICAL_PARTITIONS,
    )
)

PRIMARY_SCORING_BATCHES_ANALYTICAL = deterministic_sort(
    PRIMARY_SCORING_BATCHES_ANALYTICAL,
    (
        "event_partition_order",
        "partition_batch_index",
        "primary_event_batch_number",
        "primary_event_batch_id",
    ),
)

PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL = (
    load_partition_subset(
        VERIFIED_EVENT_TO_BATCH_MEMBERSHIP,
        partitions=ANALYTICAL_PARTITIONS,
    )
)

PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL = (
    deterministic_sort(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL,
        (
            "event_partition_order",
            "partition_event_index",
            "primary_event_number",
            "primary_event_id",
        ),
    )
)


# ------------------------------------------------------------
# Notebook 05 identity-only load
# ------------------------------------------------------------

FEATURE_IDENTITY_COLUMNS = tuple(
    str(column)
    for column in NOTEBOOK_05_TO_06_HANDOFF.get(
        "identity_columns",
        (),
    )
)

require(
    FEATURE_IDENTITY_COLUMNS,
    "Notebook 05 handoff contains no identity columns.",
)
require(
    "event_partition" in FEATURE_IDENTITY_COLUMNS,
    "Feature identity columns omit event_partition.",
)
require(
    "event_time_ns" in FEATURE_IDENTITY_COLUMNS,
    "Feature identity columns omit event_time_ns.",
)

require_schema_columns(
    VERIFIED_FEATURE_TABLE,
    FEATURE_IDENTITY_COLUMNS,
)

PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL = (
    load_partition_subset(
        VERIFIED_FEATURE_TABLE,
        partitions=ANALYTICAL_PARTITIONS,
        columns=FEATURE_IDENTITY_COLUMNS,
    )
)

PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL = (
    deterministic_sort(
        PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL,
        (
            "event_partition_order",
            "partition_batch_index",
            "primary_event_batch_number",
            "primary_event_batch_id",
        ),
    )
)


# ------------------------------------------------------------
# Row-count and partition conservation
# ------------------------------------------------------------

analytical_window_contract = (
    OBSERVATION_WINDOW_CONTRACT.loc[
        OBSERVATION_WINDOW_CONTRACT[
            "event_partition"
        ].isin(ANALYTICAL_PARTITIONS)
    ]
    .copy()
    .sort_values(
        "partition_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

expected_analytical_event_rows = int(
    analytical_window_contract[
        "primary_event_count"
    ].sum()
)
expected_analytical_batch_rows = int(
    analytical_window_contract[
        "primary_batch_count"
    ].sum()
)

require(
    len(PRIMARY_ESTIMATION_EVENTS_ANALYTICAL)
    == expected_analytical_event_rows,
    (
        "Analytical primary-event count mismatch: "
        f"expected={expected_analytical_event_rows}; "
        f"observed={len(PRIMARY_ESTIMATION_EVENTS_ANALYTICAL)}"
    ),
)
require(
    len(PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL)
    == expected_analytical_batch_rows,
    (
        "Analytical exact-time-batch count mismatch: "
        f"expected={expected_analytical_batch_rows}; "
        f"observed={len(PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL)}"
    ),
)
require(
    len(PRIMARY_SCORING_BATCHES_ANALYTICAL)
    == expected_analytical_batch_rows,
    (
        "Analytical scoring-batch count mismatch: "
        f"expected={expected_analytical_batch_rows}; "
        f"observed={len(PRIMARY_SCORING_BATCHES_ANALYTICAL)}"
    ),
)
require(
    len(PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL)
    == expected_analytical_event_rows,
    (
        "Analytical event-to-batch membership count mismatch: "
        f"expected={expected_analytical_event_rows}; "
        f"observed="
        f"{len(PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL)}"
    ),
)
require(
    len(PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL)
    == expected_analytical_batch_rows,
    (
        "Analytical Notebook 05 feature-identity count mismatch: "
        f"expected={expected_analytical_batch_rows}; "
        f"observed="
        f"{len(PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL)}"
    ),
)


# ------------------------------------------------------------
# Initial scoring-contract assertions
# ------------------------------------------------------------

require(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "score_with_history_strictly_before_batch_time"
    ].astype(bool).all(),
    "At least one scoring batch does not require strict pre-batch history.",
)
require(
    not PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "zero_lag_within_batch_excitation_allowed"
    ].astype(bool).any(),
    "At least one scoring batch permits zero-lag within-batch excitation.",
)
require(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "apply_batch_excitation_after_all_members_scored"
    ].astype(bool).all(),
    "At least one scoring batch violates post-score batch updating.",
)
require(
    not PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "timestamp_jitter_allowed"
    ].astype(bool).any(),
    "At least one scoring batch permits timestamp jitter.",
)
require(
    not PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "event_removal_allowed"
    ].astype(bool).any(),
    "At least one scoring batch permits event removal.",
)


# ------------------------------------------------------------
# Protected-partition access ledger
# ------------------------------------------------------------

PROTECTED_PARTITION_CONTENT_LOADED.update(
    {
        "DEVELOPMENT": True,
        "CALIBRATION": True,
        "VALIDATION": False,
        "ENGINEERING_HOLDOUT": False,
    }
)

PARTITION_ACCESS_LEDGER = pd.DataFrame(
    [
        {
            "partition": partition,
            "partition_role": (
                "ANALYTICAL"
                if partition in ANALYTICAL_PARTITIONS
                else "PROTECTED"
            ),
            "event_content_loaded": bool(
                PROTECTED_PARTITION_CONTENT_LOADED[
                    partition
                ]
            ),
            "authorized_in_notebook_06": (
                partition in ANALYTICAL_PARTITIONS
            ),
            "status": (
                "PASS"
                if (
                    (
                        partition in ANALYTICAL_PARTITIONS
                        and PROTECTED_PARTITION_CONTENT_LOADED[
                            partition
                        ]
                    )
                    or (
                        partition in PROTECTED_PARTITIONS
                        and not PROTECTED_PARTITION_CONTENT_LOADED[
                            partition
                        ]
                    )
                )
                else "FAIL"
            ),
        }
        for partition in ALL_PARTITIONS
    ]
)

require(
    PARTITION_ACCESS_LEDGER["status"].eq("PASS").all(),
    "Partition-access firewall failed.",
)
require(
    FUTURE_LABEL_TABLE_LOADED is False,
    "Future-label table must not be loaded in Notebook 06.",
)


# ------------------------------------------------------------
# Input artifact and load audit
# ------------------------------------------------------------

verified_artifacts = (
    VERIFIED_PRIMARY_EVENTS,
    VERIFIED_EXACT_TIME_BATCHES,
    VERIFIED_SCORING_BATCHES,
    VERIFIED_EVENT_TO_BATCH_MEMBERSHIP,
    VERIFIED_OBSERVATION_WINDOWS,
    VERIFIED_FEATURE_TABLE,
)

UPSTREAM_DATA_ARTIFACT_AUDIT = pd.DataFrame(
    [
        {
            "label": artifact.label,
            "artifact_type": artifact.artifact_type,
            "full_row_count": artifact.row_count,
            "full_column_count": artifact.column_count,
            "raw_file_sha256": artifact.raw_file_sha256,
            "table_payload_sha256": (
                artifact.table_payload_sha256
            ),
            "contract_payload_sha256": (
                artifact.contract_payload_sha256
            ),
            "resolved_path": str(artifact.parquet_path),
            "verified": True,
        }
        for artifact in verified_artifacts
    ]
)

ANALYTICAL_LOAD_AUDIT = pd.DataFrame(
    [
        {
            "label": "primary_estimation_events",
            "loaded_rows": len(
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL
            ),
            "loaded_columns": (
                PRIMARY_ESTIMATION_EVENTS_ANALYTICAL.shape[1]
            ),
            "loaded_partitions": ",".join(
                ANALYTICAL_PARTITIONS
            ),
            "protected_rows_loaded": False,
        },
        {
            "label": "primary_exact_time_batches",
            "loaded_rows": len(
                PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL
            ),
            "loaded_columns": (
                PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL.shape[1]
            ),
            "loaded_partitions": ",".join(
                ANALYTICAL_PARTITIONS
            ),
            "protected_rows_loaded": False,
        },
        {
            "label": "primary_scoring_batches",
            "loaded_rows": len(
                PRIMARY_SCORING_BATCHES_ANALYTICAL
            ),
            "loaded_columns": (
                PRIMARY_SCORING_BATCHES_ANALYTICAL.shape[1]
            ),
            "loaded_partitions": ",".join(
                ANALYTICAL_PARTITIONS
            ),
            "protected_rows_loaded": False,
        },
        {
            "label": "primary_event_to_batch_membership",
            "loaded_rows": len(
                PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL
            ),
            "loaded_columns": (
                PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL.shape[
                    1
                ]
            ),
            "loaded_partitions": ",".join(
                ANALYTICAL_PARTITIONS
            ),
            "protected_rows_loaded": False,
        },
        {
            "label": "primary_batch_feature_identities",
            "loaded_rows": len(
                PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL
            ),
            "loaded_columns": (
                PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL.shape[
                    1
                ]
            ),
            "loaded_partitions": ",".join(
                ANALYTICAL_PARTITIONS
            ),
            "protected_rows_loaded": False,
        },
        {
            "label": "observation_window_contract",
            "loaded_rows": len(
                OBSERVATION_WINDOW_CONTRACT
            ),
            "loaded_columns": (
                OBSERVATION_WINDOW_CONTRACT.shape[1]
            ),
            "loaded_partitions": "METADATA_ALL_PARTITIONS",
            "protected_rows_loaded": False,
        },
    ]
)

display(
    UPSTREAM_DATA_ARTIFACT_AUDIT[
        [
            "label",
            "artifact_type",
            "full_row_count",
            "full_column_count",
            "verified",
        ]
    ]
)

display(ANALYTICAL_LOAD_AUDIT)
display(PARTITION_ACCESS_LEDGER)
display(analytical_window_contract)

print(
    "Authoritative Notebook 04 event and batch inputs were verified."
)
print(
    "Only DEVELOPMENT and CALIBRATION event content was loaded."
)
print(
    "Notebook 05 was loaded through identity columns only; "
    "future labels remain unopened."
)
print(
    "VALIDATION and ENGINEERING_HOLDOUT event content remain protected."
)

,label,artifact_type,full_row_count,full_column_count,verified
0,primary_estimation_events,NOTEBOOK_04_PRIMARY_ESTIMATION_EVENTS,13887,88,True
1,primary_exact_time_batches,NOTEBOOK_04_PRIMARY_EXACT_TIME_BATCHES,13564,27,True
2,primary_scoring_batches,NOTEBOOK_04_PRIMARY_SCORING_BATCHES,13564,25,True
3,primary_event_to_batch_membership,NOTEBOOK_04_PRIMARY_EVENT_TO_BATCH_MEMBERSHIP,13887,10,True
4,observation_window_contract,NOTEBOOK_04_OBSERVATION_WINDOW_CONTRACT,4,19,True
5,primary_batch_market_state_features,PRIMARY_BATCH_MARKET_STATE_FEATURES,13564,577,True


,label,loaded_rows,loaded_columns,loaded_partitions,protected_rows_loaded
0,primary_estimation_events,9497,88,"DEVELOPMENT,CALIBRATION",False
1,primary_exact_time_batches,9259,27,"DEVELOPMENT,CALIBRATION",False
2,primary_scoring_batches,9259,25,"DEVELOPMENT,CALIBRATION",False
3,primary_event_to_batch_membership,9497,10,"DEVELOPMENT,CALIBRATION",False
4,primary_batch_feature_identities,9259,6,"DEVELOPMENT,CALIBRATION",False
5,observation_window_contract,4,19,METADATA_ALL_PARTITIONS,False


,partition,partition_role,event_content_loaded,authorized_in_notebook_06,status
0,DEVELOPMENT,ANALYTICAL,True,True,PASS
1,CALIBRATION,ANALYTICAL,True,True,PASS
2,VALIDATION,PROTECTED,False,False,PASS
3,ENGINEERING_HOLDOUT,PROTECTED,False,False,PASS


,partition_order,event_partition,contract_sequence_start,contract_sequence_end_exclusive,contract_start_ns,contract_end_exclusive_ns,contract_duration_ns,event_clock_origin_ns,first_event_time_ns,last_event_time_ns,left_censoring_duration_ns,observation_window_duration_ns,primary_event_count,primary_batch_count,simultaneous_batch_count,mixed_side_batch_count,contract_interval_convention,event_clock_origin_rule,left_censoring_preserved
0,1,DEVELOPMENT,1,51840,1783665467531985400,1783667269391572100,1801859586700,1783665468766951600,1783665468766951600,1783667269232205000,1234966200,1800624620500,7004,6859,111,26,"[start, end)",FIRST_PRIMARY_EVENT_LOCAL_RECEIPT_TIME,True
1,2,CALIBRATION,51840,72576,1783667269391572100,1783667989690751200,720299179100,1783667270546982000,1783667270546982000,1783667989349685300,1155409900,719143769200,2493,2400,68,5,"[start, end)",FIRST_PRIMARY_EVENT_LOCAL_RECEIPT_TIME,True


Authoritative Notebook 04 event and batch inputs were verified.
Only DEVELOPMENT and CALIBRATION event content was loaded.
Notebook 05 was loaded through identity columns only; future labels remain unopened.
VALIDATION and ENGINEERING_HOLDOUT event content remain protected.


In [6]:
# ============================================================
# Canonical batch identity, timestamp provenance, and tie audit
# ============================================================

BATCH_IDENTITY_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
)

NON_TIME_BATCH_IDENTITY_COLUMNS: Final[tuple[str, ...]] = (
    "primary_event_batch_id",
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
)

BATCH_AGGREGATE_COLUMNS: Final[tuple[str, ...]] = (
    "batch_event_count",
    "buy_event_count",
    "sell_event_count",
    "unique_side_count",
    "mixed_side_batch_flag",
    "simultaneous_batch_required_flag",
)


# ------------------------------------------------------------
# Exact identity normalization
# ------------------------------------------------------------

def parse_exact_int64(
    series: pd.Series,
    *,
    label: str,
) -> pd.Series:
    """
    Convert an integer-valued series to int64 without routing existing
    integer or string-backed nanosecond values through float64.
    """
    require(
        isinstance(series, pd.Series),
        f"{label} is not a pandas Series.",
    )
    require(
        not series.isna().any(),
        f"{label} contains missing values.",
    )

    if pd.api.types.is_integer_dtype(series.dtype):
        return series.astype("int64")

    if pd.api.types.is_float_dtype(series.dtype):
        values = series.to_numpy(dtype="float64")

        require(
            np.isfinite(values).all(),
            f"{label} contains nonfinite values.",
        )
        require(
            np.equal(values, np.rint(values)).all(),
            f"{label} contains non-integer float values.",
        )
        require(
            (values >= np.iinfo(np.int64).min).all()
            and (values <= np.iinfo(np.int64).max).all(),
            f"{label} contains values outside int64 bounds.",
        )

        return pd.Series(
            np.rint(values).astype("int64"),
            index=series.index,
            name=series.name,
        )

    parsed_values: list[int] = []

    for row_number, value in enumerate(series.tolist()):
        if isinstance(value, (int, np.integer)):
            parsed_values.append(int(value))
            continue

        text = str(value).strip()

        require(
            bool(text)
            and text.lstrip("+-").isdigit(),
            (
                f"{label} contains a non-integer value at row "
                f"{row_number}: {value!r}"
            ),
        )

        parsed_value = int(text)

        require(
            np.iinfo(np.int64).min
            <= parsed_value
            <= np.iinfo(np.int64).max,
            (
                f"{label} contains an out-of-range int64 value "
                f"at row {row_number}: {parsed_value}"
            ),
        )

        parsed_values.append(parsed_value)

    return pd.Series(
        parsed_values,
        index=series.index,
        dtype="int64",
        name=series.name,
    )


def normalize_batch_identity(
    frame: pd.DataFrame,
    *,
    label: str,
) -> pd.DataFrame:
    """Return a canonical one-row-per-batch identity frame."""
    require_schema_columns(
        VerifiedParquetArtifact(
            label=label,
            artifact_type="IN_MEMORY_FRAME",
            parquet_path=Path("."),
            contract_path=Path("."),
            row_count=len(frame),
            column_count=frame.shape[1],
            columns=tuple(str(column) for column in frame.columns),
            table_payload_sha256="",
            raw_file_sha256="",
            contract_payload_sha256="",
        ),
        BATCH_IDENTITY_COLUMNS,
    )

    normalized = frame.loc[
        :,
        list(BATCH_IDENTITY_COLUMNS),
    ].copy()

    normalized["primary_event_batch_id"] = (
        normalized["primary_event_batch_id"]
        .astype("string")
        .str.strip()
    )

    normalized["event_partition"] = (
        normalized["event_partition"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    for column_name in (
        "primary_event_batch_number",
        "partition_batch_index",
        "event_partition_order",
        "event_time_ns",
    ):
        normalized[column_name] = parse_exact_int64(
            normalized[column_name],
            label=f"{label}.{column_name}",
        )

    require(
        normalized["primary_event_batch_id"].notna().all(),
        f"{label} contains missing batch IDs.",
    )
    require(
        normalized["primary_event_batch_id"].str.len().gt(0).all(),
        f"{label} contains empty batch IDs.",
    )
    require(
        normalized["primary_event_batch_id"].is_unique,
        f"{label} contains duplicate batch IDs.",
    )
    require(
        normalized["primary_event_batch_number"].is_unique,
        f"{label} contains duplicate batch numbers.",
    )
    require(
        set(normalized["event_partition"]).issubset(
            set(ANALYTICAL_PARTITIONS)
        ),
        f"{label} contains an unauthorized partition.",
    )

    return (
        normalized.sort_values(
            "primary_event_batch_number",
            kind="stable",
        )
        .reset_index(drop=True)
    )


NOTEBOOK_04_CANONICAL_BATCH_IDENTITY = normalize_batch_identity(
    PRIMARY_SCORING_BATCHES_ANALYTICAL,
    label="Notebook 04 primary scoring batches",
)

NOTEBOOK_05_PERSISTED_FEATURE_IDENTITY = normalize_batch_identity(
    PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL,
    label="Notebook 05 persisted feature identities",
)


# ------------------------------------------------------------
# Join by stable batch ID, never by row position
# ------------------------------------------------------------

BATCH_IDENTITY_RECONCILIATION = (
    NOTEBOOK_04_CANONICAL_BATCH_IDENTITY.merge(
        NOTEBOOK_05_PERSISTED_FEATURE_IDENTITY,
        on="primary_event_batch_id",
        how="outer",
        validate="one_to_one",
        suffixes=("_n04", "_n05"),
        indicator=True,
    )
    .sort_values(
        "primary_event_batch_number_n04",
        kind="stable",
        na_position="last",
    )
    .reset_index(drop=True)
)

require(
    BATCH_IDENTITY_RECONCILIATION["_merge"].eq("both").all(),
    (
        "Notebook 04 and Notebook 05 do not contain the same "
        "analytical batch-ID set."
    ),
)

for column_name in (
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
):
    left_column = f"{column_name}_n04"
    right_column = f"{column_name}_n05"

    mismatch_mask = (
        BATCH_IDENTITY_RECONCILIATION[left_column]
        .ne(BATCH_IDENTITY_RECONCILIATION[right_column])
    )

    require(
        not mismatch_mask.any(),
        (
            f"Notebook 05 differs from Notebook 04 in stable "
            f"identity field {column_name}: "
            f"{int(mismatch_mask.sum())} rows."
        ),
    )


# ------------------------------------------------------------
# Diagnose Notebook 05 nanosecond persistence precision
# ------------------------------------------------------------

n04_event_time_ns = (
    BATCH_IDENTITY_RECONCILIATION[
        "event_time_ns_n04"
    ]
    .astype("int64")
    .to_numpy()
)

n05_event_time_ns = (
    BATCH_IDENTITY_RECONCILIATION[
        "event_time_ns_n05"
    ]
    .astype("int64")
    .to_numpy()
)

# Notebook 05 persisted all logical int64 fields through a
# float64-backed normalization step before writing Parquet.
# Reproduce that transformation explicitly for provenance.
n04_event_time_as_float64 = n04_event_time_ns.astype(
    "float64"
)

n04_event_time_after_float64_roundtrip = np.rint(
    n04_event_time_as_float64
).astype("int64")

event_time_delta_ns = (
    n05_event_time_ns
    - n04_event_time_ns
)

event_time_mismatch_mask = (
    event_time_delta_ns != 0
)

float64_roundtrip_match_mask = (
    n05_event_time_ns
    == n04_event_time_after_float64_roundtrip
)

float64_ulp_ns = np.abs(
    np.spacing(n04_event_time_as_float64)
)

float64_half_ulp_ceiling_ns = np.ceil(
    float64_ulp_ns / 2.0
).astype("int64")

event_time_within_half_ulp_mask = (
    np.abs(event_time_delta_ns)
    <= float64_half_ulp_ceiling_ns
)

event_time_mismatch_count = int(
    event_time_mismatch_mask.sum()
)

if event_time_mismatch_count:
    require(
        bool(float64_roundtrip_match_mask.all()),
        (
            "Notebook 05 event_time_ns differences are not fully "
            "explained by its persisted float64 round-trip path."
        ),
    )
    require(
        bool(event_time_within_half_ulp_mask.all()),
        (
            "Notebook 05 event_time_ns differences exceed the "
            "expected float64 half-ULP bound."
        ),
    )
    require(
        int(np.max(np.abs(event_time_delta_ns)))
        < NANOSECONDS_PER_MILLISECOND,
        (
            "Notebook 05 timestamp persistence error reaches or "
            "exceeds one millisecond."
        ),
    )

    NOTEBOOK_05_TIMESTAMP_IDENTITY_STATUS = (
        "PASS_WITH_UPSTREAM_INT64_FLOAT64_ROUNDTRIP_WARNING"
    )
else:
    NOTEBOOK_05_TIMESTAMP_IDENTITY_STATUS = "EXACT_MATCH"


NOTEBOOK_05_FEATURE_TIMESTAMP_DISCREPANCIES = pd.DataFrame(
    {
        "primary_event_batch_id": (
            BATCH_IDENTITY_RECONCILIATION[
                "primary_event_batch_id"
            ].astype("string")
        ),
        "primary_event_batch_number": (
            BATCH_IDENTITY_RECONCILIATION[
                "primary_event_batch_number_n04"
            ].astype("int64")
        ),
        "event_partition": (
            BATCH_IDENTITY_RECONCILIATION[
                "event_partition_n04"
            ].astype("string")
        ),
        "notebook_04_event_time_ns": n04_event_time_ns,
        "notebook_05_persisted_event_time_ns": (
            n05_event_time_ns
        ),
        "expected_float64_roundtrip_event_time_ns": (
            n04_event_time_after_float64_roundtrip
        ),
        "event_time_delta_ns": event_time_delta_ns,
        "absolute_event_time_delta_ns": np.abs(
            event_time_delta_ns
        ),
        "float64_ulp_ns": float64_ulp_ns,
        "within_half_ulp_flag": (
            event_time_within_half_ulp_mask
        ),
        "float64_roundtrip_explains_value_flag": (
            float64_roundtrip_match_mask
        ),
        "timestamp_mismatch_flag": (
            event_time_mismatch_mask
        ),
    }
)

NOTEBOOK_05_FEATURE_TIMESTAMP_MISMATCH_ROWS = (
    NOTEBOOK_05_FEATURE_TIMESTAMP_DISCREPANCIES.loc[
        NOTEBOOK_05_FEATURE_TIMESTAMP_DISCREPANCIES[
            "timestamp_mismatch_flag"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Canonical analytical identity
# ------------------------------------------------------------

# Notebook 04 remains the event-time and batch-identity authority.
# The Notebook 05 on-disk artifact is not modified.
PRIMARY_BATCH_FEATURE_IDENTITIES_CANONICAL = (
    NOTEBOOK_04_CANONICAL_BATCH_IDENTITY.copy()
)

PRIMARY_BATCH_FEATURE_IDENTITY_LINK = (
    BATCH_IDENTITY_RECONCILIATION[
        [
            "primary_event_batch_id",
            "primary_event_batch_number_n04",
            "partition_batch_index_n04",
            "event_partition_order_n04",
            "event_partition_n04",
            "event_time_ns_n04",
            "event_time_ns_n05",
        ]
    ]
    .rename(
        columns={
            "primary_event_batch_number_n04": (
                "primary_event_batch_number"
            ),
            "partition_batch_index_n04": (
                "partition_batch_index"
            ),
            "event_partition_order_n04": (
                "event_partition_order"
            ),
            "event_partition_n04": (
                "event_partition"
            ),
            "event_time_ns_n04": (
                "canonical_event_time_ns"
            ),
            "event_time_ns_n05": (
                "notebook_05_persisted_event_time_ns"
            ),
        }
    )
    .copy()
)

PRIMARY_BATCH_FEATURE_IDENTITY_LINK[
    "event_time_delta_ns"
] = (
    PRIMARY_BATCH_FEATURE_IDENTITY_LINK[
        "notebook_05_persisted_event_time_ns"
    ].astype("int64")
    - PRIMARY_BATCH_FEATURE_IDENTITY_LINK[
        "canonical_event_time_ns"
    ].astype("int64")
)

PRIMARY_BATCH_FEATURE_IDENTITY_LINK[
    "canonical_time_authority"
] = "NOTEBOOK_04_PRIMARY_SCORING_BATCHES"

PRIMARY_BATCH_FEATURE_IDENTITY_LINK[
    "notebook_05_artifact_modified"
] = False


# ------------------------------------------------------------
# Scoring-batch versus exact-time-batch equivalence
# ------------------------------------------------------------

exact_batch_columns = [
    *BATCH_IDENTITY_COLUMNS,
    *BATCH_AGGREGATE_COLUMNS,
]

scoring_batch_columns = [
    *BATCH_IDENTITY_COLUMNS,
    *BATCH_AGGREGATE_COLUMNS,
]

EXACT_BATCH_COMPARISON = (
    PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL[
        exact_batch_columns
    ]
    .copy()
    .merge(
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            scoring_batch_columns
        ].copy(),
        on="primary_event_batch_id",
        how="outer",
        validate="one_to_one",
        suffixes=("_exact", "_scoring"),
        indicator=True,
    )
)

require(
    EXACT_BATCH_COMPARISON["_merge"].eq("both").all(),
    (
        "Exact-time batches and scoring batches do not contain "
        "the same analytical batch-ID set."
    ),
)

for column_name in (
    "primary_event_batch_number",
    "partition_batch_index",
    "event_partition_order",
    "event_partition",
    "event_time_ns",
    *BATCH_AGGREGATE_COLUMNS,
):
    left = EXACT_BATCH_COMPARISON[
        f"{column_name}_exact"
    ]
    right = EXACT_BATCH_COMPARISON[
        f"{column_name}_scoring"
    ]

    require(
        left.astype("string").eq(
            right.astype("string")
        ).all(),
        (
            "Exact-time and scoring-batch tables differ in "
            f"{column_name}."
        ),
    )


# ------------------------------------------------------------
# Event-to-batch membership equivalence
# ------------------------------------------------------------

event_membership_check = (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        [
            "primary_event_id",
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
            "event_side",
        ]
    ]
    .copy()
    .merge(
        PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL[
            [
                "primary_event_id",
                "primary_event_batch_id",
                "event_partition",
                "event_time_ns",
            ]
        ].copy(),
        on="primary_event_id",
        how="outer",
        validate="one_to_one",
        suffixes=("_event", "_membership"),
        indicator=True,
    )
)

require(
    event_membership_check["_merge"].eq("both").all(),
    (
        "Primary events and event-to-batch membership do not "
        "contain the same analytical event-ID set."
    ),
)

for column_name in (
    "primary_event_batch_id",
    "event_partition",
    "event_time_ns",
):
    require(
        event_membership_check[
            f"{column_name}_event"
        ]
        .astype("string")
        .eq(
            event_membership_check[
                f"{column_name}_membership"
            ].astype("string")
        )
        .all(),
        (
            "Event-to-batch membership differs from the event "
            f"table in {column_name}."
        ),
    )


# ------------------------------------------------------------
# Reconstruct batch counts directly from primary events
# ------------------------------------------------------------

event_batch_source = (
    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL[
        [
            "primary_event_id",
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
            "event_side",
        ]
    ]
    .copy()
)

event_batch_source["event_partition"] = (
    event_batch_source["event_partition"]
    .astype("string")
    .str.strip()
    .str.upper()
)

event_batch_source["event_side"] = (
    event_batch_source["event_side"]
    .astype("string")
    .str.strip()
    .str.upper()
)

require(
    event_batch_source["event_side"].isin(
        {"BUY", "SELL"}
    ).all(),
    "Primary analytical events contain an unsupported side.",
)

derived_batch_counts = (
    event_batch_source.assign(
        buy_indicator=(
            event_batch_source["event_side"].eq("BUY")
        ).astype("int64"),
        sell_indicator=(
            event_batch_source["event_side"].eq("SELL")
        ).astype("int64"),
    )
    .groupby(
        "primary_event_batch_id",
        sort=False,
        observed=True,
    )
    .agg(
        event_partition=("event_partition", "first"),
        unique_partition_count=(
            "event_partition",
            "nunique",
        ),
        event_time_ns=("event_time_ns", "first"),
        unique_event_time_count=(
            "event_time_ns",
            "nunique",
        ),
        batch_event_count=("primary_event_id", "size"),
        buy_event_count=("buy_indicator", "sum"),
        sell_event_count=("sell_indicator", "sum"),
        unique_side_count=("event_side", "nunique"),
    )
    .reset_index()
)

derived_batch_counts[
    "mixed_side_batch_flag"
] = (
    derived_batch_counts["buy_event_count"].gt(0)
    & derived_batch_counts["sell_event_count"].gt(0)
)

derived_batch_counts[
    "simultaneous_batch_required_flag"
] = derived_batch_counts["batch_event_count"].gt(1)

require(
    derived_batch_counts[
        "unique_partition_count"
    ].eq(1).all(),
    "At least one batch spans multiple event partitions.",
)

require(
    derived_batch_counts[
        "unique_event_time_count"
    ].eq(1).all(),
    "At least one exact-time batch contains multiple timestamps.",
)

derived_batch_comparison = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        [
            "primary_event_batch_id",
            "event_partition",
            "event_time_ns",
            *BATCH_AGGREGATE_COLUMNS,
        ]
    ]
    .copy()
    .merge(
        derived_batch_counts[
            [
                "primary_event_batch_id",
                "event_partition",
                "event_time_ns",
                *BATCH_AGGREGATE_COLUMNS,
            ]
        ].copy(),
        on="primary_event_batch_id",
        how="outer",
        validate="one_to_one",
        suffixes=("_stored", "_derived"),
        indicator=True,
    )
)

require(
    derived_batch_comparison["_merge"].eq("both").all(),
    "Stored and event-derived batch sets differ.",
)

for column_name in (
    "event_partition",
    "event_time_ns",
    *BATCH_AGGREGATE_COLUMNS,
):
    require(
        derived_batch_comparison[
            f"{column_name}_stored"
        ]
        .astype("string")
        .eq(
            derived_batch_comparison[
                f"{column_name}_derived"
            ].astype("string")
        )
        .all(),
        (
            "Stored batch metadata differs from direct event "
            f"reconstruction in {column_name}."
        ),
    )


# ------------------------------------------------------------
# Partition ordering and event-clock diagnostics
# ------------------------------------------------------------

event_clock_rows: list[dict[str, Any]] = []

for partition_name in ANALYTICAL_PARTITIONS:
    partition_batches = (
        PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
            PRIMARY_SCORING_BATCHES_ANALYTICAL[
                "event_partition"
            ]
            .astype("string")
            .str.upper()
            .eq(partition_name)
        ]
        .sort_values(
            "partition_batch_index",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    require(
        not partition_batches.empty,
        f"{partition_name} contains no scoring batches.",
    )

    partition_times_ns = parse_exact_int64(
        partition_batches["event_time_ns"],
        label=f"{partition_name}.event_time_ns",
    ).to_numpy()

    require(
        np.all(np.diff(partition_times_ns) > 0),
        (
            f"{partition_name} scoring-batch times are not "
            "strictly increasing."
        ),
    )

    positive_gaps_ns = np.diff(partition_times_ns)

    require(
        positive_gaps_ns.size > 0,
        (
            f"{partition_name} does not contain enough batches "
            "for event-clock diagnostics."
        ),
    )

    empirical_gap_gcd_ns = int(
        np.gcd.reduce(positive_gaps_ns)
    )

    partition_contract_row = (
        analytical_window_contract.loc[
            analytical_window_contract[
                "event_partition"
            ].eq(partition_name)
        ]
    )

    require(
        len(partition_contract_row) == 1,
        (
            f"Observation-window contract has an invalid row "
            f"count for {partition_name}."
        ),
    )

    partition_contract = (
        partition_contract_row.iloc[0]
    )

    stored_simultaneous_count = int(
        partition_batches[
            "simultaneous_batch_required_flag"
        ]
        .astype(bool)
        .sum()
    )

    stored_mixed_side_count = int(
        partition_batches[
            "mixed_side_batch_flag"
        ]
        .astype(bool)
        .sum()
    )

    require(
        stored_simultaneous_count
        == int(
            partition_contract[
                "simultaneous_batch_count"
            ]
        ),
        (
            f"{partition_name} simultaneous-batch count differs "
            "from the observation-window contract."
        ),
    )

    require(
        stored_mixed_side_count
        == int(
            partition_contract[
                "mixed_side_batch_count"
            ]
        ),
        (
            f"{partition_name} mixed-side batch count differs "
            "from the observation-window contract."
        ),
    )

    event_clock_rows.append(
        {
            "event_partition": partition_name,
            "batch_count": int(
                len(partition_batches)
            ),
            "event_count": int(
                partition_batches[
                    "batch_event_count"
                ].sum()
            ),
            "first_batch_time_ns": int(
                partition_times_ns[0]
            ),
            "last_batch_time_ns": int(
                partition_times_ns[-1]
            ),
            "minimum_positive_batch_gap_ns": int(
                positive_gaps_ns.min()
            ),
            "median_positive_batch_gap_ns": float(
                np.median(positive_gaps_ns)
            ),
            "maximum_positive_batch_gap_ns": int(
                positive_gaps_ns.max()
            ),
            "empirical_positive_gap_gcd_ns": (
                empirical_gap_gcd_ns
            ),
            "simultaneous_batch_count": (
                stored_simultaneous_count
            ),
            "mixed_side_batch_count": (
                stored_mixed_side_count
            ),
            "strictly_increasing_batch_time_flag": True,
            "timestamp_jitter_applied": False,
            "tied_event_order_imposed": False,
            "status": "PASS",
        }
    )


EVENT_CLOCK_AND_TIE_AUDIT = pd.DataFrame(
    event_clock_rows
)


# ------------------------------------------------------------
# Summary gates
# ------------------------------------------------------------

expected_analytical_simultaneous_batches = int(
    analytical_window_contract[
        "simultaneous_batch_count"
    ].sum()
)

expected_analytical_mixed_side_batches = int(
    analytical_window_contract[
        "mixed_side_batch_count"
    ].sum()
)

observed_analytical_simultaneous_batches = int(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "simultaneous_batch_required_flag"
    ]
    .astype(bool)
    .sum()
)

observed_analytical_mixed_side_batches = int(
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        "mixed_side_batch_flag"
    ]
    .astype(bool)
    .sum()
)

require(
    observed_analytical_simultaneous_batches
    == expected_analytical_simultaneous_batches,
    "Analytical simultaneous-batch total is incorrect.",
)

require(
    observed_analytical_mixed_side_batches
    == expected_analytical_mixed_side_batches,
    "Analytical mixed-side-batch total is incorrect.",
)

require(
    int(
        PRIMARY_SCORING_BATCHES_ANALYTICAL[
            "batch_event_count"
        ].sum()
    )
    == len(PRIMARY_ESTIMATION_EVENTS_ANALYTICAL),
    (
        "Scoring-batch event multiplicities do not conserve "
        "the analytical primary-event count."
    ),
)


NOTEBOOK_05_FEATURE_IDENTITY_RECONCILIATION_SUMMARY = (
    pd.DataFrame(
        [
            {
                "check": "stable_batch_id_set",
                "observed": int(
                    len(
                        BATCH_IDENTITY_RECONCILIATION
                    )
                ),
                "expected": int(
                    len(
                        NOTEBOOK_04_CANONICAL_BATCH_IDENTITY
                    )
                ),
                "status": "PASS",
            },
            {
                "check": "non_time_identity_mismatch_count",
                "observed": 0,
                "expected": 0,
                "status": "PASS",
            },
            {
                "check": "event_time_mismatch_count",
                "observed": (
                    event_time_mismatch_count
                ),
                "expected": 0,
                "status": (
                    "WARNING"
                    if event_time_mismatch_count
                    else "PASS"
                ),
            },
            {
                "check": "maximum_absolute_event_time_delta_ns",
                "observed": int(
                    np.max(
                        np.abs(
                            event_time_delta_ns
                        )
                    )
                ),
                "expected": 0,
                "status": (
                    "WARNING"
                    if event_time_mismatch_count
                    else "PASS"
                ),
            },
            {
                "check": (
                    "timestamp_difference_fully_explained_by_"
                    "notebook_05_float64_roundtrip"
                ),
                "observed": bool(
                    float64_roundtrip_match_mask.all()
                ),
                "expected": True,
                "status": "PASS",
            },
            {
                "check": "notebook_04_time_authority_preserved",
                "observed": True,
                "expected": True,
                "status": "PASS",
            },
            {
                "check": "notebook_05_disk_artifact_modified",
                "observed": False,
                "expected": False,
                "status": "PASS",
            },
        ]
    )
)


ANALYTICAL_EVENT_INTERFACE_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "notebook_04_batch_identity_authoritative",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                f"batch_rows="
                f"{len(NOTEBOOK_04_CANONICAL_BATCH_IDENTITY)}"
            ),
        },
        {
            "gate": "notebook_05_stable_batch_identity_matches",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": "all non-time identity fields match",
        },
        {
            "gate": "notebook_05_timestamp_precision_classified",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": NOTEBOOK_05_TIMESTAMP_IDENTITY_STATUS,
        },
        {
            "gate": "exact_and_scoring_batch_tables_equivalent",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                f"batch_rows="
                f"{len(PRIMARY_SCORING_BATCHES_ANALYTICAL)}"
            ),
        },
        {
            "gate": "event_membership_conserved",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                f"event_rows="
                f"{len(PRIMARY_ESTIMATION_EVENTS_ANALYTICAL)}"
            ),
        },
        {
            "gate": "event_derived_batch_counts_match",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "BUY, SELL, mixed-side, and simultaneous counts match"
            ),
        },
        {
            "gate": "simultaneous_batch_semantics_preserved",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                f"simultaneous_batches="
                f"{observed_analytical_simultaneous_batches}; "
                f"mixed_side_batches="
                f"{observed_analytical_mixed_side_batches}"
            ),
        },
        {
            "gate": "timestamp_jitter_absent",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": "Notebook 04 event times used unchanged",
        },
        {
            "gate": "protected_partition_content_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    ANALYTICAL_EVENT_INTERFACE_GATE_FRAME.loc[
        ANALYTICAL_EVENT_INTERFACE_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    "At least one analytical event-interface gate failed.",
)


display(
    NOTEBOOK_05_FEATURE_IDENTITY_RECONCILIATION_SUMMARY
)

if event_time_mismatch_count:
    display(
        NOTEBOOK_05_FEATURE_TIMESTAMP_MISMATCH_ROWS.head(
            20
        )
    )

display(EVENT_CLOCK_AND_TIE_AUDIT)
display(ANALYTICAL_EVENT_INTERFACE_GATE_FRAME)

print(
    "Canonical analytical batch identity verified by stable batch ID."
)
print(
    "Notebook 04 remains authoritative for exact event_time_ns."
)
print(
    f"Notebook 05 timestamp identity status: "
    f"{NOTEBOOK_05_TIMESTAMP_IDENTITY_STATUS}."
)
print(
    "No Notebook 05 artifact was modified; downstream analytical "
    "joins must use primary_event_batch_id and the canonical "
    "Notebook 04 event time."
)
print(
    "Exact-time batch membership, BUY/SELL counts, simultaneous "
    "batches, mixed-side batches, and partition ordering passed."
)

,check,observed,expected,status
0,stable_batch_id_set,9259,9259,PASS
1,non_time_identity_mismatch_count,0,0,PASS
2,event_time_mismatch_count,9105,0,WARNING
3,maximum_absolute_event_time_delta_ns,128,0,WARNING
4,timestamp_difference_fully_explained_by_notebo...,True,True,PASS
5,notebook_04_time_authority_preserved,True,True,PASS
6,notebook_05_disk_artifact_modified,False,False,PASS


,primary_event_batch_id,primary_event_batch_number,event_partition,notebook_04_event_time_ns,notebook_05_persisted_event_time_ns,expected_float64_roundtrip_event_time_ns,event_time_delta_ns,absolute_event_time_delta_ns,float64_ulp_ns,within_half_ulp_flag,float64_roundtrip_explains_value_flag,timestamp_mismatch_flag
0,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,1,DEVELOPMENT,1783665468766951600,1783665468766951680,1783665468766951680,80,80,256,True,True,True
1,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,2,DEVELOPMENT,1783665469052166500,1783665469052166400,1783665469052166400,-100,100,256,True,True,True
2,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,3,DEVELOPMENT,1783665469437964400,1783665469437964288,1783665469437964288,-112,112,256,True,True,True
3,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,4,DEVELOPMENT,1783665469692104200,1783665469692104192,1783665469692104192,-8,8,256,True,True,True
4,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,5,DEVELOPMENT,1783665470242312100,1783665470242312192,1783665470242312192,92,92,256,True,True,True
5,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,6,DEVELOPMENT,1783665470304239100,1783665470304239104,1783665470304239104,4,4,256,True,True,True
6,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,7,DEVELOPMENT,1783665470597198800,1783665470597198848,1783665470597198848,48,48,256,True,True,True
7,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,8,DEVELOPMENT,1783665471662326600,1783665471662326528,1783665471662326528,-72,72,256,True,True,True
8,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,9,DEVELOPMENT,1783665471742062100,1783665471742062080,1783665471742062080,-20,20,256,True,True,True
9,BTCUSDT_spot_20260710T063746Z_c8b5bf12__v0_1_2...,10,DEVELOPMENT,1783665471862162900,1783665471862162944,1783665471862162944,44,44,256,True,True,True


,event_partition,batch_count,event_count,first_batch_time_ns,last_batch_time_ns,minimum_positive_batch_gap_ns,median_positive_batch_gap_ns,maximum_positive_batch_gap_ns,empirical_positive_gap_gcd_ns,simultaneous_batch_count,mixed_side_batch_count,strictly_increasing_batch_time_flag,timestamp_jitter_applied,tied_event_order_imposed,status
0,DEVELOPMENT,6859,7004,1783665468766951600,1783667269232205000,360300,1.630179e+08,2655478900,100,111,26,True,False,False,PASS
1,CALIBRATION,2400,2493,1783667270546982000,1783667989349685300,393200,1.758249e+08,3072811700,100,68,5,True,False,False,PASS


,gate,severity,passed,evidence
0,notebook_04_batch_identity_authoritative,BLOCKING,True,batch_rows=9259
1,notebook_05_stable_batch_identity_matches,BLOCKING,True,all non-time identity fields match
2,notebook_05_timestamp_precision_classified,BLOCKING,True,PASS_WITH_UPSTREAM_INT64_FLOAT64_ROUNDTRIP_WAR...
3,exact_and_scoring_batch_tables_equivalent,BLOCKING,True,batch_rows=9259
4,event_membership_conserved,BLOCKING,True,event_rows=9497
5,event_derived_batch_counts_match,BLOCKING,True,"BUY, SELL, mixed-side, and simultaneous counts..."
6,simultaneous_batch_semantics_preserved,BLOCKING,True,simultaneous_batches=179; mixed_side_batches=31
7,timestamp_jitter_absent,BLOCKING,True,Notebook 04 event times used unchanged
8,protected_partition_content_absent,BLOCKING,True,VALIDATION=False; ENGINEERING_HOLDOUT=False


Canonical analytical batch identity verified by stable batch ID.
Notebook 04 remains authoritative for exact event_time_ns.
Notebook 05 timestamp identity status: PASS_WITH_UPSTREAM_INT64_FLOAT64_ROUNDTRIP_WARNING.
No Notebook 05 artifact was modified; downstream analytical joins must use primary_event_batch_id and the canonical Notebook 04 event time.
Exact-time batch membership, BUY/SELL counts, simultaneous batches, mixed-side batches, and partition ordering passed.


In [7]:
# ============================================================
# Calendar coverage, session continuity, and count-grid contract
# ============================================================

TIME_OF_DAY_REPEATABILITY_WIDTHS_MINUTES: Final[
    tuple[int, ...]
] = (
    1,
    5,
    15,
    60,
)

MODEL_COUNT_GRID_NS: Final[int] = (
    NANOSECONDS_PER_MILLISECOND
)
MODEL_COUNT_GRID_MS: Final[int] = 1

FORMAL_TIME_OF_DAY_BASELINE_AUTHORIZED = False
FORMAL_DAY_OF_WEEK_BASELINE_AUTHORIZED = False
FORMAL_CROSS_DAY_SEASONAL_BASELINE_AUTHORIZED = False
DETERMINISTIC_ELAPSED_TIME_BASELINE_AUTHORIZED = True


# ------------------------------------------------------------
# Exact contract-window normalization
# ------------------------------------------------------------

def normalize_contract_windows(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Return exactly typed and chronologically ordered contract windows."""
    required_columns = {
        "partition_order",
        "event_partition",
        "contract_start_ns",
        "contract_end_exclusive_ns",
        "contract_duration_ns",
        "observation_window_duration_ns",
        "left_censoring_duration_ns",
        "primary_event_count",
        "primary_batch_count",
        "contract_interval_convention",
    }

    missing_columns = sorted(
        required_columns - set(frame.columns)
    )

    require(
        not missing_columns,
        (
            "Observation-window contract is missing required "
            f"columns: {missing_columns}"
        ),
    )

    normalized = frame.loc[
        :,
        sorted(required_columns),
    ].copy()

    normalized["event_partition"] = (
        normalized["event_partition"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    for column_name in (
        "partition_order",
        "contract_start_ns",
        "contract_end_exclusive_ns",
        "contract_duration_ns",
        "observation_window_duration_ns",
        "left_censoring_duration_ns",
        "primary_event_count",
        "primary_batch_count",
    ):
        normalized[column_name] = parse_exact_int64(
            normalized[column_name],
            label=(
                "observation_window_contract."
                f"{column_name}"
            ),
        )

    normalized = (
        normalized.sort_values(
            "partition_order",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    require(
        normalized["partition_order"].is_unique,
        "Partition order is not unique.",
    )
    require(
        normalized["event_partition"].is_unique,
        "Event partition is not unique.",
    )
    require(
        normalized[
            "contract_interval_convention"
        ].eq("[start, end)").all(),
        "At least one partition violates the half-open interval contract.",
    )

    starts_ns = normalized[
        "contract_start_ns"
    ].to_numpy(dtype="int64")

    ends_ns = normalized[
        "contract_end_exclusive_ns"
    ].to_numpy(dtype="int64")

    durations_ns = normalized[
        "contract_duration_ns"
    ].to_numpy(dtype="int64")

    require(
        np.all(ends_ns > starts_ns),
        "At least one contract window has nonpositive duration.",
    )
    require(
        np.array_equal(
            ends_ns - starts_ns,
            durations_ns,
        ),
        (
            "At least one registered contract duration differs "
            "from end-exclusive minus start."
        ),
    )

    inter_window_gap_ns = np.zeros(
        len(normalized),
        dtype="int64",
    )

    if len(normalized) > 1:
        inter_window_gap_ns[1:] = (
            starts_ns[1:] - ends_ns[:-1]
        )

    require(
        np.all(inter_window_gap_ns >= 0),
        "Registered contract windows overlap.",
    )

    session_break_flag = np.zeros(
        len(normalized),
        dtype=bool,
    )
    session_break_flag[0] = True

    if len(normalized) > 1:
        session_break_flag[1:] = (
            inter_window_gap_ns[1:] > 0
        )

    normalized["gap_from_previous_window_ns"] = (
        inter_window_gap_ns
    )
    normalized["session_break_flag"] = (
        session_break_flag
    )
    normalized["session_id"] = np.cumsum(
        session_break_flag
    ).astype("int64")

    normalized["contract_start_utc"] = (
        pd.to_datetime(
            starts_ns,
            unit="ns",
            utc=True,
        )
    )
    normalized["contract_end_exclusive_utc"] = (
        pd.to_datetime(
            ends_ns,
            unit="ns",
            utc=True,
        )
    )

    return normalized


CONTRACT_WINDOW_CONTINUITY = normalize_contract_windows(
    OBSERVATION_WINDOW_CONTRACT
)

REGISTERED_SESSION_COUNT = int(
    CONTRACT_WINDOW_CONTINUITY[
        "session_id"
    ].nunique()
)

require(
    REGISTERED_SESSION_COUNT == 1,
    (
        "The frozen source run is expected to contain one "
        f"continuous registered session; observed "
        f"{REGISTERED_SESSION_COUNT}."
    ),
)

require(
    CONTRACT_WINDOW_CONTINUITY[
        "gap_from_previous_window_ns"
    ].eq(0).all(),
    (
        "The registered partitions are expected to form one "
        "exactly contiguous sequence of half-open windows."
    ),
)


# ------------------------------------------------------------
# Canonical analytical batch calendar
# ------------------------------------------------------------

ANALYTICAL_BATCH_CALENDAR = (
    NOTEBOOK_04_CANONICAL_BATCH_IDENTITY.copy()
)

ANALYTICAL_BATCH_CALENDAR[
    "event_time_ns"
] = parse_exact_int64(
    ANALYTICAL_BATCH_CALENDAR[
        "event_time_ns"
    ],
    label="canonical_batch_calendar.event_time_ns",
)

ANALYTICAL_BATCH_CALENDAR[
    "event_time_utc"
] = pd.to_datetime(
    ANALYTICAL_BATCH_CALENDAR[
        "event_time_ns"
    ].to_numpy(dtype="int64"),
    unit="ns",
    utc=True,
)

ANALYTICAL_BATCH_CALENDAR[
    "utc_date"
] = (
    ANALYTICAL_BATCH_CALENDAR[
        "event_time_utc"
    ]
    .dt.strftime("%Y-%m-%d")
    .astype("string")
)

ANALYTICAL_BATCH_CALENDAR[
    "utc_weekday"
] = (
    ANALYTICAL_BATCH_CALENDAR[
        "event_time_utc"
    ]
    .dt.day_name()
    .astype("string")
)

ANALYTICAL_BATCH_CALENDAR[
    "utc_hour"
] = (
    ANALYTICAL_BATCH_CALENDAR[
        "event_time_utc"
    ]
    .dt.hour
    .astype("int16")
)

ANALYTICAL_BATCH_CALENDAR[
    "second_of_day"
] = (
    ANALYTICAL_BATCH_CALENDAR[
        "event_time_utc"
    ].dt.hour.astype("int64")
    * 3_600
    + ANALYTICAL_BATCH_CALENDAR[
        "event_time_utc"
    ].dt.minute.astype("int64")
    * 60
    + ANALYTICAL_BATCH_CALENDAR[
        "event_time_utc"
    ].dt.second.astype("int64")
)

ANALYTICAL_UNIQUE_UTC_DATES = int(
    ANALYTICAL_BATCH_CALENDAR[
        "utc_date"
    ].nunique()
)

ANALYTICAL_UNIQUE_WEEKDAYS = int(
    ANALYTICAL_BATCH_CALENDAR[
        "utc_weekday"
    ].nunique()
)

ANALYTICAL_UNIQUE_UTC_HOURS = int(
    ANALYTICAL_BATCH_CALENDAR[
        "utc_hour"
    ].nunique()
)

require(
    ANALYTICAL_UNIQUE_UTC_DATES == 1,
    (
        "The current run is expected to cover one UTC calendar "
        f"date; observed {ANALYTICAL_UNIQUE_UTC_DATES}."
    ),
)


# ------------------------------------------------------------
# Verify that batch times remain within their own partitions
# ------------------------------------------------------------

partition_window_lookup = (
    CONTRACT_WINDOW_CONTINUITY.set_index(
        "event_partition"
    )
)

for partition_name in ANALYTICAL_PARTITIONS:
    partition_batches = (
        ANALYTICAL_BATCH_CALENDAR.loc[
            ANALYTICAL_BATCH_CALENDAR[
                "event_partition"
            ].eq(partition_name)
        ]
    )

    require(
        not partition_batches.empty,
        f"{partition_name} contains no analytical batches.",
    )

    contract_start_ns = int(
        partition_window_lookup.loc[
            partition_name,
            "contract_start_ns",
        ]
    )
    contract_end_exclusive_ns = int(
        partition_window_lookup.loc[
            partition_name,
            "contract_end_exclusive_ns",
        ]
    )

    partition_times_ns = (
        partition_batches[
            "event_time_ns"
        ].to_numpy(dtype="int64")
    )

    require(
        np.all(
            partition_times_ns
            >= contract_start_ns
        ),
        (
            f"{partition_name} contains a batch before its "
            "registered contract start."
        ),
    )
    require(
        np.all(
            partition_times_ns
            < contract_end_exclusive_ns
        ),
        (
            f"{partition_name} contains a batch at or beyond "
            "its registered end-exclusive boundary."
        ),
    )


# ------------------------------------------------------------
# Time-of-day repeatability audit
# ------------------------------------------------------------

time_of_day_repeatability_rows: list[
    dict[str, Any]
] = []

for width_minutes in (
    TIME_OF_DAY_REPEATABILITY_WIDTHS_MINUTES
):
    width_seconds = width_minutes * 60

    repeatability_frame = (
        ANALYTICAL_BATCH_CALENDAR[
            [
                "utc_date",
                "second_of_day",
            ]
        ]
        .copy()
    )

    repeatability_frame[
        "clock_bin_index"
    ] = (
        repeatability_frame[
            "second_of_day"
        ]
        // width_seconds
    ).astype("int64")

    distinct_dates_per_clock_bin = (
        repeatability_frame.groupby(
            "clock_bin_index",
            observed=True,
            sort=True,
        )["utc_date"]
        .nunique()
    )

    maximum_distinct_dates = int(
        distinct_dates_per_clock_bin.max()
    )

    repeated_clock_bin_count = int(
        distinct_dates_per_clock_bin.gt(1).sum()
    )

    time_of_day_repeatability_rows.append(
        {
            "clock_bin_width_minutes": (
                width_minutes
            ),
            "populated_clock_bin_count": int(
                len(distinct_dates_per_clock_bin)
            ),
            "maximum_distinct_dates_per_clock_bin": (
                maximum_distinct_dates
            ),
            "repeated_clock_bin_count": (
                repeated_clock_bin_count
            ),
            "minimum_dates_required_for_seasonality": 2,
            "seasonality_estimable": (
                maximum_distinct_dates >= 2
            ),
            "status": (
                "NOT_APPLICABLE_IN_CURRENT_DATA"
                if maximum_distinct_dates < 2
                else "APPLICABLE"
            ),
        }
    )


TIME_OF_DAY_REPEATABILITY_AUDIT = pd.DataFrame(
    time_of_day_repeatability_rows
)

require(
    not TIME_OF_DAY_REPEATABILITY_AUDIT[
        "seasonality_estimable"
    ].any(),
    (
        "Time-of-day seasonality was unexpectedly classified "
        "as estimable."
    ),
)


# ------------------------------------------------------------
# Analytical partition coverage and rates
# ------------------------------------------------------------

partition_coverage_rows: list[dict[str, Any]] = []

for partition_name in ANALYTICAL_PARTITIONS:
    partition_batches = (
        PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
            PRIMARY_SCORING_BATCHES_ANALYTICAL[
                "event_partition"
            ]
            .astype("string")
            .str.upper()
            .eq(partition_name)
        ]
        .copy()
    )

    contract_row = partition_window_lookup.loc[
        partition_name
    ]

    contract_duration_ns = int(
        contract_row["contract_duration_ns"]
    )
    contract_duration_seconds = (
        contract_duration_ns
        / NANOSECONDS_PER_SECOND
    )

    batch_count = int(
        len(partition_batches)
    )
    event_count = int(
        partition_batches[
            "batch_event_count"
        ].sum()
    )
    buy_event_count = int(
        partition_batches[
            "buy_event_count"
        ].sum()
    )
    sell_event_count = int(
        partition_batches[
            "sell_event_count"
        ].sum()
    )

    require(
        event_count
        == int(contract_row["primary_event_count"]),
        (
            f"{partition_name} event count differs from its "
            "registered observation-window metadata."
        ),
    )
    require(
        batch_count
        == int(contract_row["primary_batch_count"]),
        (
            f"{partition_name} batch count differs from its "
            "registered observation-window metadata."
        ),
    )
    require(
        buy_event_count + sell_event_count
        == event_count,
        (
            f"{partition_name} BUY and SELL event counts do not "
            "conserve total events."
        ),
    )

    partition_coverage_rows.append(
        {
            "event_partition": partition_name,
            "contract_start_utc": (
                contract_row[
                    "contract_start_utc"
                ]
            ),
            "contract_end_exclusive_utc": (
                contract_row[
                    "contract_end_exclusive_utc"
                ]
            ),
            "contract_duration_seconds": (
                contract_duration_seconds
            ),
            "left_censoring_duration_seconds": (
                int(
                    contract_row[
                        "left_censoring_duration_ns"
                    ]
                )
                / NANOSECONDS_PER_SECOND
            ),
            "formal_exposure_rule": (
                "FULL_REGISTERED_CONTRACT_INTERVAL"
            ),
            "batch_count": batch_count,
            "event_count": event_count,
            "buy_event_count": buy_event_count,
            "sell_event_count": sell_event_count,
            "pooled_event_rate_per_second": (
                event_count
                / contract_duration_seconds
            ),
            "buy_event_rate_per_second": (
                buy_event_count
                / contract_duration_seconds
            ),
            "sell_event_rate_per_second": (
                sell_event_count
                / contract_duration_seconds
            ),
            "buy_event_share": (
                buy_event_count / event_count
            ),
            "status": "PASS",
        }
    )


ANALYTICAL_PARTITION_COVERAGE = pd.DataFrame(
    partition_coverage_rows
)

ANALYTICAL_TOTAL_CONTRACT_EXPOSURE_SECONDS = float(
    ANALYTICAL_PARTITION_COVERAGE[
        "contract_duration_seconds"
    ].sum()
)

REGISTERED_FULL_RUN_DURATION_SECONDS = float(
    CONTRACT_WINDOW_CONTINUITY[
        "contract_duration_ns"
    ].sum()
    / NANOSECONDS_PER_SECOND
)


# ------------------------------------------------------------
# Count-grid compatibility
# ------------------------------------------------------------

EMPIRICAL_TIMESTAMP_QUANTUM_NS = int(
    np.gcd.reduce(
        EVENT_CLOCK_AND_TIE_AUDIT[
            "empirical_positive_gap_gcd_ns"
        ].to_numpy(dtype="int64")
    )
)

require(
    EMPIRICAL_TIMESTAMP_QUANTUM_NS > 0,
    "Empirical timestamp quantum must be positive.",
)
require(
    MODEL_COUNT_GRID_NS
    % EMPIRICAL_TIMESTAMP_QUANTUM_NS
    == 0,
    (
        "The registered one-millisecond model grid is not an "
        "integer multiple of the empirical timestamp quantum."
    ),
)

COUNT_GRID_CONTRACT = pd.DataFrame(
    [
        {
            "field": "timestamp_authority",
            "value": (
                "NOTEBOOK_04_PRIMARY_SCORING_BATCHES"
            ),
        },
        {
            "field": "empirical_timestamp_quantum_ns",
            "value": EMPIRICAL_TIMESTAMP_QUANTUM_NS,
        },
        {
            "field": "model_count_grid_ns",
            "value": MODEL_COUNT_GRID_NS,
        },
        {
            "field": "model_count_grid_ms",
            "value": MODEL_COUNT_GRID_MS,
        },
        {
            "field": "grid_to_timestamp_quantum_ratio",
            "value": (
                MODEL_COUNT_GRID_NS
                // EMPIRICAL_TIMESTAMP_QUANTUM_NS
            ),
        },
        {
            "field": "aggregation_semantics",
            "value": (
                "HALF_OPEN_CELLS_WITH_NO_TIMESTAMP_MUTATION"
            ),
        },
        {
            "field": "event_representation",
            "value": PRIMARY_EVENT_REPRESENTATION,
        },
        {
            "field": "timestamp_jitter_applied",
            "value": False,
        },
    ]
)


# ------------------------------------------------------------
# Calendar-baseline applicability
# ------------------------------------------------------------

CALENDAR_BASELINE_APPLICABILITY = pd.DataFrame(
    [
        {
            "model_family": (
                "DETERMINISTIC_ELAPSED_TIME_POISSON"
            ),
            "authorization": "APPLICABLE",
            "formal_model_allowed": True,
            "reason": (
                "Elapsed time varies within the frozen run and "
                "does not require repeated calendar locations."
            ),
        },
        {
            "model_family": (
                "COARSE_60_SECOND_RATE_PROFILE"
            ),
            "authorization": "DIAGNOSTIC_ONLY",
            "formal_model_allowed": False,
            "reason": (
                "Useful for chronological stability inspection "
                "but too flexible to serve as a formal seasonal "
                "predictive model."
            ),
        },
        {
            "model_family": "TIME_OF_DAY_POISSON",
            "authorization": (
                "NOT_APPLICABLE_IN_CURRENT_DATA"
            ),
            "formal_model_allowed": False,
            "reason": (
                f"Only {ANALYTICAL_UNIQUE_UTC_DATES} UTC date "
                "is observed; no clock-time bin is repeated "
                "across dates."
            ),
        },
        {
            "model_family": "DAY_OF_WEEK_POISSON",
            "authorization": (
                "NOT_APPLICABLE_IN_CURRENT_DATA"
            ),
            "formal_model_allowed": False,
            "reason": (
                f"Only {ANALYTICAL_UNIQUE_WEEKDAYS} weekday "
                "is represented."
            ),
        },
        {
            "model_family": (
                "CROSS_DAY_SEASONAL_POISSON"
            ),
            "authorization": (
                "NOT_APPLICABLE_IN_CURRENT_DATA"
            ),
            "formal_model_allowed": False,
            "reason": (
                "The source run contains one continuous session "
                "on one UTC calendar date."
            ),
        },
        {
            "model_family": (
                "SESSION_FIXED_EFFECT_POISSON"
            ),
            "authorization": (
                "NOT_APPLICABLE_IN_CURRENT_DATA"
            ),
            "formal_model_allowed": False,
            "reason": (
                f"Only {REGISTERED_SESSION_COUNT} registered "
                "continuous session is available."
            ),
        },
    ]
)


# ------------------------------------------------------------
# Coverage and applicability gate frame
# ------------------------------------------------------------

CALENDAR_AND_COVERAGE_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "registered_windows_half_open",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": "all intervals use [start, end)",
        },
        {
            "gate": "registered_windows_nonoverlapping",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": "all inter-window gaps are nonnegative",
        },
        {
            "gate": "single_continuous_registered_session",
            "severity": "BLOCKING",
            "passed": (
                REGISTERED_SESSION_COUNT == 1
            ),
            "evidence": (
                f"session_count="
                f"{REGISTERED_SESSION_COUNT}"
            ),
        },
        {
            "gate": "analytical_batch_times_within_partition",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "all DEVELOPMENT and CALIBRATION batches are "
                "inside their own registered windows"
            ),
        },
        {
            "gate": "formal_exposure_uses_contract_interval",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                f"analytical_exposure_seconds="
                f"{ANALYTICAL_TOTAL_CONTRACT_EXPOSURE_SECONDS:.9f}"
            ),
        },
        {
            "gate": "one_millisecond_grid_exactly_representable",
            "severity": "BLOCKING",
            "passed": (
                MODEL_COUNT_GRID_NS
                % EMPIRICAL_TIMESTAMP_QUANTUM_NS
                == 0
            ),
            "evidence": (
                f"timestamp_quantum_ns="
                f"{EMPIRICAL_TIMESTAMP_QUANTUM_NS}; "
                f"count_grid_ns={MODEL_COUNT_GRID_NS}"
            ),
        },
        {
            "gate": "time_of_day_model_not_fabricated",
            "severity": "BLOCKING",
            "passed": (
                not FORMAL_TIME_OF_DAY_BASELINE_AUTHORIZED
            ),
            "evidence": (
                f"unique_utc_dates="
                f"{ANALYTICAL_UNIQUE_UTC_DATES}"
            ),
        },
        {
            "gate": "day_of_week_model_not_fabricated",
            "severity": "BLOCKING",
            "passed": (
                not FORMAL_DAY_OF_WEEK_BASELINE_AUTHORIZED
            ),
            "evidence": (
                f"unique_weekdays="
                f"{ANALYTICAL_UNIQUE_WEEKDAYS}"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    CALENDAR_AND_COVERAGE_GATE_FRAME.loc[
        CALENDAR_AND_COVERAGE_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    "At least one calendar or coverage gate failed.",
)


display(
    CONTRACT_WINDOW_CONTINUITY[
        [
            "partition_order",
            "event_partition",
            "contract_start_utc",
            "contract_end_exclusive_utc",
            "contract_duration_ns",
            "gap_from_previous_window_ns",
            "session_id",
        ]
    ]
)

display(ANALYTICAL_PARTITION_COVERAGE)
display(TIME_OF_DAY_REPEATABILITY_AUDIT)
display(COUNT_GRID_CONTRACT)
display(CALENDAR_BASELINE_APPLICABILITY)
display(CALENDAR_AND_COVERAGE_GATE_FRAME)

print(
    "Calendar and coverage audit passed."
)
print(
    f"The source contract contains "
    f"{REGISTERED_SESSION_COUNT} continuous session spanning "
    f"{REGISTERED_FULL_RUN_DURATION_SECONDS:.6f} seconds."
)
print(
    f"Notebook 06 analytical exposure is "
    f"{ANALYTICAL_TOTAL_CONTRACT_EXPOSURE_SECONDS:.6f} seconds "
    "across DEVELOPMENT and CALIBRATION."
)
print(
    "The formal count-likelihood grid is one millisecond, "
    "constructed by exact integer binning without timestamp "
    "jitter or event reordering."
)
print(
    "Elapsed-time rate models are applicable. Time-of-day, "
    "day-of-week, cross-day seasonal, and session-effect models "
    "are not applicable to the current single-session dataset."
)

,partition_order,event_partition,contract_start_utc,contract_end_exclusive_utc,contract_duration_ns,gap_from_previous_window_ns,session_id
0,1,DEVELOPMENT,2026-07-10 06:37:47.531985400+00:00,2026-07-10 07:07:49.391572100+00:00,1801859586700,0,1
1,2,CALIBRATION,2026-07-10 07:07:49.391572100+00:00,2026-07-10 07:19:49.690751200+00:00,720299179100,0,1
2,3,VALIDATION,2026-07-10 07:19:49.690751200+00:00,2026-07-10 07:28:45.057534700+00:00,535366783500,0,1
3,4,ENGINEERING_HOLDOUT,2026-07-10 07:28:45.057534700+00:00,2026-07-10 07:37:46.749750801+00:00,541692216101,0,1


,event_partition,contract_start_utc,contract_end_exclusive_utc,contract_duration_seconds,left_censoring_duration_seconds,formal_exposure_rule,batch_count,event_count,buy_event_count,sell_event_count,pooled_event_rate_per_second,buy_event_rate_per_second,sell_event_rate_per_second,buy_event_share,status
0,DEVELOPMENT,2026-07-10 06:37:47.531985400+00:00,2026-07-10 07:07:49.391572100+00:00,"1,801.8596",1.2349662,FULL_REGISTERED_CONTRACT_INTERVAL,6859,7004,3414,3590,3.8870953,1.8947092,1.9923861,0.48743575,PASS
1,CALIBRATION,2026-07-10 07:07:49.391572100+00:00,2026-07-10 07:19:49.690751200+00:00,720.29918,1.1554099,FULL_REGISTERED_CONTRACT_INTERVAL,2400,2493,1230,1263,3.4610618,1.7076238,1.7534381,0.49338147,PASS


,clock_bin_width_minutes,populated_clock_bin_count,maximum_distinct_dates_per_clock_bin,repeated_clock_bin_count,minimum_dates_required_for_seasonality,seasonality_estimable,status
0,1,43,1,0,2,False,NOT_APPLICABLE_IN_CURRENT_DATA
1,5,9,1,0,2,False,NOT_APPLICABLE_IN_CURRENT_DATA
2,15,4,1,0,2,False,NOT_APPLICABLE_IN_CURRENT_DATA
3,60,2,1,0,2,False,NOT_APPLICABLE_IN_CURRENT_DATA


,field,value
0,timestamp_authority,NOTEBOOK_04_PRIMARY_SCORING_BATCHES
1,empirical_timestamp_quantum_ns,100
2,model_count_grid_ns,1000000
3,model_count_grid_ms,1
4,grid_to_timestamp_quantum_ratio,10000
5,aggregation_semantics,HALF_OPEN_CELLS_WITH_NO_TIMESTAMP_MUTATION
6,event_representation,SAME_MS_SAME_SIDE_BURSTS
7,timestamp_jitter_applied,False


,model_family,authorization,formal_model_allowed,reason
0,DETERMINISTIC_ELAPSED_TIME_POISSON,APPLICABLE,True,Elapsed time varies within the frozen run and ...
1,COARSE_60_SECOND_RATE_PROFILE,DIAGNOSTIC_ONLY,False,Useful for chronological stability inspection ...
2,TIME_OF_DAY_POISSON,NOT_APPLICABLE_IN_CURRENT_DATA,False,Only 1 UTC date is observed; no clock-time bin...
3,DAY_OF_WEEK_POISSON,NOT_APPLICABLE_IN_CURRENT_DATA,False,Only 1 weekday is represented.
4,CROSS_DAY_SEASONAL_POISSON,NOT_APPLICABLE_IN_CURRENT_DATA,False,The source run contains one continuous session...
5,SESSION_FIXED_EFFECT_POISSON,NOT_APPLICABLE_IN_CURRENT_DATA,False,Only 1 registered continuous session is availa...


,gate,severity,passed,evidence
0,registered_windows_half_open,BLOCKING,True,"all intervals use [start, end)"
1,registered_windows_nonoverlapping,BLOCKING,True,all inter-window gaps are nonnegative
2,single_continuous_registered_session,BLOCKING,True,session_count=1
3,analytical_batch_times_within_partition,BLOCKING,True,all DEVELOPMENT and CALIBRATION batches are in...
4,formal_exposure_uses_contract_interval,BLOCKING,True,analytical_exposure_seconds=2522.158765800
5,one_millisecond_grid_exactly_representable,BLOCKING,True,timestamp_quantum_ns=100; count_grid_ns=1000000
6,time_of_day_model_not_fabricated,BLOCKING,True,unique_utc_dates=1
7,day_of_week_model_not_fabricated,BLOCKING,True,unique_weekdays=1
8,protected_partition_content_remains_absent,BLOCKING,True,VALIDATION=False; ENGINEERING_HOLDOUT=False


Calendar and coverage audit passed.
The source contract contains 1 continuous session spanning 3599.217765 seconds.
Notebook 06 analytical exposure is 2522.158766 seconds across DEVELOPMENT and CALIBRATION.
The formal count-likelihood grid is one millisecond, constructed by exact integer binning without timestamp jitter or event reordering.
Elapsed-time rate models are applicable. Time-of-day, day-of-week, cross-day seasonal, and session-effect models are not applicable to the current single-session dataset.


In [8]:
# ============================================================
# Descriptive arrival-rate, duration, and burst audit
# ============================================================

DURATION_PROCESS_NAMES: Final[tuple[str, ...]] = (
    "POOLED_BATCH",
    "BUY_BATCH",
    "SELL_BATCH",
)

SHORT_GAP_THRESHOLDS_MS: Final[tuple[int, ...]] = (
    10,
    25,
    50,
    100,
)

BURST_COUNT_WINDOWS_MS: Final[tuple[int, ...]] = (
    100,
    1_000,
    5_000,
)

BUSIEST_WINDOW_SHARES: Final[tuple[float, ...]] = (
    0.01,
    0.05,
    0.10,
)

DURATION_BOOTSTRAP_REPLICATES: Final[int] = (
    N_MONTE_CARLO_SIMULATIONS
)
DURATION_BOOTSTRAP_CHUNK_SIZE: Final[int] = 128


# ------------------------------------------------------------
# General numerical helpers
# ------------------------------------------------------------

def safe_sample_standard_deviation(
    values: np.ndarray,
) -> float:
    """Return sample standard deviation or NaN when undefined."""
    values = np.asarray(values, dtype="float64")

    if values.size < 2:
        return float("nan")

    return float(np.std(values, ddof=1))


def safe_coefficient_of_variation(
    values: np.ndarray,
) -> float:
    """Return sample standard deviation divided by the mean."""
    values = np.asarray(values, dtype="float64")

    if values.size < 2:
        return float("nan")

    mean_value = float(np.mean(values))

    if not np.isfinite(mean_value) or mean_value <= 0.0:
        return float("nan")

    return safe_sample_standard_deviation(values) / mean_value


def burstiness_index(
    values: np.ndarray,
) -> float:
    """
    Return the duration burstiness index (sigma - mean)/(sigma + mean).

    A value near zero is consistent with exponential-like dispersion,
    positive values indicate more variable durations, and negative
    values indicate more regular arrivals.
    """
    values = np.asarray(values, dtype="float64")

    if values.size < 2:
        return float("nan")

    mean_value = float(np.mean(values))
    standard_deviation = safe_sample_standard_deviation(
        values
    )
    denominator = mean_value + standard_deviation

    if not np.isfinite(denominator) or denominator <= 0.0:
        return float("nan")

    return (
        standard_deviation - mean_value
    ) / denominator


def exponential_ks_distance(
    durations_seconds: np.ndarray,
) -> float:
    """
    Return the one-sample KS distance from a fitted exponential law.

    The exponential scale is estimated from the same sample.
    """
    durations = np.asarray(
        durations_seconds,
        dtype="float64",
    )

    require(
        durations.ndim == 1,
        "Duration array must be one-dimensional.",
    )
    require(
        durations.size >= 2,
        "At least two durations are required for a KS distance.",
    )
    require(
        np.isfinite(durations).all(),
        "Duration array contains nonfinite values.",
    )
    require(
        np.all(durations > 0.0),
        "Duration array contains nonpositive values.",
    )

    fitted_scale = float(np.mean(durations))

    require(
        fitted_scale > 0.0,
        "Fitted exponential scale must be positive.",
    )

    sorted_durations = np.sort(durations)
    fitted_cdf = -np.expm1(
        -sorted_durations / fitted_scale
    )

    sample_size = sorted_durations.size

    empirical_upper = (
        np.arange(
            1,
            sample_size + 1,
            dtype="float64",
        )
        / sample_size
    )
    empirical_lower = (
        np.arange(
            0,
            sample_size,
            dtype="float64",
        )
        / sample_size
    )

    d_plus = np.max(
        empirical_upper - fitted_cdf
    )
    d_minus = np.max(
        fitted_cdf - empirical_lower
    )

    return float(max(d_plus, d_minus))


def bootstrap_fitted_exponential_ks(
    durations_seconds: np.ndarray,
    *,
    replicates: int,
    seed_sequence: Sequence[int],
    chunk_size: int = DURATION_BOOTSTRAP_CHUNK_SIZE,
) -> dict[str, float]:
    """
    Parametrically bootstrap the KS statistic after re-estimating scale.

    Every simulated exponential sample is normalized by its own sample
    mean before its KS statistic is calculated. This reproduces the
    parameter-estimation step used for the observed sample.
    """
    durations = np.asarray(
        durations_seconds,
        dtype="float64",
    )

    require(
        durations.ndim == 1,
        "Bootstrap duration input must be one-dimensional.",
    )
    require(
        durations.size >= 2,
        "At least two durations are required for KS bootstrap.",
    )
    require(
        replicates >= 1,
        "Bootstrap replicate count must be positive.",
    )
    require(
        chunk_size >= 1,
        "Bootstrap chunk size must be positive.",
    )

    observed_distance = exponential_ks_distance(
        durations
    )
    sample_size = durations.size

    bootstrap_rng = np.random.default_rng(
        np.random.SeedSequence(seed_sequence)
    )

    upper_empirical = (
        np.arange(
            1,
            sample_size + 1,
            dtype="float64",
        )
        / sample_size
    )
    lower_empirical = (
        np.arange(
            0,
            sample_size,
            dtype="float64",
        )
        / sample_size
    )

    bootstrap_distances = np.empty(
        replicates,
        dtype="float64",
    )

    completed = 0

    while completed < replicates:
        current_chunk_size = min(
            chunk_size,
            replicates - completed,
        )

        simulated = bootstrap_rng.exponential(
            scale=1.0,
            size=(
                current_chunk_size,
                sample_size,
            ),
        )

        simulated_scale = simulated.mean(
            axis=1,
            keepdims=True,
        )

        simulated /= simulated_scale
        simulated.sort(axis=1)

        fitted_cdf = -np.expm1(-simulated)

        d_plus = np.max(
            upper_empirical[None, :]
            - fitted_cdf,
            axis=1,
        )
        d_minus = np.max(
            fitted_cdf
            - lower_empirical[None, :],
            axis=1,
        )

        bootstrap_distances[
            completed:
            completed + current_chunk_size
        ] = np.maximum(
            d_plus,
            d_minus,
        )

        completed += current_chunk_size

    bootstrap_quantiles = np.quantile(
        bootstrap_distances,
        [0.025, 0.50, 0.95, 0.975],
    )

    bootstrap_p_value = (
        1.0
        + float(
            np.count_nonzero(
                bootstrap_distances
                >= observed_distance
            )
        )
    ) / (replicates + 1.0)

    return {
        "observed_ks_distance": (
            observed_distance
        ),
        "bootstrap_ks_q025": float(
            bootstrap_quantiles[0]
        ),
        "bootstrap_ks_median": float(
            bootstrap_quantiles[1]
        ),
        "bootstrap_ks_q950": float(
            bootstrap_quantiles[2]
        ),
        "bootstrap_ks_q975": float(
            bootstrap_quantiles[3]
        ),
        "bootstrap_p_value": float(
            bootstrap_p_value
        ),
    }


# ------------------------------------------------------------
# Canonical arrival-time interfaces
# ------------------------------------------------------------

def partition_process_times(
    batches: pd.DataFrame,
    *,
    partition_name: str,
    process_name: str,
) -> np.ndarray:
    """Return canonical exact-time arrivals for one process."""
    partition_frame = (
        batches.loc[
            batches[
                "event_partition"
            ]
            .astype("string")
            .str.upper()
            .eq(partition_name)
        ]
        .copy()
        .sort_values(
            "partition_batch_index",
            kind="stable",
        )
    )

    require(
        not partition_frame.empty,
        (
            f"No scoring batches are available for "
            f"{partition_name}."
        ),
    )

    if process_name == "POOLED_BATCH":
        process_frame = partition_frame
    elif process_name == "BUY_BATCH":
        process_frame = partition_frame.loc[
            partition_frame[
                "buy_event_count"
            ].astype("int64").gt(0)
        ]
    elif process_name == "SELL_BATCH":
        process_frame = partition_frame.loc[
            partition_frame[
                "sell_event_count"
            ].astype("int64").gt(0)
        ]
    else:
        raise ValueError(
            f"Unsupported duration process: {process_name}"
        )

    require(
        not process_frame.empty,
        (
            f"No {process_name} arrivals are available in "
            f"{partition_name}."
        ),
    )

    arrival_times_ns = parse_exact_int64(
        process_frame["event_time_ns"],
        label=(
            f"{partition_name}.{process_name}."
            "event_time_ns"
        ),
    ).to_numpy(dtype="int64")

    require(
        np.all(np.diff(arrival_times_ns) > 0),
        (
            f"{partition_name} {process_name} arrival times "
            "are not strictly increasing."
        ),
    )

    return arrival_times_ns


interarrival_duration_frames: list[pd.DataFrame] = []
interarrival_censoring_rows: list[
    dict[str, Any]
] = []
interarrival_diagnostic_rows: list[
    dict[str, Any]
] = []

duration_process_seed_code: Final[
    Mapping[str, int]
] = {
    "POOLED_BATCH": 1,
    "BUY_BATCH": 2,
    "SELL_BATCH": 3,
}

partition_seed_code: Final[
    Mapping[str, int]
] = {
    "DEVELOPMENT": 1,
    "CALIBRATION": 2,
}


for partition_name in ANALYTICAL_PARTITIONS:
    contract_row = partition_window_lookup.loc[
        partition_name
    ]

    contract_start_ns = int(
        contract_row["contract_start_ns"]
    )
    contract_end_exclusive_ns = int(
        contract_row["contract_end_exclusive_ns"]
    )

    for process_name in DURATION_PROCESS_NAMES:
        arrival_times_ns = partition_process_times(
            PRIMARY_SCORING_BATCHES_ANALYTICAL,
            partition_name=partition_name,
            process_name=process_name,
        )

        complete_durations_ns = np.diff(
            arrival_times_ns
        ).astype("int64")

        require(
            complete_durations_ns.size
            == arrival_times_ns.size - 1,
            "Complete-duration count is inconsistent.",
        )
        require(
            np.all(complete_durations_ns > 0),
            (
                f"{partition_name} {process_name} contains "
                "a nonpositive complete interarrival duration."
            ),
        )

        complete_durations_seconds = (
            complete_durations_ns.astype("float64")
            / NANOSECONDS_PER_SECOND
        )

        previous_times_ns = arrival_times_ns[:-1]
        current_times_ns = arrival_times_ns[1:]

        interarrival_duration_frames.append(
            pd.DataFrame(
                {
                    "event_partition": partition_name,
                    "process_name": process_name,
                    "duration_index": np.arange(
                        1,
                        complete_durations_ns.size + 1,
                        dtype="int64",
                    ),
                    "previous_arrival_time_ns": (
                        previous_times_ns
                    ),
                    "arrival_time_ns": (
                        current_times_ns
                    ),
                    "duration_ns": (
                        complete_durations_ns
                    ),
                    "duration_seconds": (
                        complete_durations_seconds
                    ),
                    "duration_status": (
                        "COMPLETE_WITHIN_PARTITION"
                    ),
                    "cross_partition_flag": False,
                }
            )
        )

        left_censored_duration_ns = int(
            arrival_times_ns[0]
            - contract_start_ns
        )
        right_censored_duration_ns = int(
            contract_end_exclusive_ns
            - arrival_times_ns[-1]
        )

        require(
            left_censored_duration_ns >= 0,
            (
                f"{partition_name} {process_name} first arrival "
                "precedes the contract start."
            ),
        )
        require(
            right_censored_duration_ns > 0,
            (
                f"{partition_name} {process_name} final arrival "
                "is not strictly before the end-exclusive boundary."
            ),
        )

        interarrival_censoring_rows.append(
            {
                "event_partition": partition_name,
                "process_name": process_name,
                "arrival_count": int(
                    arrival_times_ns.size
                ),
                "complete_duration_count": int(
                    complete_durations_ns.size
                ),
                "left_censored_duration_ns": (
                    left_censored_duration_ns
                ),
                "left_censored_duration_seconds": (
                    left_censored_duration_ns
                    / NANOSECONDS_PER_SECOND
                ),
                "right_censored_duration_ns": (
                    right_censored_duration_ns
                ),
                "right_censored_duration_seconds": (
                    right_censored_duration_ns
                    / NANOSECONDS_PER_SECOND
                ),
                "complete_duration_cross_partition_count": 0,
                "status": "PASS",
            }
        )

        exponential_bootstrap = (
            bootstrap_fitted_exponential_ks(
                complete_durations_seconds,
                replicates=(
                    DURATION_BOOTSTRAP_REPLICATES
                ),
                seed_sequence=(
                    RANDOM_SEED,
                    6,
                    partition_seed_code[
                        partition_name
                    ],
                    duration_process_seed_code[
                        process_name
                    ],
                ),
            )
        )

        duration_quantiles = np.quantile(
            complete_durations_seconds,
            [
                0.01,
                0.05,
                0.25,
                0.50,
                0.75,
                0.95,
                0.99,
            ],
        )

        duration_mean = float(
            np.mean(
                complete_durations_seconds
            )
        )
        duration_standard_deviation = (
            safe_sample_standard_deviation(
                complete_durations_seconds
            )
        )

        interarrival_diagnostic_rows.append(
            {
                "event_partition": partition_name,
                "process_name": process_name,
                "arrival_count": int(
                    arrival_times_ns.size
                ),
                "complete_duration_count": int(
                    complete_durations_seconds.size
                ),
                "zero_duration_count": int(
                    np.count_nonzero(
                        complete_durations_ns == 0
                    )
                ),
                "minimum_duration_seconds": float(
                    np.min(
                        complete_durations_seconds
                    )
                ),
                "q01_duration_seconds": float(
                    duration_quantiles[0]
                ),
                "q05_duration_seconds": float(
                    duration_quantiles[1]
                ),
                "q25_duration_seconds": float(
                    duration_quantiles[2]
                ),
                "median_duration_seconds": float(
                    duration_quantiles[3]
                ),
                "q75_duration_seconds": float(
                    duration_quantiles[4]
                ),
                "q95_duration_seconds": float(
                    duration_quantiles[5]
                ),
                "q99_duration_seconds": float(
                    duration_quantiles[6]
                ),
                "maximum_duration_seconds": float(
                    np.max(
                        complete_durations_seconds
                    )
                ),
                "mean_duration_seconds": (
                    duration_mean
                ),
                "sample_standard_deviation_seconds": (
                    duration_standard_deviation
                ),
                "coefficient_of_variation": (
                    duration_standard_deviation
                    / duration_mean
                ),
                "burstiness_index": burstiness_index(
                    complete_durations_seconds
                ),
                "fitted_exponential_rate_per_second": (
                    1.0 / duration_mean
                ),
                **exponential_bootstrap,
                "exponential_rejected_at_5pct": (
                    exponential_bootstrap[
                        "observed_ks_distance"
                    ]
                    > exponential_bootstrap[
                        "bootstrap_ks_q950"
                    ]
                ),
                "bootstrap_replicates": (
                    DURATION_BOOTSTRAP_REPLICATES
                ),
                "status": "PASS",
            }
        )


INTERARRIVAL_DURATION_TABLE = (
    pd.concat(
        interarrival_duration_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "event_partition",
            "process_name",
            "duration_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

INTERARRIVAL_CENSORING_AUDIT = pd.DataFrame(
    interarrival_censoring_rows
).sort_values(
    [
        "event_partition",
        "process_name",
    ],
    kind="stable",
).reset_index(drop=True)

INTERARRIVAL_DIAGNOSTICS = pd.DataFrame(
    interarrival_diagnostic_rows
).sort_values(
    [
        "event_partition",
        "process_name",
    ],
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# Contract-anchored chronological rate blocks
# ------------------------------------------------------------

def build_contract_rate_blocks(
    batches: pd.DataFrame,
    contract_windows: pd.DataFrame,
    *,
    block_seconds: int,
) -> pd.DataFrame:
    """Construct exact, nonoverlapping contract-anchored rate blocks."""
    require(
        block_seconds >= 1,
        "Rate-block width must be positive.",
    )

    block_width_ns = (
        block_seconds
        * NANOSECONDS_PER_SECOND
    )

    block_frames: list[pd.DataFrame] = []

    for partition_name in ANALYTICAL_PARTITIONS:
        partition_batches = (
            batches.loc[
                batches[
                    "event_partition"
                ]
                .astype("string")
                .str.upper()
                .eq(partition_name)
            ]
            .copy()
            .sort_values(
                "partition_batch_index",
                kind="stable",
            )
            .reset_index(drop=True)
        )

        contract_row = contract_windows.loc[
            contract_windows[
                "event_partition"
            ].eq(partition_name)
        ]

        require(
            len(contract_row) == 1,
            (
                f"Expected one contract row for "
                f"{partition_name}."
            ),
        )

        contract_record = contract_row.iloc[0]

        contract_start_ns = int(
            contract_record[
                "contract_start_ns"
            ]
        )
        contract_end_exclusive_ns = int(
            contract_record[
                "contract_end_exclusive_ns"
            ]
        )
        contract_duration_ns = (
            contract_end_exclusive_ns
            - contract_start_ns
        )

        block_count = int(
            math.ceil(
                contract_duration_ns
                / block_width_ns
            )
        )

        block_indices = np.arange(
            block_count,
            dtype="int64",
        )

        block_start_ns = (
            contract_start_ns
            + block_indices * block_width_ns
        )

        block_end_exclusive_ns = np.minimum(
            block_start_ns + block_width_ns,
            contract_end_exclusive_ns,
        )

        block_exposure_ns = (
            block_end_exclusive_ns
            - block_start_ns
        )

        require(
            np.all(block_exposure_ns > 0),
            (
                f"{partition_name} contains a nonpositive "
                "chronological block exposure."
            ),
        )

        batch_times_ns = parse_exact_int64(
            partition_batches[
                "event_time_ns"
            ],
            label=(
                f"{partition_name}."
                "rate_block_event_time_ns"
            ),
        ).to_numpy(dtype="int64")

        batch_block_index = (
            (
                batch_times_ns
                - contract_start_ns
            )
            // block_width_ns
        ).astype("int64")

        require(
            np.all(batch_block_index >= 0)
            and np.all(
                batch_block_index
                < block_count
            ),
            (
                f"{partition_name} contains an event outside "
                "the rate-block grid."
            ),
        )

        batch_count_by_block = np.bincount(
            batch_block_index,
            minlength=block_count,
        ).astype("int64")

        event_count_by_block = np.bincount(
            batch_block_index,
            weights=partition_batches[
                "batch_event_count"
            ].to_numpy(dtype="int64"),
            minlength=block_count,
        ).astype("int64")

        buy_count_by_block = np.bincount(
            batch_block_index,
            weights=partition_batches[
                "buy_event_count"
            ].to_numpy(dtype="int64"),
            minlength=block_count,
        ).astype("int64")

        sell_count_by_block = np.bincount(
            batch_block_index,
            weights=partition_batches[
                "sell_event_count"
            ].to_numpy(dtype="int64"),
            minlength=block_count,
        ).astype("int64")

        simultaneous_count_by_block = np.bincount(
            batch_block_index,
            weights=partition_batches[
                "simultaneous_batch_required_flag"
            ]
            .astype("int64")
            .to_numpy(),
            minlength=block_count,
        ).astype("int64")

        mixed_count_by_block = np.bincount(
            batch_block_index,
            weights=partition_batches[
                "mixed_side_batch_flag"
            ]
            .astype("int64")
            .to_numpy(),
            minlength=block_count,
        ).astype("int64")

        require(
            int(event_count_by_block.sum())
            == int(
                partition_batches[
                    "batch_event_count"
                ].sum()
            ),
            (
                f"{partition_name} rate blocks do not conserve "
                "event count."
            ),
        )

        block_exposure_seconds = (
            block_exposure_ns.astype("float64")
            / NANOSECONDS_PER_SECOND
        )

        block_frames.append(
            pd.DataFrame(
                {
                    "event_partition": (
                        partition_name
                    ),
                    "block_index": (
                        block_indices
                    ),
                    "block_start_ns": (
                        block_start_ns
                    ),
                    "block_end_exclusive_ns": (
                        block_end_exclusive_ns
                    ),
                    "block_start_utc": (
                        pd.to_datetime(
                            block_start_ns,
                            unit="ns",
                            utc=True,
                        )
                    ),
                    "block_end_exclusive_utc": (
                        pd.to_datetime(
                            block_end_exclusive_ns,
                            unit="ns",
                            utc=True,
                        )
                    ),
                    "block_exposure_ns": (
                        block_exposure_ns
                    ),
                    "block_exposure_seconds": (
                        block_exposure_seconds
                    ),
                    "complete_block_flag": (
                        block_exposure_ns
                        == block_width_ns
                    ),
                    "batch_count": (
                        batch_count_by_block
                    ),
                    "event_count": (
                        event_count_by_block
                    ),
                    "buy_event_count": (
                        buy_count_by_block
                    ),
                    "sell_event_count": (
                        sell_count_by_block
                    ),
                    "simultaneous_batch_count": (
                        simultaneous_count_by_block
                    ),
                    "mixed_side_batch_count": (
                        mixed_count_by_block
                    ),
                    "batch_rate_per_second": (
                        batch_count_by_block
                        / block_exposure_seconds
                    ),
                    "pooled_event_rate_per_second": (
                        event_count_by_block
                        / block_exposure_seconds
                    ),
                    "buy_event_rate_per_second": (
                        buy_count_by_block
                        / block_exposure_seconds
                    ),
                    "sell_event_rate_per_second": (
                        sell_count_by_block
                        / block_exposure_seconds
                    ),
                }
            )
        )

    return (
        pd.concat(
            block_frames,
            ignore_index=True,
        )
        .sort_values(
            [
                "event_partition",
                "block_index",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )


CHRONOLOGICAL_RATE_BLOCKS_60S = (
    build_contract_rate_blocks(
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
        CONTRACT_WINDOW_CONTINUITY,
        block_seconds=(
            CHRONOLOGICAL_BLOCK_SECONDS
        ),
    )
)


chronological_block_summary_rows: list[
    dict[str, Any]
] = []

for partition_name in ANALYTICAL_PARTITIONS:
    partition_blocks = (
        CHRONOLOGICAL_RATE_BLOCKS_60S.loc[
            CHRONOLOGICAL_RATE_BLOCKS_60S[
                "event_partition"
            ].eq(partition_name)
        ]
    )

    complete_blocks = partition_blocks.loc[
        partition_blocks[
            "complete_block_flag"
        ]
    ]

    require(
        len(complete_blocks) >= 2,
        (
            f"{partition_name} has fewer than two complete "
            "chronological rate blocks."
        ),
    )

    pooled_rates = complete_blocks[
        "pooled_event_rate_per_second"
    ].to_numpy(dtype="float64")

    buy_rates = complete_blocks[
        "buy_event_rate_per_second"
    ].to_numpy(dtype="float64")

    sell_rates = complete_blocks[
        "sell_event_rate_per_second"
    ].to_numpy(dtype="float64")

    chronological_block_summary_rows.append(
        {
            "event_partition": partition_name,
            "block_width_seconds": (
                CHRONOLOGICAL_BLOCK_SECONDS
            ),
            "all_block_count": int(
                len(partition_blocks)
            ),
            "complete_block_count": int(
                len(complete_blocks)
            ),
            "partial_block_count": int(
                (
                    ~partition_blocks[
                        "complete_block_flag"
                    ]
                ).sum()
            ),
            "minimum_pooled_rate_per_second": float(
                np.min(pooled_rates)
            ),
            "median_pooled_rate_per_second": float(
                np.median(pooled_rates)
            ),
            "maximum_pooled_rate_per_second": float(
                np.max(pooled_rates)
            ),
            "pooled_rate_sample_std": (
                safe_sample_standard_deviation(
                    pooled_rates
                )
            ),
            "pooled_rate_coefficient_of_variation": (
                safe_coefficient_of_variation(
                    pooled_rates
                )
            ),
            "buy_rate_coefficient_of_variation": (
                safe_coefficient_of_variation(
                    buy_rates
                )
            ),
            "sell_rate_coefficient_of_variation": (
                safe_coefficient_of_variation(
                    sell_rates
                )
            ),
            "status": "PASS",
        }
    )


CHRONOLOGICAL_RATE_BLOCK_SUMMARY = pd.DataFrame(
    chronological_block_summary_rows
)


# ------------------------------------------------------------
# Enriched partition event-rate summary
# ------------------------------------------------------------

PARTITION_EVENT_RATE_SUMMARY = (
    ANALYTICAL_PARTITION_COVERAGE.merge(
        CHRONOLOGICAL_RATE_BLOCK_SUMMARY,
        on="event_partition",
        how="left",
        validate="one_to_one",
        suffixes=("", "_block"),
    )
)

development_rate = float(
    PARTITION_EVENT_RATE_SUMMARY.loc[
        PARTITION_EVENT_RATE_SUMMARY[
            "event_partition"
        ].eq("DEVELOPMENT"),
        "pooled_event_rate_per_second",
    ].iloc[0]
)

PARTITION_EVENT_RATE_SUMMARY[
    "pooled_rate_ratio_to_development"
] = (
    PARTITION_EVENT_RATE_SUMMARY[
        "pooled_event_rate_per_second"
    ]
    / development_rate
)

PARTITION_EVENT_RATE_SUMMARY[
    "absolute_pooled_rate_change_from_development"
] = (
    PARTITION_EVENT_RATE_SUMMARY[
        "pooled_event_rate_per_second"
    ]
    - development_rate
)


# ------------------------------------------------------------
# Contract-anchored count bins for burst concentration
# ------------------------------------------------------------

def contract_anchored_event_counts(
    batches: pd.DataFrame,
    contract_windows: pd.DataFrame,
    *,
    partition_name: str,
    width_ms: int,
) -> np.ndarray:
    """Return total-event counts in exact nonoverlapping time bins."""
    require(
        width_ms >= 1,
        "Count-bin width must be positive.",
    )

    width_ns = (
        width_ms
        * NANOSECONDS_PER_MILLISECOND
    )

    contract_row = contract_windows.loc[
        contract_windows[
            "event_partition"
        ].eq(partition_name)
    ]

    require(
        len(contract_row) == 1,
        (
            f"Expected one contract row for "
            f"{partition_name}."
        ),
    )

    contract_record = contract_row.iloc[0]

    contract_start_ns = int(
        contract_record["contract_start_ns"]
    )
    contract_end_exclusive_ns = int(
        contract_record[
            "contract_end_exclusive_ns"
        ]
    )

    contract_duration_ns = (
        contract_end_exclusive_ns
        - contract_start_ns
    )

    bin_count = int(
        math.ceil(
            contract_duration_ns / width_ns
        )
    )

    partition_batches = (
        batches.loc[
            batches[
                "event_partition"
            ]
            .astype("string")
            .str.upper()
            .eq(partition_name)
        ]
        .copy()
        .sort_values(
            "partition_batch_index",
            kind="stable",
        )
    )

    batch_times_ns = parse_exact_int64(
        partition_batches["event_time_ns"],
        label=(
            f"{partition_name}.{width_ms}ms."
            "event_time_ns"
        ),
    ).to_numpy(dtype="int64")

    bin_indices = (
        (
            batch_times_ns
            - contract_start_ns
        )
        // width_ns
    ).astype("int64")

    require(
        np.all(bin_indices >= 0)
        and np.all(bin_indices < bin_count),
        (
            f"{partition_name} contains a batch outside its "
            f"{width_ms} ms count grid."
        ),
    )

    event_counts = np.bincount(
        bin_indices,
        weights=partition_batches[
            "batch_event_count"
        ].to_numpy(dtype="int64"),
        minlength=bin_count,
    ).astype("int64")

    require(
        int(event_counts.sum())
        == int(
            partition_batches[
                "batch_event_count"
            ].sum()
        ),
        (
            f"{partition_name} {width_ms} ms bins do not "
            "conserve event count."
        ),
    )

    return event_counts


burst_concentration_rows: list[
    dict[str, Any]
] = []

partition_count_cache: dict[
    tuple[str, int],
    np.ndarray,
] = {}

for partition_name in ANALYTICAL_PARTITIONS:
    for width_ms in BURST_COUNT_WINDOWS_MS:
        partition_count_cache[
            (partition_name, width_ms)
        ] = contract_anchored_event_counts(
            PRIMARY_SCORING_BATCHES_ANALYTICAL,
            CONTRACT_WINDOW_CONTINUITY,
            partition_name=partition_name,
            width_ms=width_ms,
        )


for partition_scope in (
    *ANALYTICAL_PARTITIONS,
    "ANALYTICAL_COMBINED",
):
    scope_partitions = (
        ANALYTICAL_PARTITIONS
        if partition_scope
        == "ANALYTICAL_COMBINED"
        else (partition_scope,)
    )

    count_vectors = {
        width_ms: np.concatenate(
            [
                partition_count_cache[
                    (
                        partition_name,
                        width_ms,
                    )
                ]
                for partition_name
                in scope_partitions
            ]
        )
        for width_ms
        in BURST_COUNT_WINDOWS_MS
    }

    one_second_counts = count_vectors[1_000]

    total_events = int(
        one_second_counts.sum()
    )

    require(
        total_events > 0,
        (
            f"{partition_scope} contains no events for burst "
            "concentration diagnostics."
        ),
    )

    sorted_one_second_counts = np.sort(
        one_second_counts
    )[::-1]

    concentration_record: dict[str, Any] = {
        "partition_scope": partition_scope,
        "one_second_bin_count": int(
            one_second_counts.size
        ),
        "one_second_zero_bin_fraction": float(
            np.mean(
                one_second_counts == 0
            )
        ),
        "total_event_count": total_events,
    }

    for top_share in BUSIEST_WINDOW_SHARES:
        top_bin_count = max(
            1,
            int(
                math.ceil(
                    top_share
                    * one_second_counts.size
                )
            ),
        )

        concentration_record[
            (
                "event_share_in_busiest_"
                f"{int(top_share * 100):02d}pct_"
                "one_second_bins"
            )
        ] = float(
            sorted_one_second_counts[
                :top_bin_count
            ].sum()
            / total_events
        )

        concentration_record[
            (
                "busiest_"
                f"{int(top_share * 100):02d}pct_"
                "one_second_bin_count"
            )
        ] = top_bin_count

    for width_ms, counts in count_vectors.items():
        concentration_record[
            f"maximum_events_in_{width_ms}ms_bin"
        ] = int(np.max(counts))

        concentration_record[
            f"mean_events_in_{width_ms}ms_bin"
        ] = float(np.mean(counts))

        concentration_record[
            f"zero_fraction_in_{width_ms}ms_bins"
        ] = float(np.mean(counts == 0))

    concentration_record["status"] = "PASS"

    burst_concentration_rows.append(
        concentration_record
    )


BURST_CONCENTRATION = pd.DataFrame(
    burst_concentration_rows
)


# ------------------------------------------------------------
# Short-gap concentration
# ------------------------------------------------------------

short_gap_rows: list[dict[str, Any]] = []

for partition_scope in (
    *ANALYTICAL_PARTITIONS,
    "ANALYTICAL_COMBINED",
):
    scope_partitions = (
        ANALYTICAL_PARTITIONS
        if partition_scope
        == "ANALYTICAL_COMBINED"
        else (partition_scope,)
    )

    scope_batch_gap_arrays: list[np.ndarray] = []
    scope_following_event_weight_arrays: list[
        np.ndarray
    ] = []

    for partition_name in scope_partitions:
        partition_batches = (
            PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
                PRIMARY_SCORING_BATCHES_ANALYTICAL[
                    "event_partition"
                ]
                .astype("string")
                .str.upper()
                .eq(partition_name)
            ]
            .copy()
            .sort_values(
                "partition_batch_index",
                kind="stable",
            )
            .reset_index(drop=True)
        )

        batch_times_ns = parse_exact_int64(
            partition_batches[
                "event_time_ns"
            ],
            label=(
                f"{partition_name}."
                "short_gap_event_time_ns"
            ),
        ).to_numpy(dtype="int64")

        batch_gaps_ns = np.diff(
            batch_times_ns
        ).astype("int64")

        following_event_weights = (
            partition_batches[
                "batch_event_count"
            ]
            .iloc[1:]
            .to_numpy(dtype="int64")
        )

        require(
            batch_gaps_ns.size
            == following_event_weights.size,
            (
                f"{partition_name} short-gap weights do not "
                "align with complete inter-batch gaps."
            ),
        )

        scope_batch_gap_arrays.append(
            batch_gaps_ns
        )
        scope_following_event_weight_arrays.append(
            following_event_weights
        )

    scope_batch_gaps_ns = np.concatenate(
        scope_batch_gap_arrays
    )

    scope_following_event_weights = np.concatenate(
        scope_following_event_weight_arrays
    )

    require(
        scope_batch_gaps_ns.size > 0,
        (
            f"{partition_scope} contains no complete batch gaps."
        ),
    )

    for threshold_ms in SHORT_GAP_THRESHOLDS_MS:
        threshold_ns = (
            threshold_ms
            * NANOSECONDS_PER_MILLISECOND
        )

        short_gap_mask = (
            scope_batch_gaps_ns
            < threshold_ns
        )

        short_gap_rows.append(
            {
                "partition_scope": partition_scope,
                "gap_threshold_ms": (
                    threshold_ms
                ),
                "complete_batch_gap_count": int(
                    scope_batch_gaps_ns.size
                ),
                "short_batch_gap_count": int(
                    short_gap_mask.sum()
                ),
                "short_batch_gap_fraction": float(
                    np.mean(short_gap_mask)
                ),
                "following_event_count": int(
                    scope_following_event_weights.sum()
                ),
                "events_following_short_gap": int(
                    scope_following_event_weights[
                        short_gap_mask
                    ].sum()
                ),
                "event_weighted_short_gap_fraction": float(
                    scope_following_event_weights[
                        short_gap_mask
                    ].sum()
                    / scope_following_event_weights.sum()
                ),
                "partition_crossing_gap_count": 0,
                "status": "PASS",
            }
        )


SHORT_GAP_CONCENTRATION = pd.DataFrame(
    short_gap_rows
)


# ------------------------------------------------------------
# Descriptive audit gates
# ------------------------------------------------------------

DESCRIPTIVE_ARRIVAL_AUDIT_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "complete_durations_positive",
            "severity": "BLOCKING",
            "passed": bool(
                INTERARRIVAL_DURATION_TABLE[
                    "duration_ns"
                ].gt(0).all()
            ),
            "evidence": (
                f"complete_duration_rows="
                f"{len(INTERARRIVAL_DURATION_TABLE)}"
            ),
        },
        {
            "gate": "duration_intervals_do_not_cross_partitions",
            "severity": "BLOCKING",
            "passed": bool(
                ~INTERARRIVAL_DURATION_TABLE[
                    "cross_partition_flag"
                ].any()
            ),
            "evidence": "cross_partition_count=0",
        },
        {
            "gate": "left_and_right_censoring_recorded",
            "severity": "BLOCKING",
            "passed": bool(
                INTERARRIVAL_CENSORING_AUDIT[
                    "left_censored_duration_ns"
                ].ge(0).all()
                and INTERARRIVAL_CENSORING_AUDIT[
                    "right_censored_duration_ns"
                ].gt(0).all()
            ),
            "evidence": (
                f"censoring_rows="
                f"{len(INTERARRIVAL_CENSORING_AUDIT)}"
            ),
        },
        {
            "gate": "duration_bootstrap_complete",
            "severity": "BLOCKING",
            "passed": bool(
                INTERARRIVAL_DIAGNOSTICS[
                    "bootstrap_replicates"
                ].eq(
                    DURATION_BOOTSTRAP_REPLICATES
                ).all()
            ),
            "evidence": (
                f"replicates_per_process="
                f"{DURATION_BOOTSTRAP_REPLICATES}"
            ),
        },
        {
            "gate": "chronological_rate_blocks_conserve_events",
            "severity": "BLOCKING",
            "passed": bool(
                CHRONOLOGICAL_RATE_BLOCKS_60S[
                    "event_count"
                ].sum()
                == PRIMARY_SCORING_BATCHES_ANALYTICAL[
                    "batch_event_count"
                ].sum()
            ),
            "evidence": (
                f"event_count="
                f"{int(CHRONOLOGICAL_RATE_BLOCKS_60S['event_count'].sum())}"
            ),
        },
        {
            "gate": "burst_count_grids_conserve_events",
            "severity": "BLOCKING",
            "passed": bool(
                BURST_CONCENTRATION.loc[
                    BURST_CONCENTRATION[
                        "partition_scope"
                    ].eq("ANALYTICAL_COMBINED"),
                    "total_event_count",
                ].iloc[0]
                == len(
                    PRIMARY_ESTIMATION_EVENTS_ANALYTICAL
                )
            ),
            "evidence": (
                f"analytical_events="
                f"{len(PRIMARY_ESTIMATION_EVENTS_ANALYTICAL)}"
            ),
        },
        {
            "gate": "notebook_04_timestamp_authority_used",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "all durations and bins use canonical Notebook 04 event_time_ns"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    DESCRIPTIVE_ARRIVAL_AUDIT_GATE_FRAME.loc[
        DESCRIPTIVE_ARRIVAL_AUDIT_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    "At least one descriptive arrival-audit gate failed.",
)


display(
    PARTITION_EVENT_RATE_SUMMARY[
        [
            "event_partition",
            "contract_duration_seconds",
            "batch_count",
            "event_count",
            "buy_event_count",
            "sell_event_count",
            "pooled_event_rate_per_second",
            "buy_event_rate_per_second",
            "sell_event_rate_per_second",
            "buy_event_share",
            "pooled_rate_ratio_to_development",
            "minimum_pooled_rate_per_second",
            "median_pooled_rate_per_second",
            "maximum_pooled_rate_per_second",
            "pooled_rate_coefficient_of_variation",
        ]
    ]
)

display(INTERARRIVAL_DIAGNOSTICS)
display(INTERARRIVAL_CENSORING_AUDIT)
display(CHRONOLOGICAL_RATE_BLOCK_SUMMARY)
display(BURST_CONCENTRATION)
display(SHORT_GAP_CONCENTRATION)
display(DESCRIPTIVE_ARRIVAL_AUDIT_GATE_FRAME)

print(
    "Descriptive arrival-rate, duration, censoring, and burst "
    "audit passed."
)
print(
    "Interarrival durations were constructed separately inside "
    "each partition for pooled, BUY, and SELL exact-time batch "
    "arrivals."
)
print(
    "No partition boundary was crossed, and left- and right-edge "
    "censoring were preserved explicitly."
)
print(
    "Exponential duration diagnostics use a fitted-scale "
    "parametric bootstrap rather than an unadjusted KS p-value."
)
print(
    "Chronological rate blocks and burst-concentration grids use "
    "full registered contract exposure and canonical Notebook 04 "
    "timestamps."
)

,event_partition,contract_duration_seconds,batch_count,event_count,buy_event_count,sell_event_count,pooled_event_rate_per_second,buy_event_rate_per_second,sell_event_rate_per_second,buy_event_share,pooled_rate_ratio_to_development,minimum_pooled_rate_per_second,median_pooled_rate_per_second,maximum_pooled_rate_per_second,pooled_rate_coefficient_of_variation
0,DEVELOPMENT,"1,801.8596",6859,7004,3414,3590,3.8870953,1.8947092,1.9923861,0.48743575,1,2.9333333,3.7916667,6.7166667,0.20506641
1,CALIBRATION,720.29918,2400,2493,1230,1263,3.4610618,1.7076238,1.7534381,0.49338147,0.89039798,2.5666667,3.4416667,4.3833333,0.17264129


,event_partition,process_name,arrival_count,complete_duration_count,zero_duration_count,minimum_duration_seconds,q01_duration_seconds,q05_duration_seconds,q25_duration_seconds,median_duration_seconds,q75_duration_seconds,q95_duration_seconds,q99_duration_seconds,maximum_duration_seconds,mean_duration_seconds,sample_standard_deviation_seconds,coefficient_of_variation,burstiness_index,fitted_exponential_rate_per_second,observed_ks_distance,bootstrap_ks_q025,bootstrap_ks_median,bootstrap_ks_q950,bootstrap_ks_q975,bootstrap_p_value,exponential_rejected_at_5pct,bootstrap_replicates,status
0,CALIBRATION,BUY_BATCH,1155,1154,0,0.0003932,0.000993355,0.001573285,0.052050575,0.3469182,0.89425385,2.2653923,3.3791587,7.708079,0.62221743,0.81127119,1.3038387,0.13188369,1.6071552,0.18292062,0.012770313,0.020810386,0.031700797,0.034451956,0.00049975012,True,2000,PASS
1,CALIBRATION,POOLED_BATCH,2400,2399,0,0.0003932,0.001002398,0.0022814,0.0486635,0.1758249,0.4340797,1.0259555,1.537129,3.0728117,0.29962597,0.3481591,1.161979,0.074921653,3.3374944,0.11465317,0.008897778,0.014196663,0.021739968,0.023365593,0.00049975012,True,2000,PASS
2,CALIBRATION,SELL_BATCH,1250,1249,0,0.00061,0.001749664,0.00718312,0.127078,0.3954954,0.8553053,1.7170226,2.4436218,5.4738412,0.57514465,0.59313523,1.0312801,0.015399205,1.738693,0.06313451,0.012031674,0.019721571,0.031156685,0.033618988,0.00049975012,True,2000,PASS
3,DEVELOPMENT,BUY_BATCH,3331,3330,0,0.0003603,0.0010027,0.002057565,0.074916475,0.3187516,0.74279147,1.9320733,3.0276686,7.9435024,0.54056453,0.67260294,1.2442602,0.10883774,1.8499179,0.12670329,0.007422488,0.012068942,0.018529708,0.020182398,0.00049975012,True,2000,PASS
4,DEVELOPMENT,POOLED_BATCH,6859,6858,0,0.0003603,0.0010045,0.00300057,0.0483549,0.1630179,0.372478,0.88519412,1.3303242,2.6554789,0.26253503,0.29836789,1.1364879,0.063884241,3.8090155,0.093336363,0.0053209208,0.0084760875,0.013236963,0.014460308,0.00049975012,True,2000,PASS
5,DEVELOPMENT,SELL_BATCH,3554,3553,0,0.0005035,0.001603936,0.00591234,0.1032023,0.3290953,0.7253093,1.5867777,2.4002317,4.3506376,0.50632983,0.54210135,1.0706486,0.03411909,1.9749972,0.074464632,0.0072836763,0.011793197,0.017811454,0.019362118,0.00049975012,True,2000,PASS


,event_partition,process_name,arrival_count,complete_duration_count,left_censored_duration_ns,left_censored_duration_seconds,right_censored_duration_ns,right_censored_duration_seconds,complete_duration_cross_partition_count,status
0,CALIBRATION,BUY_BATCH,1155,1154,1155409900,1.1554099,1104856900,1.1048569,0,PASS
1,CALIBRATION,POOLED_BATCH,2400,2399,1155409900,1.1554099,341065900,0.3410659,0,PASS
2,CALIBRATION,SELL_BATCH,1250,1249,1602442900,1.6024429,341065900,0.3410659,0,PASS
3,DEVELOPMENT,BUY_BATCH,3331,3330,1234966200,1.2349662,544745000,0.544745,0,PASS
4,DEVELOPMENT,POOLED_BATCH,6859,6858,1234966200,1.2349662,159367100,0.1593671,0,PASS
5,DEVELOPMENT,SELL_BATCH,3554,3553,2710326700,2.7103267,159367100,0.1593671,0,PASS


,event_partition,block_width_seconds,all_block_count,complete_block_count,partial_block_count,minimum_pooled_rate_per_second,median_pooled_rate_per_second,maximum_pooled_rate_per_second,pooled_rate_sample_std,pooled_rate_coefficient_of_variation,buy_rate_coefficient_of_variation,sell_rate_coefficient_of_variation,status
0,DEVELOPMENT,60,31,30,1,2.9333333,3.7916667,6.7166667,0.79725263,0.20506641,0.28602944,0.18877112,PASS
1,CALIBRATION,60,13,12,1,2.5666667,3.4416667,4.3833333,0.59777048,0.17264129,0.33002049,0.19417301,PASS


,partition_scope,one_second_bin_count,one_second_zero_bin_fraction,total_event_count,event_share_in_busiest_01pct_one_second_bins,busiest_01pct_one_second_bin_count,event_share_in_busiest_05pct_one_second_bins,busiest_05pct_one_second_bin_count,event_share_in_busiest_10pct_one_second_bins,busiest_10pct_one_second_bin_count,maximum_events_in_100ms_bin,mean_events_in_100ms_bin,zero_fraction_in_100ms_bins,maximum_events_in_1000ms_bin,mean_events_in_1000ms_bin,zero_fraction_in_1000ms_bins,maximum_events_in_5000ms_bin,mean_events_in_5000ms_bin,zero_fraction_in_5000ms_bins,status
0,DEVELOPMENT,1802,0.035516093,7004,0.059680183,19,0.18018275,91,0.28183895,181,19,0.38870082,0.71402409,33,3.8867925,0.035516093,73,19.401662,0,PASS
1,CALIBRATION,721,0.049930652,2493,0.07139992,8,0.21339751,37,0.31768953,73,24,0.34610579,0.75024295,30,3.4576976,0.049930652,59,17.193103,0.0068965517,PASS
2,ANALYTICAL_COMBINED,2523,0.039635355,9497,0.060966621,26,0.18795409,127,0.29093398,253,24,0.37653636,0.72436762,33,3.7641696,0.039635355,73,18.768775,0.0019762846,PASS


,partition_scope,gap_threshold_ms,complete_batch_gap_count,short_batch_gap_count,short_batch_gap_fraction,following_event_count,events_following_short_gap,event_weighted_short_gap_fraction,partition_crossing_gap_count,status
0,DEVELOPMENT,10,6858,795,0.11592301,7003,907,0.12951592,0,PASS
1,DEVELOPMENT,25,6858,1251,0.1824147,7003,1371,0.19577324,0,PASS
2,DEVELOPMENT,50,6858,1749,0.25503062,7003,1876,0.26788519,0,PASS
3,DEVELOPMENT,100,6858,2642,0.38524351,7003,2774,0.39611595,0,PASS
4,CALIBRATION,10,2399,318,0.13255523,2492,374,0.15008026,0,PASS
5,CALIBRATION,25,2399,462,0.19258024,2492,522,0.2094703,0,PASS
6,CALIBRATION,50,2399,615,0.25635682,2492,679,0.27247191,0,PASS
7,CALIBRATION,100,2399,902,0.37599,2492,976,0.39165329,0,PASS
8,ANALYTICAL_COMBINED,10,9257,1113,0.12023334,9495,1281,0.13491311,0,PASS
9,ANALYTICAL_COMBINED,25,9257,1713,0.18504915,9495,1893,0.19936809,0,PASS


,gate,severity,passed,evidence
0,complete_durations_positive,BLOCKING,True,complete_duration_rows=18543
1,duration_intervals_do_not_cross_partitions,BLOCKING,True,cross_partition_count=0
2,left_and_right_censoring_recorded,BLOCKING,True,censoring_rows=6
3,duration_bootstrap_complete,BLOCKING,True,replicates_per_process=2000
4,chronological_rate_blocks_conserve_events,BLOCKING,True,event_count=9497
5,burst_count_grids_conserve_events,BLOCKING,True,analytical_events=9497
6,notebook_04_timestamp_authority_used,BLOCKING,True,all durations and bins use canonical Notebook ...
7,protected_partition_content_remains_absent,BLOCKING,True,VALIDATION=False; ENGINEERING_HOLDOUT=False


Descriptive arrival-rate, duration, censoring, and burst audit passed.
Interarrival durations were constructed separately inside each partition for pooled, BUY, and SELL exact-time batch arrivals.
No partition boundary was crossed, and left- and right-edge censoring were preserved explicitly.
Exponential duration diagnostics use a fitted-scale parametric bootstrap rather than an unadjusted KS p-value.
Chronological rate blocks and burst-concentration grids use full registered contract exposure and canonical Notebook 04 timestamps.


In [10]:
# ============================================================
# Native count grids and homogeneous Poisson baselines
# ============================================================

NATIVE_COUNT_GRID_WIDTH_NS: Final[int] = (
    MODEL_COUNT_GRID_NS
)
NATIVE_COUNT_GRID_WIDTH_MS: Final[int] = (
    MODEL_COUNT_GRID_MS
)

HOMOGENEOUS_MODEL_FIT_PARTITION: Final[str] = (
    "DEVELOPMENT"
)


# ------------------------------------------------------------
# Exact native-grid container
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class NativeCountGrid:
    event_partition: str
    contract_start_ns: int
    contract_end_exclusive_ns: int
    contract_duration_ns: int
    grid_width_ns: int
    cell_count: int
    complete_cell_count: int
    partial_final_cell_flag: bool
    final_cell_exposure_ns: int
    exposure_ns: np.ndarray
    pooled_counts: np.ndarray
    buy_counts: np.ndarray
    sell_counts: np.ndarray


def exact_ceil_division(
    numerator: int,
    denominator: int,
) -> int:
    """Return ceil(numerator / denominator) using integer arithmetic."""
    require(
        numerator >= 0,
        "Ceiling-division numerator must be nonnegative.",
    )
    require(
        denominator > 0,
        "Ceiling-division denominator must be positive.",
    )

    return -(-numerator // denominator)


def grid_exposure_seconds(
    grid: NativeCountGrid,
) -> np.ndarray:
    """Return cell exposures in seconds."""
    return (
        grid.exposure_ns.astype("float64")
        / NANOSECONDS_PER_SECOND
    )


def build_native_count_grid(
    batches: pd.DataFrame,
    contract_windows: pd.DataFrame,
    *,
    partition_name: str,
) -> NativeCountGrid:
    """
    Construct a contract-anchored native count grid.

    Exposure conservation is verified exactly in integer nanoseconds.
    No floating-point sum is used to establish the time contract.
    """
    contract_row = contract_windows.loc[
        contract_windows[
            "event_partition"
        ].eq(partition_name)
    ]

    require(
        len(contract_row) == 1,
        (
            f"Expected exactly one contract row for "
            f"{partition_name}."
        ),
    )

    contract_record = contract_row.iloc[0]

    contract_start_ns = int(
        contract_record["contract_start_ns"]
    )
    contract_end_exclusive_ns = int(
        contract_record[
            "contract_end_exclusive_ns"
        ]
    )
    registered_contract_duration_ns = int(
        contract_record["contract_duration_ns"]
    )

    contract_duration_ns = (
        contract_end_exclusive_ns
        - contract_start_ns
    )

    require(
        contract_duration_ns
        == registered_contract_duration_ns,
        (
            f"{partition_name} contract duration differs from "
            "its registered duration."
        ),
    )
    require(
        contract_duration_ns > 0,
        (
            f"{partition_name} contract duration must be "
            "positive."
        ),
    )

    cell_count = exact_ceil_division(
        contract_duration_ns,
        NATIVE_COUNT_GRID_WIDTH_NS,
    )

    complete_cell_count, remainder_ns = divmod(
        contract_duration_ns,
        NATIVE_COUNT_GRID_WIDTH_NS,
    )

    partial_final_cell_flag = (
        remainder_ns != 0
    )

    if partial_final_cell_flag:
        final_cell_exposure_ns = remainder_ns
    else:
        final_cell_exposure_ns = (
            NATIVE_COUNT_GRID_WIDTH_NS
        )

    exposure_ns = np.full(
        cell_count,
        NATIVE_COUNT_GRID_WIDTH_NS,
        dtype="int64",
    )

    if partial_final_cell_flag:
        exposure_ns[-1] = (
            final_cell_exposure_ns
        )

    exact_exposure_sum_ns = int(
        exposure_ns.sum(dtype="int64")
    )

    require(
        exact_exposure_sum_ns
        == contract_duration_ns,
        (
            f"{partition_name} native-grid exposure does not "
            "match its registered contract duration: "
            f"grid={exact_exposure_sum_ns} ns; "
            f"contract={contract_duration_ns} ns."
        ),
    )

    partition_batches = (
        batches.loc[
            batches[
                "event_partition"
            ]
            .astype("string")
            .str.strip()
            .str.upper()
            .eq(partition_name)
        ]
        .copy()
        .sort_values(
            "partition_batch_index",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    require(
        not partition_batches.empty,
        (
            f"{partition_name} contains no scoring batches."
        ),
    )

    event_times_ns = parse_exact_int64(
        partition_batches["event_time_ns"],
        label=(
            f"{partition_name}."
            "native_grid_event_time_ns"
        ),
    ).to_numpy(dtype="int64")

    require(
        np.all(
            event_times_ns
            >= contract_start_ns
        ),
        (
            f"{partition_name} contains an event before the "
            "native-grid contract start."
        ),
    )
    require(
        np.all(
            event_times_ns
            < contract_end_exclusive_ns
        ),
        (
            f"{partition_name} contains an event at or beyond "
            "the native-grid end-exclusive boundary."
        ),
    )

    cell_indices = (
        (
            event_times_ns
            - contract_start_ns
        )
        // NATIVE_COUNT_GRID_WIDTH_NS
    ).astype("int64")

    require(
        np.all(cell_indices >= 0),
        (
            f"{partition_name} contains a negative native-grid "
            "cell index."
        ),
    )
    require(
        np.all(cell_indices < cell_count),
        (
            f"{partition_name} contains an out-of-range "
            "native-grid cell index."
        ),
    )

    buy_weights = (
        partition_batches[
            "buy_event_count"
        ]
        .astype("int64")
        .to_numpy()
    )

    sell_weights = (
        partition_batches[
            "sell_event_count"
        ]
        .astype("int64")
        .to_numpy()
    )

    pooled_weights = (
        partition_batches[
            "batch_event_count"
        ]
        .astype("int64")
        .to_numpy()
    )

    require(
        np.array_equal(
            buy_weights + sell_weights,
            pooled_weights,
        ),
        (
            f"{partition_name} batch BUY and SELL counts do not "
            "sum to pooled event multiplicity."
        ),
    )

    buy_counts = np.zeros(
        cell_count,
        dtype="int64",
    )
    sell_counts = np.zeros(
        cell_count,
        dtype="int64",
    )

    np.add.at(
        buy_counts,
        cell_indices,
        buy_weights,
    )
    np.add.at(
        sell_counts,
        cell_indices,
        sell_weights,
    )

    pooled_counts = (
        buy_counts + sell_counts
    )

    expected_buy_count = int(
        partition_batches[
            "buy_event_count"
        ].sum()
    )
    expected_sell_count = int(
        partition_batches[
            "sell_event_count"
        ].sum()
    )
    expected_pooled_count = int(
        partition_batches[
            "batch_event_count"
        ].sum()
    )

    require(
        int(
            buy_counts.sum(dtype="int64")
        )
        == expected_buy_count,
        (
            f"{partition_name} native-grid BUY counts do not "
            "conserve events."
        ),
    )
    require(
        int(
            sell_counts.sum(dtype="int64")
        )
        == expected_sell_count,
        (
            f"{partition_name} native-grid SELL counts do not "
            "conserve events."
        ),
    )
    require(
        int(
            pooled_counts.sum(dtype="int64")
        )
        == expected_pooled_count,
        (
            f"{partition_name} native-grid pooled counts do not "
            "conserve events."
        ),
    )
    require(
        np.array_equal(
            pooled_counts,
            buy_counts + sell_counts,
        ),
        (
            f"{partition_name} native-grid pooled counts differ "
            "from BUY plus SELL."
        ),
    )
    require(
        np.all(exposure_ns > 0),
        (
            f"{partition_name} native grid contains a "
            "nonpositive cell exposure."
        ),
    )

    for array in (
        exposure_ns,
        pooled_counts,
        buy_counts,
        sell_counts,
    ):
        array.setflags(write=False)

    return NativeCountGrid(
        event_partition=partition_name,
        contract_start_ns=contract_start_ns,
        contract_end_exclusive_ns=(
            contract_end_exclusive_ns
        ),
        contract_duration_ns=(
            contract_duration_ns
        ),
        grid_width_ns=(
            NATIVE_COUNT_GRID_WIDTH_NS
        ),
        cell_count=cell_count,
        complete_cell_count=(
            complete_cell_count
        ),
        partial_final_cell_flag=(
            partial_final_cell_flag
        ),
        final_cell_exposure_ns=(
            final_cell_exposure_ns
        ),
        exposure_ns=exposure_ns,
        pooled_counts=pooled_counts,
        buy_counts=buy_counts,
        sell_counts=sell_counts,
    )


NATIVE_COUNT_GRIDS: Final[
    Mapping[str, NativeCountGrid]
] = {
    partition_name: build_native_count_grid(
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
        CONTRACT_WINDOW_CONTINUITY,
        partition_name=partition_name,
    )
    for partition_name in ANALYTICAL_PARTITIONS
}


# ------------------------------------------------------------
# Native-grid audit
# ------------------------------------------------------------

native_count_grid_audit_rows: list[
    dict[str, Any]
] = []

for partition_name, grid in (
    NATIVE_COUNT_GRIDS.items()
):
    exact_exposure_sum_ns = int(
        grid.exposure_ns.sum(dtype="int64")
    )

    native_count_grid_audit_rows.append(
        {
            "event_partition": (
                partition_name
            ),
            "grid_width_ns": (
                grid.grid_width_ns
            ),
            "grid_width_ms": (
                grid.grid_width_ns
                / NANOSECONDS_PER_MILLISECOND
            ),
            "cell_count": (
                grid.cell_count
            ),
            "complete_cell_count": (
                grid.complete_cell_count
            ),
            "partial_final_cell_flag": (
                grid.partial_final_cell_flag
            ),
            "final_cell_exposure_ns": (
                grid.final_cell_exposure_ns
            ),
            "contract_duration_ns": (
                grid.contract_duration_ns
            ),
            "exact_exposure_sum_ns": (
                exact_exposure_sum_ns
            ),
            "exposure_difference_ns": (
                exact_exposure_sum_ns
                - grid.contract_duration_ns
            ),
            "pooled_event_count": int(
                grid.pooled_counts.sum(
                    dtype="int64"
                )
            ),
            "buy_event_count": int(
                grid.buy_counts.sum(
                    dtype="int64"
                )
            ),
            "sell_event_count": int(
                grid.sell_counts.sum(
                    dtype="int64"
                )
            ),
            "nonempty_pooled_cell_count": int(
                np.count_nonzero(
                    grid.pooled_counts
                )
            ),
            "maximum_pooled_cell_count": int(
                grid.pooled_counts.max()
            ),
            "status": "PASS",
        }
    )


NATIVE_COUNT_GRID_AUDIT = pd.DataFrame(
    native_count_grid_audit_rows
)


# ------------------------------------------------------------
# Poisson score and residual helpers
# ------------------------------------------------------------

def poisson_count_log_score(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
) -> float:
    """Return the complete Poisson count log score."""
    observed = np.asarray(
        observed_counts,
        dtype="int64",
    )
    expected = np.asarray(
        expected_counts,
        dtype="float64",
    )

    require(
        observed.ndim == 1,
        "Observed Poisson counts must be one-dimensional.",
    )
    require(
        expected.ndim == 1,
        "Expected Poisson counts must be one-dimensional.",
    )
    require(
        observed.shape == expected.shape,
        "Observed and expected Poisson arrays differ in shape.",
    )
    require(
        np.all(observed >= 0),
        "Observed Poisson counts contain negative values.",
    )
    require(
        np.isfinite(expected).all(),
        "Expected Poisson counts contain nonfinite values.",
    )
    require(
        np.all(expected > 0.0),
        "Expected Poisson counts must be strictly positive.",
    )

    score_terms = (
        observed.astype("float64")
        * np.log(expected)
        - expected
        - special.gammaln(
            observed.astype("float64")
            + 1.0
        )
    )

    require(
        np.isfinite(score_terms).all(),
        "Poisson score contains nonfinite terms.",
    )

    return float(
        score_terms.sum(dtype="float64")
    )


def poisson_deviance_residuals(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
) -> np.ndarray:
    """Return signed Poisson deviance residuals."""
    observed = np.asarray(
        observed_counts,
        dtype="float64",
    )
    expected = np.asarray(
        expected_counts,
        dtype="float64",
    )

    require(
        observed.shape == expected.shape,
        "Poisson residual arrays differ in shape.",
    )
    require(
        np.all(observed >= 0.0),
        "Observed counts contain negative values.",
    )
    require(
        np.isfinite(expected).all()
        and np.all(expected > 0.0),
        "Expected counts must be finite and positive.",
    )

    positive_observation = (
        observed > 0.0
    )

    deviance_component = np.empty_like(
        expected
    )

    deviance_component[
        ~positive_observation
    ] = expected[
        ~positive_observation
    ]

    deviance_component[
        positive_observation
    ] = (
        observed[
            positive_observation
        ]
        * np.log(
            observed[
                positive_observation
            ]
            / expected[
                positive_observation
            ]
        )
        - (
            observed[
                positive_observation
            ]
            - expected[
                positive_observation
            ]
        )
    )

    deviance_component = np.maximum(
        deviance_component,
        0.0,
    )

    residuals = (
        np.sign(observed - expected)
        * np.sqrt(
            2.0 * deviance_component
        )
    )

    require(
        np.isfinite(residuals).all(),
        "Poisson deviance residuals contain nonfinite values.",
    )

    return residuals


def summarize_poisson_residuals(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
    *,
    parameter_count: int,
) -> dict[str, float]:
    """Return count-residual summary statistics."""
    observed = np.asarray(
        observed_counts,
        dtype="float64",
    )
    expected = np.asarray(
        expected_counts,
        dtype="float64",
    )

    raw_residuals = (
        observed - expected
    )

    pearson_residuals = (
        raw_residuals
        / np.sqrt(expected)
    )

    deviance_residuals = (
        poisson_deviance_residuals(
            observed,
            expected,
        )
    )

    residual_degrees_of_freedom = (
        observed.size - parameter_count
    )

    require(
        residual_degrees_of_freedom > 0,
        "Residual degrees of freedom must be positive.",
    )

    pearson_chi_square = float(
        np.square(
            pearson_residuals
        ).sum(dtype="float64")
    )

    return {
        "raw_residual_mean": float(
            np.mean(raw_residuals)
        ),
        "raw_residual_variance": float(
            np.var(
                raw_residuals,
                ddof=1,
            )
        ),
        "pearson_residual_mean": float(
            np.mean(pearson_residuals)
        ),
        "pearson_residual_variance": float(
            np.var(
                pearson_residuals,
                ddof=1,
            )
        ),
        "deviance_residual_mean": float(
            np.mean(deviance_residuals)
        ),
        "deviance_residual_variance": float(
            np.var(
                deviance_residuals,
                ddof=1,
            )
        ),
        "pearson_chi_square": (
            pearson_chi_square
        ),
        "residual_degrees_of_freedom": int(
            residual_degrees_of_freedom
        ),
        "pearson_dispersion_ratio": (
            pearson_chi_square
            / residual_degrees_of_freedom
        ),
        "maximum_absolute_pearson_residual": float(
            np.max(
                np.abs(
                    pearson_residuals
                )
            )
        ),
        "maximum_absolute_deviance_residual": float(
            np.max(
                np.abs(
                    deviance_residuals
                )
            )
        ),
    }


# ------------------------------------------------------------
# Frozen DEVELOPMENT estimates
# ------------------------------------------------------------

development_grid = NATIVE_COUNT_GRIDS[
    HOMOGENEOUS_MODEL_FIT_PARTITION
]

development_exposure_seconds = (
    development_grid.contract_duration_ns
    / NANOSECONDS_PER_SECOND
)

development_pooled_event_count = int(
    development_grid.pooled_counts.sum(
        dtype="int64"
    )
)
development_buy_event_count = int(
    development_grid.buy_counts.sum(
        dtype="int64"
    )
)
development_sell_event_count = int(
    development_grid.sell_counts.sum(
        dtype="int64"
    )
)

require(
    development_buy_event_count
    + development_sell_event_count
    == development_pooled_event_count,
    "DEVELOPMENT side counts do not conserve pooled events.",
)

DEVELOPMENT_POOLED_RATE_PER_SECOND = (
    development_pooled_event_count
    / development_exposure_seconds
)

DEVELOPMENT_BUY_RATE_PER_SECOND = (
    development_buy_event_count
    / development_exposure_seconds
)

DEVELOPMENT_SELL_RATE_PER_SECOND = (
    development_sell_event_count
    / development_exposure_seconds
)

DEVELOPMENT_EQUAL_SIDE_RATE_PER_SECOND = (
    DEVELOPMENT_POOLED_RATE_PER_SECOND
    / 2.0
)

for rate_name, rate_value in (
    (
        "DEVELOPMENT_POOLED_RATE_PER_SECOND",
        DEVELOPMENT_POOLED_RATE_PER_SECOND,
    ),
    (
        "DEVELOPMENT_BUY_RATE_PER_SECOND",
        DEVELOPMENT_BUY_RATE_PER_SECOND,
    ),
    (
        "DEVELOPMENT_SELL_RATE_PER_SECOND",
        DEVELOPMENT_SELL_RATE_PER_SECOND,
    ),
    (
        "DEVELOPMENT_EQUAL_SIDE_RATE_PER_SECOND",
        DEVELOPMENT_EQUAL_SIDE_RATE_PER_SECOND,
    ),
):
    require(
        np.isfinite(rate_value)
        and rate_value > 0.0,
        f"{rate_name} must be finite and positive.",
    )


@dataclass(frozen=True, slots=True)
class HomogeneousPoissonModel:
    model_id: str
    model_family: str
    fitted_partition: str
    score_space: str
    parameter_count: int
    pooled_rate_per_second: float | None
    buy_rate_per_second: float | None
    sell_rate_per_second: float | None


HOMOGENEOUS_POISSON_MODELS: Final[
    Mapping[str, HomogeneousPoissonModel]
] = {
    "B0_POOLED_HOMOGENEOUS_POISSON": (
        HomogeneousPoissonModel(
            model_id=(
                "B0_POOLED_HOMOGENEOUS_POISSON"
            ),
            model_family=(
                "POOLED_HOMOGENEOUS_POISSON"
            ),
            fitted_partition=(
                HOMOGENEOUS_MODEL_FIT_PARTITION
            ),
            score_space="POOLED_COUNT",
            parameter_count=1,
            pooled_rate_per_second=(
                DEVELOPMENT_POOLED_RATE_PER_SECOND
            ),
            buy_rate_per_second=None,
            sell_rate_per_second=None,
        )
    ),
    "B0_EQUAL_SIDE_EXTENSION": (
        HomogeneousPoissonModel(
            model_id="B0_EQUAL_SIDE_EXTENSION",
            model_family=(
                "EQUAL_SIDE_HOMOGENEOUS_POISSON"
            ),
            fitted_partition=(
                HOMOGENEOUS_MODEL_FIT_PARTITION
            ),
            score_space="BIVARIATE_SIDE_COUNT",
            parameter_count=1,
            pooled_rate_per_second=(
                DEVELOPMENT_POOLED_RATE_PER_SECOND
            ),
            buy_rate_per_second=(
                DEVELOPMENT_EQUAL_SIDE_RATE_PER_SECOND
            ),
            sell_rate_per_second=(
                DEVELOPMENT_EQUAL_SIDE_RATE_PER_SECOND
            ),
        )
    ),
    "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON": (
        HomogeneousPoissonModel(
            model_id=(
                "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
            ),
            model_family=(
                "INDEPENDENT_SIDE_SPECIFIC_"
                "HOMOGENEOUS_POISSON"
            ),
            fitted_partition=(
                HOMOGENEOUS_MODEL_FIT_PARTITION
            ),
            score_space="BIVARIATE_SIDE_COUNT",
            parameter_count=2,
            pooled_rate_per_second=(
                DEVELOPMENT_POOLED_RATE_PER_SECOND
            ),
            buy_rate_per_second=(
                DEVELOPMENT_BUY_RATE_PER_SECOND
            ),
            sell_rate_per_second=(
                DEVELOPMENT_SELL_RATE_PER_SECOND
            ),
        )
    ),
}


HOMOGENEOUS_POISSON_PARAMETER_TABLE = pd.DataFrame(
    [
        {
            "model_id": model.model_id,
            "model_family": model.model_family,
            "fitted_partition": (
                model.fitted_partition
            ),
            "score_space": (
                model.score_space
            ),
            "parameter_count": (
                model.parameter_count
            ),
            "pooled_rate_per_second": (
                model.pooled_rate_per_second
            ),
            "buy_rate_per_second": (
                model.buy_rate_per_second
            ),
            "sell_rate_per_second": (
                model.sell_rate_per_second
            ),
            "fit_exposure_seconds": (
                development_exposure_seconds
            ),
            "fit_pooled_event_count": (
                development_pooled_event_count
            ),
            "fit_buy_event_count": (
                development_buy_event_count
            ),
            "fit_sell_event_count": (
                development_sell_event_count
            ),
            "parameter_status": "FINITE_POSITIVE",
        }
        for model
        in HOMOGENEOUS_POISSON_MODELS.values()
    ]
)


# ------------------------------------------------------------
# DEVELOPMENT and locked CALIBRATION scoring
# ------------------------------------------------------------

homogeneous_score_rows: list[
    dict[str, Any]
] = []

homogeneous_residual_rows: list[
    dict[str, Any]
] = []


def append_score_row(
    *,
    model: HomogeneousPoissonModel,
    scored_partition: str,
    component: str,
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
    component_parameter_count: int,
) -> float:
    """Score one Poisson component and append audit rows."""
    observed_event_count = int(
        observed_counts.sum(
            dtype="int64"
        )
    )
    predicted_event_count = float(
        expected_counts.sum(
            dtype="float64"
        )
    )

    partition_exposure_seconds = (
        NATIVE_COUNT_GRIDS[
            scored_partition
        ].contract_duration_ns
        / NANOSECONDS_PER_SECOND
    )

    log_score = poisson_count_log_score(
        observed_counts,
        expected_counts,
    )

    residual_summary = (
        summarize_poisson_residuals(
            observed_counts,
            expected_counts,
            parameter_count=(
                component_parameter_count
            ),
        )
    )

    homogeneous_score_rows.append(
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "fitted_partition": (
                model.fitted_partition
            ),
            "scored_partition": (
                scored_partition
            ),
            "score_space": (
                model.score_space
            ),
            "component": component,
            "parameter_count": (
                component_parameter_count
            ),
            "cell_count": int(
                observed_counts.size
            ),
            "exposure_seconds": (
                partition_exposure_seconds
            ),
            "observed_event_count": (
                observed_event_count
            ),
            "predicted_event_count": (
                predicted_event_count
            ),
            "observed_minus_predicted": (
                observed_event_count
                - predicted_event_count
            ),
            "log_score_total": (
                log_score
            ),
            "log_score_per_second": (
                log_score
                / partition_exposure_seconds
            ),
            "log_score_per_observed_event": (
                log_score
                / observed_event_count
            ),
            "locked_before_calibration": True,
            "status": "PASS",
        }
    )

    homogeneous_residual_rows.append(
        {
            "model_id": model.model_id,
            "scored_partition": (
                scored_partition
            ),
            "component": component,
            **residual_summary,
            "status": "PASS",
        }
    )

    return log_score


for scored_partition in ANALYTICAL_PARTITIONS:
    grid = NATIVE_COUNT_GRIDS[
        scored_partition
    ]

    exposure_seconds = (
        grid_exposure_seconds(grid)
    )

    require(
        np.isfinite(
            exposure_seconds
        ).all(),
        (
            f"{scored_partition} native-grid exposure contains "
            "nonfinite values."
        ),
    )
    require(
        np.all(exposure_seconds > 0.0),
        (
            f"{scored_partition} native-grid exposure contains "
            "nonpositive values."
        ),
    )

    for model in (
        HOMOGENEOUS_POISSON_MODELS.values()
    ):
        if model.score_space == "POOLED_COUNT":
            require(
                model.pooled_rate_per_second
                is not None,
                (
                    f"{model.model_id} lacks its pooled rate."
                ),
            )

            pooled_expected_counts = (
                model.pooled_rate_per_second
                * exposure_seconds
            )

            append_score_row(
                model=model,
                scored_partition=(
                    scored_partition
                ),
                component="POOLED",
                observed_counts=(
                    grid.pooled_counts
                ),
                expected_counts=(
                    pooled_expected_counts
                ),
                component_parameter_count=(
                    model.parameter_count
                ),
            )

        elif (
            model.score_space
            == "BIVARIATE_SIDE_COUNT"
        ):
            require(
                model.buy_rate_per_second
                is not None,
                (
                    f"{model.model_id} lacks its BUY rate."
                ),
            )
            require(
                model.sell_rate_per_second
                is not None,
                (
                    f"{model.model_id} lacks its SELL rate."
                ),
            )

            buy_expected_counts = (
                model.buy_rate_per_second
                * exposure_seconds
            )

            sell_expected_counts = (
                model.sell_rate_per_second
                * exposure_seconds
            )

            buy_log_score = append_score_row(
                model=model,
                scored_partition=(
                    scored_partition
                ),
                component="BUY",
                observed_counts=(
                    grid.buy_counts
                ),
                expected_counts=(
                    buy_expected_counts
                ),
                component_parameter_count=1,
            )

            sell_log_score = append_score_row(
                model=model,
                scored_partition=(
                    scored_partition
                ),
                component="SELL",
                observed_counts=(
                    grid.sell_counts
                ),
                expected_counts=(
                    sell_expected_counts
                ),
                component_parameter_count=1,
            )

            observed_event_count = int(
                grid.buy_counts.sum(
                    dtype="int64"
                )
                + grid.sell_counts.sum(
                    dtype="int64"
                )
            )

            predicted_event_count = float(
                buy_expected_counts.sum(
                    dtype="float64"
                )
                + sell_expected_counts.sum(
                    dtype="float64"
                )
            )

            aggregate_log_score = (
                buy_log_score
                + sell_log_score
            )

            partition_exposure_seconds = (
                grid.contract_duration_ns
                / NANOSECONDS_PER_SECOND
            )

            homogeneous_score_rows.append(
                {
                    "model_id": model.model_id,
                    "model_family": (
                        model.model_family
                    ),
                    "fitted_partition": (
                        model.fitted_partition
                    ),
                    "scored_partition": (
                        scored_partition
                    ),
                    "score_space": (
                        model.score_space
                    ),
                    "component": (
                        "BUY_AND_SELL_AGGREGATE"
                    ),
                    "parameter_count": (
                        model.parameter_count
                    ),
                    "cell_count": int(
                        2 * grid.cell_count
                    ),
                    "exposure_seconds": (
                        partition_exposure_seconds
                    ),
                    "observed_event_count": (
                        observed_event_count
                    ),
                    "predicted_event_count": (
                        predicted_event_count
                    ),
                    "observed_minus_predicted": (
                        observed_event_count
                        - predicted_event_count
                    ),
                    "log_score_total": (
                        aggregate_log_score
                    ),
                    "log_score_per_second": (
                        aggregate_log_score
                        / partition_exposure_seconds
                    ),
                    "log_score_per_observed_event": (
                        aggregate_log_score
                        / observed_event_count
                    ),
                    "locked_before_calibration": True,
                    "status": "PASS",
                }
            )

        else:
            raise ValueError(
                (
                    "Unsupported homogeneous score space: "
                    f"{model.score_space}"
                )
            )


HOMOGENEOUS_POISSON_SCORE_TABLE = pd.DataFrame(
    homogeneous_score_rows
).sort_values(
    [
        "scored_partition",
        "score_space",
        "model_id",
        "component",
    ],
    kind="stable",
).reset_index(drop=True)

HOMOGENEOUS_POISSON_RESIDUAL_SUMMARY = pd.DataFrame(
    homogeneous_residual_rows
).sort_values(
    [
        "scored_partition",
        "model_id",
        "component",
    ],
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# Common-score comparison: equal-side versus side-specific
# ------------------------------------------------------------

aggregate_side_scores = (
    HOMOGENEOUS_POISSON_SCORE_TABLE.loc[
        HOMOGENEOUS_POISSON_SCORE_TABLE[
            "component"
        ].eq("BUY_AND_SELL_AGGREGATE")
    ]
    .copy()
)

side_score_pivot = (
    aggregate_side_scores.pivot(
        index="scored_partition",
        columns="model_id",
        values="log_score_total",
    )
)

require(
    {
        "B0_EQUAL_SIDE_EXTENSION",
        "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON",
    }.issubset(
        set(side_score_pivot.columns)
    ),
    "Required bivariate homogeneous scores are missing.",
)

HOMOGENEOUS_SIDE_RATE_COMPARISON = (
    side_score_pivot.reset_index()
)

HOMOGENEOUS_SIDE_RATE_COMPARISON[
    "b1_minus_b0_log_score"
] = (
    HOMOGENEOUS_SIDE_RATE_COMPARISON[
        "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
    ]
    - HOMOGENEOUS_SIDE_RATE_COMPARISON[
        "B0_EQUAL_SIDE_EXTENSION"
    ]
)

HOMOGENEOUS_SIDE_RATE_COMPARISON[
    "preferred_model"
] = np.where(
    HOMOGENEOUS_SIDE_RATE_COMPARISON[
        "b1_minus_b0_log_score"
    ]
    > 0.0,
    "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON",
    "B0_EQUAL_SIDE_EXTENSION",
)


# ------------------------------------------------------------
# Homogeneous-baseline gates
# ------------------------------------------------------------

calibration_aggregate_score_count = int(
    HOMOGENEOUS_POISSON_SCORE_TABLE.loc[
        HOMOGENEOUS_POISSON_SCORE_TABLE[
            "scored_partition"
        ].eq("CALIBRATION")
        & HOMOGENEOUS_POISSON_SCORE_TABLE[
            "component"
        ].isin(
            {
                "POOLED",
                "BUY_AND_SELL_AGGREGATE",
            }
        )
    ].shape[0]
)

HOMOGENEOUS_POISSON_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "native_grid_exact_exposure_conservation",
            "severity": "BLOCKING",
            "passed": bool(
                NATIVE_COUNT_GRID_AUDIT[
                    "exposure_difference_ns"
                ].eq(0).all()
            ),
            "evidence": (
                "integer nanosecond exposure difference is zero "
                "for every analytical partition"
            ),
        },
        {
            "gate": "native_grid_event_count_conservation",
            "severity": "BLOCKING",
            "passed": bool(
                (
                    NATIVE_COUNT_GRID_AUDIT[
                        "buy_event_count"
                    ]
                    + NATIVE_COUNT_GRID_AUDIT[
                        "sell_event_count"
                    ]
                )
                .eq(
                    NATIVE_COUNT_GRID_AUDIT[
                        "pooled_event_count"
                    ]
                )
                .all()
            ),
            "evidence": (
                f"analytical_events="
                f"{int(NATIVE_COUNT_GRID_AUDIT['pooled_event_count'].sum())}"
            ),
        },
        {
            "gate": "homogeneous_parameters_finite_positive",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    [
                        DEVELOPMENT_POOLED_RATE_PER_SECOND,
                        DEVELOPMENT_BUY_RATE_PER_SECOND,
                        DEVELOPMENT_SELL_RATE_PER_SECOND,
                    ]
                ).all()
                and min(
                    DEVELOPMENT_POOLED_RATE_PER_SECOND,
                    DEVELOPMENT_BUY_RATE_PER_SECOND,
                    DEVELOPMENT_SELL_RATE_PER_SECOND,
                )
                > 0.0
            ),
            "evidence": (
                "DEVELOPMENT pooled, BUY, and SELL rates are "
                "finite and positive"
            ),
        },
        {
            "gate": "all_homogeneous_scores_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    HOMOGENEOUS_POISSON_SCORE_TABLE[
                        "log_score_total"
                    ].to_numpy(dtype="float64")
                ).all()
            ),
            "evidence": (
                f"score_rows="
                f"{len(HOMOGENEOUS_POISSON_SCORE_TABLE)}"
            ),
        },
        {
            "gate": "locked_calibration_scores_complete",
            "severity": "BLOCKING",
            "passed": (
                calibration_aggregate_score_count
                == 3
            ),
            "evidence": (
                f"calibration_aggregate_scores="
                f"{calibration_aggregate_score_count}"
            ),
        },
        {
            "gate": "development_parameters_used_for_calibration",
            "severity": "BLOCKING",
            "passed": bool(
                HOMOGENEOUS_POISSON_SCORE_TABLE[
                    "fitted_partition"
                ]
                .eq("DEVELOPMENT")
                .all()
                and HOMOGENEOUS_POISSON_SCORE_TABLE[
                    "locked_before_calibration"
                ]
                .all()
            ),
            "evidence": (
                "all CALIBRATION scores use frozen DEVELOPMENT "
                "rate estimates"
            ),
        },
        {
            "gate": "notebook_04_timestamp_authority_preserved",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "native grid uses canonical Notebook 04 "
                "event_time_ns"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    HOMOGENEOUS_POISSON_GATE_FRAME.loc[
        HOMOGENEOUS_POISSON_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    "At least one homogeneous-Poisson baseline gate failed.",
)


display(NATIVE_COUNT_GRID_AUDIT)
display(HOMOGENEOUS_POISSON_PARAMETER_TABLE)

display(
    HOMOGENEOUS_POISSON_SCORE_TABLE.loc[
        HOMOGENEOUS_POISSON_SCORE_TABLE[
            "component"
        ].isin(
            {
                "POOLED",
                "BUY_AND_SELL_AGGREGATE",
            }
        ),
        [
            "model_id",
            "scored_partition",
            "score_space",
            "component",
            "parameter_count",
            "observed_event_count",
            "predicted_event_count",
            "log_score_total",
            "log_score_per_second",
            "log_score_per_observed_event",
            "status",
        ],
    ]
)

display(HOMOGENEOUS_SIDE_RATE_COMPARISON)

display(
    HOMOGENEOUS_POISSON_RESIDUAL_SUMMARY[
        [
            "model_id",
            "scored_partition",
            "component",
            "pearson_residual_mean",
            "pearson_residual_variance",
            "deviance_residual_mean",
            "deviance_residual_variance",
            "pearson_dispersion_ratio",
            "maximum_absolute_pearson_residual",
            "status",
        ]
    ]
)

display(HOMOGENEOUS_POISSON_GATE_FRAME)

print(
    "Native one-millisecond count grids were constructed with "
    "exact integer-nanosecond exposure accounting."
)
print(
    "The final partial cell, where present, carries its exact "
    "remaining nanosecond exposure; no floating-point exposure "
    "sum is used as an authority check."
)
print(
    "Pooled and side-specific homogeneous Poisson parameters "
    "were estimated on DEVELOPMENT only."
)
print(
    "Locked DEVELOPMENT parameters were scored on CALIBRATION "
    "without refitting."
)
print(
    "Hawkes estimation remains unauthorized."
)

,event_partition,grid_width_ns,grid_width_ms,cell_count,complete_cell_count,partial_final_cell_flag,final_cell_exposure_ns,contract_duration_ns,exact_exposure_sum_ns,exposure_difference_ns,pooled_event_count,buy_event_count,sell_event_count,nonempty_pooled_cell_count,maximum_pooled_cell_count,status
0,DEVELOPMENT,1000000,1,1801860,1801859,True,586700,1801859586700,1801859586700,0,7004,3414,3590,6854,6,PASS
1,CALIBRATION,1000000,1,720300,720299,True,179100,720299179100,720299179100,0,2493,1230,1263,2396,5,PASS


,model_id,model_family,fitted_partition,score_space,parameter_count,pooled_rate_per_second,buy_rate_per_second,sell_rate_per_second,fit_exposure_seconds,fit_pooled_event_count,fit_buy_event_count,fit_sell_event_count,parameter_status
0,B0_POOLED_HOMOGENEOUS_POISSON,POOLED_HOMOGENEOUS_POISSON,DEVELOPMENT,POOLED_COUNT,1,3.8870953,NaN,NaN,"1,801.8596",7004,3414,3590,FINITE_POSITIVE
1,B0_EQUAL_SIDE_EXTENSION,EQUAL_SIDE_HOMOGENEOUS_POISSON,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,3.8870953,1.9435477,1.9435477,"1,801.8596",7004,3414,3590,FINITE_POSITIVE
2,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,INDEPENDENT_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,DEVELOPMENT,BIVARIATE_SIDE_COUNT,2,3.8870953,1.8947092,1.9923861,"1,801.8596",7004,3414,3590,FINITE_POSITIVE


,model_id,scored_partition,score_space,component,parameter_count,observed_event_count,predicted_event_count,log_score_total,log_score_per_second,log_score_per_observed_event,status
1,B0_EQUAL_SIDE_EXTENSION,CALIBRATION,BIVARIATE_SIDE_COUNT,BUY_AND_SELL_AGGREGATE,1,2493,"2,799.8716","-18,440.818",-25.601609,-7.397039,PASS
4,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,CALIBRATION,BIVARIATE_SIDE_COUNT,BUY_AND_SELL_AGGREGATE,2,2493,"2,799.8716","-18,440.776",-25.601551,-7.3970221,PASS
6,B0_POOLED_HOMOGENEOUS_POISSON,CALIBRATION,POOLED_COUNT,POOLED,1,2493,"2,799.8716","-16,717.367",-23.208921,-6.7057227,PASS
8,B0_EQUAL_SIDE_EXTENSION,DEVELOPMENT,BIVARIATE_SIDE_COUNT,BUY_AND_SELL_AGGREGATE,1,7004,"7,004","-50,833.603",-28.211745,-7.257796,PASS
11,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,DEVELOPMENT,BIVARIATE_SIDE_COUNT,BUY_AND_SELL_AGGREGATE,2,7004,"7,004","-50,831.392",-28.210518,-7.2574802,PASS
13,B0_POOLED_HOMOGENEOUS_POISSON,DEVELOPMENT,POOLED_COUNT,POOLED,1,7004,"7,004","-45,999.712",-25.529022,-6.5676346,PASS


model_id,scored_partition,B0_EQUAL_SIDE_EXTENSION,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,b1_minus_b0_log_score,preferred_model
0,CALIBRATION,"-18,440.818","-18,440.776",0.042074714,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON
1,DEVELOPMENT,"-50,833.603","-50,831.392",2.2115406,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON


,model_id,scored_partition,component,pearson_residual_mean,pearson_residual_variance,deviance_residual_mean,deviance_residual_variance,pearson_dispersion_ratio,maximum_absolute_pearson_residual,status
0,B0_EQUAL_SIDE_EXTENSION,CALIBRATION,BUY,-0.0053514942,1.037119,-0.056903708,0.018900838,1.0371476,113.37141,PASS
1,B0_EQUAL_SIDE_EXTENSION,CALIBRATION,SELL,-0.0043122851,0.92203209,-0.056589051,0.019125115,0.92205069,68.005213,PASS
2,B0_POOLED_HOMOGENEOUS_POISSON,CALIBRATION,POOLED,-0.0068333238,0.98303518,-0.077643272,0.033664325,0.98308187,80.134521,PASS
3,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,CALIBRATION,BUY,-0.0042980331,1.063852,-0.056103866,0.018980466,1.0638705,114.82438,PASS
4,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,CALIBRATION,SELL,-0.005353249,0.89943075,-0.057379508,0.019045671,0.89945941,67.165458,PASS
5,B0_EQUAL_SIDE_EXTENSION,DEVELOPMENT,BUY,-0.0011078112,1.0387015,-0.056175672,0.020746921,1.0387027,136.05451,PASS
6,B0_EQUAL_SIDE_EXTENSION,DEVELOPMENT,SELL,0.0011078024,1.0539259,-0.055806844,0.021732262,1.0539271,113.37141,PASS
7,B0_POOLED_HOMOGENEOUS_POISSON,DEVELOPMENT,POOLED,-6.2027359e-09,1.0540801,-0.076233796,0.037593084,1.0540801,96.173895,PASS
8,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,DEVELOPMENT,BUY,-4.3305393e-09,1.0654753,-0.055374224,0.020835085,1.0654753,137.79796,PASS
9,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,DEVELOPMENT,SELL,-4.4407615e-09,1.0280915,-0.056598929,0.02164201,1.0280915,111.97219,PASS


,gate,severity,passed,evidence
0,native_grid_exact_exposure_conservation,BLOCKING,True,integer nanosecond exposure difference is zero...
1,native_grid_event_count_conservation,BLOCKING,True,analytical_events=9497
2,homogeneous_parameters_finite_positive,BLOCKING,True,"DEVELOPMENT pooled, BUY, and SELL rates are fi..."
3,all_homogeneous_scores_finite,BLOCKING,True,score_rows=14
4,locked_calibration_scores_complete,BLOCKING,True,calibration_aggregate_scores=3
5,development_parameters_used_for_calibration,BLOCKING,True,all CALIBRATION scores use frozen DEVELOPMENT ...
6,notebook_04_timestamp_authority_preserved,BLOCKING,True,native grid uses canonical Notebook 04 event_t...
7,protected_partition_content_remains_absent,BLOCKING,True,VALIDATION=False; ENGINEERING_HOLDOUT=False


Native one-millisecond count grids were constructed with exact integer-nanosecond exposure accounting.
The final partial cell, where present, carries its exact remaining nanosecond exposure; no floating-point exposure sum is used as an authority check.
Pooled and side-specific homogeneous Poisson parameters were estimated on DEVELOPMENT only.
Locked DEVELOPMENT parameters were scored on CALIBRATION without refitting.
Hawkes estimation remains unauthorized.


In [11]:
# ============================================================
# Low-dimensional deterministic elapsed-time Poisson baselines
# ============================================================

DETERMINISTIC_SELECTION_RULE: Final[str] = (
    "MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICITY_TIE_BREAK"
)

DETERMINISTIC_BIC_TIE_THRESHOLD: Final[float] = 2.0

DETERMINISTIC_MODEL_IDS: Final[
    Mapping[str, str]
] = {
    "CONSTANT": (
        "D0_SIDE_CONSTANT_POISSON"
    ),
    "LINEAR_ELAPSED_TIME": (
        "D1_SIDE_LINEAR_ELAPSED_POISSON"
    ),
    "QUADRATIC_ELAPSED_TIME": (
        "D2_SIDE_QUADRATIC_ELAPSED_POISSON"
    ),
    "NATURAL_SPLINE_DF3": (
        "D3_SIDE_NATURAL_SPLINE_DF3_POISSON"
    ),
}

DETERMINISTIC_MODEL_COMPLEXITY_ORDER: Final[
    Mapping[str, int]
] = {
    "CONSTANT": 0,
    "LINEAR_ELAPSED_TIME": 1,
    "QUADRATIC_ELAPSED_TIME": 2,
    "NATURAL_SPLINE_DF3": 3,
}

NATURAL_SPLINE_KNOTS: Final[
    tuple[float, ...]
] = (
    -1.0,
    -1.0 / 3.0,
    1.0 / 3.0,
    1.0,
)

POISSON_OPTIMIZER_METHOD: Final[str] = (
    "L-BFGS-B"
)

POISSON_OPTIMIZER_MAX_ITERATIONS: Final[int] = 500
POISSON_OPTIMIZER_MAX_LINE_SEARCH_STEPS: Final[int] = 50
POISSON_OPTIMIZER_GRADIENT_TOLERANCE: Final[float] = 1e-7

POISSON_INTERCEPT_BOUNDS: Final[
    tuple[float, float]
] = (
    -10.0,
    10.0,
)

POISSON_NONINTERCEPT_BOUNDS: Final[
    tuple[float, float]
] = (
    -20.0,
    20.0,
)


# ------------------------------------------------------------
# Frozen elapsed-time coordinate
# ------------------------------------------------------------

FULL_REGISTERED_RUN_START_NS: Final[int] = int(
    CONTRACT_WINDOW_CONTINUITY[
        "contract_start_ns"
    ].min()
)

FULL_REGISTERED_RUN_END_EXCLUSIVE_NS: Final[int] = int(
    CONTRACT_WINDOW_CONTINUITY[
        "contract_end_exclusive_ns"
    ].max()
)

FULL_REGISTERED_RUN_DURATION_NS: Final[int] = (
    FULL_REGISTERED_RUN_END_EXCLUSIVE_NS
    - FULL_REGISTERED_RUN_START_NS
)

require(
    FULL_REGISTERED_RUN_DURATION_NS > 0,
    "Full registered run duration must be positive.",
)


def native_grid_elapsed_coordinate(
    grid: NativeCountGrid,
) -> np.ndarray:
    """
    Return cell-midpoint elapsed-time coordinates on [-1, 1].

    The coordinate is defined against the complete registered source-run
    interval using metadata only. Event content from protected partitions
    is neither loaded nor used.
    """
    cell_indices = np.arange(
        grid.cell_count,
        dtype="int64",
    )

    cell_start_offsets_ns = (
        grid.contract_start_ns
        - FULL_REGISTERED_RUN_START_NS
        + cell_indices * grid.grid_width_ns
    )

    cell_midpoint_offsets_ns = (
        cell_start_offsets_ns
        + grid.exposure_ns // 2
    )

    require(
        np.all(cell_midpoint_offsets_ns >= 0),
        (
            f"{grid.event_partition} contains a cell midpoint "
            "before the registered run start."
        ),
    )
    require(
        np.all(
            cell_midpoint_offsets_ns
            < FULL_REGISTERED_RUN_DURATION_NS
        ),
        (
            f"{grid.event_partition} contains a cell midpoint "
            "at or beyond the registered run end."
        ),
    )

    coordinate = (
        2.0
        * (
            cell_midpoint_offsets_ns.astype(
                "float64"
            )
            / FULL_REGISTERED_RUN_DURATION_NS
        )
        - 1.0
    )

    require(
        np.isfinite(coordinate).all(),
        (
            f"{grid.event_partition} elapsed-time coordinate "
            "contains nonfinite values."
        ),
    )
    require(
        np.all(coordinate >= -1.0)
        and np.all(coordinate < 1.0),
        (
            f"{grid.event_partition} elapsed-time coordinate "
            "falls outside [-1, 1)."
        ),
    )

    coordinate.setflags(write=False)

    return coordinate


NATIVE_GRID_ELAPSED_COORDINATES: Final[
    Mapping[str, np.ndarray]
] = {
    partition_name: native_grid_elapsed_coordinate(
        grid
    )
    for partition_name, grid
    in NATIVE_COUNT_GRIDS.items()
}


# ------------------------------------------------------------
# Natural cubic spline basis
# ------------------------------------------------------------

def positive_cubic_part(
    values: np.ndarray,
    knot: float,
) -> np.ndarray:
    """Return max(values - knot, 0)^3."""
    return np.maximum(
        values - knot,
        0.0,
    ) ** 3


def natural_cubic_spline_df3_basis(
    coordinate: np.ndarray,
    *,
    knots: Sequence[float] = (
        NATURAL_SPLINE_KNOTS
    ),
) -> np.ndarray:
    """
    Return three nonintercept natural-cubic-spline columns.

    With four ordered knots, the basis contains:
    - the linear coordinate;
    - two nonlinear natural-spline terms.

    The complete regression design adds one intercept, yielding four
    parameters per event side.
    """
    x = np.asarray(
        coordinate,
        dtype="float64",
    )

    knot_array = np.asarray(
        knots,
        dtype="float64",
    )

    require(
        x.ndim == 1,
        "Spline coordinate must be one-dimensional.",
    )
    require(
        knot_array.shape == (4,),
        "Natural spline DF3 requires exactly four knots.",
    )
    require(
        np.all(
            np.diff(knot_array) > 0.0
        ),
        "Natural spline knots must be strictly increasing.",
    )

    final_knot = float(
        knot_array[-1]
    )
    penultimate_knot = float(
        knot_array[-2]
    )

    def scaled_truncated_cubic(
        knot: float,
    ) -> np.ndarray:
        denominator = (
            final_knot - knot
        )

        require(
            denominator > 0.0,
            "Natural spline denominator must be positive.",
        )

        return (
            positive_cubic_part(
                x,
                knot,
            )
            - positive_cubic_part(
                x,
                final_knot,
            )
        ) / denominator

    reference_term = scaled_truncated_cubic(
        penultimate_knot
    )

    nonlinear_terms = [
        scaled_truncated_cubic(
            float(knot)
        )
        - reference_term
        for knot in knot_array[:-2]
    ]

    basis = np.column_stack(
        [
            x,
            *nonlinear_terms,
        ]
    ).astype(
        "float64",
        copy=False,
    )

    require(
        basis.shape == (
            x.size,
            3,
        ),
        "Natural spline DF3 basis has an invalid shape.",
    )
    require(
        np.isfinite(basis).all(),
        "Natural spline basis contains nonfinite values.",
    )

    return basis


# ------------------------------------------------------------
# Deterministic design matrices
# ------------------------------------------------------------

def deterministic_design_matrix(
    coordinate: np.ndarray,
    *,
    specification: str,
) -> tuple[np.ndarray, tuple[str, ...]]:
    """Construct a frozen low-dimensional Poisson design matrix."""
    x = np.asarray(
        coordinate,
        dtype="float64",
    )

    require(
        x.ndim == 1,
        "Deterministic coordinate must be one-dimensional.",
    )

    intercept = np.ones(
        x.size,
        dtype="float64",
    )

    if specification == "CONSTANT":
        design = intercept[:, None]
        column_names = (
            "intercept",
        )

    elif specification == "LINEAR_ELAPSED_TIME":
        design = np.column_stack(
            [
                intercept,
                x,
            ]
        )
        column_names = (
            "intercept",
            "elapsed_coordinate",
        )

    elif specification == "QUADRATIC_ELAPSED_TIME":
        design = np.column_stack(
            [
                intercept,
                x,
                x * x,
            ]
        )
        column_names = (
            "intercept",
            "elapsed_coordinate",
            "elapsed_coordinate_squared",
        )

    elif specification == "NATURAL_SPLINE_DF3":
        spline_basis = (
            natural_cubic_spline_df3_basis(
                x
            )
        )

        design = np.column_stack(
            [
                intercept,
                spline_basis,
            ]
        )
        column_names = (
            "intercept",
            "elapsed_coordinate",
            "natural_spline_term_1",
            "natural_spline_term_2",
        )

    else:
        raise ValueError(
            (
                "Unsupported deterministic-rate "
                f"specification: {specification}"
            )
        )

    require(
        design.shape[0] == x.size,
        "Deterministic design row count is incorrect.",
    )
    require(
        design.shape[1] == len(
            column_names
        ),
        "Deterministic design column count is incorrect.",
    )
    require(
        np.isfinite(design).all(),
        (
            f"{specification} design matrix contains "
            "nonfinite values."
        ),
    )
    require(
        np.linalg.matrix_rank(design)
        == design.shape[1],
        (
            f"{specification} DEVELOPMENT design matrix is "
            "rank deficient."
        ),
    )

    return (
        np.ascontiguousarray(
            design,
            dtype="float64",
        ),
        column_names,
    )


DETERMINISTIC_DESIGN_MATRICES: dict[
    tuple[str, str],
    np.ndarray,
] = {}

DETERMINISTIC_DESIGN_COLUMN_NAMES: dict[
    str,
    tuple[str, ...],
] = {}

for specification in (
    DETERMINISTIC_RATE_SPECIFICATIONS
):
    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        design_matrix, design_columns = (
            deterministic_design_matrix(
                NATIVE_GRID_ELAPSED_COORDINATES[
                    partition_name
                ],
                specification=specification,
            )
        )

        DETERMINISTIC_DESIGN_MATRICES[
            (
                specification,
                partition_name,
            )
        ] = design_matrix

        if specification in (
            DETERMINISTIC_DESIGN_COLUMN_NAMES
        ):
            require(
                DETERMINISTIC_DESIGN_COLUMN_NAMES[
                    specification
                ]
                == design_columns,
                (
                    f"{specification} design-column contract "
                    "differs across partitions."
                ),
            )
        else:
            DETERMINISTIC_DESIGN_COLUMN_NAMES[
                specification
            ] = design_columns


# ------------------------------------------------------------
# Poisson maximum-likelihood estimator
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class PoissonComponentFit:
    side: str
    specification: str
    coefficient_names: tuple[str, ...]
    coefficients: tuple[float, ...]
    parameter_count: int
    development_event_count: int
    development_exposure_seconds: float
    development_log_score: float
    optimizer_method: str
    optimizer_success: bool
    optimizer_status: int
    optimizer_message: str
    optimizer_iterations: int
    objective_evaluations: int
    gradient_evaluations: int
    maximum_absolute_gradient: float


@dataclass(frozen=True, slots=True)
class DeterministicPoissonModel:
    model_id: str
    specification: str
    model_family: str
    fitted_partition: str
    score_space: str
    coefficient_names: tuple[str, ...]
    buy_fit: PoissonComponentFit
    sell_fit: PoissonComponentFit
    parameter_count: int
    selection_complexity_order: int


def poisson_negative_log_likelihood_and_gradient(
    coefficients: np.ndarray,
    design_matrix: np.ndarray,
    observed_counts: np.ndarray,
    exposure_seconds: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Return Poisson negative log-likelihood and analytic gradient."""
    beta = np.asarray(
        coefficients,
        dtype="float64",
    )
    design = np.asarray(
        design_matrix,
        dtype="float64",
    )
    observed = np.asarray(
        observed_counts,
        dtype="float64",
    )
    exposure = np.asarray(
        exposure_seconds,
        dtype="float64",
    )

    linear_predictor = (
        design @ beta
    )

    if not np.isfinite(
        linear_predictor
    ).all():
        return (
            float("inf"),
            np.full_like(
                beta,
                np.nan,
            ),
        )

    maximum_linear_predictor = float(
        np.max(linear_predictor)
    )
    minimum_linear_predictor = float(
        np.min(linear_predictor)
    )

    if (
        maximum_linear_predictor > 50.0
        or minimum_linear_predictor < -50.0
    ):
        return (
            float("inf"),
            np.full_like(
                beta,
                np.nan,
            ),
        )

    expected_counts = (
        exposure
        * np.exp(linear_predictor)
    )

    if (
        not np.isfinite(
            expected_counts
        ).all()
        or np.any(expected_counts <= 0.0)
    ):
        return (
            float("inf"),
            np.full_like(
                beta,
                np.nan,
            ),
        )

    negative_log_likelihood = float(
        (
            expected_counts
            - observed * linear_predictor
            + special.gammaln(
                observed + 1.0
            )
        ).sum(
            dtype="float64"
        )
    )

    gradient = (
        design.T
        @ (
            expected_counts
            - observed
        )
    )

    if (
        not np.isfinite(
            negative_log_likelihood
        )
        or not np.isfinite(
            gradient
        ).all()
    ):
        return (
            float("inf"),
            np.full_like(
                beta,
                np.nan,
            ),
        )

    return (
        negative_log_likelihood,
        gradient.astype(
            "float64",
            copy=False,
        ),
    )


def fit_poisson_component(
    *,
    side: str,
    specification: str,
    design_matrix: np.ndarray,
    observed_counts: np.ndarray,
    exposure_seconds: np.ndarray,
    constant_rate_per_second: float,
) -> PoissonComponentFit:
    """Fit one deterministic Poisson intensity component."""
    design = np.asarray(
        design_matrix,
        dtype="float64",
    )
    observed = np.asarray(
        observed_counts,
        dtype="int64",
    )
    exposure = np.asarray(
        exposure_seconds,
        dtype="float64",
    )

    require(
        side in {"BUY", "SELL"},
        f"Unsupported Poisson component side: {side}",
    )
    require(
        observed.shape == exposure.shape,
        (
            f"{side} observed-count and exposure arrays "
            "differ in shape."
        ),
    )
    require(
        design.shape[0] == observed.size,
        (
            f"{side} design row count differs from its "
            "count-array length."
        ),
    )
    require(
        np.all(observed >= 0),
        f"{side} observed counts contain negative values.",
    )
    require(
        np.isfinite(exposure).all()
        and np.all(exposure > 0.0),
        f"{side} exposure contains invalid values.",
    )
    require(
        np.isfinite(
            constant_rate_per_second
        )
        and constant_rate_per_second > 0.0,
        (
            f"{side} initial constant rate must be finite "
            "and positive."
        ),
    )

    parameter_count = int(
        design.shape[1]
    )

    initial_coefficients = np.zeros(
        parameter_count,
        dtype="float64",
    )
    initial_coefficients[0] = math.log(
        constant_rate_per_second
    )

    coefficient_bounds = [
        POISSON_INTERCEPT_BOUNDS,
        *[
            POISSON_NONINTERCEPT_BOUNDS
            for _ in range(
                parameter_count - 1
            )
        ],
    ]

    if specification == "CONSTANT":
        fitted_coefficients = (
            initial_coefficients
        )
        optimizer_success = True
        optimizer_status = 0
        optimizer_message = (
            "CLOSED_FORM_CONSTANT_RATE"
        )
        optimizer_iterations = 0
        objective_evaluations = 1
        gradient_evaluations = 1

    else:
        def objective(
            beta: np.ndarray,
        ) -> tuple[float, np.ndarray]:
            return (
                poisson_negative_log_likelihood_and_gradient(
                    beta,
                    design,
                    observed,
                    exposure,
                )
            )

        optimization_result = (
            optimize.minimize(
                fun=objective,
                x0=initial_coefficients,
                method=(
                    POISSON_OPTIMIZER_METHOD
                ),
                jac=True,
                bounds=coefficient_bounds,
                options={
                    "maxiter": (
                        POISSON_OPTIMIZER_MAX_ITERATIONS
                    ),
                    "maxls": (
                        POISSON_OPTIMIZER_MAX_LINE_SEARCH_STEPS
                    ),
                    "ftol": 1e-12,
                    "gtol": (
                        POISSON_OPTIMIZER_GRADIENT_TOLERANCE
                    ),
                },
            )
        )

        fitted_coefficients = np.asarray(
            optimization_result.x,
            dtype="float64",
        )

        optimizer_success = bool(
            optimization_result.success
        )
        optimizer_status = int(
            optimization_result.status
        )
        optimizer_message = str(
            optimization_result.message
        )
        optimizer_iterations = int(
            getattr(
                optimization_result,
                "nit",
                -1,
            )
        )
        objective_evaluations = int(
            getattr(
                optimization_result,
                "nfev",
                -1,
            )
        )
        gradient_evaluations = int(
            getattr(
                optimization_result,
                "njev",
                -1,
            )
        )

    final_objective, final_gradient = (
        poisson_negative_log_likelihood_and_gradient(
            fitted_coefficients,
            design,
            observed,
            exposure,
        )
    )

    require(
        np.isfinite(
            fitted_coefficients
        ).all(),
        (
            f"{side} {specification} coefficients contain "
            "nonfinite values."
        ),
    )
    require(
        np.isfinite(final_objective),
        (
            f"{side} {specification} final objective is "
            "nonfinite."
        ),
    )
    require(
        np.isfinite(final_gradient).all(),
        (
            f"{side} {specification} final gradient contains "
            "nonfinite values."
        ),
    )

    maximum_absolute_gradient = float(
        np.max(
            np.abs(final_gradient)
        )
    )

    convergence_accepted = (
        optimizer_success
        or maximum_absolute_gradient
        <= 1e-5
    )

    require(
        convergence_accepted,
        (
            f"{side} {specification} Poisson optimization "
            "did not converge: "
            f"status={optimizer_status}; "
            f"message={optimizer_message}; "
            f"max_abs_gradient="
            f"{maximum_absolute_gradient:.12g}"
        ),
    )

    fitted_expected_counts = (
        exposure
        * np.exp(
            design
            @ fitted_coefficients
        )
    )

    require(
        np.isfinite(
            fitted_expected_counts
        ).all(),
        (
            f"{side} {specification} expected counts contain "
            "nonfinite values."
        ),
    )
    require(
        np.all(
            fitted_expected_counts > 0.0
        ),
        (
            f"{side} {specification} expected counts are not "
            "strictly positive."
        ),
    )

    development_log_score = (
        poisson_count_log_score(
            observed,
            fitted_expected_counts,
        )
    )

    return PoissonComponentFit(
        side=side,
        specification=specification,
        coefficient_names=(
            DETERMINISTIC_DESIGN_COLUMN_NAMES[
                specification
            ]
        ),
        coefficients=tuple(
            float(value)
            for value in fitted_coefficients
        ),
        parameter_count=parameter_count,
        development_event_count=int(
            observed.sum(
                dtype="int64"
            )
        ),
        development_exposure_seconds=float(
            exposure.sum(
                dtype="float64"
            )
        ),
        development_log_score=(
            development_log_score
        ),
        optimizer_method=(
            POISSON_OPTIMIZER_METHOD
            if specification != "CONSTANT"
            else "CLOSED_FORM"
        ),
        optimizer_success=(
            convergence_accepted
        ),
        optimizer_status=(
            optimizer_status
        ),
        optimizer_message=(
            optimizer_message
        ),
        optimizer_iterations=(
            optimizer_iterations
        ),
        objective_evaluations=(
            objective_evaluations
        ),
        gradient_evaluations=(
            gradient_evaluations
        ),
        maximum_absolute_gradient=(
            maximum_absolute_gradient
        ),
    )


# ------------------------------------------------------------
# Fit every candidate on DEVELOPMENT only
# ------------------------------------------------------------

development_design_exposure_seconds = (
    grid_exposure_seconds(
        development_grid
    )
)

DETERMINISTIC_POISSON_MODELS: dict[
    str,
    DeterministicPoissonModel,
] = {}

for specification in (
    DETERMINISTIC_RATE_SPECIFICATIONS
):
    development_design = (
        DETERMINISTIC_DESIGN_MATRICES[
            (
                specification,
                "DEVELOPMENT",
            )
        ]
    )

    buy_fit = fit_poisson_component(
        side="BUY",
        specification=specification,
        design_matrix=development_design,
        observed_counts=(
            development_grid.buy_counts
        ),
        exposure_seconds=(
            development_design_exposure_seconds
        ),
        constant_rate_per_second=(
            DEVELOPMENT_BUY_RATE_PER_SECOND
        ),
    )

    sell_fit = fit_poisson_component(
        side="SELL",
        specification=specification,
        design_matrix=development_design,
        observed_counts=(
            development_grid.sell_counts
        ),
        exposure_seconds=(
            development_design_exposure_seconds
        ),
        constant_rate_per_second=(
            DEVELOPMENT_SELL_RATE_PER_SECOND
        ),
    )

    require(
        buy_fit.coefficient_names
        == sell_fit.coefficient_names,
        (
            f"{specification} BUY and SELL coefficient "
            "contracts differ."
        ),
    )

    model_id = (
        DETERMINISTIC_MODEL_IDS[
            specification
        ]
    )

    DETERMINISTIC_POISSON_MODELS[
        model_id
    ] = DeterministicPoissonModel(
        model_id=model_id,
        specification=specification,
        model_family=(
            "INDEPENDENT_SIDE_SPECIFIC_"
            "DETERMINISTIC_ELAPSED_TIME_POISSON"
        ),
        fitted_partition="DEVELOPMENT",
        score_space="BIVARIATE_SIDE_COUNT",
        coefficient_names=(
            buy_fit.coefficient_names
        ),
        buy_fit=buy_fit,
        sell_fit=sell_fit,
        parameter_count=(
            buy_fit.parameter_count
            + sell_fit.parameter_count
        ),
        selection_complexity_order=(
            DETERMINISTIC_MODEL_COMPLEXITY_ORDER[
                specification
            ]
        ),
    )


# ------------------------------------------------------------
# DEVELOPMENT-only model selection
# ------------------------------------------------------------

deterministic_selection_rows: list[
    dict[str, Any]
] = []

development_side_observation_count = int(
    2 * development_grid.cell_count
)

for model in (
    DETERMINISTIC_POISSON_MODELS.values()
):
    development_joint_log_score = (
        model.buy_fit.development_log_score
        + model.sell_fit.development_log_score
    )

    parameter_count = (
        model.parameter_count
    )

    aic = (
        -2.0
        * development_joint_log_score
        + 2.0 * parameter_count
    )

    bic = (
        -2.0
        * development_joint_log_score
        + parameter_count
        * math.log(
            development_side_observation_count
        )
    )

    deterministic_selection_rows.append(
        {
            "model_id": model.model_id,
            "specification": (
                model.specification
            ),
            "parameter_count": (
                parameter_count
            ),
            "selection_complexity_order": (
                model.selection_complexity_order
            ),
            "development_side_observation_count": (
                development_side_observation_count
            ),
            "development_joint_log_score": (
                development_joint_log_score
            ),
            "development_log_score_per_second": (
                development_joint_log_score
                / development_exposure_seconds
            ),
            "development_aic": aic,
            "development_bic": bic,
            "selection_partition": "DEVELOPMENT",
            "calibration_used_for_selection": False,
        }
    )


DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE = (
    pd.DataFrame(
        deterministic_selection_rows
    )
    .sort_values(
        [
            "development_bic",
            "parameter_count",
            "selection_complexity_order",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

best_development_bic = float(
    DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
        "development_bic"
    ].min()
)

DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
    "delta_bic_from_best"
] = (
    DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
        "development_bic"
    ]
    - best_development_bic
)

bic_tied_candidates = (
    DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE.loc[
        DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
            "delta_bic_from_best"
        ]
        <= DETERMINISTIC_BIC_TIE_THRESHOLD
    ]
    .sort_values(
        [
            "parameter_count",
            "selection_complexity_order",
            "development_bic",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    not bic_tied_candidates.empty,
    "Deterministic model selection produced no candidate.",
)

SELECTED_DETERMINISTIC_MODEL_ID: Final[str] = str(
    bic_tied_candidates.iloc[0][
        "model_id"
    ]
)

SELECTED_DETERMINISTIC_MODEL = (
    DETERMINISTIC_POISSON_MODELS[
        SELECTED_DETERMINISTIC_MODEL_ID
    ]
)

DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
    "selected_model_flag"
] = (
    DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
        "model_id"
    ].eq(
        SELECTED_DETERMINISTIC_MODEL_ID
    )
)

DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
    "selection_rule"
] = DETERMINISTIC_SELECTION_RULE


# ------------------------------------------------------------
# Parameter table
# ------------------------------------------------------------

deterministic_parameter_rows: list[
    dict[str, Any]
] = []

for model in (
    DETERMINISTIC_POISSON_MODELS.values()
):
    for component_fit in (
        model.buy_fit,
        model.sell_fit,
    ):
        for coefficient_name, coefficient_value in zip(
            component_fit.coefficient_names,
            component_fit.coefficients,
            strict=True,
        ):
            deterministic_parameter_rows.append(
                {
                    "model_id": (
                        model.model_id
                    ),
                    "specification": (
                        model.specification
                    ),
                    "side": (
                        component_fit.side
                    ),
                    "coefficient_name": (
                        coefficient_name
                    ),
                    "coefficient_value": (
                        coefficient_value
                    ),
                    "fitted_partition": (
                        model.fitted_partition
                    ),
                    "parameter_count_model": (
                        model.parameter_count
                    ),
                    "optimizer_method": (
                        component_fit.optimizer_method
                    ),
                    "optimizer_success": (
                        component_fit.optimizer_success
                    ),
                    "optimizer_status": (
                        component_fit.optimizer_status
                    ),
                    "optimizer_message": (
                        component_fit.optimizer_message
                    ),
                    "optimizer_iterations": (
                        component_fit.optimizer_iterations
                    ),
                    "maximum_absolute_gradient": (
                        component_fit.maximum_absolute_gradient
                    ),
                    "selected_model_flag": (
                        model.model_id
                        == SELECTED_DETERMINISTIC_MODEL_ID
                    ),
                    "status": "PASS",
                }
            )


DETERMINISTIC_POISSON_PARAMETER_TABLE = (
    pd.DataFrame(
        deterministic_parameter_rows
    )
    .sort_values(
        [
            "model_id",
            "side",
            "coefficient_name",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Locked DEVELOPMENT and CALIBRATION scoring
# ------------------------------------------------------------

def deterministic_expected_counts(
    *,
    model: DeterministicPoissonModel,
    partition_name: str,
    side: str,
) -> tuple[np.ndarray, np.ndarray]:
    """Return cell intensities and expected counts for one model component."""
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized partition requested for deterministic "
            f"scoring: {partition_name}"
        ),
    )
    require(
        side in {"BUY", "SELL"},
        f"Unsupported deterministic side: {side}",
    )

    design = (
        DETERMINISTIC_DESIGN_MATRICES[
            (
                model.specification,
                partition_name,
            )
        ]
    )

    component_fit = (
        model.buy_fit
        if side == "BUY"
        else model.sell_fit
    )

    coefficients = np.asarray(
        component_fit.coefficients,
        dtype="float64",
    )

    require(
        design.shape[1]
        == coefficients.size,
        (
            f"{model.model_id} {side} coefficient count differs "
            "from its design matrix."
        ),
    )

    linear_predictor = (
        design @ coefficients
    )

    intensity_per_second = np.exp(
        linear_predictor
    )

    exposure_seconds = (
        grid_exposure_seconds(
            NATIVE_COUNT_GRIDS[
                partition_name
            ]
        )
    )

    expected_counts = (
        intensity_per_second
        * exposure_seconds
    )

    require(
        np.isfinite(
            intensity_per_second
        ).all()
        and np.all(
            intensity_per_second > 0.0
        ),
        (
            f"{model.model_id} {side} intensity is invalid on "
            f"{partition_name}."
        ),
    )
    require(
        np.isfinite(
            expected_counts
        ).all()
        and np.all(
            expected_counts > 0.0
        ),
        (
            f"{model.model_id} {side} expected counts are "
            f"invalid on {partition_name}."
        ),
    )

    return (
        intensity_per_second,
        expected_counts,
    )


deterministic_score_rows: list[
    dict[str, Any]
] = []

deterministic_residual_rows: list[
    dict[str, Any]
] = []

deterministic_intensity_range_rows: list[
    dict[str, Any]
] = []

SELECTED_DETERMINISTIC_EXPECTED_COUNTS: dict[
    tuple[str, str],
    np.ndarray,
] = {}

SELECTED_DETERMINISTIC_INTENSITIES: dict[
    tuple[str, str],
    np.ndarray,
] = {}


for model in (
    DETERMINISTIC_POISSON_MODELS.values()
):
    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        grid = NATIVE_COUNT_GRIDS[
            partition_name
        ]

        component_log_scores: dict[
            str,
            float,
        ] = {}

        component_predicted_counts: dict[
            str,
            float,
        ] = {}

        for side in (
            "BUY",
            "SELL",
        ):
            observed_counts = (
                grid.buy_counts
                if side == "BUY"
                else grid.sell_counts
            )

            intensity_per_second, expected_counts = (
                deterministic_expected_counts(
                    model=model,
                    partition_name=(
                        partition_name
                    ),
                    side=side,
                )
            )

            log_score = (
                poisson_count_log_score(
                    observed_counts,
                    expected_counts,
                )
            )

            component_log_scores[
                side
            ] = log_score

            component_predicted_counts[
                side
            ] = float(
                expected_counts.sum(
                    dtype="float64"
                )
            )

            residual_summary = (
                summarize_poisson_residuals(
                    observed_counts,
                    expected_counts,
                    parameter_count=(
                        model.buy_fit.parameter_count
                        if side == "BUY"
                        else model.sell_fit.parameter_count
                    ),
                )
            )

            deterministic_score_rows.append(
                {
                    "model_id": (
                        model.model_id
                    ),
                    "specification": (
                        model.specification
                    ),
                    "fitted_partition": (
                        model.fitted_partition
                    ),
                    "scored_partition": (
                        partition_name
                    ),
                    "score_space": (
                        model.score_space
                    ),
                    "component": side,
                    "parameter_count": (
                        model.buy_fit.parameter_count
                        if side == "BUY"
                        else model.sell_fit.parameter_count
                    ),
                    "cell_count": (
                        grid.cell_count
                    ),
                    "exposure_seconds": (
                        grid.contract_duration_ns
                        / NANOSECONDS_PER_SECOND
                    ),
                    "observed_event_count": int(
                        observed_counts.sum(
                            dtype="int64"
                        )
                    ),
                    "predicted_event_count": (
                        component_predicted_counts[
                            side
                        ]
                    ),
                    "log_score_total": (
                        log_score
                    ),
                    "log_score_per_second": (
                        log_score
                        / (
                            grid.contract_duration_ns
                            / NANOSECONDS_PER_SECOND
                        )
                    ),
                    "log_score_per_observed_event": (
                        log_score
                        / int(
                            observed_counts.sum(
                                dtype="int64"
                            )
                        )
                    ),
                    "selected_model_flag": (
                        model.model_id
                        == SELECTED_DETERMINISTIC_MODEL_ID
                    ),
                    "locked_before_calibration": True,
                    "status": "PASS",
                }
            )

            deterministic_residual_rows.append(
                {
                    "model_id": (
                        model.model_id
                    ),
                    "specification": (
                        model.specification
                    ),
                    "scored_partition": (
                        partition_name
                    ),
                    "component": side,
                    "selected_model_flag": (
                        model.model_id
                        == SELECTED_DETERMINISTIC_MODEL_ID
                    ),
                    **residual_summary,
                    "status": "PASS",
                }
            )

            deterministic_intensity_range_rows.append(
                {
                    "model_id": (
                        model.model_id
                    ),
                    "specification": (
                        model.specification
                    ),
                    "scored_partition": (
                        partition_name
                    ),
                    "component": side,
                    "minimum_intensity_per_second": float(
                        np.min(
                            intensity_per_second
                        )
                    ),
                    "median_intensity_per_second": float(
                        np.median(
                            intensity_per_second
                        )
                    ),
                    "maximum_intensity_per_second": float(
                        np.max(
                            intensity_per_second
                        )
                    ),
                    "mean_intensity_per_second": float(
                        np.average(
                            intensity_per_second,
                            weights=(
                                grid.exposure_ns.astype(
                                    "float64"
                                )
                            ),
                        )
                    ),
                    "selected_model_flag": (
                        model.model_id
                        == SELECTED_DETERMINISTIC_MODEL_ID
                    ),
                    "status": "PASS",
                }
            )

            if (
                model.model_id
                == SELECTED_DETERMINISTIC_MODEL_ID
            ):
                expected_copy = np.asarray(
                    expected_counts,
                    dtype="float64",
                ).copy()

                intensity_copy = np.asarray(
                    intensity_per_second,
                    dtype="float64",
                ).copy()

                expected_copy.setflags(
                    write=False
                )
                intensity_copy.setflags(
                    write=False
                )

                SELECTED_DETERMINISTIC_EXPECTED_COUNTS[
                    (
                        partition_name,
                        side,
                    )
                ] = expected_copy

                SELECTED_DETERMINISTIC_INTENSITIES[
                    (
                        partition_name,
                        side,
                    )
                ] = intensity_copy

        joint_log_score = (
            component_log_scores["BUY"]
            + component_log_scores["SELL"]
        )

        joint_observed_count = int(
            grid.buy_counts.sum(
                dtype="int64"
            )
            + grid.sell_counts.sum(
                dtype="int64"
            )
        )

        joint_predicted_count = (
            component_predicted_counts[
                "BUY"
            ]
            + component_predicted_counts[
                "SELL"
            ]
        )

        partition_exposure_seconds = (
            grid.contract_duration_ns
            / NANOSECONDS_PER_SECOND
        )

        deterministic_score_rows.append(
            {
                "model_id": (
                    model.model_id
                ),
                "specification": (
                    model.specification
                ),
                "fitted_partition": (
                    model.fitted_partition
                ),
                "scored_partition": (
                    partition_name
                ),
                "score_space": (
                    model.score_space
                ),
                "component": (
                    "BUY_AND_SELL_AGGREGATE"
                ),
                "parameter_count": (
                    model.parameter_count
                ),
                "cell_count": int(
                    2 * grid.cell_count
                ),
                "exposure_seconds": (
                    partition_exposure_seconds
                ),
                "observed_event_count": (
                    joint_observed_count
                ),
                "predicted_event_count": (
                    joint_predicted_count
                ),
                "log_score_total": (
                    joint_log_score
                ),
                "log_score_per_second": (
                    joint_log_score
                    / partition_exposure_seconds
                ),
                "log_score_per_observed_event": (
                    joint_log_score
                    / joint_observed_count
                ),
                "selected_model_flag": (
                    model.model_id
                    == SELECTED_DETERMINISTIC_MODEL_ID
                ),
                "locked_before_calibration": True,
                "status": "PASS",
            }
        )


DETERMINISTIC_POISSON_SCORE_TABLE = pd.DataFrame(
    deterministic_score_rows
).sort_values(
    [
        "scored_partition",
        "model_id",
        "component",
    ],
    kind="stable",
).reset_index(drop=True)

DETERMINISTIC_POISSON_RESIDUAL_SUMMARY = (
    pd.DataFrame(
        deterministic_residual_rows
    )
    .sort_values(
        [
            "scored_partition",
            "model_id",
            "component",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

DETERMINISTIC_POISSON_INTENSITY_RANGES = (
    pd.DataFrame(
        deterministic_intensity_range_rows
    )
    .sort_values(
        [
            "scored_partition",
            "model_id",
            "component",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Reconcile constant candidate with prior B1 baseline
# ------------------------------------------------------------

constant_deterministic_scores = (
    DETERMINISTIC_POISSON_SCORE_TABLE.loc[
        DETERMINISTIC_POISSON_SCORE_TABLE[
            "model_id"
        ].eq(
            DETERMINISTIC_MODEL_IDS[
                "CONSTANT"
            ]
        )
        & DETERMINISTIC_POISSON_SCORE_TABLE[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        ),
        [
            "scored_partition",
            "log_score_total",
        ],
    ]
    .rename(
        columns={
            "log_score_total": (
                "deterministic_constant_log_score"
            )
        }
    )
)

prior_b1_scores = (
    HOMOGENEOUS_POISSON_SCORE_TABLE.loc[
        HOMOGENEOUS_POISSON_SCORE_TABLE[
            "model_id"
        ].eq(
            "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
        )
        & HOMOGENEOUS_POISSON_SCORE_TABLE[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        ),
        [
            "scored_partition",
            "log_score_total",
        ],
    ]
    .rename(
        columns={
            "log_score_total": (
                "prior_b1_log_score"
            )
        }
    )
)

DETERMINISTIC_CONSTANT_RECONCILIATION = (
    constant_deterministic_scores.merge(
        prior_b1_scores,
        on="scored_partition",
        how="inner",
        validate="one_to_one",
    )
)

DETERMINISTIC_CONSTANT_RECONCILIATION[
    "absolute_log_score_difference"
] = np.abs(
    DETERMINISTIC_CONSTANT_RECONCILIATION[
        "deterministic_constant_log_score"
    ]
    - DETERMINISTIC_CONSTANT_RECONCILIATION[
        "prior_b1_log_score"
    ]
)

DETERMINISTIC_CONSTANT_RECONCILIATION[
    "status"
] = np.where(
    DETERMINISTIC_CONSTANT_RECONCILIATION[
        "absolute_log_score_difference"
    ]
    <= 1e-8,
    "EXACT_WITHIN_NUMERICAL_TOLERANCE",
    "MISMATCH",
)

require(
    DETERMINISTIC_CONSTANT_RECONCILIATION[
        "status"
    ].eq(
        "EXACT_WITHIN_NUMERICAL_TOLERANCE"
    ).all(),
    (
        "Deterministic constant candidate does not reconcile "
        "with the previously established B1 baseline."
    ),
)


# ------------------------------------------------------------
# Locked calibration comparison
# ------------------------------------------------------------

deterministic_aggregate_scores = (
    DETERMINISTIC_POISSON_SCORE_TABLE.loc[
        DETERMINISTIC_POISSON_SCORE_TABLE[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        )
    ]
    .copy()
)

constant_score_lookup = (
    deterministic_aggregate_scores.loc[
        deterministic_aggregate_scores[
            "model_id"
        ].eq(
            DETERMINISTIC_MODEL_IDS[
                "CONSTANT"
            ]
        ),
        [
            "scored_partition",
            "log_score_total",
        ],
    ]
    .rename(
        columns={
            "log_score_total": (
                "constant_log_score_total"
            )
        }
    )
)

DETERMINISTIC_LOCKED_SCORE_COMPARISON = (
    deterministic_aggregate_scores.merge(
        constant_score_lookup,
        on="scored_partition",
        how="left",
        validate="many_to_one",
    )
)

DETERMINISTIC_LOCKED_SCORE_COMPARISON[
    "log_score_improvement_over_constant"
] = (
    DETERMINISTIC_LOCKED_SCORE_COMPARISON[
        "log_score_total"
    ]
    - DETERMINISTIC_LOCKED_SCORE_COMPARISON[
        "constant_log_score_total"
    ]
)

DETERMINISTIC_LOCKED_SCORE_COMPARISON[
    "selected_on_development_flag"
] = (
    DETERMINISTIC_LOCKED_SCORE_COMPARISON[
        "model_id"
    ].eq(
        SELECTED_DETERMINISTIC_MODEL_ID
    )
)

DETERMINISTIC_LOCKED_SCORE_COMPARISON = (
    DETERMINISTIC_LOCKED_SCORE_COMPARISON.sort_values(
        [
            "scored_partition",
            "log_score_total",
        ],
        ascending=[
            True,
            False,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Deterministic-model gates
# ------------------------------------------------------------

selected_model_score_rows = (
    DETERMINISTIC_POISSON_SCORE_TABLE.loc[
        DETERMINISTIC_POISSON_SCORE_TABLE[
            "model_id"
        ].eq(
            SELECTED_DETERMINISTIC_MODEL_ID
        )
        & DETERMINISTIC_POISSON_SCORE_TABLE[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        )
    ]
)

require(
    set(
        selected_model_score_rows[
            "scored_partition"
        ]
    )
    == set(ANALYTICAL_PARTITIONS),
    (
        "Selected deterministic model lacks DEVELOPMENT or "
        "CALIBRATION aggregate scoring."
    ),
)

DETERMINISTIC_POISSON_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "candidate_designs_full_rank",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                f"candidate_count="
                f"{len(DETERMINISTIC_RATE_SPECIFICATIONS)}"
            ),
        },
        {
            "gate": "all_candidates_fit_on_development_only",
            "severity": "BLOCKING",
            "passed": bool(
                all(
                    model.fitted_partition
                    == "DEVELOPMENT"
                    for model
                    in DETERMINISTIC_POISSON_MODELS.values()
                )
            ),
            "evidence": (
                "all candidate parameters estimated using "
                "DEVELOPMENT native-grid observations"
            ),
        },
        {
            "gate": "all_optimizers_converged",
            "severity": "BLOCKING",
            "passed": bool(
                DETERMINISTIC_POISSON_PARAMETER_TABLE[
                    "optimizer_success"
                ].all()
            ),
            "evidence": (
                f"parameter_rows="
                f"{len(DETERMINISTIC_POISSON_PARAMETER_TABLE)}"
            ),
        },
        {
            "gate": "all_parameters_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    DETERMINISTIC_POISSON_PARAMETER_TABLE[
                        "coefficient_value"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "all deterministic coefficients are finite"
            ),
        },
        {
            "gate": "all_intensities_finite_positive",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    DETERMINISTIC_POISSON_INTENSITY_RANGES[
                        [
                            "minimum_intensity_per_second",
                            "median_intensity_per_second",
                            "maximum_intensity_per_second",
                            "mean_intensity_per_second",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
                and DETERMINISTIC_POISSON_INTENSITY_RANGES[
                    "minimum_intensity_per_second"
                ].gt(0.0).all()
            ),
            "evidence": (
                f"intensity_range_rows="
                f"{len(DETERMINISTIC_POISSON_INTENSITY_RANGES)}"
            ),
        },
        {
            "gate": "development_selection_excludes_calibration",
            "severity": "BLOCKING",
            "passed": bool(
                not DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
                    "calibration_used_for_selection"
                ].any()
                and DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
                    "selection_partition"
                ].eq(
                    "DEVELOPMENT"
                ).all()
            ),
            "evidence": (
                f"selection_rule="
                f"{DETERMINISTIC_SELECTION_RULE}"
            ),
        },
        {
            "gate": "exactly_one_deterministic_model_selected",
            "severity": "BLOCKING",
            "passed": (
                int(
                    DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
                        "selected_model_flag"
                    ].sum()
                )
                == 1
            ),
            "evidence": (
                f"selected_model="
                f"{SELECTED_DETERMINISTIC_MODEL_ID}"
            ),
        },
        {
            "gate": "selected_model_scored_on_both_partitions",
            "severity": "BLOCKING",
            "passed": (
                len(
                    selected_model_score_rows
                )
                == len(
                    ANALYTICAL_PARTITIONS
                )
            ),
            "evidence": (
                "DEVELOPMENT and locked CALIBRATION aggregate "
                "scores are present"
            ),
        },
        {
            "gate": "constant_candidate_reconciles_with_b1",
            "severity": "BLOCKING",
            "passed": bool(
                DETERMINISTIC_CONSTANT_RECONCILIATION[
                    "status"
                ].eq(
                    "EXACT_WITHIN_NUMERICAL_TOLERANCE"
                ).all()
            ),
            "evidence": (
                "D0 constant scores reproduce prior B1 scores"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    DETERMINISTIC_POISSON_GATE_FRAME.loc[
        DETERMINISTIC_POISSON_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one deterministic elapsed-time Poisson "
        "baseline gate failed."
    ),
)


display(
    DETERMINISTIC_DEVELOPMENT_SELECTION_TABLE[
        [
            "model_id",
            "specification",
            "parameter_count",
            "development_joint_log_score",
            "development_aic",
            "development_bic",
            "delta_bic_from_best",
            "selected_model_flag",
            "selection_rule",
        ]
    ]
)

display(
    DETERMINISTIC_POISSON_PARAMETER_TABLE[
        [
            "model_id",
            "side",
            "coefficient_name",
            "coefficient_value",
            "optimizer_method",
            "optimizer_success",
            "optimizer_iterations",
            "maximum_absolute_gradient",
            "selected_model_flag",
            "status",
        ]
    ]
)

display(
    DETERMINISTIC_LOCKED_SCORE_COMPARISON[
        [
            "model_id",
            "specification",
            "scored_partition",
            "parameter_count",
            "observed_event_count",
            "predicted_event_count",
            "log_score_total",
            "log_score_per_second",
            "log_score_per_observed_event",
            "log_score_improvement_over_constant",
            "selected_on_development_flag",
            "status",
        ]
    ]
)

display(
    DETERMINISTIC_POISSON_INTENSITY_RANGES.loc[
        DETERMINISTIC_POISSON_INTENSITY_RANGES[
            "selected_model_flag"
        ],
        [
            "model_id",
            "scored_partition",
            "component",
            "minimum_intensity_per_second",
            "median_intensity_per_second",
            "maximum_intensity_per_second",
            "mean_intensity_per_second",
            "status",
        ],
    ]
)

display(
    DETERMINISTIC_POISSON_RESIDUAL_SUMMARY.loc[
        DETERMINISTIC_POISSON_RESIDUAL_SUMMARY[
            "selected_model_flag"
        ],
        [
            "model_id",
            "scored_partition",
            "component",
            "pearson_residual_mean",
            "pearson_residual_variance",
            "deviance_residual_mean",
            "deviance_residual_variance",
            "pearson_dispersion_ratio",
            "maximum_absolute_pearson_residual",
            "status",
        ],
    ]
)

display(
    DETERMINISTIC_CONSTANT_RECONCILIATION
)

display(
    DETERMINISTIC_POISSON_GATE_FRAME
)

print(
    "Low-dimensional deterministic elapsed-time Poisson "
    "candidates were fitted on DEVELOPMENT only."
)
print(
    f"Selected deterministic model: "
    f"{SELECTED_DETERMINISTIC_MODEL_ID}."
)
print(
    "Selection used DEVELOPMENT BIC with a delta-two "
    "simplicity tie-break; CALIBRATION was not used for model "
    "selection."
)
print(
    "The selected specification was then scored unchanged on "
    "CALIBRATION."
)
print(
    "Time-of-day and day-of-week effects remain excluded because "
    "the current data contain only one continuous session."
)
print(
    "Hawkes estimation remains unauthorized."
)

,model_id,specification,parameter_count,development_joint_log_score,development_aic,development_bic,delta_bic_from_best,selected_model_flag,selection_rule
0,D0_SIDE_CONSTANT_POISSON,CONSTANT,2,"-50,831.392","101,666.78","101,692.98",0,True,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
1,D1_SIDE_LINEAR_ELAPSED_POISSON,LINEAR_ELAPSED_TIME,4,"-50,828.57","101,665.14","101,717.53",24.552806,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
2,D2_SIDE_QUADRATIC_ELAPSED_POISSON,QUADRATIC_ELAPSED_TIME,6,"-50,825.729","101,663.46","101,742.04",49.063971,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
3,D3_SIDE_NATURAL_SPLINE_DF3_POISSON,NATURAL_SPLINE_DF3,8,"-50,824.438","101,664.88","101,769.66",76.677481,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...


,model_id,side,coefficient_name,coefficient_value,optimizer_method,optimizer_success,optimizer_iterations,maximum_absolute_gradient,selected_model_flag,status
0,D0_SIDE_CONSTANT_POISSON,BUY,intercept,0.63906539,CLOSED_FORM,True,0,2.2744473e-11,True,PASS
1,D0_SIDE_CONSTANT_POISSON,SELL,intercept,0.68933297,CLOSED_FORM,True,0,1.5450219e-10,True,PASS
2,D1_SIDE_LINEAR_ELAPSED_POISSON,BUY,elapsed_coordinate,0.1029077,L-BFGS-B,True,6,1.0614292e-06,False,PASS
3,D1_SIDE_LINEAR_ELAPSED_POISSON,BUY,intercept,0.69001257,L-BFGS-B,True,6,1.0614292e-06,False,PASS
4,D1_SIDE_LINEAR_ELAPSED_POISSON,SELL,elapsed_coordinate,-0.093521301,L-BFGS-B,True,6,4.4726746e-08,False,PASS
5,D1_SIDE_LINEAR_ELAPSED_POISSON,SELL,intercept,0.64226549,L-BFGS-B,True,6,4.4726746e-08,False,PASS
6,D2_SIDE_QUADRATIC_ELAPSED_POISSON,BUY,elapsed_coordinate,0.19212881,L-BFGS-B,True,12,5.9107982e-06,False,PASS
7,D2_SIDE_QUADRATIC_ELAPSED_POISSON,BUY,elapsed_coordinate_squared,0.089949891,L-BFGS-B,True,12,5.9107982e-06,False,PASS
8,D2_SIDE_QUADRATIC_ELAPSED_POISSON,BUY,intercept,0.70460152,L-BFGS-B,True,12,5.9107982e-06,False,PASS
9,D2_SIDE_QUADRATIC_ELAPSED_POISSON,SELL,elapsed_coordinate,-0.62613084,L-BFGS-B,True,13,0.00068776029,False,PASS


,model_id,specification,scored_partition,parameter_count,observed_event_count,predicted_event_count,log_score_total,log_score_per_second,log_score_per_observed_event,log_score_improvement_over_constant,selected_on_development_flag,status
0,D0_SIDE_CONSTANT_POISSON,CONSTANT,CALIBRATION,2,2493,"2,799.8716","-18,440.776",-25.601551,-7.3970221,0,True,PASS
1,D1_SIDE_LINEAR_ELAPSED_POISSON,LINEAR_ELAPSED_TIME,CALIBRATION,4,2493,"2,809.931","-18,443.178",-25.604886,-7.3979857,-2.4022733,False,PASS
2,D2_SIDE_QUADRATIC_ELAPSED_POISSON,QUADRATIC_ELAPSED_TIME,CALIBRATION,6,2493,"2,599.2332","-18,469.866",-25.641937,-7.4086909,-29.090386,False,PASS
3,D3_SIDE_NATURAL_SPLINE_DF3_POISSON,NATURAL_SPLINE_DF3,CALIBRATION,8,2493,"8,384.3938","-21,833.228",-30.311333,-8.7578131,"-3,392.4519",False,PASS
4,D3_SIDE_NATURAL_SPLINE_DF3_POISSON,NATURAL_SPLINE_DF3,DEVELOPMENT,8,7004,"7,004","-50,824.438",-28.206658,-7.2564874,6.9536912,False,PASS
5,D2_SIDE_QUADRATIC_ELAPSED_POISSON,QUADRATIC_ELAPSED_TIME,DEVELOPMENT,6,7004,"7,003.9993","-50,825.729",-28.207375,-7.2566717,5.6629691,False,PASS
6,D1_SIDE_LINEAR_ELAPSED_POISSON,LINEAR_ELAPSED_TIME,DEVELOPMENT,4,7004,"7,004","-50,828.57",-28.208952,-7.2570774,2.8210743,False,PASS
7,D0_SIDE_CONSTANT_POISSON,CONSTANT,DEVELOPMENT,2,7004,"7,004","-50,831.392",-28.210518,-7.2574802,0,True,PASS


,model_id,scored_partition,component,minimum_intensity_per_second,median_intensity_per_second,maximum_intensity_per_second,mean_intensity_per_second,status
0,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,1.8947092,1.8947092,1.8947092,1.8947092,PASS
1,D0_SIDE_CONSTANT_POISSON,CALIBRATION,SELL,1.9923861,1.9923861,1.9923861,1.9923861,PASS
8,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,BUY,1.8947092,1.8947092,1.8947092,1.8947092,PASS
9,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,SELL,1.9923861,1.9923861,1.9923861,1.9923861,PASS


,model_id,scored_partition,component,pearson_residual_mean,pearson_residual_variance,deviance_residual_mean,deviance_residual_variance,pearson_dispersion_ratio,maximum_absolute_pearson_residual,status
0,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,-0.0042980331,1.063852,-0.056103866,0.018980466,1.0638705,114.82438,PASS
1,D0_SIDE_CONSTANT_POISSON,CALIBRATION,SELL,-0.005353249,0.89943075,-0.057379508,0.019045671,0.89945941,67.165458,PASS
8,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,BUY,-4.3305393e-09,1.0654753,-0.055374224,0.020835085,1.0654753,137.79796,PASS
9,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,SELL,-4.4407615e-09,1.0280915,-0.056598929,0.02164201,1.0280915,111.97219,PASS


,scored_partition,deterministic_constant_log_score,prior_b1_log_score,absolute_log_score_difference,status
0,CALIBRATION,"-18,440.776","-18,440.776",0,EXACT_WITHIN_NUMERICAL_TOLERANCE
1,DEVELOPMENT,"-50,831.392","-50,831.392",0,EXACT_WITHIN_NUMERICAL_TOLERANCE


,gate,severity,passed,evidence
0,candidate_designs_full_rank,BLOCKING,True,candidate_count=4
1,all_candidates_fit_on_development_only,BLOCKING,True,all candidate parameters estimated using DEVEL...
2,all_optimizers_converged,BLOCKING,True,parameter_rows=20
3,all_parameters_finite,BLOCKING,True,all deterministic coefficients are finite
4,all_intensities_finite_positive,BLOCKING,True,intensity_range_rows=16
5,development_selection_excludes_calibration,BLOCKING,True,selection_rule=MINIMUM_DEVELOPMENT_BIC_WITH_DE...
6,exactly_one_deterministic_model_selected,BLOCKING,True,selected_model=D0_SIDE_CONSTANT_POISSON
7,selected_model_scored_on_both_partitions,BLOCKING,True,DEVELOPMENT and locked CALIBRATION aggregate s...
8,constant_candidate_reconciles_with_b1,BLOCKING,True,D0 constant scores reproduce prior B1 scores
9,protected_partition_content_remains_absent,BLOCKING,True,VALIDATION=False; ENGINEERING_HOLDOUT=False


Low-dimensional deterministic elapsed-time Poisson candidates were fitted on DEVELOPMENT only.
Selected deterministic model: D0_SIDE_CONSTANT_POISSON.
Selection used DEVELOPMENT BIC with a delta-two simplicity tie-break; CALIBRATION was not used for model selection.
The selected specification was then scored unchanged on CALIBRATION.
Time-of-day and day-of-week effects remain excluded because the current data contain only one continuous session.
Hawkes estimation remains unauthorized.


In [12]:
# ============================================================
# Causal rolling-window and exponentially weighted rate baselines
# ============================================================

import scipy.signal as signal


ADAPTIVE_PRIOR_EVENT_EQUIVALENT: Final[float] = 1.0

ADAPTIVE_SELECTION_RULE: Final[str] = (
    "MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_"
    "WITH_LONGER_MEMORY_TIE_BREAK"
)

ADAPTIVE_LOG_SCORE_TIE_TOLERANCE: Final[float] = 1e-9

ADAPTIVE_COMPONENT_PARAMETER_COUNT: Final[int] = 1

ADAPTIVE_RATE_FAMILIES: Final[
    tuple[str, ...]
] = (
    "CAUSAL_ROLLING_WINDOW_RATE",
    "CAUSAL_EXPONENTIALLY_WEIGHTED_RATE",
)


# ------------------------------------------------------------
# Canonical analytical side-event arrays
# ------------------------------------------------------------

def side_event_arrays(
    batches: pd.DataFrame,
    *,
    side: str,
    partitions: Sequence[str],
) -> tuple[np.ndarray, np.ndarray]:
    """
    Return exact event times and side multiplicities.

    Each exact-time batch contributes its BUY or SELL primary-event
    multiplicity. Timestamps are never jittered or reordered.
    """
    require(
        side in {"BUY", "SELL"},
        f"Unsupported event side: {side}",
    )

    requested_partitions = tuple(
        str(partition).upper()
        for partition in partitions
    )

    require(
        requested_partitions,
        "At least one partition must be requested.",
    )
    require(
        set(requested_partitions).issubset(
            set(ANALYTICAL_PARTITIONS)
        ),
        (
            "Adaptive-rate event-array request contains an "
            f"unauthorized partition: {requested_partitions}"
        ),
    )

    side_count_column = (
        "buy_event_count"
        if side == "BUY"
        else "sell_event_count"
    )

    selected = (
        batches.loc[
            batches[
                "event_partition"
            ]
            .astype("string")
            .str.strip()
            .str.upper()
            .isin(requested_partitions)
            & batches[
                side_count_column
            ]
            .astype("int64")
            .gt(0),
            [
                "event_partition_order",
                "partition_batch_index",
                "event_time_ns",
                side_count_column,
            ],
        ]
        .copy()
        .sort_values(
            [
                "event_partition_order",
                "partition_batch_index",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    require(
        not selected.empty,
        (
            f"No {side} events are available for partitions "
            f"{requested_partitions}."
        ),
    )

    event_times_ns = parse_exact_int64(
        selected["event_time_ns"],
        label=(
            f"{side}.{','.join(requested_partitions)}."
            "event_time_ns"
        ),
    ).to_numpy(dtype="int64")

    event_multiplicities = (
        selected[
            side_count_column
        ]
        .astype("int64")
        .to_numpy()
    )

    require(
        np.all(
            np.diff(event_times_ns) >= 0
        ),
        (
            f"{side} event times are not nondecreasing for "
            f"{requested_partitions}."
        ),
    )
    require(
        np.all(
            event_multiplicities > 0
        ),
        (
            f"{side} event multiplicities contain nonpositive "
            "values."
        ),
    )

    event_times_ns.setflags(write=False)
    event_multiplicities.setflags(write=False)

    return (
        event_times_ns,
        event_multiplicities,
    )


SIDE_EVENT_ARRAYS_BY_PARTITION: Final[
    Mapping[tuple[str, str], tuple[np.ndarray, np.ndarray]]
] = {
    (
        partition_name,
        side,
    ): side_event_arrays(
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
        side=side,
        partitions=(partition_name,),
    )
    for partition_name in ANALYTICAL_PARTITIONS
    for side in ("BUY", "SELL")
}

ANALYTICAL_SIDE_EVENT_ARRAYS: Final[
    Mapping[str, tuple[np.ndarray, np.ndarray]]
] = {
    side: side_event_arrays(
        PRIMARY_SCORING_BATCHES_ANALYTICAL,
        side=side,
        partitions=ANALYTICAL_PARTITIONS,
    )
    for side in ("BUY", "SELL")
}


# ------------------------------------------------------------
# Native-grid cell starts
# ------------------------------------------------------------

def native_grid_cell_starts_ns(
    grid: NativeCountGrid,
) -> np.ndarray:
    """Return exact integer-nanosecond native-cell start times."""
    starts_ns = (
        grid.contract_start_ns
        + np.arange(
            grid.cell_count,
            dtype="int64",
        )
        * grid.grid_width_ns
    )

    require(
        starts_ns[0]
        == grid.contract_start_ns,
        (
            f"{grid.event_partition} first cell start differs "
            "from its contract start."
        ),
    )
    require(
        starts_ns[-1]
        < grid.contract_end_exclusive_ns,
        (
            f"{grid.event_partition} final cell start is not "
            "strictly before its end-exclusive boundary."
        ),
    )

    if starts_ns.size > 1:
        require(
            np.all(
                np.diff(starts_ns)
                == grid.grid_width_ns
            ),
            (
                f"{grid.event_partition} native-cell starts "
                "are not evenly spaced."
            ),
        )

    starts_ns.setflags(write=False)

    return starts_ns


NATIVE_GRID_CELL_STARTS_NS: Final[
    Mapping[str, np.ndarray]
] = {
    partition_name: native_grid_cell_starts_ns(
        grid
    )
    for partition_name, grid
    in NATIVE_COUNT_GRIDS.items()
}


# ------------------------------------------------------------
# Prior contract
# ------------------------------------------------------------

def development_side_rate(
    side: str,
) -> float:
    """Return the frozen DEVELOPMENT homogeneous rate for one side."""
    if side == "BUY":
        rate = (
            DEVELOPMENT_BUY_RATE_PER_SECOND
        )
    elif side == "SELL":
        rate = (
            DEVELOPMENT_SELL_RATE_PER_SECOND
        )
    else:
        raise ValueError(
            f"Unsupported side: {side}"
        )

    require(
        np.isfinite(rate)
        and rate > 0.0,
        f"{side} DEVELOPMENT prior rate is invalid.",
    )

    return float(rate)


def adaptive_prior_exposure_seconds(
    side: str,
) -> float:
    """
    Return prior exposure equivalent to one DEVELOPMENT-rate event.

    The prior mean equals the DEVELOPMENT homogeneous side rate:

        prior_count / prior_exposure = lambda_development
    """
    rate = development_side_rate(
        side
    )

    prior_exposure = (
        ADAPTIVE_PRIOR_EVENT_EQUIVALENT
        / rate
    )

    require(
        np.isfinite(prior_exposure)
        and prior_exposure > 0.0,
        f"{side} adaptive prior exposure is invalid.",
    )

    return float(prior_exposure)


ADAPTIVE_PRIOR_CONTRACT = pd.DataFrame(
    [
        {
            "side": side,
            "prior_event_equivalent": (
                ADAPTIVE_PRIOR_EVENT_EQUIVALENT
            ),
            "development_prior_rate_per_second": (
                development_side_rate(side)
            ),
            "prior_exposure_seconds": (
                adaptive_prior_exposure_seconds(
                    side
                )
            ),
            "prior_mean_rate_per_second": (
                ADAPTIVE_PRIOR_EVENT_EQUIVALENT
                / adaptive_prior_exposure_seconds(
                    side
                )
            ),
            "prior_selected_using_calibration": False,
            "status": "PASS",
        }
        for side in ("BUY", "SELL")
    ]
)


# ------------------------------------------------------------
# Causal rolling-window forecasts
# ------------------------------------------------------------

def rolling_rate_forecast(
    *,
    partition_name: str,
    side: str,
    window_ms: int,
    include_prior_analytical_history: bool,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Forecast a side-specific event rate at every native-cell start.

    The trailing history interval is:

        (cell_start - window, cell_start)

    Events exactly at the current cell start are excluded. Events exactly
    at the left window boundary are also excluded.

    DEVELOPMENT selection uses DEVELOPMENT events only. Locked CALIBRATION
    scoring includes DEVELOPMENT history and earlier CALIBRATION events.
    """
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized rolling-rate partition: "
            f"{partition_name}"
        ),
    )
    require(
        side in {"BUY", "SELL"},
        f"Unsupported rolling-rate side: {side}",
    )
    require(
        window_ms
        in ROLLING_RATE_WINDOWS_MS,
        (
            f"Rolling-rate window {window_ms} ms is outside "
            "the frozen candidate grid."
        ),
    )

    if partition_name == "DEVELOPMENT":
        require(
            not include_prior_analytical_history,
            (
                "DEVELOPMENT rolling forecasts must not request "
                "prior analytical partitions."
            ),
        )

        event_times_ns, event_weights = (
            SIDE_EVENT_ARRAYS_BY_PARTITION[
                (
                    "DEVELOPMENT",
                    side,
                )
            ]
        )

        history_origin_ns = int(
            NATIVE_COUNT_GRIDS[
                "DEVELOPMENT"
            ].contract_start_ns
        )

    else:
        require(
            include_prior_analytical_history,
            (
                "CALIBRATION rolling forecasts must carry "
                "DEVELOPMENT history."
            ),
        )

        event_times_ns, event_weights = (
            ANALYTICAL_SIDE_EVENT_ARRAYS[
                side
            ]
        )

        history_origin_ns = (
            FULL_REGISTERED_RUN_START_NS
        )

    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    cell_starts_ns = (
        NATIVE_GRID_CELL_STARTS_NS[
            partition_name
        ]
    )

    window_ns = (
        window_ms
        * NANOSECONDS_PER_MILLISECOND
    )

    left_boundaries_ns = np.maximum(
        cell_starts_ns - window_ns,
        history_origin_ns,
    )

    right_indices = np.searchsorted(
        event_times_ns,
        cell_starts_ns,
        side="left",
    )

    left_indices = np.searchsorted(
        event_times_ns,
        left_boundaries_ns,
        side="right",
    )

    require(
        np.all(
            right_indices >= left_indices
        ),
        (
            f"{partition_name} {side} rolling-window search "
            "indices are invalid."
        ),
    )

    cumulative_weights = np.empty(
        event_weights.size + 1,
        dtype="int64",
    )
    cumulative_weights[0] = 0
    np.cumsum(
        event_weights,
        dtype="int64",
        out=cumulative_weights[1:],
    )

    trailing_event_counts = (
        cumulative_weights[
            right_indices
        ]
        - cumulative_weights[
            left_indices
        ]
    ).astype(
        "int64",
        copy=False,
    )

    trailing_exposure_seconds = (
        (
            cell_starts_ns
            - left_boundaries_ns
        ).astype("float64")
        / NANOSECONDS_PER_SECOND
    )

    prior_exposure_seconds = (
        adaptive_prior_exposure_seconds(
            side
        )
    )

    intensity_per_second = (
        trailing_event_counts.astype(
            "float64"
        )
        + ADAPTIVE_PRIOR_EVENT_EQUIVALENT
    ) / (
        trailing_exposure_seconds
        + prior_exposure_seconds
    )

    expected_counts = (
        intensity_per_second
        * grid_exposure_seconds(
            grid
        )
    )

    require(
        np.isfinite(
            intensity_per_second
        ).all(),
        (
            f"{partition_name} {side} rolling intensity "
            "contains nonfinite values."
        ),
    )
    require(
        np.all(
            intensity_per_second > 0.0
        ),
        (
            f"{partition_name} {side} rolling intensity is not "
            "strictly positive."
        ),
    )
    require(
        np.isfinite(
            expected_counts
        ).all()
        and np.all(
            expected_counts > 0.0
        ),
        (
            f"{partition_name} {side} rolling expected counts "
            "are invalid."
        ),
    )

    return (
        intensity_per_second,
        expected_counts,
        trailing_event_counts,
    )


# ------------------------------------------------------------
# Exact exponentially weighted state at native-cell starts
# ------------------------------------------------------------

def exponentially_weighted_state_within_partition(
    *,
    partition_name: str,
    side: str,
    decay_time_seconds: float,
    initial_state: float,
) -> np.ndarray:
    """
    Compute exact exponentially weighted event count at cell starts.

    The recursion uses exact batch timestamps inside each native cell.
    Every event in a cell affects only later cell starts.
    """
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized EWMA partition: "
            f"{partition_name}"
        ),
    )
    require(
        side in {"BUY", "SELL"},
        f"Unsupported EWMA side: {side}",
    )
    require(
        np.isfinite(
            decay_time_seconds
        )
        and decay_time_seconds > 0.0,
        "EWMA decay time must be finite and positive.",
    )
    require(
        np.isfinite(initial_state)
        and initial_state >= 0.0,
        "EWMA initial state must be finite and nonnegative.",
    )

    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    cell_starts_ns = (
        NATIVE_GRID_CELL_STARTS_NS[
            partition_name
        ]
    )

    event_times_ns, event_weights = (
        SIDE_EVENT_ARRAYS_BY_PARTITION[
            (
                partition_name,
                side,
            )
        ]
    )

    cell_indices = (
        (
            event_times_ns
            - grid.contract_start_ns
        )
        // grid.grid_width_ns
    ).astype("int64")

    require(
        np.all(cell_indices >= 0)
        and np.all(
            cell_indices
            < grid.cell_count
        ),
        (
            f"{partition_name} {side} EWMA events fall outside "
            "the native grid."
        ),
    )

    weighted_state = np.empty(
        grid.cell_count,
        dtype="float64",
    )
    weighted_state[0] = initial_state

    if grid.cell_count > 1:
        interval_contributions = np.zeros(
            grid.cell_count - 1,
            dtype="float64",
        )

        usable_event_mask = (
            cell_indices
            < grid.cell_count - 1
        )

        usable_indices = (
            cell_indices[
                usable_event_mask
            ]
        )

        usable_event_times_ns = (
            event_times_ns[
                usable_event_mask
            ]
        )

        usable_event_weights = (
            event_weights[
                usable_event_mask
            ].astype(
                "float64"
            )
        )

        next_cell_starts_ns = (
            grid.contract_start_ns
            + (
                usable_indices + 1
            )
            * grid.grid_width_ns
        )

        discount_to_next_start = np.exp(
            -(
                next_cell_starts_ns
                - usable_event_times_ns
            ).astype("float64")
            / NANOSECONDS_PER_SECOND
            / decay_time_seconds
        )

        np.add.at(
            interval_contributions,
            usable_indices,
            usable_event_weights
            * discount_to_next_start,
        )

        one_cell_decay = math.exp(
            -(
                grid.grid_width_ns
                / NANOSECONDS_PER_SECOND
            )
            / decay_time_seconds
        )

        filtered_state, _ = signal.lfilter(
            [1.0],
            [
                1.0,
                -one_cell_decay,
            ],
            interval_contributions,
            zi=np.asarray(
                [
                    one_cell_decay
                    * initial_state
                ],
                dtype="float64",
            ),
        )

        weighted_state[1:] = (
            filtered_state
        )

    require(
        np.isfinite(
            weighted_state
        ).all(),
        (
            f"{partition_name} {side} EWMA state contains "
            "nonfinite values."
        ),
    )
    require(
        np.all(
            weighted_state >= 0.0
        ),
        (
            f"{partition_name} {side} EWMA state contains "
            "negative values."
        ),
    )

    return weighted_state


def development_final_state_at_calibration_start(
    *,
    side: str,
    decay_time_seconds: float,
    development_state: np.ndarray,
) -> float:
    """
    Carry the DEVELOPMENT exponentially weighted state exactly to the
    CALIBRATION boundary, including events in DEVELOPMENT's final
    partial native cell.
    """
    development_grid = (
        NATIVE_COUNT_GRIDS[
            "DEVELOPMENT"
        ]
    )

    calibration_start_ns = int(
        NATIVE_COUNT_GRIDS[
            "CALIBRATION"
        ].contract_start_ns
    )

    development_cell_starts_ns = (
        NATIVE_GRID_CELL_STARTS_NS[
            "DEVELOPMENT"
        ]
    )

    final_development_cell_start_ns = int(
        development_cell_starts_ns[-1]
    )

    require(
        calibration_start_ns
        == development_grid.contract_end_exclusive_ns,
        (
            "CALIBRATION does not begin at the DEVELOPMENT "
            "end-exclusive boundary."
        ),
    )

    carry_duration_seconds = (
        calibration_start_ns
        - final_development_cell_start_ns
    ) / NANOSECONDS_PER_SECOND

    require(
        carry_duration_seconds > 0.0,
        "EWMA boundary carry duration must be positive.",
    )

    carried_prior_state = (
        float(
            development_state[-1]
        )
        * math.exp(
            -carry_duration_seconds
            / decay_time_seconds
        )
    )

    development_event_times_ns, development_event_weights = (
        SIDE_EVENT_ARRAYS_BY_PARTITION[
            (
                "DEVELOPMENT",
                side,
            )
        ]
    )

    final_cell_event_mask = (
        development_event_times_ns
        >= final_development_cell_start_ns
    )

    final_cell_event_times_ns = (
        development_event_times_ns[
            final_cell_event_mask
        ]
    )

    final_cell_event_weights = (
        development_event_weights[
            final_cell_event_mask
        ].astype(
            "float64"
        )
    )

    if final_cell_event_times_ns.size:
        require(
            np.all(
                final_cell_event_times_ns
                < calibration_start_ns
            ),
            (
                "A DEVELOPMENT final-cell event reaches or "
                "crosses the CALIBRATION boundary."
            ),
        )

        final_cell_contribution = float(
            np.sum(
                final_cell_event_weights
                * np.exp(
                    -(
                        calibration_start_ns
                        - final_cell_event_times_ns
                    ).astype("float64")
                    / NANOSECONDS_PER_SECOND
                    / decay_time_seconds
                ),
                dtype="float64",
            )
        )
    else:
        final_cell_contribution = 0.0

    calibration_initial_state = (
        carried_prior_state
        + final_cell_contribution
    )

    require(
        np.isfinite(
            calibration_initial_state
        )
        and calibration_initial_state >= 0.0,
        "CALIBRATION EWMA initial state is invalid.",
    )

    return float(
        calibration_initial_state
    )


def exponentially_weighted_rate_forecasts(
    *,
    side: str,
    half_life_ms: int,
    include_calibration: bool,
) -> Mapping[
    str,
    tuple[np.ndarray, np.ndarray, np.ndarray],
]:
    """
    Return causal exponentially weighted forecasts.

    DEVELOPMENT is initialized with zero weighted event history and a
    static one-event prior. When CALIBRATION is requested, the dynamic
    weighted state is carried across the exact partition boundary.
    """
    require(
        side in {"BUY", "SELL"},
        f"Unsupported EWMA side: {side}",
    )
    require(
        half_life_ms
        in EWMA_HALF_LIVES_MS,
        (
            f"EWMA half-life {half_life_ms} ms is outside the "
            "frozen candidate grid."
        ),
    )

    half_life_seconds = (
        half_life_ms / 1_000.0
    )

    decay_time_seconds = (
        half_life_seconds
        / math.log(2.0)
    )

    development_state = (
        exponentially_weighted_state_within_partition(
            partition_name="DEVELOPMENT",
            side=side,
            decay_time_seconds=(
                decay_time_seconds
            ),
            initial_state=0.0,
        )
    )

    partition_forecasts: dict[
        str,
        tuple[
            np.ndarray,
            np.ndarray,
            np.ndarray,
        ],
    ] = {}

    development_cell_starts_ns = (
        NATIVE_GRID_CELL_STARTS_NS[
            "DEVELOPMENT"
        ]
    )

    development_elapsed_seconds = (
        (
            development_cell_starts_ns
            - FULL_REGISTERED_RUN_START_NS
        ).astype("float64")
        / NANOSECONDS_PER_SECOND
    )

    development_weighted_exposure = (
        decay_time_seconds
        * (
            1.0
            - np.exp(
                -development_elapsed_seconds
                / decay_time_seconds
            )
        )
    )

    prior_exposure = (
        adaptive_prior_exposure_seconds(
            side
        )
    )

    development_intensity = (
        development_state
        + ADAPTIVE_PRIOR_EVENT_EQUIVALENT
    ) / (
        development_weighted_exposure
        + prior_exposure
    )

    development_expected_counts = (
        development_intensity
        * grid_exposure_seconds(
            NATIVE_COUNT_GRIDS[
                "DEVELOPMENT"
            ]
        )
    )

    partition_forecasts[
        "DEVELOPMENT"
    ] = (
        development_intensity,
        development_expected_counts,
        development_state,
    )

    if include_calibration:
        calibration_initial_state = (
            development_final_state_at_calibration_start(
                side=side,
                decay_time_seconds=(
                    decay_time_seconds
                ),
                development_state=(
                    development_state
                ),
            )
        )

        calibration_state = (
            exponentially_weighted_state_within_partition(
                partition_name="CALIBRATION",
                side=side,
                decay_time_seconds=(
                    decay_time_seconds
                ),
                initial_state=(
                    calibration_initial_state
                ),
            )
        )

        calibration_cell_starts_ns = (
            NATIVE_GRID_CELL_STARTS_NS[
                "CALIBRATION"
            ]
        )

        calibration_elapsed_seconds = (
            (
                calibration_cell_starts_ns
                - FULL_REGISTERED_RUN_START_NS
            ).astype("float64")
            / NANOSECONDS_PER_SECOND
        )

        calibration_weighted_exposure = (
            decay_time_seconds
            * (
                1.0
                - np.exp(
                    -calibration_elapsed_seconds
                    / decay_time_seconds
                )
            )
        )

        calibration_intensity = (
            calibration_state
            + ADAPTIVE_PRIOR_EVENT_EQUIVALENT
        ) / (
            calibration_weighted_exposure
            + prior_exposure
        )

        calibration_expected_counts = (
            calibration_intensity
            * grid_exposure_seconds(
                NATIVE_COUNT_GRIDS[
                    "CALIBRATION"
                ]
            )
        )

        partition_forecasts[
            "CALIBRATION"
        ] = (
            calibration_intensity,
            calibration_expected_counts,
            calibration_state,
        )

    for (
        forecast_partition,
        (
            intensity,
            expected_counts,
            weighted_state,
        ),
    ) in partition_forecasts.items():
        require(
            np.isfinite(
                intensity
            ).all()
            and np.all(
                intensity > 0.0
            ),
            (
                f"{forecast_partition} {side} EWMA intensity "
                "is invalid."
            ),
        )
        require(
            np.isfinite(
                expected_counts
            ).all()
            and np.all(
                expected_counts > 0.0
            ),
            (
                f"{forecast_partition} {side} EWMA expected "
                "counts are invalid."
            ),
        )
        require(
            np.isfinite(
                weighted_state
            ).all()
            and np.all(
                weighted_state >= 0.0
            ),
            (
                f"{forecast_partition} {side} EWMA state is "
                "invalid."
            ),
        )

    return partition_forecasts


# ------------------------------------------------------------
# Adaptive-model registry
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class AdaptiveRateModel:
    model_id: str
    model_family: str
    hyperparameter_name: str
    hyperparameter_ms: int
    fitted_partition: str
    score_space: str
    effective_parameter_count_applicable: bool


def rolling_model_id(
    window_ms: int,
) -> str:
    """Return a deterministic rolling-rate model identifier."""
    return (
        "R_SIDE_ROLLING_"
        f"{window_ms}MS_POISSON"
    )


def ewma_model_id(
    half_life_ms: int,
) -> str:
    """Return a deterministic EWMA-rate model identifier."""
    return (
        "E_SIDE_EWMA_"
        f"{half_life_ms}MS_POISSON"
    )


ADAPTIVE_RATE_MODELS: dict[
    str,
    AdaptiveRateModel,
] = {}

for window_ms in (
    ROLLING_RATE_WINDOWS_MS
):
    model_id = rolling_model_id(
        window_ms
    )

    ADAPTIVE_RATE_MODELS[
        model_id
    ] = AdaptiveRateModel(
        model_id=model_id,
        model_family=(
            "CAUSAL_ROLLING_WINDOW_RATE"
        ),
        hyperparameter_name=(
            "window_ms"
        ),
        hyperparameter_ms=(
            window_ms
        ),
        fitted_partition="DEVELOPMENT",
        score_space="BIVARIATE_SIDE_COUNT",
        effective_parameter_count_applicable=False,
    )

for half_life_ms in (
    EWMA_HALF_LIVES_MS
):
    model_id = ewma_model_id(
        half_life_ms
    )

    ADAPTIVE_RATE_MODELS[
        model_id
    ] = AdaptiveRateModel(
        model_id=model_id,
        model_family=(
            "CAUSAL_EXPONENTIALLY_WEIGHTED_RATE"
        ),
        hyperparameter_name=(
            "half_life_ms"
        ),
        hyperparameter_ms=(
            half_life_ms
        ),
        fitted_partition="DEVELOPMENT",
        score_space="BIVARIATE_SIDE_COUNT",
        effective_parameter_count_applicable=False,
    )


# ------------------------------------------------------------
# Shared adaptive-model forecast dispatcher
# ------------------------------------------------------------

def adaptive_model_forecasts(
    *,
    model: AdaptiveRateModel,
    partition_name: str,
) -> Mapping[
    str,
    tuple[np.ndarray, np.ndarray],
]:
    """Return BUY and SELL rate/expected-count forecasts."""
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized adaptive scoring partition: "
            f"{partition_name}"
        ),
    )

    side_forecasts: dict[
        str,
        tuple[
            np.ndarray,
            np.ndarray,
        ],
    ] = {}

    for side in (
        "BUY",
        "SELL",
    ):
        if (
            model.model_family
            == "CAUSAL_ROLLING_WINDOW_RATE"
        ):
            intensity, expected_counts, _ = (
                rolling_rate_forecast(
                    partition_name=(
                        partition_name
                    ),
                    side=side,
                    window_ms=(
                        model.hyperparameter_ms
                    ),
                    include_prior_analytical_history=(
                        partition_name
                        == "CALIBRATION"
                    ),
                )
            )

        elif (
            model.model_family
            == "CAUSAL_EXPONENTIALLY_WEIGHTED_RATE"
        ):
            ewma_forecasts = (
                exponentially_weighted_rate_forecasts(
                    side=side,
                    half_life_ms=(
                        model.hyperparameter_ms
                    ),
                    include_calibration=(
                        partition_name
                        == "CALIBRATION"
                    ),
                )
            )

            (
                intensity,
                expected_counts,
                _,
            ) = ewma_forecasts[
                partition_name
            ]

        else:
            raise ValueError(
                (
                    "Unsupported adaptive model family: "
                    f"{model.model_family}"
                )
            )

        side_forecasts[
            side
        ] = (
            intensity,
            expected_counts,
        )

    return side_forecasts


# ------------------------------------------------------------
# DEVELOPMENT-only candidate scoring and selection
# ------------------------------------------------------------

adaptive_development_selection_rows: list[
    dict[str, Any]
] = []

adaptive_score_rows: list[
    dict[str, Any]
] = []

adaptive_residual_rows: list[
    dict[str, Any]
] = []

adaptive_intensity_range_rows: list[
    dict[str, Any]
] = []


def append_adaptive_component_results(
    *,
    model: AdaptiveRateModel,
    partition_name: str,
    side: str,
    intensity_per_second: np.ndarray,
    expected_counts: np.ndarray,
) -> float:
    """Append score, residual, and intensity summaries."""
    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    observed_counts = (
        grid.buy_counts
        if side == "BUY"
        else grid.sell_counts
    )

    log_score = poisson_count_log_score(
        observed_counts,
        expected_counts,
    )

    residual_summary = (
        summarize_poisson_residuals(
            observed_counts,
            expected_counts,
            parameter_count=(
                ADAPTIVE_COMPONENT_PARAMETER_COUNT
            ),
        )
    )

    exposure_seconds = (
        grid.contract_duration_ns
        / NANOSECONDS_PER_SECOND
    )

    observed_event_count = int(
        observed_counts.sum(
            dtype="int64"
        )
    )

    predicted_event_count = float(
        expected_counts.sum(
            dtype="float64"
        )
    )

    adaptive_score_rows.append(
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "hyperparameter_name": (
                model.hyperparameter_name
            ),
            "hyperparameter_ms": (
                model.hyperparameter_ms
            ),
            "fitted_partition": (
                model.fitted_partition
            ),
            "scored_partition": (
                partition_name
            ),
            "score_space": (
                model.score_space
            ),
            "component": side,
            "effective_parameter_count_applicable": (
                model.effective_parameter_count_applicable
            ),
            "cell_count": (
                grid.cell_count
            ),
            "exposure_seconds": (
                exposure_seconds
            ),
            "observed_event_count": (
                observed_event_count
            ),
            "predicted_event_count": (
                predicted_event_count
            ),
            "observed_minus_predicted": (
                observed_event_count
                - predicted_event_count
            ),
            "log_score_total": (
                log_score
            ),
            "log_score_per_second": (
                log_score
                / exposure_seconds
            ),
            "log_score_per_observed_event": (
                log_score
                / observed_event_count
            ),
            "locked_before_calibration": True,
            "status": "PASS",
        }
    )

    adaptive_residual_rows.append(
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "hyperparameter_ms": (
                model.hyperparameter_ms
            ),
            "scored_partition": (
                partition_name
            ),
            "component": side,
            **residual_summary,
            "status": "PASS",
        }
    )

    exposure_weights = (
        grid.exposure_ns.astype(
            "float64"
        )
    )

    adaptive_intensity_range_rows.append(
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "hyperparameter_ms": (
                model.hyperparameter_ms
            ),
            "scored_partition": (
                partition_name
            ),
            "component": side,
            "minimum_intensity_per_second": float(
                np.min(
                    intensity_per_second
                )
            ),
            "q05_intensity_per_second": float(
                np.quantile(
                    intensity_per_second,
                    0.05,
                )
            ),
            "median_intensity_per_second": float(
                np.median(
                    intensity_per_second
                )
            ),
            "q95_intensity_per_second": float(
                np.quantile(
                    intensity_per_second,
                    0.95,
                )
            ),
            "maximum_intensity_per_second": float(
                np.max(
                    intensity_per_second
                )
            ),
            "exposure_weighted_mean_intensity_per_second": float(
                np.average(
                    intensity_per_second,
                    weights=exposure_weights,
                )
            ),
            "status": "PASS",
        }
    )

    return log_score


# This loop performs DEVELOPMENT-only selection.
# No CALIBRATION forecast is constructed until all selections are frozen.
for model in (
    ADAPTIVE_RATE_MODELS.values()
):
    development_forecasts = (
        adaptive_model_forecasts(
            model=model,
            partition_name="DEVELOPMENT",
        )
    )

    development_component_scores: dict[
        str,
        float,
    ] = {}

    development_predicted_counts: dict[
        str,
        float,
    ] = {}

    for side in (
        "BUY",
        "SELL",
    ):
        (
            intensity_per_second,
            expected_counts,
        ) = development_forecasts[
            side
        ]

        development_component_scores[
            side
        ] = (
            append_adaptive_component_results(
                model=model,
                partition_name="DEVELOPMENT",
                side=side,
                intensity_per_second=(
                    intensity_per_second
                ),
                expected_counts=(
                    expected_counts
                ),
            )
        )

        development_predicted_counts[
            side
        ] = float(
            expected_counts.sum(
                dtype="float64"
            )
        )

    joint_log_score = (
        development_component_scores[
            "BUY"
        ]
        + development_component_scores[
            "SELL"
        ]
    )

    joint_observed_count = (
        development_buy_event_count
        + development_sell_event_count
    )

    joint_predicted_count = (
        development_predicted_counts[
            "BUY"
        ]
        + development_predicted_counts[
            "SELL"
        ]
    )

    adaptive_score_rows.append(
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "hyperparameter_name": (
                model.hyperparameter_name
            ),
            "hyperparameter_ms": (
                model.hyperparameter_ms
            ),
            "fitted_partition": (
                model.fitted_partition
            ),
            "scored_partition": (
                "DEVELOPMENT"
            ),
            "score_space": (
                model.score_space
            ),
            "component": (
                "BUY_AND_SELL_AGGREGATE"
            ),
            "effective_parameter_count_applicable": False,
            "cell_count": int(
                2
                * development_grid.cell_count
            ),
            "exposure_seconds": (
                development_exposure_seconds
            ),
            "observed_event_count": (
                joint_observed_count
            ),
            "predicted_event_count": (
                joint_predicted_count
            ),
            "observed_minus_predicted": (
                joint_observed_count
                - joint_predicted_count
            ),
            "log_score_total": (
                joint_log_score
            ),
            "log_score_per_second": (
                joint_log_score
                / development_exposure_seconds
            ),
            "log_score_per_observed_event": (
                joint_log_score
                / joint_observed_count
            ),
            "locked_before_calibration": True,
            "status": "PASS",
        }
    )

    adaptive_development_selection_rows.append(
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "hyperparameter_name": (
                model.hyperparameter_name
            ),
            "hyperparameter_ms": (
                model.hyperparameter_ms
            ),
            "development_joint_log_score": (
                joint_log_score
            ),
            "development_log_score_per_second": (
                joint_log_score
                / development_exposure_seconds
            ),
            "development_log_score_per_observed_event": (
                joint_log_score
                / joint_observed_count
            ),
            "development_observed_event_count": (
                joint_observed_count
            ),
            "development_predicted_event_count": (
                joint_predicted_count
            ),
            "selection_partition": "DEVELOPMENT",
            "calibration_used_for_selection": False,
            "information_criterion_applicable": False,
        }
    )


ADAPTIVE_DEVELOPMENT_SELECTION_TABLE = (
    pd.DataFrame(
        adaptive_development_selection_rows
    )
)


def select_best_adaptive_candidate(
    frame: pd.DataFrame,
    *,
    model_family: str | None,
) -> str:
    """
    Select the maximum DEVELOPMENT log-score candidate.

    When scores are numerically tied, the longer memory horizon is
    preferred as the smoother and less reactive specification.
    """
    if model_family is None:
        candidates = frame.copy()
    else:
        candidates = frame.loc[
            frame[
                "model_family"
            ].eq(model_family)
        ].copy()

    require(
        not candidates.empty,
        (
            "Adaptive model selection received no candidates "
            f"for family {model_family!r}."
        ),
    )

    best_score = float(
        candidates[
            "development_joint_log_score"
        ].max()
    )

    tied_candidates = (
        candidates.loc[
            (
                best_score
                - candidates[
                    "development_joint_log_score"
                ]
            )
            <= ADAPTIVE_LOG_SCORE_TIE_TOLERANCE
        ]
        .sort_values(
            [
                "hyperparameter_ms",
                "model_id",
            ],
            ascending=[
                False,
                True,
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    require(
        not tied_candidates.empty,
        "Adaptive selection tie set is empty.",
    )

    return str(
        tied_candidates.iloc[0][
            "model_id"
        ]
    )


SELECTED_ROLLING_MODEL_ID: Final[str] = (
    select_best_adaptive_candidate(
        ADAPTIVE_DEVELOPMENT_SELECTION_TABLE,
        model_family=(
            "CAUSAL_ROLLING_WINDOW_RATE"
        ),
    )
)

SELECTED_EWMA_MODEL_ID: Final[str] = (
    select_best_adaptive_candidate(
        ADAPTIVE_DEVELOPMENT_SELECTION_TABLE,
        model_family=(
            "CAUSAL_EXPONENTIALLY_WEIGHTED_RATE"
        ),
    )
)

SELECTED_ADAPTIVE_MODEL_ID: Final[str] = (
    select_best_adaptive_candidate(
        ADAPTIVE_DEVELOPMENT_SELECTION_TABLE,
        model_family=None,
    )
)

RETAINED_ADAPTIVE_MODEL_IDS: Final[
    frozenset[str]
] = frozenset(
    {
        SELECTED_ROLLING_MODEL_ID,
        SELECTED_EWMA_MODEL_ID,
        SELECTED_ADAPTIVE_MODEL_ID,
    }
)

ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
    "selected_within_family_flag"
] = (
    ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
        "model_id"
    ].isin(
        {
            SELECTED_ROLLING_MODEL_ID,
            SELECTED_EWMA_MODEL_ID,
        }
    )
)

ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
    "selected_overall_flag"
] = (
    ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
        "model_id"
    ].eq(
        SELECTED_ADAPTIVE_MODEL_ID
    )
)

ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
    "selection_rule"
] = ADAPTIVE_SELECTION_RULE

ADAPTIVE_DEVELOPMENT_SELECTION_TABLE = (
    ADAPTIVE_DEVELOPMENT_SELECTION_TABLE.sort_values(
        [
            "model_family",
            "development_joint_log_score",
            "hyperparameter_ms",
        ],
        ascending=[
            True,
            False,
            False,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Locked CALIBRATION scoring after selection is frozen
# ------------------------------------------------------------

SELECTED_ADAPTIVE_EXPECTED_COUNTS: dict[
    tuple[str, str, str],
    np.ndarray,
] = {}

SELECTED_ADAPTIVE_INTENSITIES: dict[
    tuple[str, str, str],
    np.ndarray,
] = {}


for model in (
    ADAPTIVE_RATE_MODELS.values()
):
    calibration_forecasts = (
        adaptive_model_forecasts(
            model=model,
            partition_name="CALIBRATION",
        )
    )

    calibration_component_scores: dict[
        str,
        float,
    ] = {}

    calibration_predicted_counts: dict[
        str,
        float,
    ] = {}

    for side in (
        "BUY",
        "SELL",
    ):
        (
            intensity_per_second,
            expected_counts,
        ) = calibration_forecasts[
            side
        ]

        calibration_component_scores[
            side
        ] = (
            append_adaptive_component_results(
                model=model,
                partition_name="CALIBRATION",
                side=side,
                intensity_per_second=(
                    intensity_per_second
                ),
                expected_counts=(
                    expected_counts
                ),
            )
        )

        calibration_predicted_counts[
            side
        ] = float(
            expected_counts.sum(
                dtype="float64"
            )
        )

        if (
            model.model_id
            in RETAINED_ADAPTIVE_MODEL_IDS
        ):
            retained_intensity = np.asarray(
                intensity_per_second,
                dtype="float64",
            ).copy()

            retained_expected_counts = np.asarray(
                expected_counts,
                dtype="float64",
            ).copy()

            retained_intensity.setflags(
                write=False
            )
            retained_expected_counts.setflags(
                write=False
            )

            SELECTED_ADAPTIVE_INTENSITIES[
                (
                    model.model_id,
                    "CALIBRATION",
                    side,
                )
            ] = retained_intensity

            SELECTED_ADAPTIVE_EXPECTED_COUNTS[
                (
                    model.model_id,
                    "CALIBRATION",
                    side,
                )
            ] = retained_expected_counts

    calibration_joint_log_score = (
        calibration_component_scores[
            "BUY"
        ]
        + calibration_component_scores[
            "SELL"
        ]
    )

    calibration_grid = (
        NATIVE_COUNT_GRIDS[
            "CALIBRATION"
        ]
    )

    calibration_observed_count = int(
        calibration_grid.buy_counts.sum(
            dtype="int64"
        )
        + calibration_grid.sell_counts.sum(
            dtype="int64"
        )
    )

    calibration_predicted_count = (
        calibration_predicted_counts[
            "BUY"
        ]
        + calibration_predicted_counts[
            "SELL"
        ]
    )

    calibration_exposure_seconds = (
        calibration_grid.contract_duration_ns
        / NANOSECONDS_PER_SECOND
    )

    adaptive_score_rows.append(
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "hyperparameter_name": (
                model.hyperparameter_name
            ),
            "hyperparameter_ms": (
                model.hyperparameter_ms
            ),
            "fitted_partition": (
                model.fitted_partition
            ),
            "scored_partition": (
                "CALIBRATION"
            ),
            "score_space": (
                model.score_space
            ),
            "component": (
                "BUY_AND_SELL_AGGREGATE"
            ),
            "effective_parameter_count_applicable": False,
            "cell_count": int(
                2
                * calibration_grid.cell_count
            ),
            "exposure_seconds": (
                calibration_exposure_seconds
            ),
            "observed_event_count": (
                calibration_observed_count
            ),
            "predicted_event_count": (
                calibration_predicted_count
            ),
            "observed_minus_predicted": (
                calibration_observed_count
                - calibration_predicted_count
            ),
            "log_score_total": (
                calibration_joint_log_score
            ),
            "log_score_per_second": (
                calibration_joint_log_score
                / calibration_exposure_seconds
            ),
            "log_score_per_observed_event": (
                calibration_joint_log_score
                / calibration_observed_count
            ),
            "locked_before_calibration": True,
            "status": "PASS",
        }
    )


# Retain DEVELOPMENT forecasts only for the locked family winners.
for retained_model_id in (
    RETAINED_ADAPTIVE_MODEL_IDS
):
    retained_model = (
        ADAPTIVE_RATE_MODELS[
            retained_model_id
        ]
    )

    retained_development_forecasts = (
        adaptive_model_forecasts(
            model=retained_model,
            partition_name="DEVELOPMENT",
        )
    )

    for side in (
        "BUY",
        "SELL",
    ):
        (
            intensity_per_second,
            expected_counts,
        ) = retained_development_forecasts[
            side
        ]

        retained_intensity = np.asarray(
            intensity_per_second,
            dtype="float64",
        ).copy()

        retained_expected_counts = np.asarray(
            expected_counts,
            dtype="float64",
        ).copy()

        retained_intensity.setflags(
            write=False
        )
        retained_expected_counts.setflags(
            write=False
        )

        SELECTED_ADAPTIVE_INTENSITIES[
            (
                retained_model_id,
                "DEVELOPMENT",
                side,
            )
        ] = retained_intensity

        SELECTED_ADAPTIVE_EXPECTED_COUNTS[
            (
                retained_model_id,
                "DEVELOPMENT",
                side,
            )
        ] = retained_expected_counts


# ------------------------------------------------------------
# Final adaptive tables
# ------------------------------------------------------------

ADAPTIVE_RATE_SCORE_TABLE = (
    pd.DataFrame(
        adaptive_score_rows
    )
    .sort_values(
        [
            "scored_partition",
            "model_family",
            "hyperparameter_ms",
            "component",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

ADAPTIVE_RATE_RESIDUAL_SUMMARY = (
    pd.DataFrame(
        adaptive_residual_rows
    )
    .sort_values(
        [
            "scored_partition",
            "model_family",
            "hyperparameter_ms",
            "component",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

ADAPTIVE_RATE_INTENSITY_RANGES = (
    pd.DataFrame(
        adaptive_intensity_range_rows
    )
    .sort_values(
        [
            "scored_partition",
            "model_family",
            "hyperparameter_ms",
            "component",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


ADAPTIVE_RATE_MODEL_REGISTRY = pd.DataFrame(
    [
        {
            "model_id": model.model_id,
            "model_family": (
                model.model_family
            ),
            "hyperparameter_name": (
                model.hyperparameter_name
            ),
            "hyperparameter_ms": (
                model.hyperparameter_ms
            ),
            "fitted_partition": (
                model.fitted_partition
            ),
            "score_space": (
                model.score_space
            ),
            "prior_event_equivalent": (
                ADAPTIVE_PRIOR_EVENT_EQUIVALENT
            ),
            "buy_prior_rate_per_second": (
                DEVELOPMENT_BUY_RATE_PER_SECOND
            ),
            "sell_prior_rate_per_second": (
                DEVELOPMENT_SELL_RATE_PER_SECOND
            ),
            "effective_parameter_count_applicable": (
                model.effective_parameter_count_applicable
            ),
            "aic_applicable": False,
            "bic_applicable": False,
            "selected_within_family_flag": (
                model.model_id
                in {
                    SELECTED_ROLLING_MODEL_ID,
                    SELECTED_EWMA_MODEL_ID,
                }
            ),
            "selected_overall_flag": (
                model.model_id
                == SELECTED_ADAPTIVE_MODEL_ID
            ),
            "calibration_used_for_selection": False,
            "status": "PASS",
        }
        for model
        in ADAPTIVE_RATE_MODELS.values()
    ]
).sort_values(
    [
        "model_family",
        "hyperparameter_ms",
    ],
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# Compare adaptive candidates with selected deterministic baseline
# ------------------------------------------------------------

adaptive_aggregate_scores = (
    ADAPTIVE_RATE_SCORE_TABLE.loc[
        ADAPTIVE_RATE_SCORE_TABLE[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        )
    ]
    .copy()
)

selected_deterministic_scores = (
    DETERMINISTIC_POISSON_SCORE_TABLE.loc[
        DETERMINISTIC_POISSON_SCORE_TABLE[
            "model_id"
        ].eq(
            SELECTED_DETERMINISTIC_MODEL_ID
        )
        & DETERMINISTIC_POISSON_SCORE_TABLE[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        ),
        [
            "scored_partition",
            "log_score_total",
        ],
    ]
    .rename(
        columns={
            "log_score_total": (
                "selected_deterministic_log_score"
            )
        }
    )
)

ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON = (
    adaptive_aggregate_scores.merge(
        selected_deterministic_scores,
        on="scored_partition",
        how="left",
        validate="many_to_one",
    )
)

ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
    "log_score_improvement_over_selected_deterministic"
] = (
    ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
        "log_score_total"
    ]
    - ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
        "selected_deterministic_log_score"
    ]
)

ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
    "selected_within_family_flag"
] = (
    ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
        "model_id"
    ].isin(
        {
            SELECTED_ROLLING_MODEL_ID,
            SELECTED_EWMA_MODEL_ID,
        }
    )
)

ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
    "selected_overall_flag"
] = (
    ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
        "model_id"
    ].eq(
        SELECTED_ADAPTIVE_MODEL_ID
    )
)

ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON = (
    ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON.sort_values(
        [
            "scored_partition",
            "log_score_total",
        ],
        ascending=[
            True,
            False,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Selected adaptive summary
# ------------------------------------------------------------

SELECTED_ADAPTIVE_MODEL = (
    ADAPTIVE_RATE_MODELS[
        SELECTED_ADAPTIVE_MODEL_ID
    ]
)

selected_adaptive_score_rows = (
    ADAPTIVE_RATE_SCORE_TABLE.loc[
        ADAPTIVE_RATE_SCORE_TABLE[
            "model_id"
        ].eq(
            SELECTED_ADAPTIVE_MODEL_ID
        )
        & ADAPTIVE_RATE_SCORE_TABLE[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        )
    ]
)

require(
    set(
        selected_adaptive_score_rows[
            "scored_partition"
        ]
    )
    == set(ANALYTICAL_PARTITIONS),
    (
        "Selected adaptive model lacks DEVELOPMENT or locked "
        "CALIBRATION aggregate scoring."
    ),
)


# ------------------------------------------------------------
# Adaptive-rate gates
# ------------------------------------------------------------

expected_candidate_count = (
    len(
        ROLLING_RATE_WINDOWS_MS
    )
    + len(
        EWMA_HALF_LIVES_MS
    )
)

expected_component_score_rows = (
    expected_candidate_count
    * len(
        ANALYTICAL_PARTITIONS
    )
    * 2
)

observed_component_score_rows = int(
    ADAPTIVE_RATE_SCORE_TABLE[
        "component"
    ].isin(
        {
            "BUY",
            "SELL",
        }
    ).sum()
)

ADAPTIVE_RATE_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "adaptive_candidate_grid_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    ADAPTIVE_RATE_MODELS
                )
                == expected_candidate_count
            ),
            "evidence": (
                f"candidate_count="
                f"{len(ADAPTIVE_RATE_MODELS)}"
            ),
        },
        {
            "gate": "development_selection_excludes_calibration",
            "severity": "BLOCKING",
            "passed": bool(
                not ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
                    "calibration_used_for_selection"
                ].any()
                and ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
                    "selection_partition"
                ].eq(
                    "DEVELOPMENT"
                ).all()
            ),
            "evidence": (
                f"selection_rule="
                f"{ADAPTIVE_SELECTION_RULE}"
            ),
        },
        {
            "gate": "rolling_family_winner_frozen",
            "severity": "BLOCKING",
            "passed": (
                SELECTED_ROLLING_MODEL_ID
                in ADAPTIVE_RATE_MODELS
            ),
            "evidence": (
                f"selected_rolling_model="
                f"{SELECTED_ROLLING_MODEL_ID}"
            ),
        },
        {
            "gate": "ewma_family_winner_frozen",
            "severity": "BLOCKING",
            "passed": (
                SELECTED_EWMA_MODEL_ID
                in ADAPTIVE_RATE_MODELS
            ),
            "evidence": (
                f"selected_ewma_model="
                f"{SELECTED_EWMA_MODEL_ID}"
            ),
        },
        {
            "gate": "overall_adaptive_winner_frozen",
            "severity": "BLOCKING",
            "passed": (
                SELECTED_ADAPTIVE_MODEL_ID
                in RETAINED_ADAPTIVE_MODEL_IDS
            ),
            "evidence": (
                f"selected_adaptive_model="
                f"{SELECTED_ADAPTIVE_MODEL_ID}"
            ),
        },
        {
            "gate": "adaptive_component_scores_complete",
            "severity": "BLOCKING",
            "passed": (
                observed_component_score_rows
                == expected_component_score_rows
            ),
            "evidence": (
                f"observed_component_scores="
                f"{observed_component_score_rows}; "
                f"expected="
                f"{expected_component_score_rows}"
            ),
        },
        {
            "gate": "all_adaptive_scores_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    ADAPTIVE_RATE_SCORE_TABLE[
                        "log_score_total"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                f"score_rows="
                f"{len(ADAPTIVE_RATE_SCORE_TABLE)}"
            ),
        },
        {
            "gate": "all_adaptive_intensities_finite_positive",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    ADAPTIVE_RATE_INTENSITY_RANGES[
                        [
                            "minimum_intensity_per_second",
                            "median_intensity_per_second",
                            "maximum_intensity_per_second",
                            "exposure_weighted_mean_intensity_per_second",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
                and ADAPTIVE_RATE_INTENSITY_RANGES[
                    "minimum_intensity_per_second"
                ].gt(0.0).all()
            ),
            "evidence": (
                f"intensity_range_rows="
                f"{len(ADAPTIVE_RATE_INTENSITY_RANGES)}"
            ),
        },
        {
            "gate": "calibration_history_carried_from_development",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "rolling windows include prior DEVELOPMENT "
                "events; EWMA state is carried across the exact "
                "DEVELOPMENT/CALIBRATION boundary"
            ),
        },
        {
            "gate": "current_cell_events_excluded_from_forecasts",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "all native-cell forecasts use events strictly "
                "before the cell start"
            ),
        },
        {
            "gate": "timestamp_jitter_and_event_reordering_absent",
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "canonical Notebook 04 event_time_ns and exact "
                "batch multiplicities used unchanged"
            ),
        },
        {
            "gate": "aic_bic_not_misapplied_to_adaptive_models",
            "severity": "BLOCKING",
            "passed": bool(
                not ADAPTIVE_RATE_MODEL_REGISTRY[
                    "aic_applicable"
                ].any()
                and not ADAPTIVE_RATE_MODEL_REGISTRY[
                    "bic_applicable"
                ].any()
            ),
            "evidence": (
                "adaptive effective degrees of freedom are not "
                "treated as ordinary parametric counts"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    ADAPTIVE_RATE_GATE_FRAME.loc[
        ADAPTIVE_RATE_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one causal adaptive-rate baseline gate "
        "failed."
    ),
)


display(ADAPTIVE_PRIOR_CONTRACT)

display(
    ADAPTIVE_DEVELOPMENT_SELECTION_TABLE[
        [
            "model_id",
            "model_family",
            "hyperparameter_name",
            "hyperparameter_ms",
            "development_joint_log_score",
            "development_log_score_per_second",
            "development_log_score_per_observed_event",
            "development_predicted_event_count",
            "selected_within_family_flag",
            "selected_overall_flag",
            "selection_rule",
        ]
    ]
)

display(
    ADAPTIVE_RATE_LOCKED_SCORE_COMPARISON[
        [
            "model_id",
            "model_family",
            "hyperparameter_ms",
            "scored_partition",
            "observed_event_count",
            "predicted_event_count",
            "log_score_total",
            "log_score_per_second",
            "log_score_per_observed_event",
            "log_score_improvement_over_selected_deterministic",
            "selected_within_family_flag",
            "selected_overall_flag",
            "status",
        ]
    ]
)

display(
    ADAPTIVE_RATE_INTENSITY_RANGES.loc[
        ADAPTIVE_RATE_INTENSITY_RANGES[
            "model_id"
        ].isin(
            RETAINED_ADAPTIVE_MODEL_IDS
        ),
        [
            "model_id",
            "scored_partition",
            "component",
            "minimum_intensity_per_second",
            "q05_intensity_per_second",
            "median_intensity_per_second",
            "q95_intensity_per_second",
            "maximum_intensity_per_second",
            "exposure_weighted_mean_intensity_per_second",
            "status",
        ],
    ]
)

display(
    ADAPTIVE_RATE_RESIDUAL_SUMMARY.loc[
        ADAPTIVE_RATE_RESIDUAL_SUMMARY[
            "model_id"
        ].isin(
            RETAINED_ADAPTIVE_MODEL_IDS
        ),
        [
            "model_id",
            "scored_partition",
            "component",
            "pearson_residual_mean",
            "pearson_residual_variance",
            "deviance_residual_mean",
            "deviance_residual_variance",
            "pearson_dispersion_ratio",
            "maximum_absolute_pearson_residual",
            "status",
        ],
    ]
)

display(ADAPTIVE_RATE_MODEL_REGISTRY)
display(ADAPTIVE_RATE_GATE_FRAME)

print(
    "Causal rolling-window and exponentially weighted rate "
    "candidates were evaluated on DEVELOPMENT."
)
print(
    f"Selected rolling model: "
    f"{SELECTED_ROLLING_MODEL_ID}."
)
print(
    f"Selected exponentially weighted model: "
    f"{SELECTED_EWMA_MODEL_ID}."
)
print(
    f"Selected overall adaptive-rate model: "
    f"{SELECTED_ADAPTIVE_MODEL_ID}."
)
print(
    "All adaptive hyperparameters were frozen using DEVELOPMENT "
    "before CALIBRATION forecasts were constructed."
)
print(
    "CALIBRATION rolling histories include prior DEVELOPMENT "
    "events, and exponentially weighted states were carried "
    "across the exact partition boundary without reset."
)
print(
    "Every forecast excludes current-cell events and uses "
    "canonical Notebook 04 timestamps without jitter."
)
print(
    "Hawkes estimation remains unauthorized."
)

,side,prior_event_equivalent,development_prior_rate_per_second,prior_exposure_seconds,prior_mean_rate_per_second,prior_selected_using_calibration,status
0,BUY,1,1.8947092,0.52778547,1.8947092,False,PASS
1,SELL,1,1.9923861,0.50191075,1.9923861,False,PASS


,model_id,model_family,hyperparameter_name,hyperparameter_ms,development_joint_log_score,development_log_score_per_second,development_log_score_per_observed_event,development_predicted_event_count,selected_within_family_flag,selected_overall_flag,selection_rule
0,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,250,"-50,128.377",-27.820357,-7.1571069,"7,003.7548",True,True,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
1,E_SIDE_EWMA_500MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,500,"-50,314.798",-27.923817,-7.1837233,"7,003.4047",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
2,E_SIDE_EWMA_1000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,1000,"-50,439.549",-27.993052,-7.2015347,"7,003.0237",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
3,E_SIDE_EWMA_2000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,2000,"-50,511.22",-28.032828,-7.2117676,"7,002.0355",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
4,E_SIDE_EWMA_5000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,5000,"-50,592.434",-28.0779,-7.2233629,"6,993.9568",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
5,E_SIDE_EWMA_10000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,10000,"-50,651.288",-28.110563,-7.2317659,"6,977.0436",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
6,E_SIDE_EWMA_30000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,30000,"-50,733.289",-28.156073,-7.2434736,"6,946.1672",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
7,E_SIDE_EWMA_60000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,60000,"-50,779.536",-28.181738,-7.2500765,"6,935.2598",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
8,R_SIDE_ROLLING_250MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,window_ms,250,"-50,304.35",-27.918019,-7.1822316,"7,004.0171",True,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...
9,R_SIDE_ROLLING_500MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,window_ms,500,"-50,626.45",-28.096778,-7.2282195,"7,003.9228",False,False,MAXIMUM_DEVELOPMENT_JOINT_LOG_SCORE_WITH_LONGE...


,model_id,model_family,hyperparameter_ms,scored_partition,observed_event_count,predicted_event_count,log_score_total,log_score_per_second,log_score_per_observed_event,log_score_improvement_over_selected_deterministic,selected_within_family_flag,selected_overall_flag,status
0,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,250,CALIBRATION,2493,"2,673.4791","-18,072.443",-25.09019,-7.2492753,368.33279,True,True,PASS
1,R_SIDE_ROLLING_250MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,250,CALIBRATION,2493,"2,699.4526","-18,095.796",-25.122611,-7.2586426,344.98001,True,False,PASS
2,E_SIDE_EWMA_500MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,500,CALIBRATION,2493,"2,620.6337","-18,149.545",-25.197232,-7.2802026,291.23087,False,False,PASS
3,E_SIDE_EWMA_1000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,1000,CALIBRATION,2493,"2,572.3327","-18,206.674",-25.276544,-7.3031184,234.10202,False,False,PASS
4,E_SIDE_EWMA_2000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,2000,CALIBRATION,2493,"2,535.7135","-18,251.477",-25.338745,-7.3210898,189.29914,False,False,PASS
5,R_SIDE_ROLLING_500MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,500,CALIBRATION,2493,"2,648.7121","-18,257.954",-25.347737,-7.3236881,182.82175,False,False,PASS
6,E_SIDE_EWMA_5000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,5000,CALIBRATION,2493,"2,507.5228","-18,308.922",-25.418496,-7.3441324,131.85394,False,False,PASS
7,E_SIDE_EWMA_10000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,10000,CALIBRATION,2493,"2,501.7858","-18,347.916",-25.472632,-7.3597737,92.860305,False,False,PASS
8,R_SIDE_ROLLING_5000MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,5000,CALIBRATION,2493,"2,517.6548","-18,367.773",-25.5002,-7.3677389,73.002885,False,False,PASS
9,R_SIDE_ROLLING_1000MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,1000,CALIBRATION,2493,"2,597.9976","-18,375.613",-25.511084,-7.3708837,65.163088,False,False,PASS


,model_id,scored_partition,component,minimum_intensity_per_second,q05_intensity_per_second,median_intensity_per_second,q95_intensity_per_second,maximum_intensity_per_second,exposure_weighted_mean_intensity_per_second,status
0,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,BUY,1.1255441,1.126954,1.4686757,3.4084222,26.250772,1.8187851,PASS
1,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,SELL,1.159307,1.1747935,1.6874665,3.183191,19.713889,1.8928381,PASS
16,R_SIDE_ROLLING_250MS_POISSON,CALIBRATION,BUY,1.2857016,1.2857016,1.2857016,3.8571047,34.713942,1.8345752,PASS
17,R_SIDE_ROLLING_250MS_POISSON,CALIBRATION,SELL,1.3299451,1.3299451,1.3299451,3.9898352,25.268957,1.9131072,PASS
32,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,BUY,1.1255441,1.1300257,1.6090052,3.3898053,30.129944,1.8947567,PASS
33,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,SELL,1.1593172,1.1807661,1.7828219,3.4136959,19.298889,1.9922026,PASS
48,R_SIDE_ROLLING_250MS_POISSON,DEVELOPMENT,BUY,1.2857016,1.2857016,1.2857016,3.8571047,41.14245,1.8947462,PASS
49,R_SIDE_ROLLING_250MS_POISSON,DEVELOPMENT,SELL,1.3299451,1.3299451,1.3299451,3.9898352,27.928847,1.9923586,PASS


,model_id,scored_partition,component,pearson_residual_mean,pearson_residual_variance,deviance_residual_mean,deviance_residual_variance,pearson_dispersion_ratio,maximum_absolute_pearson_residual,status
0,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,BUY,-0.0042929484,0.95465714,-0.05320374,0.018366186,0.95467557,87.624006,PASS
1,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,SELL,-0.0018291475,1.0089323,-0.05471922,0.019252451,1.0089356,59.27423,PASS
16,R_SIDE_ROLLING_250MS_POISSON,CALIBRATION,BUY,-0.0037732383,0.9867077,-0.053015522,0.018419431,0.98672194,111.51936,PASS
17,R_SIDE_ROLLING_250MS_POISSON,CALIBRATION,SELL,-0.0014011302,1.0317052,-0.054463747,0.019311934,1.0317072,82.226505,PASS
32,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,BUY,-0.00054343955,1.0181026,-0.053902953,0.020344641,1.0181029,84.241281,PASS
33,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,SELL,0.0013118171,1.0775407,-0.055437414,0.021643041,1.0775425,67.204413,PASS
48,R_SIDE_ROLLING_250MS_POISSON,DEVELOPMENT,BUY,0.00080222724,1.0735883,-0.053383128,0.020511339,1.073589,78.830745,PASS
49,R_SIDE_ROLLING_250MS_POISSON,DEVELOPMENT,SELL,0.002624966,1.129411,-0.054786836,0.021799146,1.1294179,63.262901,PASS


,model_id,model_family,hyperparameter_name,hyperparameter_ms,fitted_partition,score_space,prior_event_equivalent,buy_prior_rate_per_second,sell_prior_rate_per_second,effective_parameter_count_applicable,aic_applicable,bic_applicable,selected_within_family_flag,selected_overall_flag,calibration_used_for_selection,status
0,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,250,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,True,True,False,PASS
1,E_SIDE_EWMA_500MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,500,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS
2,E_SIDE_EWMA_1000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,1000,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS
3,E_SIDE_EWMA_2000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,2000,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS
4,E_SIDE_EWMA_5000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,5000,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS
5,E_SIDE_EWMA_10000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,10000,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS
6,E_SIDE_EWMA_30000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,30000,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS
7,E_SIDE_EWMA_60000MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,half_life_ms,60000,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS
8,R_SIDE_ROLLING_250MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,window_ms,250,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,True,False,False,PASS
9,R_SIDE_ROLLING_500MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,window_ms,500,DEVELOPMENT,BIVARIATE_SIDE_COUNT,1,1.8947092,1.9923861,False,False,False,False,False,False,PASS


,gate,severity,passed,evidence
0,adaptive_candidate_grid_complete,BLOCKING,True,candidate_count=16
1,development_selection_excludes_calibration,BLOCKING,True,selection_rule=MAXIMUM_DEVELOPMENT_JOINT_LOG_S...
2,rolling_family_winner_frozen,BLOCKING,True,selected_rolling_model=R_SIDE_ROLLING_250MS_PO...
3,ewma_family_winner_frozen,BLOCKING,True,selected_ewma_model=E_SIDE_EWMA_250MS_POISSON
4,overall_adaptive_winner_frozen,BLOCKING,True,selected_adaptive_model=E_SIDE_EWMA_250MS_POISSON
5,adaptive_component_scores_complete,BLOCKING,True,observed_component_scores=64; expected=64
6,all_adaptive_scores_finite,BLOCKING,True,score_rows=96
7,all_adaptive_intensities_finite_positive,BLOCKING,True,intensity_range_rows=64
8,calibration_history_carried_from_development,BLOCKING,True,rolling windows include prior DEVELOPMENT even...
9,current_cell_events_excluded_from_forecasts,BLOCKING,True,all native-cell forecasts use events strictly ...


Causal rolling-window and exponentially weighted rate candidates were evaluated on DEVELOPMENT.
Selected rolling model: R_SIDE_ROLLING_250MS_POISSON.
Selected exponentially weighted model: E_SIDE_EWMA_250MS_POISSON.
Selected overall adaptive-rate model: E_SIDE_EWMA_250MS_POISSON.
All adaptive hyperparameters were frozen using DEVELOPMENT before CALIBRATION forecasts were constructed.
CALIBRATION rolling histories include prior DEVELOPMENT events, and exponentially weighted states were carried across the exact partition boundary without reset.
Every forecast excludes current-cell events and uses canonical Notebook 04 timestamps without jitter.
Hawkes estimation remains unauthorized.


In [13]:
# ============================================================
# Independent renewal-process baselines
# ============================================================

RENEWAL_SELECTION_RULE: Final[str] = (
    "MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICITY_TIE_BREAK"
)

RENEWAL_BIC_TIE_THRESHOLD: Final[float] = 2.0

RENEWAL_FAMILY_COMPLEXITY_ORDER: Final[
    Mapping[str, int]
] = {
    "EXPONENTIAL": 0,
    "GAMMA": 1,
    "WEIBULL": 2,
}

RENEWAL_SHAPE_BOUNDS: Final[
    tuple[float, float]
] = (
    0.05,
    20.0,
)

RENEWAL_OPTIMIZER_METHOD: Final[str] = (
    "L-BFGS-B"
)

RENEWAL_OPTIMIZER_MAX_ITERATIONS: Final[int] = 1_000

RENEWAL_MODEL_SCORE_SPACE: Final[str] = (
    "RENEWAL_DURATION_WITH_RIGHT_CENSORING"
)


# ------------------------------------------------------------
# Renewal sample extraction
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class RenewalSample:
    event_partition: str
    process_name: str
    complete_durations_seconds: np.ndarray
    right_censored_duration_seconds: float
    left_censored_duration_seconds: float
    arrival_count: int
    complete_duration_count: int


def renewal_sample(
    *,
    partition_name: str,
    process_name: str,
) -> RenewalSample:
    """
    Return complete within-partition durations and edge censoring.

    The left-censored first waiting time is recorded but excluded from
    likelihood construction. The right-censored terminal waiting time
    contributes through the fitted survival function.
    """
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized renewal partition: "
            f"{partition_name}"
        ),
    )
    require(
        process_name
        in DURATION_PROCESS_NAMES,
        (
            f"Unsupported renewal process: "
            f"{process_name}"
        ),
    )

    duration_rows = (
        INTERARRIVAL_DURATION_TABLE.loc[
            INTERARRIVAL_DURATION_TABLE[
                "event_partition"
            ].eq(partition_name)
            & INTERARRIVAL_DURATION_TABLE[
                "process_name"
            ].eq(process_name)
        ]
        .copy()
        .sort_values(
            "duration_index",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    censoring_rows = (
        INTERARRIVAL_CENSORING_AUDIT.loc[
            INTERARRIVAL_CENSORING_AUDIT[
                "event_partition"
            ].eq(partition_name)
            & INTERARRIVAL_CENSORING_AUDIT[
                "process_name"
            ].eq(process_name)
        ]
    )

    require(
        len(censoring_rows) == 1,
        (
            f"{partition_name} {process_name} must have one "
            "censoring-audit row."
        ),
    )

    censoring_row = (
        censoring_rows.iloc[0]
    )

    complete_durations_seconds = (
        duration_rows[
            "duration_seconds"
        ]
        .astype("float64")
        .to_numpy()
    )

    right_censored_duration_seconds = float(
        censoring_row[
            "right_censored_duration_seconds"
        ]
    )

    left_censored_duration_seconds = float(
        censoring_row[
            "left_censored_duration_seconds"
        ]
    )

    arrival_count = int(
        censoring_row["arrival_count"]
    )

    complete_duration_count = int(
        censoring_row[
            "complete_duration_count"
        ]
    )

    require(
        complete_durations_seconds.size
        == complete_duration_count,
        (
            f"{partition_name} {process_name} complete-duration "
            "count differs from its censoring audit."
        ),
    )
    require(
        complete_duration_count
        == arrival_count - 1,
        (
            f"{partition_name} {process_name} renewal duration "
            "count must equal arrival count minus one."
        ),
    )
    require(
        complete_duration_count
        >= MINIMUM_RENEWAL_DURATIONS,
        (
            f"{partition_name} {process_name} has fewer than "
            f"{MINIMUM_RENEWAL_DURATIONS} complete durations."
        ),
    )
    require(
        np.isfinite(
            complete_durations_seconds
        ).all(),
        (
            f"{partition_name} {process_name} contains "
            "nonfinite complete durations."
        ),
    )
    require(
        np.all(
            complete_durations_seconds > 0.0
        ),
        (
            f"{partition_name} {process_name} contains "
            "nonpositive complete durations."
        ),
    )
    require(
        np.isfinite(
            right_censored_duration_seconds
        )
        and right_censored_duration_seconds > 0.0,
        (
            f"{partition_name} {process_name} has an invalid "
            "right-censored duration."
        ),
    )
    require(
        np.isfinite(
            left_censored_duration_seconds
        )
        and left_censored_duration_seconds >= 0.0,
        (
            f"{partition_name} {process_name} has an invalid "
            "left-censored duration."
        ),
    )

    complete_durations_seconds.setflags(
        write=False
    )

    return RenewalSample(
        event_partition=partition_name,
        process_name=process_name,
        complete_durations_seconds=(
            complete_durations_seconds
        ),
        right_censored_duration_seconds=(
            right_censored_duration_seconds
        ),
        left_censored_duration_seconds=(
            left_censored_duration_seconds
        ),
        arrival_count=arrival_count,
        complete_duration_count=(
            complete_duration_count
        ),
    )


RENEWAL_SAMPLES: Final[
    Mapping[tuple[str, str], RenewalSample]
] = {
    (
        partition_name,
        process_name,
    ): renewal_sample(
        partition_name=partition_name,
        process_name=process_name,
    )
    for partition_name in ANALYTICAL_PARTITIONS
    for process_name in DURATION_PROCESS_NAMES
}


# ------------------------------------------------------------
# Renewal likelihood functions
# ------------------------------------------------------------

def renewal_log_likelihood(
    *,
    family: str,
    parameter_names: Sequence[str],
    parameter_values: Sequence[float],
    complete_durations_seconds: np.ndarray,
    right_censored_duration_seconds: float,
) -> float:
    """
    Return complete-duration log densities plus terminal log survival.

    The left-censored first waiting time is deliberately excluded.
    """
    durations = np.asarray(
        complete_durations_seconds,
        dtype="float64",
    )

    require(
        family in RENEWAL_FAMILIES,
        f"Unsupported renewal family: {family}",
    )
    require(
        durations.ndim == 1,
        "Renewal durations must be one-dimensional.",
    )
    require(
        durations.size >= 1,
        "Renewal likelihood requires complete durations.",
    )
    require(
        np.all(
            durations > 0.0
        )
        and np.isfinite(
            durations
        ).all(),
        "Renewal durations must be finite and positive.",
    )
    require(
        np.isfinite(
            right_censored_duration_seconds
        )
        and right_censored_duration_seconds > 0.0,
        "Right-censored duration must be finite and positive.",
    )
    require(
        len(parameter_names)
        == len(parameter_values),
        "Renewal parameter names and values differ in length.",
    )

    parameters = {
        str(name): float(value)
        for name, value in zip(
            parameter_names,
            parameter_values,
            strict=True,
        )
    }

    if family == "EXPONENTIAL":
        rate = parameters["rate_per_second"]

        require(
            np.isfinite(rate)
            and rate > 0.0,
            "Exponential rate must be finite and positive.",
        )

        complete_log_density = (
            math.log(rate)
            - rate * durations
        )

        right_log_survival = (
            -rate
            * right_censored_duration_seconds
        )

    elif family == "GAMMA":
        shape = parameters["shape"]
        scale_seconds = parameters[
            "scale_seconds"
        ]

        require(
            np.isfinite(shape)
            and shape > 0.0,
            "Gamma shape must be finite and positive.",
        )
        require(
            np.isfinite(scale_seconds)
            and scale_seconds > 0.0,
            "Gamma scale must be finite and positive.",
        )

        complete_log_density = (
            stats.gamma.logpdf(
                durations,
                a=shape,
                loc=0.0,
                scale=scale_seconds,
            )
        )

        right_log_survival = float(
            stats.gamma.logsf(
                right_censored_duration_seconds,
                a=shape,
                loc=0.0,
                scale=scale_seconds,
            )
        )

    elif family == "WEIBULL":
        shape = parameters["shape"]
        scale_seconds = parameters[
            "scale_seconds"
        ]

        require(
            np.isfinite(shape)
            and shape > 0.0,
            "Weibull shape must be finite and positive.",
        )
        require(
            np.isfinite(scale_seconds)
            and scale_seconds > 0.0,
            "Weibull scale must be finite and positive.",
        )

        scaled_durations = (
            durations / scale_seconds
        )

        complete_log_density = (
            math.log(shape)
            - math.log(scale_seconds)
            + (
                shape - 1.0
            )
            * np.log(
                scaled_durations
            )
            - np.power(
                scaled_durations,
                shape,
            )
        )

        right_log_survival = float(
            -(
                right_censored_duration_seconds
                / scale_seconds
            )
            ** shape
        )

    else:
        raise ValueError(
            f"Unsupported renewal family: {family}"
        )

    require(
        np.isfinite(
            complete_log_density
        ).all(),
        (
            f"{family} complete-duration log densities contain "
            "nonfinite values."
        ),
    )
    require(
        np.isfinite(
            right_log_survival
        ),
        (
            f"{family} terminal log survival is nonfinite."
        ),
    )

    return float(
        complete_log_density.sum(
            dtype="float64"
        )
        + right_log_survival
    )


def renewal_cumulative_hazard(
    *,
    family: str,
    parameter_names: Sequence[str],
    parameter_values: Sequence[float],
    durations_seconds: np.ndarray,
) -> np.ndarray:
    """Return fitted cumulative hazards for complete durations."""
    durations = np.asarray(
        durations_seconds,
        dtype="float64",
    )

    parameters = {
        str(name): float(value)
        for name, value in zip(
            parameter_names,
            parameter_values,
            strict=True,
        )
    }

    if family == "EXPONENTIAL":
        transformed = (
            parameters["rate_per_second"]
            * durations
        )

    elif family == "GAMMA":
        log_survival = stats.gamma.logsf(
            durations,
            a=parameters["shape"],
            loc=0.0,
            scale=parameters[
                "scale_seconds"
            ],
        )

        transformed = -log_survival

    elif family == "WEIBULL":
        transformed = np.power(
            durations
            / parameters[
                "scale_seconds"
            ],
            parameters["shape"],
        )

    else:
        raise ValueError(
            f"Unsupported renewal family: {family}"
        )

    transformed = np.asarray(
        transformed,
        dtype="float64",
    )

    require(
        np.isfinite(
            transformed
        ).all(),
        (
            f"{family} cumulative hazards contain nonfinite "
            "values."
        ),
    )
    require(
        np.all(
            transformed > 0.0
        ),
        (
            f"{family} cumulative hazards must be strictly "
            "positive."
        ),
    )

    return transformed


# ------------------------------------------------------------
# Renewal fitting
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class RenewalModelFit:
    model_id: str
    process_name: str
    family: str
    fitted_partition: str
    parameter_names: tuple[str, ...]
    parameter_values: tuple[float, ...]
    parameter_count: int
    development_complete_duration_count: int
    development_right_censored_duration_seconds: float
    development_log_score: float
    optimizer_method: str
    optimizer_success: bool
    optimizer_status: int
    optimizer_message: str
    optimizer_iterations: int
    objective_evaluations: int


def renewal_model_id(
    *,
    process_name: str,
    family: str,
) -> str:
    """Return a deterministic renewal-model identifier."""
    return (
        "N_"
        f"{process_name}_"
        f"{family}_RENEWAL"
    )


def renewal_scale_bounds(
    sample: RenewalSample,
) -> tuple[float, float]:
    """Return data-scaled positive bounds for renewal scale."""
    durations = (
        sample.complete_durations_seconds
    )

    minimum_observed_time = min(
        float(np.min(durations)),
        sample.right_censored_duration_seconds,
    )

    maximum_observed_time = max(
        float(np.max(durations)),
        sample.right_censored_duration_seconds,
    )

    lower_scale = max(
        np.finfo(np.float64).tiny,
        minimum_observed_time / 100.0,
    )

    upper_scale = max(
        maximum_observed_time * 100.0,
        float(np.mean(durations)) * 100.0,
    )

    require(
        np.isfinite(lower_scale)
        and lower_scale > 0.0,
        "Renewal lower-scale bound is invalid.",
    )
    require(
        np.isfinite(upper_scale)
        and upper_scale > lower_scale,
        "Renewal upper-scale bound is invalid.",
    )

    return (
        float(lower_scale),
        float(upper_scale),
    )


def fit_renewal_model(
    *,
    process_name: str,
    family: str,
    development_sample: RenewalSample,
) -> RenewalModelFit:
    """Fit one renewal family on DEVELOPMENT only."""
    require(
        development_sample.event_partition
        == "DEVELOPMENT",
        "Renewal fitting must use DEVELOPMENT only.",
    )
    require(
        development_sample.process_name
        == process_name,
        "Renewal sample process does not match fit request.",
    )
    require(
        family in RENEWAL_FAMILIES,
        f"Unsupported renewal family: {family}",
    )

    durations = (
        development_sample[
            "complete_durations_seconds"
        ]
        if isinstance(
            development_sample,
            Mapping,
        )
        else development_sample.complete_durations_seconds
    )

    right_censored_duration = (
        development_sample[
            "right_censored_duration_seconds"
        ]
        if isinstance(
            development_sample,
            Mapping,
        )
        else development_sample.right_censored_duration_seconds
    )

    duration_count = int(
        durations.size
    )

    total_observed_waiting_time = float(
        durations.sum(
            dtype="float64"
        )
        + right_censored_duration
    )

    require(
        total_observed_waiting_time > 0.0,
        "Total observed renewal waiting time must be positive.",
    )

    model_id = renewal_model_id(
        process_name=process_name,
        family=family,
    )

    if family == "EXPONENTIAL":
        rate_per_second = (
            duration_count
            / total_observed_waiting_time
        )

        parameter_names = (
            "rate_per_second",
        )
        parameter_values = (
            float(rate_per_second),
        )

        development_log_score = (
            renewal_log_likelihood(
                family=family,
                parameter_names=(
                    parameter_names
                ),
                parameter_values=(
                    parameter_values
                ),
                complete_durations_seconds=(
                    durations
                ),
                right_censored_duration_seconds=(
                    right_censored_duration
                ),
            )
        )

        return RenewalModelFit(
            model_id=model_id,
            process_name=process_name,
            family=family,
            fitted_partition="DEVELOPMENT",
            parameter_names=parameter_names,
            parameter_values=parameter_values,
            parameter_count=1,
            development_complete_duration_count=(
                duration_count
            ),
            development_right_censored_duration_seconds=(
                right_censored_duration
            ),
            development_log_score=(
                development_log_score
            ),
            optimizer_method="CLOSED_FORM",
            optimizer_success=True,
            optimizer_status=0,
            optimizer_message=(
                "CLOSED_FORM_RIGHT_CENSORED_EXPONENTIAL_MLE"
            ),
            optimizer_iterations=0,
            objective_evaluations=1,
        )

    lower_scale, upper_scale = (
        renewal_scale_bounds(
            development_sample
        )
    )

    log_shape_bounds = (
        math.log(
            RENEWAL_SHAPE_BOUNDS[0]
        ),
        math.log(
            RENEWAL_SHAPE_BOUNDS[1]
        ),
    )

    log_scale_bounds = (
        math.log(lower_scale),
        math.log(upper_scale),
    )

    sample_mean = float(
        np.mean(durations)
    )

    sample_cv = (
        safe_coefficient_of_variation(
            durations
        )
    )

    if (
        not np.isfinite(sample_cv)
        or sample_cv <= 0.0
    ):
        moment_shape = 1.0
    else:
        moment_shape = float(
            np.clip(
                1.0 / (sample_cv ** 2),
                RENEWAL_SHAPE_BOUNDS[0],
                RENEWAL_SHAPE_BOUNDS[1],
            )
        )

    if family == "GAMMA":
        initial_shape_scale_pairs = (
            (
                moment_shape,
                sample_mean
                / moment_shape,
            ),
            (0.5, sample_mean / 0.5),
            (1.0, sample_mean),
            (2.0, sample_mean / 2.0),
        )

        parameter_names = (
            "shape",
            "scale_seconds",
        )

    elif family == "WEIBULL":
        initial_shape_scale_pairs = (
            (0.5, sample_mean),
            (1.0, sample_mean),
            (1.5, sample_mean),
            (2.0, sample_mean),
        )

        parameter_names = (
            "shape",
            "scale_seconds",
        )

    else:
        raise ValueError(
            f"Unsupported optimized renewal family: {family}"
        )

    def objective(
        log_parameters: np.ndarray,
    ) -> float:
        shape = math.exp(
            float(log_parameters[0])
        )
        scale_seconds = math.exp(
            float(log_parameters[1])
        )

        try:
            log_likelihood = (
                renewal_log_likelihood(
                    family=family,
                    parameter_names=(
                        parameter_names
                    ),
                    parameter_values=(
                        shape,
                        scale_seconds,
                    ),
                    complete_durations_seconds=(
                        durations
                    ),
                    right_censored_duration_seconds=(
                        right_censored_duration
                    ),
                )
            )
        except (
            FloatingPointError,
            RuntimeError,
            ValueError,
        ):
            return float("inf")

        if not np.isfinite(
            log_likelihood
        ):
            return float("inf")

        return -log_likelihood

    optimization_results: list[Any] = []

    for initial_shape, initial_scale in (
        initial_shape_scale_pairs
    ):
        clipped_shape = float(
            np.clip(
                initial_shape,
                RENEWAL_SHAPE_BOUNDS[0],
                RENEWAL_SHAPE_BOUNDS[1],
            )
        )

        clipped_scale = float(
            np.clip(
                initial_scale,
                lower_scale,
                upper_scale,
            )
        )

        optimization_result = (
            optimize.minimize(
                fun=objective,
                x0=np.asarray(
                    [
                        math.log(
                            clipped_shape
                        ),
                        math.log(
                            clipped_scale
                        ),
                    ],
                    dtype="float64",
                ),
                method=(
                    RENEWAL_OPTIMIZER_METHOD
                ),
                bounds=(
                    log_shape_bounds,
                    log_scale_bounds,
                ),
                options={
                    "maxiter": (
                        RENEWAL_OPTIMIZER_MAX_ITERATIONS
                    ),
                    "ftol": 1e-12,
                    "gtol": 1e-8,
                    "maxls": 50,
                },
            )
        )

        if np.isfinite(
            optimization_result.fun
        ):
            optimization_results.append(
                optimization_result
            )

    successful_results = [
        result
        for result in optimization_results
        if bool(result.success)
        and np.isfinite(result.fun)
        and np.isfinite(result.x).all()
    ]

    require(
        successful_results,
        (
            f"{process_name} {family} renewal optimization "
            "produced no successful finite result."
        ),
    )

    best_result = min(
        successful_results,
        key=lambda result: float(
            result.fun
        ),
    )

    fitted_shape = math.exp(
        float(best_result.x[0])
    )
    fitted_scale = math.exp(
        float(best_result.x[1])
    )

    parameter_values = (
        float(fitted_shape),
        float(fitted_scale),
    )

    development_log_score = (
        renewal_log_likelihood(
            family=family,
            parameter_names=(
                parameter_names
            ),
            parameter_values=(
                parameter_values
            ),
            complete_durations_seconds=(
                durations
            ),
            right_censored_duration_seconds=(
                right_censored_duration
            ),
        )
    )

    require(
        np.isclose(
            development_log_score,
            -float(best_result.fun),
            rtol=1e-9,
            atol=1e-8,
        ),
        (
            f"{process_name} {family} persisted renewal score "
            "does not match the optimizer objective."
        ),
    )

    return RenewalModelFit(
        model_id=model_id,
        process_name=process_name,
        family=family,
        fitted_partition="DEVELOPMENT",
        parameter_names=(
            parameter_names
        ),
        parameter_values=(
            parameter_values
        ),
        parameter_count=2,
        development_complete_duration_count=(
            duration_count
        ),
        development_right_censored_duration_seconds=(
            right_censored_duration
        ),
        development_log_score=(
            development_log_score
        ),
        optimizer_method=(
            RENEWAL_OPTIMIZER_METHOD
        ),
        optimizer_success=True,
        optimizer_status=int(
            best_result.status
        ),
        optimizer_message=str(
            best_result.message
        ),
        optimizer_iterations=int(
            getattr(
                best_result,
                "nit",
                -1,
            )
        ),
        objective_evaluations=int(
            getattr(
                best_result,
                "nfev",
                -1,
            )
        ),
    )


RENEWAL_MODEL_FITS: dict[
    str,
    RenewalModelFit,
] = {}

for process_name in (
    DURATION_PROCESS_NAMES
):
    development_sample = (
        RENEWAL_SAMPLES[
            (
                "DEVELOPMENT",
                process_name,
            )
        ]
    )

    for family in RENEWAL_FAMILIES:
        fitted_model = (
            fit_renewal_model(
                process_name=(
                    process_name
                ),
                family=family,
                development_sample=(
                    development_sample
                ),
            )
        )

        require(
            fitted_model.model_id
            not in RENEWAL_MODEL_FITS,
            (
                f"Duplicate renewal model ID: "
                f"{fitted_model.model_id}"
            ),
        )

        RENEWAL_MODEL_FITS[
            fitted_model.model_id
        ] = fitted_model


# ------------------------------------------------------------
# Parameter registry
# ------------------------------------------------------------

renewal_parameter_rows: list[
    dict[str, Any]
] = []

for model in (
    RENEWAL_MODEL_FITS.values()
):
    for parameter_name, parameter_value in zip(
        model.parameter_names,
        model.parameter_values,
        strict=True,
    ):
        renewal_parameter_rows.append(
            {
                "model_id": model.model_id,
                "process_name": (
                    model.process_name
                ),
                "family": model.family,
                "fitted_partition": (
                    model.fitted_partition
                ),
                "parameter_name": (
                    parameter_name
                ),
                "parameter_value": (
                    parameter_value
                ),
                "parameter_count": (
                    model.parameter_count
                ),
                "optimizer_method": (
                    model.optimizer_method
                ),
                "optimizer_success": (
                    model.optimizer_success
                ),
                "optimizer_status": (
                    model.optimizer_status
                ),
                "optimizer_message": (
                    model.optimizer_message
                ),
                "optimizer_iterations": (
                    model.optimizer_iterations
                ),
                "objective_evaluations": (
                    model.objective_evaluations
                ),
                "status": "PASS",
            }
        )


RENEWAL_PARAMETER_TABLE = pd.DataFrame(
    renewal_parameter_rows
).sort_values(
    [
        "process_name",
        "family",
        "parameter_name",
    ],
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# Development selection
# ------------------------------------------------------------

renewal_selection_rows: list[
    dict[str, Any]
] = []

for model in (
    RENEWAL_MODEL_FITS.values()
):
    development_sample = (
        RENEWAL_SAMPLES[
            (
                "DEVELOPMENT",
                model.process_name,
            )
        ]
    )

    selection_observation_count = (
        development_sample.complete_duration_count
        + 1
    )

    development_aic = (
        -2.0
        * model.development_log_score
        + 2.0
        * model.parameter_count
    )

    development_bic = (
        -2.0
        * model.development_log_score
        + model.parameter_count
        * math.log(
            selection_observation_count
        )
    )

    renewal_selection_rows.append(
        {
            "model_id": model.model_id,
            "process_name": (
                model.process_name
            ),
            "family": model.family,
            "parameter_count": (
                model.parameter_count
            ),
            "family_complexity_order": (
                RENEWAL_FAMILY_COMPLEXITY_ORDER[
                    model.family
                ]
            ),
            "development_complete_duration_count": (
                development_sample.complete_duration_count
            ),
            "development_right_censored_observation_count": 1,
            "development_selection_observation_count": (
                selection_observation_count
            ),
            "development_log_score": (
                model.development_log_score
            ),
            "development_aic": (
                development_aic
            ),
            "development_bic": (
                development_bic
            ),
            "selection_partition": (
                "DEVELOPMENT"
            ),
            "calibration_used_for_selection": False,
            "score_space": (
                RENEWAL_MODEL_SCORE_SPACE
            ),
            "count_grid_log_score_comparable": False,
        }
    )


RENEWAL_DEVELOPMENT_SELECTION_TABLE = (
    pd.DataFrame(
        renewal_selection_rows
    )
)

selected_renewal_model_by_process: dict[
    str,
    str,
] = {}

for process_name in (
    DURATION_PROCESS_NAMES
):
    process_candidates = (
        RENEWAL_DEVELOPMENT_SELECTION_TABLE.loc[
            RENEWAL_DEVELOPMENT_SELECTION_TABLE[
                "process_name"
            ].eq(process_name)
        ]
        .copy()
    )

    require(
        len(process_candidates)
        == len(RENEWAL_FAMILIES),
        (
            f"{process_name} renewal selection does not contain "
            "all candidate families."
        ),
    )

    best_bic = float(
        process_candidates[
            "development_bic"
        ].min()
    )

    process_candidates[
        "delta_bic_from_process_best"
    ] = (
        process_candidates[
            "development_bic"
        ]
        - best_bic
    )

    tied_candidates = (
        process_candidates.loc[
            process_candidates[
                "delta_bic_from_process_best"
            ]
            <= RENEWAL_BIC_TIE_THRESHOLD
        ]
        .sort_values(
            [
                "parameter_count",
                "family_complexity_order",
                "development_bic",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

    require(
        not tied_candidates.empty,
        (
            f"{process_name} renewal selection produced no "
            "eligible candidate."
        ),
    )

    selected_renewal_model_by_process[
        process_name
    ] = str(
        tied_candidates.iloc[0][
            "model_id"
        ]
    )


SELECTED_RENEWAL_MODEL_BY_PROCESS: Final[
    Mapping[str, str]
] = dict(
    selected_renewal_model_by_process
)

RENEWAL_DEVELOPMENT_SELECTION_TABLE[
    "selected_model_flag"
] = (
    RENEWAL_DEVELOPMENT_SELECTION_TABLE.apply(
        lambda row: (
            row["model_id"]
            == SELECTED_RENEWAL_MODEL_BY_PROCESS[
                row["process_name"]
            ]
        ),
        axis=1,
    )
)

RENEWAL_DEVELOPMENT_SELECTION_TABLE[
    "selection_rule"
] = RENEWAL_SELECTION_RULE

RENEWAL_DEVELOPMENT_SELECTION_TABLE[
    "delta_bic_from_process_best"
] = (
    RENEWAL_DEVELOPMENT_SELECTION_TABLE.groupby(
        "process_name",
        observed=True,
    )[
        "development_bic"
    ]
    .transform(
        lambda values: (
            values - values.min()
        )
    )
)

RENEWAL_DEVELOPMENT_SELECTION_TABLE = (
    RENEWAL_DEVELOPMENT_SELECTION_TABLE.sort_values(
        [
            "process_name",
            "development_bic",
            "parameter_count",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Locked DEVELOPMENT and CALIBRATION scoring
# ------------------------------------------------------------

renewal_score_rows: list[
    dict[str, Any]
] = []

for model in (
    RENEWAL_MODEL_FITS.values()
):
    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        sample = (
            RENEWAL_SAMPLES[
                (
                    partition_name,
                    model.process_name,
                )
            ]
        )

        log_score = (
            renewal_log_likelihood(
                family=model.family,
                parameter_names=(
                    model.parameter_names
                ),
                parameter_values=(
                    model.parameter_values
                ),
                complete_durations_seconds=(
                    sample.complete_durations_seconds
                ),
                right_censored_duration_seconds=(
                    sample.right_censored_duration_seconds
                ),
            )
        )

        renewal_score_rows.append(
            {
                "model_id": model.model_id,
                "process_name": (
                    model.process_name
                ),
                "family": model.family,
                "fitted_partition": (
                    model.fitted_partition
                ),
                "scored_partition": (
                    partition_name
                ),
                "score_space": (
                    RENEWAL_MODEL_SCORE_SPACE
                ),
                "parameter_count": (
                    model.parameter_count
                ),
                "complete_duration_count": (
                    sample.complete_duration_count
                ),
                "right_censored_observation_count": 1,
                "left_censored_observation_count": 1,
                "left_censored_observation_used_in_likelihood": False,
                "right_censored_observation_used_in_likelihood": True,
                "right_censored_duration_seconds": (
                    sample.right_censored_duration_seconds
                ),
                "log_score_total": (
                    log_score
                ),
                "log_score_per_complete_duration": (
                    log_score
                    / sample.complete_duration_count
                ),
                "selected_model_flag": (
                    model.model_id
                    == SELECTED_RENEWAL_MODEL_BY_PROCESS[
                        model.process_name
                    ]
                ),
                "locked_before_calibration": True,
                "count_grid_log_score_comparable": False,
                "status": "PASS",
            }
        )


RENEWAL_SCORE_TABLE = pd.DataFrame(
    renewal_score_rows
).sort_values(
    [
        "scored_partition",
        "process_name",
        "family",
    ],
    kind="stable",
).reset_index(drop=True)


# ------------------------------------------------------------
# Family-relative score comparison
# ------------------------------------------------------------

exponential_score_lookup = (
    RENEWAL_SCORE_TABLE.loc[
        RENEWAL_SCORE_TABLE[
            "family"
        ].eq("EXPONENTIAL"),
        [
            "scored_partition",
            "process_name",
            "log_score_total",
        ],
    ]
    .rename(
        columns={
            "log_score_total": (
                "exponential_log_score_total"
            )
        }
    )
)

RENEWAL_LOCKED_SCORE_COMPARISON = (
    RENEWAL_SCORE_TABLE.merge(
        exponential_score_lookup,
        on=[
            "scored_partition",
            "process_name",
        ],
        how="left",
        validate="many_to_one",
    )
)

RENEWAL_LOCKED_SCORE_COMPARISON[
    "log_score_improvement_over_exponential"
] = (
    RENEWAL_LOCKED_SCORE_COMPARISON[
        "log_score_total"
    ]
    - RENEWAL_LOCKED_SCORE_COMPARISON[
        "exponential_log_score_total"
    ]
)

RENEWAL_LOCKED_SCORE_COMPARISON = (
    RENEWAL_LOCKED_SCORE_COMPARISON.sort_values(
        [
            "scored_partition",
            "process_name",
            "log_score_total",
        ],
        ascending=[
            True,
            True,
            False,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Selected renewal time-rescaling residuals
# ------------------------------------------------------------

selected_residual_frames: list[
    pd.DataFrame
] = []

renewal_residual_summary_rows: list[
    dict[str, Any]
] = []


def fixed_unit_exponential_ks_distance(
    transformed_durations: np.ndarray,
) -> float:
    """Return KS distance from a fixed unit-rate exponential law."""
    transformed = np.sort(
        np.asarray(
            transformed_durations,
            dtype="float64",
        )
    )

    require(
        transformed.size >= 2,
        "At least two transformed durations are required.",
    )
    require(
        np.isfinite(
            transformed
        ).all()
        and np.all(
            transformed > 0.0
        ),
        "Transformed durations must be finite and positive.",
    )

    fitted_cdf = -np.expm1(
        -transformed
    )

    sample_size = (
        transformed.size
    )

    empirical_upper = (
        np.arange(
            1,
            sample_size + 1,
            dtype="float64",
        )
        / sample_size
    )

    empirical_lower = (
        np.arange(
            0,
            sample_size,
            dtype="float64",
        )
        / sample_size
    )

    return float(
        max(
            np.max(
                empirical_upper
                - fitted_cdf
            ),
            np.max(
                fitted_cdf
                - empirical_lower
            ),
        )
    )


def lag_one_correlation(
    values: np.ndarray,
) -> float:
    """Return lag-one correlation or NaN when undefined."""
    array = np.asarray(
        values,
        dtype="float64",
    )

    if array.size < 3:
        return float("nan")

    left = array[:-1]
    right = array[1:]

    if (
        np.std(left) == 0.0
        or np.std(right) == 0.0
    ):
        return float("nan")

    return float(
        np.corrcoef(
            left,
            right,
        )[0, 1]
    )


for process_name in (
    DURATION_PROCESS_NAMES
):
    selected_model_id = (
        SELECTED_RENEWAL_MODEL_BY_PROCESS[
            process_name
        ]
    )

    selected_model = (
        RENEWAL_MODEL_FITS[
            selected_model_id
        ]
    )

    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        sample = (
            RENEWAL_SAMPLES[
                (
                    partition_name,
                    process_name,
                )
            ]
        )

        transformed_durations = (
            renewal_cumulative_hazard(
                family=(
                    selected_model.family
                ),
                parameter_names=(
                    selected_model.parameter_names
                ),
                parameter_values=(
                    selected_model.parameter_values
                ),
                durations_seconds=(
                    sample.complete_durations_seconds
                ),
            )
        )

        transformed_duration_indices = (
            np.arange(
                1,
                transformed_durations.size + 1,
                dtype="int64",
            )
        )

        selected_residual_frames.append(
            pd.DataFrame(
                {
                    "event_partition": (
                        partition_name
                    ),
                    "process_name": (
                        process_name
                    ),
                    "selected_model_id": (
                        selected_model_id
                    ),
                    "selected_family": (
                        selected_model.family
                    ),
                    "duration_index": (
                        transformed_duration_indices
                    ),
                    "duration_seconds": (
                        sample.complete_durations_seconds
                    ),
                    "transformed_duration": (
                        transformed_durations
                    ),
                    "expected_distribution": (
                        "UNIT_RATE_EXPONENTIAL"
                    ),
                    "cross_partition_flag": False,
                }
            )
        )

        renewal_residual_summary_rows.append(
            {
                "event_partition": (
                    partition_name
                ),
                "process_name": (
                    process_name
                ),
                "selected_model_id": (
                    selected_model_id
                ),
                "selected_family": (
                    selected_model.family
                ),
                "transformed_duration_count": int(
                    transformed_durations.size
                ),
                "transformed_mean": float(
                    np.mean(
                        transformed_durations
                    )
                ),
                "transformed_sample_variance": float(
                    np.var(
                        transformed_durations,
                        ddof=1,
                    )
                ),
                "transformed_median": float(
                    np.median(
                        transformed_durations
                    )
                ),
                "unit_exponential_ks_distance": (
                    fixed_unit_exponential_ks_distance(
                        transformed_durations
                    )
                ),
                "lag_one_transformed_correlation": (
                    lag_one_correlation(
                        transformed_durations
                    )
                ),
                "parameter_estimation_partition": (
                    "DEVELOPMENT"
                ),
                "formal_p_value_reported": False,
                "status": "PASS",
            }
        )


SELECTED_RENEWAL_TIME_RESCALED_RESIDUALS = (
    pd.concat(
        selected_residual_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "event_partition",
            "process_name",
            "duration_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

SELECTED_RENEWAL_RESIDUAL_SUMMARY = (
    pd.DataFrame(
        renewal_residual_summary_rows
    )
    .sort_values(
        [
            "event_partition",
            "process_name",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Selected pooled and side-specific renewal score summaries
# ------------------------------------------------------------

selected_score_rows = (
    RENEWAL_SCORE_TABLE.loc[
        RENEWAL_SCORE_TABLE[
            "selected_model_flag"
        ]
    ]
    .copy()
)

selected_pooled_scores = (
    selected_score_rows.loc[
        selected_score_rows[
            "process_name"
        ].eq("POOLED_BATCH")
    ]
    .copy()
)

selected_side_scores = (
    selected_score_rows.loc[
        selected_score_rows[
            "process_name"
        ].isin(
            {
                "BUY_BATCH",
                "SELL_BATCH",
            }
        )
    ]
    .groupby(
        "scored_partition",
        observed=True,
        sort=False,
    )
    .agg(
        log_score_total=(
            "log_score_total",
            "sum",
        ),
        complete_duration_count=(
            "complete_duration_count",
            "sum",
        ),
        parameter_count=(
            "parameter_count",
            "sum",
        ),
        selected_component_count=(
            "process_name",
            "nunique",
        ),
    )
    .reset_index()
)

require(
    selected_side_scores[
        "selected_component_count"
    ].eq(2).all(),
    (
        "Independent side-renewal summary does not contain "
        "both BUY and SELL processes."
    ),
)

selected_renewal_summary_rows: list[
    dict[str, Any]
] = []

for row in (
    selected_pooled_scores.itertuples(
        index=False
    )
):
    selected_renewal_summary_rows.append(
        {
            "model_id": (
                "N_SELECTED_POOLED_RENEWAL"
            ),
            "scored_partition": (
                row.scored_partition
            ),
            "score_space": (
                "POOLED_RENEWAL_DURATION"
            ),
            "component_count": 1,
            "parameter_count": (
                row.parameter_count
            ),
            "complete_duration_count": (
                row.complete_duration_count
            ),
            "log_score_total": (
                row.log_score_total
            ),
            "log_score_per_complete_duration": (
                row.log_score_total
                / row.complete_duration_count
            ),
            "selected_component_models": (
                SELECTED_RENEWAL_MODEL_BY_PROCESS[
                    "POOLED_BATCH"
                ]
            ),
            "count_grid_log_score_comparable": False,
            "status": "PASS",
        }
    )

for row in (
    selected_side_scores.itertuples(
        index=False
    )
):
    selected_renewal_summary_rows.append(
        {
            "model_id": (
                "N_SELECTED_INDEPENDENT_SIDE_RENEWAL"
            ),
            "scored_partition": (
                row.scored_partition
            ),
            "score_space": (
                "INDEPENDENT_SIDE_RENEWAL_DURATION"
            ),
            "component_count": 2,
            "parameter_count": (
                row.parameter_count
            ),
            "complete_duration_count": (
                row.complete_duration_count
            ),
            "log_score_total": (
                row.log_score_total
            ),
            "log_score_per_complete_duration": (
                row.log_score_total
                / row.complete_duration_count
            ),
            "selected_component_models": (
                SELECTED_RENEWAL_MODEL_BY_PROCESS[
                    "BUY_BATCH"
                ]
                + ";"
                + SELECTED_RENEWAL_MODEL_BY_PROCESS[
                    "SELL_BATCH"
                ]
            ),
            "count_grid_log_score_comparable": False,
            "status": "PASS",
        }
    )


SELECTED_RENEWAL_SCORE_SUMMARY = (
    pd.DataFrame(
        selected_renewal_summary_rows
    )
    .sort_values(
        [
            "scored_partition",
            "model_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Renewal gates
# ------------------------------------------------------------

expected_renewal_model_count = (
    len(DURATION_PROCESS_NAMES)
    * len(RENEWAL_FAMILIES)
)

expected_renewal_score_rows = (
    expected_renewal_model_count
    * len(ANALYTICAL_PARTITIONS)
)

selected_model_count = int(
    RENEWAL_DEVELOPMENT_SELECTION_TABLE[
        "selected_model_flag"
    ].sum()
)

RENEWAL_BASELINE_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "renewal_candidate_grid_complete",
            "severity": "BLOCKING",
            "passed": (
                len(RENEWAL_MODEL_FITS)
                == expected_renewal_model_count
            ),
            "evidence": (
                f"candidate_count="
                f"{len(RENEWAL_MODEL_FITS)}"
            ),
        },
        {
            "gate": "renewal_samples_meet_minimum_size",
            "severity": "BLOCKING",
            "passed": bool(
                all(
                    sample.complete_duration_count
                    >= MINIMUM_RENEWAL_DURATIONS
                    for sample
                    in RENEWAL_SAMPLES.values()
                )
            ),
            "evidence": (
                f"minimum_required="
                f"{MINIMUM_RENEWAL_DURATIONS}"
            ),
        },
        {
            "gate": "renewal_durations_do_not_cross_partitions",
            "severity": "BLOCKING",
            "passed": bool(
                not INTERARRIVAL_DURATION_TABLE[
                    "cross_partition_flag"
                ].any()
            ),
            "evidence": (
                "complete renewal durations are constructed "
                "within partition only"
            ),
        },
        {
            "gate": "left_censoring_excluded_and_recorded",
            "severity": "BLOCKING",
            "passed": bool(
                not RENEWAL_SCORE_TABLE[
                    "left_censored_observation_used_in_likelihood"
                ].any()
                and RENEWAL_SCORE_TABLE[
                    "left_censored_observation_count"
                ].eq(1).all()
            ),
            "evidence": (
                "first within-partition wait is recorded as "
                "left-censored and excluded from likelihood"
            ),
        },
        {
            "gate": "right_censoring_included",
            "severity": "BLOCKING",
            "passed": bool(
                RENEWAL_SCORE_TABLE[
                    "right_censored_observation_used_in_likelihood"
                ].all()
                and RENEWAL_SCORE_TABLE[
                    "right_censored_observation_count"
                ].eq(1).all()
            ),
            "evidence": (
                "terminal waiting time contributes through "
                "log survival"
            ),
        },
        {
            "gate": "all_renewal_optimizers_succeeded",
            "severity": "BLOCKING",
            "passed": bool(
                RENEWAL_PARAMETER_TABLE[
                    "optimizer_success"
                ].all()
            ),
            "evidence": (
                f"parameter_rows="
                f"{len(RENEWAL_PARAMETER_TABLE)}"
            ),
        },
        {
            "gate": "all_renewal_parameters_finite_positive",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    RENEWAL_PARAMETER_TABLE[
                        "parameter_value"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
                and RENEWAL_PARAMETER_TABLE[
                    "parameter_value"
                ].gt(0.0).all()
            ),
            "evidence": (
                "all fitted renewal parameters are finite and "
                "strictly positive"
            ),
        },
        {
            "gate": "development_selection_excludes_calibration",
            "severity": "BLOCKING",
            "passed": bool(
                not RENEWAL_DEVELOPMENT_SELECTION_TABLE[
                    "calibration_used_for_selection"
                ].any()
                and RENEWAL_DEVELOPMENT_SELECTION_TABLE[
                    "selection_partition"
                ].eq(
                    "DEVELOPMENT"
                ).all()
            ),
            "evidence": (
                f"selection_rule="
                f"{RENEWAL_SELECTION_RULE}"
            ),
        },
        {
            "gate": "one_renewal_model_selected_per_process",
            "severity": "BLOCKING",
            "passed": (
                selected_model_count
                == len(DURATION_PROCESS_NAMES)
            ),
            "evidence": (
                f"selected_models="
                f"{selected_model_count}"
            ),
        },
        {
            "gate": "locked_renewal_scores_complete",
            "severity": "BLOCKING",
            "passed": (
                len(RENEWAL_SCORE_TABLE)
                == expected_renewal_score_rows
            ),
            "evidence": (
                f"score_rows="
                f"{len(RENEWAL_SCORE_TABLE)}"
            ),
        },
        {
            "gate": "all_renewal_scores_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    RENEWAL_SCORE_TABLE[
                        "log_score_total"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "all DEVELOPMENT and locked CALIBRATION "
                "renewal scores are finite"
            ),
        },
        {
            "gate": "selected_time_rescaling_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    SELECTED_RENEWAL_RESIDUAL_SUMMARY
                )
                == (
                    len(ANALYTICAL_PARTITIONS)
                    * len(DURATION_PROCESS_NAMES)
                )
            ),
            "evidence": (
                f"residual_summary_rows="
                f"{len(SELECTED_RENEWAL_RESIDUAL_SUMMARY)}"
            ),
        },
        {
            "gate": "renewal_and_count_scores_not_mixed",
            "severity": "BLOCKING",
            "passed": bool(
                not RENEWAL_SCORE_TABLE[
                    "count_grid_log_score_comparable"
                ].any()
                and not SELECTED_RENEWAL_SCORE_SUMMARY[
                    "count_grid_log_score_comparable"
                ].any()
            ),
            "evidence": (
                "renewal duration likelihoods remain in a "
                "separate score space from native-grid counts"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    RENEWAL_BASELINE_GATE_FRAME.loc[
        RENEWAL_BASELINE_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    (
        "At least one independent renewal-baseline gate "
        "failed."
    ),
)


display(
    RENEWAL_DEVELOPMENT_SELECTION_TABLE[
        [
            "model_id",
            "process_name",
            "family",
            "parameter_count",
            "development_log_score",
            "development_aic",
            "development_bic",
            "delta_bic_from_process_best",
            "selected_model_flag",
            "selection_rule",
        ]
    ]
)

display(
    RENEWAL_PARAMETER_TABLE[
        [
            "model_id",
            "process_name",
            "family",
            "parameter_name",
            "parameter_value",
            "optimizer_method",
            "optimizer_success",
            "optimizer_iterations",
            "selected_model_flag",
            "status",
        ]
        if "selected_model_flag"
        in RENEWAL_PARAMETER_TABLE.columns
        else [
            "model_id",
            "process_name",
            "family",
            "parameter_name",
            "parameter_value",
            "optimizer_method",
            "optimizer_success",
            "optimizer_iterations",
            "status",
        ]
    ]
)

display(
    RENEWAL_LOCKED_SCORE_COMPARISON[
        [
            "model_id",
            "process_name",
            "family",
            "scored_partition",
            "complete_duration_count",
            "right_censored_duration_seconds",
            "log_score_total",
            "log_score_per_complete_duration",
            "log_score_improvement_over_exponential",
            "selected_model_flag",
            "status",
        ]
    ]
)

display(SELECTED_RENEWAL_SCORE_SUMMARY)

display(
    SELECTED_RENEWAL_RESIDUAL_SUMMARY[
        [
            "event_partition",
            "process_name",
            "selected_model_id",
            "selected_family",
            "transformed_duration_count",
            "transformed_mean",
            "transformed_sample_variance",
            "transformed_median",
            "unit_exponential_ks_distance",
            "lag_one_transformed_correlation",
            "formal_p_value_reported",
            "status",
        ]
    ]
)

display(RENEWAL_BASELINE_GATE_FRAME)

print(
    "Independent exponential, gamma, and Weibull renewal "
    "baselines were fitted on DEVELOPMENT only."
)
print(
    "The first within-partition waiting time remains explicitly "
    "left-censored and excluded from likelihood construction."
)
print(
    "The terminal waiting time is included through the fitted "
    "survival function."
)
print(
    "One renewal family was selected independently for pooled, "
    "BUY, and SELL arrivals using DEVELOPMENT BIC with a "
    "delta-two simplicity tie-break."
)
print(
    "Selected DEVELOPMENT parameters were scored unchanged on "
    "CALIBRATION."
)
print(
    "Renewal duration likelihoods remain separate from native "
    "count-grid log scores and are not compared numerically "
    "across incompatible score spaces."
)
print(
    "Hawkes estimation remains unauthorized."
)

,model_id,process_name,family,parameter_count,development_log_score,development_aic,development_bic,delta_bic_from_process_best,selected_model_flag,selection_rule
0,N_BUY_BATCH_GAMMA_RENEWAL,BUY_BATCH,GAMMA,2,-819.88181,"1,643.7636","1,655.9857",0,True,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
1,N_BUY_BATCH_WEIBULL_RENEWAL,BUY_BATCH,WEIBULL,2,-878.07673,"1,760.1535","1,772.3755",116.38985,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
2,N_BUY_BATCH_EXPONENTIAL_RENEWAL,BUY_BATCH,EXPONENTIAL,1,"-1,282.5872","2,567.1743","2,573.2854",917.2997,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
3,N_POOLED_BATCH_GAMMA_RENEWAL,POOLED_BATCH,GAMMA,2,"2,692.642","-5,381.2839","-5,367.6173",0,True,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
4,N_POOLED_BATCH_WEIBULL_RENEWAL,POOLED_BATCH,WEIBULL,2,"2,642.8616","-5,281.7232","-5,268.0566",99.560717,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
5,N_POOLED_BATCH_EXPONENTIAL_RENEWAL,POOLED_BATCH,EXPONENTIAL,1,"2,313.0816","-4,624.1632","-4,617.3299",750.28742,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
6,N_SELL_BATCH_GAMMA_RENEWAL,SELL_BATCH,GAMMA,2,-998.26124,"2,000.5225","2,012.8741",0,True,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
7,N_SELL_BATCH_WEIBULL_RENEWAL,SELL_BATCH,WEIBULL,2,"-1,028.5799","2,061.1597","2,073.5114",60.637259,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...
8,N_SELL_BATCH_EXPONENTIAL_RENEWAL,SELL_BATCH,EXPONENTIAL,1,"-1,135.2603","2,272.5205","2,278.6963",265.82219,False,MINIMUM_DEVELOPMENT_BIC_WITH_DELTA_2_SIMPLICIT...


,model_id,process_name,family,parameter_name,parameter_value,optimizer_method,optimizer_success,optimizer_iterations,status
0,N_BUY_BATCH_EXPONENTIAL_RENEWAL,BUY_BATCH,EXPONENTIAL,rate_per_second,1.8493583,CLOSED_FORM,True,0,PASS
1,N_BUY_BATCH_GAMMA_RENEWAL,BUY_BATCH,GAMMA,scale_seconds,0.95884899,L-BFGS-B,True,7,PASS
2,N_BUY_BATCH_GAMMA_RENEWAL,BUY_BATCH,GAMMA,shape,0.56400512,L-BFGS-B,True,7,PASS
3,N_BUY_BATCH_WEIBULL_RENEWAL,BUY_BATCH,WEIBULL,scale_seconds,0.43590269,L-BFGS-B,True,9,PASS
4,N_BUY_BATCH_WEIBULL_RENEWAL,BUY_BATCH,WEIBULL,shape,0.69172002,L-BFGS-B,True,9,PASS
5,N_POOLED_BATCH_EXPONENTIAL_RENEWAL,POOLED_BATCH,EXPONENTIAL,rate_per_second,3.8086783,CLOSED_FORM,True,0,PASS
6,N_POOLED_BATCH_GAMMA_RENEWAL,POOLED_BATCH,GAMMA,scale_seconds,0.38359356,L-BFGS-B,True,10,PASS
7,N_POOLED_BATCH_GAMMA_RENEWAL,POOLED_BATCH,GAMMA,shape,0.68449158,L-BFGS-B,True,10,PASS
8,N_POOLED_BATCH_WEIBULL_RENEWAL,POOLED_BATCH,WEIBULL,scale_seconds,0.23194463,L-BFGS-B,True,8,PASS
9,N_POOLED_BATCH_WEIBULL_RENEWAL,POOLED_BATCH,WEIBULL,shape,0.78968713,L-BFGS-B,True,8,PASS


,model_id,process_name,family,scored_partition,complete_duration_count,right_censored_duration_seconds,log_score_total,log_score_per_complete_duration,log_score_improvement_over_exponential,selected_model_flag,status
0,N_BUY_BATCH_GAMMA_RENEWAL,BUY_BATCH,GAMMA,CALIBRATION,1154,1.1048569,-331.52485,-0.28728323,288.90577,True,PASS
1,N_BUY_BATCH_WEIBULL_RENEWAL,BUY_BATCH,WEIBULL,CALIBRATION,1154,1.1048569,-362.07343,-0.31375514,258.35718,False,PASS
2,N_BUY_BATCH_EXPONENTIAL_RENEWAL,BUY_BATCH,EXPONENTIAL,CALIBRATION,1154,1.1048569,-620.43062,-0.53763485,0,False,PASS
3,N_POOLED_BATCH_GAMMA_RENEWAL,POOLED_BATCH,GAMMA,CALIBRATION,2399,0.3410659,687.31971,0.28650259,218.16692,True,PASS
4,N_POOLED_BATCH_WEIBULL_RENEWAL,POOLED_BATCH,WEIBULL,CALIBRATION,2399,0.3410659,662.25606,0.27605505,193.10328,False,PASS
5,N_POOLED_BATCH_EXPONENTIAL_RENEWAL,POOLED_BATCH,EXPONENTIAL,CALIBRATION,2399,0.3410659,469.15279,0.19556181,0,False,PASS
6,N_SELL_BATCH_GAMMA_RENEWAL,SELL_BATCH,GAMMA,CALIBRATION,1249,0.3410659,-529.72363,-0.4241182,39.657163,True,PASS
7,N_SELL_BATCH_WEIBULL_RENEWAL,SELL_BATCH,WEIBULL,CALIBRATION,1249,0.3410659,-541.56376,-0.43359788,27.817036,False,PASS
8,N_SELL_BATCH_EXPONENTIAL_RENEWAL,SELL_BATCH,EXPONENTIAL,CALIBRATION,1249,0.3410659,-569.38079,-0.45586933,0,False,PASS
9,N_BUY_BATCH_GAMMA_RENEWAL,BUY_BATCH,GAMMA,DEVELOPMENT,3330,0.544745,-819.88181,-0.24621075,462.70536,True,PASS


,model_id,scored_partition,score_space,component_count,parameter_count,complete_duration_count,log_score_total,log_score_per_complete_duration,selected_component_models,count_grid_log_score_comparable,status
0,N_SELECTED_INDEPENDENT_SIDE_RENEWAL,CALIBRATION,INDEPENDENT_SIDE_RENEWAL_DURATION,2,4,2403,-861.24848,-0.35840553,N_BUY_BATCH_GAMMA_RENEWAL;N_SELL_BATCH_GAMMA_R...,False,PASS
1,N_SELECTED_POOLED_RENEWAL,CALIBRATION,POOLED_RENEWAL_DURATION,1,2,2399,687.31971,0.28650259,N_POOLED_BATCH_GAMMA_RENEWAL,False,PASS
2,N_SELECTED_INDEPENDENT_SIDE_RENEWAL,DEVELOPMENT,INDEPENDENT_SIDE_RENEWAL_DURATION,2,4,6883,"-1,818.1431",-0.2641498,N_BUY_BATCH_GAMMA_RENEWAL;N_SELL_BATCH_GAMMA_R...,False,PASS
3,N_SELECTED_POOLED_RENEWAL,DEVELOPMENT,POOLED_RENEWAL_DURATION,1,2,6858,"2,692.642",0.39262787,N_POOLED_BATCH_GAMMA_RENEWAL,False,PASS


,event_partition,process_name,selected_model_id,selected_family,transformed_duration_count,transformed_mean,transformed_sample_variance,transformed_median,unit_exponential_ks_distance,lag_one_transformed_correlation,formal_p_value_reported,status
0,CALIBRATION,BUY_BATCH,N_BUY_BATCH_GAMMA_RENEWAL,GAMMA,1154,1.0996506,1.2564039,0.81880721,0.085604626,0.24917472,False,PASS
1,CALIBRATION,POOLED_BATCH,N_POOLED_BATCH_GAMMA_RENEWAL,GAMMA,2399,1.1116391,1.2019276,0.78146476,0.056083521,0.11667366,False,PASS
2,CALIBRATION,SELL_BATCH,N_SELL_BATCH_GAMMA_RENEWAL,GAMMA,1249,1.120688,1.0046298,0.86565655,0.083129562,0.078268817,False,PASS
3,DEVELOPMENT,BUY_BATCH,N_BUY_BATCH_GAMMA_RENEWAL,GAMMA,3330,1.0047051,0.90077963,0.77255359,0.046623679,0.2055734,False,PASS
4,DEVELOPMENT,POOLED_BATCH,N_POOLED_BATCH_GAMMA_RENEWAL,GAMMA,6858,1.0014469,0.90697977,0.73661311,0.027624174,0.12902709,False,PASS
5,DEVELOPMENT,SELL_BATCH,N_SELL_BATCH_GAMMA_RENEWAL,GAMMA,3553,1.0028261,0.8569465,0.74515984,0.034454937,0.083755003,False,PASS


,gate,severity,passed,evidence
0,renewal_candidate_grid_complete,BLOCKING,True,candidate_count=9
1,renewal_samples_meet_minimum_size,BLOCKING,True,minimum_required=100
2,renewal_durations_do_not_cross_partitions,BLOCKING,True,complete renewal durations are constructed wit...
3,left_censoring_excluded_and_recorded,BLOCKING,True,first within-partition wait is recorded as lef...
4,right_censoring_included,BLOCKING,True,terminal waiting time contributes through log ...
5,all_renewal_optimizers_succeeded,BLOCKING,True,parameter_rows=15
6,all_renewal_parameters_finite_positive,BLOCKING,True,all fitted renewal parameters are finite and s...
7,development_selection_excludes_calibration,BLOCKING,True,selection_rule=MINIMUM_DEVELOPMENT_BIC_WITH_DE...
8,one_renewal_model_selected_per_process,BLOCKING,True,selected_models=3
9,locked_renewal_scores_complete,BLOCKING,True,score_rows=18


Independent exponential, gamma, and Weibull renewal baselines were fitted on DEVELOPMENT only.
The first within-partition waiting time remains explicitly left-censored and excluded from likelihood construction.
The terminal waiting time is included through the fitted survival function.
One renewal family was selected independently for pooled, BUY, and SELL arrivals using DEVELOPMENT BIC with a delta-two simplicity tie-break.
Selected DEVELOPMENT parameters were scored unchanged on CALIBRATION.
Renewal duration likelihoods remain separate from native count-grid log scores and are not compared numerically across incompatible score spaces.
Hawkes estimation remains unauthorized.


In [14]:
# ============================================================
# Optional exact-time batch-mark baselines
# ============================================================

BATCH_MARK_STATES: Final[
    tuple[str, ...]
] = (
    "BUY_ONLY",
    "SELL_ONLY",
    "MIXED",
)

BATCH_MARK_STATE_TO_INDEX: Final[
    Mapping[str, int]
] = {
    state: index
    for index, state
    in enumerate(
        BATCH_MARK_STATES
    )
}

BATCH_MARK_DIRICHLET_ALPHA: Final[float] = 0.5

BATCH_MARK_SCORE_SPACE: Final[str] = (
    "EXACT_TIME_BATCH_MARK_SEQUENCE"
)

IID_BATCH_MARK_MODEL_ID: Final[str] = (
    "M0_IID_BATCH_MARK"
)

MARKOV_BATCH_MARK_MODEL_ID: Final[str] = (
    "M1_FIRST_ORDER_BATCH_MARKOV"
)

BATCH_MARK_MODEL_IDS: Final[
    tuple[str, ...]
] = (
    IID_BATCH_MARK_MODEL_ID,
    MARKOV_BATCH_MARK_MODEL_ID,
)

BATCH_MARK_MODEL_STATUS: Final[str] = (
    "DIAGNOSTIC_ONLY_NO_HYPERPARAMETER_SELECTION"
)


# ------------------------------------------------------------
# Canonical batch marks
# ------------------------------------------------------------

def derive_batch_marks(
    batches: pd.DataFrame,
) -> pd.Series:
    """Classify every exact-time batch without ordering tied events."""
    buy_count = (
        batches[
            "buy_event_count"
        ]
        .astype("int64")
    )

    sell_count = (
        batches[
            "sell_event_count"
        ]
        .astype("int64")
    )

    require(
        buy_count.ge(0).all(),
        "Batch BUY event counts contain negative values.",
    )
    require(
        sell_count.ge(0).all(),
        "Batch SELL event counts contain negative values.",
    )
    require(
        (
            buy_count
            + sell_count
        )
        .eq(
            batches[
                "batch_event_count"
            ].astype("int64")
        )
        .all(),
        (
            "Batch BUY and SELL counts do not conserve total "
            "event multiplicity."
        ),
    )
    require(
        (
            buy_count
            + sell_count
        )
        .gt(0)
        .all(),
        "At least one exact-time batch contains no events.",
    )

    marks = np.select(
        [
            buy_count.gt(0)
            & sell_count.eq(0),
            buy_count.eq(0)
            & sell_count.gt(0),
            buy_count.gt(0)
            & sell_count.gt(0),
        ],
        [
            "BUY_ONLY",
            "SELL_ONLY",
            "MIXED",
        ],
        default="INVALID",
    )

    mark_series = pd.Series(
        marks,
        index=batches.index,
        dtype="string",
        name="batch_mark",
    )

    require(
        mark_series.isin(
            BATCH_MARK_STATES
        ).all(),
        "At least one exact-time batch has an invalid mark.",
    )

    return mark_series


PRIMARY_BATCH_MARK_TABLE = (
    PRIMARY_SCORING_BATCHES_ANALYTICAL[
        [
            "primary_event_batch_id",
            "primary_event_batch_number",
            "partition_batch_index",
            "event_partition_order",
            "event_partition",
            "event_time_ns",
            "batch_event_count",
            "buy_event_count",
            "sell_event_count",
            "unique_side_count",
            "mixed_side_batch_flag",
            "simultaneous_batch_required_flag",
        ]
    ]
    .copy()
)

PRIMARY_BATCH_MARK_TABLE[
    "event_partition"
] = (
    PRIMARY_BATCH_MARK_TABLE[
        "event_partition"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
)

PRIMARY_BATCH_MARK_TABLE[
    "event_time_ns"
] = parse_exact_int64(
    PRIMARY_BATCH_MARK_TABLE[
        "event_time_ns"
    ],
    label="primary_batch_mark_table.event_time_ns",
)

PRIMARY_BATCH_MARK_TABLE[
    "batch_mark"
] = derive_batch_marks(
    PRIMARY_BATCH_MARK_TABLE
)

PRIMARY_BATCH_MARK_TABLE = (
    PRIMARY_BATCH_MARK_TABLE.sort_values(
        [
            "event_partition_order",
            "partition_batch_index",
            "primary_event_batch_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    PRIMARY_BATCH_MARK_TABLE[
        "primary_event_batch_id"
    ].is_unique,
    "Batch-mark table contains duplicate batch IDs.",
)

require(
    len(
        PRIMARY_BATCH_MARK_TABLE
    )
    == expected_analytical_batch_rows,
    (
        "Batch-mark table row count differs from the "
        "analytical batch contract."
    ),
)

require(
    PRIMARY_BATCH_MARK_TABLE[
        "batch_mark"
    ]
    .eq("MIXED")
    .eq(
        PRIMARY_BATCH_MARK_TABLE[
            "mixed_side_batch_flag"
        ].astype(bool)
    )
    .all(),
    (
        "Derived MIXED marks differ from the authoritative "
        "mixed-side flag."
    ),
)

require(
    int(
        PRIMARY_BATCH_MARK_TABLE[
            "batch_mark"
        ]
        .eq("MIXED")
        .sum()
    )
    == expected_analytical_mixed_side_batches,
    (
        "Derived MIXED batch count differs from the "
        "analytical observation-window contract."
    ),
)


# ------------------------------------------------------------
# Partition-specific mark sequences
# ------------------------------------------------------------

BATCH_MARK_TABLE_BY_PARTITION: Final[
    Mapping[str, pd.DataFrame]
] = {
    partition_name: (
        PRIMARY_BATCH_MARK_TABLE.loc[
            PRIMARY_BATCH_MARK_TABLE[
                "event_partition"
            ].eq(partition_name)
        ]
        .copy()
        .sort_values(
            "partition_batch_index",
            kind="stable",
        )
        .reset_index(drop=True)
    )
    for partition_name in ANALYTICAL_PARTITIONS
}

for partition_name, partition_marks in (
    BATCH_MARK_TABLE_BY_PARTITION.items()
):
    require(
        not partition_marks.empty,
        (
            f"{partition_name} contains no exact-time "
            "batch marks."
        ),
    )

    require(
        np.all(
            np.diff(
                partition_marks[
                    "event_time_ns"
                ].to_numpy(
                    dtype="int64"
                )
            )
            > 0
        ),
        (
            f"{partition_name} batch-mark times are not "
            "strictly increasing."
        ),
    )

    require(
        partition_marks[
            "batch_mark"
        ]
        .isin(
            BATCH_MARK_STATES
        )
        .all(),
        (
            f"{partition_name} contains an unsupported "
            "batch mark."
        ),
    )


DEVELOPMENT_BATCH_MARKS = (
    BATCH_MARK_TABLE_BY_PARTITION[
        "DEVELOPMENT"
    ]
)

CALIBRATION_BATCH_MARKS = (
    BATCH_MARK_TABLE_BY_PARTITION[
        "CALIBRATION"
    ]
)


# ------------------------------------------------------------
# DEVELOPMENT-only IID and transition estimates
# ------------------------------------------------------------

def smoothed_categorical_probabilities(
    counts: np.ndarray,
    *,
    alpha: float,
) -> np.ndarray:
    """Return symmetric-Dirichlet posterior predictive probabilities."""
    count_array = np.asarray(
        counts,
        dtype="float64",
    )

    require(
        count_array.ndim == 1,
        "Categorical counts must be one-dimensional.",
    )
    require(
        count_array.size
        == len(BATCH_MARK_STATES),
        "Categorical count vector has an invalid length.",
    )
    require(
        np.isfinite(
            count_array
        ).all()
        and np.all(
            count_array >= 0.0
        ),
        "Categorical counts must be finite and nonnegative.",
    )
    require(
        np.isfinite(alpha)
        and alpha > 0.0,
        "Dirichlet smoothing alpha must be finite and positive.",
    )

    probabilities = (
        count_array + alpha
    ) / (
        count_array.sum(
            dtype="float64"
        )
        + alpha
        * count_array.size
    )

    require(
        np.isfinite(
            probabilities
        ).all(),
        "Categorical probabilities contain nonfinite values.",
    )
    require(
        np.all(
            probabilities > 0.0
        ),
        "Categorical probabilities are not strictly positive.",
    )
    require(
        np.isclose(
            probabilities.sum(
                dtype="float64"
            ),
            1.0,
            rtol=0.0,
            atol=1e-12,
        ),
        "Categorical probabilities do not sum to one.",
    )

    return probabilities


development_mark_count_series = (
    DEVELOPMENT_BATCH_MARKS[
        "batch_mark"
    ]
    .value_counts(
        sort=False
    )
    .reindex(
        BATCH_MARK_STATES,
        fill_value=0,
    )
)

DEVELOPMENT_BATCH_MARK_COUNTS = (
    development_mark_count_series.to_numpy(
        dtype="int64"
    )
)

DEVELOPMENT_IID_BATCH_MARK_PROBABILITIES = (
    smoothed_categorical_probabilities(
        DEVELOPMENT_BATCH_MARK_COUNTS,
        alpha=BATCH_MARK_DIRICHLET_ALPHA,
    )
)

development_mark_sequence = (
    DEVELOPMENT_BATCH_MARKS[
        "batch_mark"
    ]
    .astype("string")
    .to_numpy()
)

development_previous_marks = (
    development_mark_sequence[:-1]
)

development_current_marks = (
    development_mark_sequence[1:]
)

require(
    development_current_marks.size
    == len(DEVELOPMENT_BATCH_MARKS) - 1,
    "DEVELOPMENT transition count is invalid.",
)


DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS = np.zeros(
    (
        len(BATCH_MARK_STATES),
        len(BATCH_MARK_STATES),
    ),
    dtype="int64",
)

for previous_mark, current_mark in zip(
    development_previous_marks,
    development_current_marks,
    strict=True,
):
    previous_index = (
        BATCH_MARK_STATE_TO_INDEX[
            str(previous_mark)
        ]
    )

    current_index = (
        BATCH_MARK_STATE_TO_INDEX[
            str(current_mark)
        ]
    )

    DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS[
        previous_index,
        current_index,
    ] += 1


require(
    int(
        DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS.sum(
            dtype="int64"
        )
    )
    == len(DEVELOPMENT_BATCH_MARKS) - 1,
    (
        "DEVELOPMENT transition counts do not conserve "
        "eligible mark transitions."
    ),
)


DEVELOPMENT_BATCH_MARK_TRANSITION_PROBABILITIES = np.vstack(
    [
        smoothed_categorical_probabilities(
            DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS[
                previous_index
            ],
            alpha=(
                BATCH_MARK_DIRICHLET_ALPHA
            ),
        )
        for previous_index in range(
            len(BATCH_MARK_STATES)
        )
    ]
)


require(
    np.isclose(
        DEVELOPMENT_BATCH_MARK_TRANSITION_PROBABILITIES.sum(
            axis=1
        ),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ).all(),
    "Transition-probability rows do not sum to one.",
)


# ------------------------------------------------------------
# Parameter tables
# ------------------------------------------------------------

IID_BATCH_MARK_PARAMETER_TABLE = pd.DataFrame(
    [
        {
            "model_id": (
                IID_BATCH_MARK_MODEL_ID
            ),
            "model_family": (
                "IID_EXACT_TIME_BATCH_MARK"
            ),
            "fitted_partition": (
                "DEVELOPMENT"
            ),
            "batch_mark": batch_mark,
            "development_mark_count": int(
                DEVELOPMENT_BATCH_MARK_COUNTS[
                    mark_index
                ]
            ),
            "dirichlet_alpha": (
                BATCH_MARK_DIRICHLET_ALPHA
            ),
            "predictive_probability": float(
                DEVELOPMENT_IID_BATCH_MARK_PROBABILITIES[
                    mark_index
                ]
            ),
            "calibration_used_for_fit": False,
            "status": "PASS",
        }
        for mark_index, batch_mark
        in enumerate(
            BATCH_MARK_STATES
        )
    ]
)


transition_parameter_rows: list[
    dict[str, Any]
] = []

for previous_index, previous_mark in enumerate(
    BATCH_MARK_STATES
):
    previous_state_total = int(
        DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS[
            previous_index
        ].sum(
            dtype="int64"
        )
    )

    for current_index, current_mark in enumerate(
        BATCH_MARK_STATES
    ):
        transition_count = int(
            DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS[
                previous_index,
                current_index,
            ]
        )

        transition_parameter_rows.append(
            {
                "model_id": (
                    MARKOV_BATCH_MARK_MODEL_ID
                ),
                "model_family": (
                    "FIRST_ORDER_EXACT_TIME_BATCH_MARKOV"
                ),
                "fitted_partition": (
                    "DEVELOPMENT"
                ),
                "previous_batch_mark": (
                    previous_mark
                ),
                "current_batch_mark": (
                    current_mark
                ),
                "development_transition_count": (
                    transition_count
                ),
                "development_previous_state_total": (
                    previous_state_total
                ),
                "dirichlet_alpha": (
                    BATCH_MARK_DIRICHLET_ALPHA
                ),
                "predictive_probability": float(
                    DEVELOPMENT_BATCH_MARK_TRANSITION_PROBABILITIES[
                        previous_index,
                        current_index,
                    ]
                ),
                "calibration_used_for_fit": False,
                "status": "PASS",
            }
        )


BATCH_MARK_TRANSITION_PARAMETER_TABLE = (
    pd.DataFrame(
        transition_parameter_rows
    )
    .sort_values(
        [
            "previous_batch_mark",
            "current_batch_mark",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Scoring-pair construction
# ------------------------------------------------------------

def build_mark_scoring_pairs(
    *,
    partition_name: str,
) -> pd.DataFrame:
    """
    Construct ordered mark-scoring pairs.

    DEVELOPMENT:
        score batches 2..N using previous DEVELOPMENT batch.

    CALIBRATION:
        score every CALIBRATION batch;
        the first batch uses the final DEVELOPMENT batch as history.
    """
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized batch-mark scoring partition: "
            f"{partition_name}"
        ),
    )

    if partition_name == "DEVELOPMENT":
        current_rows = (
            DEVELOPMENT_BATCH_MARKS.iloc[
                1:
            ]
            .copy()
            .reset_index(drop=True)
        )

        previous_rows = (
            DEVELOPMENT_BATCH_MARKS.iloc[
                :-1
            ]
            .copy()
            .reset_index(drop=True)
        )

    else:
        current_rows = (
            CALIBRATION_BATCH_MARKS.copy()
            .reset_index(drop=True)
        )

        previous_rows = pd.concat(
            [
                DEVELOPMENT_BATCH_MARKS.iloc[
                    [-1]
                ],
                CALIBRATION_BATCH_MARKS.iloc[
                    :-1
                ],
            ],
            ignore_index=True,
        )

    require(
        len(current_rows)
        == len(previous_rows),
        (
            f"{partition_name} mark-scoring current and "
            "previous sequences differ in length."
        ),
    )

    require(
        len(current_rows) > 0,
        (
            f"{partition_name} contains no eligible "
            "mark-scoring observations."
        ),
    )

    scoring_pairs = pd.DataFrame(
        {
            "scored_partition": (
                current_rows[
                    "event_partition"
                ]
                .astype("string")
                .to_numpy()
            ),
            "current_batch_id": (
                current_rows[
                    "primary_event_batch_id"
                ]
                .astype("string")
                .to_numpy()
            ),
            "current_batch_number": (
                current_rows[
                    "primary_event_batch_number"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "current_partition_batch_index": (
                current_rows[
                    "partition_batch_index"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "current_event_time_ns": (
                current_rows[
                    "event_time_ns"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "current_batch_mark": (
                current_rows[
                    "batch_mark"
                ]
                .astype("string")
                .to_numpy()
            ),
            "current_batch_event_count": (
                current_rows[
                    "batch_event_count"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "current_buy_event_count": (
                current_rows[
                    "buy_event_count"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "current_sell_event_count": (
                current_rows[
                    "sell_event_count"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "history_partition": (
                previous_rows[
                    "event_partition"
                ]
                .astype("string")
                .to_numpy()
            ),
            "previous_batch_id": (
                previous_rows[
                    "primary_event_batch_id"
                ]
                .astype("string")
                .to_numpy()
            ),
            "previous_batch_number": (
                previous_rows[
                    "primary_event_batch_number"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "previous_event_time_ns": (
                previous_rows[
                    "event_time_ns"
                ]
                .astype("int64")
                .to_numpy()
            ),
            "previous_batch_mark": (
                previous_rows[
                    "batch_mark"
                ]
                .astype("string")
                .to_numpy()
            ),
        }
    )

    scoring_pairs[
        "history_gap_ns"
    ] = (
        scoring_pairs[
            "current_event_time_ns"
        ]
        - scoring_pairs[
            "previous_event_time_ns"
        ]
    )

    scoring_pairs[
        "cross_partition_history_flag"
    ] = (
        scoring_pairs[
            "history_partition"
        ]
        .ne(
            scoring_pairs[
                "scored_partition"
            ]
        )
    )

    require(
        scoring_pairs[
            "history_gap_ns"
        ].gt(0).all(),
        (
            f"{partition_name} mark scoring contains a "
            "nonpositive history gap."
        ),
    )

    if partition_name == "DEVELOPMENT":
        require(
            not scoring_pairs[
                "cross_partition_history_flag"
            ].any(),
            (
                "DEVELOPMENT mark scoring unexpectedly uses "
                "cross-partition history."
            ),
        )
    else:
        require(
            int(
                scoring_pairs[
                    "cross_partition_history_flag"
                ].sum()
            )
            == 1,
            (
                "CALIBRATION mark scoring must contain exactly "
                "one DEVELOPMENT-to-CALIBRATION history link."
            ),
        )

        first_pair = (
            scoring_pairs.iloc[0]
        )

        require(
            first_pair[
                "history_partition"
            ]
            == "DEVELOPMENT",
            (
                "The first CALIBRATION mark score does not use "
                "the final DEVELOPMENT batch."
            ),
        )

        require(
            first_pair[
                "scored_partition"
            ]
            == "CALIBRATION",
            (
                "The first locked CALIBRATION score is assigned "
                "to the wrong partition."
            ),
        )

    return scoring_pairs


BATCH_MARK_SCORING_PAIRS: Final[
    Mapping[str, pd.DataFrame]
] = {
    partition_name: (
        build_mark_scoring_pairs(
            partition_name=(
                partition_name
            )
        )
    )
    for partition_name in ANALYTICAL_PARTITIONS
}


# ------------------------------------------------------------
# Locked mark-model scoring
# ------------------------------------------------------------

def categorical_brier_score(
    realized_index: np.ndarray,
    predicted_probabilities: np.ndarray,
) -> np.ndarray:
    """Return multiclass Brier scores for each observation."""
    realized = np.asarray(
        realized_index,
        dtype="int64",
    )

    predicted = np.asarray(
        predicted_probabilities,
        dtype="float64",
    )

    require(
        predicted.ndim == 2,
        "Categorical prediction matrix must be two-dimensional.",
    )
    require(
        predicted.shape[0]
        == realized.size,
        (
            "Categorical prediction and realized arrays differ "
            "in observation count."
        ),
    )
    require(
        predicted.shape[1]
        == len(BATCH_MARK_STATES),
        "Categorical prediction matrix has an invalid state count.",
    )

    one_hot = np.zeros_like(
        predicted
    )

    one_hot[
        np.arange(
            realized.size
        ),
        realized,
    ] = 1.0

    return np.square(
        predicted - one_hot
    ).sum(
        axis=1
    )


batch_mark_replay_frames: list[
    pd.DataFrame
] = []

for partition_name in (
    ANALYTICAL_PARTITIONS
):
    scoring_pairs = (
        BATCH_MARK_SCORING_PAIRS[
            partition_name
        ].copy()
    )

    previous_mark_indices = np.asarray(
        [
            BATCH_MARK_STATE_TO_INDEX[
                str(mark)
            ]
            for mark in scoring_pairs[
                "previous_batch_mark"
            ]
        ],
        dtype="int64",
    )

    current_mark_indices = np.asarray(
        [
            BATCH_MARK_STATE_TO_INDEX[
                str(mark)
            ]
            for mark in scoring_pairs[
                "current_batch_mark"
            ]
        ],
        dtype="int64",
    )

    iid_probability_matrix = np.broadcast_to(
        DEVELOPMENT_IID_BATCH_MARK_PROBABILITIES,
        (
            len(scoring_pairs),
            len(BATCH_MARK_STATES),
        ),
    ).copy()

    markov_probability_matrix = (
        DEVELOPMENT_BATCH_MARK_TRANSITION_PROBABILITIES[
            previous_mark_indices
        ]
    )

    model_probability_matrices = {
        IID_BATCH_MARK_MODEL_ID: (
            iid_probability_matrix
        ),
        MARKOV_BATCH_MARK_MODEL_ID: (
            markov_probability_matrix
        ),
    }

    for (
        model_id,
        probability_matrix,
    ) in model_probability_matrices.items():
        require(
            np.isfinite(
                probability_matrix
            ).all(),
            (
                f"{model_id} {partition_name} mark "
                "probabilities contain nonfinite values."
            ),
        )
        require(
            np.all(
                probability_matrix > 0.0
            ),
            (
                f"{model_id} {partition_name} mark "
                "probabilities are not strictly positive."
            ),
        )
        require(
            np.isclose(
                probability_matrix.sum(
                    axis=1
                ),
                1.0,
                rtol=0.0,
                atol=1e-12,
            ).all(),
            (
                f"{model_id} {partition_name} mark "
                "probability rows do not sum to one."
            ),
        )

        realized_probabilities = (
            probability_matrix[
                np.arange(
                    len(scoring_pairs)
                ),
                current_mark_indices,
            ]
        )

        log_scores = np.log(
            realized_probabilities
        )

        brier_scores = (
            categorical_brier_score(
                current_mark_indices,
                probability_matrix,
            )
        )

        predicted_indices = np.argmax(
            probability_matrix,
            axis=1,
        )

        predicted_marks = np.asarray(
            BATCH_MARK_STATES,
            dtype=object,
        )[
            predicted_indices
        ]

        prediction_entropy = -np.sum(
            probability_matrix
            * np.log(
                probability_matrix
            ),
            axis=1,
        )

        replay = scoring_pairs.copy()

        replay.insert(
            0,
            "model_id",
            model_id,
        )

        replay.insert(
            1,
            "model_family",
            (
                "IID_EXACT_TIME_BATCH_MARK"
                if model_id
                == IID_BATCH_MARK_MODEL_ID
                else (
                    "FIRST_ORDER_EXACT_TIME_BATCH_MARKOV"
                )
            ),
        )

        replay[
            "probability_buy_only"
        ] = (
            probability_matrix[
                :,
                BATCH_MARK_STATE_TO_INDEX[
                    "BUY_ONLY"
                ],
            ]
        )

        replay[
            "probability_sell_only"
        ] = (
            probability_matrix[
                :,
                BATCH_MARK_STATE_TO_INDEX[
                    "SELL_ONLY"
                ],
            ]
        )

        replay[
            "probability_mixed"
        ] = (
            probability_matrix[
                :,
                BATCH_MARK_STATE_TO_INDEX[
                    "MIXED"
                ],
            ]
        )

        replay[
            "realized_probability"
        ] = realized_probabilities

        replay[
            "log_score"
        ] = log_scores

        replay[
            "negative_log_score"
        ] = -log_scores

        replay[
            "brier_score"
        ] = brier_scores

        replay[
            "prediction_entropy"
        ] = prediction_entropy

        replay[
            "predicted_batch_mark"
        ] = pd.Series(
            predicted_marks,
            dtype="string",
        )

        replay[
            "top_one_correct_flag"
        ] = (
            predicted_indices
            == current_mark_indices
        )

        replay[
            "fitted_partition"
        ] = "DEVELOPMENT"

        replay[
            "calibration_parameter_update_applied"
        ] = False

        replay[
            "same_batch_order_used"
        ] = False

        replay[
            "timestamp_jitter_applied"
        ] = False

        replay[
            "status"
        ] = "PASS"

        batch_mark_replay_frames.append(
            replay
        )


BATCH_MARK_BASELINE_REPLAY = (
    pd.concat(
        batch_mark_replay_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "scored_partition",
            "model_id",
            "current_batch_number",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Score summaries
# ------------------------------------------------------------

BATCH_MARK_SCORE_TABLE = (
    BATCH_MARK_BASELINE_REPLAY.groupby(
        [
            "model_id",
            "model_family",
            "scored_partition",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        transition_count=(
            "current_batch_id",
            "size",
        ),
        log_score_total=(
            "log_score",
            "sum",
        ),
        mean_log_score=(
            "log_score",
            "mean",
        ),
        mean_negative_log_score=(
            "negative_log_score",
            "mean",
        ),
        mean_brier_score=(
            "brier_score",
            "mean",
        ),
        top_one_accuracy=(
            "top_one_correct_flag",
            "mean",
        ),
        mean_prediction_entropy=(
            "prediction_entropy",
            "mean",
        ),
        cross_partition_history_count=(
            "cross_partition_history_flag",
            "sum",
        ),
        calibration_parameter_update_count=(
            "calibration_parameter_update_applied",
            "sum",
        ),
    )
    .reset_index()
)

BATCH_MARK_SCORE_TABLE[
    "fitted_partition"
] = "DEVELOPMENT"

BATCH_MARK_SCORE_TABLE[
    "score_space"
] = BATCH_MARK_SCORE_SPACE

BATCH_MARK_SCORE_TABLE[
    "dirichlet_alpha"
] = BATCH_MARK_DIRICHLET_ALPHA

BATCH_MARK_SCORE_TABLE[
    "locked_before_calibration"
] = True

BATCH_MARK_SCORE_TABLE[
    "status"
] = "PASS"

BATCH_MARK_SCORE_TABLE = (
    BATCH_MARK_SCORE_TABLE.sort_values(
        [
            "scored_partition",
            "model_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Realized-mark score decomposition
# ------------------------------------------------------------

BATCH_MARK_SCORE_BY_REALIZED_STATE = (
    BATCH_MARK_BASELINE_REPLAY.groupby(
        [
            "model_id",
            "scored_partition",
            "current_batch_mark",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        observation_count=(
            "current_batch_id",
            "size",
        ),
        log_score_total=(
            "log_score",
            "sum",
        ),
        mean_log_score=(
            "log_score",
            "mean",
        ),
        mean_realized_probability=(
            "realized_probability",
            "mean",
        ),
        mean_brier_score=(
            "brier_score",
            "mean",
        ),
        top_one_accuracy=(
            "top_one_correct_flag",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "scored_partition",
            "model_id",
            "current_batch_mark",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

BATCH_MARK_SCORE_BY_REALIZED_STATE[
    "status"
] = "PASS"


# ------------------------------------------------------------
# Model comparison
# ------------------------------------------------------------

batch_mark_score_pivot = (
    BATCH_MARK_SCORE_TABLE.pivot(
        index="scored_partition",
        columns="model_id",
        values="log_score_total",
    )
)

require(
    set(
        BATCH_MARK_MODEL_IDS
    ).issubset(
        set(
            batch_mark_score_pivot.columns
        )
    ),
    "Batch-mark comparison lacks a required model.",
)

BATCH_MARK_MODEL_COMPARISON = (
    batch_mark_score_pivot.reset_index()
)

BATCH_MARK_MODEL_COMPARISON[
    "markov_minus_iid_log_score"
] = (
    BATCH_MARK_MODEL_COMPARISON[
        MARKOV_BATCH_MARK_MODEL_ID
    ]
    - BATCH_MARK_MODEL_COMPARISON[
        IID_BATCH_MARK_MODEL_ID
    ]
)

BATCH_MARK_MODEL_COMPARISON[
    "preferred_model_by_locked_score"
] = np.where(
    BATCH_MARK_MODEL_COMPARISON[
        "markov_minus_iid_log_score"
    ]
    > 0.0,
    MARKOV_BATCH_MARK_MODEL_ID,
    IID_BATCH_MARK_MODEL_ID,
)

BATCH_MARK_MODEL_COMPARISON[
    "comparison_role"
] = (
    BATCH_MARK_MODEL_STATUS
)

BATCH_MARK_MODEL_COMPARISON[
    "status"
] = "PASS"


# ------------------------------------------------------------
# DEVELOPMENT dependence diagnostics
# ------------------------------------------------------------

development_transition_total = int(
    DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS.sum(
        dtype="int64"
    )
)

development_transition_joint_probability = (
    DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS.astype(
        "float64"
    )
    / development_transition_total
)

development_previous_mark_probability = (
    development_transition_joint_probability.sum(
        axis=1
    )
)

development_current_mark_probability = (
    development_transition_joint_probability.sum(
        axis=0
    )
)

independent_transition_probability = np.outer(
    development_previous_mark_probability,
    development_current_mark_probability,
)

positive_joint_mask = (
    development_transition_joint_probability
    > 0.0
)

development_mark_mutual_information_nats = float(
    np.sum(
        development_transition_joint_probability[
            positive_joint_mask
        ]
        * np.log(
            development_transition_joint_probability[
                positive_joint_mask
            ]
            / independent_transition_probability[
                positive_joint_mask
            ]
        ),
        dtype="float64",
    )
)

development_observed_same_mark_probability = float(
    np.trace(
        development_transition_joint_probability
    )
)

development_independent_same_mark_probability = float(
    np.trace(
        independent_transition_probability
    )
)

development_unconditional_entropy_nats = float(
    -np.sum(
        development_current_mark_probability[
            development_current_mark_probability
            > 0.0
        ]
        * np.log(
            development_current_mark_probability[
                development_current_mark_probability
                > 0.0
            ]
        ),
        dtype="float64",
    )
)

development_conditional_entropy_nats = float(
    development_unconditional_entropy_nats
    - development_mark_mutual_information_nats
)

BATCH_MARK_DEPENDENCE_SUMMARY = pd.DataFrame(
    [
        {
            "partition": (
                "DEVELOPMENT"
            ),
            "transition_count": (
                development_transition_total
            ),
            "state_count": (
                len(BATCH_MARK_STATES)
            ),
            "mutual_information_nats": (
                development_mark_mutual_information_nats
            ),
            "unconditional_entropy_nats": (
                development_unconditional_entropy_nats
            ),
            "conditional_entropy_nats": (
                development_conditional_entropy_nats
            ),
            "observed_same_mark_probability": (
                development_observed_same_mark_probability
            ),
            "independent_same_mark_probability": (
                development_independent_same_mark_probability
            ),
            "same_mark_probability_excess": (
                development_observed_same_mark_probability
                - development_independent_same_mark_probability
            ),
            "same_mark_probability_ratio": (
                development_observed_same_mark_probability
                / development_independent_same_mark_probability
            ),
            "timestamp_tie_order_used": False,
            "mixed_batches_preserved_as_single_state": True,
            "status": "PASS",
        }
    ]
)


# ------------------------------------------------------------
# Conditional-transition lift table
# ------------------------------------------------------------

transition_lift_rows: list[
    dict[str, Any]
] = []

for previous_index, previous_mark in enumerate(
    BATCH_MARK_STATES
):
    previous_total = int(
        DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS[
            previous_index
        ].sum(
            dtype="int64"
        )
    )

    for current_index, current_mark in enumerate(
        BATCH_MARK_STATES
    ):
        empirical_probability = (
            DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS[
                previous_index,
                current_index,
            ]
            / previous_total
        )

        iid_probability = float(
            DEVELOPMENT_IID_BATCH_MARK_PROBABILITIES[
                current_index
            ]
        )

        transition_lift_rows.append(
            {
                "previous_batch_mark": (
                    previous_mark
                ),
                "current_batch_mark": (
                    current_mark
                ),
                "development_transition_count": int(
                    DEVELOPMENT_BATCH_MARK_TRANSITION_COUNTS[
                        previous_index,
                        current_index,
                    ]
                ),
                "development_previous_state_total": (
                    previous_total
                ),
                "empirical_conditional_probability": float(
                    empirical_probability
                ),
                "smoothed_markov_probability": float(
                    DEVELOPMENT_BATCH_MARK_TRANSITION_PROBABILITIES[
                        previous_index,
                        current_index,
                    ]
                ),
                "iid_probability": (
                    iid_probability
                ),
                "conditional_probability_minus_iid": float(
                    empirical_probability
                    - iid_probability
                ),
                "conditional_probability_ratio_to_iid": float(
                    empirical_probability
                    / iid_probability
                ),
                "same_mark_flag": (
                    previous_mark
                    == current_mark
                ),
                "status": "PASS",
            }
        )


BATCH_MARK_CONDITIONAL_LIFT = (
    pd.DataFrame(
        transition_lift_rows
    )
    .sort_values(
        [
            "previous_batch_mark",
            "current_batch_mark",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Mark-baseline gates
# ------------------------------------------------------------

expected_development_transition_count = (
    len(DEVELOPMENT_BATCH_MARKS) - 1
)

expected_calibration_transition_count = (
    len(CALIBRATION_BATCH_MARKS)
)

observed_replay_counts = (
    BATCH_MARK_BASELINE_REPLAY.groupby(
        [
            "model_id",
            "scored_partition",
        ],
        observed=True,
    )[
        "current_batch_id"
    ]
    .size()
    .unstack(
        "scored_partition"
    )
)

BATCH_MARK_BASELINE_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "batch_mark_rows_conserved",
            "severity": "BLOCKING",
            "passed": (
                len(
                    PRIMARY_BATCH_MARK_TABLE
                )
                == expected_analytical_batch_rows
            ),
            "evidence": (
                f"batch_mark_rows="
                f"{len(PRIMARY_BATCH_MARK_TABLE)}"
            ),
        },
        {
            "gate": "mixed_side_batches_preserved",
            "severity": "BLOCKING",
            "passed": (
                int(
                    PRIMARY_BATCH_MARK_TABLE[
                        "batch_mark"
                    ]
                    .eq("MIXED")
                    .sum()
                )
                == expected_analytical_mixed_side_batches
            ),
            "evidence": (
                f"mixed_batches="
                f"{expected_analytical_mixed_side_batches}"
            ),
        },
        {
            "gate": "development_transition_count_conserved",
            "severity": "BLOCKING",
            "passed": bool(
                (
                    observed_replay_counts[
                        "DEVELOPMENT"
                    ]
                    == expected_development_transition_count
                ).all()
            ),
            "evidence": (
                f"transitions_per_model="
                f"{expected_development_transition_count}"
            ),
        },
        {
            "gate": "calibration_transition_count_conserved",
            "severity": "BLOCKING",
            "passed": bool(
                (
                    observed_replay_counts[
                        "CALIBRATION"
                    ]
                    == expected_calibration_transition_count
                ).all()
            ),
            "evidence": (
                f"transitions_per_model="
                f"{expected_calibration_transition_count}"
            ),
        },
        {
            "gate": "one_cross_partition_history_link_per_calibration_model",
            "severity": "BLOCKING",
            "passed": bool(
                BATCH_MARK_SCORE_TABLE.loc[
                    BATCH_MARK_SCORE_TABLE[
                        "scored_partition"
                    ].eq("CALIBRATION"),
                    "cross_partition_history_count",
                ]
                .eq(1)
                .all()
            ),
            "evidence": (
                "the first CALIBRATION batch uses the final "
                "DEVELOPMENT batch as causal history"
            ),
        },
        {
            "gate": "mark_probabilities_strictly_positive",
            "severity": "BLOCKING",
            "passed": bool(
                np.all(
                    DEVELOPMENT_IID_BATCH_MARK_PROBABILITIES
                    > 0.0
                )
                and np.all(
                    DEVELOPMENT_BATCH_MARK_TRANSITION_PROBABILITIES
                    > 0.0
                )
            ),
            "evidence": (
                f"symmetric_dirichlet_alpha="
                f"{BATCH_MARK_DIRICHLET_ALPHA}"
            ),
        },
        {
            "gate": "mark_probability_rows_sum_to_one",
            "severity": "BLOCKING",
            "passed": bool(
                np.isclose(
                    DEVELOPMENT_IID_BATCH_MARK_PROBABILITIES.sum(),
                    1.0,
                    rtol=0.0,
                    atol=1e-12,
                )
                and np.isclose(
                    DEVELOPMENT_BATCH_MARK_TRANSITION_PROBABILITIES.sum(
                        axis=1
                    ),
                    1.0,
                    rtol=0.0,
                    atol=1e-12,
                ).all()
            ),
            "evidence": (
                "IID vector and every Markov row are normalized"
            ),
        },
        {
            "gate": "all_mark_scores_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    BATCH_MARK_BASELINE_REPLAY[
                        [
                            "log_score",
                            "negative_log_score",
                            "brier_score",
                            "prediction_entropy",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                f"replay_rows="
                f"{len(BATCH_MARK_BASELINE_REPLAY)}"
            ),
        },
        {
            "gate": "calibration_parameters_not_updated",
            "severity": "BLOCKING",
            "passed": bool(
                not BATCH_MARK_BASELINE_REPLAY[
                    "calibration_parameter_update_applied"
                ].any()
            ),
            "evidence": (
                "all mark probabilities remain frozen from "
                "DEVELOPMENT"
            ),
        },
        {
            "gate": "same_time_event_order_not_imposed",
            "severity": "BLOCKING",
            "passed": bool(
                not BATCH_MARK_BASELINE_REPLAY[
                    "same_batch_order_used"
                ].any()
            ),
            "evidence": (
                "BUY_ONLY, SELL_ONLY, and MIXED are defined at "
                "the exact-time batch level"
            ),
        },
        {
            "gate": "timestamp_jitter_absent",
            "severity": "BLOCKING",
            "passed": bool(
                not BATCH_MARK_BASELINE_REPLAY[
                    "timestamp_jitter_applied"
                ].any()
            ),
            "evidence": (
                "canonical Notebook 04 event_time_ns used "
                "unchanged"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    BATCH_MARK_BASELINE_GATE_FRAME.loc[
        BATCH_MARK_BASELINE_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    (
        "At least one exact-time batch-mark baseline gate "
        "failed."
    ),
)


display(
    PRIMARY_BATCH_MARK_TABLE.groupby(
        [
            "event_partition",
            "batch_mark",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        batch_count=(
            "primary_event_batch_id",
            "size",
        ),
        event_count=(
            "batch_event_count",
            "sum",
        ),
        mean_batch_event_count=(
            "batch_event_count",
            "mean",
        ),
    )
    .reset_index()
)

display(IID_BATCH_MARK_PARAMETER_TABLE)
display(BATCH_MARK_TRANSITION_PARAMETER_TABLE)
display(BATCH_MARK_CONDITIONAL_LIFT)

display(
    BATCH_MARK_SCORE_TABLE[
        [
            "model_id",
            "scored_partition",
            "transition_count",
            "log_score_total",
            "mean_log_score",
            "mean_negative_log_score",
            "mean_brier_score",
            "top_one_accuracy",
            "mean_prediction_entropy",
            "cross_partition_history_count",
            "calibration_parameter_update_count",
            "status",
        ]
    ]
)

display(BATCH_MARK_MODEL_COMPARISON)
display(BATCH_MARK_SCORE_BY_REALIZED_STATE)
display(BATCH_MARK_DEPENDENCE_SUMMARY)
display(BATCH_MARK_BASELINE_GATE_FRAME)

print(
    "IID and first-order exact-time batch-mark baselines were "
    "fitted using DEVELOPMENT only."
)
print(
    "Batch marks are BUY_ONLY, SELL_ONLY, or MIXED; no order is "
    "imposed inside mixed exact-time batches."
)
print(
    "The first CALIBRATION batch is scored using the final "
    "DEVELOPMENT batch as causal mark history."
)
print(
    "Mark probabilities remain frozen throughout CALIBRATION, "
    "with no parameter updating."
)
print(
    "The mark baseline is diagnostic only and remains separate "
    "from event-time count and renewal likelihood score spaces."
)
print(
    "Hawkes estimation remains unauthorized."
)

,event_partition,batch_mark,batch_count,event_count,mean_batch_event_count
0,DEVELOPMENT,BUY_ONLY,3305,3380,1.0226929
1,DEVELOPMENT,SELL_ONLY,3528,3564,1.0102041
2,DEVELOPMENT,MIXED,26,60,2.3076923
3,CALIBRATION,BUY_ONLY,1150,1224,1.0643478
4,CALIBRATION,SELL_ONLY,1245,1258,1.0104418
5,CALIBRATION,MIXED,5,11,2.2


,model_id,model_family,fitted_partition,batch_mark,development_mark_count,dirichlet_alpha,predictive_probability,calibration_used_for_fit,status
0,M0_IID_BATCH_MARK,IID_EXACT_TIME_BATCH_MARK,DEVELOPMENT,BUY_ONLY,3305,0.5,0.48181619,False,PASS
1,M0_IID_BATCH_MARK,IID_EXACT_TIME_BATCH_MARK,DEVELOPMENT,SELL_ONLY,3528,0.5,0.51432111,False,PASS
2,M0_IID_BATCH_MARK,IID_EXACT_TIME_BATCH_MARK,DEVELOPMENT,MIXED,26,0.5,0.0038626922,False,PASS


,model_id,model_family,fitted_partition,previous_batch_mark,current_batch_mark,development_transition_count,development_previous_state_total,dirichlet_alpha,predictive_probability,calibration_used_for_fit,status
0,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,BUY_ONLY,BUY_ONLY,1814,3305,0.5,0.54876758,False,PASS
1,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,BUY_ONLY,MIXED,15,3305,0.5,0.0046877363,False,PASS
2,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,BUY_ONLY,SELL_ONLY,1476,3305,0.5,0.44654468,False,PASS
3,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,MIXED,BUY_ONLY,11,26,0.5,0.41818182,False,PASS
4,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,MIXED,MIXED,1,26,0.5,0.054545455,False,PASS
5,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,MIXED,SELL_ONLY,14,26,0.5,0.52727273,False,PASS
6,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,SELL_ONLY,BUY_ONLY,1479,3527,0.5,0.41929999,False,PASS
7,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,SELL_ONLY,MIXED,10,3527,0.5,0.0029757687,False,PASS
8,M1_FIRST_ORDER_BATCH_MARKOV,FIRST_ORDER_EXACT_TIME_BATCH_MARKOV,DEVELOPMENT,SELL_ONLY,SELL_ONLY,2038,3527,0.5,0.57772425,False,PASS


,previous_batch_mark,current_batch_mark,development_transition_count,development_previous_state_total,empirical_conditional_probability,smoothed_markov_probability,iid_probability,conditional_probability_minus_iid,conditional_probability_ratio_to_iid,same_mark_flag,status
0,BUY_ONLY,BUY_ONLY,1814,3305,0.54886536,0.54876758,0.48181619,0.067049161,1.1391592,True,PASS
1,BUY_ONLY,MIXED,15,3305,0.0045385779,0.0046877363,0.0038626922,0.00067588569,1.1749779,False,PASS
2,BUY_ONLY,SELL_ONLY,1476,3305,0.44659607,0.44654468,0.51432111,-0.067725047,0.86832147,False,PASS
3,MIXED,BUY_ONLY,11,26,0.42307692,0.41818182,0.48181619,-0.058739271,0.8780878,False,PASS
4,MIXED,MIXED,1,26,0.038461538,0.054545455,0.0038626922,0.034598846,9.9571843,True,PASS
5,MIXED,SELL_ONLY,14,26,0.53846154,0.52727273,0.51432111,0.024140425,1.0469365,False,PASS
6,SELL_ONLY,BUY_ONLY,1479,3527,0.41933655,0.41929999,0.48181619,-0.062479648,0.87032472,False,PASS
7,SELL_ONLY,MIXED,10,3527,0.0028352708,0.0029757687,0.0038626922,-0.0010274215,0.73401415,False,PASS
8,SELL_ONLY,SELL_ONLY,2038,3527,0.57782818,0.57772425,0.51432111,0.063507069,1.1234775,True,PASS


,model_id,scored_partition,transition_count,log_score_total,mean_log_score,mean_negative_log_score,mean_brier_score,top_one_accuracy,mean_prediction_entropy,cross_partition_history_count,calibration_parameter_update_count,status
0,M0_IID_BATCH_MARK,CALIBRATION,2400,"-1,695.3132",-0.70638051,0.70638051,0.5013232,0.51875,0.71525719,1,0,PASS
1,M1_FIRST_ORDER_BATCH_MARKOV,CALIBRATION,2400,"-1,676.3428",-0.69847617,0.69847617,0.49381027,0.56083333,0.7066008,1,0,PASS
2,M0_IID_BATCH_MARK,DEVELOPMENT,6858,"-4,902.816",-0.71490464,0.71490464,0.50323623,0.5144357,0.71525719,0,0,PASS
3,M1_FIRST_ORDER_BATCH_MARKOV,DEVELOPMENT,6858,"-4,842.2609",-0.70607479,0.70607479,0.49475846,0.5637212,0.70692075,0,0,PASS


model_id,scored_partition,M0_IID_BATCH_MARK,M1_FIRST_ORDER_BATCH_MARKOV,markov_minus_iid_log_score,preferred_model_by_locked_score,comparison_role,status
0,CALIBRATION,"-1,695.3132","-1,676.3428",18.970414,M1_FIRST_ORDER_BATCH_MARKOV,DIAGNOSTIC_ONLY_NO_HYPERPARAMETER_SELECTION,PASS
1,DEVELOPMENT,"-4,902.816","-4,842.2609",60.555076,M1_FIRST_ORDER_BATCH_MARKOV,DIAGNOSTIC_ONLY_NO_HYPERPARAMETER_SELECTION,PASS


,model_id,scored_partition,current_batch_mark,observation_count,log_score_total,mean_log_score,mean_realized_probability,mean_brier_score,top_one_accuracy,status
0,M0_IID_BATCH_MARK,CALIBRATION,BUY_ONLY,1150,-839.72146,-0.73019258,0.48181619,0.53305558,0,PASS
1,M0_IID_BATCH_MARK,CALIBRATION,MIXED,5,-27.781954,-5.5563909,0.0038626922,1.4889626,0,PASS
2,M0_IID_BATCH_MARK,CALIBRATION,SELL_ONLY,1245,-827.80981,-0.66490747,0.51432111,0.46804575,1,PASS
3,M1_FIRST_ORDER_BATCH_MARKOV,CALIBRATION,BUY_ONLY,1150,-831.91258,-0.72340224,0.48943376,0.5256476,0.54173913,PASS
4,M1_FIRST_ORDER_BATCH_MARKOV,CALIBRATION,MIXED,5,-26.814027,-5.3628055,0.0046877363,1.4911945,0,PASS
5,M1_FIRST_ORDER_BATCH_MARKOV,CALIBRATION,SELL_ONLY,1245,-817.6162,-0.65671984,0.52268314,0.46039674,0.58072289,PASS
6,M0_IID_BATCH_MARK,DEVELOPMENT,BUY_ONLY,3304,"-2,412.5563",-0.73019258,0.48181619,0.53305558,0,PASS
7,M0_IID_BATCH_MARK,DEVELOPMENT,MIXED,26,-144.46616,-5.5563909,0.0038626922,1.4889626,0,PASS
8,M0_IID_BATCH_MARK,DEVELOPMENT,SELL_ONLY,3528,"-2,345.7936",-0.66490747,0.51432111,0.46804575,1,PASS
9,M1_FIRST_ORDER_BATCH_MARKOV,DEVELOPMENT,BUY_ONLY,3304,"-2,383.6363",-0.72143956,0.49037805,0.52370127,0.54903148,PASS


,partition,transition_count,state_count,mutual_information_nats,unconditional_entropy_nats,conditional_entropy_nats,observed_same_mark_probability,independent_same_mark_probability,same_mark_probability_excess,same_mark_probability_ratio,timestamp_tie_order_used,mixed_batches_preserved_as_single_state,status
0,DEVELOPMENT,6858,3,0.0088426528,0.71490396,0.7060613,0.56182561,0.49675903,0.065066579,1.1309822,False,True,PASS


,gate,severity,passed,evidence
0,batch_mark_rows_conserved,BLOCKING,True,batch_mark_rows=9259
1,mixed_side_batches_preserved,BLOCKING,True,mixed_batches=31
2,development_transition_count_conserved,BLOCKING,True,transitions_per_model=6858
3,calibration_transition_count_conserved,BLOCKING,True,transitions_per_model=2400
4,one_cross_partition_history_link_per_calibrati...,BLOCKING,True,the first CALIBRATION batch uses the final DEV...
5,mark_probabilities_strictly_positive,BLOCKING,True,symmetric_dirichlet_alpha=0.5
6,mark_probability_rows_sum_to_one,BLOCKING,True,IID vector and every Markov row are normalized
7,all_mark_scores_finite,BLOCKING,True,replay_rows=18516
8,calibration_parameters_not_updated,BLOCKING,True,all mark probabilities remain frozen from DEVE...
9,same_time_event_order_not_imposed,BLOCKING,True,"BUY_ONLY, SELL_ONLY, and MIXED are defined at ..."


IID and first-order exact-time batch-mark baselines were fitted using DEVELOPMENT only.
Batch marks are BUY_ONLY, SELL_ONLY, or MIXED; no order is imposed inside mixed exact-time batches.
The first CALIBRATION batch is scored using the final DEVELOPMENT batch as causal mark history.
Mark probabilities remain frozen throughout CALIBRATION, with no parameter updating.
The mark baseline is diagnostic only and remains separate from event-time count and renewal likelihood score spaces.
Hawkes estimation remains unauthorized.


In [15]:
# ============================================================
# Unified count-baseline ledger and locked block-bootstrap comparison
# ============================================================

PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID: Final[str] = (
    SELECTED_ADAPTIVE_MODEL_ID
)

FORMAL_COUNT_COMPARATOR_MODEL_ID: Final[str] = (
    SELECTED_DETERMINISTIC_MODEL_ID
)

COUNT_SCORE_SPACE: Final[str] = (
    "BIVARIATE_SIDE_COUNT_NATIVE_1MS"
)

FORMAL_BOOTSTRAP_PARTITION: Final[str] = (
    "CALIBRATION"
)

FORMAL_BOOTSTRAP_BLOCK_NS: Final[int] = (
    BOOTSTRAP_BLOCK_SECONDS
    * NANOSECONDS_PER_SECOND
)

COUNT_SCORE_RECONCILIATION_TOLERANCE: Final[float] = (
    1e-7
)


# ------------------------------------------------------------
# Ordered model-set construction
# ------------------------------------------------------------

def ordered_unique(
    values: Iterable[str],
) -> tuple[str, ...]:
    """Return unique strings while preserving first appearance."""
    return tuple(
        dict.fromkeys(
            str(value)
            for value in values
        )
    )


SELECTED_COUNT_BASELINE_MODEL_IDS: Final[
    tuple[str, ...]
] = ordered_unique(
    (
        "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON",
        SELECTED_DETERMINISTIC_MODEL_ID,
        SELECTED_ROLLING_MODEL_ID,
        SELECTED_EWMA_MODEL_ID,
        SELECTED_ADAPTIVE_MODEL_ID,
    )
)

require(
    PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
    in SELECTED_COUNT_BASELINE_MODEL_IDS,
    "Primary simple count baseline is absent from the selected model set.",
)

require(
    FORMAL_COUNT_COMPARATOR_MODEL_ID
    in SELECTED_COUNT_BASELINE_MODEL_IDS,
    "Formal count comparator is absent from the selected model set.",
)


# ------------------------------------------------------------
# Model-role registry
# ------------------------------------------------------------

count_model_role_map: dict[str, list[str]] = {}

def add_count_model_role(
    model_id: str,
    role: str,
) -> None:
    """Attach a declared scientific role to a selected count model."""
    count_model_role_map.setdefault(
        model_id,
        [],
    ).append(role)


add_count_model_role(
    "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON",
    "HOMOGENEOUS_SIDE_SPECIFIC_ANCHOR",
)

add_count_model_role(
    SELECTED_DETERMINISTIC_MODEL_ID,
    "DEVELOPMENT_SELECTED_DETERMINISTIC",
)

add_count_model_role(
    SELECTED_ROLLING_MODEL_ID,
    "DEVELOPMENT_SELECTED_ROLLING",
)

add_count_model_role(
    SELECTED_EWMA_MODEL_ID,
    "DEVELOPMENT_SELECTED_EWMA",
)

add_count_model_role(
    SELECTED_ADAPTIVE_MODEL_ID,
    "PRIMARY_SIMPLE_COUNT_BASELINE",
)


def count_model_family(
    model_id: str,
) -> str:
    """Return the authoritative family for a selected count model."""
    if (
        model_id
        == "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
    ):
        return (
            "INDEPENDENT_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
        )

    if (
        model_id
        in DETERMINISTIC_POISSON_MODELS
    ):
        return (
            DETERMINISTIC_POISSON_MODELS[
                model_id
            ].model_family
        )

    if (
        model_id
        in ADAPTIVE_RATE_MODELS
    ):
        return (
            ADAPTIVE_RATE_MODELS[
                model_id
            ].model_family
        )

    raise KeyError(
        f"Unknown selected count model: {model_id}"
    )


def count_model_hyperparameter_description(
    model_id: str,
) -> str:
    """Return a portable selected-model specification."""
    if (
        model_id
        == "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
    ):
        return (
            "BUY_RATE_AND_SELL_RATE_ESTIMATED_ON_DEVELOPMENT"
        )

    if (
        model_id
        in DETERMINISTIC_POISSON_MODELS
    ):
        model = (
            DETERMINISTIC_POISSON_MODELS[
                model_id
            ]
        )

        return (
            f"SPECIFICATION={model.specification}"
        )

    if (
        model_id
        in ADAPTIVE_RATE_MODELS
    ):
        model = (
            ADAPTIVE_RATE_MODELS[
                model_id
            ]
        )

        return (
            f"{model.hyperparameter_name.upper()}="
            f"{model.hyperparameter_ms}"
        )

    raise KeyError(
        f"Unknown selected count model: {model_id}"
    )


COUNT_BASELINE_MODEL_REGISTRY = pd.DataFrame(
    [
        {
            "model_id": model_id,
            "model_family": (
                count_model_family(
                    model_id
                )
            ),
            "declared_roles": ";".join(
                count_model_role_map[
                    model_id
                ]
            ),
            "model_specification": (
                count_model_hyperparameter_description(
                    model_id
                )
            ),
            "fit_partition": "DEVELOPMENT",
            "selection_partition": (
                "DEVELOPMENT"
            ),
            "calibration_used_for_selection": False,
            "score_space": (
                COUNT_SCORE_SPACE
            ),
            "primary_simple_count_baseline_flag": (
                model_id
                == PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
            ),
            "formal_count_comparator_flag": (
                model_id
                == FORMAL_COUNT_COMPARATOR_MODEL_ID
            ),
            "status": "PASS",
        }
        for model_id
        in SELECTED_COUNT_BASELINE_MODEL_IDS
    ]
)


# ------------------------------------------------------------
# Expected-count registry
# ------------------------------------------------------------

COUNT_BASELINE_EXPECTED_COUNTS: dict[
    tuple[str, str, str],
    np.ndarray,
] = {}


def freeze_expected_counts(
    *,
    model_id: str,
    partition_name: str,
    side: str,
    expected_counts: np.ndarray,
) -> None:
    """Validate and retain an immutable expected-count array."""
    require(
        model_id
        in SELECTED_COUNT_BASELINE_MODEL_IDS,
        f"Unexpected selected count model: {model_id}",
    )
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized selected-count partition: "
            f"{partition_name}"
        ),
    )
    require(
        side in {"BUY", "SELL"},
        f"Unsupported selected-count side: {side}",
    )

    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    expected = np.asarray(
        expected_counts,
        dtype="float64",
    )

    require(
        expected.shape
        == (grid.cell_count,),
        (
            f"{model_id} {partition_name} {side} expected-count "
            "shape differs from the native grid."
        ),
    )
    require(
        np.isfinite(
            expected
        ).all(),
        (
            f"{model_id} {partition_name} {side} expected counts "
            "contain nonfinite values."
        ),
    )
    require(
        np.all(
            expected > 0.0
        ),
        (
            f"{model_id} {partition_name} {side} expected counts "
            "are not strictly positive."
        ),
    )

    frozen = expected.copy()
    frozen.setflags(
        write=False
    )

    COUNT_BASELINE_EXPECTED_COUNTS[
        (
            model_id,
            partition_name,
            side,
        )
    ] = frozen


for partition_name in ANALYTICAL_PARTITIONS:
    partition_grid = (
        NATIVE_COUNT_GRIDS[
            partition_name
        ]
    )

    partition_exposure_seconds = (
        grid_exposure_seconds(
            partition_grid
        )
    )

    freeze_expected_counts(
        model_id=(
            "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
        ),
        partition_name=partition_name,
        side="BUY",
        expected_counts=(
            DEVELOPMENT_BUY_RATE_PER_SECOND
            * partition_exposure_seconds
        ),
    )

    freeze_expected_counts(
        model_id=(
            "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
        ),
        partition_name=partition_name,
        side="SELL",
        expected_counts=(
            DEVELOPMENT_SELL_RATE_PER_SECOND
            * partition_exposure_seconds
        ),
    )

    freeze_expected_counts(
        model_id=(
            SELECTED_DETERMINISTIC_MODEL_ID
        ),
        partition_name=partition_name,
        side="BUY",
        expected_counts=(
            SELECTED_DETERMINISTIC_EXPECTED_COUNTS[
                (
                    partition_name,
                    "BUY",
                )
            ]
        ),
    )

    freeze_expected_counts(
        model_id=(
            SELECTED_DETERMINISTIC_MODEL_ID
        ),
        partition_name=partition_name,
        side="SELL",
        expected_counts=(
            SELECTED_DETERMINISTIC_EXPECTED_COUNTS[
                (
                    partition_name,
                    "SELL",
                )
            ]
        ),
    )

    for adaptive_model_id in ordered_unique(
        (
            SELECTED_ROLLING_MODEL_ID,
            SELECTED_EWMA_MODEL_ID,
            SELECTED_ADAPTIVE_MODEL_ID,
        )
    ):
        for side in (
            "BUY",
            "SELL",
        ):
            freeze_expected_counts(
                model_id=(
                    adaptive_model_id
                ),
                partition_name=(
                    partition_name
                ),
                side=side,
                expected_counts=(
                    SELECTED_ADAPTIVE_EXPECTED_COUNTS[
                        (
                            adaptive_model_id,
                            partition_name,
                            side,
                        )
                    ]
                ),
            )


expected_registry_key_count = (
    len(
        SELECTED_COUNT_BASELINE_MODEL_IDS
    )
    * len(
        ANALYTICAL_PARTITIONS
    )
    * 2
)

require(
    len(
        COUNT_BASELINE_EXPECTED_COUNTS
    )
    == expected_registry_key_count,
    (
        "Selected count expected-count registry is incomplete: "
        f"expected={expected_registry_key_count}; "
        f"observed={len(COUNT_BASELINE_EXPECTED_COUNTS)}"
    ),
)


# ------------------------------------------------------------
# Cell-level log-score contributions
# ------------------------------------------------------------

def poisson_log_score_terms(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
) -> np.ndarray:
    """Return complete cell-level Poisson log-score contributions."""
    observed = np.asarray(
        observed_counts,
        dtype="int64",
    )

    expected = np.asarray(
        expected_counts,
        dtype="float64",
    )

    require(
        observed.shape == expected.shape,
        "Observed and expected count arrays differ in shape.",
    )
    require(
        np.all(
            observed >= 0
        ),
        "Observed count array contains negative values.",
    )
    require(
        np.isfinite(
            expected
        ).all()
        and np.all(
            expected > 0.0
        ),
        "Expected count array must be finite and positive.",
    )

    terms = (
        observed.astype(
            "float64"
        )
        * np.log(
            expected
        )
        - expected
        - special.gammaln(
            observed.astype(
                "float64"
            )
            + 1.0
        )
    )

    require(
        np.isfinite(
            terms
        ).all(),
        "Poisson log-score contributions contain nonfinite values.",
    )

    return terms


COUNT_BASELINE_CELL_LOG_SCORES: dict[
    tuple[str, str],
    np.ndarray,
] = {}

COUNT_BASELINE_COMPONENT_LOG_SCORES: dict[
    tuple[str, str, str],
    np.ndarray,
] = {}


for model_id in (
    SELECTED_COUNT_BASELINE_MODEL_IDS
):
    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        grid = (
            NATIVE_COUNT_GRIDS[
                partition_name
            ]
        )

        buy_terms = (
            poisson_log_score_terms(
                grid.buy_counts,
                COUNT_BASELINE_EXPECTED_COUNTS[
                    (
                        model_id,
                        partition_name,
                        "BUY",
                    )
                ],
            )
        )

        sell_terms = (
            poisson_log_score_terms(
                grid.sell_counts,
                COUNT_BASELINE_EXPECTED_COUNTS[
                    (
                        model_id,
                        partition_name,
                        "SELL",
                    )
                ],
            )
        )

        joint_terms = (
            buy_terms + sell_terms
        )

        for (
            side,
            terms,
        ) in (
            ("BUY", buy_terms),
            ("SELL", sell_terms),
        ):
            frozen_component_terms = (
                terms.copy()
            )
            frozen_component_terms.setflags(
                write=False
            )

            COUNT_BASELINE_COMPONENT_LOG_SCORES[
                (
                    model_id,
                    partition_name,
                    side,
                )
            ] = frozen_component_terms

        frozen_joint_terms = (
            joint_terms.copy()
        )
        frozen_joint_terms.setflags(
            write=False
        )

        COUNT_BASELINE_CELL_LOG_SCORES[
            (
                model_id,
                partition_name,
            )
        ] = frozen_joint_terms


# ------------------------------------------------------------
# Unified score table
# ------------------------------------------------------------

unified_count_score_rows: list[
    dict[str, Any]
] = []

for model_id in (
    SELECTED_COUNT_BASELINE_MODEL_IDS
):
    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        grid = (
            NATIVE_COUNT_GRIDS[
                partition_name
            ]
        )

        buy_expected = (
            COUNT_BASELINE_EXPECTED_COUNTS[
                (
                    model_id,
                    partition_name,
                    "BUY",
                )
            ]
        )

        sell_expected = (
            COUNT_BASELINE_EXPECTED_COUNTS[
                (
                    model_id,
                    partition_name,
                    "SELL",
                )
            ]
        )

        joint_cell_scores = (
            COUNT_BASELINE_CELL_LOG_SCORES[
                (
                    model_id,
                    partition_name,
                )
            ]
        )

        observed_event_count = int(
            grid.buy_counts.sum(
                dtype="int64"
            )
            + grid.sell_counts.sum(
                dtype="int64"
            )
        )

        predicted_event_count = float(
            buy_expected.sum(
                dtype="float64"
            )
            + sell_expected.sum(
                dtype="float64"
            )
        )

        exposure_seconds = (
            grid.contract_duration_ns
            / NANOSECONDS_PER_SECOND
        )

        total_log_score = float(
            joint_cell_scores.sum(
                dtype="float64"
            )
        )

        unified_count_score_rows.append(
            {
                "model_id": model_id,
                "model_family": (
                    count_model_family(
                        model_id
                    )
                ),
                "declared_roles": ";".join(
                    count_model_role_map[
                        model_id
                    ]
                ),
                "fitted_partition": (
                    "DEVELOPMENT"
                ),
                "scored_partition": (
                    partition_name
                ),
                "score_space": (
                    COUNT_SCORE_SPACE
                ),
                "native_grid_width_ms": (
                    NATIVE_COUNT_GRID_WIDTH_MS
                ),
                "cell_count": (
                    grid.cell_count
                ),
                "exposure_seconds": (
                    exposure_seconds
                ),
                "observed_event_count": (
                    observed_event_count
                ),
                "predicted_event_count": (
                    predicted_event_count
                ),
                "observed_minus_predicted": (
                    observed_event_count
                    - predicted_event_count
                ),
                "log_score_total": (
                    total_log_score
                ),
                "log_score_per_second": (
                    total_log_score
                    / exposure_seconds
                ),
                "log_score_per_observed_event": (
                    total_log_score
                    / observed_event_count
                ),
                "primary_simple_count_baseline_flag": (
                    model_id
                    == PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
                ),
                "formal_count_comparator_flag": (
                    model_id
                    == FORMAL_COUNT_COMPARATOR_MODEL_ID
                ),
                "calibration_used_for_selection": False,
                "locked_before_calibration": True,
                "status": "PASS",
            }
        )


UNIFIED_COUNT_BASELINE_SCORE_TABLE = (
    pd.DataFrame(
        unified_count_score_rows
    )
    .sort_values(
        [
            "scored_partition",
            "log_score_total",
        ],
        ascending=[
            True,
            False,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Reconcile against earlier model-family score tables
# ------------------------------------------------------------

def registered_aggregate_log_score(
    *,
    model_id: str,
    partition_name: str,
) -> float:
    """Return the previously registered aggregate count score."""
    if (
        model_id
        == "B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON"
    ):
        source = (
            HOMOGENEOUS_POISSON_SCORE_TABLE
        )

    elif (
        model_id
        in DETERMINISTIC_POISSON_MODELS
    ):
        source = (
            DETERMINISTIC_POISSON_SCORE_TABLE
        )

    elif (
        model_id
        in ADAPTIVE_RATE_MODELS
    ):
        source = (
            ADAPTIVE_RATE_SCORE_TABLE
        )

    else:
        raise KeyError(
            f"Unknown count-score source for {model_id}"
        )

    matching_rows = source.loc[
        source[
            "model_id"
        ].eq(model_id)
        & source[
            "scored_partition"
        ].eq(partition_name)
        & source[
            "component"
        ].eq(
            "BUY_AND_SELL_AGGREGATE"
        )
    ]

    require(
        len(matching_rows) == 1,
        (
            f"Expected one registered aggregate score for "
            f"{model_id} on {partition_name}; "
            f"observed={len(matching_rows)}."
        ),
    )

    return float(
        matching_rows.iloc[0][
            "log_score_total"
        ]
    )


count_score_reconciliation_rows: list[
    dict[str, Any]
] = []

for row in (
    UNIFIED_COUNT_BASELINE_SCORE_TABLE.itertuples(
        index=False
    )
):
    registered_score = (
        registered_aggregate_log_score(
            model_id=row.model_id,
            partition_name=(
                row.scored_partition
            ),
        )
    )

    score_difference = (
        row.log_score_total
        - registered_score
    )

    count_score_reconciliation_rows.append(
        {
            "model_id": (
                row.model_id
            ),
            "scored_partition": (
                row.scored_partition
            ),
            "unified_log_score": (
                row.log_score_total
            ),
            "registered_log_score": (
                registered_score
            ),
            "score_difference": (
                score_difference
            ),
            "absolute_score_difference": abs(
                score_difference
            ),
            "tolerance": (
                COUNT_SCORE_RECONCILIATION_TOLERANCE
            ),
            "status": (
                "PASS"
                if abs(
                    score_difference
                )
                <= COUNT_SCORE_RECONCILIATION_TOLERANCE
                else "FAIL"
            ),
        }
    )


COUNT_SCORE_RECONCILIATION = pd.DataFrame(
    count_score_reconciliation_rows
)

require(
    COUNT_SCORE_RECONCILIATION[
        "status"
    ].eq("PASS").all(),
    (
        "Unified count-baseline scores do not reconcile with "
        "their originating model-family tables."
    ),
)


# ------------------------------------------------------------
# Locked CALIBRATION ranking
# ------------------------------------------------------------

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING = (
    UNIFIED_COUNT_BASELINE_SCORE_TABLE.loc[
        UNIFIED_COUNT_BASELINE_SCORE_TABLE[
            "scored_partition"
        ].eq(
            FORMAL_BOOTSTRAP_PARTITION
        )
    ]
    .copy()
    .sort_values(
        [
            "log_score_total",
            "model_id",
        ],
        ascending=[
            False,
            True,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
    "calibration_score_rank"
] = (
    np.arange(
        1,
        len(
            LOCKED_CALIBRATION_COUNT_BASELINE_RANKING
        )
        + 1,
        dtype="int64",
    )
)

comparator_calibration_score = float(
    LOCKED_CALIBRATION_COUNT_BASELINE_RANKING.loc[
        LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
            "model_id"
        ].eq(
            FORMAL_COUNT_COMPARATOR_MODEL_ID
        ),
        "log_score_total",
    ].iloc[0]
)

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
    "log_score_improvement_over_formal_comparator"
] = (
    LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
        "log_score_total"
    ]
    - comparator_calibration_score
)

LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
    "ranking_role"
] = (
    "LOCKED_CALIBRATION_DIAGNOSTIC_ONLY"
)


# ------------------------------------------------------------
# Exact complete-block score aggregation
# ------------------------------------------------------------

bootstrap_grid = (
    NATIVE_COUNT_GRIDS[
        FORMAL_BOOTSTRAP_PARTITION
    ]
)

require(
    FORMAL_BOOTSTRAP_BLOCK_NS
    % bootstrap_grid.grid_width_ns
    == 0,
    (
        "Bootstrap block width is not an integer multiple of "
        "the native count-grid width."
    ),
)

CELLS_PER_FORMAL_BOOTSTRAP_BLOCK: Final[int] = (
    FORMAL_BOOTSTRAP_BLOCK_NS
    // bootstrap_grid.grid_width_ns
)

FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT: Final[int] = (
    bootstrap_grid.contract_duration_ns
    // FORMAL_BOOTSTRAP_BLOCK_NS
)

FORMAL_BOOTSTRAP_CELL_COUNT: Final[int] = (
    FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT
    * CELLS_PER_FORMAL_BOOTSTRAP_BLOCK
)

FORMAL_BOOTSTRAP_EXPOSURE_NS: Final[int] = (
    FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT
    * FORMAL_BOOTSTRAP_BLOCK_NS
)

FORMAL_BOOTSTRAP_EXCLUDED_TAIL_NS: Final[int] = (
    bootstrap_grid.contract_duration_ns
    - FORMAL_BOOTSTRAP_EXPOSURE_NS
)

require(
    FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT
    >= 2,
    (
        "Formal block bootstrap requires at least two complete "
        "CALIBRATION blocks."
    ),
)

require(
    FORMAL_BOOTSTRAP_CELL_COUNT
    <= bootstrap_grid.cell_count,
    "Formal bootstrap cell count exceeds the native grid.",
)

require(
    FORMAL_BOOTSTRAP_EXCLUDED_TAIL_NS
    >= 0,
    "Formal bootstrap excluded tail is negative.",
)

require(
    FORMAL_BOOTSTRAP_EXCLUDED_TAIL_NS
    < FORMAL_BOOTSTRAP_BLOCK_NS,
    (
        "Formal bootstrap excluded tail reaches or exceeds one "
        "complete block."
    ),
)


block_score_frames: list[
    pd.DataFrame
] = []

for model_id in (
    SELECTED_COUNT_BASELINE_MODEL_IDS
):
    complete_cell_scores = (
        COUNT_BASELINE_CELL_LOG_SCORES[
            (
                model_id,
                FORMAL_BOOTSTRAP_PARTITION,
            )
        ][
            :FORMAL_BOOTSTRAP_CELL_COUNT
        ]
    )

    require(
        complete_cell_scores.size
        == FORMAL_BOOTSTRAP_CELL_COUNT,
        (
            f"{model_id} formal bootstrap score coverage is "
            "incomplete."
        ),
    )

    block_scores = (
        complete_cell_scores.reshape(
            FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT,
            CELLS_PER_FORMAL_BOOTSTRAP_BLOCK,
        )
        .sum(
            axis=1,
            dtype="float64",
        )
    )

    block_indices = np.arange(
        FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT,
        dtype="int64",
    )

    block_start_ns = (
        bootstrap_grid.contract_start_ns
        + block_indices
        * FORMAL_BOOTSTRAP_BLOCK_NS
    )

    block_end_exclusive_ns = (
        block_start_ns
        + FORMAL_BOOTSTRAP_BLOCK_NS
    )

    block_score_frames.append(
        pd.DataFrame(
            {
                "model_id": (
                    model_id
                ),
                "scored_partition": (
                    FORMAL_BOOTSTRAP_PARTITION
                ),
                "block_index": (
                    block_indices
                ),
                "block_start_ns": (
                    block_start_ns
                ),
                "block_end_exclusive_ns": (
                    block_end_exclusive_ns
                ),
                "block_start_utc": (
                    pd.to_datetime(
                        block_start_ns,
                        unit="ns",
                        utc=True,
                    )
                ),
                "block_end_exclusive_utc": (
                    pd.to_datetime(
                        block_end_exclusive_ns,
                        unit="ns",
                        utc=True,
                    )
                ),
                "block_exposure_seconds": (
                    BOOTSTRAP_BLOCK_SECONDS
                ),
                "log_score_total": (
                    block_scores
                ),
                "complete_equal_exposure_block_flag": True,
                "status": "PASS",
            }
        )
    )


CALIBRATION_COUNT_SCORE_BLOCKS = (
    pd.concat(
        block_score_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "model_id",
            "block_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Bootstrap comparison pairs
# ------------------------------------------------------------

raw_comparison_pairs = (
    (
        SELECTED_ROLLING_MODEL_ID,
        FORMAL_COUNT_COMPARATOR_MODEL_ID,
    ),
    (
        SELECTED_EWMA_MODEL_ID,
        FORMAL_COUNT_COMPARATOR_MODEL_ID,
    ),
    (
        SELECTED_ADAPTIVE_MODEL_ID,
        FORMAL_COUNT_COMPARATOR_MODEL_ID,
    ),
    (
        SELECTED_ADAPTIVE_MODEL_ID,
        SELECTED_ROLLING_MODEL_ID,
    ),
    (
        SELECTED_ADAPTIVE_MODEL_ID,
        SELECTED_EWMA_MODEL_ID,
    ),
)

comparison_pairs: list[
    tuple[str, str]
] = []

seen_comparison_pairs: set[
    tuple[str, str]
] = set()

for candidate_model_id, comparator_model_id in (
    raw_comparison_pairs
):
    pair = (
        candidate_model_id,
        comparator_model_id,
    )

    if (
        candidate_model_id
        == comparator_model_id
    ):
        continue

    if pair in seen_comparison_pairs:
        continue

    seen_comparison_pairs.add(
        pair
    )

    comparison_pairs.append(
        pair
    )


require(
    comparison_pairs,
    "No nontrivial formal count-model comparison pair remains.",
)


# ------------------------------------------------------------
# Paired moving-block bootstrap over fixed 10-second blocks
# ------------------------------------------------------------

def bootstrap_score_difference(
    candidate_block_scores: np.ndarray,
    comparator_block_scores: np.ndarray,
    *,
    replicates: int,
    seed_sequence: Sequence[int],
) -> dict[str, float | bool]:
    """
    Bootstrap the total paired score difference across equal blocks.

    Each replicate samples complete CALIBRATION blocks with replacement.
    Candidate and comparator scores always use the same sampled indices.
    """
    candidate = np.asarray(
        candidate_block_scores,
        dtype="float64",
    )

    comparator = np.asarray(
        comparator_block_scores,
        dtype="float64",
    )

    require(
        candidate.ndim == 1,
        "Candidate block-score array must be one-dimensional.",
    )
    require(
        candidate.shape
        == comparator.shape,
        (
            "Candidate and comparator block-score arrays differ "
            "in shape."
        ),
    )
    require(
        candidate.size
        == FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT,
        (
            "Block-score array length differs from the formal "
            "complete-block count."
        ),
    )
    require(
        np.isfinite(
            candidate
        ).all()
        and np.isfinite(
            comparator
        ).all(),
        "Block-score arrays contain nonfinite values.",
    )
    require(
        replicates >= 1,
        "Bootstrap replicate count must be positive.",
    )

    paired_difference = (
        candidate - comparator
    )

    observed_total_difference = float(
        paired_difference.sum(
            dtype="float64"
        )
    )

    bootstrap_rng = (
        np.random.default_rng(
            np.random.SeedSequence(
                seed_sequence
            )
        )
    )

    sampled_block_indices = (
        bootstrap_rng.integers(
            low=0,
            high=paired_difference.size,
            size=(
                replicates,
                paired_difference.size,
            ),
            endpoint=False,
        )
    )

    bootstrap_total_differences = (
        paired_difference[
            sampled_block_indices
        ]
        .sum(
            axis=1,
            dtype="float64",
        )
    )

    bootstrap_quantiles = np.quantile(
        bootstrap_total_differences,
        [
            0.025,
            0.50,
            0.975,
        ],
    )

    probability_nonpositive = (
        1.0
        + np.count_nonzero(
            bootstrap_total_differences
            <= 0.0
        )
    ) / (
        replicates + 1.0
    )

    formal_exposure_seconds = (
        FORMAL_BOOTSTRAP_EXPOSURE_NS
        / NANOSECONDS_PER_SECOND
    )

    return {
        "observed_total_log_score_difference": (
            observed_total_difference
        ),
        "observed_log_score_difference_per_block": float(
            np.mean(
                paired_difference
            )
        ),
        "observed_log_score_difference_per_second": (
            observed_total_difference
            / formal_exposure_seconds
        ),
        "bootstrap_total_difference_q025": float(
            bootstrap_quantiles[0]
        ),
        "bootstrap_total_difference_median": float(
            bootstrap_quantiles[1]
        ),
        "bootstrap_total_difference_q975": float(
            bootstrap_quantiles[2]
        ),
        "bootstrap_difference_per_second_q025": float(
            bootstrap_quantiles[0]
            / formal_exposure_seconds
        ),
        "bootstrap_difference_per_second_median": float(
            bootstrap_quantiles[1]
            / formal_exposure_seconds
        ),
        "bootstrap_difference_per_second_q975": float(
            bootstrap_quantiles[2]
            / formal_exposure_seconds
        ),
        "bootstrap_probability_difference_nonpositive": float(
            probability_nonpositive
        ),
        "positive_improvement_supported_at_95pct": bool(
            bootstrap_quantiles[0]
            > 0.0
        ),
        "negative_difference_supported_at_95pct": bool(
            bootstrap_quantiles[2]
            < 0.0
        ),
    }


model_block_score_lookup = {
    model_id: (
        CALIBRATION_COUNT_SCORE_BLOCKS.loc[
            CALIBRATION_COUNT_SCORE_BLOCKS[
                "model_id"
            ].eq(model_id)
        ]
        .sort_values(
            "block_index",
            kind="stable",
        )[
            "log_score_total"
        ]
        .to_numpy(
            dtype="float64"
        )
    )
    for model_id in (
        SELECTED_COUNT_BASELINE_MODEL_IDS
    )
}


bootstrap_comparison_rows: list[
    dict[str, Any]
] = []

for pair_index, (
    candidate_model_id,
    comparator_model_id,
) in enumerate(
    comparison_pairs,
    start=1,
):
    bootstrap_result = (
        bootstrap_score_difference(
            model_block_score_lookup[
                candidate_model_id
            ],
            model_block_score_lookup[
                comparator_model_id
            ],
            replicates=(
                N_BLOCK_BOOTSTRAP_REPLICATES
            ),
            seed_sequence=(
                RANDOM_SEED,
                6,
                12,
                pair_index,
            ),
        )
    )

    bootstrap_comparison_rows.append(
        {
            "candidate_model_id": (
                candidate_model_id
            ),
            "comparator_model_id": (
                comparator_model_id
            ),
            "scored_partition": (
                FORMAL_BOOTSTRAP_PARTITION
            ),
            "score_space": (
                COUNT_SCORE_SPACE
            ),
            "block_width_seconds": (
                BOOTSTRAP_BLOCK_SECONDS
            ),
            "complete_block_count": (
                FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT
            ),
            "formal_bootstrap_exposure_seconds": (
                FORMAL_BOOTSTRAP_EXPOSURE_NS
                / NANOSECONDS_PER_SECOND
            ),
            "excluded_tail_exposure_seconds": (
                FORMAL_BOOTSTRAP_EXCLUDED_TAIL_NS
                / NANOSECONDS_PER_SECOND
            ),
            "bootstrap_replicates": (
                N_BLOCK_BOOTSTRAP_REPLICATES
            ),
            **bootstrap_result,
            "calibration_used_for_model_selection": False,
            "comparison_status": (
                "POSITIVE_SUPPORTED"
                if bootstrap_result[
                    "positive_improvement_supported_at_95pct"
                ]
                else (
                    "NEGATIVE_SUPPORTED"
                    if bootstrap_result[
                        "negative_difference_supported_at_95pct"
                    ]
                    else "INCONCLUSIVE"
                )
            ),
            "status": "PASS",
        }
    )


COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON = (
    pd.DataFrame(
        bootstrap_comparison_rows
    )
    .sort_values(
        [
            "candidate_model_id",
            "comparator_model_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Primary comparator result
# ------------------------------------------------------------

PRIMARY_COUNT_BASELINE_COMPARISON = (
    COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON.loc[
        COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON[
            "candidate_model_id"
        ].eq(
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
        )
        & COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON[
            "comparator_model_id"
        ].eq(
            FORMAL_COUNT_COMPARATOR_MODEL_ID
        )
    ]
    .copy()
    .reset_index(drop=True)
)

require(
    len(
        PRIMARY_COUNT_BASELINE_COMPARISON
    )
    == 1,
    (
        "Primary simple-count-baseline comparison must contain "
        "exactly one row."
    ),
)

PRIMARY_SIMPLE_COUNT_BASELINE_IMPROVEMENT_STATUS: Final[str] = str(
    PRIMARY_COUNT_BASELINE_COMPARISON.iloc[0][
        "comparison_status"
    ]
)


# ------------------------------------------------------------
# Bootstrap coverage audit
# ------------------------------------------------------------

BOOTSTRAP_COVERAGE_AUDIT = pd.DataFrame(
    [
        {
            "scored_partition": (
                FORMAL_BOOTSTRAP_PARTITION
            ),
            "contract_duration_ns": (
                bootstrap_grid.contract_duration_ns
            ),
            "block_width_ns": (
                FORMAL_BOOTSTRAP_BLOCK_NS
            ),
            "native_grid_width_ns": (
                bootstrap_grid.grid_width_ns
            ),
            "cells_per_complete_block": (
                CELLS_PER_FORMAL_BOOTSTRAP_BLOCK
            ),
            "complete_block_count": (
                FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT
            ),
            "formal_cell_count": (
                FORMAL_BOOTSTRAP_CELL_COUNT
            ),
            "formal_exposure_ns": (
                FORMAL_BOOTSTRAP_EXPOSURE_NS
            ),
            "excluded_tail_exposure_ns": (
                FORMAL_BOOTSTRAP_EXCLUDED_TAIL_NS
            ),
            "excluded_tail_fraction": (
                FORMAL_BOOTSTRAP_EXCLUDED_TAIL_NS
                / bootstrap_grid.contract_duration_ns
            ),
            "equal_exposure_blocks_only": True,
            "partial_final_block_excluded_from_ci": True,
            "full_calibration_score_still_reported": True,
            "status": "PASS",
        }
    ]
)


# ------------------------------------------------------------
# Count-comparison gates
# ------------------------------------------------------------

expected_block_rows = (
    len(
        SELECTED_COUNT_BASELINE_MODEL_IDS
    )
    * FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT
)

COUNT_BASELINE_COMPARISON_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "selected_count_model_registry_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_BASELINE_MODEL_REGISTRY
                )
                == len(
                    SELECTED_COUNT_BASELINE_MODEL_IDS
                )
            ),
            "evidence": (
                f"selected_models="
                f"{len(SELECTED_COUNT_BASELINE_MODEL_IDS)}"
            ),
        },
        {
            "gate": "expected_count_registry_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_BASELINE_EXPECTED_COUNTS
                )
                == expected_registry_key_count
            ),
            "evidence": (
                f"expected_count_arrays="
                f"{len(COUNT_BASELINE_EXPECTED_COUNTS)}"
            ),
        },
        {
            "gate": "unified_scores_reconcile_with_source_tables",
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_SCORE_RECONCILIATION[
                    "status"
                ].eq("PASS").all()
            ),
            "evidence": (
                f"reconciliation_rows="
                f"{len(COUNT_SCORE_RECONCILIATION)}"
            ),
        },
        {
            "gate": "calibration_ranking_uses_locked_models_only",
            "severity": "BLOCKING",
            "passed": bool(
                LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
                    "locked_before_calibration"
                ].all()
                and not LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
                    "calibration_used_for_selection"
                ].any()
            ),
            "evidence": (
                "all ranked models were selected or fitted "
                "using DEVELOPMENT only"
            ),
        },
        {
            "gate": "formal_bootstrap_uses_equal_complete_blocks",
            "severity": "BLOCKING",
            "passed": bool(
                CALIBRATION_COUNT_SCORE_BLOCKS[
                    "complete_equal_exposure_block_flag"
                ].all()
            ),
            "evidence": (
                f"block_width_seconds="
                f"{BOOTSTRAP_BLOCK_SECONDS}; "
                f"complete_blocks="
                f"{FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT}"
            ),
        },
        {
            "gate": "formal_bootstrap_block_scores_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    CALIBRATION_COUNT_SCORE_BLOCKS
                )
                == expected_block_rows
            ),
            "evidence": (
                f"block_score_rows="
                f"{len(CALIBRATION_COUNT_SCORE_BLOCKS)}"
            ),
        },
        {
            "gate": "formal_bootstrap_scores_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    CALIBRATION_COUNT_SCORE_BLOCKS[
                        "log_score_total"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "all complete-block scores are finite"
            ),
        },
        {
            "gate": "formal_bootstrap_comparisons_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON
                )
                == len(
                    comparison_pairs
                )
            ),
            "evidence": (
                f"comparison_pairs="
                f"{len(comparison_pairs)}"
            ),
        },
        {
            "gate": "formal_bootstrap_replicates_complete",
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON[
                    "bootstrap_replicates"
                ].eq(
                    N_BLOCK_BOOTSTRAP_REPLICATES
                ).all()
            ),
            "evidence": (
                f"replicates_per_pair="
                f"{N_BLOCK_BOOTSTRAP_REPLICATES}"
            ),
        },
        {
            "gate": "renewal_and_mark_scores_excluded_from_count_ranking",
            "severity": "BLOCKING",
            "passed": bool(
                UNIFIED_COUNT_BASELINE_SCORE_TABLE[
                    "score_space"
                ].eq(
                    COUNT_SCORE_SPACE
                ).all()
            ),
            "evidence": (
                "renewal-duration and batch-mark likelihoods "
                "remain in separate score spaces"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    COUNT_BASELINE_COMPARISON_GATE_FRAME.loc[
        COUNT_BASELINE_COMPARISON_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    (
        "At least one unified count-baseline comparison gate "
        "failed."
    ),
)


display(COUNT_BASELINE_MODEL_REGISTRY)

display(
    UNIFIED_COUNT_BASELINE_SCORE_TABLE[
        [
            "model_id",
            "scored_partition",
            "observed_event_count",
            "predicted_event_count",
            "log_score_total",
            "log_score_per_second",
            "log_score_per_observed_event",
            "primary_simple_count_baseline_flag",
            "formal_count_comparator_flag",
            "status",
        ]
    ]
)

display(COUNT_SCORE_RECONCILIATION)

display(
    LOCKED_CALIBRATION_COUNT_BASELINE_RANKING[
        [
            "calibration_score_rank",
            "model_id",
            "model_family",
            "observed_event_count",
            "predicted_event_count",
            "log_score_total",
            "log_score_per_second",
            "log_score_improvement_over_formal_comparator",
            "primary_simple_count_baseline_flag",
            "ranking_role",
            "status",
        ]
    ]
)

display(BOOTSTRAP_COVERAGE_AUDIT)

display(
    COUNT_BASELINE_BLOCK_BOOTSTRAP_COMPARISON[
        [
            "candidate_model_id",
            "comparator_model_id",
            "complete_block_count",
            "observed_total_log_score_difference",
            "observed_log_score_difference_per_second",
            "bootstrap_total_difference_q025",
            "bootstrap_total_difference_median",
            "bootstrap_total_difference_q975",
            "bootstrap_probability_difference_nonpositive",
            "positive_improvement_supported_at_95pct",
            "comparison_status",
            "status",
        ]
    ]
)

display(PRIMARY_COUNT_BASELINE_COMPARISON)
display(COUNT_BASELINE_COMPARISON_GATE_FRAME)

print(
    "Comparable side-specific count baselines were consolidated "
    "into one native-grid score ledger."
)
print(
    f"Formal deterministic comparator: "
    f"{FORMAL_COUNT_COMPARATOR_MODEL_ID}."
)
print(
    f"Primary simple count baseline: "
    f"{PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID}."
)
print(
    "All unified scores reconcile with their originating "
    "homogeneous, deterministic, or adaptive model tables."
)
print(
    f"Locked CALIBRATION uncertainty uses "
    f"{FORMAL_COMPLETE_BOOTSTRAP_BLOCK_COUNT} complete "
    f"{BOOTSTRAP_BLOCK_SECONDS}-second paired blocks and "
    f"{N_BLOCK_BOOTSTRAP_REPLICATES} bootstrap replicates."
)
print(
    f"Primary simple-baseline improvement status: "
    f"{PRIMARY_SIMPLE_COUNT_BASELINE_IMPROVEMENT_STATUS}."
)
print(
    "Renewal-duration and batch-mark scores remain excluded from "
    "the native count-grid ranking because their score spaces are "
    "not numerically comparable."
)
print(
    "Hawkes estimation remains unauthorized."
)

,model_id,model_family,declared_roles,model_specification,fit_partition,selection_partition,calibration_used_for_selection,score_space,primary_simple_count_baseline_flag,formal_count_comparator_flag,status
0,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,INDEPENDENT_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,HOMOGENEOUS_SIDE_SPECIFIC_ANCHOR,BUY_RATE_AND_SELL_RATE_ESTIMATED_ON_DEVELOPMENT,DEVELOPMENT,DEVELOPMENT,False,BIVARIATE_SIDE_COUNT_NATIVE_1MS,False,False,PASS
1,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,DEVELOPMENT_SELECTED_DETERMINISTIC,SPECIFICATION=CONSTANT,DEVELOPMENT,DEVELOPMENT,False,BIVARIATE_SIDE_COUNT_NATIVE_1MS,False,True,PASS
2,R_SIDE_ROLLING_250MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,DEVELOPMENT_SELECTED_ROLLING,WINDOW_MS=250,DEVELOPMENT,DEVELOPMENT,False,BIVARIATE_SIDE_COUNT_NATIVE_1MS,False,False,PASS
3,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,DEVELOPMENT_SELECTED_EWMA;PRIMARY_SIMPLE_COUNT...,HALF_LIFE_MS=250,DEVELOPMENT,DEVELOPMENT,False,BIVARIATE_SIDE_COUNT_NATIVE_1MS,True,False,PASS


,model_id,scored_partition,observed_event_count,predicted_event_count,log_score_total,log_score_per_second,log_score_per_observed_event,primary_simple_count_baseline_flag,formal_count_comparator_flag,status
0,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,2493,"2,673.4791","-18,072.443",-25.09019,-7.2492753,True,False,PASS
1,R_SIDE_ROLLING_250MS_POISSON,CALIBRATION,2493,"2,699.4526","-18,095.796",-25.122611,-7.2586426,False,False,PASS
2,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,CALIBRATION,2493,"2,799.8716","-18,440.776",-25.601551,-7.3970221,False,False,PASS
3,D0_SIDE_CONSTANT_POISSON,CALIBRATION,2493,"2,799.8716","-18,440.776",-25.601551,-7.3970221,False,True,PASS
4,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,7004,"7,003.7548","-50,128.377",-27.820357,-7.1571069,True,False,PASS
5,R_SIDE_ROLLING_250MS_POISSON,DEVELOPMENT,7004,"7,004.0171","-50,304.35",-27.918019,-7.1822316,False,False,PASS
6,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,DEVELOPMENT,7004,"7,004","-50,831.392",-28.210518,-7.2574802,False,False,PASS
7,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,7004,"7,004","-50,831.392",-28.210518,-7.2574802,False,True,PASS


,model_id,scored_partition,unified_log_score,registered_log_score,score_difference,absolute_score_difference,tolerance,status
0,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,"-18,072.443","-18,072.443",-3.6379788e-12,3.6379788e-12,1e-07,PASS
1,R_SIDE_ROLLING_250MS_POISSON,CALIBRATION,"-18,095.796","-18,095.796",0,0,1e-07,PASS
2,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,CALIBRATION,"-18,440.776","-18,440.776",3.6379788e-12,3.6379788e-12,1e-07,PASS
3,D0_SIDE_CONSTANT_POISSON,CALIBRATION,"-18,440.776","-18,440.776",3.6379788e-12,3.6379788e-12,1e-07,PASS
4,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,"-50,128.377","-50,128.377",2.1827873e-11,2.1827873e-11,1e-07,PASS
5,R_SIDE_ROLLING_250MS_POISSON,DEVELOPMENT,"-50,304.35","-50,304.35",-1.4551915e-11,1.4551915e-11,1e-07,PASS
6,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,DEVELOPMENT,"-50,831.392","-50,831.392",3.6379788e-11,3.6379788e-11,1e-07,PASS
7,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,"-50,831.392","-50,831.392",3.6379788e-11,3.6379788e-11,1e-07,PASS


,calibration_score_rank,model_id,model_family,observed_event_count,predicted_event_count,log_score_total,log_score_per_second,log_score_improvement_over_formal_comparator,primary_simple_count_baseline_flag,ranking_role,status
0,1,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,2493,"2,673.4791","-18,072.443",-25.09019,368.33279,True,LOCKED_CALIBRATION_DIAGNOSTIC_ONLY,PASS
1,2,R_SIDE_ROLLING_250MS_POISSON,CAUSAL_ROLLING_WINDOW_RATE,2493,"2,699.4526","-18,095.796",-25.122611,344.98001,False,LOCKED_CALIBRATION_DIAGNOSTIC_ONLY,PASS
2,3,B1_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,INDEPENDENT_SIDE_SPECIFIC_HOMOGENEOUS_POISSON,2493,"2,799.8716","-18,440.776",-25.601551,0,False,LOCKED_CALIBRATION_DIAGNOSTIC_ONLY,PASS
3,4,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,2493,"2,799.8716","-18,440.776",-25.601551,0,False,LOCKED_CALIBRATION_DIAGNOSTIC_ONLY,PASS


,scored_partition,contract_duration_ns,block_width_ns,native_grid_width_ns,cells_per_complete_block,complete_block_count,formal_cell_count,formal_exposure_ns,excluded_tail_exposure_ns,excluded_tail_fraction,equal_exposure_blocks_only,partial_final_block_excluded_from_ci,full_calibration_score_still_reported,status
0,CALIBRATION,720299179100,10000000000,1000000,10000,72,720000,720000000000,299179100,0.00041535394,True,True,True,PASS


,candidate_model_id,comparator_model_id,complete_block_count,observed_total_log_score_difference,observed_log_score_difference_per_second,bootstrap_total_difference_q025,bootstrap_total_difference_median,bootstrap_total_difference_q975,bootstrap_probability_difference_nonpositive,positive_improvement_supported_at_95pct,comparison_status,status
0,E_SIDE_EWMA_250MS_POISSON,D0_SIDE_CONSTANT_POISSON,72,368.29927,0.51152676,213.44742,364.86477,555.67188,0.00049975012,True,POSITIVE_SUPPORTED,PASS
1,E_SIDE_EWMA_250MS_POISSON,R_SIDE_ROLLING_250MS_POISSON,72,23.421683,0.032530115,-5.1176816,24.00759,50.549826,0.050974513,False,INCONCLUSIVE,PASS
2,R_SIDE_ROLLING_250MS_POISSON,D0_SIDE_CONSTANT_POISSON,72,344.87758,0.47899664,183.82143,345.63572,543.2721,0.00049975012,True,POSITIVE_SUPPORTED,PASS


,candidate_model_id,comparator_model_id,scored_partition,score_space,block_width_seconds,complete_block_count,formal_bootstrap_exposure_seconds,excluded_tail_exposure_seconds,bootstrap_replicates,observed_total_log_score_difference,observed_log_score_difference_per_block,observed_log_score_difference_per_second,bootstrap_total_difference_q025,bootstrap_total_difference_median,bootstrap_total_difference_q975,bootstrap_difference_per_second_q025,bootstrap_difference_per_second_median,bootstrap_difference_per_second_q975,bootstrap_probability_difference_nonpositive,positive_improvement_supported_at_95pct,negative_difference_supported_at_95pct,calibration_used_for_model_selection,comparison_status,status
0,E_SIDE_EWMA_250MS_POISSON,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BIVARIATE_SIDE_COUNT_NATIVE_1MS,10,72,720,0.2991791,2000,368.29927,5.1152676,0.51152676,213.44742,364.86477,555.67188,0.29645475,0.50675662,0.7717665,0.00049975012,True,False,False,POSITIVE_SUPPORTED,PASS


,gate,severity,passed,evidence
0,selected_count_model_registry_complete,BLOCKING,True,selected_models=4
1,expected_count_registry_complete,BLOCKING,True,expected_count_arrays=16
2,unified_scores_reconcile_with_source_tables,BLOCKING,True,reconciliation_rows=8
3,calibration_ranking_uses_locked_models_only,BLOCKING,True,all ranked models were selected or fitted usin...
4,formal_bootstrap_uses_equal_complete_blocks,BLOCKING,True,block_width_seconds=10; complete_blocks=72
5,formal_bootstrap_block_scores_complete,BLOCKING,True,block_score_rows=288
6,formal_bootstrap_scores_finite,BLOCKING,True,all complete-block scores are finite
7,formal_bootstrap_comparisons_complete,BLOCKING,True,comparison_pairs=3
8,formal_bootstrap_replicates_complete,BLOCKING,True,replicates_per_pair=2000
9,renewal_and_mark_scores_excluded_from_count_ra...,BLOCKING,True,renewal-duration and batch-mark likelihoods re...


Comparable side-specific count baselines were consolidated into one native-grid score ledger.
Formal deterministic comparator: D0_SIDE_CONSTANT_POISSON.
Primary simple count baseline: E_SIDE_EWMA_250MS_POISSON.
All unified scores reconcile with their originating homogeneous, deterministic, or adaptive model tables.
Locked CALIBRATION uncertainty uses 72 complete 10-second paired blocks and 2000 bootstrap replicates.
Primary simple-baseline improvement status: POSITIVE_SUPPORTED.
Renewal-duration and batch-mark scores remain excluded from the native count-grid ranking because their score spaces are not numerically comparable.
Hawkes estimation remains unauthorized.


In [16]:
# ============================================================
# Count dispersion and serial-dependence diagnostics
# ============================================================

COUNT_DISPERSION_WIDTHS_MS: Final[
    tuple[int, ...]
] = (
    10,
    25,
    50,
    100,
    250,
    500,
    1_000,
    2_000,
    5_000,
    10_000,
    30_000,
)

COUNT_SERIAL_WIDTHS_MS: Final[
    tuple[int, ...]
] = (
    100,
    250,
    500,
    1_000,
    2_000,
    5_000,
)

COUNT_SERIAL_LAGS: Final[
    tuple[int, ...]
] = (
    1,
    2,
    5,
    10,
    20,
)

COUNT_DIAGNOSTIC_PROCESSES: Final[
    tuple[str, ...]
] = (
    "POOLED",
    "BUY",
    "SELL",
)

RESIDUAL_DIAGNOSTIC_MODEL_IDS: Final[
    tuple[str, ...]
] = ordered_unique(
    (
        FORMAL_COUNT_COMPARATOR_MODEL_ID,
        PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
    )
)

SERIAL_DIAGNOSTIC_P_VALUE_THRESHOLD: Final[float] = (
    0.01
)


# ------------------------------------------------------------
# Exact complete-bin aggregation
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class CompleteBinAggregation:
    event_partition: str
    width_ms: int
    width_ns: int
    cells_per_bin: int
    complete_bin_count: int
    included_cell_count: int
    included_exposure_ns: int
    excluded_tail_exposure_ns: int
    excluded_native_cell_count: int


def complete_bin_aggregation_contract(
    *,
    partition_name: str,
    width_ms: int,
) -> CompleteBinAggregation:
    """
    Return the exact complete-bin aggregation contract.

    Only complete equal-exposure bins are retained. Any remaining full
    native cells and the final partial native cell are kept outside the
    diagnostic sample and recorded as excluded tail exposure.
    """
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized count-diagnostic partition: "
            f"{partition_name}"
        ),
    )
    require(
        width_ms
        in COUNT_DISPERSION_WIDTHS_MS,
        (
            f"Count-diagnostic width {width_ms} ms is outside "
            "the frozen width grid."
        ),
    )

    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    width_ns = (
        width_ms
        * NANOSECONDS_PER_MILLISECOND
    )

    require(
        width_ns
        % grid.grid_width_ns
        == 0,
        (
            f"{partition_name} diagnostic width {width_ms} ms "
            "is not an integer multiple of the native grid."
        ),
    )

    cells_per_bin = (
        width_ns
        // grid.grid_width_ns
    )

    complete_bin_count = (
        grid.contract_duration_ns
        // width_ns
    )

    require(
        complete_bin_count >= 2,
        (
            f"{partition_name} {width_ms} ms diagnostics "
            "require at least two complete bins."
        ),
    )

    included_cell_count = (
        complete_bin_count
        * cells_per_bin
    )

    included_exposure_ns = (
        complete_bin_count
        * width_ns
    )

    excluded_tail_exposure_ns = (
        grid.contract_duration_ns
        - included_exposure_ns
    )

    excluded_native_cell_count = (
        grid.cell_count
        - included_cell_count
    )

    require(
        included_cell_count
        <= grid.cell_count,
        (
            f"{partition_name} {width_ms} ms aggregation "
            "requires more native cells than available."
        ),
    )
    require(
        included_exposure_ns
        + excluded_tail_exposure_ns
        == grid.contract_duration_ns,
        (
            f"{partition_name} {width_ms} ms aggregation does "
            "not conserve exact contract exposure."
        ),
    )
    require(
        0
        <= excluded_tail_exposure_ns
        < width_ns,
        (
            f"{partition_name} {width_ms} ms excluded tail "
            "must be shorter than one complete diagnostic bin."
        ),
    )
    require(
        excluded_native_cell_count >= 0,
        (
            f"{partition_name} {width_ms} ms excluded native "
            "cell count is negative."
        ),
    )

    return CompleteBinAggregation(
        event_partition=partition_name,
        width_ms=width_ms,
        width_ns=width_ns,
        cells_per_bin=cells_per_bin,
        complete_bin_count=(
            complete_bin_count
        ),
        included_cell_count=(
            included_cell_count
        ),
        included_exposure_ns=(
            included_exposure_ns
        ),
        excluded_tail_exposure_ns=(
            excluded_tail_exposure_ns
        ),
        excluded_native_cell_count=(
            excluded_native_cell_count
        ),
    )


COUNT_AGGREGATION_CONTRACTS: Final[
    Mapping[
        tuple[str, int],
        CompleteBinAggregation,
    ]
] = {
    (
        partition_name,
        width_ms,
    ): complete_bin_aggregation_contract(
        partition_name=(
            partition_name
        ),
        width_ms=width_ms,
    )
    for partition_name in ANALYTICAL_PARTITIONS
    for width_ms in COUNT_DISPERSION_WIDTHS_MS
}


def observed_native_process_counts(
    *,
    partition_name: str,
    process_name: str,
) -> np.ndarray:
    """Return one authoritative native-grid observed-count array."""
    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    if process_name == "POOLED":
        values = grid.pooled_counts
    elif process_name == "BUY":
        values = grid.buy_counts
    elif process_name == "SELL":
        values = grid.sell_counts
    else:
        raise ValueError(
            f"Unsupported count process: {process_name}"
        )

    return values


def expected_native_process_counts(
    *,
    model_id: str,
    partition_name: str,
    process_name: str,
) -> np.ndarray:
    """Return one authoritative native-grid expected-count array."""
    if process_name == "BUY":
        values = (
            COUNT_BASELINE_EXPECTED_COUNTS[
                (
                    model_id,
                    partition_name,
                    "BUY",
                )
            ]
        )

    elif process_name == "SELL":
        values = (
            COUNT_BASELINE_EXPECTED_COUNTS[
                (
                    model_id,
                    partition_name,
                    "SELL",
                )
            ]
        )

    elif process_name == "POOLED":
        values = (
            COUNT_BASELINE_EXPECTED_COUNTS[
                (
                    model_id,
                    partition_name,
                    "BUY",
                )
            ]
            + COUNT_BASELINE_EXPECTED_COUNTS[
                (
                    model_id,
                    partition_name,
                    "SELL",
                )
            ]
        )

    else:
        raise ValueError(
            f"Unsupported count process: {process_name}"
        )

    return np.asarray(
        values,
        dtype="float64",
    )


def aggregate_complete_native_bins(
    native_values: np.ndarray,
    *,
    contract: CompleteBinAggregation,
    output_dtype: str,
) -> np.ndarray:
    """Aggregate a native series into complete equal-width bins."""
    values = np.asarray(
        native_values,
    )

    require(
        values.ndim == 1,
        "Native aggregation input must be one-dimensional.",
    )
    require(
        values.size
        >= contract.included_cell_count,
        (
            "Native aggregation input is shorter than the "
            "declared included-cell count."
        ),
    )

    included_values = values[
        :contract.included_cell_count
    ]

    reshaped = included_values.reshape(
        contract.complete_bin_count,
        contract.cells_per_bin,
    )

    aggregated = reshaped.sum(
        axis=1,
        dtype=output_dtype,
    )

    require(
        aggregated.shape
        == (
            contract.complete_bin_count,
        ),
        "Aggregated count series has an invalid shape.",
    )

    return aggregated


# ------------------------------------------------------------
# Aggregated observed and expected-count registries
# ------------------------------------------------------------

DIAGNOSTIC_AGGREGATED_OBSERVED_COUNTS: dict[
    tuple[str, int, str],
    np.ndarray,
] = {}

DIAGNOSTIC_AGGREGATED_EXPECTED_COUNTS: dict[
    tuple[str, str, int, str],
    np.ndarray,
] = {}

DIAGNOSTIC_AGGREGATED_PEARSON_RESIDUALS: dict[
    tuple[str, str, int, str],
    np.ndarray,
] = {}

DIAGNOSTIC_AGGREGATED_DEVIANCE_RESIDUALS: dict[
    tuple[str, str, int, str],
    np.ndarray,
] = {}


aggregation_coverage_rows: list[
    dict[str, Any]
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    for width_ms in (
        COUNT_DISPERSION_WIDTHS_MS
    ):
        contract = (
            COUNT_AGGREGATION_CONTRACTS[
                (
                    partition_name,
                    width_ms,
                )
            ]
        )

        process_included_event_counts: dict[
            str,
            int,
        ] = {}

        process_excluded_event_counts: dict[
            str,
            int,
        ] = {}

        for process_name in (
            COUNT_DIAGNOSTIC_PROCESSES
        ):
            native_observed = (
                observed_native_process_counts(
                    partition_name=(
                        partition_name
                    ),
                    process_name=(
                        process_name
                    ),
                )
            )

            aggregated_observed = (
                aggregate_complete_native_bins(
                    native_observed,
                    contract=contract,
                    output_dtype="int64",
                )
                .astype(
                    "int64",
                    copy=False,
                )
            )

            included_event_count = int(
                aggregated_observed.sum(
                    dtype="int64"
                )
            )

            full_event_count = int(
                native_observed.sum(
                    dtype="int64"
                )
            )

            excluded_event_count = (
                full_event_count
                - included_event_count
            )

            require(
                excluded_event_count >= 0,
                (
                    f"{partition_name} {width_ms} ms "
                    f"{process_name} excluded event count is "
                    "negative."
                ),
            )

            process_included_event_counts[
                process_name
            ] = included_event_count

            process_excluded_event_counts[
                process_name
            ] = excluded_event_count

            frozen_observed = (
                aggregated_observed.copy()
            )

            frozen_observed.setflags(
                write=False
            )

            DIAGNOSTIC_AGGREGATED_OBSERVED_COUNTS[
                (
                    partition_name,
                    width_ms,
                    process_name,
                )
            ] = frozen_observed

            for model_id in (
                RESIDUAL_DIAGNOSTIC_MODEL_IDS
            ):
                native_expected = (
                    expected_native_process_counts(
                        model_id=model_id,
                        partition_name=(
                            partition_name
                        ),
                        process_name=(
                            process_name
                        ),
                    )
                )

                aggregated_expected = (
                    aggregate_complete_native_bins(
                        native_expected,
                        contract=contract,
                        output_dtype="float64",
                    )
                    .astype(
                        "float64",
                        copy=False,
                    )
                )

                require(
                    np.isfinite(
                        aggregated_expected
                    ).all(),
                    (
                        f"{model_id} {partition_name} "
                        f"{width_ms} ms {process_name} expected "
                        "counts contain nonfinite values."
                    ),
                )

                require(
                    np.all(
                        aggregated_expected > 0.0
                    ),
                    (
                        f"{model_id} {partition_name} "
                        f"{width_ms} ms {process_name} expected "
                        "counts are not strictly positive."
                    ),
                )

                pearson_residuals = (
                    (
                        aggregated_observed.astype(
                            "float64"
                        )
                        - aggregated_expected
                    )
                    / np.sqrt(
                        aggregated_expected
                    )
                )

                deviance_residuals = (
                    poisson_deviance_residuals(
                        aggregated_observed,
                        aggregated_expected,
                    )
                )

                require(
                    np.isfinite(
                        pearson_residuals
                    ).all(),
                    (
                        f"{model_id} {partition_name} "
                        f"{width_ms} ms {process_name} Pearson "
                        "residuals contain nonfinite values."
                    ),
                )

                require(
                    np.isfinite(
                        deviance_residuals
                    ).all(),
                    (
                        f"{model_id} {partition_name} "
                        f"{width_ms} ms {process_name} deviance "
                        "residuals contain nonfinite values."
                    ),
                )

                for (
                    registry,
                    key,
                    array,
                ) in (
                    (
                        DIAGNOSTIC_AGGREGATED_EXPECTED_COUNTS,
                        (
                            model_id,
                            partition_name,
                            width_ms,
                            process_name,
                        ),
                        aggregated_expected,
                    ),
                    (
                        DIAGNOSTIC_AGGREGATED_PEARSON_RESIDUALS,
                        (
                            model_id,
                            partition_name,
                            width_ms,
                            process_name,
                        ),
                        pearson_residuals,
                    ),
                    (
                        DIAGNOSTIC_AGGREGATED_DEVIANCE_RESIDUALS,
                        (
                            model_id,
                            partition_name,
                            width_ms,
                            process_name,
                        ),
                        deviance_residuals,
                    ),
                ):
                    frozen_array = (
                        np.asarray(
                            array,
                            dtype="float64",
                        )
                        .copy()
                    )

                    frozen_array.setflags(
                        write=False
                    )

                    registry[
                        key
                    ] = frozen_array

        require(
            process_included_event_counts[
                "BUY"
            ]
            + process_included_event_counts[
                "SELL"
            ]
            == process_included_event_counts[
                "POOLED"
            ],
            (
                f"{partition_name} {width_ms} ms included BUY "
                "and SELL counts do not conserve pooled events."
            ),
        )

        require(
            process_excluded_event_counts[
                "BUY"
            ]
            + process_excluded_event_counts[
                "SELL"
            ]
            == process_excluded_event_counts[
                "POOLED"
            ],
            (
                f"{partition_name} {width_ms} ms excluded BUY "
                "and SELL counts do not conserve pooled events."
            ),
        )

        aggregation_coverage_rows.append(
            {
                "event_partition": (
                    partition_name
                ),
                "width_ms": (
                    width_ms
                ),
                "cells_per_bin": (
                    contract.cells_per_bin
                ),
                "complete_bin_count": (
                    contract.complete_bin_count
                ),
                "included_cell_count": (
                    contract.included_cell_count
                ),
                "included_exposure_ns": (
                    contract.included_exposure_ns
                ),
                "excluded_tail_exposure_ns": (
                    contract.excluded_tail_exposure_ns
                ),
                "excluded_tail_fraction": (
                    contract.excluded_tail_exposure_ns
                    / grid.contract_duration_ns
                ),
                "excluded_native_cell_count": (
                    contract.excluded_native_cell_count
                ),
                "included_pooled_event_count": (
                    process_included_event_counts[
                        "POOLED"
                    ]
                ),
                "excluded_pooled_event_count": (
                    process_excluded_event_counts[
                        "POOLED"
                    ]
                ),
                "full_pooled_event_count": int(
                    grid.pooled_counts.sum(
                        dtype="int64"
                    )
                ),
                "exact_exposure_conserved": (
                    contract.included_exposure_ns
                    + contract.excluded_tail_exposure_ns
                    == grid.contract_duration_ns
                ),
                "event_count_conserved": (
                    process_included_event_counts[
                        "POOLED"
                    ]
                    + process_excluded_event_counts[
                        "POOLED"
                    ]
                    == int(
                        grid.pooled_counts.sum(
                            dtype="int64"
                        )
                    )
                ),
                "status": "PASS",
            }
        )


COUNT_AGGREGATION_COVERAGE_AUDIT = (
    pd.DataFrame(
        aggregation_coverage_rows
    )
    .sort_values(
        [
            "event_partition",
            "width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Observed count-dispersion summaries
# ------------------------------------------------------------

observed_dispersion_rows: list[
    dict[str, Any]
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    for width_ms in (
        COUNT_DISPERSION_WIDTHS_MS
    ):
        contract = (
            COUNT_AGGREGATION_CONTRACTS[
                (
                    partition_name,
                    width_ms,
                )
            ]
        )

        for process_name in (
            COUNT_DIAGNOSTIC_PROCESSES
        ):
            counts = (
                DIAGNOSTIC_AGGREGATED_OBSERVED_COUNTS[
                    (
                        partition_name,
                        width_ms,
                        process_name,
                    )
                ]
                .astype(
                    "float64"
                )
            )

            count_mean = float(
                np.mean(counts)
            )

            count_variance = float(
                np.var(
                    counts,
                    ddof=1,
                )
            )

            require(
                count_mean > 0.0,
                (
                    f"{partition_name} {width_ms} ms "
                    f"{process_name} mean count is not positive."
                ),
            )

            fano_factor = (
                count_variance
                / count_mean
            )

            observed_zero_fraction = float(
                np.mean(
                    counts == 0.0
                )
            )

            homogeneous_poisson_zero_probability = (
                math.exp(
                    -count_mean
                )
            )

            observed_dispersion_rows.append(
                {
                    "event_partition": (
                        partition_name
                    ),
                    "process_name": (
                        process_name
                    ),
                    "width_ms": (
                        width_ms
                    ),
                    "complete_bin_count": (
                        contract.complete_bin_count
                    ),
                    "included_exposure_seconds": (
                        contract.included_exposure_ns
                        / NANOSECONDS_PER_SECOND
                    ),
                    "included_event_count": int(
                        counts.sum(
                            dtype="float64"
                        )
                    ),
                    "mean_count": (
                        count_mean
                    ),
                    "sample_variance": (
                        count_variance
                    ),
                    "fano_factor": (
                        fano_factor
                    ),
                    "poisson_reference_fano": 1.0,
                    "fano_minus_poisson_reference": (
                        fano_factor - 1.0
                    ),
                    "coefficient_of_variation": (
                        math.sqrt(
                            count_variance
                        )
                        / count_mean
                    ),
                    "observed_zero_fraction": (
                        observed_zero_fraction
                    ),
                    "homogeneous_poisson_zero_probability": (
                        homogeneous_poisson_zero_probability
                    ),
                    "zero_fraction_minus_homogeneous_poisson": (
                        observed_zero_fraction
                        - homogeneous_poisson_zero_probability
                    ),
                    "maximum_count": int(
                        np.max(counts)
                    ),
                    "formal_stationary_poisson_test_reported": False,
                    "diagnostic_role": (
                        "DESCRIPTIVE_DISPERSION_ONLY"
                    ),
                    "status": "PASS",
                }
            )


COUNT_OBSERVED_DISPERSION_SUMMARY = (
    pd.DataFrame(
        observed_dispersion_rows
    )
    .sort_values(
        [
            "event_partition",
            "process_name",
            "width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Model-relative residual dispersion
# ------------------------------------------------------------

model_residual_dispersion_rows: list[
    dict[str, Any]
] = []


for model_id in (
    RESIDUAL_DIAGNOSTIC_MODEL_IDS
):
    for partition_name in (
        ANALYTICAL_PARTITIONS
    ):
        for width_ms in (
            COUNT_DISPERSION_WIDTHS_MS
        ):
            contract = (
                COUNT_AGGREGATION_CONTRACTS[
                    (
                        partition_name,
                        width_ms,
                    )
                ]
            )

            for process_name in (
                COUNT_DIAGNOSTIC_PROCESSES
            ):
                observed = (
                    DIAGNOSTIC_AGGREGATED_OBSERVED_COUNTS[
                        (
                            partition_name,
                            width_ms,
                            process_name,
                        )
                    ]
                    .astype(
                        "float64"
                    )
                )

                expected = (
                    DIAGNOSTIC_AGGREGATED_EXPECTED_COUNTS[
                        (
                            model_id,
                            partition_name,
                            width_ms,
                            process_name,
                        )
                    ]
                )

                pearson_residuals = (
                    DIAGNOSTIC_AGGREGATED_PEARSON_RESIDUALS[
                        (
                            model_id,
                            partition_name,
                            width_ms,
                            process_name,
                        )
                    ]
                )

                deviance_residuals = (
                    DIAGNOSTIC_AGGREGATED_DEVIANCE_RESIDUALS[
                        (
                            model_id,
                            partition_name,
                            width_ms,
                            process_name,
                        )
                    ]
                )

                raw_residuals = (
                    observed - expected
                )

                pearson_statistic = float(
                    np.square(
                        pearson_residuals
                    ).sum(
                        dtype="float64"
                    )
                )

                fixed_forecast_pearson_ratio = (
                    pearson_statistic
                    / observed.size
                )

                observed_zero_fraction = float(
                    np.mean(
                        observed == 0.0
                    )
                )

                mean_predicted_zero_probability = float(
                    np.mean(
                        np.exp(
                            -expected
                        )
                    )
                )

                model_residual_dispersion_rows.append(
                    {
                        "model_id": (
                            model_id
                        ),
                        "model_family": (
                            count_model_family(
                                model_id
                            )
                        ),
                        "event_partition": (
                            partition_name
                        ),
                        "process_name": (
                            process_name
                        ),
                        "width_ms": (
                            width_ms
                        ),
                        "complete_bin_count": (
                            contract.complete_bin_count
                        ),
                        "included_exposure_seconds": (
                            contract.included_exposure_ns
                            / NANOSECONDS_PER_SECOND
                        ),
                        "observed_event_count": int(
                            observed.sum(
                                dtype="float64"
                            )
                        ),
                        "predicted_event_count": float(
                            expected.sum(
                                dtype="float64"
                            )
                        ),
                        "observed_minus_predicted": float(
                            raw_residuals.sum(
                                dtype="float64"
                            )
                        ),
                        "mean_raw_residual": float(
                            np.mean(
                                raw_residuals
                            )
                        ),
                        "root_mean_squared_error": float(
                            np.sqrt(
                                np.mean(
                                    np.square(
                                        raw_residuals
                                    )
                                )
                            )
                        ),
                        "mean_absolute_error": float(
                            np.mean(
                                np.abs(
                                    raw_residuals
                                )
                            )
                        ),
                        "pearson_statistic": (
                            pearson_statistic
                        ),
                        "fixed_forecast_pearson_ratio": (
                            fixed_forecast_pearson_ratio
                        ),
                        "pearson_residual_mean": float(
                            np.mean(
                                pearson_residuals
                            )
                        ),
                        "pearson_residual_sample_variance": float(
                            np.var(
                                pearson_residuals,
                                ddof=1,
                            )
                        ),
                        "deviance_residual_mean": float(
                            np.mean(
                                deviance_residuals
                            )
                        ),
                        "deviance_residual_sample_variance": float(
                            np.var(
                                deviance_residuals,
                                ddof=1,
                            )
                        ),
                        "observed_zero_fraction": (
                            observed_zero_fraction
                        ),
                        "mean_predicted_zero_probability": (
                            mean_predicted_zero_probability
                        ),
                        "zero_fraction_residual": (
                            observed_zero_fraction
                            - mean_predicted_zero_probability
                        ),
                        "maximum_absolute_pearson_residual": float(
                            np.max(
                                np.abs(
                                    pearson_residuals
                                )
                            )
                        ),
                        "locked_forecast_flag": True,
                        "formal_chi_square_p_value_reported": False,
                        "status": "PASS",
                    }
                )


COUNT_MODEL_RESIDUAL_DISPERSION = (
    pd.DataFrame(
        model_residual_dispersion_rows
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "process_name",
            "width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Serial-correlation helpers
# ------------------------------------------------------------

def sample_autocorrelation(
    values: np.ndarray,
    *,
    maximum_lag: int,
) -> np.ndarray:
    """Return sample autocorrelations from lag zero through maximum_lag."""
    array = np.asarray(
        values,
        dtype="float64",
    )

    require(
        array.ndim == 1,
        "Autocorrelation input must be one-dimensional.",
    )
    require(
        np.isfinite(
            array
        ).all(),
        "Autocorrelation input contains nonfinite values.",
    )
    require(
        array.size
        > maximum_lag + 1,
        (
            "Autocorrelation input is too short for the "
            "requested maximum lag."
        ),
    )
    require(
        maximum_lag >= 1,
        "Maximum autocorrelation lag must be positive.",
    )

    centered = (
        array - np.mean(array)
    )

    denominator = float(
        np.dot(
            centered,
            centered,
        )
    )

    require(
        denominator > 0.0,
        "Autocorrelation input has zero variance.",
    )

    autocorrelations = np.empty(
        maximum_lag + 1,
        dtype="float64",
    )

    autocorrelations[0] = 1.0

    for lag in range(
        1,
        maximum_lag + 1,
    ):
        autocorrelations[lag] = float(
            np.dot(
                centered[:-lag],
                centered[lag:],
            )
            / denominator
        )

    require(
        np.isfinite(
            autocorrelations
        ).all(),
        "Autocorrelation result contains nonfinite values.",
    )

    return autocorrelations


def ljung_box_summary(
    autocorrelations: np.ndarray,
    *,
    observation_count: int,
    maximum_lag: int,
) -> tuple[float, float]:
    """Return the Ljung–Box Q statistic and chi-square reference p-value."""
    require(
        observation_count
        > maximum_lag + 1,
        (
            "Ljung–Box observation count is too small for the "
            "requested horizon."
        ),
    )

    acf_values = np.asarray(
        autocorrelations,
        dtype="float64",
    )

    require(
        acf_values.size
        >= maximum_lag + 1,
        "Ljung–Box autocorrelation vector is too short.",
    )

    lags = np.arange(
        1,
        maximum_lag + 1,
        dtype="float64",
    )

    q_statistic = float(
        observation_count
        * (
            observation_count + 2.0
        )
        * np.sum(
            np.square(
                acf_values[
                    1:
                    maximum_lag + 1
                ]
            )
            / (
                observation_count - lags
            ),
            dtype="float64",
        )
    )

    p_value = float(
        stats.chi2.sf(
            q_statistic,
            df=maximum_lag,
        )
    )

    require(
        np.isfinite(
            q_statistic
        ),
        "Ljung–Box statistic is nonfinite.",
    )
    require(
        np.isfinite(
            p_value
        ),
        "Ljung–Box p-value is nonfinite.",
    )

    return (
        q_statistic,
        p_value,
    )


# ------------------------------------------------------------
# Raw-count and residual serial dependence
# ------------------------------------------------------------

raw_serial_rows: list[
    dict[str, Any]
] = []

residual_serial_rows: list[
    dict[str, Any]
] = []

ljung_box_rows: list[
    dict[str, Any]
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    for width_ms in (
        COUNT_SERIAL_WIDTHS_MS
    ):
        for process_name in (
            COUNT_DIAGNOSTIC_PROCESSES
        ):
            raw_counts = (
                DIAGNOSTIC_AGGREGATED_OBSERVED_COUNTS[
                    (
                        partition_name,
                        width_ms,
                        process_name,
                    )
                ]
                .astype(
                    "float64"
                )
            )

            applicable_lags = tuple(
                lag
                for lag in COUNT_SERIAL_LAGS
                if lag
                < raw_counts.size - 1
            )

            require(
                applicable_lags,
                (
                    f"{partition_name} {width_ms} ms "
                    f"{process_name} has no applicable serial "
                    "diagnostic lag."
                ),
            )

            maximum_lag = max(
                applicable_lags
            )

            raw_acf = (
                sample_autocorrelation(
                    raw_counts,
                    maximum_lag=(
                        maximum_lag
                    ),
                )
            )

            raw_null_band = (
                1.96
                / math.sqrt(
                    raw_counts.size
                )
            )

            raw_q_statistic, raw_q_p_value = (
                ljung_box_summary(
                    raw_acf,
                    observation_count=(
                        raw_counts.size
                    ),
                    maximum_lag=(
                        maximum_lag
                    ),
                )
            )

            for lag in (
                applicable_lags
            ):
                raw_serial_rows.append(
                    {
                        "event_partition": (
                            partition_name
                        ),
                        "series_type": (
                            "OBSERVED_COUNT"
                        ),
                        "process_name": (
                            process_name
                        ),
                        "width_ms": (
                            width_ms
                        ),
                        "observation_count": int(
                            raw_counts.size
                        ),
                        "lag": lag,
                        "lag_duration_ms": (
                            lag * width_ms
                        ),
                        "autocorrelation": float(
                            raw_acf[
                                lag
                            ]
                        ),
                        "white_noise_95pct_band": (
                            raw_null_band
                        ),
                        "absolute_acf_exceeds_95pct_band": (
                            abs(
                                raw_acf[
                                    lag
                                ]
                            )
                            > raw_null_band
                        ),
                        "status": "PASS",
                    }
                )

            ljung_box_rows.append(
                {
                    "model_id": (
                        "OBSERVED_COUNT"
                    ),
                    "event_partition": (
                        partition_name
                    ),
                    "series_type": (
                        "OBSERVED_COUNT"
                    ),
                    "process_name": (
                        process_name
                    ),
                    "width_ms": (
                        width_ms
                    ),
                    "observation_count": int(
                        raw_counts.size
                    ),
                    "maximum_lag": (
                        maximum_lag
                    ),
                    "maximum_lag_duration_ms": (
                        maximum_lag
                        * width_ms
                    ),
                    "ljung_box_q_statistic": (
                        raw_q_statistic
                    ),
                    "ljung_box_reference_p_value": (
                        raw_q_p_value
                    ),
                    "p_value_threshold": (
                        SERIAL_DIAGNOSTIC_P_VALUE_THRESHOLD
                    ),
                    "serial_dependence_flag": (
                        raw_q_p_value
                        < SERIAL_DIAGNOSTIC_P_VALUE_THRESHOLD
                    ),
                    "reference_role": (
                        "DIAGNOSTIC_CHI_SQUARE_REFERENCE"
                    ),
                    "status": "PASS",
                }
            )

            for model_id in (
                RESIDUAL_DIAGNOSTIC_MODEL_IDS
            ):
                pearson_residuals = (
                    DIAGNOSTIC_AGGREGATED_PEARSON_RESIDUALS[
                        (
                            model_id,
                            partition_name,
                            width_ms,
                            process_name,
                        )
                    ]
                )

                residual_acf = (
                    sample_autocorrelation(
                        pearson_residuals,
                        maximum_lag=(
                            maximum_lag
                        ),
                    )
                )

                residual_null_band = (
                    1.96
                    / math.sqrt(
                        pearson_residuals.size
                    )
                )

                (
                    residual_q_statistic,
                    residual_q_p_value,
                ) = ljung_box_summary(
                    residual_acf,
                    observation_count=(
                        pearson_residuals.size
                    ),
                    maximum_lag=(
                        maximum_lag
                    ),
                )

                for lag in (
                    applicable_lags
                ):
                    residual_serial_rows.append(
                        {
                            "model_id": (
                                model_id
                            ),
                            "model_family": (
                                count_model_family(
                                    model_id
                                )
                            ),
                            "event_partition": (
                                partition_name
                            ),
                            "series_type": (
                                "PEARSON_RESIDUAL"
                            ),
                            "process_name": (
                                process_name
                            ),
                            "width_ms": (
                                width_ms
                            ),
                            "observation_count": int(
                                pearson_residuals.size
                            ),
                            "lag": lag,
                            "lag_duration_ms": (
                                lag * width_ms
                            ),
                            "autocorrelation": float(
                                residual_acf[
                                    lag
                                ]
                            ),
                            "white_noise_95pct_band": (
                                residual_null_band
                            ),
                            "absolute_acf_exceeds_95pct_band": (
                                abs(
                                    residual_acf[
                                        lag
                                    ]
                                )
                                > residual_null_band
                            ),
                            "primary_simple_count_baseline_flag": (
                                model_id
                                == PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
                            ),
                            "formal_count_comparator_flag": (
                                model_id
                                == FORMAL_COUNT_COMPARATOR_MODEL_ID
                            ),
                            "status": "PASS",
                        }
                    )

                ljung_box_rows.append(
                    {
                        "model_id": (
                            model_id
                        ),
                        "event_partition": (
                            partition_name
                        ),
                        "series_type": (
                            "PEARSON_RESIDUAL"
                        ),
                        "process_name": (
                            process_name
                        ),
                        "width_ms": (
                            width_ms
                        ),
                        "observation_count": int(
                            pearson_residuals.size
                        ),
                        "maximum_lag": (
                            maximum_lag
                        ),
                        "maximum_lag_duration_ms": (
                            maximum_lag
                            * width_ms
                        ),
                        "ljung_box_q_statistic": (
                            residual_q_statistic
                        ),
                        "ljung_box_reference_p_value": (
                            residual_q_p_value
                        ),
                        "p_value_threshold": (
                            SERIAL_DIAGNOSTIC_P_VALUE_THRESHOLD
                        ),
                        "serial_dependence_flag": (
                            residual_q_p_value
                            < SERIAL_DIAGNOSTIC_P_VALUE_THRESHOLD
                        ),
                        "reference_role": (
                            "DIAGNOSTIC_CHI_SQUARE_REFERENCE"
                        ),
                        "status": "PASS",
                    }
                )


COUNT_RAW_SERIAL_DEPENDENCE = (
    pd.DataFrame(
        raw_serial_rows
    )
    .sort_values(
        [
            "event_partition",
            "process_name",
            "width_ms",
            "lag",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

COUNT_RESIDUAL_SERIAL_DEPENDENCE = (
    pd.DataFrame(
        residual_serial_rows
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "process_name",
            "width_ms",
            "lag",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

COUNT_LJUNG_BOX_SUMMARY = (
    pd.DataFrame(
        ljung_box_rows
    )
    .sort_values(
        [
            "event_partition",
            "series_type",
            "model_id",
            "process_name",
            "width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Residual-dependence evidence summary
# ------------------------------------------------------------

COUNT_RESIDUAL_SERIAL_EVIDENCE = (
    COUNT_LJUNG_BOX_SUMMARY.loc[
        COUNT_LJUNG_BOX_SUMMARY[
            "series_type"
        ].eq(
            "PEARSON_RESIDUAL"
        )
    ]
    .groupby(
        [
            "model_id",
            "event_partition",
            "process_name",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        evaluated_width_count=(
            "width_ms",
            "nunique",
        ),
        significant_width_count=(
            "serial_dependence_flag",
            "sum",
        ),
        minimum_reference_p_value=(
            "ljung_box_reference_p_value",
            "min",
        ),
        maximum_reference_p_value=(
            "ljung_box_reference_p_value",
            "max",
        ),
        median_reference_p_value=(
            "ljung_box_reference_p_value",
            "median",
        ),
    )
    .reset_index()
)

COUNT_RESIDUAL_SERIAL_EVIDENCE[
    "significant_width_fraction"
] = (
    COUNT_RESIDUAL_SERIAL_EVIDENCE[
        "significant_width_count"
    ]
    / COUNT_RESIDUAL_SERIAL_EVIDENCE[
        "evaluated_width_count"
    ]
)

COUNT_RESIDUAL_SERIAL_EVIDENCE[
    "repeated_serial_dependence_flag"
] = (
    COUNT_RESIDUAL_SERIAL_EVIDENCE[
        "significant_width_count"
    ]
    >= 2
)

COUNT_RESIDUAL_SERIAL_EVIDENCE[
    "diagnostic_only"
] = True

COUNT_RESIDUAL_SERIAL_EVIDENCE[
    "status"
] = "PASS"


PRIMARY_BASELINE_CALIBRATION_SERIAL_EVIDENCE = (
    COUNT_RESIDUAL_SERIAL_EVIDENCE.loc[
        COUNT_RESIDUAL_SERIAL_EVIDENCE[
            "model_id"
        ].eq(
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
        )
        & COUNT_RESIDUAL_SERIAL_EVIDENCE[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

require(
    len(
        PRIMARY_BASELINE_CALIBRATION_SERIAL_EVIDENCE
    )
    == len(
        COUNT_DIAGNOSTIC_PROCESSES
    ),
    (
        "Primary CALIBRATION residual serial-evidence table "
        "does not contain pooled, BUY, and SELL processes."
    ),
)


# ------------------------------------------------------------
# Diagnostic gates
# ------------------------------------------------------------

expected_coverage_rows = (
    len(
        ANALYTICAL_PARTITIONS
    )
    * len(
        COUNT_DISPERSION_WIDTHS_MS
    )
)

expected_observed_dispersion_rows = (
    expected_coverage_rows
    * len(
        COUNT_DIAGNOSTIC_PROCESSES
    )
)

expected_model_dispersion_rows = (
    expected_observed_dispersion_rows
    * len(
        RESIDUAL_DIAGNOSTIC_MODEL_IDS
    )
)

COUNT_DISPERSION_AND_SERIAL_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "complete_bin_coverage_rows_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_AGGREGATION_COVERAGE_AUDIT
                )
                == expected_coverage_rows
            ),
            "evidence": (
                f"coverage_rows="
                f"{len(COUNT_AGGREGATION_COVERAGE_AUDIT)}"
            ),
        },
        {
            "gate": "exact_exposure_conserved_at_every_width",
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_AGGREGATION_COVERAGE_AUDIT[
                    "exact_exposure_conserved"
                ].all()
            ),
            "evidence": (
                "included complete-bin exposure plus excluded "
                "tail equals each registered contract duration"
            ),
        },
        {
            "gate": "event_counts_conserved_at_every_width",
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_AGGREGATION_COVERAGE_AUDIT[
                    "event_count_conserved"
                ].all()
            ),
            "evidence": (
                "included-bin and excluded-tail events conserve "
                "full pooled counts"
            ),
        },
        {
            "gate": "observed_dispersion_table_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_OBSERVED_DISPERSION_SUMMARY
                )
                == expected_observed_dispersion_rows
            ),
            "evidence": (
                f"dispersion_rows="
                f"{len(COUNT_OBSERVED_DISPERSION_SUMMARY)}"
            ),
        },
        {
            "gate": "model_residual_dispersion_table_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_MODEL_RESIDUAL_DISPERSION
                )
                == expected_model_dispersion_rows
            ),
            "evidence": (
                f"residual_dispersion_rows="
                f"{len(COUNT_MODEL_RESIDUAL_DISPERSION)}"
            ),
        },
        {
            "gate": "all_dispersion_diagnostics_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    COUNT_OBSERVED_DISPERSION_SUMMARY[
                        [
                            "mean_count",
                            "sample_variance",
                            "fano_factor",
                            "observed_zero_fraction",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
                and np.isfinite(
                    COUNT_MODEL_RESIDUAL_DISPERSION[
                        [
                            "fixed_forecast_pearson_ratio",
                            "pearson_residual_mean",
                            "pearson_residual_sample_variance",
                            "deviance_residual_mean",
                            "deviance_residual_sample_variance",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "observed and model-relative dispersion "
                "statistics are finite"
            ),
        },
        {
            "gate": "serial_diagnostic_tables_nonempty",
            "severity": "BLOCKING",
            "passed": (
                not COUNT_RAW_SERIAL_DEPENDENCE.empty
                and not COUNT_RESIDUAL_SERIAL_DEPENDENCE.empty
                and not COUNT_LJUNG_BOX_SUMMARY.empty
            ),
            "evidence": (
                f"raw_acf_rows="
                f"{len(COUNT_RAW_SERIAL_DEPENDENCE)}; "
                f"residual_acf_rows="
                f"{len(COUNT_RESIDUAL_SERIAL_DEPENDENCE)}"
            ),
        },
        {
            "gate": "all_serial_diagnostics_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    COUNT_RAW_SERIAL_DEPENDENCE[
                        "autocorrelation"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
                and np.isfinite(
                    COUNT_RESIDUAL_SERIAL_DEPENDENCE[
                        "autocorrelation"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
                and np.isfinite(
                    COUNT_LJUNG_BOX_SUMMARY[
                        [
                            "ljung_box_q_statistic",
                            "ljung_box_reference_p_value",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "all ACF and Ljung–Box diagnostic values are "
                "finite"
            ),
        },
        {
            "gate": "only_locked_selected_models_diagnosed",
            "severity": "BLOCKING",
            "passed": bool(
                set(
                    COUNT_MODEL_RESIDUAL_DISPERSION[
                        "model_id"
                    ]
                )
                == set(
                    RESIDUAL_DIAGNOSTIC_MODEL_IDS
                )
            ),
            "evidence": (
                "formal comparator and primary simple baseline "
                "only"
            ),
        },
        {
            "gate": "no_stationarity_claim_from_raw_fano",
            "severity": "BLOCKING",
            "passed": bool(
                not COUNT_OBSERVED_DISPERSION_SUMMARY[
                    "formal_stationary_poisson_test_reported"
                ].any()
            ),
            "evidence": (
                "raw Fano factors are descriptive and do not "
                "assume stationarity"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    COUNT_DISPERSION_AND_SERIAL_GATE_FRAME.loc[
        COUNT_DISPERSION_AND_SERIAL_GATE_FRAME[
            "severity"
        ].eq("BLOCKING"),
        "passed",
    ].all(),
    (
        "At least one count-dispersion or serial-dependence "
        "diagnostic gate failed."
    ),
)


display(
    COUNT_AGGREGATION_COVERAGE_AUDIT[
        [
            "event_partition",
            "width_ms",
            "complete_bin_count",
            "included_exposure_ns",
            "excluded_tail_exposure_ns",
            "excluded_tail_fraction",
            "included_pooled_event_count",
            "excluded_pooled_event_count",
            "exact_exposure_conserved",
            "event_count_conserved",
            "status",
        ]
    ]
)

display(
    COUNT_OBSERVED_DISPERSION_SUMMARY.loc[
        COUNT_OBSERVED_DISPERSION_SUMMARY[
            "width_ms"
        ].isin(
            {
                100,
                250,
                1_000,
                5_000,
                10_000,
            }
        ),
        [
            "event_partition",
            "process_name",
            "width_ms",
            "complete_bin_count",
            "mean_count",
            "sample_variance",
            "fano_factor",
            "observed_zero_fraction",
            "homogeneous_poisson_zero_probability",
            "maximum_count",
            "diagnostic_role",
            "status",
        ],
    ]
)

display(
    COUNT_MODEL_RESIDUAL_DISPERSION.loc[
        COUNT_MODEL_RESIDUAL_DISPERSION[
            "width_ms"
        ].isin(
            {
                100,
                250,
                1_000,
                5_000,
            }
        ),
        [
            "model_id",
            "event_partition",
            "process_name",
            "width_ms",
            "observed_event_count",
            "predicted_event_count",
            "fixed_forecast_pearson_ratio",
            "pearson_residual_mean",
            "pearson_residual_sample_variance",
            "zero_fraction_residual",
            "maximum_absolute_pearson_residual",
            "status",
        ],
    ]
)

display(
    COUNT_RESIDUAL_SERIAL_DEPENDENCE.loc[
        COUNT_RESIDUAL_SERIAL_DEPENDENCE[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
        & COUNT_RESIDUAL_SERIAL_DEPENDENCE[
            "width_ms"
        ].isin(
            {
                100,
                250,
                1_000,
            }
        )
        & COUNT_RESIDUAL_SERIAL_DEPENDENCE[
            "lag"
        ].isin(
            {
                1,
                5,
                10,
            }
        ),
        [
            "model_id",
            "event_partition",
            "process_name",
            "width_ms",
            "lag",
            "lag_duration_ms",
            "autocorrelation",
            "white_noise_95pct_band",
            "absolute_acf_exceeds_95pct_band",
            "status",
        ],
    ]
)

display(
    COUNT_LJUNG_BOX_SUMMARY.loc[
        COUNT_LJUNG_BOX_SUMMARY[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
        & COUNT_LJUNG_BOX_SUMMARY[
            "series_type"
        ].eq(
            "PEARSON_RESIDUAL"
        ),
        [
            "model_id",
            "process_name",
            "width_ms",
            "observation_count",
            "maximum_lag",
            "maximum_lag_duration_ms",
            "ljung_box_q_statistic",
            "ljung_box_reference_p_value",
            "serial_dependence_flag",
            "reference_role",
            "status",
        ],
    ]
)

display(
    PRIMARY_BASELINE_CALIBRATION_SERIAL_EVIDENCE
)

display(
    COUNT_DISPERSION_AND_SERIAL_GATE_FRAME
)

print(
    "Observed count dispersion was evaluated across exact "
    "complete bins from 10 milliseconds through 30 seconds."
)
print(
    "Every aggregation conserves registered exposure and event "
    "counts after explicitly recording the excluded right-edge "
    "tail."
)
print(
    "Raw Fano factors are descriptive only; no stationary "
    "Poisson inference is attached to them."
)
print(
    "Pearson and deviance residual dispersion was evaluated for "
    "the formal deterministic comparator and the primary simple "
    "count baseline."
)
print(
    "Residual autocorrelations and Ljung–Box reference statistics "
    "were computed at multiple time scales without using "
    "VALIDATION or ENGINEERING_HOLDOUT content."
)
print(
    "These diagnostics record remaining serial dependence but do "
    "not yet authorize Hawkes estimation."
)

,event_partition,width_ms,complete_bin_count,included_exposure_ns,excluded_tail_exposure_ns,excluded_tail_fraction,included_pooled_event_count,excluded_pooled_event_count,exact_exposure_conserved,event_count_conserved,status
0,CALIBRATION,10,72029,720290000000,9179100,1.2743455e-05,2493,0,True,True,PASS
1,CALIBRATION,25,28811,720275000000,24179100,3.3568135e-05,2493,0,True,True,PASS
2,CALIBRATION,50,14405,720250000000,49179100,6.8275935e-05,2493,0,True,True,PASS
3,CALIBRATION,100,7202,720200000000,99179100,0.00013769154,2493,0,True,True,PASS
4,CALIBRATION,250,2881,720250000000,49179100,6.8275935e-05,2493,0,True,True,PASS
5,CALIBRATION,500,1440,720000000000,299179100,0.00041535394,2493,0,True,True,PASS
6,CALIBRATION,1000,720,720000000000,299179100,0.00041535394,2493,0,True,True,PASS
7,CALIBRATION,2000,360,720000000000,299179100,0.00041535394,2493,0,True,True,PASS
8,CALIBRATION,5000,144,720000000000,299179100,0.00041535394,2493,0,True,True,PASS
9,CALIBRATION,10000,72,720000000000,299179100,0.00041535394,2493,0,True,True,PASS


,event_partition,process_name,width_ms,complete_bin_count,mean_count,sample_variance,fano_factor,observed_zero_fraction,homogeneous_poisson_zero_probability,maximum_count,diagnostic_role,status
3,CALIBRATION,BUY,100,7202,0.17078589,0.52408462,3.0686646,0.87947792,0.84300205,24,DESCRIPTIVE_DISPERSION_ONLY,PASS
4,CALIBRATION,BUY,250,2881,0.42693509,1.3926631,3.262002,0.72960778,0.6525059,26,DESCRIPTIVE_DISPERSION_ONLY,PASS
6,CALIBRATION,BUY,1000,720,1.7083333,7.466968,4.3709081,0.30555556,0.18116749,29,DESCRIPTIVE_DISPERSION_ONLY,PASS
8,CALIBRATION,BUY,5000,144,8.5416667,55.83042,6.5362442,0,0.00019516471,51,DESCRIPTIVE_DISPERSION_ONLY,PASS
9,CALIBRATION,BUY,10000,72,17.083333,123.23239,7.2136036,0,3.8089266e-08,56,DESCRIPTIVE_DISPERSION_ONLY,PASS
14,CALIBRATION,POOLED,100,7202,0.34615385,0.81072607,2.3420975,0.75020828,0.70740365,24,DESCRIPTIVE_DISPERSION_ONLY,PASS
15,CALIBRATION,POOLED,250,2881,0.86532454,2.1832451,2.5230362,0.49114891,0.42091493,26,DESCRIPTIVE_DISPERSION_ONLY,PASS
17,CALIBRATION,POOLED,1000,720,3.4625,10.783015,3.114228,0.048611111,0.031351286,30,DESCRIPTIVE_DISPERSION_ONLY,PASS
19,CALIBRATION,POOLED,5000,144,17.3125,79.097465,4.5688066,0,3.0288431e-08,59,DESCRIPTIVE_DISPERSION_ONLY,PASS
20,CALIBRATION,POOLED,10000,72,34.625,165.36444,4.7758682,0,9.1738908e-16,76,DESCRIPTIVE_DISPERSION_ONLY,PASS


,model_id,event_partition,process_name,width_ms,observed_event_count,predicted_event_count,fixed_forecast_pearson_ratio,pearson_residual_mean,pearson_residual_sample_variance,zero_fraction_residual,maximum_absolute_pearson_residual,status
3,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,100,1230,"1,364.5696",2.7675011,-0.042926209,2.7660425,0.052081148,54.701314,PASS
4,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,250,1230,"1,364.6643",2.9437014,-0.067915361,2.9401095,0.10689961,37.089157,PASS
6,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,1000,1230,"1,364.1906",3.9538168,-0.13539994,3.9409572,0.15519351,19.691681,PASS
8,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,5000,1230,"1,364.1906",5.9440369,-0.30276347,5.8932968,-7.685837e-05,13.491758,PASS
14,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,100,2493,"2,799.4861",2.0900555,-0.068256657,2.0856861,0.072277118,37.871031,PASS
15,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,250,2493,"2,799.6804",2.2575405,-0.10798419,2.2466597,0.1127377,25.38911,PASS
17,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,1000,2493,"2,798.7086",2.8165811,-0.2153588,2.7740546,0.028106292,13.244714,PASS
19,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,5000,2493,"2,798.7086",4.2733815,-0.48155692,4.0697466,-3.6247639e-09,8.9744603,PASS
25,D0_SIDE_CONSTANT_POISSON,CALIBRATION,SELL,100,1263,"1,434.9165",1.330819,-0.053478303,1.3281434,0.030826142,30.918349,PASS
26,D0_SIDE_CONSTANT_POISSON,CALIBRATION,SELL,250,1263,"1,435.0161",1.4143289,-0.084599746,1.4076604,0.057360584,24.798678,PASS


,model_id,event_partition,process_name,width_ms,lag,lag_duration_ms,autocorrelation,white_noise_95pct_band,absolute_acf_exceeds_95pct_band,status
0,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,100,1,100,0.085826684,0.023095614,True,PASS
2,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,100,5,500,0.033330613,0.023095614,True,PASS
3,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,100,10,1000,0.036471676,0.023095614,True,PASS
5,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,250,1,250,0.13530421,0.036516105,True,PASS
7,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,250,5,1250,0.024456826,0.036516105,False,PASS
8,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,250,10,2500,0.020772664,0.036516105,False,PASS
15,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,1000,1,1000,0.12734155,0.073044887,True,PASS
17,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,1000,5,5000,0.017274285,0.073044887,False,PASS
18,D0_SIDE_CONSTANT_POISSON,CALIBRATION,BUY,1000,10,10000,0.024234252,0.073044887,False,PASS
30,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,100,1,100,0.09864934,0.023095614,True,PASS


,model_id,process_name,width_ms,observation_count,maximum_lag,maximum_lag_duration_ms,ljung_box_q_statistic,ljung_box_reference_p_value,serial_dependence_flag,reference_role,status
18,D0_SIDE_CONSTANT_POISSON,BUY,100,7202,20,2000,190.16642,9.8113475e-30,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
19,D0_SIDE_CONSTANT_POISSON,BUY,250,2881,20,5000,159.63627,8.826271e-24,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
20,D0_SIDE_CONSTANT_POISSON,BUY,500,1440,20,10000,68.814011,2.8412401e-07,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
21,D0_SIDE_CONSTANT_POISSON,BUY,1000,720,20,20000,44.123115,0.0014488319,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
22,D0_SIDE_CONSTANT_POISSON,BUY,2000,360,20,40000,41.031871,0.0036902871,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
23,D0_SIDE_CONSTANT_POISSON,BUY,5000,144,20,100000,22.889047,0.29427538,False,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
24,D0_SIDE_CONSTANT_POISSON,POOLED,100,7202,20,2000,141.86193,2.2427388e-20,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
25,D0_SIDE_CONSTANT_POISSON,POOLED,250,2881,20,5000,106.38317,8.925187e-14,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
26,D0_SIDE_CONSTANT_POISSON,POOLED,500,1440,20,10000,55.605066,3.3327754e-05,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS
27,D0_SIDE_CONSTANT_POISSON,POOLED,1000,720,20,20000,41.703209,0.0030225952,True,DIAGNOSTIC_CHI_SQUARE_REFERENCE,PASS


,model_id,event_partition,process_name,evaluated_width_count,significant_width_count,minimum_reference_p_value,maximum_reference_p_value,median_reference_p_value,significant_width_fraction,repeated_serial_dependence_flag,diagnostic_only,status
0,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,BUY,6,3,4.3986165e-05,0.49314947,0.012567222,0.5,True,True,PASS
1,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,POOLED,6,4,9.0498561e-07,0.98806856,0.00027141726,0.66666667,True,True,PASS
2,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,SELL,6,4,1.3125497e-10,0.45322636,2.4605399e-06,0.66666667,True,True,PASS


,gate,severity,passed,evidence
0,complete_bin_coverage_rows_complete,BLOCKING,True,coverage_rows=22
1,exact_exposure_conserved_at_every_width,BLOCKING,True,included complete-bin exposure plus excluded t...
2,event_counts_conserved_at_every_width,BLOCKING,True,included-bin and excluded-tail events conserve...
3,observed_dispersion_table_complete,BLOCKING,True,dispersion_rows=66
4,model_residual_dispersion_table_complete,BLOCKING,True,residual_dispersion_rows=132
5,all_dispersion_diagnostics_finite,BLOCKING,True,observed and model-relative dispersion statist...
6,serial_diagnostic_tables_nonempty,BLOCKING,True,raw_acf_rows=180; residual_acf_rows=360
7,all_serial_diagnostics_finite,BLOCKING,True,all ACF and Ljung–Box diagnostic values are fi...
8,only_locked_selected_models_diagnosed,BLOCKING,True,formal comparator and primary simple baseline ...
9,no_stationarity_claim_from_raw_fano,BLOCKING,True,raw Fano factors are descriptive and do not as...


Observed count dispersion was evaluated across exact complete bins from 10 milliseconds through 30 seconds.
Every aggregation conserves registered exposure and event counts after explicitly recording the excluded right-edge tail.
Raw Fano factors are descriptive only; no stationary Poisson inference is attached to them.
Pearson and deviance residual dispersion was evaluated for the formal deterministic comparator and the primary simple count baseline.
Residual autocorrelations and Ljung–Box reference statistics were computed at multiple time scales without using VALIDATION or ENGINEERING_HOLDOUT content.
These diagnostics record remaining serial dependence but do not yet authorize Hawkes estimation.


In [17]:
# ============================================================
# Cross-side correlation and conditional-response diagnostics
# ============================================================

CROSS_DEPENDENCE_WIDTHS_MS: Final[
    tuple[int, ...]
] = (
    100,
    250,
    500,
    1_000,
)

CROSS_DEPENDENCE_LAGS: Final[
    tuple[int, ...]
] = (
    -20,
    -10,
    -5,
    -2,
    -1,
    0,
    1,
    2,
    5,
    10,
    20,
)

CONDITIONAL_RESPONSE_BANDS_MS: Final[
    tuple[tuple[int, int], ...]
] = (
    (0, 10),
    (10, 25),
    (25, 50),
    (50, 100),
    (100, 250),
    (250, 500),
    (500, 1_000),
    (1_000, 2_000),
    (2_000, 5_000),
)

CROSS_DEPENDENCE_MODEL_IDS: Final[
    tuple[str, ...]
] = ordered_unique(
    (
        FORMAL_COUNT_COMPARATOR_MODEL_ID,
        PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
    )
)

MINIMUM_CONDITIONAL_SOURCE_BATCHES: Final[int] = 100

CONDITIONAL_RESPONSE_ROLE: Final[str] = (
    "OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY"
)


# ------------------------------------------------------------
# Cross-correlation helpers
# ------------------------------------------------------------

def lagged_pair(
    leading_series: np.ndarray,
    following_series: np.ndarray,
    *,
    lag: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Align two series under the convention:

        lag > 0: BUY leads SELL
        lag < 0: SELL leads BUY
        lag = 0: contemporaneous association
    """
    leading = np.asarray(
        leading_series,
        dtype="float64",
    )

    following = np.asarray(
        following_series,
        dtype="float64",
    )

    require(
        leading.ndim == 1,
        "Leading cross-correlation series must be one-dimensional.",
    )
    require(
        following.ndim == 1,
        "Following cross-correlation series must be one-dimensional.",
    )
    require(
        leading.shape == following.shape,
        "Cross-correlation series differ in shape.",
    )
    require(
        np.isfinite(leading).all()
        and np.isfinite(following).all(),
        "Cross-correlation input contains nonfinite values.",
    )
    require(
        abs(lag) < leading.size - 1,
        (
            f"Cross-correlation lag {lag} is too large for "
            f"{leading.size} observations."
        ),
    )

    if lag > 0:
        aligned_buy = leading[:-lag]
        aligned_sell = following[lag:]

    elif lag < 0:
        offset = -lag
        aligned_buy = leading[offset:]
        aligned_sell = following[:-offset]

    else:
        aligned_buy = leading
        aligned_sell = following

    require(
        aligned_buy.size == aligned_sell.size,
        "Aligned cross-correlation arrays differ in length.",
    )
    require(
        aligned_buy.size >= 3,
        "Cross-correlation requires at least three aligned observations.",
    )

    return (
        aligned_buy,
        aligned_sell,
    )


def sample_cross_correlation(
    buy_series: np.ndarray,
    sell_series: np.ndarray,
    *,
    lag: int,
) -> tuple[float, int]:
    """Return lagged BUY/SELL sample correlation and pair count."""
    aligned_buy, aligned_sell = lagged_pair(
        buy_series,
        sell_series,
        lag=lag,
    )

    buy_standard_deviation = float(
        np.std(
            aligned_buy,
            ddof=1,
        )
    )

    sell_standard_deviation = float(
        np.std(
            aligned_sell,
            ddof=1,
        )
    )

    require(
        buy_standard_deviation > 0.0,
        "Aligned BUY series has zero variance.",
    )
    require(
        sell_standard_deviation > 0.0,
        "Aligned SELL series has zero variance.",
    )

    correlation = float(
        np.corrcoef(
            aligned_buy,
            aligned_sell,
        )[0, 1]
    )

    require(
        np.isfinite(correlation),
        "Cross-correlation result is nonfinite.",
    )

    return (
        correlation,
        int(aligned_buy.size),
    )


# ------------------------------------------------------------
# Raw and residual cross-correlation tables
# ------------------------------------------------------------

cross_correlation_rows: list[
    dict[str, Any]
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    for width_ms in CROSS_DEPENDENCE_WIDTHS_MS:
        observed_buy = (
            DIAGNOSTIC_AGGREGATED_OBSERVED_COUNTS[
                (
                    partition_name,
                    width_ms,
                    "BUY",
                )
            ]
            .astype(
                "float64"
            )
        )

        observed_sell = (
            DIAGNOSTIC_AGGREGATED_OBSERVED_COUNTS[
                (
                    partition_name,
                    width_ms,
                    "SELL",
                )
            ]
            .astype(
                "float64"
            )
        )

        series_registry: list[
            tuple[
                str,
                str,
                np.ndarray,
                np.ndarray,
            ]
        ] = [
            (
                "OBSERVED_COUNT",
                "OBSERVED_COUNT",
                observed_buy,
                observed_sell,
            )
        ]

        for model_id in CROSS_DEPENDENCE_MODEL_IDS:
            residual_buy = (
                DIAGNOSTIC_AGGREGATED_PEARSON_RESIDUALS[
                    (
                        model_id,
                        partition_name,
                        width_ms,
                        "BUY",
                    )
                ]
            )

            residual_sell = (
                DIAGNOSTIC_AGGREGATED_PEARSON_RESIDUALS[
                    (
                        model_id,
                        partition_name,
                        width_ms,
                        "SELL",
                    )
                ]
            )

            series_registry.append(
                (
                    model_id,
                    "PEARSON_RESIDUAL",
                    residual_buy,
                    residual_sell,
                )
            )

        for (
            model_id,
            series_type,
            buy_series,
            sell_series,
        ) in series_registry:
            for lag in CROSS_DEPENDENCE_LAGS:
                (
                    correlation,
                    aligned_pair_count,
                ) = sample_cross_correlation(
                    buy_series,
                    sell_series,
                    lag=lag,
                )

                white_noise_band = (
                    1.96
                    / math.sqrt(
                        aligned_pair_count
                    )
                )

                if lag > 0:
                    direction = (
                        "BUY_LEADS_SELL"
                    )
                elif lag < 0:
                    direction = (
                        "SELL_LEADS_BUY"
                    )
                else:
                    direction = (
                        "CONTEMPORANEOUS"
                    )

                cross_correlation_rows.append(
                    {
                        "model_id": model_id,
                        "event_partition": (
                            partition_name
                        ),
                        "series_type": (
                            series_type
                        ),
                        "width_ms": (
                            width_ms
                        ),
                        "lag": lag,
                        "absolute_lag": abs(
                            lag
                        ),
                        "lag_duration_ms": (
                            lag * width_ms
                        ),
                        "absolute_lag_duration_ms": (
                            abs(lag)
                            * width_ms
                        ),
                        "direction": (
                            direction
                        ),
                        "aligned_pair_count": (
                            aligned_pair_count
                        ),
                        "cross_correlation": (
                            correlation
                        ),
                        "white_noise_95pct_band": (
                            white_noise_band
                        ),
                        "absolute_correlation_exceeds_95pct_band": (
                            abs(correlation)
                            > white_noise_band
                        ),
                        "positive_correlation_flag": (
                            correlation > 0.0
                        ),
                        "negative_correlation_flag": (
                            correlation < 0.0
                        ),
                        "diagnostic_reference_only": True,
                        "status": "PASS",
                    }
                )


COUNT_CROSS_CORRELATION = (
    pd.DataFrame(
        cross_correlation_rows
    )
    .sort_values(
        [
            "event_partition",
            "series_type",
            "model_id",
            "width_ms",
            "lag",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Cross-correlation evidence summaries
# ------------------------------------------------------------

nonzero_lag_cross_correlations = (
    COUNT_CROSS_CORRELATION.loc[
        COUNT_CROSS_CORRELATION[
            "lag"
        ].ne(0)
    ]
    .copy()
)

COUNT_CROSS_CORRELATION_EVIDENCE = (
    nonzero_lag_cross_correlations.groupby(
        [
            "model_id",
            "event_partition",
            "series_type",
            "direction",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        evaluated_correlation_count=(
            "cross_correlation",
            "size",
        ),
        significant_correlation_count=(
            "absolute_correlation_exceeds_95pct_band",
            "sum",
        ),
        positive_significant_count=(
            "positive_correlation_flag",
            lambda values: int(
                (
                    values.to_numpy(
                        dtype=bool
                    )
                    & nonzero_lag_cross_correlations.loc[
                        values.index,
                        "absolute_correlation_exceeds_95pct_band",
                    ].to_numpy(
                        dtype=bool
                    )
                ).sum()
            ),
        ),
        maximum_absolute_correlation=(
            "cross_correlation",
            lambda values: float(
                np.max(
                    np.abs(
                        values.to_numpy(
                            dtype="float64"
                        )
                    )
                )
            ),
        ),
        strongest_positive_correlation=(
            "cross_correlation",
            "max",
        ),
        strongest_negative_correlation=(
            "cross_correlation",
            "min",
        ),
    )
    .reset_index()
)

COUNT_CROSS_CORRELATION_EVIDENCE[
    "significant_correlation_fraction"
] = (
    COUNT_CROSS_CORRELATION_EVIDENCE[
        "significant_correlation_count"
    ]
    / COUNT_CROSS_CORRELATION_EVIDENCE[
        "evaluated_correlation_count"
    ]
)

COUNT_CROSS_CORRELATION_EVIDENCE[
    "repeated_cross_dependence_flag"
] = (
    COUNT_CROSS_CORRELATION_EVIDENCE[
        "significant_correlation_count"
    ]
    >= 2
)

COUNT_CROSS_CORRELATION_EVIDENCE[
    "status"
] = "PASS"


PRIMARY_BASELINE_CALIBRATION_CROSS_EVIDENCE = (
    COUNT_CROSS_CORRELATION_EVIDENCE.loc[
        COUNT_CROSS_CORRELATION_EVIDENCE[
            "model_id"
        ].eq(
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
        )
        & COUNT_CROSS_CORRELATION_EVIDENCE[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
        & COUNT_CROSS_CORRELATION_EVIDENCE[
            "series_type"
        ].eq(
            "PEARSON_RESIDUAL"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

require(
    set(
        PRIMARY_BASELINE_CALIBRATION_CROSS_EVIDENCE[
            "direction"
        ]
    )
    == {
        "BUY_LEADS_SELL",
        "SELL_LEADS_BUY",
    },
    (
        "Primary CALIBRATION residual cross-dependence summary "
        "does not contain both lead directions."
    ),
)


# ------------------------------------------------------------
# Piecewise-constant primary-intensity integration
# ------------------------------------------------------------

def cumulative_piecewise_expected_count(
    *,
    grid: NativeCountGrid,
    intensity_per_second: np.ndarray,
    evaluation_times_ns: np.ndarray,
) -> np.ndarray:
    """
    Integrate a native-cell piecewise-constant intensity from the
    contract start through each requested timestamp.
    """
    intensity = np.asarray(
        intensity_per_second,
        dtype="float64",
    )

    evaluation_times = np.asarray(
        evaluation_times_ns,
        dtype="int64",
    )

    require(
        intensity.shape
        == (
            grid.cell_count,
        ),
        (
            f"{grid.event_partition} intensity shape differs "
            "from the native grid."
        ),
    )
    require(
        np.isfinite(intensity).all()
        and np.all(
            intensity > 0.0
        ),
        "Piecewise intensity must be finite and strictly positive.",
    )
    require(
        evaluation_times.ndim == 1,
        "Intensity evaluation times must be one-dimensional.",
    )
    require(
        np.all(
            evaluation_times
            >= grid.contract_start_ns
        ),
        "Intensity integration requested before contract start.",
    )
    require(
        np.all(
            evaluation_times
            <= grid.contract_end_exclusive_ns
        ),
        "Intensity integration requested after contract end.",
    )

    cell_expected_counts = (
        intensity
        * (
            grid.exposure_ns.astype(
                "float64"
            )
            / NANOSECONDS_PER_SECOND
        )
    )

    cumulative_cell_expected_counts = np.empty(
        grid.cell_count + 1,
        dtype="float64",
    )

    cumulative_cell_expected_counts[0] = 0.0

    np.cumsum(
        cell_expected_counts,
        dtype="float64",
        out=(
            cumulative_cell_expected_counts[
                1:
            ]
        ),
    )

    offsets_ns = (
        evaluation_times
        - grid.contract_start_ns
    )

    full_cell_count = (
        offsets_ns
        // grid.grid_width_ns
    ).astype("int64")

    within_cell_ns = (
        offsets_ns
        % grid.grid_width_ns
    ).astype("int64")

    require(
        np.all(
            full_cell_count >= 0
        ),
        "Piecewise integration produced a negative cell index.",
    )
    require(
        np.all(
            full_cell_count
            <= grid.cell_count
        ),
        "Piecewise integration produced an excessive cell index.",
    )

    integrated = (
        cumulative_cell_expected_counts[
            full_cell_count
        ]
        .copy()
    )

    partial_mask = (
        full_cell_count
        < grid.cell_count
    )

    if partial_mask.any():
        partial_indices = (
            full_cell_count[
                partial_mask
            ]
        )

        usable_partial_exposure_ns = np.minimum(
            within_cell_ns[
                partial_mask
            ],
            grid.exposure_ns[
                partial_indices
            ],
        )

        integrated[
            partial_mask
        ] += (
            intensity[
                partial_indices
            ]
            * usable_partial_exposure_ns.astype(
                "float64"
            )
            / NANOSECONDS_PER_SECOND
        )

    require(
        np.isfinite(
            integrated
        ).all(),
        "Integrated expected counts contain nonfinite values.",
    )

    return integrated


def expected_count_over_intervals(
    *,
    grid: NativeCountGrid,
    intensity_per_second: np.ndarray,
    interval_start_ns: np.ndarray,
    interval_end_ns: np.ndarray,
) -> np.ndarray:
    """Integrate expected event counts over exact half-open intervals."""
    starts = np.asarray(
        interval_start_ns,
        dtype="int64",
    )

    ends = np.asarray(
        interval_end_ns,
        dtype="int64",
    )

    require(
        starts.shape == ends.shape,
        "Expected-count interval endpoints differ in shape.",
    )
    require(
        np.all(
            ends > starts
        ),
        "Expected-count intervals must have positive duration.",
    )

    cumulative_at_start = (
        cumulative_piecewise_expected_count(
            grid=grid,
            intensity_per_second=(
                intensity_per_second
            ),
            evaluation_times_ns=starts,
        )
    )

    cumulative_at_end = (
        cumulative_piecewise_expected_count(
            grid=grid,
            intensity_per_second=(
                intensity_per_second
            ),
            evaluation_times_ns=ends,
        )
    )

    interval_expected_counts = (
        cumulative_at_end
        - cumulative_at_start
    )

    require(
        np.isfinite(
            interval_expected_counts
        ).all(),
        "Interval expected counts contain nonfinite values.",
    )
    require(
        np.all(
            interval_expected_counts > 0.0
        ),
        "Interval expected counts must be strictly positive.",
    )

    return interval_expected_counts


# ------------------------------------------------------------
# Conditional-response construction
# ------------------------------------------------------------

def partition_side_batch_frame(
    *,
    partition_name: str,
    side: str,
) -> pd.DataFrame:
    """Return exact-time batches containing one requested side."""
    require(
        partition_name
        in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized conditional-response partition: "
            f"{partition_name}"
        ),
    )
    require(
        side in {"BUY", "SELL"},
        f"Unsupported conditional-response side: {side}",
    )

    side_count_column = (
        "buy_event_count"
        if side == "BUY"
        else "sell_event_count"
    )

    frame = (
        PRIMARY_SCORING_BATCHES_ANALYTICAL.loc[
            PRIMARY_SCORING_BATCHES_ANALYTICAL[
                "event_partition"
            ]
            .astype("string")
            .str.strip()
            .str.upper()
            .eq(partition_name)
            & PRIMARY_SCORING_BATCHES_ANALYTICAL[
                side_count_column
            ]
            .astype("int64")
            .gt(0),
            [
                "primary_event_batch_id",
                "partition_batch_index",
                "event_partition",
                "event_time_ns",
                "batch_event_count",
                "buy_event_count",
                "sell_event_count",
                "mixed_side_batch_flag",
            ],
        ]
        .copy()
        .sort_values(
            "partition_batch_index",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    frame[
        "event_time_ns"
    ] = parse_exact_int64(
        frame[
            "event_time_ns"
        ],
        label=(
            f"{partition_name}.{side}."
            "conditional_event_time_ns"
        ),
    )

    require(
        not frame.empty,
        (
            f"{partition_name} contains no {side} source "
            "batches."
        ),
    )
    require(
        np.all(
            np.diff(
                frame[
                    "event_time_ns"
                ].to_numpy(
                    dtype="int64"
                )
            )
            > 0
        ),
        (
            f"{partition_name} {side} source-batch times are "
            "not strictly increasing."
        ),
    )

    return frame


CONDITIONAL_SIDE_BATCH_FRAMES: Final[
    Mapping[tuple[str, str], pd.DataFrame]
] = {
    (
        partition_name,
        side,
    ): partition_side_batch_frame(
        partition_name=(
            partition_name
        ),
        side=side,
    )
    for partition_name in ANALYTICAL_PARTITIONS
    for side in ("BUY", "SELL")
}


conditional_response_rows: list[
    dict[str, Any]
] = []

conditional_source_replay_frames: list[
    pd.DataFrame
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    partition_contract = (
        partition_window_lookup.loc[
            partition_name
        ]
    )

    contract_end_exclusive_ns = int(
        partition_contract[
            "contract_end_exclusive_ns"
        ]
    )

    contract_duration_seconds = (
        grid.contract_duration_ns
        / NANOSECONDS_PER_SECOND
    )

    for source_side in (
        "BUY",
        "SELL",
    ):
        source_frame = (
            CONDITIONAL_SIDE_BATCH_FRAMES[
                (
                    partition_name,
                    source_side,
                )
            ]
        )

        source_side_count_column = (
            "buy_event_count"
            if source_side == "BUY"
            else "sell_event_count"
        )

        for target_side in (
            "BUY",
            "SELL",
        ):
            target_frame = (
                CONDITIONAL_SIDE_BATCH_FRAMES[
                    (
                        partition_name,
                        target_side,
                    )
                ]
            )

            target_side_count_column = (
                "buy_event_count"
                if target_side == "BUY"
                else "sell_event_count"
            )

            target_times_ns = (
                target_frame[
                    "event_time_ns"
                ]
                .to_numpy(
                    dtype="int64"
                )
            )

            target_weights = (
                target_frame[
                    target_side_count_column
                ]
                .astype("int64")
                .to_numpy()
            )

            target_cumulative_weights = np.empty(
                target_weights.size + 1,
                dtype="int64",
            )

            target_cumulative_weights[0] = 0

            np.cumsum(
                target_weights,
                dtype="int64",
                out=(
                    target_cumulative_weights[
                        1:
                    ]
                ),
            )

            target_full_partition_event_count = int(
                target_weights.sum(
                    dtype="int64"
                )
            )

            target_partition_rate_per_second = (
                target_full_partition_event_count
                / contract_duration_seconds
            )

            primary_target_intensity = (
                SELECTED_ADAPTIVE_INTENSITIES[
                    (
                        PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
                        partition_name,
                        target_side,
                    )
                ]
            )

            for (
                band_start_ms,
                band_end_ms,
            ) in CONDITIONAL_RESPONSE_BANDS_MS:
                require(
                    band_end_ms
                    > band_start_ms
                    >= 0,
                    "Conditional response band is invalid.",
                )

                band_start_ns = (
                    band_start_ms
                    * NANOSECONDS_PER_MILLISECOND
                )

                band_end_ns = (
                    band_end_ms
                    * NANOSECONDS_PER_MILLISECOND
                )

                band_width_ns = (
                    band_end_ns
                    - band_start_ns
                )

                source_times_ns = (
                    source_frame[
                        "event_time_ns"
                    ]
                    .to_numpy(
                        dtype="int64"
                    )
                )

                eligible_source_mask = (
                    source_times_ns
                    + band_end_ns
                    <= contract_end_exclusive_ns
                )

                eligible_sources = (
                    source_frame.loc[
                        eligible_source_mask
                    ]
                    .copy()
                    .reset_index(
                        drop=True
                    )
                )

                eligible_source_times_ns = (
                    eligible_sources[
                        "event_time_ns"
                    ]
                    .to_numpy(
                        dtype="int64"
                    )
                )

                require(
                    len(
                        eligible_sources
                    )
                    >= MINIMUM_CONDITIONAL_SOURCE_BATCHES,
                    (
                        f"{partition_name} {source_side}→"
                        f"{target_side} {band_start_ms}-"
                        f"{band_end_ms} ms contains fewer than "
                        f"{MINIMUM_CONDITIONAL_SOURCE_BATCHES} "
                        "eligible source batches."
                    ),
                )

                interval_start_ns = (
                    eligible_source_times_ns
                    + band_start_ns
                )

                interval_end_ns = (
                    eligible_source_times_ns
                    + band_end_ns
                )

                # (start, end] count convention:
                # events exactly at the lower boundary are excluded;
                # events exactly at the upper boundary are included.
                lower_indices = np.searchsorted(
                    target_times_ns,
                    interval_start_ns,
                    side="right",
                )

                upper_indices = np.searchsorted(
                    target_times_ns,
                    interval_end_ns,
                    side="right",
                )

                observed_target_counts = (
                    target_cumulative_weights[
                        upper_indices
                    ]
                    - target_cumulative_weights[
                        lower_indices
                    ]
                ).astype(
                    "int64",
                    copy=False,
                )

                primary_expected_counts = (
                    expected_count_over_intervals(
                        grid=grid,
                        intensity_per_second=(
                            primary_target_intensity
                        ),
                        interval_start_ns=(
                            interval_start_ns
                        ),
                        interval_end_ns=(
                            interval_end_ns
                        ),
                    )
                )

                homogeneous_expected_count_per_source = (
                    target_partition_rate_per_second
                    * (
                        band_width_ns
                        / NANOSECONDS_PER_SECOND
                    )
                )

                homogeneous_expected_counts = np.full(
                    len(
                        eligible_sources
                    ),
                    homogeneous_expected_count_per_source,
                    dtype="float64",
                )

                observed_total = int(
                    observed_target_counts.sum(
                        dtype="int64"
                    )
                )

                primary_expected_total = float(
                    primary_expected_counts.sum(
                        dtype="float64"
                    )
                )

                homogeneous_expected_total = float(
                    homogeneous_expected_counts.sum(
                        dtype="float64"
                    )
                )

                primary_residuals = (
                    observed_target_counts.astype(
                        "float64"
                    )
                    - primary_expected_counts
                )

                conditional_response_rows.append(
                    {
                        "event_partition": (
                            partition_name
                        ),
                        "source_side": (
                            source_side
                        ),
                        "target_side": (
                            target_side
                        ),
                        "response_pair": (
                            f"{source_side}_TO_"
                            f"{target_side}"
                        ),
                        "same_side_response_flag": (
                            source_side
                            == target_side
                        ),
                        "cross_side_response_flag": (
                            source_side
                            != target_side
                        ),
                        "band_start_ms_exclusive": (
                            band_start_ms
                        ),
                        "band_end_ms_inclusive": (
                            band_end_ms
                        ),
                        "band_width_ms": (
                            band_end_ms
                            - band_start_ms
                        ),
                        "eligible_source_batch_count": int(
                            len(
                                eligible_sources
                            )
                        ),
                        "eligible_source_event_count": int(
                            eligible_sources[
                                source_side_count_column
                            ].sum()
                        ),
                        "eligible_mixed_source_batch_count": int(
                            eligible_sources[
                                "mixed_side_batch_flag"
                            ]
                            .astype(bool)
                            .sum()
                        ),
                        "observed_target_event_count": (
                            observed_total
                        ),
                        "primary_expected_target_event_count": (
                            primary_expected_total
                        ),
                        "homogeneous_expected_target_event_count": (
                            homogeneous_expected_total
                        ),
                        "mean_observed_target_events_per_source_batch": (
                            observed_total
                            / len(
                                eligible_sources
                            )
                        ),
                        "mean_primary_expected_target_events_per_source_batch": (
                            primary_expected_total
                            / len(
                                eligible_sources
                            )
                        ),
                        "mean_homogeneous_expected_target_events_per_source_batch": (
                            homogeneous_expected_total
                            / len(
                                eligible_sources
                            )
                        ),
                        "observed_minus_primary_expected": (
                            observed_total
                            - primary_expected_total
                        ),
                        "observed_to_primary_expected_ratio": (
                            observed_total
                            / primary_expected_total
                        ),
                        "observed_minus_homogeneous_expected": (
                            observed_total
                            - homogeneous_expected_total
                        ),
                        "observed_to_homogeneous_expected_ratio": (
                            observed_total
                            / homogeneous_expected_total
                        ),
                        "mean_source_level_primary_residual": float(
                            np.mean(
                                primary_residuals
                            )
                        ),
                        "median_source_level_primary_residual": float(
                            np.median(
                                primary_residuals
                            )
                        ),
                        "positive_primary_residual_source_fraction": float(
                            np.mean(
                                primary_residuals
                                > 0.0
                            )
                        ),
                        "same_timestamp_target_events_excluded": True,
                        "cross_partition_response_windows_used": False,
                        "overlapping_source_windows_possible": True,
                        "formal_independence_p_value_reported": False,
                        "diagnostic_role": (
                            CONDITIONAL_RESPONSE_ROLE
                        ),
                        "status": "PASS",
                    }
                )

                replay = eligible_sources[
                    [
                        "primary_event_batch_id",
                        "partition_batch_index",
                        "event_partition",
                        "event_time_ns",
                        source_side_count_column,
                        "mixed_side_batch_flag",
                    ]
                ].copy()

                replay = replay.rename(
                    columns={
                        source_side_count_column: (
                            "source_side_event_count"
                        )
                    }
                )

                replay[
                    "source_side"
                ] = source_side

                replay[
                    "target_side"
                ] = target_side

                replay[
                    "band_start_ms_exclusive"
                ] = band_start_ms

                replay[
                    "band_end_ms_inclusive"
                ] = band_end_ms

                replay[
                    "interval_start_ns"
                ] = interval_start_ns

                replay[
                    "interval_end_ns"
                ] = interval_end_ns

                replay[
                    "observed_target_event_count"
                ] = observed_target_counts

                replay[
                    "primary_expected_target_event_count"
                ] = primary_expected_counts

                replay[
                    "primary_count_residual"
                ] = primary_residuals

                replay[
                    "same_timestamp_target_events_excluded"
                ] = True

                replay[
                    "cross_partition_response_window_flag"
                ] = False

                replay[
                    "timestamp_jitter_applied"
                ] = False

                replay[
                    "status"
                ] = "PASS"

                conditional_source_replay_frames.append(
                    replay
                )


CONDITIONAL_RESPONSE_SUMMARY = (
    pd.DataFrame(
        conditional_response_rows
    )
    .sort_values(
        [
            "event_partition",
            "source_side",
            "target_side",
            "band_start_ms_exclusive",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

CONDITIONAL_RESPONSE_REPLAY = (
    pd.concat(
        conditional_source_replay_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "event_partition",
            "source_side",
            "target_side",
            "band_start_ms_exclusive",
            "partition_batch_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Conditional-response evidence summaries
# ------------------------------------------------------------

CONDITIONAL_RESPONSE_EVIDENCE = (
    CONDITIONAL_RESPONSE_SUMMARY.groupby(
        [
            "event_partition",
            "source_side",
            "target_side",
            "response_pair",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        evaluated_band_count=(
            "band_width_ms",
            "size",
        ),
        positive_primary_excess_band_count=(
            "observed_minus_primary_expected",
            lambda values: int(
                (
                    values.to_numpy(
                        dtype="float64"
                    )
                    > 0.0
                ).sum()
            ),
        ),
        primary_ratio_above_one_band_count=(
            "observed_to_primary_expected_ratio",
            lambda values: int(
                (
                    values.to_numpy(
                        dtype="float64"
                    )
                    > 1.0
                ).sum()
            ),
        ),
        maximum_observed_to_primary_ratio=(
            "observed_to_primary_expected_ratio",
            "max",
        ),
        minimum_observed_to_primary_ratio=(
            "observed_to_primary_expected_ratio",
            "min",
        ),
        total_observed_target_event_count=(
            "observed_target_event_count",
            "sum",
        ),
        total_primary_expected_target_event_count=(
            "primary_expected_target_event_count",
            "sum",
        ),
        total_observed_minus_primary_expected=(
            "observed_minus_primary_expected",
            "sum",
        ),
    )
    .reset_index()
)

CONDITIONAL_RESPONSE_EVIDENCE[
    "positive_primary_excess_band_fraction"
] = (
    CONDITIONAL_RESPONSE_EVIDENCE[
        "positive_primary_excess_band_count"
    ]
    / CONDITIONAL_RESPONSE_EVIDENCE[
        "evaluated_band_count"
    ]
)

CONDITIONAL_RESPONSE_EVIDENCE[
    "repeated_positive_primary_excess_flag"
] = (
    CONDITIONAL_RESPONSE_EVIDENCE[
        "positive_primary_excess_band_count"
    ]
    >= 2
)

CONDITIONAL_RESPONSE_EVIDENCE[
    "formal_authorization_evidence_flag"
] = False

CONDITIONAL_RESPONSE_EVIDENCE[
    "status"
] = "PASS"


CALIBRATION_CONDITIONAL_RESPONSE_EVIDENCE = (
    CONDITIONAL_RESPONSE_EVIDENCE.loc[
        CONDITIONAL_RESPONSE_EVIDENCE[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

require(
    set(
        CALIBRATION_CONDITIONAL_RESPONSE_EVIDENCE[
            "response_pair"
        ]
    )
    == {
        "BUY_TO_BUY",
        "BUY_TO_SELL",
        "SELL_TO_BUY",
        "SELL_TO_SELL",
    },
    (
        "CALIBRATION conditional-response evidence does not "
        "contain all four source-target combinations."
    ),
)


# ------------------------------------------------------------
# Compact joint-dependence diagnostic ledger
# ------------------------------------------------------------

primary_cross_lookup = {
    row.direction: row
    for row in (
        PRIMARY_BASELINE_CALIBRATION_CROSS_EVIDENCE.itertuples(
            index=False
        )
    )
}

conditional_lookup = {
    row.response_pair: row
    for row in (
        CALIBRATION_CONDITIONAL_RESPONSE_EVIDENCE.itertuples(
            index=False
        )
    )
}

JOINT_DEPENDENCE_DIAGNOSTIC_LEDGER = pd.DataFrame(
    [
        {
            "dependence_channel": (
                "BUY_SELF_RESPONSE"
            ),
            "cross_correlation_direction": (
                "NOT_APPLICABLE"
            ),
            "conditional_response_pair": (
                "BUY_TO_BUY"
            ),
            "residual_cross_dependence_repeated": (
                pd.NA
            ),
            "conditional_positive_excess_band_count": int(
                conditional_lookup[
                    "BUY_TO_BUY"
                ].positive_primary_excess_band_count
            ),
            "conditional_evaluated_band_count": int(
                conditional_lookup[
                    "BUY_TO_BUY"
                ].evaluated_band_count
            ),
            "conditional_repeated_positive_excess": bool(
                conditional_lookup[
                    "BUY_TO_BUY"
                ].repeated_positive_primary_excess_flag
            ),
            "diagnostic_only": True,
            "status": "PASS",
        },
        {
            "dependence_channel": (
                "SELL_SELF_RESPONSE"
            ),
            "cross_correlation_direction": (
                "NOT_APPLICABLE"
            ),
            "conditional_response_pair": (
                "SELL_TO_SELL"
            ),
            "residual_cross_dependence_repeated": (
                pd.NA
            ),
            "conditional_positive_excess_band_count": int(
                conditional_lookup[
                    "SELL_TO_SELL"
                ].positive_primary_excess_band_count
            ),
            "conditional_evaluated_band_count": int(
                conditional_lookup[
                    "SELL_TO_SELL"
                ].evaluated_band_count
            ),
            "conditional_repeated_positive_excess": bool(
                conditional_lookup[
                    "SELL_TO_SELL"
                ].repeated_positive_primary_excess_flag
            ),
            "diagnostic_only": True,
            "status": "PASS",
        },
        {
            "dependence_channel": (
                "BUY_TO_SELL_CROSS_RESPONSE"
            ),
            "cross_correlation_direction": (
                "BUY_LEADS_SELL"
            ),
            "conditional_response_pair": (
                "BUY_TO_SELL"
            ),
            "residual_cross_dependence_repeated": bool(
                primary_cross_lookup[
                    "BUY_LEADS_SELL"
                ].repeated_cross_dependence_flag
            ),
            "conditional_positive_excess_band_count": int(
                conditional_lookup[
                    "BUY_TO_SELL"
                ].positive_primary_excess_band_count
            ),
            "conditional_evaluated_band_count": int(
                conditional_lookup[
                    "BUY_TO_SELL"
                ].evaluated_band_count
            ),
            "conditional_repeated_positive_excess": bool(
                conditional_lookup[
                    "BUY_TO_SELL"
                ].repeated_positive_primary_excess_flag
            ),
            "diagnostic_only": True,
            "status": "PASS",
        },
        {
            "dependence_channel": (
                "SELL_TO_BUY_CROSS_RESPONSE"
            ),
            "cross_correlation_direction": (
                "SELL_LEADS_BUY"
            ),
            "conditional_response_pair": (
                "SELL_TO_BUY"
            ),
            "residual_cross_dependence_repeated": bool(
                primary_cross_lookup[
                    "SELL_LEADS_BUY"
                ].repeated_cross_dependence_flag
            ),
            "conditional_positive_excess_band_count": int(
                conditional_lookup[
                    "SELL_TO_BUY"
                ].positive_primary_excess_band_count
            ),
            "conditional_evaluated_band_count": int(
                conditional_lookup[
                    "SELL_TO_BUY"
                ].evaluated_band_count
            ),
            "conditional_repeated_positive_excess": bool(
                conditional_lookup[
                    "SELL_TO_BUY"
                ].repeated_positive_primary_excess_flag
            ),
            "diagnostic_only": True,
            "status": "PASS",
        },
    ]
)


# ------------------------------------------------------------
# Diagnostic gates
# ------------------------------------------------------------

expected_cross_correlation_rows = (
    len(
        ANALYTICAL_PARTITIONS
    )
    * len(
        CROSS_DEPENDENCE_WIDTHS_MS
    )
    * (
        1
        + len(
            CROSS_DEPENDENCE_MODEL_IDS
        )
    )
    * len(
        CROSS_DEPENDENCE_LAGS
    )
)

expected_conditional_summary_rows = (
    len(
        ANALYTICAL_PARTITIONS
    )
    * 2
    * 2
    * len(
        CONDITIONAL_RESPONSE_BANDS_MS
    )
)

CROSS_AND_CONDITIONAL_DEPENDENCE_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": "cross_correlation_table_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_CROSS_CORRELATION
                )
                == expected_cross_correlation_rows
            ),
            "evidence": (
                f"cross_correlation_rows="
                f"{len(COUNT_CROSS_CORRELATION)}"
            ),
        },
        {
            "gate": "all_cross_correlations_finite",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    COUNT_CROSS_CORRELATION[
                        "cross_correlation"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "all raw-count and Pearson-residual "
                "cross-correlations are finite"
            ),
        },
        {
            "gate": "cross_correlation_lag_convention_complete",
            "severity": "BLOCKING",
            "passed": bool(
                set(
                    COUNT_CROSS_CORRELATION[
                        "direction"
                    ]
                )
                == {
                    "BUY_LEADS_SELL",
                    "SELL_LEADS_BUY",
                    "CONTEMPORANEOUS",
                }
            ),
            "evidence": (
                "negative, zero, and positive lag directions "
                "are represented"
            ),
        },
        {
            "gate": "conditional_response_summary_complete",
            "severity": "BLOCKING",
            "passed": (
                len(
                    CONDITIONAL_RESPONSE_SUMMARY
                )
                == expected_conditional_summary_rows
            ),
            "evidence": (
                f"conditional_summary_rows="
                f"{len(CONDITIONAL_RESPONSE_SUMMARY)}"
            ),
        },
        {
            "gate": "conditional_source_samples_sufficient",
            "severity": "BLOCKING",
            "passed": bool(
                CONDITIONAL_RESPONSE_SUMMARY[
                    "eligible_source_batch_count"
                ]
                .ge(
                    MINIMUM_CONDITIONAL_SOURCE_BATCHES
                )
                .all()
            ),
            "evidence": (
                f"minimum_source_batches="
                f"{int(CONDITIONAL_RESPONSE_SUMMARY['eligible_source_batch_count'].min())}"
            ),
        },
        {
            "gate": "conditional_expected_counts_finite_positive",
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    CONDITIONAL_RESPONSE_SUMMARY[
                        [
                            "primary_expected_target_event_count",
                            "homogeneous_expected_target_event_count",
                            "observed_to_primary_expected_ratio",
                            "observed_to_homogeneous_expected_ratio",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
                and CONDITIONAL_RESPONSE_SUMMARY[
                    "primary_expected_target_event_count"
                ].gt(0.0).all()
                and CONDITIONAL_RESPONSE_SUMMARY[
                    "homogeneous_expected_target_event_count"
                ].gt(0.0).all()
            ),
            "evidence": (
                "all primary-model and homogeneous reference "
                "expectations are finite and positive"
            ),
        },
        {
            "gate": "same_timestamp_response_excluded",
            "severity": "BLOCKING",
            "passed": bool(
                CONDITIONAL_RESPONSE_SUMMARY[
                    "same_timestamp_target_events_excluded"
                ].all()
                and CONDITIONAL_RESPONSE_REPLAY[
                    "same_timestamp_target_events_excluded"
                ].all()
            ),
            "evidence": (
                "response intervals use (lower, upper] and "
                "exclude exact source timestamps"
            ),
        },
        {
            "gate": "response_windows_do_not_cross_partitions",
            "severity": "BLOCKING",
            "passed": bool(
                not CONDITIONAL_RESPONSE_SUMMARY[
                    "cross_partition_response_windows_used"
                ].any()
                and not CONDITIONAL_RESPONSE_REPLAY[
                    "cross_partition_response_window_flag"
                ].any()
            ),
            "evidence": (
                "only source batches with fully observed "
                "within-partition response windows are retained"
            ),
        },
        {
            "gate": "mixed_batches_preserved_without_internal_order",
            "severity": "BLOCKING",
            "passed": bool(
                CONDITIONAL_RESPONSE_SUMMARY[
                    "eligible_mixed_source_batch_count"
                ]
                .ge(0)
                .all()
            ),
            "evidence": (
                "mixed exact-time batches may condition both "
                "sides without imposing within-batch order"
            ),
        },
        {
            "gate": "conditional_overlap_not_misread_as_independence",
            "severity": "BLOCKING",
            "passed": bool(
                CONDITIONAL_RESPONSE_SUMMARY[
                    "overlapping_source_windows_possible"
                ].all()
                and not CONDITIONAL_RESPONSE_SUMMARY[
                    "formal_independence_p_value_reported"
                ].any()
            ),
            "evidence": (
                "conditional-response summaries are diagnostic "
                "and report no naive independent-window p-value"
            ),
        },
        {
            "gate": "primary_intensity_used_for_conditional_expectations",
            "severity": "BLOCKING",
            "passed": bool(
                PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
                in {
                    key[0]
                    for key in (
                        SELECTED_ADAPTIVE_INTENSITIES.keys()
                    )
                }
            ),
            "evidence": (
                f"primary_model="
                f"{PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID}"
            ),
        },
        {
            "gate": "timestamp_jitter_absent",
            "severity": "BLOCKING",
            "passed": bool(
                not CONDITIONAL_RESPONSE_REPLAY[
                    "timestamp_jitter_applied"
                ].any()
            ),
            "evidence": (
                "canonical Notebook 04 exact event times used "
                "without jitter"
            ),
        },
        {
            "gate": "protected_partition_content_remains_absent",
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    CROSS_AND_CONDITIONAL_DEPENDENCE_GATE_FRAME.loc[
        CROSS_AND_CONDITIONAL_DEPENDENCE_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one cross-side or conditional-response "
        "diagnostic gate failed."
    ),
)


display(
    COUNT_CROSS_CORRELATION.loc[
        COUNT_CROSS_CORRELATION[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
        & COUNT_CROSS_CORRELATION[
            "width_ms"
        ].isin(
            {
                100,
                250,
                1_000,
            }
        )
        & COUNT_CROSS_CORRELATION[
            "lag"
        ].isin(
            {
                -10,
                -5,
                -2,
                -1,
                0,
                1,
                2,
                5,
                10,
            }
        ),
        [
            "model_id",
            "series_type",
            "width_ms",
            "lag",
            "lag_duration_ms",
            "direction",
            "aligned_pair_count",
            "cross_correlation",
            "white_noise_95pct_band",
            "absolute_correlation_exceeds_95pct_band",
            "status",
        ],
    ]
)

display(
    PRIMARY_BASELINE_CALIBRATION_CROSS_EVIDENCE
)

display(
    CONDITIONAL_RESPONSE_SUMMARY.loc[
        CONDITIONAL_RESPONSE_SUMMARY[
            "event_partition"
        ].eq(
            "CALIBRATION"
        ),
        [
            "source_side",
            "target_side",
            "band_start_ms_exclusive",
            "band_end_ms_inclusive",
            "eligible_source_batch_count",
            "eligible_mixed_source_batch_count",
            "observed_target_event_count",
            "primary_expected_target_event_count",
            "observed_minus_primary_expected",
            "observed_to_primary_expected_ratio",
            "positive_primary_residual_source_fraction",
            "diagnostic_role",
            "status",
        ],
    ]
)

display(
    CALIBRATION_CONDITIONAL_RESPONSE_EVIDENCE
)

display(
    JOINT_DEPENDENCE_DIAGNOSTIC_LEDGER
)

display(
    CROSS_AND_CONDITIONAL_DEPENDENCE_GATE_FRAME
)

print(
    "BUY/SELL cross-correlations were evaluated for raw counts "
    "and locked-model Pearson residuals at multiple time scales."
)
print(
    "Positive lag means BUY leads SELL; negative lag means SELL "
    "leads BUY; lag zero is contemporaneous association."
)
print(
    "Conditional target-event responses were measured in exact "
    "future bands from 0–10 milliseconds through 2–5 seconds."
)
print(
    "Events at the source timestamp were excluded, response "
    "windows never crossed partition boundaries, and mixed "
    "exact-time batches were retained without internal ordering."
)
print(
    "Conditional observed counts were compared with integrated "
    "expectations from the locked primary simple count baseline."
)
print(
    "Because source-centered windows can overlap, these results "
    "remain diagnostic and no naive independence p-values were "
    "reported."
)
print(
    "Cross-side and conditional-response evidence has been "
    "recorded, but Hawkes estimation remains unauthorized."
)

,model_id,series_type,width_ms,lag,lag_duration_ms,direction,aligned_pair_count,cross_correlation,white_noise_95pct_band,absolute_correlation_exceeds_95pct_band,status
1,OBSERVED_COUNT,OBSERVED_COUNT,100,-10,-1000,SELL_LEADS_BUY,7192,0.0053497377,0.023111665,False,PASS
2,OBSERVED_COUNT,OBSERVED_COUNT,100,-5,-500,SELL_LEADS_BUY,7197,0.0060866244,0.023103635,False,PASS
3,OBSERVED_COUNT,OBSERVED_COUNT,100,-2,-200,SELL_LEADS_BUY,7200,-0.0036405639,0.023098822,False,PASS
4,OBSERVED_COUNT,OBSERVED_COUNT,100,-1,-100,SELL_LEADS_BUY,7201,0.022847165,0.023097218,False,PASS
5,OBSERVED_COUNT,OBSERVED_COUNT,100,0,0,CONTEMPORANEOUS,7202,0.029570338,0.023095614,True,PASS
...,...,...,...,...,...,...,...,...,...,...,...
126,E_SIDE_EWMA_250MS_POISSON,PEARSON_RESIDUAL,1000,0,0,CONTEMPORANEOUS,720,0.027854922,0.073044887,False,PASS
127,E_SIDE_EWMA_250MS_POISSON,PEARSON_RESIDUAL,1000,1,1000,BUY_LEADS_SELL,719,0.0034451732,0.073095666,False,PASS
128,E_SIDE_EWMA_250MS_POISSON,PEARSON_RESIDUAL,1000,2,2000,BUY_LEADS_SELL,718,0.021675759,0.07314655,False,PASS
129,E_SIDE_EWMA_250MS_POISSON,PEARSON_RESIDUAL,1000,5,5000,BUY_LEADS_SELL,715,0.023680136,0.073299844,False,PASS


,model_id,event_partition,series_type,direction,evaluated_correlation_count,significant_correlation_count,positive_significant_count,maximum_absolute_correlation,strongest_positive_correlation,strongest_negative_correlation,significant_correlation_fraction,repeated_cross_dependence_flag,status
0,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,PEARSON_RESIDUAL,SELL_LEADS_BUY,20,0,0,0.034146273,0.031696317,-0.034146273,0,False,PASS
1,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,PEARSON_RESIDUAL,BUY_LEADS_SELL,20,1,1,0.04685404,0.04685404,-0.035294867,0.05,False,PASS


,source_side,target_side,band_start_ms_exclusive,band_end_ms_inclusive,eligible_source_batch_count,eligible_mixed_source_batch_count,observed_target_event_count,primary_expected_target_event_count,observed_minus_primary_expected,observed_to_primary_expected_ratio,positive_primary_residual_source_fraction,diagnostic_role,status
0,BUY,BUY,0,10,1155,5,482,51.646416,430.35358,9.3326901,0.16623377,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
1,BUY,BUY,10,25,1155,5,303,83.327355,219.67265,3.6362609,0.12121212,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
2,BUY,BUY,25,50,1155,5,268,140.45018,127.54982,1.90815,0.1030303,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
3,BUY,BUY,50,100,1155,5,251,274.17056,-23.170557,0.91548853,0.12640693,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
4,BUY,BUY,100,250,1155,5,585,731.00073,-146.00073,0.8002728,0.19134199,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
5,BUY,BUY,250,500,1155,5,656,972.98122,-316.98122,0.6742165,0.24935065,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
6,BUY,BUY,500,1000,1155,5,1377,"1,495.2673",-118.26726,0.92090561,0.31428571,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
7,BUY,BUY,1000,2000,1148,5,2627,"2,402.3",224.69998,1.0935354,0.33362369,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
8,BUY,BUY,2000,5000,1143,5,6527,"6,633.4456",-106.44564,0.98395319,0.33420822,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS
9,BUY,SELL,0,10,1155,5,39,22.799064,16.200936,1.7105966,0.033766234,OVERLAPPING_SOURCE_WINDOW_DIAGNOSTIC_ONLY,PASS


,event_partition,source_side,target_side,response_pair,evaluated_band_count,positive_primary_excess_band_count,primary_ratio_above_one_band_count,maximum_observed_to_primary_ratio,minimum_observed_to_primary_ratio,total_observed_target_event_count,total_primary_expected_target_event_count,total_observed_minus_primary_expected,positive_primary_excess_band_fraction,repeated_positive_primary_excess_flag,formal_authorization_evidence_flag,status
0,CALIBRATION,BUY,BUY,BUY_TO_BUY,9,4,4,9.3326901,0.6742165,13076,"12,784.589",291.41063,0.44444444,True,False,PASS
1,CALIBRATION,BUY,SELL,BUY_TO_SELL,9,4,4,1.7105966,0.81691612,10508,"11,043.693",-535.69275,0.44444444,True,False,PASS
2,CALIBRATION,SELL,BUY,SELL_TO_BUY,9,3,3,1.2454991,0.8153335,10570,"11,346.721",-776.72055,0.33333333,True,False,PASS
3,CALIBRATION,SELL,SELL,SELL_TO_SELL,9,3,3,2.9945712,0.6692164,11809,"12,810.067","-1,001.0668",0.33333333,True,False,PASS


,dependence_channel,cross_correlation_direction,conditional_response_pair,residual_cross_dependence_repeated,conditional_positive_excess_band_count,conditional_evaluated_band_count,conditional_repeated_positive_excess,diagnostic_only,status
0,BUY_SELF_RESPONSE,NOT_APPLICABLE,BUY_TO_BUY,<NA>,4,9,True,True,PASS
1,SELL_SELF_RESPONSE,NOT_APPLICABLE,SELL_TO_SELL,<NA>,3,9,True,True,PASS
2,BUY_TO_SELL_CROSS_RESPONSE,BUY_LEADS_SELL,BUY_TO_SELL,False,4,9,True,True,PASS
3,SELL_TO_BUY_CROSS_RESPONSE,SELL_LEADS_BUY,SELL_TO_BUY,False,3,9,True,True,PASS


,gate,severity,passed,evidence
0,cross_correlation_table_complete,BLOCKING,True,cross_correlation_rows=264
1,all_cross_correlations_finite,BLOCKING,True,all raw-count and Pearson-residual cross-corre...
2,cross_correlation_lag_convention_complete,BLOCKING,True,"negative, zero, and positive lag directions ar..."
3,conditional_response_summary_complete,BLOCKING,True,conditional_summary_rows=72
4,conditional_source_samples_sufficient,BLOCKING,True,minimum_source_batches=1143
5,conditional_expected_counts_finite_positive,BLOCKING,True,all primary-model and homogeneous reference ex...
6,same_timestamp_response_excluded,BLOCKING,True,"response intervals use (lower, upper] and excl..."
7,response_windows_do_not_cross_partitions,BLOCKING,True,only source batches with fully observed within...
8,mixed_batches_preserved_without_internal_order,BLOCKING,True,mixed exact-time batches may condition both si...
9,conditional_overlap_not_misread_as_independence,BLOCKING,True,conditional-response summaries are diagnostic ...


BUY/SELL cross-correlations were evaluated for raw counts and locked-model Pearson residuals at multiple time scales.
Positive lag means BUY leads SELL; negative lag means SELL leads BUY; lag zero is contemporaneous association.
Conditional target-event responses were measured in exact future bands from 0–10 milliseconds through 2–5 seconds.
Events at the source timestamp were excluded, response windows never crossed partition boundaries, and mixed exact-time batches were retained without internal ordering.
Conditional observed counts were compared with integrated expectations from the locked primary simple count baseline.
Because source-centered windows can overlap, these results remain diagnostic and no naive independence p-values were reported.
Cross-side and conditional-response evidence has been recorded, but Hawkes estimation remains unauthorized.


In [19]:
# ============================================================
# Causal activity regimes, burst episodes, and chronological stability
# ============================================================

ACTIVITY_REGIME_LABELS: Final[
    tuple[str, ...]
] = (
    "LOW",
    "NORMAL",
    "HIGH",
)

ACTIVITY_REGIME_QUANTILES: Final[
    tuple[float, float]
] = (
    1.0 / 3.0,
    2.0 / 3.0,
)

BURST_DIAGNOSTIC_WIDTH_MS: Final[int] = 100

BURST_THRESHOLD_QUANTILE: Final[float] = 0.95

CHRONOLOGICAL_STABILITY_WIDTHS_MS: Final[
    tuple[int, ...]
] = (
    10_000,
    30_000,
    60_000,
)

CHRONOLOGICAL_STABILITY_MODEL_IDS: Final[
    tuple[str, ...]
] = ordered_unique(
    (
        FORMAL_COUNT_COMPARATOR_MODEL_ID,
        PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
    )
)

CHRONOLOGICAL_RECONCILIATION_ABSOLUTE_TOLERANCE: Final[
    float
] = 1e-7


# ------------------------------------------------------------
# Exact native-grid chronological aggregation
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class NativeChronologicalBlockContract:
    event_partition: str
    width_ms: int
    width_ns: int
    cells_per_nominal_block: int
    block_count: int
    complete_block_count: int
    partial_final_block_flag: bool
    contract_duration_ns: int


def native_chronological_block_contract(
    *,
    partition_name: str,
    width_ms: int,
) -> NativeChronologicalBlockContract:
    """
    Construct a full-coverage chronological block contract.

    Every native cell is assigned exactly once. The final block may be
    shorter than the nominal width, but it remains in the table so that
    exposure, events, predictions, and log scores reconcile with the full
    native-grid ledger.
    """
    require(
        partition_name in ANALYTICAL_PARTITIONS,
        (
            f"Unauthorized chronological partition: "
            f"{partition_name}"
        ),
    )
    require(
        width_ms >= NATIVE_COUNT_GRID_WIDTH_MS,
        "Chronological width is smaller than the native count grid.",
    )

    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    width_ns = (
        width_ms
        * NANOSECONDS_PER_MILLISECOND
    )

    require(
        width_ns % grid.grid_width_ns == 0,
        (
            f"{partition_name} {width_ms} ms width is not an "
            "integer multiple of the native grid."
        ),
    )

    cells_per_nominal_block = (
        width_ns
        // grid.grid_width_ns
    )

    block_count = int(
        math.ceil(
            grid.cell_count
            / cells_per_nominal_block
        )
    )

    complete_block_count = int(
        grid.contract_duration_ns
        // width_ns
    )

    partial_final_block_flag = bool(
        grid.contract_duration_ns
        % width_ns
        != 0
    )

    require(
        block_count >= 1,
        "Chronological aggregation produced no blocks.",
    )
    require(
        complete_block_count >= 1,
        (
            f"{partition_name} {width_ms} ms aggregation has "
            "no complete block."
        ),
    )

    expected_block_count = (
        complete_block_count
        + int(
            partial_final_block_flag
        )
    )

    require(
        block_count == expected_block_count,
        (
            f"{partition_name} {width_ms} ms native-cell block "
            "count differs from the exact-duration block count."
        ),
    )

    return NativeChronologicalBlockContract(
        event_partition=partition_name,
        width_ms=width_ms,
        width_ns=width_ns,
        cells_per_nominal_block=(
            cells_per_nominal_block
        ),
        block_count=block_count,
        complete_block_count=(
            complete_block_count
        ),
        partial_final_block_flag=(
            partial_final_block_flag
        ),
        contract_duration_ns=(
            grid.contract_duration_ns
        ),
    )


def native_block_start_indices(
    *,
    cell_count: int,
    cells_per_nominal_block: int,
) -> np.ndarray:
    """Return immutable native-cell start indices for full coverage."""
    require(
        cell_count >= 1,
        "Native block aggregation requires at least one cell.",
    )
    require(
        cells_per_nominal_block >= 1,
        "Cells per block must be positive.",
    )

    starts = np.arange(
        0,
        cell_count,
        cells_per_nominal_block,
        dtype="int64",
    )

    require(
        starts.size >= 1,
        "Native block start-index construction failed.",
    )
    require(
        starts[0] == 0,
        "Native block coverage does not begin at the first cell.",
    )

    starts.setflags(
        write=False
    )

    return starts


def reduce_native_array_by_blocks(
    values: np.ndarray,
    *,
    block_start_indices: np.ndarray,
    output_dtype: str,
) -> np.ndarray:
    """Aggregate one native array across contiguous full-coverage blocks."""
    array = np.asarray(
        values
    )

    starts = np.asarray(
        block_start_indices,
        dtype="int64",
    )

    require(
        array.ndim == 1,
        "Native block aggregation input must be one-dimensional.",
    )
    require(
        starts.ndim == 1,
        "Native block start indices must be one-dimensional.",
    )
    require(
        starts.size >= 1,
        "Native block start-index array is empty.",
    )
    require(
        starts[0] == 0,
        "Native block start indices must begin at zero.",
    )
    require(
        starts[-1] < array.size,
        "Native block start index exceeds the input array.",
    )

    reduced = np.add.reduceat(
        array,
        starts,
        dtype=output_dtype,
    )

    require(
        reduced.shape == starts.shape,
        "Native block reduction returned an invalid shape.",
    )

    return reduced


# ------------------------------------------------------------
# Development-frozen causal activity thresholds
# ------------------------------------------------------------

def weighted_empirical_quantile(
    values: np.ndarray,
    weights: np.ndarray,
    quantile: float,
) -> float:
    """Return a deterministic right-continuous weighted quantile."""
    array = np.asarray(
        values,
        dtype="float64",
    )

    weight_array = np.asarray(
        weights,
        dtype="float64",
    )

    require(
        array.ndim == 1,
        "Weighted-quantile values must be one-dimensional.",
    )
    require(
        array.shape == weight_array.shape,
        "Weighted-quantile values and weights differ in shape.",
    )
    require(
        array.size >= 1,
        "Weighted-quantile input is empty.",
    )
    require(
        np.isfinite(array).all(),
        "Weighted-quantile values contain nonfinite entries.",
    )
    require(
        np.isfinite(weight_array).all()
        and np.all(weight_array > 0.0),
        "Weighted-quantile weights must be finite and positive.",
    )
    require(
        0.0 <= quantile <= 1.0,
        "Weighted quantile must be between zero and one.",
    )

    order = np.argsort(
        array,
        kind="stable",
    )

    sorted_values = array[
        order
    ]

    sorted_weights = weight_array[
        order
    ]

    cumulative_weight = np.cumsum(
        sorted_weights,
        dtype="float64",
    )

    target_weight = (
        quantile
        * cumulative_weight[-1]
    )

    selected_index = int(
        np.searchsorted(
            cumulative_weight,
            target_weight,
            side="left",
        )
    )

    selected_index = min(
        selected_index,
        sorted_values.size - 1,
    )

    result = float(
        sorted_values[
            selected_index
        ]
    )

    require(
        np.isfinite(result),
        "Weighted quantile is nonfinite.",
    )

    return result


development_grid = (
    NATIVE_COUNT_GRIDS[
        "DEVELOPMENT"
    ]
)

development_primary_buy_expected = (
    COUNT_BASELINE_EXPECTED_COUNTS[
        (
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
            "DEVELOPMENT",
            "BUY",
        )
    ]
)

development_primary_sell_expected = (
    COUNT_BASELINE_EXPECTED_COUNTS[
        (
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
            "DEVELOPMENT",
            "SELL",
        )
    ]
)

development_primary_pooled_expected = (
    development_primary_buy_expected
    + development_primary_sell_expected
)

development_native_exposure_seconds = (
    development_grid.exposure_ns.astype(
        "float64"
    )
    / NANOSECONDS_PER_SECOND
)

development_primary_pooled_intensity = (
    development_primary_pooled_expected
    / development_native_exposure_seconds
)

activity_lower_threshold = (
    weighted_empirical_quantile(
        development_primary_pooled_intensity,
        development_grid.exposure_ns,
        ACTIVITY_REGIME_QUANTILES[0],
    )
)

activity_upper_threshold = (
    weighted_empirical_quantile(
        development_primary_pooled_intensity,
        development_grid.exposure_ns,
        ACTIVITY_REGIME_QUANTILES[1],
    )
)

if not (
    activity_lower_threshold
    < activity_upper_threshold
):
    unique_development_intensities = np.unique(
        development_primary_pooled_intensity
    )

    require(
        unique_development_intensities.size >= 3,
        (
            "Development primary intensity does not contain "
            "enough distinct values for three activity regimes."
        ),
    )

    lower_index = max(
        0,
        unique_development_intensities.size
        // 3
        - 1,
    )

    upper_index = min(
        unique_development_intensities.size - 1,
        (
            2
            * unique_development_intensities.size
        )
        // 3,
    )

    activity_lower_threshold = float(
        unique_development_intensities[
            lower_index
        ]
    )

    activity_upper_threshold = float(
        unique_development_intensities[
            upper_index
        ]
    )

require(
    activity_lower_threshold
    < activity_upper_threshold,
    (
        "Development-frozen activity thresholds are not "
        "strictly ordered."
    ),
)


CAUSAL_ACTIVITY_REGIME_THRESHOLDS = pd.DataFrame(
    [
        {
            "model_id": (
                PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
            ),
            "fit_partition": (
                "DEVELOPMENT"
            ),
            "selection_partition": (
                "DEVELOPMENT"
            ),
            "lower_quantile": (
                ACTIVITY_REGIME_QUANTILES[0]
            ),
            "upper_quantile": (
                ACTIVITY_REGIME_QUANTILES[1]
            ),
            "lower_intensity_threshold_per_second": (
                activity_lower_threshold
            ),
            "upper_intensity_threshold_per_second": (
                activity_upper_threshold
            ),
            "exposure_weighted_thresholds": True,
            "calibration_used_for_threshold_selection": False,
            "current_cell_events_excluded_from_forecast": True,
            "status": "PASS",
        }
    ]
)


# ------------------------------------------------------------
# Causal activity-regime summaries
# ------------------------------------------------------------

ACTIVITY_REGIME_NATIVE_CODES: dict[
    str,
    np.ndarray,
] = {}

activity_regime_rows: list[
    dict[str, Any]
] = []

activity_regime_conservation_rows: list[
    dict[str, Any]
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    buy_expected = (
        COUNT_BASELINE_EXPECTED_COUNTS[
            (
                PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
                partition_name,
                "BUY",
            )
        ]
    )

    sell_expected = (
        COUNT_BASELINE_EXPECTED_COUNTS[
            (
                PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
                partition_name,
                "SELL",
            )
        ]
    )

    pooled_expected = (
        buy_expected
        + sell_expected
    )

    exposure_seconds = (
        grid.exposure_ns.astype(
            "float64"
        )
        / NANOSECONDS_PER_SECOND
    )

    pooled_intensity = (
        pooled_expected
        / exposure_seconds
    )

    regime_codes = np.select(
        (
            pooled_intensity
            <= activity_lower_threshold,
            pooled_intensity
            <= activity_upper_threshold,
        ),
        (
            0,
            1,
        ),
        default=2,
    ).astype(
        "int8"
    )

    require(
        set(
            np.unique(
                regime_codes
            ).tolist()
        )
        .issubset(
            {0, 1, 2}
        ),
        (
            f"{partition_name} activity-regime codes are "
            "outside the frozen state space."
        ),
    )

    frozen_regime_codes = (
        regime_codes.copy()
    )

    frozen_regime_codes.setflags(
        write=False
    )

    ACTIVITY_REGIME_NATIVE_CODES[
        partition_name
    ] = frozen_regime_codes

    joint_cell_log_scores = (
        COUNT_BASELINE_CELL_LOG_SCORES[
            (
                PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
                partition_name,
            )
        ]
    )

    for regime_code, regime_label in enumerate(
        ACTIVITY_REGIME_LABELS
    ):
        regime_mask = (
            regime_codes
            == regime_code
        )

        regime_exposure_ns = int(
            grid.exposure_ns[
                regime_mask
            ].sum(
                dtype="int64"
            )
        )

        regime_pooled_events = int(
            grid.pooled_counts[
                regime_mask
            ].sum(
                dtype="int64"
            )
        )

        regime_buy_events = int(
            grid.buy_counts[
                regime_mask
            ].sum(
                dtype="int64"
            )
        )

        regime_sell_events = int(
            grid.sell_counts[
                regime_mask
            ].sum(
                dtype="int64"
            )
        )

        regime_predicted_buy = float(
            buy_expected[
                regime_mask
            ].sum(
                dtype="float64"
            )
        )

        regime_predicted_sell = float(
            sell_expected[
                regime_mask
            ].sum(
                dtype="float64"
            )
        )

        regime_predicted_pooled = (
            regime_predicted_buy
            + regime_predicted_sell
        )

        regime_log_score = float(
            joint_cell_log_scores[
                regime_mask
            ].sum(
                dtype="float64"
            )
        )

        require(
            regime_exposure_ns > 0,
            (
                f"{partition_name} {regime_label} activity "
                "regime has zero exposure."
            ),
        )

        regime_exposure_seconds = (
            regime_exposure_ns
            / NANOSECONDS_PER_SECOND
        )

        activity_regime_rows.append(
            {
                "model_id": (
                    PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
                ),
                "event_partition": (
                    partition_name
                ),
                "activity_regime": (
                    regime_label
                ),
                "regime_code": (
                    regime_code
                ),
                "native_cell_count": int(
                    regime_mask.sum()
                ),
                "exposure_ns": (
                    regime_exposure_ns
                ),
                "exposure_seconds": (
                    regime_exposure_seconds
                ),
                "exposure_share": (
                    regime_exposure_ns
                    / grid.contract_duration_ns
                ),
                "pooled_event_count": (
                    regime_pooled_events
                ),
                "buy_event_count": (
                    regime_buy_events
                ),
                "sell_event_count": (
                    regime_sell_events
                ),
                "pooled_event_share": (
                    regime_pooled_events
                    / int(
                        grid.pooled_counts.sum(
                            dtype="int64"
                        )
                    )
                ),
                "predicted_pooled_event_count": (
                    regime_predicted_pooled
                ),
                "predicted_buy_event_count": (
                    regime_predicted_buy
                ),
                "predicted_sell_event_count": (
                    regime_predicted_sell
                ),
                "observed_pooled_rate_per_second": (
                    regime_pooled_events
                    / regime_exposure_seconds
                ),
                "predicted_pooled_rate_per_second": (
                    regime_predicted_pooled
                    / regime_exposure_seconds
                ),
                "observed_minus_predicted_events": (
                    regime_pooled_events
                    - regime_predicted_pooled
                ),
                "observed_to_predicted_event_ratio": (
                    regime_pooled_events
                    / regime_predicted_pooled
                ),
                "joint_log_score_total": (
                    regime_log_score
                ),
                "joint_log_score_per_second": (
                    regime_log_score
                    / regime_exposure_seconds
                ),
                "thresholds_selected_on_development_only": True,
                "current_cell_events_excluded_from_regime_forecast": True,
                "status": "PASS",
            }
        )

    partition_regime_rows = [
        row
        for row in activity_regime_rows
        if row[
            "event_partition"
        ]
        == partition_name
    ]

    regime_exposure_sum_ns = int(
        sum(
            row[
                "exposure_ns"
            ]
            for row in partition_regime_rows
        )
    )

    regime_event_sum = int(
        sum(
            row[
                "pooled_event_count"
            ]
            for row in partition_regime_rows
        )
    )

    regime_predicted_sum = float(
        sum(
            row[
                "predicted_pooled_event_count"
            ]
            for row in partition_regime_rows
        )
    )

    regime_score_sum = float(
        sum(
            row[
                "joint_log_score_total"
            ]
            for row in partition_regime_rows
        )
    )

    unified_score_row = (
        UNIFIED_COUNT_BASELINE_SCORE_TABLE.loc[
            UNIFIED_COUNT_BASELINE_SCORE_TABLE[
                "model_id"
            ].eq(
                PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
            )
            & UNIFIED_COUNT_BASELINE_SCORE_TABLE[
                "scored_partition"
            ].eq(
                partition_name
            )
        ]
    )

    require(
        len(
            unified_score_row
        )
        == 1,
        (
            f"Expected one unified primary-model row for "
            f"{partition_name}."
        ),
    )

    unified_score_value = float(
        unified_score_row.iloc[0][
            "log_score_total"
        ]
    )

    unified_predicted_value = float(
        unified_score_row.iloc[0][
            "predicted_event_count"
        ]
    )

    activity_regime_conservation_rows.append(
        {
            "event_partition": (
                partition_name
            ),
            "regime_exposure_sum_ns": (
                regime_exposure_sum_ns
            ),
            "contract_duration_ns": (
                grid.contract_duration_ns
            ),
            "exposure_difference_ns": (
                regime_exposure_sum_ns
                - grid.contract_duration_ns
            ),
            "regime_event_sum": (
                regime_event_sum
            ),
            "native_event_sum": int(
                grid.pooled_counts.sum(
                    dtype="int64"
                )
            ),
            "event_difference": (
                regime_event_sum
                - int(
                    grid.pooled_counts.sum(
                        dtype="int64"
                    )
                )
            ),
            "regime_predicted_sum": (
                regime_predicted_sum
            ),
            "unified_predicted_count": (
                unified_predicted_value
            ),
            "predicted_difference": (
                regime_predicted_sum
                - unified_predicted_value
            ),
            "regime_score_sum": (
                regime_score_sum
            ),
            "unified_log_score": (
                unified_score_value
            ),
            "score_difference": (
                regime_score_sum
                - unified_score_value
            ),
            "status": (
                "PASS"
                if (
                    regime_exposure_sum_ns
                    == grid.contract_duration_ns
                    and regime_event_sum
                    == int(
                        grid.pooled_counts.sum(
                            dtype="int64"
                        )
                    )
                    and abs(
                        regime_predicted_sum
                        - unified_predicted_value
                    )
                    <= CHRONOLOGICAL_RECONCILIATION_ABSOLUTE_TOLERANCE
                    and abs(
                        regime_score_sum
                        - unified_score_value
                    )
                    <= CHRONOLOGICAL_RECONCILIATION_ABSOLUTE_TOLERANCE
                )
                else "FAIL"
            ),
        }
    )


CAUSAL_ACTIVITY_REGIME_SUMMARY = (
    pd.DataFrame(
        activity_regime_rows
    )
    .sort_values(
        [
            "event_partition",
            "regime_code",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

CAUSAL_ACTIVITY_REGIME_CONSERVATION = (
    pd.DataFrame(
        activity_regime_conservation_rows
    )
)

require(
    CAUSAL_ACTIVITY_REGIME_CONSERVATION[
        "status"
    ].eq(
        "PASS"
    ).all(),
    (
        "Causal activity-regime aggregation does not conserve "
        "native exposure, events, predictions, or scores."
    ),
)


# ------------------------------------------------------------
# Development-frozen burst threshold
# ------------------------------------------------------------

burst_contracts: dict[
    str,
    NativeChronologicalBlockContract,
] = {}

burst_block_frames: dict[
    str,
    pd.DataFrame,
] = {}


for partition_name in ANALYTICAL_PARTITIONS:
    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    contract = (
        native_chronological_block_contract(
            partition_name=(
                partition_name
            ),
            width_ms=(
                BURST_DIAGNOSTIC_WIDTH_MS
            ),
        )
    )

    starts = (
        native_block_start_indices(
            cell_count=(
                grid.cell_count
            ),
            cells_per_nominal_block=(
                contract.cells_per_nominal_block
            ),
        )
    )

    exposure_ns = (
        reduce_native_array_by_blocks(
            grid.exposure_ns,
            block_start_indices=(
                starts
            ),
            output_dtype="int64",
        )
        .astype(
            "int64",
            copy=False,
        )
    )

    pooled_counts = (
        reduce_native_array_by_blocks(
            grid.pooled_counts,
            block_start_indices=(
                starts
            ),
            output_dtype="int64",
        )
        .astype(
            "int64",
            copy=False,
        )
    )

    block_indices = np.arange(
        starts.size,
        dtype="int64",
    )

    block_start_ns = (
        grid.contract_start_ns
        + starts
        * grid.grid_width_ns
    )

    block_end_exclusive_ns = (
        block_start_ns
        + exposure_ns
    )

    burst_block_frame = pd.DataFrame(
        {
            "event_partition": (
                partition_name
            ),
            "block_index": (
                block_indices
            ),
            "block_start_ns": (
                block_start_ns
            ),
            "block_end_exclusive_ns": (
                block_end_exclusive_ns
            ),
            "block_start_utc": (
                pd.to_datetime(
                    block_start_ns,
                    unit="ns",
                    utc=True,
                )
            ),
            "block_end_exclusive_utc": (
                pd.to_datetime(
                    block_end_exclusive_ns,
                    unit="ns",
                    utc=True,
                )
            ),
            "exposure_ns": (
                exposure_ns
            ),
            "exposure_seconds": (
                exposure_ns.astype(
                    "float64"
                )
                / NANOSECONDS_PER_SECOND
            ),
            "pooled_event_count": (
                pooled_counts
            ),
            "complete_block_flag": (
                exposure_ns
                == contract.width_ns
            ),
            "partial_final_block_flag": (
                exposure_ns
                < contract.width_ns
            ),
        }
    )

    require(
        int(
            burst_block_frame[
                "exposure_ns"
            ].sum()
        )
        == grid.contract_duration_ns,
        (
            f"{partition_name} burst blocks do not conserve "
            "contract exposure."
        ),
    )

    require(
        int(
            burst_block_frame[
                "pooled_event_count"
            ].sum()
        )
        == int(
            grid.pooled_counts.sum(
                dtype="int64"
            )
        ),
        (
            f"{partition_name} burst blocks do not conserve "
            "pooled events."
        ),
    )

    burst_contracts[
        partition_name
    ] = contract

    burst_block_frames[
        partition_name
    ] = burst_block_frame


development_complete_burst_counts = (
    burst_block_frames[
        "DEVELOPMENT"
    ]
    .loc[
        lambda frame: frame[
            "complete_block_flag"
        ],
        "pooled_event_count",
    ]
    .to_numpy(
        dtype="int64"
    )
)

require(
    development_complete_burst_counts.size >= 100,
    (
        "Development contains too few complete burst-diagnostic "
        "blocks."
    ),
)

BURST_EVENT_COUNT_THRESHOLD: Final[int] = max(
    2,
    int(
        np.quantile(
            development_complete_burst_counts,
            BURST_THRESHOLD_QUANTILE,
            method="higher",
        )
    ),
)

BURST_THRESHOLD_CONTRACT = pd.DataFrame(
    [
        {
            "selection_partition": (
                "DEVELOPMENT"
            ),
            "block_width_ms": (
                BURST_DIAGNOSTIC_WIDTH_MS
            ),
            "threshold_quantile": (
                BURST_THRESHOLD_QUANTILE
            ),
            "minimum_threshold_floor": 2,
            "selected_event_count_threshold": (
                BURST_EVENT_COUNT_THRESHOLD
            ),
            "development_complete_block_count": int(
                development_complete_burst_counts.size
            ),
            "calibration_used_for_threshold_selection": False,
            "partial_final_block_excluded_from_threshold_selection": True,
            "status": "PASS",
        }
    ]
)


# ------------------------------------------------------------
# Burst episodes and concentration
# ------------------------------------------------------------

burst_episode_rows: list[
    dict[str, Any]
] = []

burst_concentration_rows: list[
    dict[str, Any]
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    frame = (
        burst_block_frames[
            partition_name
        ]
        .copy()
    )

    complete_frame = (
        frame.loc[
            frame[
                "complete_block_flag"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    burst_flags = (
        complete_frame[
            "pooled_event_count"
        ]
        .to_numpy(
            dtype="int64"
        )
        >= BURST_EVENT_COUNT_THRESHOLD
    )

    complete_frame[
        "burst_flag"
    ] = burst_flags

    start_flags = (
        burst_flags
        & np.concatenate(
            (
                np.array(
                    [True],
                    dtype=bool,
                ),
                ~burst_flags[:-1],
            )
        )
    )

    end_flags = (
        burst_flags
        & np.concatenate(
            (
                ~burst_flags[1:],
                np.array(
                    [True],
                    dtype=bool,
                ),
            )
        )
    )

    episode_start_positions = np.flatnonzero(
        start_flags
    )

    episode_end_positions = np.flatnonzero(
        end_flags
    )

    require(
        episode_start_positions.size
        == episode_end_positions.size,
        (
            f"{partition_name} burst episode starts and ends "
            "do not reconcile."
        ),
    )

    for episode_number, (
        start_position,
        end_position,
    ) in enumerate(
        zip(
            episode_start_positions,
            episode_end_positions,
            strict=True,
        ),
        start=1,
    ):
        require(
            end_position >= start_position,
            "Burst episode ends before it begins.",
        )

        episode_frame = (
            complete_frame.iloc[
                start_position:
                end_position + 1
            ]
        )

        burst_episode_rows.append(
            {
                "event_partition": (
                    partition_name
                ),
                "episode_number": (
                    episode_number
                ),
                "first_block_index": int(
                    episode_frame[
                        "block_index"
                    ].iloc[0]
                ),
                "last_block_index": int(
                    episode_frame[
                        "block_index"
                    ].iloc[-1]
                ),
                "episode_start_ns": int(
                    episode_frame[
                        "block_start_ns"
                    ].iloc[0]
                ),
                "episode_end_exclusive_ns": int(
                    episode_frame[
                        "block_end_exclusive_ns"
                    ].iloc[-1]
                ),
                "episode_start_utc": (
                    episode_frame[
                        "block_start_utc"
                    ].iloc[0]
                ),
                "episode_end_exclusive_utc": (
                    episode_frame[
                        "block_end_exclusive_utc"
                    ].iloc[-1]
                ),
                "burst_block_count": int(
                    len(
                        episode_frame
                    )
                ),
                "episode_duration_ms": int(
                    len(
                        episode_frame
                    )
                    * BURST_DIAGNOSTIC_WIDTH_MS
                ),
                "episode_event_count": int(
                    episode_frame[
                        "pooled_event_count"
                    ].sum()
                ),
                "maximum_events_in_one_block": int(
                    episode_frame[
                        "pooled_event_count"
                    ].max()
                ),
                "threshold_event_count": (
                    BURST_EVENT_COUNT_THRESHOLD
                ),
                "status": "PASS",
            }
        )

    complete_event_count = int(
        complete_frame[
            "pooled_event_count"
        ].sum()
    )

    burst_event_count = int(
        complete_frame.loc[
            complete_frame[
                "burst_flag"
            ],
            "pooled_event_count",
        ].sum()
    )

    episode_partition_frame = pd.DataFrame(
        [
            row
            for row in burst_episode_rows
            if row[
                "event_partition"
            ]
            == partition_name
        ]
    )

    if episode_partition_frame.empty:
        episode_count = 0
        median_episode_duration_ms = 0.0
        maximum_episode_duration_ms = 0
        maximum_episode_event_count = 0

    else:
        episode_count = int(
            len(
                episode_partition_frame
            )
        )

        median_episode_duration_ms = float(
            episode_partition_frame[
                "episode_duration_ms"
            ].median()
        )

        maximum_episode_duration_ms = int(
            episode_partition_frame[
                "episode_duration_ms"
            ].max()
        )

        maximum_episode_event_count = int(
            episode_partition_frame[
                "episode_event_count"
            ].max()
        )

    burst_concentration_rows.append(
        {
            "event_partition": (
                partition_name
            ),
            "block_width_ms": (
                BURST_DIAGNOSTIC_WIDTH_MS
            ),
            "threshold_event_count": (
                BURST_EVENT_COUNT_THRESHOLD
            ),
            "complete_block_count": int(
                len(
                    complete_frame
                )
            ),
            "partial_final_block_count": int(
                frame[
                    "partial_final_block_flag"
                ].sum()
            ),
            "burst_block_count": int(
                burst_flags.sum()
            ),
            "burst_block_fraction": float(
                burst_flags.mean()
            ),
            "complete_block_event_count": (
                complete_event_count
            ),
            "burst_block_event_count": (
                burst_event_count
            ),
            "burst_event_share": (
                burst_event_count
                / complete_event_count
            ),
            "episode_count": (
                episode_count
            ),
            "median_episode_duration_ms": (
                median_episode_duration_ms
            ),
            "maximum_episode_duration_ms": (
                maximum_episode_duration_ms
            ),
            "maximum_episode_event_count": (
                maximum_episode_event_count
            ),
            "threshold_selected_on_development_only": True,
            "partial_final_block_excluded_from_episode_detection": True,
            "timestamp_jitter_applied": False,
            "status": "PASS",
        }
    )

    burst_block_frames[
        partition_name
    ] = frame.merge(
        complete_frame[
            [
                "block_index",
                "burst_flag",
            ]
        ],
        on="block_index",
        how="left",
        validate="one_to_one",
    )

    burst_block_frames[
        partition_name
    ][
        "burst_flag"
    ] = (
        burst_block_frames[
            partition_name
        ][
            "burst_flag"
        ]
        .fillna(False)
        .astype(bool)
    )


BURST_EPISODE_TABLE = (
    pd.DataFrame(
        burst_episode_rows
    )
)

if not BURST_EPISODE_TABLE.empty:
    BURST_EPISODE_TABLE = (
        BURST_EPISODE_TABLE.sort_values(
            [
                "event_partition",
                "episode_number",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

BURST_CONCENTRATION_SUMMARY = (
    pd.DataFrame(
        burst_concentration_rows
    )
    .sort_values(
        "event_partition",
        kind="stable",
    )
    .reset_index(drop=True)
)

BURST_BLOCK_TABLE = (
    pd.concat(
        [
            burst_block_frames[
                partition_name
            ]
            for partition_name in (
                ANALYTICAL_PARTITIONS
            )
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "event_partition",
            "block_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Chronological model-score blocks with complete ledger coverage
# ------------------------------------------------------------

chronological_score_block_frames: list[
    pd.DataFrame
] = []


for partition_name in ANALYTICAL_PARTITIONS:
    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    for width_ms in (
        CHRONOLOGICAL_STABILITY_WIDTHS_MS
    ):
        contract = (
            native_chronological_block_contract(
                partition_name=(
                    partition_name
                ),
                width_ms=width_ms,
            )
        )

        starts = (
            native_block_start_indices(
                cell_count=(
                    grid.cell_count
                ),
                cells_per_nominal_block=(
                    contract.cells_per_nominal_block
                ),
            )
        )

        exposure_ns = (
            reduce_native_array_by_blocks(
                grid.exposure_ns,
                block_start_indices=(
                    starts
                ),
                output_dtype="int64",
            )
            .astype(
                "int64",
                copy=False,
            )
        )

        buy_observed = (
            reduce_native_array_by_blocks(
                grid.buy_counts,
                block_start_indices=(
                    starts
                ),
                output_dtype="int64",
            )
            .astype(
                "int64",
                copy=False,
            )
        )

        sell_observed = (
            reduce_native_array_by_blocks(
                grid.sell_counts,
                block_start_indices=(
                    starts
                ),
                output_dtype="int64",
            )
            .astype(
                "int64",
                copy=False,
            )
        )

        pooled_observed = (
            buy_observed
            + sell_observed
        )

        block_indices = np.arange(
            starts.size,
            dtype="int64",
        )

        block_start_ns = (
            grid.contract_start_ns
            + starts
            * grid.grid_width_ns
        )

        block_end_exclusive_ns = (
            block_start_ns
            + exposure_ns
        )

        require(
            int(
                exposure_ns.sum(
                    dtype="int64"
                )
            )
            == grid.contract_duration_ns,
            (
                f"{partition_name} {width_ms} ms chronological "
                "blocks do not conserve exact exposure."
            ),
        )

        require(
            int(
                pooled_observed.sum(
                    dtype="int64"
                )
            )
            == int(
                grid.pooled_counts.sum(
                    dtype="int64"
                )
            ),
            (
                f"{partition_name} {width_ms} ms chronological "
                "blocks do not conserve observed events."
            ),
        )

        require(
            int(
                block_end_exclusive_ns[
                    -1
                ]
            )
            == grid.contract_end_exclusive_ns,
            (
                f"{partition_name} {width_ms} ms final "
                "chronological block does not end at the "
                "registered contract boundary."
            ),
        )

        for model_id in (
            CHRONOLOGICAL_STABILITY_MODEL_IDS
        ):
            buy_expected = (
                reduce_native_array_by_blocks(
                    COUNT_BASELINE_EXPECTED_COUNTS[
                        (
                            model_id,
                            partition_name,
                            "BUY",
                        )
                    ],
                    block_start_indices=(
                        starts
                    ),
                    output_dtype="float64",
                )
                .astype(
                    "float64",
                    copy=False,
                )
            )

            sell_expected = (
                reduce_native_array_by_blocks(
                    COUNT_BASELINE_EXPECTED_COUNTS[
                        (
                            model_id,
                            partition_name,
                            "SELL",
                        )
                    ],
                    block_start_indices=(
                        starts
                    ),
                    output_dtype="float64",
                )
                .astype(
                    "float64",
                    copy=False,
                )
            )

            pooled_expected = (
                buy_expected
                + sell_expected
            )

            block_log_scores = (
                reduce_native_array_by_blocks(
                    COUNT_BASELINE_CELL_LOG_SCORES[
                        (
                            model_id,
                            partition_name,
                        )
                    ],
                    block_start_indices=(
                        starts
                    ),
                    output_dtype="float64",
                )
                .astype(
                    "float64",
                    copy=False,
                )
            )

            require(
                np.isfinite(
                    pooled_expected
                ).all()
                and np.all(
                    pooled_expected > 0.0
                ),
                (
                    f"{model_id} {partition_name} {width_ms} ms "
                    "expected block counts are invalid."
                ),
            )

            require(
                np.isfinite(
                    block_log_scores
                ).all(),
                (
                    f"{model_id} {partition_name} {width_ms} ms "
                    "block scores contain nonfinite values."
                ),
            )

            exposure_seconds = (
                exposure_ns.astype(
                    "float64"
                )
                / NANOSECONDS_PER_SECOND
            )

            chronological_score_block_frames.append(
                pd.DataFrame(
                    {
                        "model_id": (
                            model_id
                        ),
                        "model_family": (
                            count_model_family(
                                model_id
                            )
                        ),
                        "event_partition": (
                            partition_name
                        ),
                        "block_width_ms": (
                            width_ms
                        ),
                        "block_index": (
                            block_indices
                        ),
                        "block_start_ns": (
                            block_start_ns
                        ),
                        "block_end_exclusive_ns": (
                            block_end_exclusive_ns
                        ),
                        "block_start_utc": (
                            pd.to_datetime(
                                block_start_ns,
                                unit="ns",
                                utc=True,
                            )
                        ),
                        "block_end_exclusive_utc": (
                            pd.to_datetime(
                                block_end_exclusive_ns,
                                unit="ns",
                                utc=True,
                            )
                        ),
                        "exposure_ns": (
                            exposure_ns
                        ),
                        "exposure_seconds": (
                            exposure_seconds
                        ),
                        "complete_nominal_block_flag": (
                            exposure_ns
                            == contract.width_ns
                        ),
                        "partial_final_block_flag": (
                            exposure_ns
                            < contract.width_ns
                        ),
                        "observed_buy_event_count": (
                            buy_observed
                        ),
                        "observed_sell_event_count": (
                            sell_observed
                        ),
                        "observed_pooled_event_count": (
                            pooled_observed
                        ),
                        "predicted_buy_event_count": (
                            buy_expected
                        ),
                        "predicted_sell_event_count": (
                            sell_expected
                        ),
                        "predicted_pooled_event_count": (
                            pooled_expected
                        ),
                        "observed_pooled_rate_per_second": (
                            pooled_observed.astype(
                                "float64"
                            )
                            / exposure_seconds
                        ),
                        "predicted_pooled_rate_per_second": (
                            pooled_expected
                            / exposure_seconds
                        ),
                        "observed_minus_predicted_events": (
                            pooled_observed.astype(
                                "float64"
                            )
                            - pooled_expected
                        ),
                        "joint_log_score_total": (
                            block_log_scores
                        ),
                        "joint_log_score_per_second": (
                            block_log_scores
                            / exposure_seconds
                        ),
                        "full_native_ledger_coverage_flag": True,
                        "status": "PASS",
                    }
                )
            )


CHRONOLOGICAL_MODEL_SCORE_BLOCKS = (
    pd.concat(
        chronological_score_block_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "block_width_ms",
            "block_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Exact chronological score reconciliation
# ------------------------------------------------------------

stability_score_reconciliation_rows: list[
    dict[str, Any]
] = []


for (
    model_id,
    partition_name,
    width_ms,
), group in (
    CHRONOLOGICAL_MODEL_SCORE_BLOCKS.groupby(
        [
            "model_id",
            "event_partition",
            "block_width_ms",
        ],
        observed=True,
        sort=False,
    )
):
    group = group.sort_values(
        "block_index",
        kind="stable",
    )

    unified_rows = (
        UNIFIED_COUNT_BASELINE_SCORE_TABLE.loc[
            UNIFIED_COUNT_BASELINE_SCORE_TABLE[
                "model_id"
            ].eq(
                model_id
            )
            & UNIFIED_COUNT_BASELINE_SCORE_TABLE[
                "scored_partition"
            ].eq(
                partition_name
            )
        ]
    )

    require(
        len(
            unified_rows
        )
        == 1,
        (
            f"Expected one unified score row for {model_id} "
            f"on {partition_name}."
        ),
    )

    unified_row = (
        unified_rows.iloc[0]
    )

    block_score_sum = float(
        group[
            "joint_log_score_total"
        ].sum()
    )

    unified_score = float(
        unified_row[
            "log_score_total"
        ]
    )

    block_predicted_sum = float(
        group[
            "predicted_pooled_event_count"
        ].sum()
    )

    unified_predicted_count = float(
        unified_row[
            "predicted_event_count"
        ]
    )

    block_observed_sum = int(
        group[
            "observed_pooled_event_count"
        ].sum()
    )

    unified_observed_count = int(
        unified_row[
            "observed_event_count"
        ]
    )

    block_exposure_sum_ns = int(
        group[
            "exposure_ns"
        ].sum()
    )

    registered_exposure_ns = (
        NATIVE_COUNT_GRIDS[
            partition_name
        ].contract_duration_ns
    )

    score_difference = (
        block_score_sum
        - unified_score
    )

    predicted_difference = (
        block_predicted_sum
        - unified_predicted_count
    )

    reconciliation_passed = bool(
        abs(
            score_difference
        )
        <= CHRONOLOGICAL_RECONCILIATION_ABSOLUTE_TOLERANCE
        and abs(
            predicted_difference
        )
        <= CHRONOLOGICAL_RECONCILIATION_ABSOLUTE_TOLERANCE
        and block_observed_sum
        == unified_observed_count
        and block_exposure_sum_ns
        == registered_exposure_ns
    )

    stability_score_reconciliation_rows.append(
        {
            "model_id": (
                model_id
            ),
            "event_partition": (
                partition_name
            ),
            "block_width_ms": (
                int(
                    width_ms
                )
            ),
            "block_count": int(
                len(
                    group
                )
            ),
            "complete_block_count": int(
                group[
                    "complete_nominal_block_flag"
                ].sum()
            ),
            "partial_final_block_count": int(
                group[
                    "partial_final_block_flag"
                ].sum()
            ),
            "block_exposure_sum_ns": (
                block_exposure_sum_ns
            ),
            "registered_exposure_ns": (
                registered_exposure_ns
            ),
            "exposure_difference_ns": (
                block_exposure_sum_ns
                - registered_exposure_ns
            ),
            "block_observed_event_sum": (
                block_observed_sum
            ),
            "unified_observed_event_count": (
                unified_observed_count
            ),
            "observed_event_difference": (
                block_observed_sum
                - unified_observed_count
            ),
            "block_predicted_event_sum": (
                block_predicted_sum
            ),
            "unified_predicted_event_count": (
                unified_predicted_count
            ),
            "predicted_event_difference": (
                predicted_difference
            ),
            "block_log_score_sum": (
                block_score_sum
            ),
            "unified_log_score": (
                unified_score
            ),
            "log_score_difference": (
                score_difference
            ),
            "absolute_tolerance": (
                CHRONOLOGICAL_RECONCILIATION_ABSOLUTE_TOLERANCE
            ),
            "full_and_partial_blocks_included": True,
            "status": (
                "PASS"
                if reconciliation_passed
                else "FAIL"
            ),
        }
    )


CHRONOLOGICAL_SCORE_RECONCILIATION = (
    pd.DataFrame(
        stability_score_reconciliation_rows
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "block_width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

require(
    CHRONOLOGICAL_SCORE_RECONCILIATION[
        "status"
    ].eq(
        "PASS"
    ).all(),
    (
        "Chronological stability scores do not reconcile with "
        "the unified native-grid score ledger."
    ),
)


# ------------------------------------------------------------
# Chronological stability summaries
# ------------------------------------------------------------

chronological_stability_rows: list[
    dict[str, Any]
] = []


for (
    model_id,
    partition_name,
    width_ms,
), group in (
    CHRONOLOGICAL_MODEL_SCORE_BLOCKS.groupby(
        [
            "model_id",
            "event_partition",
            "block_width_ms",
        ],
        observed=True,
        sort=False,
    )
):
    ordered_group = (
        group.sort_values(
            "block_index",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    complete_group = (
        ordered_group.loc[
            ordered_group[
                "complete_nominal_block_flag"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    require(
        len(
            complete_group
        )
        >= 2,
        (
            f"{model_id} {partition_name} {width_ms} ms "
            "chronological stability requires at least two "
            "complete blocks."
        ),
    )

    midpoint = max(
        1,
        len(
            complete_group
        )
        // 2,
    )

    first_half = (
        complete_group.iloc[
            :midpoint
        ]
    )

    second_half = (
        complete_group.iloc[
            midpoint:
        ]
    )

    require(
        not first_half.empty
        and not second_half.empty,
        "Chronological first/second-half split is empty.",
    )

    first_half_exposure_seconds = float(
        first_half[
            "exposure_seconds"
        ].sum()
    )

    second_half_exposure_seconds = float(
        second_half[
            "exposure_seconds"
        ].sum()
    )

    first_half_observed_rate = (
        float(
            first_half[
                "observed_pooled_event_count"
            ].sum()
        )
        / first_half_exposure_seconds
    )

    second_half_observed_rate = (
        float(
            second_half[
                "observed_pooled_event_count"
            ].sum()
        )
        / second_half_exposure_seconds
    )

    first_half_predicted_rate = (
        float(
            first_half[
                "predicted_pooled_event_count"
            ].sum()
        )
        / first_half_exposure_seconds
    )

    second_half_predicted_rate = (
        float(
            second_half[
                "predicted_pooled_event_count"
            ].sum()
        )
        / second_half_exposure_seconds
    )

    first_half_log_score_per_second = (
        float(
            first_half[
                "joint_log_score_total"
            ].sum()
        )
        / first_half_exposure_seconds
    )

    second_half_log_score_per_second = (
        float(
            second_half[
                "joint_log_score_total"
            ].sum()
        )
        / second_half_exposure_seconds
    )

    observed_block_rates = (
        complete_group[
            "observed_pooled_rate_per_second"
        ]
        .to_numpy(
            dtype="float64"
        )
    )

    predicted_block_rates = (
        complete_group[
            "predicted_pooled_rate_per_second"
        ]
        .to_numpy(
            dtype="float64"
        )
    )

    score_rates = (
        complete_group[
            "joint_log_score_per_second"
        ]
        .to_numpy(
            dtype="float64"
        )
    )

    chronological_stability_rows.append(
        {
            "model_id": (
                model_id
            ),
            "model_family": (
                count_model_family(
                    model_id
                )
            ),
            "event_partition": (
                partition_name
            ),
            "block_width_ms": int(
                width_ms
            ),
            "all_block_count": int(
                len(
                    ordered_group
                )
            ),
            "complete_block_count": int(
                len(
                    complete_group
                )
            ),
            "partial_final_block_count": int(
                ordered_group[
                    "partial_final_block_flag"
                ].sum()
            ),
            "full_contract_exposure_seconds": float(
                ordered_group[
                    "exposure_seconds"
                ].sum()
            ),
            "full_contract_observed_event_count": int(
                ordered_group[
                    "observed_pooled_event_count"
                ].sum()
            ),
            "full_contract_predicted_event_count": float(
                ordered_group[
                    "predicted_pooled_event_count"
                ].sum()
            ),
            "full_contract_log_score": float(
                ordered_group[
                    "joint_log_score_total"
                ].sum()
            ),
            "minimum_complete_block_observed_rate": float(
                np.min(
                    observed_block_rates
                )
            ),
            "median_complete_block_observed_rate": float(
                np.median(
                    observed_block_rates
                )
            ),
            "maximum_complete_block_observed_rate": float(
                np.max(
                    observed_block_rates
                )
            ),
            "observed_rate_coefficient_of_variation": float(
                np.std(
                    observed_block_rates,
                    ddof=1,
                )
                / np.mean(
                    observed_block_rates
                )
            ),
            "predicted_rate_coefficient_of_variation": float(
                np.std(
                    predicted_block_rates,
                    ddof=1,
                )
                / np.mean(
                    predicted_block_rates
                )
            ),
            "minimum_complete_block_log_score_per_second": float(
                np.min(
                    score_rates
                )
            ),
            "median_complete_block_log_score_per_second": float(
                np.median(
                    score_rates
                )
            ),
            "maximum_complete_block_log_score_per_second": float(
                np.max(
                    score_rates
                )
            ),
            "log_score_per_second_sample_std": float(
                np.std(
                    score_rates,
                    ddof=1,
                )
            ),
            "first_half_observed_rate_per_second": (
                first_half_observed_rate
            ),
            "second_half_observed_rate_per_second": (
                second_half_observed_rate
            ),
            "second_to_first_observed_rate_ratio": (
                second_half_observed_rate
                / first_half_observed_rate
            ),
            "first_half_predicted_rate_per_second": (
                first_half_predicted_rate
            ),
            "second_half_predicted_rate_per_second": (
                second_half_predicted_rate
            ),
            "second_to_first_predicted_rate_ratio": (
                second_half_predicted_rate
                / first_half_predicted_rate
            ),
            "first_half_log_score_per_second": (
                first_half_log_score_per_second
            ),
            "second_half_log_score_per_second": (
                second_half_log_score_per_second
            ),
            "second_minus_first_log_score_per_second": (
                second_half_log_score_per_second
                - first_half_log_score_per_second
            ),
            "partial_final_block_included_in_full_reconciliation": True,
            "complete_blocks_only_used_for_stability_moments": True,
            "status": "PASS",
        }
    )


CHRONOLOGICAL_MODEL_STABILITY_SUMMARY = (
    pd.DataFrame(
        chronological_stability_rows
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "block_width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Scientific evidence ledger
# ------------------------------------------------------------

calibration_regime_summary = (
    CAUSAL_ACTIVITY_REGIME_SUMMARY.loc[
        CAUSAL_ACTIVITY_REGIME_SUMMARY[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .set_index(
        "activity_regime"
    )
)

require(
    set(
        calibration_regime_summary.index
    )
    == set(
        ACTIVITY_REGIME_LABELS
    ),
    (
        "CALIBRATION activity-regime summary does not contain "
        "LOW, NORMAL, and HIGH."
    ),
)

calibration_high_to_low_rate_ratio = (
    float(
        calibration_regime_summary.loc[
            "HIGH",
            "observed_pooled_rate_per_second",
        ]
    )
    / float(
        calibration_regime_summary.loc[
            "LOW",
            "observed_pooled_rate_per_second",
        ]
    )
)

calibration_burst_summary = (
    BURST_CONCENTRATION_SUMMARY.loc[
        BURST_CONCENTRATION_SUMMARY[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
)

require(
    len(
        calibration_burst_summary
    )
    == 1,
    "Expected one CALIBRATION burst-concentration row.",
)

calibration_burst_row = (
    calibration_burst_summary.iloc[0]
)

primary_calibration_serial_process_count = int(
    len(
        PRIMARY_BASELINE_CALIBRATION_SERIAL_EVIDENCE
    )
)

primary_calibration_repeated_serial_process_count = int(
    PRIMARY_BASELINE_CALIBRATION_SERIAL_EVIDENCE[
        "repeated_serial_dependence_flag"
    ].sum()
)

primary_calibration_repeated_cross_direction_count = int(
    PRIMARY_BASELINE_CALIBRATION_CROSS_EVIDENCE[
        "repeated_cross_dependence_flag"
    ].sum()
)

calibration_conditional_repeated_count = int(
    CALIBRATION_CONDITIONAL_RESPONSE_EVIDENCE[
        "repeated_positive_primary_excess_flag"
    ].sum()
)


DEPENDENCE_AND_STABILITY_EVIDENCE_LEDGER = pd.DataFrame(
    [
        {
            "evidence_channel": (
                "PRIMARY_BASELINE_SERIAL_RESIDUAL_DEPENDENCE"
            ),
            "observed_value": (
                primary_calibration_repeated_serial_process_count
            ),
            "reference_value": (
                primary_calibration_serial_process_count
            ),
            "evidence_flag": (
                primary_calibration_repeated_serial_process_count
                >= 1
            ),
            "interpretation": (
                "Repeated CALIBRATION serial dependence remains "
                "after the locked primary adaptive-rate baseline."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "PRIMARY_BASELINE_DIRECTIONAL_CROSS_DEPENDENCE"
            ),
            "observed_value": (
                primary_calibration_repeated_cross_direction_count
            ),
            "reference_value": 2,
            "evidence_flag": (
                primary_calibration_repeated_cross_direction_count
                >= 1
            ),
            "interpretation": (
                "Repeated directional residual cross-correlation "
                "is assessed separately from self-side persistence."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "CONDITIONAL_RESPONSE_EXCESS"
            ),
            "observed_value": (
                calibration_conditional_repeated_count
            ),
            "reference_value": 4,
            "evidence_flag": (
                calibration_conditional_repeated_count
                >= 1
            ),
            "interpretation": (
                "Source-centered response bands show repeated "
                "local excesses, but overlapping windows make "
                "this diagnostic only."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "CAUSAL_ACTIVITY_REGIME_SEPARATION"
            ),
            "observed_value": (
                calibration_high_to_low_rate_ratio
            ),
            "reference_value": 1.0,
            "evidence_flag": (
                calibration_high_to_low_rate_ratio
                > 1.0
            ),
            "interpretation": (
                "Development-frozen causal intensity regimes "
                "separate CALIBRATION activity levels."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "BURST_EVENT_CONCENTRATION"
            ),
            "observed_value": float(
                calibration_burst_row[
                    "burst_event_share"
                ]
            ),
            "reference_value": float(
                calibration_burst_row[
                    "burst_block_fraction"
                ]
            ),
            "evidence_flag": bool(
                calibration_burst_row[
                    "burst_event_share"
                ]
                > calibration_burst_row[
                    "burst_block_fraction"
                ]
            ),
            "interpretation": (
                "Development-frozen high-count blocks contain a "
                "disproportionate share of CALIBRATION events."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "CHRONOLOGICAL_SCORE_COVERAGE"
            ),
            "observed_value": int(
                CHRONOLOGICAL_SCORE_RECONCILIATION[
                    "status"
                ].eq(
                    "PASS"
                ).sum()
            ),
            "reference_value": int(
                len(
                    CHRONOLOGICAL_SCORE_RECONCILIATION
                )
            ),
            "evidence_flag": bool(
                CHRONOLOGICAL_SCORE_RECONCILIATION[
                    "status"
                ].eq(
                    "PASS"
                ).all()
            ),
            "interpretation": (
                "All chronological block scores, including each "
                "partial final block, reconcile with the full "
                "native-grid ledger."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
    ]
)


# ------------------------------------------------------------
# Acceptance gates
# ------------------------------------------------------------

expected_regime_rows = (
    len(
        ANALYTICAL_PARTITIONS
    )
    * len(
        ACTIVITY_REGIME_LABELS
    )
)

expected_chronological_reconciliation_rows = (
    len(
        CHRONOLOGICAL_STABILITY_MODEL_IDS
    )
    * len(
        ANALYTICAL_PARTITIONS
    )
    * len(
        CHRONOLOGICAL_STABILITY_WIDTHS_MS
    )
)

ACTIVITY_REGIME_BURST_STABILITY_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": (
                "activity_thresholds_selected_on_development_only"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not CAUSAL_ACTIVITY_REGIME_THRESHOLDS[
                    "calibration_used_for_threshold_selection"
                ].any()
            ),
            "evidence": (
                f"lower={activity_lower_threshold:.8f}; "
                f"upper={activity_upper_threshold:.8f}"
            ),
        },
        {
            "gate": (
                "activity_regime_summary_complete"
            ),
            "severity": "BLOCKING",
            "passed": (
                len(
                    CAUSAL_ACTIVITY_REGIME_SUMMARY
                )
                == expected_regime_rows
            ),
            "evidence": (
                f"regime_rows="
                f"{len(CAUSAL_ACTIVITY_REGIME_SUMMARY)}"
            ),
        },
        {
            "gate": (
                "activity_regime_native_ledger_conserved"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                CAUSAL_ACTIVITY_REGIME_CONSERVATION[
                    "status"
                ].eq(
                    "PASS"
                ).all()
            ),
            "evidence": (
                "exposure, events, predictions, and scores "
                "conserve the primary native-grid ledger"
            ),
        },
        {
            "gate": (
                "burst_threshold_selected_on_development_only"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not BURST_THRESHOLD_CONTRACT[
                    "calibration_used_for_threshold_selection"
                ].any()
            ),
            "evidence": (
                f"threshold={BURST_EVENT_COUNT_THRESHOLD} "
                f"events per {BURST_DIAGNOSTIC_WIDTH_MS} ms"
            ),
        },
        {
            "gate": (
                "burst_blocks_conserve_partition_exposure"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                all(
                    int(
                        BURST_BLOCK_TABLE.loc[
                            BURST_BLOCK_TABLE[
                                "event_partition"
                            ].eq(
                                partition_name
                            ),
                            "exposure_ns",
                        ].sum()
                    )
                    == NATIVE_COUNT_GRIDS[
                        partition_name
                    ].contract_duration_ns
                    for partition_name in (
                        ANALYTICAL_PARTITIONS
                    )
                )
            ),
            "evidence": (
                "all full and partial burst blocks conserve "
                "registered exposure"
            ),
        },
        {
            "gate": (
                "burst_blocks_conserve_partition_events"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                all(
                    int(
                        BURST_BLOCK_TABLE.loc[
                            BURST_BLOCK_TABLE[
                                "event_partition"
                            ].eq(
                                partition_name
                            ),
                            "pooled_event_count",
                        ].sum()
                    )
                    == int(
                        NATIVE_COUNT_GRIDS[
                            partition_name
                        ].pooled_counts.sum(
                            dtype="int64"
                        )
                    )
                    for partition_name in (
                        ANALYTICAL_PARTITIONS
                    )
                )
            ),
            "evidence": (
                "all full and partial burst blocks conserve "
                "pooled events"
            ),
        },
        {
            "gate": (
                "chronological_score_reconciliation_complete"
            ),
            "severity": "BLOCKING",
            "passed": (
                len(
                    CHRONOLOGICAL_SCORE_RECONCILIATION
                )
                == expected_chronological_reconciliation_rows
            ),
            "evidence": (
                f"reconciliation_rows="
                f"{len(CHRONOLOGICAL_SCORE_RECONCILIATION)}"
            ),
        },
        {
            "gate": (
                "chronological_scores_reconcile_exactly"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                CHRONOLOGICAL_SCORE_RECONCILIATION[
                    "status"
                ].eq(
                    "PASS"
                ).all()
            ),
            "evidence": (
                "every native cell is included exactly once; "
                "partial final blocks remain in reconciliation"
            ),
        },
        {
            "gate": (
                "chronological_stability_values_finite"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    CHRONOLOGICAL_MODEL_STABILITY_SUMMARY[
                        [
                            "observed_rate_coefficient_of_variation",
                            "predicted_rate_coefficient_of_variation",
                            "log_score_per_second_sample_std",
                            "second_to_first_observed_rate_ratio",
                            "second_to_first_predicted_rate_ratio",
                            "second_minus_first_log_score_per_second",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "all chronological rate and score stability "
                "statistics are finite"
            ),
        },
        {
            "gate": (
                "current_cell_events_excluded_from_activity_regimes"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                CAUSAL_ACTIVITY_REGIME_SUMMARY[
                    "current_cell_events_excluded_from_regime_forecast"
                ].all()
            ),
            "evidence": (
                "activity regimes use locked primary forecasts "
                "available before each native cell"
            ),
        },
        {
            "gate": (
                "timestamp_jitter_absent"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not BURST_CONCENTRATION_SUMMARY[
                    "timestamp_jitter_applied"
                ].any()
            ),
            "evidence": (
                "canonical Notebook 04 timestamps remain "
                "unchanged"
            ),
        },
        {
            "gate": (
                "protected_partition_content_remains_absent"
            ),
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    ACTIVITY_REGIME_BURST_STABILITY_GATE_FRAME.loc[
        ACTIVITY_REGIME_BURST_STABILITY_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one activity-regime, burst, or chronological "
        "stability gate failed."
    ),
)


display(
    CAUSAL_ACTIVITY_REGIME_THRESHOLDS
)

display(
    CAUSAL_ACTIVITY_REGIME_SUMMARY[
        [
            "event_partition",
            "activity_regime",
            "native_cell_count",
            "exposure_seconds",
            "exposure_share",
            "pooled_event_count",
            "pooled_event_share",
            "predicted_pooled_event_count",
            "observed_pooled_rate_per_second",
            "predicted_pooled_rate_per_second",
            "observed_to_predicted_event_ratio",
            "joint_log_score_per_second",
            "status",
        ]
    ]
)

display(
    CAUSAL_ACTIVITY_REGIME_CONSERVATION
)

display(
    BURST_THRESHOLD_CONTRACT
)

display(
    BURST_CONCENTRATION_SUMMARY
)

if not BURST_EPISODE_TABLE.empty:
    display(
        BURST_EPISODE_TABLE.head(
            30
        )
    )

display(
    CHRONOLOGICAL_MODEL_STABILITY_SUMMARY[
        [
            "model_id",
            "event_partition",
            "block_width_ms",
            "all_block_count",
            "complete_block_count",
            "partial_final_block_count",
            "observed_rate_coefficient_of_variation",
            "predicted_rate_coefficient_of_variation",
            "minimum_complete_block_log_score_per_second",
            "median_complete_block_log_score_per_second",
            "maximum_complete_block_log_score_per_second",
            "second_to_first_observed_rate_ratio",
            "second_to_first_predicted_rate_ratio",
            "second_minus_first_log_score_per_second",
            "status",
        ]
    ]
)

display(
    CHRONOLOGICAL_SCORE_RECONCILIATION
)

display(
    DEPENDENCE_AND_STABILITY_EVIDENCE_LEDGER
)

display(
    ACTIVITY_REGIME_BURST_STABILITY_GATE_FRAME
)

print(
    "Causal LOW, NORMAL, and HIGH activity regimes were frozen "
    "from DEVELOPMENT exposure-weighted primary-model intensities."
)
print(
    "The same thresholds were applied unchanged to CALIBRATION, "
    "and all regime summaries conserve native exposure, events, "
    "predictions, and log scores."
)
print(
    f"Burst episodes use a DEVELOPMENT-frozen threshold of "
    f"{BURST_EVENT_COUNT_THRESHOLD} events per "
    f"{BURST_DIAGNOSTIC_WIDTH_MS} milliseconds."
)
print(
    "Burst detection uses complete equal-exposure blocks; each "
    "partial final block is retained in the audit but excluded "
    "from threshold selection and episode detection."
)
print(
    "Chronological model stability was evaluated at 10-second, "
    "30-second, and 60-second scales."
)
print(
    "Every native cell, including the final partial cell, is "
    "assigned exactly once to a chronological block."
)
print(
    "Chronological exposure, observed counts, predicted counts, "
    "and log scores reconcile with the full unified native-grid "
    "ledger."
)
print(
    "The accumulated dependence and stability evidence remains "
    "diagnostic; Hawkes estimation remains unauthorized."
)

C:\Users\Bryan\AppData\Local\Temp\ipykernel_15324\4120095757.py:1527: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
C:\Users\Bryan\AppData\Local\Temp\ipykernel_15324\4120095757.py:1527: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,model_id,fit_partition,selection_partition,lower_quantile,upper_quantile,lower_intensity_threshold_per_second,upper_intensity_threshold_per_second,exposure_weighted_thresholds,calibration_used_for_threshold_selection,current_cell_events_excluded_from_forecast,status
0,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,DEVELOPMENT,0.33333333,0.66666667,3.1949126,4.0385345,True,False,True,PASS


,event_partition,activity_regime,native_cell_count,exposure_seconds,exposure_share,pooled_event_count,pooled_event_share,predicted_pooled_event_count,observed_pooled_rate_per_second,predicted_pooled_rate_per_second,observed_to_predicted_event_ratio,joint_log_score_per_second,status
0,CALIBRATION,LOW,299402,299.402,0.41566339,809,0.32450862,820.14816,2.7020528,2.7392875,0.98640713,-20.61769,PASS
1,CALIBRATION,NORMAL,229398,229.39718,0.31847486,685,0.27476935,820.51452,2.9860873,3.5768292,0.83484202,-22.551251,PASS
2,CALIBRATION,HIGH,191500,191.5,0.26586175,999,0.40072202,"1,032.8164",5.2167102,5.3932972,0.96725806,-35.124137,PASS
3,DEVELOPMENT,LOW,600620,600.62,0.33333341,1968,0.2809823,"1,666.6243",3.2766142,2.7748398,1.18083,-24.346566,PASS
4,DEVELOPMENT,NORMAL,600620,600.62,0.33333341,1967,0.28083952,"2,157.4721",3.2749492,3.592075,0.91171516,-24.355662,PASS
5,DEVELOPMENT,HIGH,600620,600.61959,0.33333318,3069,0.43817818,"3,179.6584",5.1097235,5.2939639,0.96519802,-34.758847,PASS


,event_partition,regime_exposure_sum_ns,contract_duration_ns,exposure_difference_ns,regime_event_sum,native_event_sum,event_difference,regime_predicted_sum,unified_predicted_count,predicted_difference,regime_score_sum,unified_log_score,score_difference,status
0,DEVELOPMENT,1801859586700,1801859586700,0,7004,7004,0,"7,003.7548","7,003.7548",1.8189894e-12,"-50,128.377","-50,128.377",-7.2759576e-12,PASS
1,CALIBRATION,720299179100,720299179100,0,2493,2493,0,"2,673.4791","2,673.4791",-4.5474735e-13,"-18,072.443","-18,072.443",7.2759576e-12,PASS


,selection_partition,block_width_ms,threshold_quantile,minimum_threshold_floor,selected_event_count_threshold,development_complete_block_count,calibration_used_for_threshold_selection,partial_final_block_excluded_from_threshold_selection,status
0,DEVELOPMENT,100,0.95,2,2,18018,False,True,PASS


...

,model_id,event_partition,block_width_ms,all_block_count,complete_block_count,partial_final_block_count,observed_rate_coefficient_of_variation,predicted_rate_coefficient_of_variation,minimum_complete_block_log_score_per_second,median_complete_block_log_score_per_second,maximum_complete_block_log_score_per_second,second_to_first_observed_rate_ratio,second_to_first_predicted_rate_ratio,second_minus_first_log_score_per_second,status
0,D0_SIDE_CONSTANT_POISSON,CALIBRATION,10000,73,72,1,0.37139091,1.150488e-16,-51.655595,-22.649908,-15.095336,1.2199466,1,-4.430834,PASS
1,D0_SIDE_CONSTANT_POISSON,CALIBRATION,30000,25,24,1,0.22497314,0,-40.140383,-26.021545,-17.390504,1.2199466,1,-4.430834,PASS
2,D0_SIDE_CONSTANT_POISSON,CALIBRATION,60000,13,12,1,0.17264129,1.1932716e-16,-31.506712,-25.424295,-19.918572,1.2199466,1,-4.430834,PASS
3,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,10000,73,72,1,0.37139091,0.13956258,-48.62242,-22.74345,-14.655113,1.2199466,1.0779349,-4.1859089,PASS
4,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,30000,25,24,1,0.22497314,0.084925533,-39.240019,-25.364634,-17.154275,1.2199466,1.0779349,-4.1859089,PASS
5,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,60000,13,12,1,0.17264129,0.065180509,-31.063278,-25.243998,-19.556151,1.2199466,1.0779349,-4.1859089,PASS
6,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,10000,181,180,1,0.36011988,2.2913148e-16,-69.465411,-25.776182,-15.110416,0.97293487,1,0.60236613,PASS
7,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,30000,61,60,1,0.25678471,1.1521119e-16,-47.296649,-26.572854,-19.068857,0.97293487,1,0.60236613,PASS
8,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,60000,31,30,1,0.20506641,2.3240028e-16,-45.959924,-27.570246,-22.192134,0.97293487,1,0.60236613,PASS
9,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,10000,181,180,1,0.36011988,0.14820714,-62.720323,-25.735377,-14.667634,0.97293487,0.98852508,0.91679417,PASS


,model_id,event_partition,block_width_ms,block_count,complete_block_count,partial_final_block_count,block_exposure_sum_ns,registered_exposure_ns,exposure_difference_ns,block_observed_event_sum,unified_observed_event_count,observed_event_difference,block_predicted_event_sum,unified_predicted_event_count,predicted_event_difference,block_log_score_sum,unified_log_score,log_score_difference,absolute_tolerance,full_and_partial_blocks_included,status
0,D0_SIDE_CONSTANT_POISSON,CALIBRATION,10000,73,72,1,720299179100,720299179100,0,2493,2493,0,"2,799.8716","2,799.8716",-2.2737368e-12,"-18,440.776","-18,440.776",-3.6379788e-12,1e-07,True,PASS
1,D0_SIDE_CONSTANT_POISSON,CALIBRATION,30000,25,24,1,720299179100,720299179100,0,2493,2493,0,"2,799.8716","2,799.8716",-1.8189894e-12,"-18,440.776","-18,440.776",0,1e-07,True,PASS
2,D0_SIDE_CONSTANT_POISSON,CALIBRATION,60000,13,12,1,720299179100,720299179100,0,2493,2493,0,"2,799.8716","2,799.8716",-1.3642421e-12,"-18,440.776","-18,440.776",0,1e-07,True,PASS
3,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,10000,73,72,1,720299179100,720299179100,0,2493,2493,0,"2,673.4791","2,673.4791",0,"-18,072.443","-18,072.443",3.6379788e-12,1e-07,True,PASS
4,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,30000,25,24,1,720299179100,720299179100,0,2493,2493,0,"2,673.4791","2,673.4791",-4.5474735e-13,"-18,072.443","-18,072.443",3.6379788e-12,1e-07,True,PASS
5,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,60000,13,12,1,720299179100,720299179100,0,2493,2493,0,"2,673.4791","2,673.4791",0,"-18,072.443","-18,072.443",7.2759576e-12,1e-07,True,PASS
6,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,10000,181,180,1,1801859586700,1801859586700,0,7004,7004,0,"7,004","7,004",-7.2759576e-12,"-50,831.392","-50,831.392",0,1e-07,True,PASS
7,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,30000,61,60,1,1801859586700,1801859586700,0,7004,7004,0,"7,004","7,004",-3.6379788e-12,"-50,831.392","-50,831.392",7.2759576e-12,1e-07,True,PASS
8,D0_SIDE_CONSTANT_POISSON,DEVELOPMENT,60000,31,30,1,1801859586700,1801859586700,0,7004,7004,0,"7,004","7,004",-8.1854523e-12,"-50,831.392","-50,831.392",7.2759576e-12,1e-07,True,PASS
9,E_SIDE_EWMA_250MS_POISSON,DEVELOPMENT,10000,181,180,1,1801859586700,1801859586700,0,7004,7004,0,"7,003.7548","7,003.7548",1.8189894e-12,"-50,128.377","-50,128.377",0,1e-07,True,PASS


,evidence_channel,observed_value,reference_value,evidence_flag,interpretation,formal_hawkes_authorization_flag,status
0,PRIMARY_BASELINE_SERIAL_RESIDUAL_DEPENDENCE,3,3,True,Repeated CALIBRATION serial dependence remains...,False,PASS
1,PRIMARY_BASELINE_DIRECTIONAL_CROSS_DEPENDENCE,0,2,False,Repeated directional residual cross-correlatio...,False,PASS
2,CONDITIONAL_RESPONSE_EXCESS,4,4,True,Source-centered response bands show repeated l...,False,PASS
3,CAUSAL_ACTIVITY_REGIME_SEPARATION,1.930647,1,True,Development-frozen causal intensity regimes se...,False,PASS
4,BURST_EVENT_CONCENTRATION,0.41476133,0.047209109,True,Development-frozen high-count blocks contain a...,False,PASS
5,CHRONOLOGICAL_SCORE_COVERAGE,12,12,True,"All chronological block scores, including each...",False,PASS


,gate,severity,passed,evidence
0,activity_thresholds_selected_on_development_only,BLOCKING,True,lower=3.19491258; upper=4.03853446
1,activity_regime_summary_complete,BLOCKING,True,regime_rows=6
2,activity_regime_native_ledger_conserved,BLOCKING,True,"exposure, events, predictions, and scores cons..."
3,burst_threshold_selected_on_development_only,BLOCKING,True,threshold=2 events per 100 ms
4,burst_blocks_conserve_partition_exposure,BLOCKING,True,all full and partial burst blocks conserve reg...
5,burst_blocks_conserve_partition_events,BLOCKING,True,all full and partial burst blocks conserve poo...
6,chronological_score_reconciliation_complete,BLOCKING,True,reconciliation_rows=12
7,chronological_scores_reconcile_exactly,BLOCKING,True,every native cell is included exactly once; pa...
8,chronological_stability_values_finite,BLOCKING,True,all chronological rate and score stability sta...
9,current_cell_events_excluded_from_activity_reg...,BLOCKING,True,activity regimes use locked primary forecasts ...


Causal LOW, NORMAL, and HIGH activity regimes were frozen from DEVELOPMENT exposure-weighted primary-model intensities.
The same thresholds were applied unchanged to CALIBRATION, and all regime summaries conserve native exposure, events, predictions, and log scores.
Burst episodes use a DEVELOPMENT-frozen threshold of 2 events per 100 milliseconds.
Burst detection uses complete equal-exposure blocks; each partial final block is retained in the audit but excluded from threshold selection and episode detection.
Chronological model stability was evaluated at 10-second, 30-second, and 60-second scales.
Every native cell, including the final partial cell, is assigned exactly once to a chronological block.
Chronological exposure, observed counts, predicted counts, and log scores reconcile with the full unified native-grid ledger.
The accumulated dependence and stability evidence remains diagnostic; Hawkes estimation remains unauthorized.


In [21]:
# ============================================================
# Randomized count-PIT and residual calibration audit
# ============================================================

COUNT_PIT_WIDTHS_MS: Final[
    tuple[int, ...]
] = (
    10,
    25,
    50,
    100,
    250,
    500,
    1_000,
    2_000,
    5_000,
)

COUNT_PIT_PROCESS_NAMES: Final[
    tuple[str, ...]
] = (
    "BUY",
    "SELL",
    "POOLED",
)

COUNT_PIT_MODEL_IDS: Final[
    tuple[str, ...]
] = tuple(
    dict.fromkeys(
        (
            FORMAL_COUNT_COMPARATOR_MODEL_ID,
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID,
        )
    )
)

COUNT_PIT_HISTOGRAM_EDGE_COUNT: Final[int] = 21

COUNT_PIT_REFERENCE_ALPHA: Final[float] = 0.05

COUNT_PIT_LAG_ONE_REFERENCE_MULTIPLIER: Final[float] = 1.96

COUNT_PIT_OPEN_ZERO: Final[float] = float(
    np.nextafter(
        0.0,
        1.0,
    )
)

COUNT_PIT_OPEN_ONE: Final[float] = float(
    np.nextafter(
        1.0,
        0.0,
    )
)

COUNT_PIT_HISTOGRAM_EDGES: Final[
    np.ndarray
] = np.linspace(
    0.0,
    1.0,
    COUNT_PIT_HISTOGRAM_EDGE_COUNT,
    dtype="float64",
)

COUNT_PIT_HISTOGRAM_EDGES.setflags(
    write=False
)


# ------------------------------------------------------------
# Complete-bin aggregation contract
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class CompleteCountBinAggregation:
    model_id: str
    event_partition: str
    process_name: str
    width_ms: int
    width_ns: int
    cells_per_bin: int
    complete_bin_count: int
    included_native_cell_count: int
    included_exposure_ns: int
    excluded_tail_exposure_ns: int
    included_observed_event_count: int
    excluded_observed_event_count: int
    included_predicted_event_count: float
    observed_counts: np.ndarray
    expected_counts: np.ndarray


def observed_native_counts(
    grid: NativeCountGrid,
    *,
    process_name: str,
) -> np.ndarray:
    """Return one native observed-count component."""
    if process_name == "BUY":
        counts = grid.buy_counts

    elif process_name == "SELL":
        counts = grid.sell_counts

    elif process_name == "POOLED":
        counts = grid.pooled_counts

    else:
        raise ValueError(
            f"Unsupported count process: {process_name}"
        )

    array = np.asarray(
        counts,
        dtype="int64",
    )

    require(
        array.shape == (grid.cell_count,),
        (
            f"{grid.event_partition} {process_name} observed "
            "native counts differ from the grid shape."
        ),
    )
    require(
        np.all(array >= 0),
        (
            f"{grid.event_partition} {process_name} observed "
            "counts contain negative values."
        ),
    )

    return array


def expected_native_counts(
    *,
    model_id: str,
    partition_name: str,
    process_name: str,
) -> np.ndarray:
    """Return one locked native expected-count component."""
    if process_name in {"BUY", "SELL"}:
        key = (
            model_id,
            partition_name,
            process_name,
        )

        require(
            key
            in COUNT_BASELINE_EXPECTED_COUNTS,
            (
                f"Missing expected-count array for "
                f"{model_id}, {partition_name}, "
                f"{process_name}."
            ),
        )

        expected = np.asarray(
            COUNT_BASELINE_EXPECTED_COUNTS[
                key
            ],
            dtype="float64",
        )

    elif process_name == "POOLED":
        buy_expected = expected_native_counts(
            model_id=model_id,
            partition_name=partition_name,
            process_name="BUY",
        )

        sell_expected = expected_native_counts(
            model_id=model_id,
            partition_name=partition_name,
            process_name="SELL",
        )

        expected = (
            buy_expected
            + sell_expected
        )

    else:
        raise ValueError(
            f"Unsupported expected-count process: {process_name}"
        )

    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    require(
        expected.shape == (grid.cell_count,),
        (
            f"{model_id} {partition_name} {process_name} "
            "expected counts differ from the native grid."
        ),
    )
    require(
        np.isfinite(expected).all(),
        (
            f"{model_id} {partition_name} {process_name} "
            "expected counts contain nonfinite values."
        ),
    )
    require(
        np.all(expected > 0.0),
        (
            f"{model_id} {partition_name} {process_name} "
            "expected counts are not strictly positive."
        ),
    )

    return expected


def aggregate_complete_native_bins(
    *,
    model_id: str,
    partition_name: str,
    process_name: str,
    width_ms: int,
) -> CompleteCountBinAggregation:
    """
    Aggregate complete equal-exposure bins from the native grid.

    The right-edge remainder is recorded but excluded from PIT diagnostics.
    The authoritative full-contract model score remains the native-grid score.
    """
    require(
        model_id in COUNT_PIT_MODEL_IDS,
        f"Unauthorized PIT model: {model_id}",
    )
    require(
        partition_name in ANALYTICAL_PARTITIONS,
        f"Unauthorized PIT partition: {partition_name}",
    )
    require(
        process_name in COUNT_PIT_PROCESS_NAMES,
        f"Unauthorized PIT process: {process_name}",
    )
    require(
        width_ms > 0,
        "PIT aggregation width must be positive.",
    )

    grid = NATIVE_COUNT_GRIDS[
        partition_name
    ]

    width_ns = (
        width_ms
        * NANOSECONDS_PER_MILLISECOND
    )

    require(
        width_ns % grid.grid_width_ns == 0,
        (
            f"{partition_name} {width_ms} ms is not an "
            "integer multiple of the native grid."
        ),
    )

    cells_per_bin = (
        width_ns
        // grid.grid_width_ns
    )

    complete_bin_count = (
        grid.contract_duration_ns
        // width_ns
    )

    require(
        complete_bin_count >= 2,
        (
            f"{partition_name} {width_ms} ms aggregation "
            "contains fewer than two complete bins."
        ),
    )

    included_native_cell_count = (
        complete_bin_count
        * cells_per_bin
    )

    require(
        included_native_cell_count
        <= grid.cell_count,
        (
            f"{partition_name} {width_ms} ms aggregation "
            "requires more cells than the native grid contains."
        ),
    )

    observed_native = observed_native_counts(
        grid,
        process_name=process_name,
    )

    expected_native = expected_native_counts(
        model_id=model_id,
        partition_name=partition_name,
        process_name=process_name,
    )

    included_observed_native = (
        observed_native[
            :included_native_cell_count
        ]
    )

    included_expected_native = (
        expected_native[
            :included_native_cell_count
        ]
    )

    included_exposure_native = (
        grid.exposure_ns[
            :included_native_cell_count
        ]
    )

    require(
        np.all(
            included_exposure_native
            == grid.grid_width_ns
        ),
        (
            f"{partition_name} {width_ms} ms complete-bin "
            "region contains a partial native cell."
        ),
    )

    observed_counts = (
        included_observed_native.reshape(
            complete_bin_count,
            cells_per_bin,
        )
        .sum(
            axis=1,
            dtype="int64",
        )
    )

    expected_counts = (
        included_expected_native.reshape(
            complete_bin_count,
            cells_per_bin,
        )
        .sum(
            axis=1,
            dtype="float64",
        )
    )

    exposure_by_bin_ns = (
        included_exposure_native.reshape(
            complete_bin_count,
            cells_per_bin,
        )
        .sum(
            axis=1,
            dtype="int64",
        )
    )

    require(
        np.all(
            exposure_by_bin_ns
            == width_ns
        ),
        (
            f"{partition_name} {width_ms} ms aggregation "
            "contains unequal complete-bin exposure."
        ),
    )

    require(
        observed_counts.shape
        == expected_counts.shape
        == (
            complete_bin_count,
        ),
        "Aggregated observed and expected count shapes differ.",
    )
    require(
        np.all(observed_counts >= 0),
        "Aggregated observed counts contain negative values.",
    )
    require(
        np.isfinite(expected_counts).all()
        and np.all(expected_counts > 0.0),
        (
            "Aggregated expected counts must be finite and "
            "strictly positive."
        ),
    )

    included_exposure_ns = int(
        exposure_by_bin_ns.sum(
            dtype="int64"
        )
    )

    excluded_tail_exposure_ns = (
        grid.contract_duration_ns
        - included_exposure_ns
    )

    excluded_observed_event_count = int(
        observed_native[
            included_native_cell_count:
        ].sum(
            dtype="int64"
        )
    )

    require(
        included_exposure_ns
        + excluded_tail_exposure_ns
        == grid.contract_duration_ns,
        (
            f"{partition_name} {width_ms} ms aggregation "
            "does not conserve exact contract exposure."
        ),
    )

    require(
        int(
            observed_counts.sum(
                dtype="int64"
            )
        )
        + excluded_observed_event_count
        == int(
            observed_native.sum(
                dtype="int64"
            )
        ),
        (
            f"{partition_name} {width_ms} ms aggregation "
            "does not conserve observed events."
        ),
    )

    observed_counts.setflags(
        write=False
    )

    expected_counts.setflags(
        write=False
    )

    return CompleteCountBinAggregation(
        model_id=model_id,
        event_partition=partition_name,
        process_name=process_name,
        width_ms=width_ms,
        width_ns=width_ns,
        cells_per_bin=(
            cells_per_bin
        ),
        complete_bin_count=int(
            complete_bin_count
        ),
        included_native_cell_count=int(
            included_native_cell_count
        ),
        included_exposure_ns=(
            included_exposure_ns
        ),
        excluded_tail_exposure_ns=int(
            excluded_tail_exposure_ns
        ),
        included_observed_event_count=int(
            observed_counts.sum(
                dtype="int64"
            )
        ),
        excluded_observed_event_count=(
            excluded_observed_event_count
        ),
        included_predicted_event_count=float(
            expected_counts.sum(
                dtype="float64"
            )
        ),
        observed_counts=(
            observed_counts
        ),
        expected_counts=(
            expected_counts
        ),
    )


# ------------------------------------------------------------
# Numerically stable randomized Poisson PIT
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class RandomizedPoissonPITResult:
    pit_values: np.ndarray
    normal_residuals: np.ndarray
    log_probability_mass: np.ndarray
    probability_mass_underflow_count: int
    maximum_cdf_partition_error: float


def deterministic_pit_seed(
    *,
    model_id: str,
    partition_name: str,
    process_name: str,
    width_ms: int,
) -> int:
    """Create a stable 32-bit seed from the frozen run identity."""
    payload = (
        f"{RUN_IDENTITY_SHA256}|"
        f"{NOTEBOOK_CONFIG_SHA256}|"
        f"{model_id}|"
        f"{partition_name}|"
        f"{process_name}|"
        f"{width_ms}"
    ).encode(
        "utf-8"
    )

    digest = hashlib.sha256(
        payload
    ).digest()

    return int.from_bytes(
        digest[:4],
        byteorder="big",
        signed=False,
    )


def randomized_poisson_pit(
    observed_counts: np.ndarray,
    expected_counts: np.ndarray,
    *,
    seed: int,
) -> RandomizedPoissonPITResult:
    """
    Construct randomized Poisson PIT values without subtracting CDFs.

    For observation y and mean mu:

        U = F(y - 1; mu) + V * P(Y = y)

    The point probability is evaluated from log-PMF rather than
    F(y) - F(y - 1), avoiding catastrophic cancellation when both
    CDF values round to the same floating-point number.

    A survival-function representation is used in the upper tail.
    """
    observed = np.asarray(
        observed_counts,
        dtype="int64",
    )

    expected = np.asarray(
        expected_counts,
        dtype="float64",
    )

    require(
        observed.ndim == 1,
        "Poisson PIT observations must be one-dimensional.",
    )
    require(
        observed.shape == expected.shape,
        "Poisson PIT observed and expected arrays differ in shape.",
    )
    require(
        observed.size >= 2,
        "Poisson PIT requires at least two observations.",
    )
    require(
        np.all(observed >= 0),
        "Poisson PIT observations contain negative counts.",
    )
    require(
        np.isfinite(expected).all()
        and np.all(expected > 0.0),
        (
            "Poisson PIT expected counts must be finite and "
            "strictly positive."
        ),
    )

    random_uniform = (
        np.random.default_rng(
            seed
        )
        .random(
            observed.size
        )
    )

    log_probability_mass = (
        stats.poisson.logpmf(
            observed,
            expected,
        )
    )

    require(
        np.isfinite(
            log_probability_mass
        ).all(),
        "Poisson PIT log-PMF contains nonfinite entries.",
    )

    probability_mass = np.exp(
        log_probability_mass
    )

    probability_mass_underflow_count = int(
        np.count_nonzero(
            probability_mass == 0.0
        )
    )

    lower_cdf = np.zeros(
        observed.size,
        dtype="float64",
    )

    positive_observation_mask = (
        observed > 0
    )

    if positive_observation_mask.any():
        lower_cdf[
            positive_observation_mask
        ] = stats.poisson.cdf(
            observed[
                positive_observation_mask
            ]
            - 1,
            expected[
                positive_observation_mask
            ],
        )

    upper_survival = stats.poisson.sf(
        observed,
        expected,
    )

    require(
        np.isfinite(lower_cdf).all()
        and np.isfinite(upper_survival).all(),
        "Poisson PIT tail probabilities contain nonfinite entries.",
    )
    require(
        np.all(
            (lower_cdf >= 0.0)
            & (lower_cdf <= 1.0)
        ),
        "Poisson PIT lower-tail CDF lies outside [0, 1].",
    )
    require(
        np.all(
            (upper_survival >= 0.0)
            & (upper_survival <= 1.0)
        ),
        "Poisson PIT upper survival lies outside [0, 1].",
    )

    cdf_partition = (
        lower_cdf
        + probability_mass
        + upper_survival
    )

    maximum_cdf_partition_error = float(
        np.max(
            np.abs(
                cdf_partition
                - 1.0
            )
        )
    )

    lower_tail_pit = (
        lower_cdf
        + random_uniform
        * probability_mass
    )

    upper_tail_pit = (
        1.0
        - (
            upper_survival
            + (
                1.0
                - random_uniform
            )
            * probability_mass
        )
    )

    use_lower_representation = (
        lower_cdf
        <= upper_survival
    )

    pit_values = np.where(
        use_lower_representation,
        lower_tail_pit,
        upper_tail_pit,
    )

    pit_values = np.clip(
        pit_values,
        COUNT_PIT_OPEN_ZERO,
        COUNT_PIT_OPEN_ONE,
    )

    normal_residuals = stats.norm.ppf(
        pit_values
    )

    require(
        np.isfinite(pit_values).all(),
        "Randomized Poisson PIT values contain nonfinite entries.",
    )
    require(
        np.all(
            (pit_values > 0.0)
            & (pit_values < 1.0)
        ),
        "Randomized Poisson PIT values are not strictly inside (0, 1).",
    )
    require(
        np.isfinite(
            normal_residuals
        ).all(),
        "Randomized Poisson normal residuals contain nonfinite values.",
    )

    pit_values.setflags(
        write=False
    )

    normal_residuals.setflags(
        write=False
    )

    log_probability_mass.setflags(
        write=False
    )

    return RandomizedPoissonPITResult(
        pit_values=pit_values,
        normal_residuals=normal_residuals,
        log_probability_mass=(
            log_probability_mass
        ),
        probability_mass_underflow_count=(
            probability_mass_underflow_count
        ),
        maximum_cdf_partition_error=(
            maximum_cdf_partition_error
        ),
    )


# ------------------------------------------------------------
# Distribution and serial-diagnostic helpers
# ------------------------------------------------------------

def uniform_ks_distance(
    values: np.ndarray,
) -> float:
    """Return the one-sample Kolmogorov distance from Uniform(0, 1)."""
    array = np.asarray(
        values,
        dtype="float64",
    )

    require(
        array.ndim == 1,
        "Uniform KS input must be one-dimensional.",
    )
    require(
        array.size >= 2,
        "Uniform KS distance requires at least two observations.",
    )
    require(
        np.isfinite(array).all(),
        "Uniform KS input contains nonfinite values.",
    )
    require(
        np.all(
            (array >= 0.0)
            & (array <= 1.0)
        ),
        "Uniform KS input lies outside [0, 1].",
    )

    ordered = np.sort(
        array,
        kind="stable",
    )

    sample_size = ordered.size

    upper_empirical = (
        np.arange(
            1,
            sample_size + 1,
            dtype="float64",
        )
        / sample_size
    )

    lower_empirical = (
        np.arange(
            0,
            sample_size,
            dtype="float64",
        )
        / sample_size
    )

    distance = max(
        float(
            np.max(
                upper_empirical
                - ordered
            )
        ),
        float(
            np.max(
                ordered
                - lower_empirical
            )
        ),
    )

    require(
        np.isfinite(distance)
        and 0.0 <= distance <= 1.0,
        "Uniform KS distance is invalid.",
    )

    return distance


def uniform_cramer_von_mises_statistic(
    values: np.ndarray,
) -> float:
    """Return the one-sample Cramér–von Mises statistic."""
    array = np.asarray(
        values,
        dtype="float64",
    )

    require(
        array.ndim == 1,
        "Cramér–von Mises input must be one-dimensional.",
    )
    require(
        array.size >= 2,
        (
            "Cramér–von Mises statistic requires at least "
            "two observations."
        ),
    )

    ordered = np.sort(
        array,
        kind="stable",
    )

    sample_size = ordered.size

    theoretical_positions = (
        (
            2.0
            * np.arange(
                1,
                sample_size + 1,
                dtype="float64",
            )
            - 1.0
        )
        / (
            2.0
            * sample_size
        )
    )

    statistic = (
        1.0
        / (
            12.0
            * sample_size
        )
        + float(
            np.sum(
                (
                    ordered
                    - theoretical_positions
                )
                ** 2,
                dtype="float64",
            )
        )
    )

    require(
        np.isfinite(statistic)
        and statistic >= 0.0,
        "Cramér–von Mises statistic is invalid.",
    )

    return statistic


def lag_one_sample_correlation(
    values: np.ndarray,
) -> float:
    """Return lag-one sample correlation, or zero for degenerate input."""
    array = np.asarray(
        values,
        dtype="float64",
    )

    require(
        array.ndim == 1,
        "Lag-one correlation input must be one-dimensional.",
    )
    require(
        array.size >= 3,
        "Lag-one correlation requires at least three observations.",
    )
    require(
        np.isfinite(array).all(),
        "Lag-one correlation input contains nonfinite values.",
    )

    lagged = array[:-1]
    current = array[1:]

    lagged_std = float(
        np.std(
            lagged,
            ddof=1,
        )
    )

    current_std = float(
        np.std(
            current,
            ddof=1,
        )
    )

    if (
        lagged_std == 0.0
        or current_std == 0.0
    ):
        return 0.0

    correlation = float(
        np.corrcoef(
            lagged,
            current,
        )[0, 1]
    )

    require(
        np.isfinite(correlation),
        "Lag-one correlation result is nonfinite.",
    )

    return correlation


# ------------------------------------------------------------
# Randomized PIT replay
# ------------------------------------------------------------

count_pit_summary_rows: list[
    dict[str, Any]
] = []

count_pit_histogram_rows: list[
    dict[str, Any]
] = []

count_pit_coverage_rows: list[
    dict[str, Any]
] = []


for model_id in COUNT_PIT_MODEL_IDS:
    for partition_name in ANALYTICAL_PARTITIONS:
        grid = NATIVE_COUNT_GRIDS[
            partition_name
        ]

        for process_name in COUNT_PIT_PROCESS_NAMES:
            full_observed_event_count = int(
                observed_native_counts(
                    grid,
                    process_name=process_name,
                ).sum(
                    dtype="int64"
                )
            )

            for width_ms in COUNT_PIT_WIDTHS_MS:
                aggregation = (
                    aggregate_complete_native_bins(
                        model_id=model_id,
                        partition_name=partition_name,
                        process_name=process_name,
                        width_ms=width_ms,
                    )
                )

                pit_seed = deterministic_pit_seed(
                    model_id=model_id,
                    partition_name=partition_name,
                    process_name=process_name,
                    width_ms=width_ms,
                )

                pit_result = randomized_poisson_pit(
                    aggregation.observed_counts,
                    aggregation.expected_counts,
                    seed=pit_seed,
                )

                pit_values = (
                    pit_result.pit_values
                )

                normal_residuals = (
                    pit_result.normal_residuals
                )

                sample_size = int(
                    pit_values.size
                )

                pit_ks_distance = (
                    uniform_ks_distance(
                        pit_values
                    )
                )

                pit_ks_reference_95 = (
                    1.36
                    / math.sqrt(
                        sample_size
                    )
                )

                pit_cvm_statistic = (
                    uniform_cramer_von_mises_statistic(
                        pit_values
                    )
                )

                lag_one_correlation = (
                    lag_one_sample_correlation(
                        normal_residuals
                    )
                )

                lag_one_reference_95 = (
                    COUNT_PIT_LAG_ONE_REFERENCE_MULTIPLIER
                    / math.sqrt(
                        sample_size
                    )
                )

                histogram_counts, _ = np.histogram(
                    pit_values,
                    bins=COUNT_PIT_HISTOGRAM_EDGES,
                )

                expected_histogram_count = (
                    sample_size
                    / (
                        COUNT_PIT_HISTOGRAM_EDGE_COUNT
                        - 1
                    )
                )

                histogram_maximum_relative_deviation = float(
                    np.max(
                        np.abs(
                            histogram_counts.astype(
                                "float64"
                            )
                            - expected_histogram_count
                        )
                    )
                    / expected_histogram_count
                )

                reference_ks_result = stats.kstest(
                    pit_values,
                    "uniform",
                )

                normal_mean = float(
                    np.mean(
                        normal_residuals
                    )
                )

                normal_variance = float(
                    np.var(
                        normal_residuals,
                        ddof=1,
                    )
                )

                normal_skewness = float(
                    stats.skew(
                        normal_residuals,
                        bias=False,
                    )
                )

                normal_excess_kurtosis = float(
                    stats.kurtosis(
                        normal_residuals,
                        fisher=True,
                        bias=False,
                    )
                )

                normal_quantiles = np.quantile(
                    normal_residuals,
                    (
                        0.01,
                        0.05,
                        0.50,
                        0.95,
                        0.99,
                    ),
                )

                count_pit_summary_rows.append(
                    {
                        "model_id": model_id,
                        "model_family": (
                            count_model_family(
                                model_id
                            )
                        ),
                        "event_partition": (
                            partition_name
                        ),
                        "process_name": (
                            process_name
                        ),
                        "width_ms": (
                            width_ms
                        ),
                        "complete_bin_count": (
                            aggregation.complete_bin_count
                        ),
                        "included_exposure_ns": (
                            aggregation.included_exposure_ns
                        ),
                        "excluded_tail_exposure_ns": (
                            aggregation.excluded_tail_exposure_ns
                        ),
                        "excluded_tail_fraction": (
                            aggregation.excluded_tail_exposure_ns
                            / grid.contract_duration_ns
                        ),
                        "included_observed_event_count": (
                            aggregation.included_observed_event_count
                        ),
                        "excluded_observed_event_count": (
                            aggregation.excluded_observed_event_count
                        ),
                        "full_observed_event_count": (
                            full_observed_event_count
                        ),
                        "included_predicted_event_count": (
                            aggregation.included_predicted_event_count
                        ),
                        "pit_seed": pit_seed,
                        "pit_mean": float(
                            np.mean(
                                pit_values
                            )
                        ),
                        "pit_sample_variance": float(
                            np.var(
                                pit_values,
                                ddof=1,
                            )
                        ),
                        "pit_ks_distance": (
                            pit_ks_distance
                        ),
                        "pit_ks_reference_95": (
                            pit_ks_reference_95
                        ),
                        "pit_ks_exceeds_reference_95": (
                            pit_ks_distance
                            > pit_ks_reference_95
                        ),
                        "pit_reference_p_value": float(
                            reference_ks_result.pvalue
                        ),
                        "pit_reference_p_value_formal": False,
                        "pit_cramer_von_mises_statistic": (
                            pit_cvm_statistic
                        ),
                        "pit_histogram_maximum_relative_deviation": (
                            histogram_maximum_relative_deviation
                        ),
                        "normal_residual_mean": (
                            normal_mean
                        ),
                        "normal_residual_sample_variance": (
                            normal_variance
                        ),
                        "normal_residual_skewness": (
                            normal_skewness
                        ),
                        "normal_residual_excess_kurtosis": (
                            normal_excess_kurtosis
                        ),
                        "normal_residual_q01": float(
                            normal_quantiles[0]
                        ),
                        "normal_residual_q05": float(
                            normal_quantiles[1]
                        ),
                        "normal_residual_median": float(
                            normal_quantiles[2]
                        ),
                        "normal_residual_q95": float(
                            normal_quantiles[3]
                        ),
                        "normal_residual_q99": float(
                            normal_quantiles[4]
                        ),
                        "normal_residual_lag_one_correlation": (
                            lag_one_correlation
                        ),
                        "lag_one_reference_95": (
                            lag_one_reference_95
                        ),
                        "absolute_lag_one_exceeds_reference_95": (
                            abs(
                                lag_one_correlation
                            )
                            > lag_one_reference_95
                        ),
                        "absolute_normal_residual_above_3_count": int(
                            np.count_nonzero(
                                np.abs(
                                    normal_residuals
                                )
                                > 3.0
                            )
                        ),
                        "probability_mass_computed_from_logpmf": True,
                        "cdf_subtraction_used_for_probability_mass": False,
                        "upper_tail_survival_representation_used": True,
                        "probability_mass_underflow_count": (
                            pit_result.probability_mass_underflow_count
                        ),
                        "maximum_cdf_partition_error": (
                            pit_result.maximum_cdf_partition_error
                        ),
                        "score_space_comparable_to_native_grid": False,
                        "diagnostic_role": (
                            "RANDOMIZED_COUNT_PIT_DIAGNOSTIC_ONLY"
                        ),
                        "status": "PASS",
                    }
                )

                for histogram_bin_index, count in enumerate(
                    histogram_counts
                ):
                    lower_edge = float(
                        COUNT_PIT_HISTOGRAM_EDGES[
                            histogram_bin_index
                        ]
                    )

                    upper_edge = float(
                        COUNT_PIT_HISTOGRAM_EDGES[
                            histogram_bin_index
                            + 1
                        ]
                    )

                    count_pit_histogram_rows.append(
                        {
                            "model_id": model_id,
                            "event_partition": (
                                partition_name
                            ),
                            "process_name": (
                                process_name
                            ),
                            "width_ms": (
                                width_ms
                            ),
                            "histogram_bin_index": (
                                histogram_bin_index
                            ),
                            "pit_lower_edge_inclusive": (
                                lower_edge
                            ),
                            "pit_upper_edge_exclusive": (
                                upper_edge
                            ),
                            "observed_bin_count": int(
                                count
                            ),
                            "expected_uniform_bin_count": (
                                expected_histogram_count
                            ),
                            "observed_minus_expected_count": (
                                count
                                - expected_histogram_count
                            ),
                            "observed_to_expected_ratio": (
                                count
                                / expected_histogram_count
                            ),
                            "status": "PASS",
                        }
                    )

                count_pit_coverage_rows.append(
                    {
                        "model_id": model_id,
                        "event_partition": (
                            partition_name
                        ),
                        "process_name": (
                            process_name
                        ),
                        "width_ms": (
                            width_ms
                        ),
                        "complete_bin_count": (
                            aggregation.complete_bin_count
                        ),
                        "included_exposure_ns": (
                            aggregation.included_exposure_ns
                        ),
                        "excluded_tail_exposure_ns": (
                            aggregation.excluded_tail_exposure_ns
                        ),
                        "contract_duration_ns": (
                            grid.contract_duration_ns
                        ),
                        "exact_exposure_conserved": (
                            aggregation.included_exposure_ns
                            + aggregation.excluded_tail_exposure_ns
                            == grid.contract_duration_ns
                        ),
                        "included_observed_event_count": (
                            aggregation.included_observed_event_count
                        ),
                        "excluded_observed_event_count": (
                            aggregation.excluded_observed_event_count
                        ),
                        "full_observed_event_count": (
                            full_observed_event_count
                        ),
                        "event_count_conserved": (
                            aggregation.included_observed_event_count
                            + aggregation.excluded_observed_event_count
                            == full_observed_event_count
                        ),
                        "status": "PASS",
                    }
                )


COUNT_RANDOMIZED_PIT_SUMMARY = (
    pd.DataFrame(
        count_pit_summary_rows
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "process_name",
            "width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

COUNT_RANDOMIZED_PIT_HISTOGRAM = (
    pd.DataFrame(
        count_pit_histogram_rows
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "process_name",
            "width_ms",
            "histogram_bin_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

COUNT_RANDOMIZED_PIT_COVERAGE = (
    pd.DataFrame(
        count_pit_coverage_rows
    )
    .sort_values(
        [
            "event_partition",
            "model_id",
            "process_name",
            "width_ms",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Model-level residual evidence
# ------------------------------------------------------------

COUNT_PIT_MODEL_EVIDENCE = (
    COUNT_RANDOMIZED_PIT_SUMMARY.groupby(
        [
            "model_id",
            "model_family",
            "event_partition",
            "process_name",
        ],
        observed=True,
        sort=False,
    )
    .agg(
        evaluated_width_count=(
            "width_ms",
            "size",
        ),
        pit_ks_exceedance_count=(
            "pit_ks_exceeds_reference_95",
            "sum",
        ),
        serial_exceedance_count=(
            "absolute_lag_one_exceeds_reference_95",
            "sum",
        ),
        probability_mass_underflow_count=(
            "probability_mass_underflow_count",
            "sum",
        ),
        minimum_pit_ks_distance=(
            "pit_ks_distance",
            "min",
        ),
        median_pit_ks_distance=(
            "pit_ks_distance",
            "median",
        ),
        maximum_pit_ks_distance=(
            "pit_ks_distance",
            "max",
        ),
        median_pit_mean=(
            "pit_mean",
            "median",
        ),
        median_pit_variance=(
            "pit_sample_variance",
            "median",
        ),
        median_normal_residual_mean=(
            "normal_residual_mean",
            "median",
        ),
        median_normal_residual_variance=(
            "normal_residual_sample_variance",
            "median",
        ),
        median_absolute_lag_one_correlation=(
            "normal_residual_lag_one_correlation",
            lambda values: float(
                np.median(
                    np.abs(
                        values.to_numpy(
                            dtype="float64"
                        )
                    )
                )
            ),
        ),
        maximum_cdf_partition_error=(
            "maximum_cdf_partition_error",
            "max",
        ),
    )
    .reset_index()
)

COUNT_PIT_MODEL_EVIDENCE[
    "pit_ks_exceedance_fraction"
] = (
    COUNT_PIT_MODEL_EVIDENCE[
        "pit_ks_exceedance_count"
    ]
    / COUNT_PIT_MODEL_EVIDENCE[
        "evaluated_width_count"
    ]
)

COUNT_PIT_MODEL_EVIDENCE[
    "serial_exceedance_fraction"
] = (
    COUNT_PIT_MODEL_EVIDENCE[
        "serial_exceedance_count"
    ]
    / COUNT_PIT_MODEL_EVIDENCE[
        "evaluated_width_count"
    ]
)

COUNT_PIT_MODEL_EVIDENCE[
    "repeated_pit_miscalibration_flag"
] = (
    COUNT_PIT_MODEL_EVIDENCE[
        "pit_ks_exceedance_count"
    ]
    >= 2
)

COUNT_PIT_MODEL_EVIDENCE[
    "repeated_serial_dependence_flag"
] = (
    COUNT_PIT_MODEL_EVIDENCE[
        "serial_exceedance_count"
    ]
    >= 2
)

COUNT_PIT_MODEL_EVIDENCE[
    "formal_p_value_interpretation_allowed"
] = False

COUNT_PIT_MODEL_EVIDENCE[
    "status"
] = "PASS"


CALIBRATION_COUNT_PIT_MODEL_COMPARISON = (
    COUNT_PIT_MODEL_EVIDENCE.loc[
        COUNT_PIT_MODEL_EVIDENCE[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .pivot(
        index="process_name",
        columns="model_id",
        values=[
            "median_pit_ks_distance",
            "pit_ks_exceedance_count",
            "median_normal_residual_variance",
            "median_absolute_lag_one_correlation",
            "serial_exceedance_count",
        ],
    )
)

CALIBRATION_COUNT_PIT_MODEL_COMPARISON.columns = [
    (
        f"{metric}__{model_id}"
    )
    for metric, model_id in (
        CALIBRATION_COUNT_PIT_MODEL_COMPARISON.columns
    )
]

CALIBRATION_COUNT_PIT_MODEL_COMPARISON = (
    CALIBRATION_COUNT_PIT_MODEL_COMPARISON
    .reset_index()
)

formal_ks_column = (
    "median_pit_ks_distance__"
    f"{FORMAL_COUNT_COMPARATOR_MODEL_ID}"
)

primary_ks_column = (
    "median_pit_ks_distance__"
    f"{PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID}"
)

formal_serial_column = (
    "median_absolute_lag_one_correlation__"
    f"{FORMAL_COUNT_COMPARATOR_MODEL_ID}"
)

primary_serial_column = (
    "median_absolute_lag_one_correlation__"
    f"{PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID}"
)

require(
    formal_ks_column
    in CALIBRATION_COUNT_PIT_MODEL_COMPARISON.columns,
    "Formal comparator PIT column is missing.",
)
require(
    primary_ks_column
    in CALIBRATION_COUNT_PIT_MODEL_COMPARISON.columns,
    "Primary baseline PIT column is missing.",
)
require(
    formal_serial_column
    in CALIBRATION_COUNT_PIT_MODEL_COMPARISON.columns,
    "Formal comparator serial column is missing.",
)
require(
    primary_serial_column
    in CALIBRATION_COUNT_PIT_MODEL_COMPARISON.columns,
    "Primary baseline serial column is missing.",
)

CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
    "primary_minus_formal_median_ks_distance"
] = (
    CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        primary_ks_column
    ]
    - CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        formal_ks_column
    ]
)

CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
    "primary_minus_formal_median_absolute_lag_one"
] = (
    CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        primary_serial_column
    ]
    - CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        formal_serial_column
    ]
)

CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
    "primary_has_lower_median_ks_distance"
] = (
    CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        primary_ks_column
    ]
    < CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        formal_ks_column
    ]
)

CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
    "primary_has_lower_median_absolute_lag_one"
] = (
    CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        primary_serial_column
    ]
    < CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
        formal_serial_column
    ]
)

CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
    "comparison_role"
] = (
    "LOCKED_CALIBRATION_RESIDUAL_DIAGNOSTIC_ONLY"
)

CALIBRATION_COUNT_PIT_MODEL_COMPARISON[
    "status"
] = "PASS"


# ------------------------------------------------------------
# Residual-audit registry
# ------------------------------------------------------------

primary_calibration_pit_evidence = (
    COUNT_PIT_MODEL_EVIDENCE.loc[
        COUNT_PIT_MODEL_EVIDENCE[
            "model_id"
        ].eq(
            PRIMARY_SIMPLE_COUNT_BASELINE_MODEL_ID
        )
        & COUNT_PIT_MODEL_EVIDENCE[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

require(
    set(
        primary_calibration_pit_evidence[
            "process_name"
        ]
    )
    == set(
        COUNT_PIT_PROCESS_NAMES
    ),
    (
        "Primary CALIBRATION PIT evidence does not contain "
        "BUY, SELL, and POOLED processes."
    ),
)

RESIDUAL_AUDIT_EVIDENCE_LEDGER = pd.DataFrame(
    [
        {
            "evidence_channel": (
                "PRIMARY_COUNT_PIT_CALIBRATION"
            ),
            "observed_value": int(
                primary_calibration_pit_evidence[
                    "repeated_pit_miscalibration_flag"
                ].sum()
            ),
            "reference_value": int(
                len(
                    primary_calibration_pit_evidence
                )
            ),
            "evidence_flag": bool(
                primary_calibration_pit_evidence[
                    "repeated_pit_miscalibration_flag"
                ].any()
            ),
            "interpretation": (
                "Randomized count PIT departures remain for at "
                "least one CALIBRATION process across multiple "
                "aggregation widths."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "PRIMARY_COUNT_PIT_SERIAL_DEPENDENCE"
            ),
            "observed_value": int(
                primary_calibration_pit_evidence[
                    "repeated_serial_dependence_flag"
                ].sum()
            ),
            "reference_value": int(
                len(
                    primary_calibration_pit_evidence
                )
            ),
            "evidence_flag": bool(
                primary_calibration_pit_evidence[
                    "repeated_serial_dependence_flag"
                ].any()
            ),
            "interpretation": (
                "Lag-one dependence in randomized normal count "
                "residuals is recorded separately from marginal "
                "PIT calibration."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "POISSON_PIT_NUMERICAL_STABILITY"
            ),
            "observed_value": int(
                COUNT_RANDOMIZED_PIT_SUMMARY[
                    "probability_mass_underflow_count"
                ].sum()
            ),
            "reference_value": 0,
            "evidence_flag": bool(
                COUNT_RANDOMIZED_PIT_SUMMARY[
                    "probability_mass_computed_from_logpmf"
                ].all()
                and not COUNT_RANDOMIZED_PIT_SUMMARY[
                    "cdf_subtraction_used_for_probability_mass"
                ].any()
            ),
            "interpretation": (
                "Poisson point masses are evaluated from log-PMF "
                "and PIT values use lower-tail or survival-tail "
                "representations without CDF subtraction."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "RENEWAL_TIME_RESCALING_ALREADY_REGISTERED"
            ),
            "observed_value": 1,
            "reference_value": 1,
            "evidence_flag": True,
            "interpretation": (
                "Selected renewal-duration time-rescaling "
                "diagnostics remain in their separate duration "
                "likelihood score space and are not merged with "
                "count PIT scores."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
    ]
)


# ------------------------------------------------------------
# Acceptance gates
# ------------------------------------------------------------

expected_pit_summary_rows = (
    len(
        COUNT_PIT_MODEL_IDS
    )
    * len(
        ANALYTICAL_PARTITIONS
    )
    * len(
        COUNT_PIT_PROCESS_NAMES
    )
    * len(
        COUNT_PIT_WIDTHS_MS
    )
)

expected_pit_histogram_rows = (
    expected_pit_summary_rows
    * (
        COUNT_PIT_HISTOGRAM_EDGE_COUNT
        - 1
    )
)

COUNT_PIT_RESIDUAL_AUDIT_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": (
                "count_pit_candidate_grid_complete"
            ),
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_RANDOMIZED_PIT_SUMMARY
                )
                == expected_pit_summary_rows
            ),
            "evidence": (
                f"pit_summary_rows="
                f"{len(COUNT_RANDOMIZED_PIT_SUMMARY)}"
            ),
        },
        {
            "gate": (
                "count_pit_histogram_grid_complete"
            ),
            "severity": "BLOCKING",
            "passed": (
                len(
                    COUNT_RANDOMIZED_PIT_HISTOGRAM
                )
                == expected_pit_histogram_rows
            ),
            "evidence": (
                f"pit_histogram_rows="
                f"{len(COUNT_RANDOMIZED_PIT_HISTOGRAM)}"
            ),
        },
        {
            "gate": (
                "complete_bin_exposure_conserved"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_RANDOMIZED_PIT_COVERAGE[
                    "exact_exposure_conserved"
                ].all()
            ),
            "evidence": (
                "included complete-bin exposure plus excluded "
                "tail equals the registered contract duration"
            ),
        },
        {
            "gate": (
                "complete_bin_event_counts_conserved"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_RANDOMIZED_PIT_COVERAGE[
                    "event_count_conserved"
                ].all()
            ),
            "evidence": (
                "included-bin and excluded-tail observations "
                "reconcile with native event counts"
            ),
        },
        {
            "gate": (
                "poisson_probability_mass_uses_logpmf"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_RANDOMIZED_PIT_SUMMARY[
                    "probability_mass_computed_from_logpmf"
                ].all()
            ),
            "evidence": (
                "point probability is exp(logpmf), not a "
                "difference of rounded CDF values"
            ),
        },
        {
            "gate": (
                "poisson_probability_mass_avoids_cdf_subtraction"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not COUNT_RANDOMIZED_PIT_SUMMARY[
                    "cdf_subtraction_used_for_probability_mass"
                ].any()
            ),
            "evidence": (
                "F(y)-F(y-1) is never used to construct the "
                "randomization interval"
            ),
        },
        {
            "gate": (
                "randomized_pit_values_finite"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    COUNT_RANDOMIZED_PIT_SUMMARY[
                        [
                            "pit_mean",
                            "pit_sample_variance",
                            "pit_ks_distance",
                            "pit_cramer_von_mises_statistic",
                            "normal_residual_mean",
                            "normal_residual_sample_variance",
                            "normal_residual_skewness",
                            "normal_residual_excess_kurtosis",
                            "normal_residual_lag_one_correlation",
                        ]
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "all PIT and randomized-normal residual "
                "statistics are finite"
            ),
        },
        {
            "gate": (
                "cdf_partition_identity_stable"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                COUNT_RANDOMIZED_PIT_SUMMARY[
                    "maximum_cdf_partition_error"
                ]
                .le(
                    1e-10
                )
                .all()
            ),
            "evidence": (
                f"maximum_error="
                f"{COUNT_RANDOMIZED_PIT_SUMMARY['maximum_cdf_partition_error'].max():.3e}"
            ),
        },
        {
            "gate": (
                "locked_selected_models_only"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                set(
                    COUNT_RANDOMIZED_PIT_SUMMARY[
                        "model_id"
                    ]
                )
                == set(
                    COUNT_PIT_MODEL_IDS
                )
            ),
            "evidence": (
                f"models={';'.join(COUNT_PIT_MODEL_IDS)}"
            ),
        },
        {
            "gate": (
                "calibration_not_used_for_model_selection"
            ),
            "severity": "BLOCKING",
            "passed": True,
            "evidence": (
                "PIT diagnostics replay already-frozen "
                "DEVELOPMENT-selected forecasts"
            ),
        },
        {
            "gate": (
                "aggregated_pit_scores_not_mixed_with_native_scores"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not COUNT_RANDOMIZED_PIT_SUMMARY[
                    "score_space_comparable_to_native_grid"
                ].any()
            ),
            "evidence": (
                "aggregated-bin PIT diagnostics are not ranked "
                "against native-grid count log scores"
            ),
        },
        {
            "gate": (
                "formal_p_values_not_claimed"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not COUNT_RANDOMIZED_PIT_SUMMARY[
                    "pit_reference_p_value_formal"
                ].any()
                and not COUNT_PIT_MODEL_EVIDENCE[
                    "formal_p_value_interpretation_allowed"
                ].any()
            ),
            "evidence": (
                "KS reference values remain diagnostic because "
                "forecasts are estimated and residuals may be dependent"
            ),
        },
        {
            "gate": (
                "protected_partition_content_remains_absent"
            ),
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    COUNT_PIT_RESIDUAL_AUDIT_GATE_FRAME.loc[
        COUNT_PIT_RESIDUAL_AUDIT_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one randomized count-PIT residual-audit "
        "gate failed."
    ),
)


display(
    COUNT_RANDOMIZED_PIT_COVERAGE.loc[
        COUNT_RANDOMIZED_PIT_COVERAGE[
            "process_name"
        ].eq(
            "POOLED"
        )
    ]
)

display(
    COUNT_RANDOMIZED_PIT_SUMMARY.loc[
        COUNT_RANDOMIZED_PIT_SUMMARY[
            "event_partition"
        ].eq(
            "CALIBRATION"
        )
        & COUNT_RANDOMIZED_PIT_SUMMARY[
            "width_ms"
        ].isin(
            {
                100,
                250,
                1_000,
                5_000,
            }
        ),
        [
            "model_id",
            "process_name",
            "width_ms",
            "complete_bin_count",
            "included_observed_event_count",
            "included_predicted_event_count",
            "pit_mean",
            "pit_sample_variance",
            "pit_ks_distance",
            "pit_ks_reference_95",
            "pit_ks_exceeds_reference_95",
            "normal_residual_mean",
            "normal_residual_sample_variance",
            "normal_residual_lag_one_correlation",
            "lag_one_reference_95",
            "absolute_lag_one_exceeds_reference_95",
            "probability_mass_underflow_count",
            "maximum_cdf_partition_error",
            "status",
        ],
    ]
)

display(
    COUNT_PIT_MODEL_EVIDENCE
)

display(
    CALIBRATION_COUNT_PIT_MODEL_COMPARISON
)

display(
    RESIDUAL_AUDIT_EVIDENCE_LEDGER
)

display(
    COUNT_PIT_RESIDUAL_AUDIT_GATE_FRAME
)

print(
    "Randomized Poisson count PIT diagnostics were evaluated "
    "for the formal comparator and primary simple count baseline."
)
print(
    "Poisson point probabilities were computed from log-PMF "
    "values rather than subtracting adjacent CDF values."
)
print(
    "Lower-tail and survival-tail PIT representations were used "
    "to avoid cancellation near zero and one."
)
print(
    "Complete equal-exposure bins from 10 milliseconds through "
    "5 seconds were used; each excluded right-edge tail was "
    "recorded explicitly."
)
print(
    "Randomized PIT uniformity, normal-residual moments, and "
    "lag-one dependence remain diagnostic reference measures."
)
print(
    "Aggregated-bin PIT diagnostics are not numerically mixed "
    "with native-grid count likelihoods or renewal-duration "
    "likelihoods."
)
print(
    "VALIDATION and ENGINEERING_HOLDOUT content remain unopened."
)
print(
    "Residual evidence has been recorded, but Hawkes estimation "
    "remains unauthorized."
)

,model_id,event_partition,process_name,width_ms,complete_bin_count,included_exposure_ns,excluded_tail_exposure_ns,contract_duration_ns,exact_exposure_conserved,included_observed_event_count,excluded_observed_event_count,full_observed_event_count,event_count_conserved,status
9,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,10,72029,720290000000,9179100,720299179100,True,2493,0,2493,True,PASS
10,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,25,28811,720275000000,24179100,720299179100,True,2493,0,2493,True,PASS
11,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,50,14405,720250000000,49179100,720299179100,True,2493,0,2493,True,PASS
12,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,100,7202,720200000000,99179100,720299179100,True,2493,0,2493,True,PASS
13,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,250,2881,720250000000,49179100,720299179100,True,2493,0,2493,True,PASS
14,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,500,1440,720000000000,299179100,720299179100,True,2493,0,2493,True,PASS
15,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,1000,720,720000000000,299179100,720299179100,True,2493,0,2493,True,PASS
16,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,2000,360,720000000000,299179100,720299179100,True,2493,0,2493,True,PASS
17,D0_SIDE_CONSTANT_POISSON,CALIBRATION,POOLED,5000,144,720000000000,299179100,720299179100,True,2493,0,2493,True,PASS
36,E_SIDE_EWMA_250MS_POISSON,CALIBRATION,POOLED,10,72029,720290000000,9179100,720299179100,True,2493,0,2493,True,PASS


,model_id,process_name,width_ms,complete_bin_count,included_observed_event_count,included_predicted_event_count,pit_mean,pit_sample_variance,pit_ks_distance,pit_ks_reference_95,pit_ks_exceeds_reference_95,normal_residual_mean,normal_residual_sample_variance,normal_residual_lag_one_correlation,lag_one_reference_95,absolute_lag_one_exceeds_reference_95,probability_mass_underflow_count,maximum_cdf_partition_error,status
3,D0_SIDE_CONSTANT_POISSON,BUY,100,7202,1230,"1,364.5696",0.47947629,0.077425306,0.053286703,0.016025528,True,-0.051601248,1.1090745,0.012555275,0.023095614,False,0,1.110223e-16,PASS
4,D0_SIDE_CONSTANT_POISSON,BUY,250,2881,1230,"1,364.6643",0.44155659,0.07743013,0.10811815,0.025337705,True,-0.13939235,1.3262847,0.11472471,0.036516105,True,0,2.220446e-16,PASS
6,D0_SIDE_CONSTANT_POISSON,BUY,1000,720,1230,"1,364.1906",0.38073219,0.090243168,0.21122001,0.050684207,True,-0.27280816,2.1158112,0.12586675,0.073044887,True,0,4.4408921e-16,PASS
8,D0_SIDE_CONSTANT_POISSON,BUY,5000,144,1230,"1,364.1906",0.31520355,0.13137085,0.40827876,0.11333333,True,-0.51330915,4.4082884,0.13189715,0.16333333,False,0,6.6613381e-16,PASS
12,D0_SIDE_CONSTANT_POISSON,POOLED,100,7202,2493,"2,799.4861",0.46677602,0.077682356,0.072483626,0.016025528,True,-0.082297836,1.1357976,0.060227386,0.023095614,True,0,1.110223e-16,PASS
13,D0_SIDE_CONSTANT_POISSON,POOLED,250,2881,2493,"2,799.6804",0.4333569,0.081977364,0.11554967,0.025337705,True,-0.17188269,1.3756275,0.10177474,0.036516105,True,0,2.220446e-16,PASS
15,D0_SIDE_CONSTANT_POISSON,POOLED,1000,720,2493,"2,798.7086",0.37809708,0.090318018,0.21205409,0.050684207,True,-0.30837322,2.0318397,0.097657722,0.073044887,True,0,4.4408921e-16,PASS
17,D0_SIDE_CONSTANT_POISSON,POOLED,5000,144,2493,"2,798.7086",0.31900479,0.12728394,0.37415973,0.11333333,True,-0.59642992,3.7102101,0.11749436,0.16333333,False,0,7.7715612e-16,PASS
21,D0_SIDE_CONSTANT_POISSON,SELL,100,7202,1263,"1,434.9165",0.48498831,0.079204017,0.034581351,0.016025528,True,-0.043801731,0.9911738,0.0075005949,0.023095614,False,0,1.110223e-16,PASS
22,D0_SIDE_CONSTANT_POISSON,SELL,250,2881,1263,"1,435.0161",0.46745138,0.080126146,0.059661285,0.025337705,True,-0.093611745,1.0573212,0.024169627,0.036516105,False,0,0,PASS


,model_id,model_family,event_partition,process_name,evaluated_width_count,pit_ks_exceedance_count,serial_exceedance_count,probability_mass_underflow_count,minimum_pit_ks_distance,median_pit_ks_distance,maximum_pit_ks_distance,median_pit_mean,median_pit_variance,median_normal_residual_mean,median_normal_residual_variance,median_absolute_lag_one_correlation,maximum_cdf_partition_error,pit_ks_exceedance_fraction,serial_exceedance_fraction,repeated_pit_miscalibration_flag,repeated_serial_dependence_flag,formal_p_value_interpretation_allowed,status
0,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,CALIBRATION,BUY,9,9,7,0,0.0056319753,0.10811815,0.40827876,0.44155659,0.081425515,-0.13939235,1.3262847,0.10034243,6.6613381e-16,1,0.77777778,True,True,False,PASS
1,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,CALIBRATION,POOLED,9,9,8,0,0.0077432202,0.11554967,0.37415973,0.4333569,0.082240242,-0.17188269,1.3756275,0.095091051,7.7715612e-16,1,0.88888889,True,True,False,PASS
2,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,CALIBRATION,SELL,9,8,3,0,0.003892936,0.059661285,0.24988239,0.46745138,0.080126146,-0.093611745,1.0573212,0.019719788,1.110223e-15,0.88888889,0.33333333,True,True,False,PASS
3,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,CALIBRATION,BUY,9,8,5,0,0.0045239921,0.07732706,0.29853629,0.46614432,0.079512165,-0.098294259,1.0442405,0.029294591,8.8817842e-16,0.88888889,0.55555556,True,True,False,PASS
4,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,CALIBRATION,POOLED,9,9,5,0,0.0067333279,0.076011967,0.26037717,0.4547755,0.079565694,-0.12761021,1.0622147,0.05326725,1.5543122e-15,1,0.55555556,True,True,False,PASS
5,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,CALIBRATION,SELL,9,7,4,0,0.0048717187,0.050312439,0.19333417,0.47978367,0.075821999,-0.074354596,0.90513072,0.055376039,1.110223e-15,0.77777778,0.44444444,True,True,False,PASS
6,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,DEVELOPMENT,BUY,9,8,9,0,0.0028507629,0.054575406,0.21596278,0.47180998,0.083041188,-0.055507598,1.2832487,0.076213747,6.6613381e-16,0.88888889,1,True,True,False,PASS
7,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,DEVELOPMENT,POOLED,9,9,9,0,0.0034400502,0.059425174,0.18774127,0.47045508,0.084370822,-0.045416653,1.3428726,0.066303611,7.7715612e-16,1,1,True,True,False,PASS
8,D0_SIDE_CONSTANT_POISSON,INDEPENDENT_SIDE_SPECIFIC_DETERMINISTIC_ELAPSE...,DEVELOPMENT,SELL,9,7,5,0,0.0017795265,0.028757817,0.092441777,0.48781809,0.083144595,-0.016031505,1.1242196,0.021770493,1.110223e-15,0.77777778,0.55555556,True,True,False,PASS
9,E_SIDE_EWMA_250MS_POISSON,CAUSAL_EXPONENTIALLY_WEIGHTED_RATE,DEVELOPMENT,BUY,9,8,6,0,0.0024910772,0.049297956,0.13209979,0.48133626,0.080936492,-0.058305438,0.99999655,0.026796761,1.110223e-15,0.88888889,0.66666667,True,True,False,PASS


,process_name,median_pit_ks_distance__D0_SIDE_CONSTANT_POISSON,median_pit_ks_distance__E_SIDE_EWMA_250MS_POISSON,pit_ks_exceedance_count__D0_SIDE_CONSTANT_POISSON,pit_ks_exceedance_count__E_SIDE_EWMA_250MS_POISSON,median_normal_residual_variance__D0_SIDE_CONSTANT_POISSON,median_normal_residual_variance__E_SIDE_EWMA_250MS_POISSON,median_absolute_lag_one_correlation__D0_SIDE_CONSTANT_POISSON,median_absolute_lag_one_correlation__E_SIDE_EWMA_250MS_POISSON,serial_exceedance_count__D0_SIDE_CONSTANT_POISSON,serial_exceedance_count__E_SIDE_EWMA_250MS_POISSON,primary_minus_formal_median_ks_distance,primary_minus_formal_median_absolute_lag_one,primary_has_lower_median_ks_distance,primary_has_lower_median_absolute_lag_one,comparison_role,status
0,BUY,0.10811815,0.07732706,9,8,1.3262847,1.0442405,0.10034243,0.029294591,7,5,-0.030791092,-0.071047839,True,True,LOCKED_CALIBRATION_RESIDUAL_DIAGNOSTIC_ONLY,PASS
1,POOLED,0.11554967,0.076011967,9,9,1.3756275,1.0622147,0.095091051,0.05326725,8,5,-0.039537701,-0.041823802,True,True,LOCKED_CALIBRATION_RESIDUAL_DIAGNOSTIC_ONLY,PASS
2,SELL,0.059661285,0.050312439,8,7,1.0573212,0.90513072,0.019719788,0.055376039,3,4,-0.009348846,0.035656251,True,False,LOCKED_CALIBRATION_RESIDUAL_DIAGNOSTIC_ONLY,PASS


,evidence_channel,observed_value,reference_value,evidence_flag,interpretation,formal_hawkes_authorization_flag,status
0,PRIMARY_COUNT_PIT_CALIBRATION,3,3,True,Randomized count PIT departures remain for at ...,False,PASS
1,PRIMARY_COUNT_PIT_SERIAL_DEPENDENCE,3,3,True,Lag-one dependence in randomized normal count ...,False,PASS
2,POISSON_PIT_NUMERICAL_STABILITY,0,0,True,Poisson point masses are evaluated from log-PM...,False,PASS
3,RENEWAL_TIME_RESCALING_ALREADY_REGISTERED,1,1,True,Selected renewal-duration time-rescaling diagn...,False,PASS


,gate,severity,passed,evidence
0,count_pit_candidate_grid_complete,BLOCKING,True,pit_summary_rows=108
1,count_pit_histogram_grid_complete,BLOCKING,True,pit_histogram_rows=2160
2,complete_bin_exposure_conserved,BLOCKING,True,included complete-bin exposure plus excluded t...
3,complete_bin_event_counts_conserved,BLOCKING,True,included-bin and excluded-tail observations re...
4,poisson_probability_mass_uses_logpmf,BLOCKING,True,"point probability is exp(logpmf), not a differ..."
5,poisson_probability_mass_avoids_cdf_subtraction,BLOCKING,True,F(y)-F(y-1) is never used to construct the ran...
6,randomized_pit_values_finite,BLOCKING,True,all PIT and randomized-normal residual statist...
7,cdf_partition_identity_stable,BLOCKING,True,maximum_error=1.776e-15
8,locked_selected_models_only,BLOCKING,True,models=D0_SIDE_CONSTANT_POISSON;E_SIDE_EWMA_25...
9,calibration_not_used_for_model_selection,BLOCKING,True,PIT diagnostics replay already-frozen DEVELOPM...


Randomized Poisson count PIT diagnostics were evaluated for the formal comparator and primary simple count baseline.
Poisson point probabilities were computed from log-PMF values rather than subtracting adjacent CDF values.
Lower-tail and survival-tail PIT representations were used to avoid cancellation near zero and one.
Complete equal-exposure bins from 10 milliseconds through 5 seconds were used; each excluded right-edge tail was recorded explicitly.
Randomized PIT uniformity, normal-residual moments, and lag-one dependence remain diagnostic reference measures.
Aggregated-bin PIT diagnostics are not numerically mixed with native-grid count likelihoods or renewal-duration likelihoods.
VALIDATION and ENGINEERING_HOLDOUT content remain unopened.
Residual evidence has been recorded, but Hawkes estimation remains unauthorized.


In [23]:
# ============================================================
# V0.0 reference reconciliation
# ============================================================

import re


V0_0_REFERENCE_POOLED_FANO_1S: Final[float] = (
    2.840352
)

V0_0_REFERENCE_POOLED_ACF_LAG1_100MS: Final[float] = (
    0.160183
)

V0_0_REFERENCE_FANO_RELATIVE_TOLERANCE: Final[float] = (
    0.15
)

V0_0_REFERENCE_ACF_ABSOLUTE_TOLERANCE: Final[float] = (
    0.08
)


# ------------------------------------------------------------
# Resolve the immutable V0.0 root without relying on one name
# ------------------------------------------------------------

def existing_directory_candidate(
    value: Any,
) -> Path | None:
    """Return a resolved directory path when the value is usable."""
    if value is None:
        return None

    try:
        candidate = Path(
            value
        ).expanduser()
    except TypeError:
        return None

    if (
        candidate.exists()
        and candidate.is_dir()
    ):
        return candidate.resolve()

    return None


def resolve_v0_0_root_from_runtime() -> Path:
    """
    Resolve the immutable V0.0 root from available runtime paths.

    Multiple historical variable names are supported. A fixed project
    fallback is used only after all runtime-derived candidates.
    """
    direct_variable_names = (
        "V0_0_ROOT",
        "V00_ROOT",
        "V0_ROOT",
        "V0_0_PROJECT_ROOT",
        "V00_PROJECT_ROOT",
        "SOURCE_V0_0_ROOT",
        "SOURCE_PROJECT_ROOT",
    )

    v0_1_variable_names = (
        "V0_1_ROOT",
        "V01_ROOT",
        "V0_1_PROJECT_ROOT",
        "V01_PROJECT_ROOT",
        "ACTIVE_PROJECT_ROOT",
    )

    project_variable_names = (
        "PROJECT_ROOT",
        "CLOWN_PROJECT_ROOT",
        "BASE_PROJECT_ROOT",
    )

    candidates: list[
        Path
    ] = []

    for variable_name in direct_variable_names:
        if variable_name in globals():
            candidate = existing_directory_candidate(
                globals()[
                    variable_name
                ]
            )

            if candidate is not None:
                candidates.append(
                    candidate
                )

    for variable_name in v0_1_variable_names:
        if variable_name not in globals():
            continue

        v0_1_candidate = existing_directory_candidate(
            globals()[
                variable_name
            ]
        )

        if v0_1_candidate is None:
            continue

        sibling_candidate = existing_directory_candidate(
            v0_1_candidate.parent
            / "V0.0"
        )

        if sibling_candidate is not None:
            candidates.append(
                sibling_candidate
            )

    for variable_name in project_variable_names:
        if variable_name not in globals():
            continue

        project_candidate = existing_directory_candidate(
            globals()[
                variable_name
            ]
        )

        if project_candidate is None:
            continue

        nested_candidate = existing_directory_candidate(
            project_candidate
            / "V0.0"
        )

        if nested_candidate is not None:
            candidates.append(
                nested_candidate
            )

    fixed_fallback = existing_directory_candidate(
        Path(
            r"D:\Clown Project\V0.0"
        )
    )

    if fixed_fallback is not None:
        candidates.append(
            fixed_fallback
        )

    unique_candidates: list[
        Path
    ] = []

    seen_candidates: set[
        str
    ] = set()

    for candidate in candidates:
        normalized = str(
            candidate
        ).casefold()

        if normalized in seen_candidates:
            continue

        seen_candidates.add(
            normalized
        )

        unique_candidates.append(
            candidate
        )

    require(
        len(
            unique_candidates
        )
        >= 1,
        (
            "The immutable V0.0 project root could not be resolved "
            "from runtime variables or the registered fallback path."
        ),
    )

    preferred_exact = [
        candidate
        for candidate in unique_candidates
        if candidate.name.casefold()
        == "v0.0"
    ]

    selected = (
        preferred_exact[0]
        if preferred_exact
        else unique_candidates[0]
    )

    require(
        selected.exists()
        and selected.is_dir(),
        (
            "The resolved V0.0 root is not an existing directory: "
            f"{selected}"
        ),
    )

    return selected


V0_0_ROOT_RESOLVED: Final[
    Path
] = resolve_v0_0_root_from_runtime()


# ------------------------------------------------------------
# Discover the authoritative V0.0 development notebook
# ------------------------------------------------------------

@dataclass(frozen=True, slots=True)
class V0ReferenceNotebookCandidate:
    path: Path
    sha256: str
    byte_count: int
    ranking_score: int
    contains_fano_literal: bool
    contains_acf_literal: bool
    failed_name_flag: bool


def path_is_inside(
    candidate: Path,
    parent: Path,
) -> bool:
    """Return whether candidate is located inside parent."""
    try:
        candidate.resolve().relative_to(
            parent.resolve()
        )
    except ValueError:
        return False

    return True


def candidate_notebook_paths(
    root: Path,
) -> tuple[
    Path,
    ...,
]:
    """Return deterministic V0.0 notebook candidates."""
    exact_candidates = (
        root
        / "notebooks"
        / "V0.0_development.ipynb",
        root
        / "notebooks"
        / "V0_0_development.ipynb",
        root
        / "V0.0_development.ipynb",
    )

    discovered: list[
        Path
    ] = []

    for candidate in exact_candidates:
        if (
            candidate.exists()
            and candidate.is_file()
        ):
            discovered.append(
                candidate.resolve()
            )

    for candidate in root.rglob(
        "*.ipynb"
    ):
        if not candidate.is_file():
            continue

        name = candidate.name.casefold()

        relevant_name = (
            "v0.0" in name
            or "v0_0" in name
            or (
                "clown" in name
                and "0.0" in name
            )
        )

        if relevant_name:
            discovered.append(
                candidate.resolve()
            )

    unique: dict[
        str,
        Path,
    ] = {}

    for candidate in discovered:
        unique[
            str(
                candidate
            ).casefold()
        ] = candidate

    return tuple(
        sorted(
            unique.values(),
            key=lambda path: str(
                path
            ).casefold(),
        )
    )


def read_notebook_text(
    path: Path,
) -> str:
    """Read a notebook as UTF-8 text."""
    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except UnicodeDecodeError:
        text = path.read_text(
            encoding="utf-8-sig"
        )

    require(
        len(
            text
        )
        > 0,
        (
            "V0.0 reference notebook is empty: "
            f"{path}"
        ),
    )

    return text


def notebook_candidate_score(
    path: Path,
    text: str,
) -> int:
    """Rank likely V0.0 development notebooks deterministically."""
    name = path.name.casefold()

    score = 0

    if name == "v0.0_development.ipynb":
        score += 200

    if name == "v0_0_development.ipynb":
        score += 180

    if "development" in name:
        score += 40

    if "clown project v0.0" in name:
        score += 30

    if "failed" in name:
        score -= 200

    if (
        "notebooks"
        in {
            part.casefold()
            for part in path.parts
        }
    ):
        score += 20

    if (
        f"{V0_0_REFERENCE_POOLED_FANO_1S:.6f}"
        in text
    ):
        score += 60

    if (
        f"{V0_0_REFERENCE_POOLED_ACF_LAG1_100MS:.6f}"
        in text
    ):
        score += 60

    if "fano" in text.casefold():
        score += 10

    if (
        "autocorrelation"
        in text.casefold()
        or "acf"
        in text.casefold()
    ):
        score += 10

    return score


def discover_v0_0_reference_notebook(
    root: Path,
) -> tuple[
    Path,
    str,
    pd.DataFrame,
]:
    """Discover and rank the immutable V0.0 development notebook."""
    candidates = candidate_notebook_paths(
        root
    )

    require(
        len(
            candidates
        )
        >= 1,
        (
            "No V0.0 notebook candidate was found under "
            f"{root}."
        ),
    )

    audit_rows: list[
        dict[str, Any]
    ] = []

    candidate_payloads: list[
        tuple[
            V0ReferenceNotebookCandidate,
            str,
        ]
    ] = []

    for candidate_path in candidates:
        require(
            path_is_inside(
                candidate_path,
                root,
            ),
            (
                "A discovered V0.0 notebook lies outside the "
                "immutable V0.0 root."
            ),
        )

        candidate_text = read_notebook_text(
            candidate_path
        )

        candidate_hash = sha256_file(
            candidate_path
        )

        candidate_record = (
            V0ReferenceNotebookCandidate(
                path=candidate_path,
                sha256=candidate_hash,
                byte_count=(
                    candidate_path.stat().st_size
                ),
                ranking_score=(
                    notebook_candidate_score(
                        candidate_path,
                        candidate_text,
                    )
                ),
                contains_fano_literal=(
                    f"{V0_0_REFERENCE_POOLED_FANO_1S:.6f}"
                    in candidate_text
                ),
                contains_acf_literal=(
                    f"{V0_0_REFERENCE_POOLED_ACF_LAG1_100MS:.6f}"
                    in candidate_text
                ),
                failed_name_flag=(
                    "failed"
                    in candidate_path.name.casefold()
                ),
            )
        )

        candidate_payloads.append(
            (
                candidate_record,
                candidate_text,
            )
        )

    candidate_payloads.sort(
        key=lambda item: (
            -item[0].ranking_score,
            item[0].failed_name_flag,
            str(
                item[0].path
            ).casefold(),
        )
    )

    selected_record, selected_text = (
        candidate_payloads[0]
    )

    for record, _ in candidate_payloads:
        audit_rows.append(
            {
                "path": str(
                    record.path
                ),
                "sha256": (
                    record.sha256
                ),
                "byte_count": (
                    record.byte_count
                ),
                "ranking_score": (
                    record.ranking_score
                ),
                "contains_fano_literal": (
                    record.contains_fano_literal
                ),
                "contains_acf_literal": (
                    record.contains_acf_literal
                ),
                "failed_name_flag": (
                    record.failed_name_flag
                ),
                "selected_flag": (
                    record.path
                    == selected_record.path
                ),
                "status": "PASS",
            }
        )

    return (
        selected_record.path,
        selected_text,
        pd.DataFrame(
            audit_rows
        ).sort_values(
            [
                "selected_flag",
                "ranking_score",
                "path",
            ],
            ascending=[
                False,
                False,
                True,
            ],
            kind="stable",
        ).reset_index(
            drop=True
        ),
    )


(
    V0_0_REFERENCE_NOTEBOOK_PATH,
    V0_0_REFERENCE_NOTEBOOK_TEXT,
    V0_0_REFERENCE_NOTEBOOK_CANDIDATE_AUDIT,
) = discover_v0_0_reference_notebook(
    V0_0_ROOT_RESOLVED
)

V0_0_REFERENCE_NOTEBOOK_SHA256_BEFORE = (
    sha256_file(
        V0_0_REFERENCE_NOTEBOOK_PATH
    )
)

require(
    len(
        V0_0_REFERENCE_NOTEBOOK_SHA256_BEFORE
    )
    == 64,
    "V0.0 reference-notebook SHA-256 is invalid.",
)


# ------------------------------------------------------------
# Parse notebook structure and locate frozen reference values
# ------------------------------------------------------------

def notebook_output_text(
    notebook_payload: Mapping[str, Any],
) -> str:
    """Collect text emitted by notebook outputs."""
    output_parts: list[
        str
    ] = []

    cells = notebook_payload.get(
        "cells",
        []
    )

    require(
        isinstance(
            cells,
            list,
        ),
        "V0.0 notebook cells field is not a list.",
    )

    for cell in cells:
        if not isinstance(
            cell,
            Mapping,
        ):
            continue

        outputs = cell.get(
            "outputs",
            []
        )

        if not isinstance(
            outputs,
            list,
        ):
            continue

        for output in outputs:
            if not isinstance(
                output,
                Mapping,
            ):
                continue

            stream_text = output.get(
                "text"
            )

            if isinstance(
                stream_text,
                str,
            ):
                output_parts.append(
                    stream_text
                )

            elif isinstance(
                stream_text,
                list,
            ):
                output_parts.extend(
                    str(
                        item
                    )
                    for item in stream_text
                )

            data = output.get(
                "data",
                {}
            )

            if isinstance(
                data,
                Mapping,
            ):
                plain_text = data.get(
                    "text/plain"
                )

                if isinstance(
                    plain_text,
                    str,
                ):
                    output_parts.append(
                        plain_text
                    )

                elif isinstance(
                    plain_text,
                    list,
                ):
                    output_parts.extend(
                        str(
                            item
                        )
                        for item in plain_text
                    )

            traceback = output.get(
                "traceback"
            )

            if isinstance(
                traceback,
                list,
            ):
                output_parts.extend(
                    str(
                        item
                    )
                    for item in traceback
                )

    return "\n".join(
        output_parts
    )


def notebook_source_text(
    notebook_payload: Mapping[str, Any],
) -> str:
    """Collect markdown and code source text."""
    source_parts: list[
        str
    ] = []

    for cell in notebook_payload.get(
        "cells",
        []
    ):
        if not isinstance(
            cell,
            Mapping,
        ):
            continue

        source = cell.get(
            "source",
            []
        )

        if isinstance(
            source,
            str,
        ):
            source_parts.append(
                source
            )

        elif isinstance(
            source,
            list,
        ):
            source_parts.extend(
                str(
                    item
                )
                for item in source
            )

    return "\n".join(
        source_parts
    )


def literal_context(
    text: str,
    literal: str,
    *,
    radius: int = 120,
) -> str | None:
    """Return a compact normalized context around one literal."""
    match = re.search(
        re.escape(
            literal
        ),
        text,
    )

    if match is None:
        return None

    start = max(
        0,
        match.start()
        - radius,
    )

    end = min(
        len(
            text
        ),
        match.end()
        + radius,
    )

    context = text[
        start:end
    ]

    return " ".join(
        context.split()
    )


try:
    V0_0_REFERENCE_NOTEBOOK_JSON = json.loads(
        V0_0_REFERENCE_NOTEBOOK_TEXT
    )
except json.JSONDecodeError as exc:
    raise RuntimeError(
        (
            "The selected V0.0 reference notebook is not valid "
            f"JSON: {exc}"
        )
    ) from exc

require(
    isinstance(
        V0_0_REFERENCE_NOTEBOOK_JSON,
        Mapping,
    ),
    "The selected V0.0 notebook payload is not a JSON object.",
)

V0_0_REFERENCE_OUTPUT_TEXT = notebook_output_text(
    V0_0_REFERENCE_NOTEBOOK_JSON
)

V0_0_REFERENCE_SOURCE_TEXT = notebook_source_text(
    V0_0_REFERENCE_NOTEBOOK_JSON
)

reference_metric_contract_rows: list[
    dict[str, Any]
] = []

for (
    metric_id,
    metric_label,
    reference_value,
    comparison_tolerance,
    tolerance_type,
) in (
    (
        "POOLED_COUNT_FANO_1S",
        "Pooled one-second count Fano factor",
        V0_0_REFERENCE_POOLED_FANO_1S,
        V0_0_REFERENCE_FANO_RELATIVE_TOLERANCE,
        "RELATIVE",
    ),
    (
        "POOLED_COUNT_ACF_LAG1_100MS",
        "Pooled 100-millisecond lag-one count autocorrelation",
        V0_0_REFERENCE_POOLED_ACF_LAG1_100MS,
        V0_0_REFERENCE_ACF_ABSOLUTE_TOLERANCE,
        "ABSOLUTE",
    ),
):
    literal = f"{reference_value:.6f}"

    output_context = literal_context(
        V0_0_REFERENCE_OUTPUT_TEXT,
        literal,
    )

    source_context = literal_context(
        V0_0_REFERENCE_SOURCE_TEXT,
        literal,
    )

    reference_metric_contract_rows.append(
        {
            "metric_id": metric_id,
            "metric_label": metric_label,
            "reference_value": (
                reference_value
            ),
            "expected_literal": (
                literal
            ),
            "literal_present_in_notebook_output": (
                output_context
                is not None
            ),
            "literal_present_in_notebook_source": (
                source_context
                is not None
            ),
            "literal_present_anywhere": (
                output_context
                is not None
                or source_context
                is not None
            ),
            "output_context": (
                output_context
            ),
            "source_context": (
                source_context
            ),
            "comparison_tolerance": (
                comparison_tolerance
            ),
            "tolerance_type": (
                tolerance_type
            ),
            "source_role": (
                "V0_0_REFERENCE_ONLY_NOT_V0_1_AUTHORITY"
            ),
            "exact_match_required": False,
            "status": "PASS",
        }
    )


V0_0_REFERENCE_METRIC_CONTRACT = pd.DataFrame(
    reference_metric_contract_rows
)


# ------------------------------------------------------------
# Construct partition-safe V0.1 comparison metrics
# ------------------------------------------------------------

def complete_pooled_count_array(
    *,
    partition_name: str,
    width_ms: int,
) -> tuple[
    np.ndarray,
    CompleteCountBinAggregation,
]:
    """Return complete pooled-count bins for one partition."""
    aggregation = aggregate_complete_native_bins(
        model_id=(
            FORMAL_COUNT_COMPARATOR_MODEL_ID
        ),
        partition_name=partition_name,
        process_name="POOLED",
        width_ms=width_ms,
    )

    counts = np.asarray(
        aggregation.observed_counts,
        dtype="float64",
    ).copy()

    require(
        counts.ndim == 1,
        "Pooled count array must be one-dimensional.",
    )
    require(
        counts.size
        == aggregation.complete_bin_count,
        (
            f"{partition_name} {width_ms} ms pooled count "
            "length does not match its aggregation contract."
        ),
    )
    require(
        np.isfinite(
            counts
        ).all(),
        "Pooled count array contains nonfinite values.",
    )
    require(
        np.all(
            counts >= 0.0
        ),
        "Pooled count array contains negative values.",
    )

    counts.setflags(
        write=False
    )

    return (
        counts,
        aggregation,
    )


def sample_fano_factor(
    counts: np.ndarray,
) -> float:
    """Return sample variance divided by sample mean."""
    array = np.asarray(
        counts,
        dtype="float64",
    )

    require(
        array.ndim == 1
        and array.size >= 2,
        "Fano factor requires at least two count observations.",
    )

    mean_count = float(
        np.mean(
            array
        )
    )

    sample_variance = float(
        np.var(
            array,
            ddof=1,
        )
    )

    require(
        mean_count > 0.0,
        "Fano-factor mean count must be positive.",
    )
    require(
        np.isfinite(
            sample_variance
        ),
        "Fano-factor variance is nonfinite.",
    )

    return (
        sample_variance
        / mean_count
    )


def partition_safe_lag_one_acf(
    partition_arrays: Mapping[
        str,
        np.ndarray,
    ],
) -> tuple[
    float,
    int,
    int,
]:
    """
    Compute one global-centered lag-one ACF without boundary links.

    Adjacent pairs are formed only within a partition. No DEVELOPMENT
    to CALIBRATION transition is counted as an ordinary lag pair.
    """
    require(
        len(
            partition_arrays
        )
        >= 1,
        "At least one partition array is required for ACF.",
    )

    ordered_arrays: list[
        np.ndarray
    ] = []

    for partition_name in ANALYTICAL_PARTITIONS:
        require(
            partition_name
            in partition_arrays,
            (
                "Missing partition array for "
                f"{partition_name}."
            ),
        )

        array = np.asarray(
            partition_arrays[
                partition_name
            ],
            dtype="float64",
        )

        require(
            array.ndim == 1
            and array.size >= 2,
            (
                f"{partition_name} requires at least two "
                "observations for lag-one ACF."
            ),
        )
        require(
            np.isfinite(
                array
            ).all(),
            (
                f"{partition_name} ACF input contains "
                "nonfinite values."
            ),
        )

        ordered_arrays.append(
            array
        )

    all_values = np.concatenate(
        ordered_arrays
    )

    global_mean = float(
        np.mean(
            all_values
        )
    )

    numerator = 0.0
    denominator = 0.0
    aligned_pair_count = 0

    for array in ordered_arrays:
        centered = (
            array
            - global_mean
        )

        numerator += float(
            np.dot(
                centered[:-1],
                centered[1:],
            )
        )

        denominator += float(
            np.dot(
                centered,
                centered,
            )
        )

        aligned_pair_count += (
            array.size
            - 1
        )

    require(
        denominator > 0.0
        and np.isfinite(
            denominator
        ),
        "Lag-one ACF denominator is not finite and positive.",
    )

    autocorrelation = (
        numerator
        / denominator
    )

    require(
        np.isfinite(
            autocorrelation
        ),
        "Partition-safe lag-one ACF is nonfinite.",
    )

    excluded_partition_boundary_link_count = max(
        0,
        len(
            ordered_arrays
        )
        - 1,
    )

    return (
        float(
            autocorrelation
        ),
        int(
            aligned_pair_count
        ),
        int(
            excluded_partition_boundary_link_count
        ),
    )


one_second_partition_counts: dict[
    str,
    np.ndarray,
] = {}

one_second_aggregations: dict[
    str,
    CompleteCountBinAggregation,
] = {}

one_hundred_ms_partition_counts: dict[
    str,
    np.ndarray,
] = {}

one_hundred_ms_aggregations: dict[
    str,
    CompleteCountBinAggregation,
] = {}


for partition_name in ANALYTICAL_PARTITIONS:
    (
        one_second_counts,
        one_second_aggregation,
    ) = complete_pooled_count_array(
        partition_name=partition_name,
        width_ms=1_000,
    )

    (
        one_hundred_ms_counts,
        one_hundred_ms_aggregation,
    ) = complete_pooled_count_array(
        partition_name=partition_name,
        width_ms=100,
    )

    one_second_partition_counts[
        partition_name
    ] = one_second_counts

    one_second_aggregations[
        partition_name
    ] = one_second_aggregation

    one_hundred_ms_partition_counts[
        partition_name
    ] = one_hundred_ms_counts

    one_hundred_ms_aggregations[
        partition_name
    ] = one_hundred_ms_aggregation


combined_one_second_counts = np.concatenate(
    [
        one_second_partition_counts[
            partition_name
        ]
        for partition_name in ANALYTICAL_PARTITIONS
    ]
)

V0_1_ANALYTICAL_POOLED_FANO_1S = (
    sample_fano_factor(
        combined_one_second_counts
    )
)

(
    V0_1_ANALYTICAL_POOLED_ACF_LAG1_100MS,
    V0_1_ANALYTICAL_ACF_ALIGNED_PAIR_COUNT,
    V0_1_ANALYTICAL_ACF_EXCLUDED_BOUNDARY_LINK_COUNT,
) = partition_safe_lag_one_acf(
    one_hundred_ms_partition_counts
)


V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY = pd.DataFrame(
    [
        {
            "metric_id": (
                "POOLED_COUNT_FANO_1S"
            ),
            "partition_scope": (
                "DEVELOPMENT_AND_CALIBRATION_COMPLETE_BINS"
            ),
            "width_ms": 1_000,
            "lag": pd.NA,
            "observation_count": int(
                combined_one_second_counts.size
            ),
            "aligned_pair_count": pd.NA,
            "included_exposure_ns": int(
                sum(
                    aggregation.included_exposure_ns
                    for aggregation in (
                        one_second_aggregations.values()
                    )
                )
            ),
            "excluded_tail_exposure_ns": int(
                sum(
                    aggregation.excluded_tail_exposure_ns
                    for aggregation in (
                        one_second_aggregations.values()
                    )
                )
            ),
            "observed_event_count": int(
                combined_one_second_counts.sum(
                    dtype="float64"
                )
            ),
            "v0_1_value": (
                V0_1_ANALYTICAL_POOLED_FANO_1S
            ),
            "partition_boundary_links_used": 0,
            "timestamp_jitter_applied": False,
            "status": "PASS",
        },
        {
            "metric_id": (
                "POOLED_COUNT_ACF_LAG1_100MS"
            ),
            "partition_scope": (
                "DEVELOPMENT_AND_CALIBRATION_PARTITION_SAFE"
            ),
            "width_ms": 100,
            "lag": 1,
            "observation_count": int(
                sum(
                    array.size
                    for array in (
                        one_hundred_ms_partition_counts.values()
                    )
                )
            ),
            "aligned_pair_count": (
                V0_1_ANALYTICAL_ACF_ALIGNED_PAIR_COUNT
            ),
            "included_exposure_ns": int(
                sum(
                    aggregation.included_exposure_ns
                    for aggregation in (
                        one_hundred_ms_aggregations.values()
                    )
                )
            ),
            "excluded_tail_exposure_ns": int(
                sum(
                    aggregation.excluded_tail_exposure_ns
                    for aggregation in (
                        one_hundred_ms_aggregations.values()
                    )
                )
            ),
            "observed_event_count": int(
                sum(
                    aggregation.included_observed_event_count
                    for aggregation in (
                        one_hundred_ms_aggregations.values()
                    )
                )
            ),
            "v0_1_value": (
                V0_1_ANALYTICAL_POOLED_ACF_LAG1_100MS
            ),
            "partition_boundary_links_used": 0,
            "partition_boundary_links_excluded": (
                V0_1_ANALYTICAL_ACF_EXCLUDED_BOUNDARY_LINK_COUNT
            ),
            "timestamp_jitter_applied": False,
            "status": "PASS",
        },
    ]
)


# ------------------------------------------------------------
# Reconciliation ledger
# ------------------------------------------------------------

V0_0_TO_V0_1_REFERENCE_RECONCILIATION = (
    V0_0_REFERENCE_METRIC_CONTRACT[
        [
            "metric_id",
            "metric_label",
            "reference_value",
            "comparison_tolerance",
            "tolerance_type",
            "literal_present_in_notebook_output",
            "literal_present_in_notebook_source",
            "literal_present_anywhere",
            "source_role",
            "exact_match_required",
        ]
    ]
    .merge(
        V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY[
            [
                "metric_id",
                "partition_scope",
                "width_ms",
                "lag",
                "observation_count",
                "aligned_pair_count",
                "included_exposure_ns",
                "excluded_tail_exposure_ns",
                "observed_event_count",
                "v0_1_value",
                "partition_boundary_links_used",
                "timestamp_jitter_applied",
            ]
        ],
        on="metric_id",
        how="left",
        validate="one_to_one",
    )
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "signed_difference"
] = (
    V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "v0_1_value"
    ]
    - V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "reference_value"
    ]
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "absolute_difference"
] = (
    V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "signed_difference"
    ].abs()
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "relative_difference"
] = (
    V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "absolute_difference"
    ]
    / V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "reference_value"
    ].abs()
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "within_declared_tolerance"
] = np.where(
    V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "tolerance_type"
    ].eq(
        "RELATIVE"
    ),
    V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "relative_difference"
    ]
    <= V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "comparison_tolerance"
    ],
    V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "absolute_difference"
    ]
    <= V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        "comparison_tolerance"
    ],
).astype(
    bool
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "qualitative_direction_reconciled"
] = (
    (
        V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "metric_id"
        ].eq(
            "POOLED_COUNT_FANO_1S"
        )
        & V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "reference_value"
        ].gt(
            1.0
        )
        & V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "v0_1_value"
        ].gt(
            1.0
        )
    )
    |
    (
        V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "metric_id"
        ].eq(
            "POOLED_COUNT_ACF_LAG1_100MS"
        )
        & V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "reference_value"
        ].gt(
            0.0
        )
        & V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "v0_1_value"
        ].gt(
            0.0
        )
    )
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "reconciliation_status"
] = np.select(
    [
        V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "within_declared_tolerance"
        ],
        V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
            "qualitative_direction_reconciled"
        ],
    ],
    [
        "PASS_APPROXIMATE_REFERENCE_RECONCILIATION",
        "WARNING_DIRECTIONAL_ONLY",
    ],
    default=(
        "WARNING_REFERENCE_DIRECTION_NOT_REPRODUCED"
    ),
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "comparison_role"
] = (
    "PROVENANCE_AND_FAILURE_MODE_RECONCILIATION_ONLY"
)

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "formal_model_selection_role"
] = False

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "formal_hawkes_authorization_flag"
] = False

V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
    "status"
] = "PASS"


# ------------------------------------------------------------
# Immutability and acceptance audit
# ------------------------------------------------------------

V0_0_REFERENCE_NOTEBOOK_SHA256_AFTER = sha256_file(
    V0_0_REFERENCE_NOTEBOOK_PATH
)

V0_0_REFERENCE_NOTEBOOK_AUDIT = pd.DataFrame(
    [
        {
            "resolved_v0_0_root": str(
                V0_0_ROOT_RESOLVED
            ),
            "reference_notebook_path": str(
                V0_0_REFERENCE_NOTEBOOK_PATH
            ),
            "reference_notebook_sha256_before": (
                V0_0_REFERENCE_NOTEBOOK_SHA256_BEFORE
            ),
            "reference_notebook_sha256_after": (
                V0_0_REFERENCE_NOTEBOOK_SHA256_AFTER
            ),
            "reference_notebook_byte_count": (
                V0_0_REFERENCE_NOTEBOOK_PATH.stat().st_size
            ),
            "path_inside_v0_0_root": (
                path_is_inside(
                    V0_0_REFERENCE_NOTEBOOK_PATH,
                    V0_0_ROOT_RESOLVED,
                )
            ),
            "notebook_json_valid": True,
            "notebook_modified_by_cell": (
                V0_0_REFERENCE_NOTEBOOK_SHA256_BEFORE
                != V0_0_REFERENCE_NOTEBOOK_SHA256_AFTER
            ),
            "source_authority_role": (
                "READ_ONLY_REFERENCE"
            ),
            "status": "PASS",
        }
    ]
)


V0_0_REFERENCE_RECONCILIATION_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": (
                "v0_0_root_resolved"
            ),
            "severity": "BLOCKING",
            "passed": (
                V0_0_ROOT_RESOLVED.exists()
                and V0_0_ROOT_RESOLVED.is_dir()
            ),
            "evidence": str(
                V0_0_ROOT_RESOLVED
            ),
        },
        {
            "gate": (
                "v0_0_reference_notebook_discovered"
            ),
            "severity": "BLOCKING",
            "passed": (
                V0_0_REFERENCE_NOTEBOOK_PATH.exists()
                and V0_0_REFERENCE_NOTEBOOK_PATH.is_file()
            ),
            "evidence": str(
                V0_0_REFERENCE_NOTEBOOK_PATH
            ),
        },
        {
            "gate": (
                "reference_notebook_inside_immutable_root"
            ),
            "severity": "BLOCKING",
            "passed": path_is_inside(
                V0_0_REFERENCE_NOTEBOOK_PATH,
                V0_0_ROOT_RESOLVED,
            ),
            "evidence": (
                "selected notebook is contained under the "
                "resolved V0.0 root"
            ),
        },
        {
            "gate": (
                "reference_notebook_hash_stable"
            ),
            "severity": "BLOCKING",
            "passed": (
                V0_0_REFERENCE_NOTEBOOK_SHA256_BEFORE
                == V0_0_REFERENCE_NOTEBOOK_SHA256_AFTER
            ),
            "evidence": (
                V0_0_REFERENCE_NOTEBOOK_SHA256_AFTER
            ),
        },
        {
            "gate": (
                "reference_metrics_registered"
            ),
            "severity": "BLOCKING",
            "passed": (
                set(
                    V0_0_REFERENCE_METRIC_CONTRACT[
                        "metric_id"
                    ]
                )
                == {
                    "POOLED_COUNT_FANO_1S",
                    "POOLED_COUNT_ACF_LAG1_100MS",
                }
            ),
            "evidence": (
                f"registered_metrics="
                f"{len(V0_0_REFERENCE_METRIC_CONTRACT)}"
            ),
        },
        {
            "gate": (
                "v0_1_comparison_metrics_finite"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                np.isfinite(
                    V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY[
                        "v0_1_value"
                    ].to_numpy(
                        dtype="float64"
                    )
                ).all()
            ),
            "evidence": (
                "current Fano and partition-safe lag-one ACF "
                "are finite"
            ),
        },
        {
            "gate": (
                "comparison_uses_complete_equal_exposure_bins"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY[
                    "included_exposure_ns"
                ].gt(
                    0
                ).all()
                and V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY[
                    "excluded_tail_exposure_ns"
                ].ge(
                    0
                ).all()
            ),
            "evidence": (
                "right-edge tails are recorded and excluded "
                "from complete-bin reference metrics"
            ),
        },
        {
            "gate": (
                "acf_partition_boundaries_not_linked"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY[
                    "partition_boundary_links_used"
                ].eq(
                    0
                ).all()
            ),
            "evidence": (
                f"excluded_boundary_links="
                f"{V0_1_ANALYTICAL_ACF_EXCLUDED_BOUNDARY_LINK_COUNT}"
            ),
        },
        {
            "gate": (
                "reference_qualitative_direction_reconciled"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
                    "qualitative_direction_reconciled"
                ].all()
            ),
            "evidence": (
                "pooled counts remain overdispersed and "
                "100 ms lag-one dependence remains positive"
            ),
        },
        {
            "gate": (
                "reference_values_within_declared_tolerance"
            ),
            "severity": "ADVISORY",
            "passed": bool(
                V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
                    "within_declared_tolerance"
                ].all()
            ),
            "evidence": (
                "comparison is approximate because V0.1 opens "
                "only DEVELOPMENT and CALIBRATION content"
            ),
        },
        {
            "gate": (
                "v0_0_reference_not_used_for_model_selection"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
                    "formal_model_selection_role"
                ].any()
            ),
            "evidence": (
                "V0.0 metrics are provenance references only"
            ),
        },
        {
            "gate": (
                "timestamp_jitter_absent"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY[
                    "timestamp_jitter_applied"
                ].any()
            ),
            "evidence": (
                "canonical Notebook 04 timing and exact integer "
                "binning are preserved"
            ),
        },
        {
            "gate": (
                "protected_partition_content_remains_absent"
            ),
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)

require(
    V0_0_REFERENCE_RECONCILIATION_GATE_FRAME.loc[
        V0_0_REFERENCE_RECONCILIATION_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one blocking V0.0 reference-reconciliation "
        "gate failed."
    ),
)


display(
    V0_0_REFERENCE_NOTEBOOK_CANDIDATE_AUDIT
)

display(
    V0_0_REFERENCE_NOTEBOOK_AUDIT
)

display(
    V0_0_REFERENCE_METRIC_CONTRACT[
        [
            "metric_id",
            "metric_label",
            "reference_value",
            "literal_present_in_notebook_output",
            "literal_present_in_notebook_source",
            "literal_present_anywhere",
            "comparison_tolerance",
            "tolerance_type",
            "source_role",
            "status",
        ]
    ]
)

display(
    V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY
)

display(
    V0_0_TO_V0_1_REFERENCE_RECONCILIATION[
        [
            "metric_id",
            "reference_value",
            "v0_1_value",
            "signed_difference",
            "absolute_difference",
            "relative_difference",
            "comparison_tolerance",
            "tolerance_type",
            "within_declared_tolerance",
            "qualitative_direction_reconciled",
            "reconciliation_status",
            "comparison_role",
            "status",
        ]
    ]
)

display(
    V0_0_REFERENCE_RECONCILIATION_GATE_FRAME
)

print(
    "The immutable V0.0 development notebook was discovered "
    "without relying on a pre-existing V0_0_ROOT variable."
)
print(
    f"Resolved V0.0 root: {V0_0_ROOT_RESOLVED}"
)
print(
    f"Selected reference notebook: "
    f"{V0_0_REFERENCE_NOTEBOOK_PATH}"
)
print(
    "The reference notebook hash was unchanged before and after "
    "the reconciliation audit."
)
print(
    f"V0.1 analytical pooled one-second Fano factor: "
    f"{V0_1_ANALYTICAL_POOLED_FANO_1S:.6f}."
)
print(
    f"V0.1 analytical pooled 100-millisecond lag-one ACF: "
    f"{V0_1_ANALYTICAL_POOLED_ACF_LAG1_100MS:.6f}."
)
print(
    "The V0.0 values remain read-only provenance references and "
    "were not used for model fitting, hyperparameter selection, "
    "or CALIBRATION decisions."
)
print(
    "VALIDATION and ENGINEERING_HOLDOUT event content remain "
    "unopened."
)
print(
    "Hawkes estimation remains unauthorized pending the formal "
    "Notebook 06 decision ledger."
)

,path,sha256,byte_count,ranking_score,contains_fano_literal,contains_acf_literal,failed_name_flag,selected_flag,status
0,D:\Clown Project\V0.0\Clown Project v0.0.ipynb,79d0d6adecb7167a7a0580e5774d2baa93317ad1cc057c...,1917338,170,True,True,False,True,PASS
1,D:\Clown Project\V0.0\Clown Project v0.0 (fail...,5b3ed48e7d8742ec4c71888ca0f1cb4b8afa054fc24b54...,5891099,-150,False,False,True,False,PASS


,resolved_v0_0_root,reference_notebook_path,reference_notebook_sha256_before,reference_notebook_sha256_after,reference_notebook_byte_count,path_inside_v0_0_root,notebook_json_valid,notebook_modified_by_cell,source_authority_role,status
0,D:\Clown Project\V0.0,D:\Clown Project\V0.0\Clown Project v0.0.ipynb,79d0d6adecb7167a7a0580e5774d2baa93317ad1cc057c...,79d0d6adecb7167a7a0580e5774d2baa93317ad1cc057c...,1917338,True,True,False,READ_ONLY_REFERENCE,PASS


,metric_id,metric_label,reference_value,literal_present_in_notebook_output,literal_present_in_notebook_source,literal_present_anywhere,comparison_tolerance,tolerance_type,source_role,status
0,POOLED_COUNT_FANO_1S,Pooled one-second count Fano factor,2.840352,True,False,True,0.15,RELATIVE,V0_0_REFERENCE_ONLY_NOT_V0_1_AUTHORITY,PASS
1,POOLED_COUNT_ACF_LAG1_100MS,Pooled 100-millisecond lag-one count autocorre...,0.160183,True,False,True,0.08,ABSOLUTE,V0_0_REFERENCE_ONLY_NOT_V0_1_AUTHORITY,PASS


,metric_id,partition_scope,width_ms,lag,observation_count,aligned_pair_count,included_exposure_ns,excluded_tail_exposure_ns,observed_event_count,v0_1_value,partition_boundary_links_used,timestamp_jitter_applied,status,partition_boundary_links_excluded
0,POOLED_COUNT_FANO_1S,DEVELOPMENT_AND_CALIBRATION_COMPLETE_BINS,1000,<NA>,2521,<NA>,2521000000000,1158765800,9493,2.7503221,0,False,PASS,NaN
1,POOLED_COUNT_ACF_LAG1_100MS,DEVELOPMENT_AND_CALIBRATION_PARTITION_SAFE,100,1,25220,25218,2522000000000,158765800,9497,0.12097494,0,False,PASS,1


,metric_id,reference_value,v0_1_value,signed_difference,absolute_difference,relative_difference,comparison_tolerance,tolerance_type,within_declared_tolerance,qualitative_direction_reconciled,reconciliation_status,comparison_role,status
0,POOLED_COUNT_FANO_1S,2.840352,2.7503221,-0.090029875,0.090029875,0.031696731,0.15,RELATIVE,True,True,PASS_APPROXIMATE_REFERENCE_RECONCILIATION,PROVENANCE_AND_FAILURE_MODE_RECONCILIATION_ONLY,PASS
1,POOLED_COUNT_ACF_LAG1_100MS,0.160183,0.12097494,-0.039208062,0.039208062,0.24477043,0.08,ABSOLUTE,True,True,PASS_APPROXIMATE_REFERENCE_RECONCILIATION,PROVENANCE_AND_FAILURE_MODE_RECONCILIATION_ONLY,PASS


,gate,severity,passed,evidence
0,v0_0_root_resolved,BLOCKING,True,D:\Clown Project\V0.0
1,v0_0_reference_notebook_discovered,BLOCKING,True,D:\Clown Project\V0.0\Clown Project v0.0.ipynb
2,reference_notebook_inside_immutable_root,BLOCKING,True,selected notebook is contained under the resol...
3,reference_notebook_hash_stable,BLOCKING,True,79d0d6adecb7167a7a0580e5774d2baa93317ad1cc057c...
4,reference_metrics_registered,BLOCKING,True,registered_metrics=2
5,v0_1_comparison_metrics_finite,BLOCKING,True,current Fano and partition-safe lag-one ACF ar...
6,comparison_uses_complete_equal_exposure_bins,BLOCKING,True,right-edge tails are recorded and excluded fro...
7,acf_partition_boundaries_not_linked,BLOCKING,True,excluded_boundary_links=1
8,reference_qualitative_direction_reconciled,BLOCKING,True,pooled counts remain overdispersed and 100 ms ...
9,reference_values_within_declared_tolerance,ADVISORY,True,comparison is approximate because V0.1 opens o...


The immutable V0.0 development notebook was discovered without relying on a pre-existing V0_0_ROOT variable.
Resolved V0.0 root: D:\Clown Project\V0.0
Selected reference notebook: D:\Clown Project\V0.0\Clown Project v0.0.ipynb
The reference notebook hash was unchanged before and after the reconciliation audit.
V0.1 analytical pooled one-second Fano factor: 2.750322.
V0.1 analytical pooled 100-millisecond lag-one ACF: 0.120975.
The V0.0 values remain read-only provenance references and were not used for model fitting, hyperparameter selection, or CALIBRATION decisions.
VALIDATION and ENGINEERING_HOLDOUT event content remain unopened.
Hawkes estimation remains unauthorized pending the formal Notebook 06 decision ledger.


In [25]:
# ============================================================
# Formal Notebook 06 terminal decision and Notebook 07 handoff
# ============================================================

# ------------------------------------------------------------
# Resolve the V0.1 run ID from the frozen runtime contract
# ------------------------------------------------------------

def nonempty_runtime_text(
    value: Any,
) -> str | None:
    """Return a stripped nonempty string when one is available."""
    if value is None:
        return None

    text = str(
        value
    ).strip()

    return (
        text
        if text
        else None
    )


def resolve_v0_1_run_id() -> tuple[
    str,
    str,
]:
    """
    Resolve the V0.1 run ID without assuming one variable spelling.

    Earlier notebooks use V01_RUN_ID as the frozen identity. Additional
    aliases are accepted only when they contain a valid V0.1 identifier.
    """
    direct_candidates = (
        "V0_1_RUN_ID",
        "V01_RUN_ID",
        "CURRENT_V0_1_RUN_ID",
        "RUN_ID",
    )

    resolved_candidates: list[
        tuple[str, str]
    ] = []

    for variable_name in direct_candidates:
        if variable_name not in globals():
            continue

        candidate_text = nonempty_runtime_text(
            globals()[
                variable_name
            ]
        )

        if (
            candidate_text is not None
            and candidate_text.startswith(
                "v0_1_"
            )
        ):
            resolved_candidates.append(
                (
                    variable_name,
                    candidate_text,
                )
            )

    object_candidates = (
        "NOTEBOOK_CONFIG",
        "CONFIG",
        "RUN_CONFIG",
    )

    object_attributes = (
        "v0_1_run_id",
        "v01_run_id",
        "run_id",
    )

    for object_name in object_candidates:
        if object_name not in globals():
            continue

        runtime_object = globals()[
            object_name
        ]

        for attribute_name in object_attributes:
            if not hasattr(
                runtime_object,
                attribute_name,
            ):
                continue

            candidate_text = nonempty_runtime_text(
                getattr(
                    runtime_object,
                    attribute_name,
                )
            )

            if (
                candidate_text is not None
                and candidate_text.startswith(
                    "v0_1_"
                )
            ):
                resolved_candidates.append(
                    (
                        f"{object_name}.{attribute_name}",
                        candidate_text,
                    )
                )

    if (
        "COMBINED_OUTPUT_PREFIX"
        in globals()
    ):
        combined_prefix = nonempty_runtime_text(
            globals()[
                "COMBINED_OUTPUT_PREFIX"
            ]
        )

        if (
            combined_prefix is not None
            and "__v0_1_"
            in combined_prefix
        ):
            candidate_text = (
                "v0_1_"
                + combined_prefix.split(
                    "__v0_1_",
                    maxsplit=1,
                )[1]
            )

            resolved_candidates.append(
                (
                    "COMBINED_OUTPUT_PREFIX",
                    candidate_text,
                )
            )

    registered_fallback = (
        "v0_1_20260714T090616Z_e82325081a81"
    )

    resolved_candidates.append(
        (
            "REGISTERED_PROJECT_FALLBACK",
            registered_fallback,
        )
    )

    distinct_values = {
        candidate_value
        for _, candidate_value in resolved_candidates
    }

    require(
        len(
            distinct_values
        )
        == 1,
        (
            "Conflicting V0.1 run IDs were found in the runtime: "
            f"{sorted(distinct_values)}"
        ),
    )

    source_name, resolved_value = (
        resolved_candidates[0]
    )

    require(
        resolved_value.startswith(
            "v0_1_"
        ),
        (
            "The resolved V0.1 run ID does not use the required "
            "v0_1_ prefix."
        ),
    )

    return (
        resolved_value,
        source_name,
    )


(
    V0_1_RUN_ID,
    V0_1_RUN_ID_RESOLUTION_SOURCE,
) = resolve_v0_1_run_id()

RESOLVED_V0_1_RUN_ID: Final[
    str
] = V0_1_RUN_ID

require(
    V0_1_RUN_ID
    == "v0_1_20260714T090616Z_e82325081a81",
    (
        "The resolved V0.1 run ID differs from the frozen "
        "Notebook 00 run contract."
    ),
)

if (
    "V01_RUN_ID"
    in globals()
):
    require(
        V0_1_RUN_ID
        == str(
            V01_RUN_ID
        ),
        (
            "V0_1_RUN_ID does not reconcile with the frozen "
            "V01_RUN_ID constant."
        ),
    )

if (
    "COMBINED_OUTPUT_PREFIX"
    in globals()
):
    require(
        str(
            COMBINED_OUTPUT_PREFIX
        ).endswith(
            V0_1_RUN_ID
        ),
        (
            "The combined output prefix does not end with the "
            "resolved V0.1 run ID."
        ),
    )


V0_1_RUN_ID_RESOLUTION_AUDIT = pd.DataFrame(
    [
        {
            "canonical_field": (
                "v0_1_run_id"
            ),
            "resolved_value": (
                V0_1_RUN_ID
            ),
            "resolution_source": (
                V0_1_RUN_ID_RESOLUTION_SOURCE
            ),
            "expected_value": (
                "v0_1_20260714T090616Z_e82325081a81"
            ),
            "matches_frozen_contract": (
                V0_1_RUN_ID
                == "v0_1_20260714T090616Z_e82325081a81"
            ),
            "combined_output_prefix_reconciled": (
                str(
                    COMBINED_OUTPUT_PREFIX
                ).endswith(
                    V0_1_RUN_ID
                )
                if (
                    "COMBINED_OUTPUT_PREFIX"
                    in globals()
                )
                else pd.NA
            ),
            "status": "PASS",
        }
    ]
)


# ------------------------------------------------------------
# Verify that the scientific decision state already exists
# ------------------------------------------------------------

required_decision_runtime_names = (
    "NOTEBOOK_06_AUTHORIZATION_STATE",
    "NOTEBOOK_06_TERMINAL_STATUS",
    "DECISION_FORMAL_COMPARATOR_MODEL_ID",
    "DECISION_PRIMARY_SIMPLE_MODEL_ID",
    "PRIMARY_HAWKES_SCOPE",
    "SIMPLE_BASELINE_MATERIALLY_IMPROVES",
    "SIMPLE_BASELINE_SUFFICIENT",
    "REMAINING_SELF_DEPENDENCE_EVIDENCE",
    "DIRECTIONAL_RESIDUAL_CROSS_EVIDENCE_STRONG",
    "CONDITIONAL_CROSS_RESPONSE_EVIDENCE_PRESENT",
    "NOTEBOOK_07_HAWKES_ESTIMATION_AUTHORIZED",
    "NOTEBOOK_07_RESTRICTED_FIRST_REQUIRED",
)

missing_decision_runtime_names = [
    variable_name
    for variable_name
    in required_decision_runtime_names
    if variable_name
    not in globals()
]

require(
    not missing_decision_runtime_names,
    (
        "The terminal decision cannot be finalized because these "
        "scientific decision variables are absent: "
        f"{missing_decision_runtime_names}"
    ),
)


authorization_state = str(
    NOTEBOOK_06_AUTHORIZATION_STATE
)

terminal_status = str(
    NOTEBOOK_06_TERMINAL_STATUS
)

formal_comparator_model_id = str(
    DECISION_FORMAL_COMPARATOR_MODEL_ID
)

primary_simple_model_id = str(
    DECISION_PRIMARY_SIMPLE_MODEL_ID
)

primary_hawkes_scope = str(
    PRIMARY_HAWKES_SCOPE
)

simple_baseline_materially_improves = bool(
    SIMPLE_BASELINE_MATERIALLY_IMPROVES
)

simple_baseline_sufficient = bool(
    SIMPLE_BASELINE_SUFFICIENT
)

remaining_self_dependence_evidence = bool(
    REMAINING_SELF_DEPENDENCE_EVIDENCE
)

directional_residual_cross_evidence_strong = bool(
    DIRECTIONAL_RESIDUAL_CROSS_EVIDENCE_STRONG
)

conditional_cross_response_evidence_present = bool(
    CONDITIONAL_CROSS_RESPONSE_EVIDENCE_PRESENT
)

notebook_07_authorized = bool(
    NOTEBOOK_07_HAWKES_ESTIMATION_AUTHORIZED
)

restricted_first_required = bool(
    NOTEBOOK_07_RESTRICTED_FIRST_REQUIRED
)


allowed_authorization_status_pairs: Final[
    Mapping[str, str]
] = {
    "AUTHORIZED_BIVARIATE": (
        "PASS_HAWKES_BIVARIATE_AUTHORIZED"
    ),
    "AUTHORIZED_RESTRICTED_FIRST": (
        "PASS_HAWKES_RESTRICTED_FIRST"
    ),
    "CONDITIONAL_AUTHORIZATION": (
        "CONDITIONAL_PASS_HAWKES_DIAGNOSTIC_ONLY"
    ),
    "NOT_AUTHORIZED": (
        "PASS_SIMPLE_BASELINE_SUFFICIENT_HAWKES_NOT_AUTHORIZED"
    ),
}

require(
    authorization_state
    in allowed_authorization_status_pairs,
    (
        "Notebook 06 produced an unregistered authorization "
        f"state: {authorization_state}"
    ),
)

require(
    terminal_status
    == allowed_authorization_status_pairs[
        authorization_state
    ],
    (
        "Notebook 06 authorization state and terminal status "
        "are inconsistent."
    ),
)


# ------------------------------------------------------------
# Construct the Notebook 07 execution contract when necessary
# ------------------------------------------------------------

if (
    "NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT"
    not in globals()
):
    NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT = pd.DataFrame(
        [
            {
                "contract_section": (
                    "DOWNSTREAM_AUTHORIZATION"
                ),
                "field": (
                    "notebook_07_authorized"
                ),
                "value": (
                    notebook_07_authorized
                ),
                "blocking": True,
                "interpretation": (
                    "Notebook 07 may begin only under the "
                    "formal Notebook 06 terminal decision."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "MODEL_SCOPE"
                ),
                "field": (
                    "primary_hawkes_scope"
                ),
                "value": (
                    primary_hawkes_scope
                ),
                "blocking": True,
                "interpretation": (
                    "Notebook 07 must obey the authorized "
                    "restricted or bivariate scope."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "MODEL_SCOPE"
                ),
                "field": (
                    "restricted_first_required"
                ),
                "value": (
                    restricted_first_required
                ),
                "blocking": True,
                "interpretation": (
                    "Restricted-first estimation is mandatory "
                    "when cross-side evidence is not strong "
                    "enough for immediate unrestricted fitting."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "ESTIMATION_DATA"
                ),
                "field": (
                    "fit_partition"
                ),
                "value": "DEVELOPMENT",
                "blocking": True,
                "interpretation": (
                    "Hawkes parameters and candidate choices "
                    "must be estimated using DEVELOPMENT only."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "ESTIMATION_DATA"
                ),
                "field": (
                    "locked_evaluation_partition"
                ),
                "value": "CALIBRATION",
                "blocking": True,
                "interpretation": (
                    "CALIBRATION may evaluate frozen candidates "
                    "but may not select kernels or parameters."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "PARTITION_FIREWALL"
                ),
                "field": (
                    "validation_content_access"
                ),
                "value": "FORBIDDEN",
                "blocking": True,
                "interpretation": (
                    "VALIDATION event content remains unopened."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "PARTITION_FIREWALL"
                ),
                "field": (
                    "engineering_holdout_content_access"
                ),
                "value": "FORBIDDEN",
                "blocking": True,
                "interpretation": (
                    "ENGINEERING_HOLDOUT event content remains "
                    "unopened."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "EVENT_INTERFACE"
                ),
                "field": (
                    "timestamp_authority"
                ),
                "value": (
                    "NOTEBOOK_04_PRIMARY_SCORING_BATCHES"
                ),
                "blocking": True,
                "interpretation": (
                    "Canonical Notebook 04 event_time_ns remains "
                    "the exact timestamp authority."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "EVENT_INTERFACE"
                ),
                "field": (
                    "simultaneous_batch_scoring"
                ),
                "value": (
                    "STRICT_PRE_BATCH_HISTORY"
                ),
                "blocking": True,
                "interpretation": (
                    "All events at one exact batch time are "
                    "scored before that batch updates intensity."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "EVENT_INTERFACE"
                ),
                "field": (
                    "zero_lag_within_batch_excitation"
                ),
                "value": False,
                "blocking": True,
                "interpretation": (
                    "No fake ordering or zero-lag excitation is "
                    "permitted inside an exact-time batch."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "MODEL_CLASS"
                ),
                "field": (
                    "ordinary_hawkes_first"
                ),
                "value": True,
                "blocking": True,
                "interpretation": (
                    "Ordinary exponential-kernel Hawkes models "
                    "must be evaluated before any extension."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "MODEL_CLASS"
                ),
                "field": (
                    "state_dependent_hawkes_authorized"
                ),
                "value": False,
                "blocking": True,
                "interpretation": (
                    "State-dependent Hawkes remains outside the "
                    "authorized Notebook 07 scope."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "MODEL_VALIDITY"
                ),
                "field": (
                    "stability_requirement"
                ),
                "value": (
                    "SPECTRAL_RADIUS_STRICTLY_LESS_THAN_ONE"
                ),
                "blocking": True,
                "interpretation": (
                    "Every accepted Hawkes candidate must "
                    "satisfy the registered stability condition."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "MODEL_VALIDITY"
                ),
                "field": (
                    "intensity_requirement"
                ),
                "value": (
                    "FINITE_AND_STRICTLY_POSITIVE"
                ),
                "blocking": True,
                "interpretation": (
                    "All fitted and replayed intensities must be "
                    "finite and strictly positive."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "MODEL_SELECTION"
                ),
                "field": (
                    "calibration_used_for_selection"
                ),
                "value": False,
                "blocking": True,
                "interpretation": (
                    "Kernel families, decay grids, restrictions, "
                    "and optimizer choices are frozen using "
                    "DEVELOPMENT only."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "CLAIM_LIMIT"
                ),
                "field": (
                    "hawkes_superiority_claim_authorized"
                ),
                "value": False,
                "blocking": True,
                "interpretation": (
                    "Notebook 07 estimation alone cannot establish "
                    "incremental superiority over the simple "
                    "baseline."
                ),
                "status": "PASS",
            },
            {
                "contract_section": (
                    "CLAIM_LIMIT"
                ),
                "field": (
                    "strategy_or_market_making_claim_authorized"
                ),
                "value": False,
                "blocking": True,
                "interpretation": (
                    "No quoting, execution, fill, P&L, Sharpe, "
                    "or deployment claim is authorized."
                ),
                "status": "PASS",
            },
        ]
    )
else:
    NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT = (
        NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT.copy()
    )

    require(
        isinstance(
            NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT,
            pd.DataFrame,
        )
        and not NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT.empty,
        (
            "The existing Notebook 07 Hawkes execution contract "
            "is empty or invalid."
        ),
    )

    if (
        "status"
        not in NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT.columns
    ):
        NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT[
            "status"
        ] = "PASS"


# ------------------------------------------------------------
# Scientific terminal evidence summary
# ------------------------------------------------------------

v0_0_reference_reconciled = False

if (
    "V0_0_TO_V0_1_REFERENCE_RECONCILIATION"
    in globals()
):
    reconciliation_frame = (
        V0_0_TO_V0_1_REFERENCE_RECONCILIATION
    )

    if (
        isinstance(
            reconciliation_frame,
            pd.DataFrame,
        )
        and not reconciliation_frame.empty
        and (
            "qualitative_direction_reconciled"
            in reconciliation_frame.columns
        )
    ):
        v0_0_reference_reconciled = bool(
            reconciliation_frame[
                "qualitative_direction_reconciled"
            ].astype(
                bool
            ).all()
        )


NOTEBOOK_06_TERMINAL_EVIDENCE_LEDGER = pd.DataFrame(
    [
        {
            "evidence_channel": (
                "LOCKED_SIMPLE_BASELINE_IMPROVEMENT"
            ),
            "evidence_flag": (
                simple_baseline_materially_improves
            ),
            "decision_role": (
                "DETERMINES_WHETHER_STATIC_POISSON_IS_ADEQUATE"
            ),
            "interpretation": (
                "The DEVELOPMENT-selected adaptive simple "
                "baseline materially improves the locked "
                "CALIBRATION count score over the formal "
                "deterministic comparator."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "SIMPLE_BASELINE_SUFFICIENCY"
            ),
            "evidence_flag": (
                simple_baseline_sufficient
            ),
            "decision_role": (
                "CAN_BLOCK_HAWKES_WHEN_TRUE"
            ),
            "interpretation": (
                "A sufficient simple baseline would end the "
                "point-process escalation. Remaining diagnostics "
                "determine whether escalation is still warranted."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "REMAINING_SELF_DEPENDENCE"
            ),
            "evidence_flag": (
                remaining_self_dependence_evidence
            ),
            "decision_role": (
                "SUPPORTS_RESTRICTED_SELF_EXCITATION_TEST"
            ),
            "interpretation": (
                "Repeated serial and time-rescaling departures "
                "remain after the primary simple baseline."
            ),
            "formal_hawkes_authorization_flag": (
                authorization_state
                in {
                    "AUTHORIZED_RESTRICTED_FIRST",
                    "AUTHORIZED_BIVARIATE",
                }
            ),
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "DIRECTIONAL_RESIDUAL_CROSS_DEPENDENCE"
            ),
            "evidence_flag": (
                directional_residual_cross_evidence_strong
            ),
            "decision_role": (
                "SUPPORTS_IMMEDIATE_BIVARIATE_SCOPE_WHEN_STRONG"
            ),
            "interpretation": (
                "Repeated directional residual cross-dependence "
                "is required for immediate unrestricted "
                "bivariate authorization."
            ),
            "formal_hawkes_authorization_flag": (
                authorization_state
                == "AUTHORIZED_BIVARIATE"
            ),
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "CONDITIONAL_CROSS_RESPONSE"
            ),
            "evidence_flag": (
                conditional_cross_response_evidence_present
            ),
            "decision_role": (
                "DIAGNOSTIC_SUPPORT_ONLY"
            ),
            "interpretation": (
                "Conditional response excess is informative but "
                "cannot independently authorize cross-excitation "
                "because source-centered windows overlap."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "V0_0_REFERENCE_RECONCILIATION"
            ),
            "evidence_flag": (
                v0_0_reference_reconciled
            ),
            "decision_role": (
                "PROVENANCE_AND_FAILURE_MODE_RECONCILIATION"
            ),
            "interpretation": (
                "V0.1 reproduces the qualitative V0.0 findings "
                "of overdispersion and positive short-lag "
                "dependence without using V0.0 for selection."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
        {
            "evidence_channel": (
                "PROTECTED_PARTITION_FIREWALL"
            ),
            "evidence_flag": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "decision_role": (
                "BLOCKING_CAUSAL_RESEARCH_CONTRACT"
            ),
            "interpretation": (
                "VALIDATION and ENGINEERING_HOLDOUT event "
                "content remain unopened."
            ),
            "formal_hawkes_authorization_flag": False,
            "status": "PASS",
        },
    ]
)


# ------------------------------------------------------------
# Formal terminal decision
# ------------------------------------------------------------

NOTEBOOK_06_TERMINAL_DECISION = pd.DataFrame(
    [
        {
            "notebook_name": (
                NOTEBOOK_NAME
            ),
            "notebook_stage": (
                NOTEBOOK_STAGE
            ),
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "v0_1_run_id": (
                V0_1_RUN_ID
            ),
            "v0_1_run_id_resolution_source": (
                V0_1_RUN_ID_RESOLUTION_SOURCE
            ),
            "authorization_state": (
                authorization_state
            ),
            "terminal_status": (
                terminal_status
            ),
            "formal_count_comparator": (
                formal_comparator_model_id
            ),
            "primary_simple_count_baseline": (
                primary_simple_model_id
            ),
            "primary_hawkes_scope": (
                primary_hawkes_scope
            ),
            "simple_baseline_materially_improves": (
                simple_baseline_materially_improves
            ),
            "simple_baseline_sufficient": (
                simple_baseline_sufficient
            ),
            "remaining_self_dependence_evidence": (
                remaining_self_dependence_evidence
            ),
            "directional_residual_cross_evidence_strong": (
                directional_residual_cross_evidence_strong
            ),
            "conditional_cross_response_evidence_present": (
                conditional_cross_response_evidence_present
            ),
            "notebook_07_authorized": (
                notebook_07_authorized
            ),
            "restricted_first_required": (
                restricted_first_required
            ),
            "state_dependent_hawkes_authorized": False,
            "hawkes_superiority_claim_authorized": False,
            "strategy_claim_authorized": False,
            "validation_content_loaded": (
                PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
            ),
            "engineering_holdout_content_loaded": (
                PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "status": "PASS",
        }
    ]
)


NEXT_AUTHORIZED_NOTEBOOK = (
    "07_HAWKES_ESTIMATION.ipynb"
    if notebook_07_authorized
    else pd.NA
)

NOTEBOOK_06_FINAL_STATUS = (
    terminal_status
)

NOTEBOOK_06_DECISION_COMPLETE = True


# ------------------------------------------------------------
# Decision gates
# ------------------------------------------------------------

authorization_requires_notebook_07 = (
    authorization_state
    in {
        "AUTHORIZED_BIVARIATE",
        "AUTHORIZED_RESTRICTED_FIRST",
        "CONDITIONAL_AUTHORIZATION",
    }
)

authorization_requires_restricted_first = (
    authorization_state
    == "AUTHORIZED_RESTRICTED_FIRST"
)

authorization_requires_strong_cross_evidence = (
    authorization_state
    == "AUTHORIZED_BIVARIATE"
)

NOTEBOOK_06_DECISION_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": (
                "v0_1_run_id_resolved"
            ),
            "severity": "BLOCKING",
            "passed": (
                V0_1_RUN_ID
                == "v0_1_20260714T090616Z_e82325081a81"
            ),
            "evidence": (
                f"source={V0_1_RUN_ID_RESOLUTION_SOURCE}; "
                f"value={V0_1_RUN_ID}"
            ),
        },
        {
            "gate": (
                "authorization_state_registered"
            ),
            "severity": "BLOCKING",
            "passed": (
                authorization_state
                in allowed_authorization_status_pairs
            ),
            "evidence": (
                authorization_state
            ),
        },
        {
            "gate": (
                "authorization_and_terminal_status_consistent"
            ),
            "severity": "BLOCKING",
            "passed": (
                terminal_status
                == allowed_authorization_status_pairs[
                    authorization_state
                ]
            ),
            "evidence": (
                f"{authorization_state} -> {terminal_status}"
            ),
        },
        {
            "gate": (
                "exactly_one_terminal_decision_recorded"
            ),
            "severity": "BLOCKING",
            "passed": (
                len(
                    NOTEBOOK_06_TERMINAL_DECISION
                )
                == 1
            ),
            "evidence": (
                f"decision_rows="
                f"{len(NOTEBOOK_06_TERMINAL_DECISION)}"
            ),
        },
        {
            "gate": (
                "formal_comparator_registered"
            ),
            "severity": "BLOCKING",
            "passed": (
                formal_comparator_model_id
                == "D0_SIDE_CONSTANT_POISSON"
            ),
            "evidence": (
                formal_comparator_model_id
            ),
        },
        {
            "gate": (
                "primary_simple_baseline_registered"
            ),
            "severity": "BLOCKING",
            "passed": (
                primary_simple_model_id
                == "E_SIDE_EWMA_250MS_POISSON"
            ),
            "evidence": (
                primary_simple_model_id
            ),
        },
        {
            "gate": (
                "notebook_07_authorization_consistent"
            ),
            "severity": "BLOCKING",
            "passed": (
                notebook_07_authorized
                == authorization_requires_notebook_07
            ),
            "evidence": (
                f"authorization_state={authorization_state}; "
                f"notebook_07_authorized={notebook_07_authorized}"
            ),
        },
        {
            "gate": (
                "restricted_first_flag_consistent"
            ),
            "severity": "BLOCKING",
            "passed": (
                restricted_first_required
                == authorization_requires_restricted_first
            ),
            "evidence": (
                f"restricted_first_required="
                f"{restricted_first_required}"
            ),
        },
        {
            "gate": (
                "bivariate_authorization_requires_cross_evidence"
            ),
            "severity": "BLOCKING",
            "passed": (
                not authorization_requires_strong_cross_evidence
                or directional_residual_cross_evidence_strong
            ),
            "evidence": (
                f"authorization_state={authorization_state}; "
                "directional_residual_cross_evidence_strong="
                f"{directional_residual_cross_evidence_strong}"
            ),
        },
        {
            "gate": (
                "restricted_authorization_requires_self_dependence"
            ),
            "severity": "BLOCKING",
            "passed": (
                authorization_state
                != "AUTHORIZED_RESTRICTED_FIRST"
                or remaining_self_dependence_evidence
            ),
            "evidence": (
                "remaining_self_dependence_evidence="
                f"{remaining_self_dependence_evidence}"
            ),
        },
        {
            "gate": (
                "simple_baseline_sufficiency_blocks_authorization"
            ),
            "severity": "BLOCKING",
            "passed": (
                not simple_baseline_sufficient
                or authorization_state
                == "NOT_AUTHORIZED"
            ),
            "evidence": (
                f"simple_baseline_sufficient="
                f"{simple_baseline_sufficient}; "
                f"authorization_state={authorization_state}"
            ),
        },
        {
            "gate": (
                "not_authorized_state_blocks_notebook_07"
            ),
            "severity": "BLOCKING",
            "passed": (
                authorization_state
                != "NOT_AUTHORIZED"
                or not notebook_07_authorized
            ),
            "evidence": (
                f"authorization_state={authorization_state}; "
                f"notebook_07_authorized={notebook_07_authorized}"
            ),
        },
        {
            "gate": (
                "state_dependent_hawkes_remains_unauthorized"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not NOTEBOOK_06_TERMINAL_DECISION[
                    "state_dependent_hawkes_authorized"
                ].any()
            ),
            "evidence": (
                "state_dependent_hawkes_authorized=False"
            ),
        },
        {
            "gate": (
                "hawkes_superiority_claim_remains_unauthorized"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not NOTEBOOK_06_TERMINAL_DECISION[
                    "hawkes_superiority_claim_authorized"
                ].any()
            ),
            "evidence": (
                "Notebook 07 estimation must be followed by "
                "diagnostics and incremental-information testing"
            ),
        },
        {
            "gate": (
                "strategy_claim_remains_unauthorized"
            ),
            "severity": "BLOCKING",
            "passed": bool(
                not NOTEBOOK_06_TERMINAL_DECISION[
                    "strategy_claim_authorized"
                ].any()
            ),
            "evidence": (
                "no quoting, fills, P&L, Sharpe, or deployment "
                "claim is authorized"
            ),
        },
        {
            "gate": (
                "notebook_07_execution_contract_complete"
            ),
            "severity": "BLOCKING",
            "passed": (
                isinstance(
                    NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT,
                    pd.DataFrame,
                )
                and not NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT.empty
                and NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT[
                    "status"
                ].eq(
                    "PASS"
                ).all()
            ),
            "evidence": (
                f"contract_rows="
                f"{len(NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT)}"
            ),
        },
        {
            "gate": (
                "protected_partition_content_remains_absent"
            ),
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
    ]
)


require(
    NOTEBOOK_06_DECISION_GATE_FRAME.loc[
        NOTEBOOK_06_DECISION_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one blocking Notebook 06 terminal-decision "
        "gate failed."
    ),
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    V0_1_RUN_ID_RESOLUTION_AUDIT
)

display(
    NOTEBOOK_06_TERMINAL_EVIDENCE_LEDGER
)

display(
    NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT
)

display(
    NOTEBOOK_06_TERMINAL_DECISION
)

display(
    NOTEBOOK_06_DECISION_GATE_FRAME
)

print(
    f"Notebook 06 terminal status: "
    f"{NOTEBOOK_06_TERMINAL_STATUS}."
)
print(
    f"Formal authorization state: "
    f"{NOTEBOOK_06_AUTHORIZATION_STATE}."
)
print(
    f"Formal count comparator: "
    f"{DECISION_FORMAL_COMPARATOR_MODEL_ID}."
)
print(
    f"Primary simple count baseline: "
    f"{DECISION_PRIMARY_SIMPLE_MODEL_ID}."
)
print(
    f"Notebook 07 authorized: "
    f"{NOTEBOOK_07_HAWKES_ESTIMATION_AUTHORIZED}."
)
print(
    f"Authorized Hawkes scope: "
    f"{PRIMARY_HAWKES_SCOPE}."
)
print(
    f"Restricted-first estimation required: "
    f"{NOTEBOOK_07_RESTRICTED_FIRST_REQUIRED}."
)
print(
    "State-dependent Hawkes remains unauthorized."
)
print(
    "No Hawkes-superiority, strategy, quoting, execution, "
    "fill, P&L, Sharpe, or deployment claim is authorized."
)
print(
    "VALIDATION and ENGINEERING_HOLDOUT event content remain "
    "unopened."
)

,canonical_field,resolved_value,resolution_source,expected_value,matches_frozen_contract,combined_output_prefix_reconciled,status
0,v0_1_run_id,v0_1_20260714T090616Z_e82325081a81,V01_RUN_ID,v0_1_20260714T090616Z_e82325081a81,True,True,PASS


,evidence_channel,evidence_flag,decision_role,interpretation,formal_hawkes_authorization_flag,status
0,LOCKED_SIMPLE_BASELINE_IMPROVEMENT,True,DETERMINES_WHETHER_STATIC_POISSON_IS_ADEQUATE,The DEVELOPMENT-selected adaptive simple basel...,False,PASS
1,SIMPLE_BASELINE_SUFFICIENCY,False,CAN_BLOCK_HAWKES_WHEN_TRUE,A sufficient simple baseline would end the poi...,False,PASS
2,REMAINING_SELF_DEPENDENCE,True,SUPPORTS_RESTRICTED_SELF_EXCITATION_TEST,Repeated serial and time-rescaling departures ...,True,PASS
3,DIRECTIONAL_RESIDUAL_CROSS_DEPENDENCE,False,SUPPORTS_IMMEDIATE_BIVARIATE_SCOPE_WHEN_STRONG,Repeated directional residual cross-dependence...,False,PASS
4,CONDITIONAL_CROSS_RESPONSE,True,DIAGNOSTIC_SUPPORT_ONLY,Conditional response excess is informative but...,False,PASS
5,V0_0_REFERENCE_RECONCILIATION,True,PROVENANCE_AND_FAILURE_MODE_RECONCILIATION,V0.1 reproduces the qualitative V0.0 findings ...,False,PASS
6,PROTECTED_PARTITION_FIREWALL,True,BLOCKING_CAUSAL_RESEARCH_CONTRACT,VALIDATION and ENGINEERING_HOLDOUT event conte...,False,PASS


,contract_item,required_value,blocking,status
0,authorized_notebook,07_HAWKES_ESTIMATION.ipynb,True,PASS
1,authorization_state,AUTHORIZED_RESTRICTED_FIRST,True,PASS
2,primary_model_scope,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,True,PASS
3,event_representation,SAME_MS_SAME_SIDE_BURSTS,True,PASS
4,timestamp_authority,NOTEBOOK_04_EVENT_TIME_NS,True,PASS
5,timestamp_mutation,FORBIDDEN,True,PASS
6,simultaneous_batch_scoring,ALL_EVENTS_AT_ONE_EXACT_TIME_SCORE_AGAINST_STR...,True,PASS
7,within_batch_zero_lag_excitation,FORBIDDEN,True,PASS
8,post_score_batch_update,APPLY_ALL_BATCH_EVENTS_ONLY_AFTER_ALL_BATCH_CO...,True,PASS
9,parameter_fit_partition,DEVELOPMENT_ONLY,True,PASS


,notebook_name,notebook_stage,source_run_prefix,v0_1_run_id,v0_1_run_id_resolution_source,authorization_state,terminal_status,formal_count_comparator,primary_simple_count_baseline,primary_hawkes_scope,simple_baseline_materially_improves,simple_baseline_sufficient,remaining_self_dependence_evidence,directional_residual_cross_evidence_strong,conditional_cross_response_evidence_present,notebook_07_authorized,restricted_first_required,state_dependent_hawkes_authorized,hawkes_superiority_claim_authorized,strategy_claim_authorized,validation_content_loaded,engineering_holdout_content_loaded,status
0,06_POINT_PROCESS_BASELINES.ipynb,POINT_PROCESS_BASELINES,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,V01_RUN_ID,AUTHORIZED_RESTRICTED_FIRST,PASS_HAWKES_RESTRICTED_FIRST,D0_SIDE_CONSTANT_POISSON,E_SIDE_EWMA_250MS_POISSON,DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES,True,False,True,False,True,True,True,False,False,False,False,False,PASS


,gate,severity,passed,evidence
0,v0_1_run_id_resolved,BLOCKING,True,source=V01_RUN_ID; value=v0_1_20260714T090616Z...
1,authorization_state_registered,BLOCKING,True,AUTHORIZED_RESTRICTED_FIRST
2,authorization_and_terminal_status_consistent,BLOCKING,True,AUTHORIZED_RESTRICTED_FIRST -> PASS_HAWKES_RES...
3,exactly_one_terminal_decision_recorded,BLOCKING,True,decision_rows=1
4,formal_comparator_registered,BLOCKING,True,D0_SIDE_CONSTANT_POISSON
5,primary_simple_baseline_registered,BLOCKING,True,E_SIDE_EWMA_250MS_POISSON
6,notebook_07_authorization_consistent,BLOCKING,True,authorization_state=AUTHORIZED_RESTRICTED_FIRS...
7,restricted_first_flag_consistent,BLOCKING,True,restricted_first_required=True
8,bivariate_authorization_requires_cross_evidence,BLOCKING,True,authorization_state=AUTHORIZED_RESTRICTED_FIRS...
9,restricted_authorization_requires_self_dependence,BLOCKING,True,remaining_self_dependence_evidence=True


Notebook 06 terminal status: PASS_HAWKES_RESTRICTED_FIRST.
Formal authorization state: AUTHORIZED_RESTRICTED_FIRST.
Formal count comparator: D0_SIDE_CONSTANT_POISSON.
Primary simple count baseline: E_SIDE_EWMA_250MS_POISSON.
Notebook 07 authorized: True.
Authorized Hawkes scope: DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES.
Restricted-first estimation required: True.
State-dependent Hawkes remains unauthorized.
No Hawkes-superiority, strategy, quoting, execution, fill, P&L, Sharpe, or deployment claim is authorized.
VALIDATION and ENGINEERING_HOLDOUT event content remain unopened.


In [26]:
# ============================================================
# Persist Notebook 06 outputs, verify readback, and hand off
# to 07_HAWKES_ESTIMATION.ipynb
# ============================================================

from datetime import datetime, timezone
from tempfile import NamedTemporaryFile


# ------------------------------------------------------------
# Resolve authoritative V0.1 output directories
# ------------------------------------------------------------

def resolve_runtime_path(
    candidate_names: Sequence[str],
    fallback: Path,
) -> Path:
    """Resolve the first nonempty runtime path or use a fallback."""
    for candidate_name in candidate_names:
        if candidate_name not in globals():
            continue

        candidate_value = globals()[candidate_name]

        if candidate_value is None:
            continue

        candidate_text = str(candidate_value).strip()

        if candidate_text:
            return Path(candidate_text)

    return fallback


V0_1_PERSISTENCE_ROOT: Final[Path] = resolve_runtime_path(
    (
        "V0_1_ROOT",
        "V01_ROOT",
        "V0_1_PROJECT_ROOT",
    ),
    Path(r"D:\Clown Project\V0.1"),
)

V0_0_IMMUTABLE_ROOT: Final[Path] = resolve_runtime_path(
    (
        "V0_0_ROOT",
        "V00_ROOT",
        "V0_0_PROJECT_ROOT",
    ),
    Path(r"D:\Clown Project\V0.0"),
)

V0_1_ARTIFACT_ROOT: Final[Path] = resolve_runtime_path(
    (
        "V0_1_ARTIFACT_ROOT",
        "V01_ARTIFACT_ROOT",
        "ARTIFACT_ROOT",
    ),
    V0_1_PERSISTENCE_ROOT / "artifacts",
)

NOTEBOOK_06_TABLE_DIR: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "audit_tables"
    / "06_point_process_baselines"
)

NOTEBOOK_06_DIAGNOSTIC_DIR: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "diagnostics"
    / "06_point_process_baselines"
)

NOTEBOOK_06_MANIFEST_DIR: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "manifests"
)

NOTEBOOK_06_HANDOFF_DIR: Final[Path] = (
    V0_1_ARTIFACT_ROOT
    / "handoff"
)

NOTEBOOK_06_LOG_DIR: Final[Path] = resolve_runtime_path(
    (
        "V0_1_LOG_ROOT",
        "V01_LOG_ROOT",
        "LOG_ROOT",
    ),
    V0_1_PERSISTENCE_ROOT / "logs",
)


for directory_path in (
    NOTEBOOK_06_TABLE_DIR,
    NOTEBOOK_06_DIAGNOSTIC_DIR,
    NOTEBOOK_06_MANIFEST_DIR,
    NOTEBOOK_06_HANDOFF_DIR,
    NOTEBOOK_06_LOG_DIR,
):
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )


def path_is_inside(
    candidate_path: Path,
    parent_path: Path,
) -> bool:
    """Return whether candidate_path is contained by parent_path."""
    candidate_resolved = candidate_path.resolve()
    parent_resolved = parent_path.resolve()

    try:
        candidate_resolved.relative_to(
            parent_resolved
        )
    except ValueError:
        return False

    return True


for directory_path in (
    NOTEBOOK_06_TABLE_DIR,
    NOTEBOOK_06_DIAGNOSTIC_DIR,
    NOTEBOOK_06_MANIFEST_DIR,
    NOTEBOOK_06_HANDOFF_DIR,
    NOTEBOOK_06_LOG_DIR,
):
    require(
        path_is_inside(
            directory_path,
            V0_1_PERSISTENCE_ROOT,
        ),
        (
            "Notebook 06 attempted to create an output directory "
            "outside the V0.1 root: "
            f"{directory_path}"
        ),
    )

    require(
        not path_is_inside(
            directory_path,
            V0_0_IMMUTABLE_ROOT,
        ),
        (
            "Notebook 06 attempted to create an output directory "
            "inside the immutable V0.0 tree: "
            f"{directory_path}"
        ),
    )


# ------------------------------------------------------------
# Canonical serialization and hashing helpers
# ------------------------------------------------------------

def json_ready(
    value: Any,
) -> Any:
    """Convert common scientific Python objects to JSON values."""
    if value is None:
        return None

    if value is pd.NA:
        return None

    if isinstance(
        value,
        (
            str,
            bool,
            int,
        ),
    ):
        return value

    if isinstance(
        value,
        float,
    ):
        if not math.isfinite(value):
            return None

        return value

    if isinstance(
        value,
        (
            np.integer,
            np.bool_,
        ),
    ):
        return value.item()

    if isinstance(
        value,
        np.floating,
    ):
        scalar_value = float(value)

        if not math.isfinite(
            scalar_value
        ):
            return None

        return scalar_value

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    if isinstance(
        value,
        (
            pd.Timestamp,
            datetime,
        ),
    ):
        return value.isoformat()

    if isinstance(
        value,
        np.datetime64,
    ):
        return pd.Timestamp(
            value
        ).isoformat()

    if isinstance(
        value,
        Mapping,
    ):
        return {
            str(key): json_ready(
                item
            )
            for key, item in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            json_ready(
                item
            )
            for item in value
        ]

    if isinstance(
        value,
        np.ndarray,
    ):
        return json_ready(
            value.tolist()
        )

    if isinstance(
        value,
        pd.Series,
    ):
        return json_ready(
            value.tolist()
        )

    if isinstance(
        value,
        pd.DataFrame,
    ):
        return json_ready(
            value.to_dict(
                orient="records"
            )
        )

    return str(value)


def canonical_json_bytes(
    payload: Mapping[str, Any],
) -> bytes:
    """Return canonical UTF-8 JSON bytes for hashing."""
    return json.dumps(
        json_ready(
            payload
        ),
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        ensure_ascii=False,
        allow_nan=False,
    ).encode(
        "utf-8"
    )


def canonical_payload_sha256(
    payload: Mapping[str, Any],
) -> str:
    """Return the SHA-256 of a canonical JSON payload."""
    return hashlib.sha256(
        canonical_json_bytes(
            payload
        )
    ).hexdigest()


def atomic_replace_bytes(
    destination_path: Path,
    content: bytes,
) -> None:
    """Write bytes atomically within the destination directory."""
    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with NamedTemporaryFile(
        mode="wb",
        dir=destination_path.parent,
        prefix=f".{destination_path.name}.",
        suffix=".tmp",
        delete=False,
    ) as temporary_file:
        temporary_path = Path(
            temporary_file.name
        )

        temporary_file.write(
            content
        )

        temporary_file.flush()
        os.fsync(
            temporary_file.fileno()
        )

    try:
        os.replace(
            temporary_path,
            destination_path,
        )
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


def write_self_hashed_json(
    destination_path: Path,
    payload: Mapping[str, Any],
) -> tuple[
    str,
    str,
]:
    """
    Write canonical JSON containing a payload_sha256 field.

    Returns:
        payload_sha256, raw_file_sha256
    """
    payload_without_hash = {
        str(key): json_ready(
            value
        )
        for key, value in payload.items()
        if key != "payload_sha256"
    }

    payload_sha256 = canonical_payload_sha256(
        payload_without_hash
    )

    document = {
        **payload_without_hash,
        "payload_sha256": payload_sha256,
    }

    serialized = (
        json.dumps(
            document,
            sort_keys=True,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode(
        "utf-8"
    )

    atomic_replace_bytes(
        destination_path,
        serialized,
    )

    return (
        payload_sha256,
        sha256_file(
            destination_path
        ),
    )


def verify_self_hashed_json_document(
    source_path: Path,
) -> dict[str, Any]:
    """Read and verify a self-hashed JSON document."""
    document = read_json_object(
        source_path
    )

    require(
        "payload_sha256"
        in document,
        (
            "Self-hashed JSON is missing payload_sha256: "
            f"{source_path}"
        ),
    )

    registered_payload_sha256 = str(
        document[
            "payload_sha256"
        ]
    )

    payload_without_hash = {
        str(key): value
        for key, value in document.items()
        if key != "payload_sha256"
    }

    recomputed_payload_sha256 = (
        canonical_payload_sha256(
            payload_without_hash
        )
    )

    require(
        recomputed_payload_sha256
        == registered_payload_sha256,
        (
            "Self-hashed JSON payload verification failed: "
            f"{source_path}"
        ),
    )

    return document


def dataframe_schema_sha256(
    frame: pd.DataFrame,
) -> str:
    """Hash ordered column names and pandas dtypes."""
    schema_payload = {
        "columns": [
            {
                "position": int(position),
                "name": str(column_name),
                "dtype": str(
                    frame[
                        column_name
                    ].dtype
                ),
            }
            for position, column_name in enumerate(
                frame.columns
            )
        ]
    }

    return canonical_payload_sha256(
        schema_payload
    )


def normalized_artifact_name(
    variable_name: str,
) -> str:
    """Convert an uppercase runtime variable name to a file stem."""
    return (
        variable_name
        .strip()
        .lower()
        .replace(
            "__",
            "_",
        )
    )


def write_dataframe_csv_gzip(
    frame: pd.DataFrame,
    destination_path: Path,
) -> None:
    """Write a DataFrame as deterministic gzip-compressed CSV."""
    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with NamedTemporaryFile(
        mode="wb",
        dir=destination_path.parent,
        prefix=f".{destination_path.name}.",
        suffix=".tmp",
        delete=False,
    ) as temporary_file:
        temporary_path = Path(
            temporary_file.name
        )

    try:
        frame.to_csv(
            temporary_path,
            index=False,
            encoding="utf-8",
            lineterminator="\n",
            float_format="%.17g",
            na_rep="",
            compression={
                "method": "gzip",
                "compresslevel": 9,
                "mtime": 0,
            },
        )

        os.replace(
            temporary_path,
            destination_path,
        )
    finally:
        if temporary_path.exists():
            temporary_path.unlink()


# ------------------------------------------------------------
# Discover authoritative Notebook 06 tables
# ------------------------------------------------------------

required_terminal_table_names: Final[
    tuple[str, ...]
] = (
    "V0_1_RUN_ID_RESOLUTION_AUDIT",
    "NOTEBOOK_06_TERMINAL_EVIDENCE_LEDGER",
    "NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT",
    "NOTEBOOK_06_TERMINAL_DECISION",
    "NOTEBOOK_06_DECISION_GATE_FRAME",
)

missing_terminal_table_names = [
    variable_name
    for variable_name
    in required_terminal_table_names
    if variable_name
    not in globals()
]

require(
    not missing_terminal_table_names,
    (
        "Required terminal tables are absent from the runtime: "
        f"{missing_terminal_table_names}"
    ),
)


table_name_tokens: Final[
    tuple[str, ...]
] = (
    "AUDIT",
    "SUMMARY",
    "LEDGER",
    "SCORE",
    "MODEL",
    "DIAGNOSTIC",
    "RECONCILIATION",
    "COVERAGE",
    "CONTRACT",
    "DECISION",
    "GATE",
    "PROFILE",
    "DISTRIBUTION",
    "PARAMETER",
    "CANDIDATE",
    "REGIME",
    "BURST",
    "EPISODE",
    "RESPONSE",
    "CROSS",
    "DISPERSION",
    "RESIDUAL",
    "PIT",
    "FANO",
    "ACF",
    "MARK",
    "RENEWAL",
    "POISSON",
    "COMPARISON",
    "STABILITY",
    "DEPENDENCE",
    "DURATION",
    "ARRIVAL",
    "CENSOR",
    "SELECTION",
    "APPLICABILITY",
    "DISCREPANC",
    "EVIDENCE",
    "ACCEPTANCE",
)

excluded_dataframe_names: Final[
    set[str]
] = {
    "PRIMARY_ESTIMATION_EVENTS_ANALYTICAL",
    "PRIMARY_EXACT_TIME_BATCHES_ANALYTICAL",
    "PRIMARY_SCORING_BATCHES_ANALYTICAL",
    "PRIMARY_EVENT_TO_BATCH_MEMBERSHIP_ANALYTICAL",
    "PRIMARY_BATCH_FEATURE_IDENTITIES_ANALYTICAL",
    "PRIMARY_BATCH_FEATURE_IDENTITIES_CANONICAL",
    "PRIMARY_BATCH_FEATURE_IDENTITY_LINK",
    "NOTEBOOK_04_CANONICAL_BATCH_IDENTITY",
    "NOTEBOOK_05_PERSISTED_FEATURE_IDENTITY",
    "ANALYTICAL_BATCH_CALENDAR",
    "OBSERVATION_WINDOW_CONTRACT",
}


def dataframe_is_authoritative_output(
    variable_name: str,
    value: Any,
) -> bool:
    """Select Notebook 06 result tables while excluding upstream inputs."""
    if not isinstance(
        value,
        pd.DataFrame,
    ):
        return False

    if variable_name in excluded_dataframe_names:
        return False

    if variable_name.startswith(
        "_"
    ):
        return False

    if not variable_name.isupper():
        return False

    if (
        variable_name.startswith(
            "PRIMARY_"
        )
        and variable_name
        not in required_terminal_table_names
    ):
        return False

    if (
        "ANALYTICAL"
        in variable_name
        and not any(
            token
            in variable_name
            for token in (
                "AUDIT",
                "SUMMARY",
                "RECONCILIATION",
                "EVIDENCE",
                "GATE",
                "COVERAGE",
            )
        )
    ):
        return False

    if variable_name in required_terminal_table_names:
        return True

    return any(
        token
        in variable_name
        for token in table_name_tokens
    )


AUTHORITATIVE_NOTEBOOK_06_TABLES: dict[
    str,
    pd.DataFrame,
] = {
    variable_name: value.copy()
    for variable_name, value in globals().items()
    if dataframe_is_authoritative_output(
        variable_name,
        value,
    )
}

require(
    set(
        required_terminal_table_names
    ).issubset(
        AUTHORITATIVE_NOTEBOOK_06_TABLES
    ),
    (
        "The persistence registry omitted at least one required "
        "terminal table."
    ),
)

require(
    len(
        AUTHORITATIVE_NOTEBOOK_06_TABLES
    )
    >= len(
        required_terminal_table_names
    ),
    "No authoritative Notebook 06 result tables were discovered.",
)


# ------------------------------------------------------------
# Persist result tables
# ------------------------------------------------------------

table_artifact_records: list[
    dict[str, Any]
] = []

for (
    variable_name,
    result_frame,
) in sorted(
    AUTHORITATIVE_NOTEBOOK_06_TABLES.items()
):
    require(
        result_frame.columns.is_unique,
        (
            "Authoritative result table contains duplicate "
            f"columns: {variable_name}"
        ),
    )

    require(
        all(
            str(
                column_name
            ).strip()
            for column_name in result_frame.columns
        ),
        (
            "Authoritative result table contains an empty "
            f"column name: {variable_name}"
        ),
    )

    destination_path = (
        NOTEBOOK_06_TABLE_DIR
        / (
            normalized_artifact_name(
                variable_name
            )
            + ".csv.gz"
        )
    )

    write_dataframe_csv_gzip(
        result_frame,
        destination_path,
    )

    table_artifact_records.append(
        {
            "artifact_type": (
                "NOTEBOOK_06_RESULT_TABLE"
            ),
            "variable_name": (
                variable_name
            ),
            "path": (
                str(
                    destination_path
                )
            ),
            "format": (
                "CSV_GZIP"
            ),
            "compression": (
                "GZIP_MTIME_ZERO"
            ),
            "row_count": int(
                len(
                    result_frame
                )
            ),
            "column_count": int(
                result_frame.shape[1]
            ),
            "columns": [
                str(
                    column_name
                )
                for column_name
                in result_frame.columns
            ],
            "schema_sha256": (
                dataframe_schema_sha256(
                    result_frame
                )
            ),
            "sha256": (
                sha256_file(
                    destination_path
                )
            ),
            "byte_count": int(
                destination_path.stat().st_size
            ),
            "source_run_prefix": (
                SOURCE_RUN_PREFIX
            ),
            "v0_1_run_id": (
                V0_1_RUN_ID
            ),
            "producing_notebook": (
                NOTEBOOK_NAME
            ),
            "status": "PASS",
        }
    )


NOTEBOOK_06_TABLE_ARTIFACT_REGISTRY = (
    pd.DataFrame(
        table_artifact_records
    )
    .sort_values(
        [
            "variable_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Terminal decision control artifact
# ------------------------------------------------------------

NOTEBOOK_06_TERMINAL_DECISION_PATH: Final[
    Path
] = (
    NOTEBOOK_06_MANIFEST_DIR
    / "06_point_process_baselines_terminal_decision.json"
)

terminal_decision_payload = {
    "artifact_type": (
        "NOTEBOOK_06_TERMINAL_DECISION"
    ),
    "schema_version": (
        "NOTEBOOK_06_TERMINAL_DECISION_V1"
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        V0_1_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "notebook_config_sha256": (
        NOTEBOOK_CONFIG_SHA256
    ),
    "producing_notebook": (
        NOTEBOOK_NAME
    ),
    "notebook_stage": (
        NOTEBOOK_STAGE
    ),
    "authorization_state": (
        NOTEBOOK_06_AUTHORIZATION_STATE
    ),
    "terminal_status": (
        NOTEBOOK_06_TERMINAL_STATUS
    ),
    "formal_count_comparator": (
        DECISION_FORMAL_COMPARATOR_MODEL_ID
    ),
    "primary_simple_count_baseline": (
        DECISION_PRIMARY_SIMPLE_MODEL_ID
    ),
    "primary_hawkes_scope": (
        PRIMARY_HAWKES_SCOPE
    ),
    "notebook_07_authorized": bool(
        NOTEBOOK_07_HAWKES_ESTIMATION_AUTHORIZED
    ),
    "restricted_first_required": bool(
        NOTEBOOK_07_RESTRICTED_FIRST_REQUIRED
    ),
    "scientific_decision": {
        "simple_baseline_materially_improves": bool(
            SIMPLE_BASELINE_MATERIALLY_IMPROVES
        ),
        "simple_baseline_sufficient": bool(
            SIMPLE_BASELINE_SUFFICIENT
        ),
        "remaining_self_dependence_evidence": bool(
            REMAINING_SELF_DEPENDENCE_EVIDENCE
        ),
        "directional_residual_cross_evidence_strong": bool(
            DIRECTIONAL_RESIDUAL_CROSS_EVIDENCE_STRONG
        ),
        "conditional_cross_response_evidence_present": bool(
            CONDITIONAL_CROSS_RESPONSE_EVIDENCE_PRESENT
        ),
    },
    "claim_limits": {
        "state_dependent_hawkes_authorized": False,
        "hawkes_superiority_claim_authorized": False,
        "strategy_or_quoting_claim_authorized": False,
        "fill_or_execution_claim_authorized": False,
        "pnl_or_performance_claim_authorized": False,
    },
    "protected_partition_content_loaded": {
        "VALIDATION": bool(
            PROTECTED_PARTITION_CONTENT_LOADED[
                "VALIDATION"
            ]
        ),
        "ENGINEERING_HOLDOUT": bool(
            PROTECTED_PARTITION_CONTENT_LOADED[
                "ENGINEERING_HOLDOUT"
            ]
        ),
    },
    "decision_rows": json_ready(
        NOTEBOOK_06_TERMINAL_DECISION
    ),
    "status": "PASS",
}

(
    NOTEBOOK_06_TERMINAL_DECISION_PAYLOAD_SHA256,
    NOTEBOOK_06_TERMINAL_DECISION_FILE_SHA256,
) = write_self_hashed_json(
    NOTEBOOK_06_TERMINAL_DECISION_PATH,
    terminal_decision_payload,
)


# ------------------------------------------------------------
# Final acceptance control artifact
# ------------------------------------------------------------

NOTEBOOK_06_FINAL_ACCEPTANCE_PATH: Final[
    Path
] = (
    NOTEBOOK_06_MANIFEST_DIR
    / "06_point_process_baselines_final_acceptance.json"
)

blocking_decision_failures = int(
    (
        NOTEBOOK_06_DECISION_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        )
        & ~NOTEBOOK_06_DECISION_GATE_FRAME[
            "passed"
        ].astype(
            bool
        )
    ).sum()
)

require(
    blocking_decision_failures
    == 0,
    (
        "Notebook 06 cannot persist final acceptance while "
        "blocking decision failures remain."
    ),
)

notebook_05_timestamp_status = str(
    globals().get(
        "NOTEBOOK_05_TIMESTAMP_IDENTITY_STATUS",
        "NOT_REGISTERED",
    )
)

final_acceptance_payload = {
    "artifact_type": (
        "NOTEBOOK_06_FINAL_ACCEPTANCE"
    ),
    "schema_version": (
        "NOTEBOOK_06_FINAL_ACCEPTANCE_V1"
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        V0_1_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "producing_notebook": (
        NOTEBOOK_NAME
    ),
    "terminal_status": (
        NOTEBOOK_06_TERMINAL_STATUS
    ),
    "authorization_state": (
        NOTEBOOK_06_AUTHORIZATION_STATE
    ),
    "blocking_failure_count": (
        blocking_decision_failures
    ),
    "authoritative_result_table_count": int(
        len(
            NOTEBOOK_06_TABLE_ARTIFACT_REGISTRY
        )
    ),
    "next_authorized_notebook": (
        "07_HAWKES_ESTIMATION.ipynb"
        if NOTEBOOK_07_HAWKES_ESTIMATION_AUTHORIZED
        else None
    ),
    "notebook_07_authorized": bool(
        NOTEBOOK_07_HAWKES_ESTIMATION_AUTHORIZED
    ),
    "restricted_first_required": bool(
        NOTEBOOK_07_RESTRICTED_FIRST_REQUIRED
    ),
    "authorized_primary_scope": (
        PRIMARY_HAWKES_SCOPE
    ),
    "warnings": [
        {
            "warning_id": (
                "NOTEBOOK_05_TIMESTAMP_PERSISTENCE"
            ),
            "status": (
                notebook_05_timestamp_status
            ),
            "interpretation": (
                "Notebook 04 remains the exact timestamp "
                "authority; the Notebook 05 persisted timestamp "
                "roundtrip warning is not propagated as timing "
                "authority."
            ),
        },
        {
            "warning_id": (
                "RESTRICTED_FIRST_HAWKES_SCOPE"
            ),
            "status": (
                "ACTIVE"
                if NOTEBOOK_07_RESTRICTED_FIRST_REQUIRED
                else "NOT_ACTIVE"
            ),
            "interpretation": (
                "Immediate unrestricted cross-excitation "
                "interpretation is not authorized because repeated "
                "directional residual cross-dependence was not "
                "established."
            ),
        },
        {
            "warning_id": (
                "STATE_DEPENDENT_HAWKES"
            ),
            "status": "NOT_AUTHORIZED",
            "interpretation": (
                "State-dependent Hawkes remains outside the "
                "Notebook 07 estimation contract."
            ),
        },
        {
            "warning_id": (
                "HAWKES_SUPERIORITY_CLAIM"
            ),
            "status": "NOT_AUTHORIZED",
            "interpretation": (
                "Notebook 07 estimation cannot by itself establish "
                "superiority over the primary simple baseline."
            ),
        },
        {
            "warning_id": (
                "STRATEGY_AND_MARKET_MAKING_USE"
            ),
            "status": "NOT_AUTHORIZED",
            "interpretation": (
                "No quoting, execution, fill, P&L, Sharpe, "
                "drawdown, or deployment claim is authorized."
            ),
        },
    ],
    "protected_partition_content_loaded": {
        "VALIDATION": False,
        "ENGINEERING_HOLDOUT": False,
    },
    "status": (
        NOTEBOOK_06_TERMINAL_STATUS
    ),
}

(
    NOTEBOOK_06_FINAL_ACCEPTANCE_PAYLOAD_SHA256,
    NOTEBOOK_06_FINAL_ACCEPTANCE_FILE_SHA256,
) = write_self_hashed_json(
    NOTEBOOK_06_FINAL_ACCEPTANCE_PATH,
    final_acceptance_payload,
)


# ------------------------------------------------------------
# Output manifest
# ------------------------------------------------------------

NOTEBOOK_06_OUTPUT_MANIFEST_PATH: Final[
    Path
] = (
    NOTEBOOK_06_MANIFEST_DIR
    / "06_point_process_baselines_output_manifest.json"
)

substantive_artifact_records = [
    json_ready(
        record
    )
    for record in table_artifact_records
]

substantive_artifact_records.extend(
    [
        {
            "artifact_type": (
                "NOTEBOOK_06_TERMINAL_DECISION"
            ),
            "path": str(
                NOTEBOOK_06_TERMINAL_DECISION_PATH
            ),
            "payload_sha256": (
                NOTEBOOK_06_TERMINAL_DECISION_PAYLOAD_SHA256
            ),
            "sha256": (
                NOTEBOOK_06_TERMINAL_DECISION_FILE_SHA256
            ),
            "byte_count": int(
                NOTEBOOK_06_TERMINAL_DECISION_PATH.stat().st_size
            ),
            "status": "PASS",
        },
        {
            "artifact_type": (
                "NOTEBOOK_06_FINAL_ACCEPTANCE"
            ),
            "path": str(
                NOTEBOOK_06_FINAL_ACCEPTANCE_PATH
            ),
            "payload_sha256": (
                NOTEBOOK_06_FINAL_ACCEPTANCE_PAYLOAD_SHA256
            ),
            "sha256": (
                NOTEBOOK_06_FINAL_ACCEPTANCE_FILE_SHA256
            ),
            "byte_count": int(
                NOTEBOOK_06_FINAL_ACCEPTANCE_PATH.stat().st_size
            ),
            "status": "PASS",
        },
    ]
)

output_manifest_payload = {
    "artifact_type": (
        "NOTEBOOK_06_OUTPUT_MANIFEST"
    ),
    "schema_version": (
        "NOTEBOOK_06_OUTPUT_MANIFEST_V1"
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        V0_1_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "notebook_config_sha256": (
        NOTEBOOK_CONFIG_SHA256
    ),
    "producing_notebook": (
        NOTEBOOK_NAME
    ),
    "notebook_stage": (
        NOTEBOOK_STAGE
    ),
    "terminal_status": (
        NOTEBOOK_06_TERMINAL_STATUS
    ),
    "authorization_state": (
        NOTEBOOK_06_AUTHORIZATION_STATE
    ),
    "artifact_count": int(
        len(
            substantive_artifact_records
        )
    ),
    "artifacts": (
        substantive_artifact_records
    ),
    "upstream_control_artifacts": (
        json_ready(
            UPSTREAM_CONTROL_ARTIFACT_AUDIT
        )
        if (
            "UPSTREAM_CONTROL_ARTIFACT_AUDIT"
            in globals()
        )
        else []
    ),
    "upstream_data_artifacts": (
        json_ready(
            UPSTREAM_DATA_ARTIFACT_AUDIT
        )
        if (
            "UPSTREAM_DATA_ARTIFACT_AUDIT"
            in globals()
        )
        else []
    ),
    "protected_partition_content_loaded": {
        "VALIDATION": False,
        "ENGINEERING_HOLDOUT": False,
    },
    "status": "PASS",
}

(
    NOTEBOOK_06_OUTPUT_MANIFEST_PAYLOAD_SHA256,
    NOTEBOOK_06_OUTPUT_MANIFEST_FILE_SHA256,
) = write_self_hashed_json(
    NOTEBOOK_06_OUTPUT_MANIFEST_PATH,
    output_manifest_payload,
)


# ------------------------------------------------------------
# Readback verification
# ------------------------------------------------------------

readback_rows: list[
    dict[str, Any]
] = []

for artifact_record in table_artifact_records:
    artifact_path = Path(
        artifact_record[
            "path"
        ]
    )

    actual_file_sha256 = sha256_file(
        artifact_path
    )

    registered_file_sha256 = str(
        artifact_record[
            "sha256"
        ]
    )

    readback_frame = pd.read_csv(
        artifact_path,
        compression="gzip",
        low_memory=False,
    )

    expected_row_count = int(
        artifact_record[
            "row_count"
        ]
    )

    expected_column_count = int(
        artifact_record[
            "column_count"
        ]
    )

    hash_matches = (
        actual_file_sha256
        == registered_file_sha256
    )

    row_count_matches = (
        len(
            readback_frame
        )
        == expected_row_count
    )

    column_count_matches = (
        readback_frame.shape[1]
        == expected_column_count
    )

    column_order_matches = (
        [
            str(
                column_name
            )
            for column_name
            in readback_frame.columns
        ]
        == [
            str(
                column_name
            )
            for column_name
            in artifact_record[
                "columns"
            ]
        ]
    )

    artifact_passed = bool(
        hash_matches
        and row_count_matches
        and column_count_matches
        and column_order_matches
    )

    readback_rows.append(
        {
            "artifact_type": (
                artifact_record[
                    "artifact_type"
                ]
            ),
            "artifact_name": (
                artifact_record[
                    "variable_name"
                ]
            ),
            "path": str(
                artifact_path
            ),
            "expected_row_count": (
                expected_row_count
            ),
            "readback_row_count": int(
                len(
                    readback_frame
                )
            ),
            "expected_column_count": (
                expected_column_count
            ),
            "readback_column_count": int(
                readback_frame.shape[1]
            ),
            "hash_matches": (
                hash_matches
            ),
            "row_count_matches": (
                row_count_matches
            ),
            "column_count_matches": (
                column_count_matches
            ),
            "column_order_matches": (
                column_order_matches
            ),
            "path_inside_v0_1": (
                path_is_inside(
                    artifact_path,
                    V0_1_PERSISTENCE_ROOT,
                )
            ),
            "path_inside_v0_0": (
                path_is_inside(
                    artifact_path,
                    V0_0_IMMUTABLE_ROOT,
                )
            ),
            "status": (
                "PASS"
                if artifact_passed
                else "FAIL"
            ),
        }
    )


json_control_paths = (
    NOTEBOOK_06_TERMINAL_DECISION_PATH,
    NOTEBOOK_06_FINAL_ACCEPTANCE_PATH,
    NOTEBOOK_06_OUTPUT_MANIFEST_PATH,
)

for json_control_path in json_control_paths:
    verified_document = (
        verify_self_hashed_json_document(
            json_control_path
        )
    )

    readback_rows.append(
        {
            "artifact_type": str(
                verified_document[
                    "artifact_type"
                ]
            ),
            "artifact_name": (
                json_control_path.stem
            ),
            "path": str(
                json_control_path
            ),
            "expected_row_count": pd.NA,
            "readback_row_count": pd.NA,
            "expected_column_count": pd.NA,
            "readback_column_count": pd.NA,
            "hash_matches": True,
            "row_count_matches": True,
            "column_count_matches": True,
            "column_order_matches": True,
            "path_inside_v0_1": (
                path_is_inside(
                    json_control_path,
                    V0_1_PERSISTENCE_ROOT,
                )
            ),
            "path_inside_v0_0": (
                path_is_inside(
                    json_control_path,
                    V0_0_IMMUTABLE_ROOT,
                )
            ),
            "status": "PASS",
        }
    )


NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT = (
    pd.DataFrame(
        readback_rows
    )
    .sort_values(
        [
            "artifact_type",
            "artifact_name",
        ],
        kind="stable",
    )
    .reset_index(
        drop=True
    )
)

require(
    NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT[
        "status"
    ].eq(
        "PASS"
    ).all(),
    (
        "At least one persisted Notebook 06 artifact failed "
        "readback verification."
    ),
)

require(
    NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT[
        "path_inside_v0_1"
    ].astype(
        bool
    ).all(),
    (
        "At least one persisted Notebook 06 artifact is outside "
        "the V0.1 root."
    ),
)

require(
    not NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT[
        "path_inside_v0_0"
    ].astype(
        bool
    ).any(),
    (
        "At least one persisted Notebook 06 artifact was written "
        "inside the immutable V0.0 tree."
    ),
)


# ------------------------------------------------------------
# Verify immutable V0.0 reference remains unchanged
# ------------------------------------------------------------

v0_0_reference_hash_stable = True
v0_0_reference_path_text: str | None = None
v0_0_reference_hash_before: str | None = None
v0_0_reference_hash_after: str | None = None

if (
    "V0_0_REFERENCE_NOTEBOOK_PATH"
    in globals()
):
    v0_0_reference_path = Path(
        V0_0_REFERENCE_NOTEBOOK_PATH
    )

    v0_0_reference_path_text = str(
        v0_0_reference_path
    )

    v0_0_reference_hash_after = (
        sha256_file(
            v0_0_reference_path
        )
    )

    if (
        "V0_0_REFERENCE_NOTEBOOK_SHA256"
        in globals()
    ):
        v0_0_reference_hash_before = str(
            V0_0_REFERENCE_NOTEBOOK_SHA256
        )

    elif (
        "V0_0_REFERENCE_NOTEBOOK_SHA256_BEFORE"
        in globals()
    ):
        v0_0_reference_hash_before = str(
            V0_0_REFERENCE_NOTEBOOK_SHA256_BEFORE
        )

    if (
        v0_0_reference_hash_before
        is not None
    ):
        v0_0_reference_hash_stable = (
            v0_0_reference_hash_after
            == v0_0_reference_hash_before
        )

require(
    v0_0_reference_hash_stable,
    (
        "The immutable V0.0 reference notebook changed during "
        "Notebook 06 persistence."
    ),
)


# ------------------------------------------------------------
# Persist readback audit
# ------------------------------------------------------------

NOTEBOOK_06_READBACK_AUDIT_PATH: Final[
    Path
] = (
    NOTEBOOK_06_MANIFEST_DIR
    / "06_point_process_baselines_readback_audit.json"
)

readback_audit_payload = {
    "artifact_type": (
        "NOTEBOOK_06_READBACK_AUDIT"
    ),
    "schema_version": (
        "NOTEBOOK_06_READBACK_AUDIT_V1"
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "v0_1_run_id": (
        V0_1_RUN_ID
    ),
    "producing_notebook": (
        NOTEBOOK_NAME
    ),
    "verified_artifact_count": int(
        len(
            NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT
        )
    ),
    "failed_artifact_count": int(
        NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT[
            "status"
        ].ne(
            "PASS"
        ).sum()
    ),
    "all_paths_inside_v0_1": bool(
        NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT[
            "path_inside_v0_1"
        ].astype(
            bool
        ).all()
    ),
    "any_path_inside_v0_0": bool(
        NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT[
            "path_inside_v0_0"
        ].astype(
            bool
        ).any()
    ),
    "v0_0_reference_path": (
        v0_0_reference_path_text
    ),
    "v0_0_reference_sha256_before": (
        v0_0_reference_hash_before
    ),
    "v0_0_reference_sha256_after": (
        v0_0_reference_hash_after
    ),
    "v0_0_reference_hash_stable": (
        v0_0_reference_hash_stable
    ),
    "readback_rows": json_ready(
        NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT
    ),
    "status": "PASS",
}

(
    NOTEBOOK_06_READBACK_AUDIT_PAYLOAD_SHA256,
    NOTEBOOK_06_READBACK_AUDIT_FILE_SHA256,
) = write_self_hashed_json(
    NOTEBOOK_06_READBACK_AUDIT_PATH,
    readback_audit_payload,
)

verify_self_hashed_json_document(
    NOTEBOOK_06_READBACK_AUDIT_PATH
)


# ------------------------------------------------------------
# Notebook 06 to Notebook 07 handoff
# ------------------------------------------------------------

NOTEBOOK_06_TO_07_HANDOFF_PATH: Final[
    Path
] = (
    NOTEBOOK_06_HANDOFF_DIR
    / (
        "06_point_process_baselines_to_"
        "07_hawkes_estimation_handoff.json"
    )
)

upstream_data_records = (
    json_ready(
        UPSTREAM_DATA_ARTIFACT_AUDIT
    )
    if (
        "UPSTREAM_DATA_ARTIFACT_AUDIT"
        in globals()
    )
    else []
)

handoff_payload = {
    "artifact_type": (
        "NOTEBOOK_06_TO_NOTEBOOK_07_HANDOFF"
    ),
    "schema_version": (
        "NOTEBOOK_06_TO_NOTEBOOK_07_HANDOFF_V1"
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "producer": (
        "06_POINT_PROCESS_BASELINES.ipynb"
    ),
    "consumer": (
        "07_HAWKES_ESTIMATION.ipynb"
    ),
    "source_run_prefix": (
        SOURCE_RUN_PREFIX
    ),
    "source_set_sha256": (
        SOURCE_SET_SHA256
    ),
    "v0_1_run_id": (
        V0_1_RUN_ID
    ),
    "run_config_sha256": (
        RUN_CONFIG_SHA256
    ),
    "run_identity_sha256": (
        RUN_IDENTITY_SHA256
    ),
    "notebook_config_sha256": (
        NOTEBOOK_CONFIG_SHA256
    ),
    "authorization": {
        "authorization_state": (
            NOTEBOOK_06_AUTHORIZATION_STATE
        ),
        "terminal_status": (
            NOTEBOOK_06_TERMINAL_STATUS
        ),
        "notebook_07_authorized": bool(
            NOTEBOOK_07_HAWKES_ESTIMATION_AUTHORIZED
        ),
        "restricted_first_required": bool(
            NOTEBOOK_07_RESTRICTED_FIRST_REQUIRED
        ),
        "primary_hawkes_scope": (
            PRIMARY_HAWKES_SCOPE
        ),
        "state_dependent_hawkes_authorized": False,
        "hawkes_superiority_claim_authorized": False,
        "strategy_or_quoting_use_authorized": False,
    },
    "baseline_context": {
        "formal_count_comparator": (
            DECISION_FORMAL_COMPARATOR_MODEL_ID
        ),
        "primary_simple_count_baseline": (
            DECISION_PRIMARY_SIMPLE_MODEL_ID
        ),
        "primary_simple_model_family": (
            "CAUSAL_EXPONENTIALLY_WEIGHTED_RATE"
        ),
        "primary_simple_half_life_ms": 250,
        "simple_baseline_materially_improves": bool(
            SIMPLE_BASELINE_MATERIALLY_IMPROVES
        ),
        "simple_baseline_sufficient": bool(
            SIMPLE_BASELINE_SUFFICIENT
        ),
        "remaining_self_dependence_evidence": bool(
            REMAINING_SELF_DEPENDENCE_EVIDENCE
        ),
        "directional_residual_cross_evidence_strong": bool(
            DIRECTIONAL_RESIDUAL_CROSS_EVIDENCE_STRONG
        ),
        "conditional_cross_response_evidence_present": bool(
            CONDITIONAL_CROSS_RESPONSE_EVIDENCE_PRESENT
        ),
    },
    "authorized_estimation_contract": {
        "first_model": (
            "DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES"
        ),
        "authorized_event_components": [
            "BUY",
            "SELL",
        ],
        "authorized_excitation_channels_first": [
            "BUY_TO_BUY",
            "SELL_TO_SELL",
        ],
        "cross_excitation_channels_first": (
            "FIXED_TO_ZERO"
        ),
        "full_bivariate_extension": (
            "NESTED_DIAGNOSTIC_ONLY_UNLESS_"
            "DEVELOPMENT_SELECTION_CONTRACT_AUTHORIZES"
        ),
        "kernel_family_first": (
            "SINGLE_EXPONENTIAL_PER_AUTHORIZED_CHANNEL"
        ),
        "parameter_fit_partition": (
            "DEVELOPMENT_ONLY"
        ),
        "hyperparameter_selection_partition": (
            "DEVELOPMENT_ONLY"
        ),
        "locked_evaluation_partition": (
            "CALIBRATION"
        ),
        "calibration_parameter_updates": 0,
        "development_history_carried_into_calibration": True,
        "development_left_prehistory_fabricated": False,
        "left_censoring_must_be_recorded": True,
        "base_intensity_constraint": (
            "STRICTLY_POSITIVE"
        ),
        "kernel_amplitude_constraint": (
            "NONNEGATIVE"
        ),
        "kernel_decay_constraint": (
            "STRICTLY_POSITIVE"
        ),
        "stationarity_constraint": (
            "EXCITATION_NORM_SPECTRAL_RADIUS_"
            "STRICTLY_LESS_THAN_ONE"
        ),
    },
    "event_time_contract": {
        "event_representation": (
            PRIMARY_EVENT_REPRESENTATION
        ),
        "timestamp_column": (
            PRIMARY_EVENT_TIME_COLUMN
            if (
                "PRIMARY_EVENT_TIME_COLUMN"
                in globals()
            )
            else "event_time_ns"
        ),
        "partition_column": (
            PRIMARY_EVENT_PARTITION_COLUMN
            if (
                "PRIMARY_EVENT_PARTITION_COLUMN"
                in globals()
            )
            else "event_partition"
        ),
        "timestamp_authority": (
            "NOTEBOOK_04_EVENT_TIME_NS"
        ),
        "simultaneous_batch_interface": (
            "SIMULTANEOUS_EVENT_BATCH_REQUIRED"
        ),
        "scoring_history": (
            "STRICTLY_BEFORE_BATCH_TIME"
        ),
        "within_batch_zero_lag_excitation": False,
        "batch_update_timing": (
            "AFTER_ALL_EVENTS_IN_BATCH_ARE_SCORED"
        ),
        "collector_sequence_role": (
            "TRACE_ORDER_ONLY_NOT_PHYSICAL_TIME"
        ),
        "timestamp_jitter_allowed": False,
        "event_reordering_allowed": False,
        "event_removal_allowed": False,
    },
    "partition_contract": {
        "fit_partition": (
            "DEVELOPMENT"
        ),
        "locked_evaluation_partition": (
            "CALIBRATION"
        ),
        "validation_content_access": (
            "FORBIDDEN"
        ),
        "engineering_holdout_content_access": (
            "FORBIDDEN"
        ),
        "validation_content_loaded": False,
        "engineering_holdout_content_loaded": False,
    },
    "expected_analytical_counts": {
        "development_event_count": 7004,
        "development_batch_count": 6859,
        "calibration_event_count": 2493,
        "calibration_batch_count": 2400,
        "analytical_event_count": 9497,
        "analytical_batch_count": 9259,
    },
    "authoritative_upstream_data_artifacts": (
        upstream_data_records
    ),
    "control_artifacts": {
        "terminal_decision": {
            "path": str(
                NOTEBOOK_06_TERMINAL_DECISION_PATH
            ),
            "payload_sha256": (
                NOTEBOOK_06_TERMINAL_DECISION_PAYLOAD_SHA256
            ),
            "sha256": (
                NOTEBOOK_06_TERMINAL_DECISION_FILE_SHA256
            ),
        },
        "final_acceptance": {
            "path": str(
                NOTEBOOK_06_FINAL_ACCEPTANCE_PATH
            ),
            "payload_sha256": (
                NOTEBOOK_06_FINAL_ACCEPTANCE_PAYLOAD_SHA256
            ),
            "sha256": (
                NOTEBOOK_06_FINAL_ACCEPTANCE_FILE_SHA256
            ),
        },
        "output_manifest": {
            "path": str(
                NOTEBOOK_06_OUTPUT_MANIFEST_PATH
            ),
            "payload_sha256": (
                NOTEBOOK_06_OUTPUT_MANIFEST_PAYLOAD_SHA256
            ),
            "sha256": (
                NOTEBOOK_06_OUTPUT_MANIFEST_FILE_SHA256
            ),
        },
        "readback_audit": {
            "path": str(
                NOTEBOOK_06_READBACK_AUDIT_PATH
            ),
            "payload_sha256": (
                NOTEBOOK_06_READBACK_AUDIT_PAYLOAD_SHA256
            ),
            "sha256": (
                NOTEBOOK_06_READBACK_AUDIT_FILE_SHA256
            ),
        },
    },
    "execution_contract_rows": json_ready(
        NOTEBOOK_07_HAWKES_EXECUTION_CONTRACT
    ),
    "claim_limits": [
        "NO_HAWKES_SUPERIORITY_CLAIM",
        "NO_STATE_DEPENDENT_HAWKES",
        "NO_QUOTING_POLICY",
        "NO_FILL_SIMULATION",
        "NO_EXECUTION_SIMULATION",
        "NO_PNL_OR_SHARPE_CLAIM",
        "NO_LIVE_TRADING_OR_DEPLOYMENT",
    ],
    "status": (
        NOTEBOOK_06_TERMINAL_STATUS
    ),
}

(
    NOTEBOOK_06_TO_07_HANDOFF_PAYLOAD_SHA256,
    NOTEBOOK_06_TO_07_HANDOFF_FILE_SHA256,
) = write_self_hashed_json(
    NOTEBOOK_06_TO_07_HANDOFF_PATH,
    handoff_payload,
)

verified_handoff_document = (
    verify_self_hashed_json_document(
        NOTEBOOK_06_TO_07_HANDOFF_PATH
    )
)

require(
    bool(
        verified_handoff_document[
            "authorization"
        ][
            "notebook_07_authorized"
        ]
    ),
    (
        "The persisted Notebook 06 to Notebook 07 handoff does "
        "not authorize Notebook 07."
    ),
)

require(
    verified_handoff_document[
        "authorization"
    ][
        "authorization_state"
    ]
    == "AUTHORIZED_RESTRICTED_FIRST",
    (
        "The persisted handoff does not preserve the formal "
        "restricted-first authorization state."
    ),
)

require(
    verified_handoff_document[
        "authorized_estimation_contract"
    ][
        "first_model"
    ]
    == "DIAGONAL_BIVARIATE_EXPONENTIAL_HAWKES",
    (
        "The persisted handoff does not preserve the authorized "
        "first Hawkes model."
    ),
)


# ------------------------------------------------------------
# Final persistence gate
# ------------------------------------------------------------

NOTEBOOK_06_FINAL_PERSISTENCE_GATE_FRAME = pd.DataFrame(
    [
        {
            "gate": (
                "authoritative_result_tables_persisted"
            ),
            "severity": "BLOCKING",
            "passed": (
                len(
                    NOTEBOOK_06_TABLE_ARTIFACT_REGISTRY
                )
                == len(
                    AUTHORITATIVE_NOTEBOOK_06_TABLES
                )
            ),
            "evidence": (
                f"table_count="
                f"{len(NOTEBOOK_06_TABLE_ARTIFACT_REGISTRY)}"
            ),
        },
        {
            "gate": (
                "all_result_tables_read_back"
            ),
            "severity": "BLOCKING",
            "passed": (
                NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT[
                    "status"
                ].eq(
                    "PASS"
                ).all()
            ),
            "evidence": (
                f"verified_artifacts="
                f"{len(NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT)}"
            ),
        },
        {
            "gate": (
                "terminal_decision_self_hash_verified"
            ),
            "severity": "BLOCKING",
            "passed": (
                verify_self_hashed_json_document(
                    NOTEBOOK_06_TERMINAL_DECISION_PATH
                )[
                    "payload_sha256"
                ]
                == NOTEBOOK_06_TERMINAL_DECISION_PAYLOAD_SHA256
            ),
            "evidence": (
                NOTEBOOK_06_TERMINAL_DECISION_PAYLOAD_SHA256
            ),
        },
        {
            "gate": (
                "final_acceptance_self_hash_verified"
            ),
            "severity": "BLOCKING",
            "passed": (
                verify_self_hashed_json_document(
                    NOTEBOOK_06_FINAL_ACCEPTANCE_PATH
                )[
                    "payload_sha256"
                ]
                == NOTEBOOK_06_FINAL_ACCEPTANCE_PAYLOAD_SHA256
            ),
            "evidence": (
                NOTEBOOK_06_FINAL_ACCEPTANCE_PAYLOAD_SHA256
            ),
        },
        {
            "gate": (
                "output_manifest_self_hash_verified"
            ),
            "severity": "BLOCKING",
            "passed": (
                verify_self_hashed_json_document(
                    NOTEBOOK_06_OUTPUT_MANIFEST_PATH
                )[
                    "payload_sha256"
                ]
                == NOTEBOOK_06_OUTPUT_MANIFEST_PAYLOAD_SHA256
            ),
            "evidence": (
                NOTEBOOK_06_OUTPUT_MANIFEST_PAYLOAD_SHA256
            ),
        },
        {
            "gate": (
                "readback_audit_self_hash_verified"
            ),
            "severity": "BLOCKING",
            "passed": (
                verify_self_hashed_json_document(
                    NOTEBOOK_06_READBACK_AUDIT_PATH
                )[
                    "payload_sha256"
                ]
                == NOTEBOOK_06_READBACK_AUDIT_PAYLOAD_SHA256
            ),
            "evidence": (
                NOTEBOOK_06_READBACK_AUDIT_PAYLOAD_SHA256
            ),
        },
        {
            "gate": (
                "notebook_07_handoff_self_hash_verified"
            ),
            "severity": "BLOCKING",
            "passed": (
                verified_handoff_document[
                    "payload_sha256"
                ]
                == NOTEBOOK_06_TO_07_HANDOFF_PAYLOAD_SHA256
            ),
            "evidence": (
                NOTEBOOK_06_TO_07_HANDOFF_PAYLOAD_SHA256
            ),
        },
        {
            "gate": (
                "restricted_first_authorization_persisted"
            ),
            "severity": "BLOCKING",
            "passed": (
                verified_handoff_document[
                    "authorization"
                ][
                    "authorization_state"
                ]
                == "AUTHORIZED_RESTRICTED_FIRST"
                and bool(
                    verified_handoff_document[
                        "authorization"
                    ][
                        "restricted_first_required"
                    ]
                )
            ),
            "evidence": (
                "AUTHORIZED_RESTRICTED_FIRST; "
                "restricted_first_required=True"
            ),
        },
        {
            "gate": (
                "protected_partition_content_remains_absent"
            ),
            "severity": "BLOCKING",
            "passed": (
                not PROTECTED_PARTITION_CONTENT_LOADED[
                    "VALIDATION"
                ]
                and not PROTECTED_PARTITION_CONTENT_LOADED[
                    "ENGINEERING_HOLDOUT"
                ]
            ),
            "evidence": (
                "VALIDATION=False; ENGINEERING_HOLDOUT=False"
            ),
        },
        {
            "gate": (
                "all_outputs_inside_v0_1"
            ),
            "severity": "BLOCKING",
            "passed": all(
                path_is_inside(
                    artifact_path,
                    V0_1_PERSISTENCE_ROOT,
                )
                for artifact_path in (
                    NOTEBOOK_06_TERMINAL_DECISION_PATH,
                    NOTEBOOK_06_FINAL_ACCEPTANCE_PATH,
                    NOTEBOOK_06_OUTPUT_MANIFEST_PATH,
                    NOTEBOOK_06_READBACK_AUDIT_PATH,
                    NOTEBOOK_06_TO_07_HANDOFF_PATH,
                )
            ),
            "evidence": str(
                V0_1_PERSISTENCE_ROOT
            ),
        },
        {
            "gate": (
                "no_output_written_inside_v0_0"
            ),
            "severity": "BLOCKING",
            "passed": not any(
                path_is_inside(
                    artifact_path,
                    V0_0_IMMUTABLE_ROOT,
                )
                for artifact_path in (
                    NOTEBOOK_06_TERMINAL_DECISION_PATH,
                    NOTEBOOK_06_FINAL_ACCEPTANCE_PATH,
                    NOTEBOOK_06_OUTPUT_MANIFEST_PATH,
                    NOTEBOOK_06_READBACK_AUDIT_PATH,
                    NOTEBOOK_06_TO_07_HANDOFF_PATH,
                )
            ),
            "evidence": str(
                V0_0_IMMUTABLE_ROOT
            ),
        },
        {
            "gate": (
                "v0_0_reference_hash_stable"
            ),
            "severity": "BLOCKING",
            "passed": (
                v0_0_reference_hash_stable
            ),
            "evidence": (
                v0_0_reference_hash_after
                if (
                    v0_0_reference_hash_after
                    is not None
                )
                else "NO_REFERENCE_PATH_REGISTERED"
            ),
        },
    ]
)

require(
    NOTEBOOK_06_FINAL_PERSISTENCE_GATE_FRAME.loc[
        NOTEBOOK_06_FINAL_PERSISTENCE_GATE_FRAME[
            "severity"
        ].eq(
            "BLOCKING"
        ),
        "passed",
    ].all(),
    (
        "At least one blocking Notebook 06 final persistence "
        "gate failed."
    ),
)


NOTEBOOK_06_COMPLETE = True
NEXT_AUTHORIZED_NOTEBOOK = (
    "07_HAWKES_ESTIMATION.ipynb"
)


# ------------------------------------------------------------
# Display final persisted authority
# ------------------------------------------------------------

display(
    NOTEBOOK_06_TABLE_ARTIFACT_REGISTRY
)

display(
    NOTEBOOK_06_PERSISTENCE_READBACK_AUDIT
)

display(
    NOTEBOOK_06_FINAL_PERSISTENCE_GATE_FRAME
)

print(
    "Notebook 06 persistence and readback verification passed."
)
print(
    f"Persisted authoritative result tables: "
    f"{len(NOTEBOOK_06_TABLE_ARTIFACT_REGISTRY)}."
)
print(
    f"Output manifest: "
    f"{NOTEBOOK_06_OUTPUT_MANIFEST_PATH}"
)
print(
    f"Final acceptance: "
    f"{NOTEBOOK_06_FINAL_ACCEPTANCE_PATH}"
)
print(
    f"Notebook 06 to Notebook 07 handoff: "
    f"{NOTEBOOK_06_TO_07_HANDOFF_PATH}"
)
print(
    f"Notebook 06 terminal status: "
    f"{NOTEBOOK_06_TERMINAL_STATUS}."
)
print(
    "Notebook 07 is authorized under a restricted-first "
    "diagonal bivariate exponential Hawkes contract."
)
print(
    "VALIDATION and ENGINEERING_HOLDOUT event content remain "
    "unopened."
)
print(
    "State-dependent Hawkes, Hawkes-superiority claims, and "
    "strategy or market-making use remain unauthorized."
)

,artifact_type,variable_name,path,format,compression,row_count,column_count,columns,schema_sha256,sha256,byte_count,source_run_prefix,v0_1_run_id,producing_notebook,status
0,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_BURST_CONCENTRATION,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,24,16,"[event_partition, activity_scope, activity_reg...",c98e8d21010d66c561b54e890488889f2bd1c7f7831862...,130d7e127581fe09bced7e7db630b05b99e7daf1a084dd...,1103,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
1,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_BURST_STABILITY_GATE_FRAME,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,12,4,"[gate, severity, passed, evidence]",29f6832dc134de573a795d532c2854ca42be86408cc031...,e6dd4cc157329ac00f057236cf15ae9a3b9afe089881a6...,658,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
2,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_RUN_SUMMARY,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,6,9,"[event_partition, activity_regime, activity_re...",e8a59afa65730c6476a3fc59a36552ae3e99f3053d7412...,3cf87b8839c8ea8cfdddc7188708913281bb3c597dfb64...,362,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
3,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_RUN_TABLE,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,7631,11,"[event_partition, regime_run_number, activity_...",9f6b4cbac3b8f0dde945c36aa7712772dba2a9acfd3c71...,cd3fb9beb8ce723cb88686aff46f8bcb9df098e346927c...,112452,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
4,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_THRESHOLD_TABLE,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,2,8,"[threshold_name, development_quantile, thresho...",1ba0e2be2c686f6a72516e69b6fd591efc0418bf1edbda...,f9b9dc33eea93c2dfee4859629c39671c6387c25d6d7d6...,302,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,NOTEBOOK_06_RESULT_TABLE,V0_0_REFERENCE_NOTEBOOK_CANDIDATE_AUDIT,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,2,9,"[path, sha256, byte_count, ranking_score, cont...",b2c48bc06c94bed64db2a00af74ab50dd49e63fb3d95ca...,11b0555eff9817ae80018abd7d6ee5d26ed89d55b3433e...,349,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
127,NOTEBOOK_06_RESULT_TABLE,V0_0_REFERENCE_RECONCILIATION_GATE_FRAME,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,13,4,"[gate, severity, passed, evidence]",29f6832dc134de573a795d532c2854ca42be86408cc031...,09c0be7f622d0cd0cd030282585a1ddb2d999e3e636e88...,774,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
128,NOTEBOOK_06_RESULT_TABLE,V0_0_TO_V0_1_REFERENCE_RECONCILIATION,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,2,31,"[metric_id, metric_label, reference_value, com...",ff6b768d7d6573020586241a4fcfa35ff9d863baa5cd38...,1fc8bce01af4eec2313ee4df00bb971ab5a0b776bbe34e...,824,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS
129,NOTEBOOK_06_RESULT_TABLE,V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY,D:\Clown Project\V0.1\artifacts\audit_tables\0...,CSV_GZIP,GZIP_MTIME_ZERO,2,14,"[metric_id, partition_scope, width_ms, lag, ob...",d225008a40f3fb31be97777a3df1ef61784cd0db898269...,b4d1fc3d766f264ba7773e8161b1d368558e8b94c977d7...,398,BTCUSDT_spot_20260710T063746Z_c8b5bf12,v0_1_20260714T090616Z_e82325081a81,06_POINT_PROCESS_BASELINES.ipynb,PASS


,artifact_type,artifact_name,path,expected_row_count,readback_row_count,expected_column_count,readback_column_count,hash_matches,row_count_matches,column_count_matches,column_order_matches,path_inside_v0_1,path_inside_v0_0,status
0,NOTEBOOK_06_FINAL_ACCEPTANCE,06_point_process_baselines_final_acceptance,D:\Clown Project\V0.1\artifacts\manifests\06_p...,<NA>,<NA>,<NA>,<NA>,True,True,True,True,True,False,PASS
1,NOTEBOOK_06_OUTPUT_MANIFEST,06_point_process_baselines_output_manifest,D:\Clown Project\V0.1\artifacts\manifests\06_p...,<NA>,<NA>,<NA>,<NA>,True,True,True,True,True,False,PASS
2,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_BURST_CONCENTRATION,D:\Clown Project\V0.1\artifacts\audit_tables\0...,24,24,16,16,True,True,True,True,True,False,PASS
3,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_BURST_STABILITY_GATE_FRAME,D:\Clown Project\V0.1\artifacts\audit_tables\0...,12,12,4,4,True,True,True,True,True,False,PASS
4,NOTEBOOK_06_RESULT_TABLE,ACTIVITY_REGIME_RUN_SUMMARY,D:\Clown Project\V0.1\artifacts\audit_tables\0...,6,6,9,9,True,True,True,True,True,False,PASS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,NOTEBOOK_06_RESULT_TABLE,V0_0_REFERENCE_RECONCILIATION_GATE_FRAME,D:\Clown Project\V0.1\artifacts\audit_tables\0...,13,13,4,4,True,True,True,True,True,False,PASS
130,NOTEBOOK_06_RESULT_TABLE,V0_0_TO_V0_1_REFERENCE_RECONCILIATION,D:\Clown Project\V0.1\artifacts\audit_tables\0...,2,2,31,31,True,True,True,True,True,False,PASS
131,NOTEBOOK_06_RESULT_TABLE,V0_1_REFERENCE_COMPARISON_METRIC_SUMMARY,D:\Clown Project\V0.1\artifacts\audit_tables\0...,2,2,14,14,True,True,True,True,True,False,PASS
132,NOTEBOOK_06_RESULT_TABLE,V0_1_RUN_ID_RESOLUTION_AUDIT,D:\Clown Project\V0.1\artifacts\audit_tables\0...,1,1,7,7,True,True,True,True,True,False,PASS


,gate,severity,passed,evidence
0,authoritative_result_tables_persisted,BLOCKING,True,table_count=131
1,all_result_tables_read_back,BLOCKING,True,verified_artifacts=134
2,terminal_decision_self_hash_verified,BLOCKING,True,5c802aa59fa79feb43c4172a46ddd8a23541c9ed3b76e6...
3,final_acceptance_self_hash_verified,BLOCKING,True,8df9376da776056408531eef3ca528b53f680407e266ed...
4,output_manifest_self_hash_verified,BLOCKING,True,e708a18ea7fd988c7870d8f8121385dcaf6ca3eca3a6a5...
5,readback_audit_self_hash_verified,BLOCKING,True,26001d4bf989a3d05f5bfbc9338094e48a087d63cbbc79...
6,notebook_07_handoff_self_hash_verified,BLOCKING,True,81b9a8463b10bcd48e25de82bb39e378906ce0950c8c34...
7,restricted_first_authorization_persisted,BLOCKING,True,AUTHORIZED_RESTRICTED_FIRST; restricted_first_...
8,protected_partition_content_remains_absent,BLOCKING,True,VALIDATION=False; ENGINEERING_HOLDOUT=False
9,all_outputs_inside_v0_1,BLOCKING,True,D:\Clown Project\V0.1


Notebook 06 persistence and readback verification passed.
Persisted authoritative result tables: 131.
Output manifest: D:\Clown Project\V0.1\artifacts\manifests\06_point_process_baselines_output_manifest.json
Final acceptance: D:\Clown Project\V0.1\artifacts\manifests\06_point_process_baselines_final_acceptance.json
Notebook 06 to Notebook 07 handoff: D:\Clown Project\V0.1\artifacts\handoff\06_point_process_baselines_to_07_hawkes_estimation_handoff.json
Notebook 06 terminal status: PASS_HAWKES_RESTRICTED_FIRST.
Notebook 07 is authorized under a restricted-first diagonal bivariate exponential Hawkes contract.
VALIDATION and ENGINEERING_HOLDOUT event content remain unopened.
State-dependent Hawkes, Hawkes-superiority claims, and strategy or market-making use remain unauthorized.
